# GeoCPU v49: 外部データ6種の取得と検査（CPU だけ・GPU は使いません）

Hugging Face にある公開データ6種を Google Drive に取ってきて、検査して、GeoCPU の学習用の形（JSONL）に直します。
パソコンには何も取ってきません。重い処理はすべてこの Colab の上で動きます。

## 動くこと（上から順に）
0. **予行演習（3c）と自己診断（3d）**: 何も取る前に、Drive のつなぎ直しを数回試し（許可の画面がそのたびに出るかを先に知るため）、データ中のコードを実行する子プロセスが root ではなく nobody で動くこと、危険なコードを拒否することを、この Colab の上で確かめます（Drive が外れるのは 7a・7b のあいだだけで、3d の時点ではまだつながっています。3d は、つながっている Drive が子プロセスに見えたかを `info:` の1行で報告するだけで、合格の数には入れません）。確かめられなければ止まります。弱い代わりの方法には切り替えません（コードを実行するのは coding の python 行だけです。coding を処理しない実行では、この自己診断は飛ばします）。
1. **取得**（約1.9GB）: 6つのデータセットを、ライセンスを先に確認してから、コミットを固定して取得します。大きさと sha256 をハブの値と照合します。データセットごとに、ファイルが Drive に着いてから出所の記録（provenance.json）を書きます。
2. **検査**: 件数をカードの値と照合し（違えば `MISMATCH` と大きく表示）、モデルの語彙に無い文字、長さ、読んで確かめるための100行を調べます。
3. **変換**: 1,900文字以内の GeoCPU の行に直します。python のコードはモデル自身の実行器と同じ規則（使えるのは math・cmath・decimal・fractions・statistics・itertools・functools・operator・collections だけ、クラスや属性代入は不可）で実行し、保存されていた出力を実行結果で置き換えます。**答え自身の保存出力と実行結果が一致した行だけ**を残します。コードを実行しているあいだは Drive をアンマウントします。長い2つ（coding・tengentoppa）は小分け（シャード）にして、できた分から Drive に保存します。
4. **絞り込み**: 完全一致と近い重複（MinHash）、評価データ（JMMLU・JamC-QA・T1-T8・v48 の評価セット・python120 など登録簿の他の評価セット・JaQuAD の dev・HotpotQA の validation）との13文字一致、旧コーパス423,348行との重複、学習行に近い兄弟がいる dev / heldout の行を取り除きます。8b の任意の確認は時間の上限つきで、行は除きません。
5. **記録**: `registry.json` と、日本語の表 `REPORT_JA.md` を書きます。採用する件数は、検査のあとで決まります（事前には決めていません）。保留のデータセット（toolace・wizard）は `external_data_v49_rows/_HOLD_do_not_mix/` に置きます。

## 守っていること
- GPU は使いません。ランタイムは「ハードウェア アクセラレータ: なし（CPU）」のままにしてください。GPU が付いていると、セル3a で止まります。
- Drive には **新しいフォルダにだけ** 書きます（`external_data_v49/`、`external_data_v49_stageA/`、`external_data_v49_rows/`、`v49_acquire_20261109/`）。既にあるファイルを上書きも削除もしません。内容の違うファイルが既にあれば、そこで止まります。書き込みは `.part` に書いてから名前を付け替えるので、途中で止まっても欠けたファイルは残りません。
- パスワード・API キー・トークンは要りません。通信先は huggingface.co（とその配信元）と PyPI だけです。
- 各セルの最後に `PASS` か `FAIL` が出ます。`FAIL` のときはそこで止まり、先へは進みません。
- 途中で止まっても、もう一度「すべてのセルを実行」で続きから動きます（取得済みのファイルは検証して飛ばし、変換済みの行・シャードは Drive から戻し、終わった 7a・6c は保存した結果を使います）。実行中のセルを止めたとき（実行を中断）は、動いている変換プロセスも一緒に止めます。
- Drive は **全体で約20回以上、外してつなぎ直します**（データセットごとの取得のあと、変換の波ごと、書き込みの多いステップのあと）。許可の画面がそのたびに出るかどうかは、セル3c の予行演習で先にわかります。回数を減らしたいときはセル3a の `SHARD_RECORDS` を大きくします（切れたときにやり直す量は増えます）。`FLUSH_DRIVE_AFTER_STEPS` / `UNMOUNT_DRIVE_DURING_CODE_RUNS` を False にすると回数は減りますが、書き込みが Drive に届いたことの確認と、コード実行中に Drive を外す守りがなくなります。

## あなたが押すもの（全部）
1. Colab で開くと出る「このノートブックは Google が作成したものではありません」の画面で **「このまま実行」**（最初の1回だけ）。
2. 「ランタイム → ランタイムのタイプを変更」で **ハードウェア アクセラレータが「なし（CPU）」** になっていること（GPU が付いているとセル3a で止まります）。
3. 「ランタイム → **すべてのセルを実行**」。
4. セル3a で出る **Drive のアクセス許可**（アカウントを選んで「許可」。画面が2〜3枚出ます）。
5. セル3c の予行演習で許可の画面が **もう一度出た** なら、本番でも同じ回数（セル3c が「約N回」と表示します）出ます。そのたびに「許可」を押します（押さないとそのセルは待ち続けます）。出ないなら、あとは放っておけます。
6. 止まったとき（接続が切れた・`FAIL` が出た）は、接続が切れたときは画面の「再接続」を押してから、メッセージの指示を直して、もう一度「すべてのセルを実行」。取得済みのファイル・変換済みの行は使い回されます。セル2が「セッションを再起動」を求めたときだけ、再起動してからやり直します。
7. 7b は時間がかかるので、**ブラウザのタブを開いたままに**してください。7a（試運転）が終わると、全体の所要時間の目安が出ます（実測値ではなく見積もりです）。

## 使い方
- 最後に `REPORT_JA.md` の全文がセル10に出ます。読み方は README_JA.md にあります。
- 一部のデータセットだけ試すとき: セル3a の `ONLY` を `("jaquad", "toolace")` のようにします。試運転は **自分用の出力フォルダ**（`v49_acquire_20261109_only_...`）に書くので、あとの本番（`ONLY = None`）と同じ `RUN_SUFFIX` でぶつかりません。変換済みの行・最終の行は共有されます。

## 結果が違うフォルダに欲しいとき
- 内容が違うファイルが既にあって **出力で** 止まったときは、セル3a の `RUN_SUFFIX` を `'_2'` のように変えて、もう一度実行してください。新しい出力フォルダに書きます。変換済みの行（`external_data_v49_stageA/`）は残るので、絞り込みからやり直すだけで、数時間かかる変換はやり直しません。
- 取得した **元ファイル** が違っていて止まったときだけ、`RAW_SUFFIX` を変えます（新しいフォルダに取り直します。変換済みの行も新しいフォルダ `external_data_v49_stageA<RAW_SUFFIX>/` に作り直します）。
- **変換済みの行（`external_data_v49_stageA/`）で止まったとき**は `RUN_SUFFIX` では直りません（このフォルダは `RUN_SUFFIX` に左右されない）。メッセージの指示どおり、そのフォルダの名前をあなたの手で変えて（削除はしません）、7b をもう一度実行します。


In [ ]:
# 2. 必要なライブラリ（PyPI から。GPU は使いません）
import importlib.metadata as md
import subprocess
import sys

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24", "pyarrow"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
assert r.returncode == 0, "FAIL [2 pip] pip install が失敗しました。上の出力を見てください"
for n in ("huggingface_hub", "pyarrow"):
    loaded = getattr(sys.modules.get(n), "__version__", None)       # 読み込み済みの版と、いま入っている版が違うと、古い版のまま動いてしまう
    assert loaded is None or loaded == md.version(n), f"FAIL [2 pip] {n} は読み込み済みが {loaded}、入れ直した版が {md.version(n)} です。「ランタイム → セッションを再起動」してから、もう一度すべてのセルを実行してください"
print("PASS [2 pip] huggingface_hub", md.version("huggingface_hub"), "pyarrow", md.version("pyarrow"), "python", sys.version.split()[0])


In [ ]:
# 3a. Google Drive をつなぎ、設定を決める（Colab 専用）
import os

try:
    from google.colab import drive
except ImportError as e:
    raise AssertionError("FAIL [3a drive] これは Google Colab の上で動かすノートブックです（パソコンでは動かしません）") from e
# CPU だけで動く作業です。GPU 付きのランタイムのままだと、要らない有料の計算資源を使ってしまうので、ここで止めます
assert not (os.path.exists("/proc/driver/nvidia") or os.path.exists("/dev/nvidiactl")), "FAIL [3a cpu] このランタイムには GPU が付いています。「ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: なし（CPU）」にしてから、もう一度実行してください"
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106"
assert os.path.isdir(DRIVE_ROOT), f"FAIL [3a drive] {DRIVE_ROOT} が見つかりません。Drive のフォルダ名を確かめてください"
RUN_SUFFIX = ""                 # 内容の違うファイルで出力が止まったときだけ '_2' のように変える（新しい出力フォルダ。変換済みの行は使い回す）
RAW_SUFFIX = ""                 # 取得した元ファイルが違っていて止まったときだけ（新しい取得フォルダになります）
STOP_ON_COUNT_MISMATCH = False  # True にすると、件数がカードと違った時点で止まります（False: 大きく表示して記録し、続けます）
ALLOW_VOCAB_MISMATCH = False    # True: モデルの語彙がパソコン上の語彙と違っても進む（既定は止まる。語彙が本当に変わったときだけ）
ONLY = None                     # 試運転で一部だけ動かすとき: ("jaquad", "toolace") など。None なら6つすべて。試運転は自分用の出力フォルダ（..._only_...）に書くので、あとの本番と RUN_SUFFIX がぶつからない
RUN_SANDBOX = True              # コードを実行して検証するのは coding の python 行だけ。セル3d の自己診断が通らないときにだけ False にする（coding の python 行はすべて捨てられ、ほかの5つは進む）
UNMOUNT_DRIVE_DURING_CODE_RUNS = True   # データ中のコードを実行するあいだ Drive を外す（コードが万一抜け出しても Drive に届かないように）
FLUSH_DRIVE_AFTER_STEPS = True          # データセットごとの取得のあと・書き込みの多いステップのあと Drive を外してつなぎ直し、書き込みを確実にする（全体で約20回。セル3c で先に試せる）
SHARD_RECORDS = 20000           # coding と tengentoppa を何件ずつの小分け（シャード）にするか。大きくすると Drive をつなぎ直す回数が減るが、切れたときにやり直す量が増える
SHARD_KEYS = ("coding", "tengentoppa")   # 小分けにするデータセット
MAX_UNSHARDED_MINUTES = 120     # 小分けにしないデータセットの変換が、試運転の見積もりでこの分数を超えるときは、7b が始まる前に止まる
print("PASS [3a drive]", DRIVE_ROOT, "| RUN_SUFFIX =", repr(RUN_SUFFIX), "| RAW_SUFFIX =", repr(RAW_SUFFIX), "| ONLY =", ONLY, "| RUN_SANDBOX =", RUN_SANDBOX)


In [ ]:
#@title 3b. コードを書き出して読み込む（折りたたみ。中身は acquire.py / checks.py / convert.py / pipeline.py と同じです）
import hashlib
import sys
from pathlib import Path

FILES = {
 "acquire.py": (
  "#!/usr/bin/env python3\n"
  "\"\"\"ACQUIRE: fetch the six approved Hugging Face datasets into NEW Google Drive folders, verified and with provenance.\n"
  "\n"
  "Runs on a Google Colab CPU runtime (Drive mounted at /content/drive).  It is deliberately NOT meant to run on the owner's PC:\n"
  "a real (non-dry) run refuses to start off-Colab unless allow_local=True (that switch exists for the unit tests with fake Hub\n"
  "functions).  Nothing here needs a secret: every repo is public and ungated, and every Hub call passes token=False so a token\n"
  "that happens to be present in the environment is never sent, read back or written.\n"
  "\n"
  "Layout written under ``<root>/external_data_v49/`` (root = /content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106):\n"
  "\n"
  "    <repo with / -> __>/README.md                    the dataset card, verbatim (licence text lives here)\n"
  "    <repo with / -> __>/<path in the repo>           TRAIN-eligible files only\n"
  "    <repo with / -> __>/provenance.json              machine-readable record (see build_provenance)\n"
  "    <repo with / -> __>/PROVENANCE.md                the same, for people\n"
  "    do_not_train_dev/<repo__>/<path>                 JaQuAD dev files    (decontamination only, never a converter input)\n"
  "    do_not_train_validation/<repo__>/<path>          HotpotQA validation (decontamination only, never a converter input)\n"
  "    do_not_train_*/DO_NOT_TRAIN.txt                  marker\n"
  "\n"
  "Converters must get their input through train_files(root, repo): it can only return files whose role is \"train\", and it refuses a dataset on\n"
  "adoption hold (ToolACE, WizardLM) unless include_held=True.\n"
  "\n"
  "Rules enforced (the owner's decisions, see the task text):\n"
  "  * licence first: before ANY data file of ANY selected dataset is downloaded, every selected dataset card is fetched and its\n"
  "    front-matter licence must equal the expected one (otherwise LicenseMismatch);\n"
  "  * revision pinned to the commit sha that HfApi().dataset_info(repo).sha reports at run time (recorded);\n"
  "  * every file: size == API size and sha256 == LFS sha256; non-LFS files: size == git blob size and git blob id == API blob id\n"
  "    (sha256 is computed for the record in both cases); a data file that differs from the planning snapshot embedded in\n"
  "    DATASETS (taken from the saved API JSON) raises UpstreamDrift;\n"
  "  * nothing is ever overwritten or deleted on Drive: a final file that exists with the wrong size/sha256 raises\n"
  "    ExistingFileMismatch, a final file that exists with the right ones is skipped, files are written as *.part with open(...,'xb')\n"
  "    and renamed only after verification, a stale *.part is renamed to *.part.stale-N (never overwritten, never removed),\n"
  "    provenance files are write-once (identical record: kept; different record: ProvenanceConflict);\n"
  "  * the heavy transfer goes through a staging directory on the Colab local disk (default /content/_hf_stage_v49, refused if it\n"
  "    is inside the Drive root), so a bad download never reaches Drive;\n"
  "  * durability (Drive/FUSE writes are asynchronous): an fsync error other than 'not supported' is raised, not swallowed; a\n"
  "    commit_pin.json (the commit sha) is written before the first file of a dataset, so a re-run fetches the same commit even when the\n"
  "    card was edited upstream in between; the after_files hook (the notebook: flush_and_unmount + mount) runs after the files of a\n"
  "    dataset are in place and BEFORE provenance.json is written; a SHORT final file left by a run that died before Drive had all its\n"
  "    bytes (no provenance.json yet, bytes exactly the start of the verified file) is moved aside as .stale-N (never deleted) and the\n"
  "    file is written again; every other mismatch stays a refusal;\n"
  "  * a data-like file in the live listing that is neither in the table nor declared `not_fetched` stops the run before any download\n"
  "    (a new train shard would otherwise be silently missed);\n"
  "  * dry_run=True prints the plan and the byte total, touches no network and writes nothing.\n"
  "\n"
  "Colab cell:\n"
  "\n"
  "    !pip install -q huggingface_hub pyarrow\n"
  "    from google.colab import drive; drive.mount('/content/drive')\n"
  "    import sys; sys.path.insert(0, '<folder that holds acquire.py>')\n"
  "    import acquire\n"
  "    ROOT = '/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106'\n"
  "    acquire.acquire(ROOT, dry_run=True)        # prints what would be fetched, ~1.9 GB\n"
  "    res = acquire.acquire(ROOT)                # the real run; idempotent, safe to re-run after a disconnect\n"
  "    drive.flush_and_unmount()                  # Drive uploads in the background: let it finish before the runtime is released\n"
  "\n"
  "The only hosts contacted are huggingface.co and its CDN/xet hosts (*.huggingface.co, *.hf.co), through huggingface_hub; this module\n"
  "opens no other connection.  (hf_xet, if installed, does its own transfers to *.hf.co and cannot be sandboxed from Python.)\n"
  "\"\"\"\n"
  "from __future__ import annotations\n"
  "\n"
  "import argparse\n"
  "import datetime\n"
  "import errno\n"
  "import hashlib\n"
  "import json\n"
  "import os\n"
  "import re\n"
  "import shutil\n"
  "import sys\n"
  "import tempfile\n"
  "import time\n"
  "from pathlib import Path, PurePosixPath\n"
  "from typing import Optional\n"
  "\n"
  "ACQUIRE_VERSION = \"1\"\n"
  "SCHEMA = \"v49_acquire_provenance/1\"\n"
  "OWNER_DECISION = \"approved: all six (2026-11-09)\"\n"
  "DEFAULT_ROOT = \"/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106\"\n"
  "EXTERNAL_DIRNAME = \"external_data_v49\"\n"
  "DEFAULT_STAGE = \"/content/_hf_stage_v49\"\n"
  "ALLOWED_HOSTS = (\"huggingface.co\", \"*.huggingface.co\", \"*.hf.co\")\n"
  "PLANNING_SOURCE = \"Hugging Face API JSON saved 2026-11-09 (hf_meta/*.json) + dataset cards\"\n"
  "CHUNK = 8 << 20\n"
  "\n"
  "# role -> top-level folder (None: inside the dataset folder).  Only role \"train\" files may ever be trained on.\n"
  "ROLE_DIR = {\"train\": None, \"card\": None, \"dev\": \"do_not_train_dev\", \"validation\": \"do_not_train_validation\"}\n"
  "NO_TRAIN_ROLES = (\"dev\", \"validation\")\n"
  "\n"
  "# Hub entry points are resolved lazily so that importing this module (and dry runs) never imports huggingface_hub and the tests\n"
  "# can replace both with fakes.\n"
  "HfApi = None\n"
  "hf_hub_download = None\n"
  "\n"
  "\n"
  "class AcquireError(Exception):\n"
  "    pass\n"
  "\n"
  "\n"
  "class ExistingFileMismatch(AcquireError):\n"
  "    \"\"\"A file already exists at the destination with a different size/sha256 (never overwritten).\"\"\"\n"
  "\n"
  "\n"
  "class VerificationError(AcquireError):\n"
  "    \"\"\"A downloaded or copied file does not match the size / sha256 / git blob id the Hub reports.\"\"\"\n"
  "\n"
  "\n"
  "class UpstreamDrift(VerificationError):\n"
  "    \"\"\"The live Hub listing disagrees with the planning snapshot for a data file.\"\"\"\n"
  "\n"
  "\n"
  "class LicenseMismatch(AcquireError):\n"
  "    pass\n"
  "\n"
  "\n"
  "class ProvenanceConflict(AcquireError):\n"
  "    pass\n"
  "\n"
  "\n"
  "class HeldDataset(AcquireError):\n"
  "    \"\"\"train_files() was asked for a dataset whose adoption status is 'hold' without include_held=True.\"\"\"\n"
  "\n"
  "\n"
  "class UnsafeLayout(AcquireError):\n"
  "    pass\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ the table\n"
  "def F(path, size, role, sha256=None, blob=None, pointer_blob=None):\n"
  "    \"\"\"One file to fetch.  LFS files carry sha256 (the LFS oid); plain git files carry blob (git blob sha1).\"\"\"\n"
  "    return {\"path\": path, \"size\": size, \"role\": role, \"sha256\": sha256, \"blob\": blob, \"pointer_blob\": pointer_blob}\n"
  "\n"
  "\n"
  "_APACHE = (\"Apache License 2.0\", \"https://www.apache.org/licenses/LICENSE-2.0\")\n"
  "_BYSA3 = (\"Creative Commons Attribution-ShareAlike 3.0 (CC BY-SA 3.0)\", \"https://creativecommons.org/licenses/by-sa/3.0/\")\n"
  "_BYSA4 = (\"Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0)\", \"https://creativecommons.org/licenses/by-sa/4.0/\")\n"
  "_PENDING = {\"status\": \"pending_checks\", \"text\": \"adoption: decided after the checks (the number of rows is not fixed in advance)\"}\n"
  "\n"
  "# Sizes, sha256 (LFS oid) and git blob ids below were taken from the saved Hugging Face API JSON (planning snapshot).\n"
  "DATASETS = [\n"
  "    dict(\n"
  "        key=\"coding\", repo=\"llm-jp/Synthetic-JP-EN-Coding-Dataset\", planning_sha=\"f3b782fde6b6ebe7606b42bddea68cb75cfbb547\",\n"
  "        not_fetched=[\".gitattributes\"],\n"
  "        licenses=(\"apache-2.0\",), license_name=_APACHE[0], license_url=_APACHE[1], share_alike=False, row_license=\"Apache-2.0\",\n"
  "        row_source=\"llm-jp/Synthetic-JP-EN-Coding-Dataset (LLM-jp; subset of Aratako/Synthetic-JP-EN-Coding-Dataset-801k) - Apache-2.0 - https://huggingface.co/datasets/llm-jp/Synthetic-JP-EN-Coding-Dataset\",\n"
  "        language=\"ja, en\", rows_card=\"113,796 (train)\",\n"
  "        generator_notes=[\n"
  "            \"Card: a subset of Aratako/Synthetic-JP-EN-Coding-Dataset-801k (code SFT data, Japanese/English mixed).\",\n"
  "            \"Parent card (Aratako 801k): seed = Magpie-made Aratako/Synthetic-JP-EN-Coding-Dataset-Magpie-69k, expanded with Evol-Instruct-like methods; \"\n"
  "            \"instructions were made by nvidia/Nemotron-4-340B-Instruct, microsoft/Phi-3-medium-4k-instruct, mistralai/Mixtral-8x22B-Instruct-v0.1 and \"\n"
  "            \"cyberagent/calm3-22b-chat; the parent card says DeepInfra was used for generation.\",\n"
  "            \"Parent card: NO post-hoc filtering was applied, 'records of low quality may be included'.\",\n"
  "        ],\n"
  "        quality_notes=[\"Many programming languages and both Japanese and English; about 3.6 KB per row on average (406,143,107 bytes / 113,796 rows).\"],\n"
  "        adoption=dict(_PENDING),\n"
  "        attribution=dict(title=\"Synthetic-JP-EN-Coding-Dataset (llm-jp/Synthetic-JP-EN-Coding-Dataset)\",\n"
  "                         authors=\"LLM-jp (Hirokazu Kiyomaru, Takashi Kodama); parent dataset Aratako/Synthetic-JP-EN-Coding-Dataset-801k\",\n"
  "                         source_url=\"https://huggingface.co/datasets/llm-jp/Synthetic-JP-EN-Coding-Dataset\", extra=[]),\n"
  "        check_names=[\"Kiyomaru\", \"Kodama\"],\n"
  "        files=[\n"
  "            F(\"README.md\", 910, \"card\", blob=\"221b1f76afaf4e460edceb3cc88bd4e0c44ced86\"),\n"
  "            F(\"data/train-00000-of-00001.parquet\", 201478456, \"train\", sha256=\"aac23bec3167bdc63550c130cdba7cf80bf4e0ddec5259d62eef609ba5910beb\", pointer_blob=\"222be9ffab6977652ce332929009ca65671e996e\"),\n"
  "        ]),\n"
  "    dict(\n"
  "        key=\"tengentoppa\", repo=\"DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k\", planning_sha=\"fa2497452a9b765e8f1e2b7b08e60b276521c26f\",\n"
  "        not_fetched=[\".gitattributes\"],\n"
  "        licenses=(\"apache-2.0\",), license_name=_APACHE[0], license_url=_APACHE[1], share_alike=False, row_license=\"Apache-2.0\",\n"
  "        row_source=\"DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k (Qwen2.5-32B-Instruct generated) - Apache-2.0 - https://huggingface.co/datasets/DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k\",\n"
  "        language=\"ja\", rows_card=\"125,000 (stated on the card; not re-counted here)\",\n"
  "        generator_notes=[\n"
  "            \"Generator: Qwen2.5-32B-Instruct (card). Temperature 0.8 for instructions and 0.7 otherwise, max_tokens 1024, top_p 0.95, repetition_penalty 1.1; \"\n"
  "            \"one of 10 personas chosen at random for the system prompt.\",\n"
  "            \"Per sample: persona-based instruction -> chain-of-thought answer procedure (reasoning) -> initial_answer -> self-refined answer (refined_answer).\",\n"
  "            \"Card caveat (quoted in Japanese): auto-generated; no human check or edit apart from the instruction text \"\n"
  "            \"(\\u6307\\u793a\\u6587\\u4ee5\\u5916\\u4eba\\u9593\\u306b\\u3088\\u308b\\u78ba\\u8a8d\\u3084\\u7de8\\u96c6\\u306f\\u884c\\u308f\\u308c\\u3066\\u3044\\u307e\\u305b\\u3093); quality and accuracy are not guaranteed.\",\n"
  "        ],\n"
  "        quality_notes=[\"Keys: instruction, reasoning, initial_answer, refined_answer.  About 8 KB per row on average, so many rows will exceed 2,048 characters.\"],\n"
  "        adoption=dict(_PENDING),\n"
  "        attribution=dict(title=\"Tengentoppa-sft-qwen2.5-32b-reasoning-100k (DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k)\",\n"
  "                         authors=\"DeL-TaiseiOzaki (rows generated with Qwen2.5-32B-Instruct)\",\n"
  "                         source_url=\"https://huggingface.co/datasets/DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k\", extra=[]),\n"
  "        check_names=[],\n"
  "        files=[\n"
  "            F(\"README.md\", 2778, \"card\", blob=\"55474df179e79c0a2f102d889c3c61cb55d89c9c\"),\n"
  "            F(\"completed_sets.jsonl\", 997252762, \"train\", sha256=\"80dd010ad1b4213063acb4f9b2188a02783ad6bb546f504a2c8839f6675a4b3d\", pointer_blob=\"dbeb1e49734a8d4127aadca2eecae8a0bc8c9d17\"),\n"
  "        ]),\n"
  "    dict(\n"
  "        key=\"jaquad\", repo=\"SkelterLabsInc/JaQuAD\", planning_sha=\"05600ff310a0970823e70f82f428893b85c71ffe\",\n"
  "        not_fetched=[\".gitattributes\", \"JaQuAD.py\"],\n"
  "        licenses=(\"cc-by-sa-3.0\",), license_name=_BYSA3[0], license_url=_BYSA3[1], share_alike=True, row_license=\"CC-BY-SA-3.0\",\n"
  "        row_source=\"JaQuAD (So et al., Skelter Labs) - CC BY-SA 3.0 - https://huggingface.co/datasets/SkelterLabsInc/JaQuAD\",\n"
  "        language=\"ja\", rows_card=\"train 31,748 questions (9,713 contexts, 691 articles); dev/validation 3,939 (never trained on); test 4,009 is not public; 39,696 in total\",\n"
  "        generator_notes=[\n"
  "            \"Human annotated (crowdsourced): questions and answer spans written by fluent Japanese speakers (natives and non-natives) who were asked for non-trivial questions about a given context.\",\n"
  "            \"Contexts are Japanese Wikipedia paragraphs (88.7% of the articles come from the curated featured/good-article lists).\",\n"
  "            \"Splits are made from disjoint sets of Wikipedia articles.  Only the train files are trainable; the dev files are fetched ONLY for decontamination.\",\n"
  "        ],\n"
  "        quality_notes=[\"Card: most answers are short; every question is assumed answerable from its context; the card says the dataset is 'incomplete yet'.\",\n"
  "                       \"Plain git files (no LFS): integrity is checked with size + git blob id; sha256 is computed for the record.\"],\n"
  "        adoption=dict(_PENDING),\n"
  "        attribution=dict(title=\"JaQuAD: Japanese Question Answering Dataset for Machine Reading Comprehension (SkelterLabsInc/JaQuAD)\",\n"
  "                         authors=\"ByungHoon So, Kyuhong Byun, Kyungwon Kang, Seongjin Cho (Skelter Labs)\",\n"
  "                         source_url=\"https://huggingface.co/datasets/SkelterLabsInc/JaQuAD\",\n"
  "                         extra=[\"Contexts: Japanese Wikipedia article text (CC BY-SA); each article title is kept in the row so the source article stays traceable.\",\n"
  "                                \"ShareAlike: the converted JSONL rows must stay under CC BY-SA 3.0 (or a compatible licence) when they are redistributed.\"]),\n"
  "        check_names=[\"ByungHoon So\", \"Kyuhong Byun\", \"Kyungwon Kang\", \"Seongjin Cho\"],\n"
  "        files=[\n"
  "            F(\"README.md\", 6742, \"card\", blob=\"c81f756f80e3bda212f599390195e5393dcb2fa6\"),\n"
  "            F(\"data/train/jaquad_train_0000.json\", 790020, \"train\", blob=\"15b82703a5b063607616d1129b506c103be22ed3\"),\n"
  "            F(\"data/train/jaquad_train_0001.json\", 814618, \"train\", blob=\"18333166c669abeb549c2f9e9d8cddb4acdbd21a\"),\n"
  "            F(\"data/train/jaquad_train_0002.json\", 843609, \"train\", blob=\"bdb658167c10e55d58f839f0e1f1f7d7cdfdb3a4\"),\n"
  "            F(\"data/train/jaquad_train_0003.json\", 658016, \"train\", blob=\"752e29d000b72cb6e76ea8061180a85f114e4271\"),\n"
  "            F(\"data/train/jaquad_train_0004.json\", 790560, \"train\", blob=\"836f2dca7bbf150f5a20f80dd4d844fa16e4f829\"),\n"
  "            F(\"data/train/jaquad_train_0005.json\", 775597, \"train\", blob=\"62bc65a81a4e8b798303f9b566db27524c86a421\"),\n"
  "            F(\"data/train/jaquad_train_0006.json\", 717697, \"train\", blob=\"89e8bd28c6d744e25cd1d6aaa3acd277f92934fd\"),\n"
  "            F(\"data/train/jaquad_train_0007.json\", 800496, \"train\", blob=\"5f327ecc3f1213c59296307494dbce871d19d220\"),\n"
  "            F(\"data/train/jaquad_train_0008.json\", 752172, \"train\", blob=\"750aedf644c55ddf18e6698c492d8a4fc3df50c4\"),\n"
  "            F(\"data/train/jaquad_train_0009.json\", 796964, \"train\", blob=\"06f489ce355886e1114bc5260262930b1d8ad380\"),\n"
  "            F(\"data/train/jaquad_train_0010.json\", 797173, \"train\", blob=\"664b6993df2b7fc92f1d2b6e36c854ec46b702a6\"),\n"
  "            F(\"data/train/jaquad_train_0011.json\", 785342, \"train\", blob=\"bbc87d5253789d5783595621e54f74b4a37ff7f6\"),\n"
  "            F(\"data/train/jaquad_train_0012.json\", 579524, \"train\", blob=\"c7cdf4173bdb7a407353ac3f9dd445cc72ffd681\"),\n"
  "            F(\"data/train/jaquad_train_0013.json\", 742690, \"train\", blob=\"0e44977533980105a53c682fd0fecf2d3cb7fab8\"),\n"
  "            F(\"data/train/jaquad_train_0014.json\", 728197, \"train\", blob=\"ff8d7510f75b8c0a1b5cd4b99656acf349b7c901\"),\n"
  "            F(\"data/train/jaquad_train_0015.json\", 757375, \"train\", blob=\"15ef62ebfa58fe7209ef0db228ff34f2f93d90b9\"),\n"
  "            F(\"data/train/jaquad_train_0016.json\", 762747, \"train\", blob=\"c7c4e63c14d7a6927624d0bfbe4acaf1d0878dc2\"),\n"
  "            F(\"data/train/jaquad_train_0017.json\", 771769, \"train\", blob=\"627936d2f016c20970a06e621e3e0e38e9ce7d96\"),\n"
  "            F(\"data/train/jaquad_train_0018.json\", 807423, \"train\", blob=\"cb8453dc3530fe0330280fbbffbc51ee4569cb6e\"),\n"
  "            F(\"data/train/jaquad_train_0019.json\", 795900, \"train\", blob=\"5ed94d0ea96f236999126def51c3530aec86b3cc\"),\n"
  "            F(\"data/train/jaquad_train_0020.json\", 775251, \"train\", blob=\"7325f45e40b442ca6fab7f9563b4d6cdb0a4b9dc\"),\n"
  "            F(\"data/train/jaquad_train_0021.json\", 803654, \"train\", blob=\"0f486278419e03ac0ffe9d0dbeb68f7b00482d35\"),\n"
  "            F(\"data/train/jaquad_train_0022.json\", 855399, \"train\", blob=\"a5a9ff12ffef78a65b29b17294bb5ddf4b9e8ea7\"),\n"
  "            F(\"data/train/jaquad_train_0023.json\", 745120, \"train\", blob=\"1cdb3849833167f89ea0209a5453dbe13099dbbd\"),\n"
  "            F(\"data/train/jaquad_train_0024.json\", 770649, \"train\", blob=\"e7e1f2bd48910da34fe8b005a83b69d4edb4659a\"),\n"
  "            F(\"data/train/jaquad_train_0025.json\", 778422, \"train\", blob=\"6e3651824e35a2136eae17259271e6e80753a587\"),\n"
  "            F(\"data/train/jaquad_train_0026.json\", 763484, \"train\", blob=\"c72b52dd31d062520e4c674b8a6a4832f3d64e95\"),\n"
  "            F(\"data/train/jaquad_train_0027.json\", 762216, \"train\", blob=\"d17e706bf3f253edf83a4b32b6e60d54ed6c678d\"),\n"
  "            F(\"data/train/jaquad_train_0028.json\", 945484, \"train\", blob=\"39dab733e8fc61202820f1fa2ba28c8379038887\"),\n"
  "            F(\"data/train/jaquad_train_0029.json\", 245699, \"train\", blob=\"d2bfcbc2b2f56423b036e789e88668f9f1e16315\"),\n"
  "            F(\"data/dev/jaquad_dev_0000.json\", 855457, \"dev\", blob=\"bdc9a782b7d7f24e9ef6fb4d10e8959ebc7508df\"),\n"
  "            F(\"data/dev/jaquad_dev_0001.json\", 800689, \"dev\", blob=\"93a2708ad28959efe78d7a5c669a717debe2c59b\"),\n"
  "            F(\"data/dev/jaquad_dev_0002.json\", 745518, \"dev\", blob=\"4fe6060c912d3fc2444ba4c7eafca42548067420\"),\n"
  "            F(\"data/dev/jaquad_dev_0003.json\", 620520, \"dev\", blob=\"272703ab78b4a853fb5a6784ef9a2895f2fac3e7\"),\n"
  "        ]),\n"
  "    dict(\n"
  "        key=\"hotpotqa\", repo=\"hotpotqa/hotpot_qa\", planning_sha=\"1908d6afbbead072334abe2965f91bd2709910ab\",\n"
  "        not_fetched=[\".gitattributes\", \"fullwiki/\"],                 # the 'fullwiki' config is not part of the plan\n"
  "        licenses=(\"cc-by-sa-4.0\",), license_name=_BYSA4[0], license_url=_BYSA4[1], share_alike=True, row_license=\"CC-BY-SA-4.0\",\n"
  "        row_source=\"HotpotQA (Yang et al., EMNLP 2018) - CC BY-SA 4.0 - https://huggingface.co/datasets/hotpotqa/hotpot_qa\",\n"
  "        language=\"en\", rows_card=\"distractor: train 90,447; validation 7,405 (never trained on)\",\n"
  "        generator_notes=[\n"
  "            \"Crowdsourced multi-hop question answering over English Wikipedia paragraphs (card: annotations_creators crowdsourced, language_creators found).\",\n"
  "            \"Config 'distractor' only (the 'fullwiki' train files are the same blobs and are not fetched).\",\n"
  "        ],\n"
  "        quality_notes=[\"English only; about 6.1 KB per row (552,948,795 bytes / 90,447 rows), so rows do NOT fit 2,048 characters as they are: use only as a skeleton or as short derived rows.\",\n"
  "                       \"The validation file is fetched ONLY for decontamination (do_not_train_validation/).\"],\n"
  "        adoption=dict(_PENDING),\n"
  "        attribution=dict(title=\"HotpotQA: A Dataset for Diverse, Explainable Multi-hop Question Answering (hotpotqa/hotpot_qa)\",\n"
  "                         authors=\"Zhilin Yang, Peng Qi, Saizheng Zhang, Yoshua Bengio, William W. Cohen, Ruslan Salakhutdinov, Christopher D. Manning (arXiv:1809.09600, EMNLP 2018)\",\n"
  "                         source_url=\"https://huggingface.co/datasets/hotpotqa/hotpot_qa\",\n"
  "                         extra=[\"Contexts: English Wikipedia paragraphs (CC BY-SA); paragraph titles are kept in the row so the source article stays traceable.\",\n"
  "                                \"ShareAlike: the converted JSONL rows must stay under CC BY-SA 4.0 (or a compatible licence) when they are redistributed.\"]),\n"
  "        check_names=[\"Zhilin\", \"Saizheng\", \"Bengio\", \"Salakhutdinov\", \"Manning\"],\n"
  "        files=[\n"
  "            F(\"README.md\", 9522, \"card\", blob=\"d077a76e02586322b672e66325ee0e7ec4e3d949\"),\n"
  "            F(\"distractor/train-00000-of-00002.parquet\", 165624177, \"train\", sha256=\"76d3bb3048a7cc73c1958107c0c5872a00d7e7d00c105b81e92f6769e7822e68\", pointer_blob=\"6a5d2fbef658520bd04c00852c1de33030090934\"),\n"
  "            F(\"distractor/train-00001-of-00002.parquet\", 166162479, \"train\", sha256=\"713661628434fbb19fff7392e2e321e4ed107e3c7c7784d0690946e5f722763f\", pointer_blob=\"d52fc5e088d38d834acd60e2751759c90f291fa8\"),\n"
  "            F(\"distractor/validation-00000-of-00001.parquet\", 27452575, \"validation\", sha256=\"c20b638ca82b21d04fe12e14ff417ad05153d4d215a65de54497fca4e972f7c6\", pointer_blob=\"8df32c2341e895698ebbef0a39a055666c15b201\"),\n"
  "        ]),\n"
  "    dict(\n"
  "        key=\"toolace\", repo=\"Team-ACE/ToolACE\", planning_sha=\"6bda777c88d21e5a204703c1ee45597a8fa4f734\",\n"
  "        not_fetched=[\".gitattributes\"],\n"
  "        licenses=(\"apache-2.0\",), license_name=_APACHE[0], license_url=_APACHE[1], share_alike=False, row_license=\"Apache-2.0\",\n"
  "        row_source=\"Team-ACE/ToolACE (Liu et al., arXiv:2409.00920) - Apache-2.0 (per card; generator model unverified) - https://huggingface.co/datasets/Team-ACE/ToolACE\",\n"
  "        language=\"en, zh\", rows_card=\"not stated on the card (size category 10K<n<100K)\",\n"
  "        generator_notes=[\n"
  "            \"ToolACE pipeline (card + paper arXiv:2409.00920): Tool Self-evolution Synthesis (pool of 26,507 APIs), multi-agent Self-guided Dialog Generation, \"\n"
  "            \"Dual-Layer Verification (rule checks + model checks).\",\n"
  "            \"The generator model is NOT stated on the card, and the paper text saved in hf_meta names no generator model either: UNVERIFIED.\",\n"
  "        ],\n"
  "        quality_notes=[\"English and Chinese function-calling dialogues.  Downloaded and checked only: adoption is gated on the generator being verified.\"],\n"
  "        adoption={\"status\": \"hold\", \"text\": \"adoption: hold (generator unverified)\"},\n"
  "        attribution=dict(title=\"ToolACE (Team-ACE/ToolACE): ToolACE: Winning the Points of LLM Function Calling\",\n"
  "                         authors=\"Weiwen Liu, Xu Huang, Xingshan Zeng, et al. (Team-ACE; arXiv:2409.00920)\",\n"
  "                         source_url=\"https://huggingface.co/datasets/Team-ACE/ToolACE\", extra=[]),\n"
  "        check_names=[\"Weiwen Liu\", \"Xingshan Zeng\"],\n"
  "        files=[\n"
  "            F(\"README.md\", 1873, \"card\", blob=\"e42187577a300679b9e59dfd4d715070f05c7e7c\"),\n"
  "            F(\"data.json\", 37154735, \"train\", sha256=\"ba12c083fca7e8da48c67ad5b895e495447da7c66e39a2e19742c082e6cb537e\", pointer_blob=\"466498d8464a9a4aa9ce641edcfe6a564fb3a7b2\"),\n"
  "        ]),\n"
  "    dict(\n"
  "        key=\"wizardlm\", repo=\"llm-jp/wizardlm8x22b-logical-math-coding-sft-ja\", planning_sha=\"568b89c1f7e5b81e30096efe16b2506418fa2c4a\",\n"
  "        not_fetched=[\".gitattributes\"],\n"
  "        licenses=(\"apache-2.0\",), license_name=_APACHE[0], license_url=_APACHE[1], share_alike=False, row_license=\"Apache-2.0\",\n"
  "        row_source=\"llm-jp/wizardlm8x22b-logical-math-coding-sft-ja (LLM-jp; subset of kanhatakeyama/wizardlm8x22b-logical-math-coding-sft-ja) - Apache-2.0 - https://huggingface.co/datasets/llm-jp/wizardlm8x22b-logical-math-coding-sft-ja\",\n"
  "        language=\"ja\", rows_card=\"116,422 (train)\",\n"
  "        generator_notes=[\n"
  "            \"Card: subset of kanhatakeyama/wizardlm8x22b-logical-math-coding-sft-ja and kanhatakeyama/wizardlm8x22b-logical-math-coding-sft_additional-ja.\",\n"
  "            \"The generator is a WizardLM 8x22B model by the repo/source-dataset names only; this card does not state it.\",\n"
  "            \"The owner's own 2026-11-06 materials plan (Drive materials_plan_20261106.md, file id 1Qq2mHpS9fzlk7dpZv8dwjJFZ3cxSq9IL) lists it under 'avoid: obvious mathematical errors'.\",\n"
  "        ],\n"
  "        quality_notes=[\"Logical / math / coding SFT, Japanese.  Downloaded and checked only: use it only if the checks show that the error rate is small.\"],\n"
  "        adoption={\"status\": \"hold\", \"text\": \"adoption: low priority/hold unless the checks show the error rate is small\"},\n"
  "        attribution=dict(title=\"wizardlm8x22b-logical-math-coding-sft-ja (llm-jp/wizardlm8x22b-logical-math-coding-sft-ja)\",\n"
  "                         authors=\"LLM-jp (Hirokazu Kiyomaru, Takashi Kodama); source datasets by kanhatakeyama\",\n"
  "                         source_url=\"https://huggingface.co/datasets/llm-jp/wizardlm8x22b-logical-math-coding-sft-ja\", extra=[]),\n"
  "        check_names=[\"Kiyomaru\", \"Kodama\"],\n"
  "        files=[\n"
  "            F(\"README.md\", 1084, \"card\", blob=\"808bd2f4502178f7bb8c289af2b33ef5eab89121\"),\n"
  "            F(\"data/train-00000-of-00002.parquet\", 149685396, \"train\", sha256=\"f0f96b9a6d31150d38f096fe095007591233bcca5ddee872a7c0b85ed84632ee\", pointer_blob=\"22fb6540b22f3c3271f4b624404e1861084e494d\"),\n"
  "            F(\"data/train-00001-of-00002.parquet\", 147648584, \"train\", sha256=\"f2fd710a79b6901de3d5feee18b27c05ea4c40dc7744d86d5229efe6c7d8dbbf\", pointer_blob=\"298a93a85672d4a5e643e383526b94d37d9407b2\"),\n"
  "        ]),\n"
  "]\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ small helpers\n"
  "def folder_of(repo: str) -> str:\n"
  "    return repo.replace(\"/\", \"__\")\n"
  "\n"
  "\n"
  "def fmt_bytes(n: int) -> str:\n"
  "    return f\"{n:,} B\"\n"
  "\n"
  "\n"
  "def _safe_rel(path: str) -> str:\n"
  "    p = PurePosixPath(path)\n"
  "    if not path or \"\\\\\" in path or p.is_absolute() or any(part in (\"\", \".\", \"..\") for part in p.parts) or str(p) != path:\n"
  "        raise UnsafeLayout(f\"unsafe relative path in table: {path!r}\")\n"
  "    return path\n"
  "\n"
  "\n"
  "def dest_rel(ds: dict, f: dict) -> str:\n"
  "    \"\"\"Destination path of a file, relative to <root>/external_data_v49/ (posix separators).\"\"\"\n"
  "    rf = _safe_rel(f[\"path\"])\n"
  "    d = folder_of(ds[\"repo\"])\n"
  "    role_dir = ROLE_DIR[f[\"role\"]]\n"
  "    return f\"{role_dir}/{d}/{rf}\" if role_dir else f\"{d}/{rf}\"\n"
  "\n"
  "\n"
  "def _is_within(child, parent) -> bool:\n"
  "    try:\n"
  "        c, p = os.path.realpath(child), os.path.realpath(parent)\n"
  "        return os.path.commonpath([c, p]) == p\n"
  "    except ValueError:                                  # different drives on Windows\n"
  "        return False\n"
  "\n"
  "\n"
  "def _g(obj, name, default=None):\n"
  "    \"\"\"Attribute-or-key access: the Hub returns objects, test fakes and the saved JSON use dicts.\"\"\"\n"
  "    if obj is None:\n"
  "        return default\n"
  "    if isinstance(obj, dict):\n"
  "        return obj.get(name, default)\n"
  "    return getattr(obj, name, default)\n"
  "\n"
  "\n"
  "def validate_table(datasets) -> None:\n"
  "    seen_repo, seen_dest = set(), set()\n"
  "    for ds in datasets:\n"
  "        repo = ds[\"repo\"]\n"
  "        if repo in seen_repo:\n"
  "            raise AcquireError(f\"duplicate repo in table: {repo}\")\n"
  "        seen_repo.add(repo)\n"
  "        cards = [f for f in ds[\"files\"] if f[\"role\"] == \"card\"]\n"
  "        if len(cards) != 1:\n"
  "            raise AcquireError(f\"{repo}: exactly one card file required, found {len(cards)}\")\n"
  "        if not ds.get(\"licenses\"):\n"
  "            raise AcquireError(f\"{repo}: expected licence ids missing\")\n"
  "        for f in ds[\"files\"]:\n"
  "            if f[\"role\"] not in ROLE_DIR:\n"
  "                raise AcquireError(f\"{repo}: bad role {f['role']!r}\")\n"
  "            if not isinstance(f[\"size\"], int) or f[\"size\"] <= 0:\n"
  "                raise AcquireError(f\"{repo}:{f['path']}: bad size\")\n"
  "            if bool(f[\"sha256\"]) == bool(f[\"blob\"]):\n"
  "                raise AcquireError(f\"{repo}:{f['path']}: exactly one of sha256 (LFS) / blob (git) is required\")\n"
  "            if f[\"sha256\"] and not re.fullmatch(r\"[0-9a-f]{64}\", f[\"sha256\"]):\n"
  "                raise AcquireError(f\"{repo}:{f['path']}: bad sha256\")\n"
  "            if f[\"blob\"] and not re.fullmatch(r\"[0-9a-f]{40}\", f[\"blob\"]):\n"
  "                raise AcquireError(f\"{repo}:{f['path']}: bad git blob id\")\n"
  "            d = dest_rel(ds, f)\n"
  "            if d in seen_dest:\n"
  "                raise AcquireError(f\"destination collision: {d}\")\n"
  "            seen_dest.add(d)\n"
  "\n"
  "\n"
  "def select(datasets, only):\n"
  "    if only is None:\n"
  "        return list(datasets)\n"
  "    if isinstance(only, str):\n"
  "        only = [x for x in only.split(\",\") if x.strip()]\n"
  "    out = []\n"
  "    for o in only:\n"
  "        o = o.strip()\n"
  "        m = [d for d in datasets if o in (d[\"key\"], d[\"repo\"], folder_of(d[\"repo\"]))]\n"
  "        if len(m) != 1:\n"
  "            raise ValueError(f\"unknown dataset {o!r}; known: \" + \", \".join(f\"{d['key']} ({d['repo']})\" for d in datasets))\n"
  "        if m[0] not in out:\n"
  "            out.append(m[0])\n"
  "    return out\n"
  "\n"
  "\n"
  "def total_bytes(datasets, roles=None) -> int:\n"
  "    return sum(f[\"size\"] for ds in datasets for f in ds[\"files\"] if roles is None or f[\"role\"] in roles)\n"
  "\n"
  "\n"
  "def file_digests(path, git_blob: bool = False) -> dict:\n"
  "    \"\"\"size, sha256 and (optionally) the git blob sha1, in one streaming pass.\"\"\"\n"
  "    size = os.stat(path).st_size\n"
  "    h = hashlib.sha256()\n"
  "    g = hashlib.sha1(b\"blob %d\\0\" % size) if git_blob else None\n"
  "    with open(path, \"rb\") as fh:\n"
  "        while True:\n"
  "            b = fh.read(CHUNK)\n"
  "            if not b:\n"
  "                break\n"
  "            h.update(b)\n"
  "            if g is not None:\n"
  "                g.update(b)\n"
  "    return {\"size\": size, \"sha256\": h.hexdigest(), \"git_blob_sha1\": g.hexdigest() if g is not None else None}\n"
  "\n"
  "\n"
  "def _digest_errors(dg: dict, exp: dict) -> list:\n"
  "    errs = []\n"
  "    if dg[\"size\"] != exp[\"size\"]:\n"
  "        errs.append(f\"size {dg['size']} != expected {exp['size']}\")\n"
  "    if exp[\"sha256\"]:\n"
  "        if dg[\"sha256\"] != exp[\"sha256\"]:\n"
  "            errs.append(f\"sha256 {dg['sha256']} != expected {exp['sha256']}\")\n"
  "    elif exp[\"blob\"]:\n"
  "        if dg[\"git_blob_sha1\"] != exp[\"blob\"]:\n"
  "            errs.append(f\"git blob id {dg['git_blob_sha1']} != expected {exp['blob']}\")\n"
  "    else:\n"
  "        errs.append(\"nothing to verify against (the Hub gave neither an LFS sha256 nor a git blob id)\")\n"
  "    return errs\n"
  "\n"
  "\n"
  "def parse_card_licenses(card_text: str):\n"
  "    \"\"\"(licence ids lower-cased, front-matter text) from the YAML header of a dataset card; no PyYAML needed.\"\"\"\n"
  "    m = re.match(r\"^\\ufeff?---[ \\t]*\\r?\\n(.*?)\\r?\\n---[ \\t]*(\\r?\\n|$)\", card_text, re.S)\n"
  "    if not m:\n"
  "        return [], \"\"\n"
  "    fm = m.group(1)\n"
  "    lines = fm.splitlines()\n"
  "    ids = []\n"
  "    for i, line in enumerate(lines):\n"
  "        mm = re.match(r\"^license\\s*:\\s*(.*?)\\s*$\", line)\n"
  "        if not mm:\n"
  "            continue\n"
  "        v = mm.group(1)\n"
  "        if v.startswith(\"[\"):\n"
  "            ids.extend(x.strip().strip(\"'\\\"\") for x in v.strip(\"[]\").split(\",\") if x.strip())\n"
  "        elif v:\n"
  "            ids.append(v.strip(\"'\\\"\"))\n"
  "        else:\n"
  "            for nxt in lines[i + 1:]:\n"
  "                m2 = re.match(r\"^\\s*-\\s*(.+?)\\s*$\", nxt)\n"
  "                if not m2:\n"
  "                    break\n"
  "                ids.append(m2.group(1).strip(\"'\\\"\"))\n"
  "    return [x.lower() for x in ids], fm\n"
  "\n"
  "\n"
  "def card_license_excerpt(card_text: str, limit: int = 8) -> list:\n"
  "    out = []\n"
  "    for line in card_text.splitlines():\n"
  "        if re.search(r\"licen[sc]e|\\u30e9\\u30a4\\u30bb\\u30f3\\u30b9\", line, re.I):\n"
  "            out.append(line.strip()[:300])\n"
  "            if len(out) >= limit:\n"
  "                break\n"
  "    return out\n"
  "\n"
  "\n"
  "def attribution_lines(ds: dict) -> list:\n"
  "    a = ds[\"attribution\"]\n"
  "    lines = [f\"Title: {a['title']}\",\n"
  "             f\"Author/source: {a['authors']}; {a['source_url']}\",\n"
  "             f\"License: {ds['license_name']} <{ds['license_url']}>\",\n"
  "             \"Changes: converted to JSONL rows\"]\n"
  "    lines.extend(a.get(\"extra\", []))\n"
  "    return lines\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ Hub access (lazy)\n"
  "def _hub():\n"
  "    global HfApi, hf_hub_download\n"
  "    if HfApi is None or hf_hub_download is None:\n"
  "        try:\n"
  "            import huggingface_hub as hh\n"
  "        except ImportError as e:               # pragma: no cover - environment dependent\n"
  "            raise AcquireError(\"huggingface_hub is not installed; run: pip install -q huggingface_hub pyarrow\") from e\n"
  "        if HfApi is None:\n"
  "            HfApi = hh.HfApi\n"
  "        if hf_hub_download is None:\n"
  "            hf_hub_download = hh.hf_hub_download\n"
  "    return HfApi, hf_hub_download\n"
  "\n"
  "\n"
  "def _live_listing(info) -> dict:\n"
  "    out = {}\n"
  "    for s in (_g(info, \"siblings\") or []):\n"
  "        name = _g(s, \"rfilename\")\n"
  "        lfs = _g(s, \"lfs\")\n"
  "        out[name] = {\"size\": _g(s, \"size\"), \"blob\": _g(s, \"blob_id\") or _g(s, \"blobId\"), \"is_lfs\": lfs is not None,\n"
  "                     \"lfs_sha256\": _g(lfs, \"sha256\") if lfs is not None else None}\n"
  "    return out\n"
  "\n"
  "\n"
  "def _expected_for(ds: dict, f: dict, live: dict):\n"
  "    \"\"\"(what the downloaded bytes must match = the LIVE Hub values, drift vs the planning snapshot).  Raises for data files that drifted.\"\"\"\n"
  "    repo = ds[\"repo\"]\n"
  "    L = live.get(f[\"path\"])\n"
  "    if L is None:\n"
  "        raise UpstreamDrift(f\"{repo}:{f['path']} is not in the live Hub listing at the pinned commit\")\n"
  "    drift = []\n"
  "    if L[\"size\"] != f[\"size\"]:\n"
  "        drift.append(f\"size live {L['size']} vs planning {f['size']}\")\n"
  "    if f[\"sha256\"]:\n"
  "        if L[\"lfs_sha256\"] != f[\"sha256\"]:\n"
  "            drift.append(f\"lfs sha256 live {L['lfs_sha256']} vs planning {f['sha256']}\")\n"
  "    elif L[\"blob\"] != f[\"blob\"] or L[\"is_lfs\"]:\n"
  "        drift.append(f\"git blob id live {L['blob']} (lfs={L['is_lfs']}) vs planning {f['blob']}\")\n"
  "    if drift and f[\"role\"] != \"card\":\n"
  "        raise UpstreamDrift(f\"{repo}:{f['path']} differs from the planning snapshot ({'; '.join(drift)}); stop and re-plan, nothing was downloaded\")\n"
  "    exp = {\"size\": L[\"size\"], \"sha256\": L[\"lfs_sha256\"] if L[\"is_lfs\"] else None, \"blob\": None if L[\"is_lfs\"] else L[\"blob\"]}\n"
  "    return exp, drift\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ never-overwrite file ops\n"
  "def _quarantine(path: str) -> str:\n"
  "    n = 1\n"
  "    while os.path.lexists(f\"{path}.stale-{n}\"):\n"
  "        n += 1\n"
  "    dst = f\"{path}.stale-{n}\"\n"
  "    os.rename(path, dst)\n"
  "    return dst\n"
  "\n"
  "\n"
  "_FSYNC_UNSUPPORTED = {getattr(errno, n) for n in (\"EINVAL\", \"ENOTSUP\", \"EOPNOTSUPP\", \"ENOSYS\") if hasattr(errno, n)}\n"
  "\n"
  "\n"
  "def _fsync(fd: int) -> None:\n"
  "    \"\"\"os.fsync that ignores ONLY 'this file system cannot fsync' (EINVAL / ENOTSUP / EOPNOTSUPP / ENOSYS).  EIO, ENOSPC, EDQUOT ... are raised:\n"
  "    on a FUSE mount such as Google Drive the quota-full or I/O error of buffered writes shows up here, and swallowing it would record a\n"
  "    failed write as a successful one (the read-back after the copy is served from the local cache and cannot prove the upload).\"\"\"\n"
  "    try:\n"
  "        os.fsync(fd)\n"
  "    except OSError as e:\n"
  "        if e.errno not in _FSYNC_UNSUPPORTED:\n"
  "            raise\n"
  "\n"
  "\n"
  "def _fsync_path(path) -> None:\n"
  "    \"\"\"Flush an existing file again (a complete .part of a run whose own flush failed or never happened).  Opens it r+b: nothing is written.\"\"\"\n"
  "    with open(path, \"rb+\") as f:\n"
  "        _fsync(f.fileno())\n"
  "\n"
  "\n"
  "def _is_prefix_of(short, full) -> bool:\n"
  "    \"\"\"True when the bytes of `short` are exactly the first bytes of `full` (compared byte for byte).\"\"\"\n"
  "    n = os.path.getsize(short)\n"
  "    if n > os.path.getsize(full):\n"
  "        return False\n"
  "    with open(short, \"rb\") as a, open(full, \"rb\") as b:\n"
  "        left = n\n"
  "        while left:\n"
  "            x, y = a.read(min(CHUNK, left)), b.read(min(CHUNK, left))\n"
  "            if not x or x != y:\n"
  "                return False\n"
  "            left -= len(x)\n"
  "    return True\n"
  "\n"
  "\n"
  "def _copy_hashing(src, dst):\n"
  "    h = hashlib.sha256()\n"
  "    n = 0\n"
  "    with open(src, \"rb\") as fi, open(dst, \"xb\") as fo:         # 'x': fails instead of overwriting\n"
  "        while True:\n"
  "            b = fi.read(CHUNK)\n"
  "            if not b:\n"
  "                break\n"
  "            fo.write(b)\n"
  "            h.update(b)\n"
  "            n += len(b)\n"
  "        fo.flush()\n"
  "        _fsync(fo.fileno())\n"
  "    return n, h.hexdigest()\n"
  "\n"
  "\n"
  "def _copy_resume(src, dst):\n"
  "    \"\"\"Copy src to dst (never over a finished file).  An existing dst - the '.part' of a dead run - is CONTINUED when its bytes are exactly the first\n"
  "    bytes of src (compared byte for byte); anything else is moved aside by _quarantine (never deleted) and the copy starts again.  A runtime that\n"
  "    died after 900 MB of a 1 GB copy therefore costs 100 MB, not 1 GB, and leaves no new '.stale-N' file behind.\n"
  "    Returns (bytes in dst, sha256 of all of them, bytes that were already there).\"\"\"\n"
  "    h = hashlib.sha256()\n"
  "    n = resumed = 0\n"
  "    mode = \"xb\"\n"
  "    if os.path.lexists(dst):\n"
  "        have = os.path.getsize(dst) if os.path.isfile(dst) else -1\n"
  "        ok = 0 <= have <= os.path.getsize(src)\n"
  "        if ok:\n"
  "            with open(src, \"rb\") as fs, open(dst, \"rb\") as fd:\n"
  "                left = have\n"
  "                while left:\n"
  "                    a, b = fs.read(min(CHUNK, left)), fd.read(min(CHUNK, left))\n"
  "                    if not a or a != b:\n"
  "                        ok = False\n"
  "                        break\n"
  "                    h.update(a)\n"
  "                    n += len(a)\n"
  "                    left -= len(a)\n"
  "        if ok:\n"
  "            mode, resumed = \"ab\", n\n"
  "        else:\n"
  "            _quarantine(str(dst))\n"
  "            h, n = hashlib.sha256(), 0\n"
  "    with open(src, \"rb\") as fi, open(dst, mode) as fo:\n"
  "        fi.seek(n)\n"
  "        while True:\n"
  "            b = fi.read(CHUNK)\n"
  "            if not b:\n"
  "                break\n"
  "            fo.write(b)\n"
  "            h.update(b)\n"
  "            n += len(b)\n"
  "        fo.flush()\n"
  "        _fsync(fo.fileno())\n"
  "    return n, h.hexdigest(), resumed\n"
  "\n"
  "\n"
  "def _write_new_text(path: Path, text: str) -> None:\n"
  "    \"\"\"Create a text file that did not exist; never overwrites.  Written to <path>.part, flushed (a failed flush is raised) and renamed.\"\"\"\n"
  "    if os.path.lexists(path):\n"
  "        raise ExistingFileMismatch(f\"refusing to overwrite {path}\")\n"
  "    path.parent.mkdir(parents=True, exist_ok=True)\n"
  "    part = str(path) + \".part\"\n"
  "    if os.path.lexists(part):\n"
  "        _quarantine(part)\n"
  "    with open(part, \"x\", encoding=\"utf-8\", newline=\"\\n\") as fh:\n"
  "        fh.write(text)\n"
  "        fh.flush()\n"
  "        _fsync(fh.fileno())\n"
  "    if os.path.lexists(path):\n"
  "        raise ExistingFileMismatch(f\"{path} appeared while it was being written; NOT overwritten (the finished text stays at {part})\")\n"
  "    os.rename(part, path)\n"
  "\n"
  "\n"
  "class _Ctx:\n"
  "    pass\n"
  "\n"
  "\n"
  "def _verify_existing(final: Path, exp: dict, rel: str) -> dict:\n"
  "    if not os.path.isfile(final):\n"
  "        raise ExistingFileMismatch(f\"{rel} exists but is not a regular file: {final}\")\n"
  "    dg = file_digests(final, git_blob=exp[\"blob\"] is not None)\n"
  "    errs = _digest_errors(dg, exp)\n"
  "    if errs:\n"
  "        raise ExistingFileMismatch(f\"{rel} already exists with different content ({'; '.join(errs)}); NOT overwritten. \"\n"
  "                                   f\"Use a fresh root or move the old file by hand.\")\n"
  "    return dg\n"
  "\n"
  "\n"
  "def _stage(ctx, ds: dict, f: dict, sha: str, exp: dict):\n"
  "    \"\"\"Make a verified copy of one repo file in the staging dir (local disk).  Returns (path, digests).\"\"\"\n"
  "    local_dir = ctx.stage / folder_of(ds[\"repo\"])\n"
  "    target = local_dir / f[\"path\"]\n"
  "    want_git = exp[\"blob\"] is not None\n"
  "    if os.path.lexists(target):\n"
  "        dg = file_digests(target, want_git)\n"
  "        if not _digest_errors(dg, exp):\n"
  "            ctx.log(f\"  staged copy already verified: {f['path']}\")\n"
  "            return target, dg\n"
  "        if not _is_within(target, ctx.stage):\n"
  "            raise UnsafeLayout(f\"refusing to remove {target}: outside the staging dir\")\n"
  "        ctx.log(f\"  stale staged copy of {f['path']} (staging dir only): replacing\")\n"
  "        os.remove(target)\n"
  "    local_dir.mkdir(parents=True, exist_ok=True)\n"
  "    ctx.log(f\"  downloading {ds['repo']}:{f['path']}  {fmt_bytes(exp['size'])}  @ {sha[:12]}\")\n"
  "    got = ctx.download(repo_id=ds[\"repo\"], filename=f[\"path\"], repo_type=\"dataset\", revision=sha, local_dir=str(local_dir), token=False)\n"
  "    got = Path(got)\n"
  "    if not os.path.isfile(got):\n"
  "        raise VerificationError(f\"{ds['repo']}:{f['path']}: hf_hub_download returned {got}, which is not a file\")\n"
  "    dg = file_digests(got, want_git)\n"
  "    errs = _digest_errors(dg, exp)\n"
  "    if errs:\n"
  "        raise VerificationError(f\"{ds['repo']}:{f['path']} failed verification ({'; '.join(errs)}); nothing was written to Drive; \"\n"
  "                                f\"the bad download stays at {got} and is replaced on the next run\")\n"
  "    return got, dg\n"
  "\n"
  "\n"
  "def _commit(ctx, staged: Path, dg: dict, final: Path, rel: str) -> None:\n"
  "    \"\"\"staged -> <final>.part (never over an existing file) -> verified -> renamed to final.\"\"\"\n"
  "    final.parent.mkdir(parents=True, exist_ok=True)\n"
  "    part = str(final) + \".part\"\n"
  "    reuse = False\n"
  "    if os.path.lexists(part):\n"
  "        if os.path.isfile(part) and os.path.getsize(part) == dg[\"size\"] and file_digests(part)[\"sha256\"] == dg[\"sha256\"]:\n"
  "            reuse = True\n"
  "            ctx.log(f\"  complete {os.path.basename(part)} from an earlier run is correct: finishing it\")\n"
  "            _fsync_path(part)                                       # its own flush may have failed (or never happened) in the run that wrote it\n"
  "    if not reuse:\n"
  "        had = os.path.lexists(part)\n"
  "        n, sha, resumed = _copy_resume(staged, part)               # continues a partial of an earlier run, or moves a bad one aside\n"
  "        if had and resumed:\n"
  "            ctx.log(f\"  continued the partial copy {os.path.basename(part)} of an earlier run after {resumed:,} bytes\")\n"
  "        elif had:\n"
  "            ctx.log(f\"  the partial copy {os.path.basename(part)} of an earlier run did not match the file: moved aside (not deleted), copying again\")\n"
  "        if n != dg[\"size\"] or sha != dg[\"sha256\"]:\n"
  "            raise VerificationError(f\"{rel}: the copy on Drive differs from the verified staged file (kept as {part} for inspection)\")\n"
  "        if ctx.verify_after_copy:\n"
  "            d2 = file_digests(part)\n"
  "            if d2[\"size\"] != dg[\"size\"] or d2[\"sha256\"] != dg[\"sha256\"]:\n"
  "                raise VerificationError(f\"{rel}: read-back of the Drive copy does not match (kept as {part} for inspection)\")\n"
  "    if os.path.lexists(final):\n"
  "        raise ExistingFileMismatch(f\"{rel} appeared while it was being written; NOT overwritten (the finished copy stays at {part})\")\n"
  "    os.rename(part, final)\n"
  "\n"
  "\n"
  "def _ensure_file(ctx, st: dict, f: dict) -> dict:\n"
  "    ds = st[\"ds\"]\n"
  "    exp = st[\"exp\"][f[\"path\"]]\n"
  "    rel = dest_rel(ds, f)\n"
  "    final = ctx.ext / rel\n"
  "    t0 = time.time()\n"
  "    if os.path.lexists(final):\n"
  "        try:\n"
  "            dg = _verify_existing(final, exp, rel)\n"
  "            status = \"skipped_existing\"\n"
  "            ctx.log(f\"  present and verified, skipped: {rel}\")\n"
  "        except ExistingFileMismatch as err:\n"
  "            # A runtime that died after the rename but before Drive had uploaded all the bytes leaves a SHORT final file.  While the dataset has no\n"
  "            # provenance.json (it is written last, after the hook that flushes Drive) such a file is a remnant of this pipeline's own interrupted\n"
  "            # copy, never a certified one: if its bytes are exactly the start of the verified file it is moved aside (renamed, never deleted) and\n"
  "            # the file is written again.  Anything else (other bytes, same size, longer, or a dataset that already has provenance) stays a refusal.\n"
  "            if not (os.path.isfile(final) and os.path.getsize(final) < exp[\"size\"]\n"
  "                    and not _has_readable_provenance(ctx.ext, ds)):\n"
  "                raise\n"
  "            staged, dg = _stage(ctx, ds, f, st[\"sha\"], exp)\n"
  "            if not _is_prefix_of(final, staged):\n"
  "                raise err\n"
  "            had = os.path.getsize(final)\n"
  "            moved = _quarantine(str(final))\n"
  "            ctx.log(f\"  {rel}: a shorter copy ({had:,} of {dg['size']:,} bytes, exactly the start of the file) from a run that died before Drive had all \"\n"
  "                    f\"the bytes was moved aside as {os.path.basename(moved)} (not deleted); copying the file again\")\n"
  "            _commit(ctx, staged, dg, final, rel)\n"
  "            status = \"downloaded\"\n"
  "            if not ctx.keep_stage and _is_within(staged, ctx.stage):\n"
  "                os.remove(staged)\n"
  "            ctx.log(f\"  stored: {rel}\")\n"
  "    else:\n"
  "        staged, dg = _stage(ctx, ds, f, st[\"sha\"], exp)\n"
  "        _commit(ctx, staged, dg, final, rel)\n"
  "        status = \"downloaded\"\n"
  "        if not ctx.keep_stage and _is_within(staged, ctx.stage):\n"
  "            os.remove(staged)\n"
  "        ctx.log(f\"  stored: {rel}\")\n"
  "    checks = [\"size == Hub size\", \"sha256 == LFS oid\" if exp[\"sha256\"] else \"git blob id == Hub blob id (sha256 computed for the record)\"]\n"
  "    return {\"path\": f[\"path\"], \"role\": f[\"role\"], \"may_train\": f[\"role\"] == \"train\", \"dest_rel\": rel, \"size\": dg[\"size\"], \"sha256\": dg[\"sha256\"],\n"
  "            \"git_blob_sha1\": dg[\"git_blob_sha1\"], \"storage\": \"lfs\" if exp[\"sha256\"] else \"git\", \"verified_against_hub\": checks, \"status\": status,\n"
  "            \"seconds\": round(time.time() - t0, 2),\n"
  "            \"resolve_url\": f\"https://huggingface.co/datasets/{ds['repo']}/resolve/{st['sha']}/{f['path']}\",\n"
  "            \"drift_from_planning_snapshot\": st[\"drift\"][f[\"path\"]]}\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ provenance\n"
  "def build_provenance(ctx, st: dict, records: list) -> dict:\n"
  "    ds = st[\"ds\"]\n"
  "    by_role = {}\n"
  "    for r in records:\n"
  "        by_role[r[\"role\"]] = by_role.get(r[\"role\"], 0) + r[\"size\"]\n"
  "    names_found = {n: (n in st[\"card_text\"]) for n in ds.get(\"check_names\", [])}\n"
  "    try:\n"
  "        from importlib.metadata import version as _ver\n"
  "        hub_ver = _ver(\"huggingface_hub\")\n"
  "    except Exception:\n"
  "        hub_ver = None\n"
  "    try:\n"
  "        me = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n"
  "    except Exception:\n"
  "        me = None\n"
  "    return {\n"
  "        \"schema\": SCHEMA,\n"
  "        \"repo_id\": ds[\"repo\"],\n"
  "        \"dataset_key\": ds[\"key\"],\n"
  "        \"url\": f\"https://huggingface.co/datasets/{ds['repo']}\",\n"
  "        \"commit_sha\": st[\"sha\"],\n"
  "        \"planning_snapshot_commit_sha\": ds[\"planning_sha\"],\n"
  "        \"planning_snapshot_source\": PLANNING_SOURCE,\n"
  "        \"commit_changed_since_planning\": st[\"sha\"] != ds[\"planning_sha\"],\n"
  "        \"retrieved_utc\": ctx.now,\n"
  "        \"owner_decision\": OWNER_DECISION,\n"
  "        \"license\": {\"expected_ids\": list(ds[\"licenses\"]), \"ids_from_card\": st[\"card_ids\"], \"name\": ds[\"license_name\"], \"url\": ds[\"license_url\"],\n"
  "                    \"card_front_matter\": st[\"card_fm\"], \"card_licence_lines\": card_license_excerpt(st[\"card_text\"]),\n"
  "                    \"card_file\": f\"{folder_of(ds['repo'])}/README.md\", \"share_alike\": ds[\"share_alike\"]},\n"
  "        \"attribution\": {\"lines\": attribution_lines(ds), \"names_found_in_card\": names_found},\n"
  "        \"row_defaults\": {\"source\": ds[\"row_source\"], \"license\": ds[\"row_license\"]},\n"
  "        \"language\": ds[\"language\"],\n"
  "        \"rows_per_card\": ds[\"rows_card\"],\n"
  "        \"generator_notes\": list(ds[\"generator_notes\"]),\n"
  "        \"quality_notes\": list(ds[\"quality_notes\"]),\n"
  "        \"adoption\": dict(ds[\"adoption\"]),\n"
  "        \"files\": records,\n"
  "        \"totals\": {\"files\": len(records), \"bytes\": sum(r[\"size\"] for r in records), \"bytes_by_role\": by_role,\n"
  "                   \"train_files\": sum(1 for r in records if r[\"may_train\"])},\n"
  "        \"rules_for_converters\": [\"Read only files with may_train == true (use acquire.train_files).\",\n"
  "                                 \"Files with role dev/validation live under do_not_train_*/ and are for decontamination only.\",\n"
  "                                 \"Set each row's source and license from row_defaults; keep the attribution lines with the dataset.\"],\n"
  "        \"tool\": {\"acquire_version\": ACQUIRE_VERSION, \"acquire_py_sha256\": me, \"python\": sys.version.split()[0], \"huggingface_hub\": hub_ver},\n"
  "        \"network\": {\"allowed_hosts\": list(ALLOWED_HOSTS), \"token_used\": False, \"revision_pinned\": True},\n"
  "    }\n"
  "\n"
  "\n"
  "def _core(prov: dict):\n"
  "    return {\"repo_id\": prov[\"repo_id\"], \"files\": sorted((f[\"dest_rel\"], f[\"size\"], f[\"sha256\"], f[\"role\"]) for f in prov[\"files\"])}\n"
  "\n"
  "\n"
  "def render_markdown(prov: dict) -> str:\n"
  "    L = prov[\"license\"]\n"
  "    out = [f\"# Provenance: {prov['repo_id']}\", \"\",\n"
  "           f\"- Repo: {prov['url']}\",\n"
  "           f\"- Pinned commit: `{prov['commit_sha']}` (planning snapshot: `{prov['planning_snapshot_commit_sha']}`\"\n"
  "           + (\", CHANGED since planning\" if prov[\"commit_changed_since_planning\"] else \", unchanged\") + \")\",\n"
  "           f\"- Retrieved (UTC): {prov['retrieved_utc']}\",\n"
  "           f\"- Owner decision: {prov['owner_decision']}\",\n"
  "           f\"- Licence: {L['name']} <{L['url']}>; ids on the card: {', '.join(L['ids_from_card'])}\",\n"
  "           f\"- Language: {prov['language']}; rows per card: {prov['rows_per_card']}\",\n"
  "           f\"- {prov['adoption']['text']}\",\n"
  "           \"\", \"## Licence text from the card (lines mentioning the licence; the full card is README.md in this folder)\", \"\", \"```\"]\n"
  "    out.extend(L[\"card_licence_lines\"] or [\"(no line mentions the licence besides the front matter)\"])\n"
  "    out += [\"\", \"--- front matter ---\", L[\"card_front_matter\"], \"```\", \"\", \"## Generator and origin notes\", \"\"]\n"
  "    out.extend(f\"- {x}\" for x in prov[\"generator_notes\"])\n"
  "    out += [\"\", \"## Quality notes\", \"\"]\n"
  "    out.extend(f\"- {x}\" for x in prov[\"quality_notes\"])\n"
  "    out += [\"\", \"## Attribution (required; carry it with every converted row/dataset)\", \"\"]\n"
  "    out.extend(f\"- {x}\" for x in prov[\"attribution\"][\"lines\"])\n"
  "    out += [\"\", f\"Row defaults: source = `{prov['row_defaults']['source']}`; license = `{prov['row_defaults']['license']}`\", \"\",\n"
  "            \"## Files (paths relative to external_data_v49/)\", \"\", \"| role | may_train | path | bytes | sha256 |\", \"|---|---|---|---|---|\"]\n"
  "    for f in prov[\"files\"]:\n"
  "        out.append(f\"| {f['role']} | {str(f['may_train']).lower()} | {f['dest_rel']} | {f['size']} | {f['sha256']} |\")\n"
  "    out += [\"\", \"## Rules for converters\", \"\"]\n"
  "    out.extend(f\"- {x}\" for x in prov[\"rules_for_converters\"])\n"
  "    return \"\\n\".join(out) + \"\\n\"\n"
  "\n"
  "\n"
  "def _read_provenance_json(p) -> dict:\n"
  "    \"\"\"provenance.json as a dict, or AcquireError with the way out (Drive can lose the tail of a file that was written just before a runtime died).\"\"\"\n"
  "    try:\n"
  "        d = json.loads(Path(p).read_text(encoding=\"utf-8\"))\n"
  "    except (ValueError, OSError) as e:\n"
  "        raise AcquireError(f\"{p} cannot be read as JSON ({type(e).__name__}: {e}); Drive may have lost the end of the file.  Nothing was overwritten or \"\n"
  "                           f\"deleted.  Run step 5 (acquire) again: it moves the damaged file aside (provenance.json.stale-N, not deleted), checks every \"\n"
  "                           f\"data file again against the Hub values and writes a fresh provenance.json.\") from e\n"
  "    if not isinstance(d, dict) or not isinstance(d.get(\"files\"), list) or not isinstance(d.get(\"repo_id\"), str):\n"
  "        raise AcquireError(f\"{p} is not a provenance record (no repo_id / files).  Nothing was overwritten or deleted.  Run step 5 (acquire) again: it \"\n"
  "                           f\"moves the damaged file aside and writes a fresh one.\")\n"
  "    return d\n"
  "\n"
  "\n"
  "def _has_readable_provenance(ext, ds) -> bool:\n"
  "    \"\"\"True when the dataset has a provenance.json that reads as a provenance record.  A damaged one (Drive can lose the tail of a file written just\n"
  "    before a runtime died) counts as ABSENT where the recovery of a truncated final file asks for it: provenance.json is written last, so a dataset\n"
  "    whose record is unreadable never had a finished acquisition, and _write_provenance moves the damaged record aside (never deletes it) anyway.\"\"\"\n"
  "    p = Path(ext) / folder_of(ds[\"repo\"]) / \"provenance.json\"\n"
  "    if not os.path.lexists(p):\n"
  "        return False\n"
  "    try:\n"
  "        _read_provenance_json(p)\n"
  "    except AcquireError:\n"
  "        return False\n"
  "    return True\n"
  "\n"
  "\n"
  "def _write_provenance(ctx, st: dict, prov: dict) -> str:\n"
  "    d = ctx.ext / folder_of(st[\"ds\"][\"repo\"])\n"
  "    jp, mp = d / \"provenance.json\", d / \"PROVENANCE.md\"\n"
  "    state, old = \"written\", None\n"
  "    if os.path.lexists(jp):\n"
  "        try:\n"
  "            old = _read_provenance_json(jp)\n"
  "        except AcquireError:\n"
  "            moved = _quarantine(str(jp))                                # damaged (truncated by Drive): keep the bytes, write a fresh record below\n"
  "            ctx.log(f\"  {jp.name} of {st['ds']['repo']} was damaged: moved aside as {os.path.basename(moved)} (not deleted); writing a fresh one from the \"\n"
  "                    f\"files this run verified\")\n"
  "    if old is not None:\n"
  "        if _core(old) != _core(prov):\n"
  "            raise ProvenanceConflict(f\"{jp} records different files than this run verified; not overwritten. Compare both and use a fresh root if the data changed.\")\n"
  "        state = \"kept\"\n"
  "    else:\n"
  "        _write_new_text(jp, json.dumps(prov, ensure_ascii=False, indent=1) + \"\\n\")\n"
  "    if os.path.lexists(mp) and os.path.getsize(mp) == 0:             # an empty PROVENANCE.md is a lost write, not a record\n"
  "        ctx.log(f\"  an empty PROVENANCE.md of {st['ds']['repo']} was moved aside as {os.path.basename(_quarantine(str(mp)))} (not deleted)\")\n"
  "    if not os.path.lexists(mp):\n"
  "        _write_new_text(mp, render_markdown(old if old is not None else prov))\n"
  "        if state == \"kept\":\n"
  "            state = \"kept (PROVENANCE.md re-created)\"\n"
  "    return state\n"
  "\n"
  "\n"
  "PIN_NAME = \"commit_pin.json\"\n"
  "\n"
  "\n"
  "def _write_pin(ctx, st: dict) -> str:\n"
  "    \"\"\"Record the commit this dataset is fetched at BEFORE its first file is written (write-once).  provenance.json is written last, so a run that\n"
  "    dies half way through a dataset would otherwise leave nothing that pins the re-run to the same commit: an upstream README edit in between made\n"
  "    the re-run refuse for every dataset (and left only a 1.9 GB RAW_SUFFIX re-download).\"\"\"\n"
  "    ds = st[\"ds\"]\n"
  "    p = ctx.ext / folder_of(ds[\"repo\"]) / PIN_NAME\n"
  "    if os.path.lexists(p):\n"
  "        return \"kept\"\n"
  "    _write_new_text(p, json.dumps({\"schema\": \"v49_commit_pin/1\", \"repo_id\": ds[\"repo\"], \"commit_sha\": st[\"sha\"], \"planning_snapshot_commit_sha\": ds[\"planning_sha\"],\n"
  "                                   \"pinned_utc\": ctx.now, \"note\": \"written before the first file of this dataset: a re-run fetches this commit, not the head\"},\n"
  "                                  ensure_ascii=False, indent=1) + \"\\n\")\n"
  "    return \"written\"\n"
  "\n"
  "\n"
  "_DATA_EXT = (\".parquet\", \".jsonl\", \".ndjson\", \".arrow\", \".csv\", \".tsv\", \".feather\", \".zst\", \".gz\", \".zip\", \".tar\", \".bz2\", \".xz\")\n"
  "\n"
  "\n"
  "def _data_like(path: str) -> bool:\n"
  "    low = path.lower()\n"
  "    return low.endswith(_DATA_EXT) or (low.endswith(\".json\") and any(w in low for w in (\"train\", \"dev\", \"valid\", \"test\", \"data\")))\n"
  "\n"
  "\n"
  "def _check_unlisted(ds: dict, live: dict, log) -> None:\n"
  "    \"\"\"Every file of the live listing is either in the table, declared `not_fetched` (planning-snapshot siblings that are deliberately not fetched:\n"
  "    a path, or a prefix ending in '/'), or reported.  A data-like file that is none of these (a new train shard, a new dev file that decontamination\n"
  "    would need) stops the run before any download: the table would silently under-fetch.\"\"\"\n"
  "    listed = {f[\"path\"] for f in ds[\"files\"]}\n"
  "    declared = tuple(ds.get(\"not_fetched\", ()))\n"
  "    data, other = [], []\n"
  "    for name in sorted(live):\n"
  "        if name in listed or any(name == x or (x.endswith(\"/\") and name.startswith(x)) for x in declared):\n"
  "            continue\n"
  "        (data if _data_like(name) else other).append(name)\n"
  "    if data:\n"
  "        raise UpstreamDrift(f\"{ds['repo']}: the repo holds data file(s) that the plan does not list: {', '.join(data[:6])}\"\n"
  "                            + (f\" (+{len(data) - 6} more)\" if len(data) > 6 else \"\")\n"
  "                            + \"; stop and re-plan (add them to the table with their size and sha256, or declare them not_fetched); nothing was downloaded\")\n"
  "    if other:\n"
  "        log(f\"  note: {ds['repo']} also has file(s) that are not fetched and not data: {', '.join(other[:6])}\")\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ dry run / main entry\n"
  "def _plan(datasets, root=None, ext_dirname: str = EXTERNAL_DIRNAME) -> dict:\n"
  "    ds_out = []\n"
  "    for ds in datasets:\n"
  "        files = []\n"
  "        for f in ds[\"files\"]:\n"
  "            rel = dest_rel(ds, f)\n"
  "            present = None\n"
  "            if root is not None:\n"
  "                p = Path(root) / ext_dirname / rel\n"
  "                present = bool(os.path.isfile(p) and os.path.getsize(p) == f[\"size\"])\n"
  "            files.append({\"role\": f[\"role\"], \"path\": f[\"path\"], \"size\": f[\"size\"], \"dest_rel\": rel, \"already_present_size_ok\": present})\n"
  "        ds_out.append({\"repo\": ds[\"repo\"], \"key\": ds[\"key\"], \"planning_sha\": ds[\"planning_sha\"], \"licenses\": list(ds[\"licenses\"]),\n"
  "                       \"bytes\": sum(x[\"size\"] for x in files), \"files\": files})\n"
  "    return {\"datasets\": ds_out, \"n_files\": sum(len(d[\"files\"]) for d in ds_out), \"total_bytes\": sum(d[\"bytes\"] for d in ds_out),\n"
  "            \"card_bytes\": total_bytes(datasets, (\"card\",)), \"train_bytes\": total_bytes(datasets, (\"train\",)),\n"
  "            \"do_not_train_bytes\": total_bytes(datasets, NO_TRAIN_ROLES)}\n"
  "\n"
  "\n"
  "def _dry_run(datasets, root, log, ext_dirname: str = EXTERNAL_DIRNAME) -> dict:\n"
  "    plan = _plan(datasets, root, ext_dirname)\n"
  "    log(\"DRY RUN: no network, nothing is written.  Files that WOULD be fetched (revision = commit sha read from the Hub at run time):\")\n"
  "    for i, d in enumerate(plan[\"datasets\"], 1):\n"
  "        log(f\"{i}. {d['repo']}  [{', '.join(d['licenses'])}]  planning commit {d['planning_sha'][:12]}\")\n"
  "        for f in d[\"files\"]:\n"
  "            mark = \"\" if f[\"already_present_size_ok\"] is None else (\"  (present, size ok: would be re-verified and skipped)\" if f[\"already_present_size_ok\"] else \"\")\n"
  "            log(f\"     {f['role']:<10} {f['size']:>13,} B  {f['path']}  ->  {ext_dirname}/{f['dest_rel']}{mark}\")\n"
  "        log(f\"     subtotal {d['bytes']:,} B\")\n"
  "    tb = plan[\"total_bytes\"]\n"
  "    log(f\"TOTAL: {plan['n_files']} files, {tb:,} bytes = {tb / 1e9:.2f} GB = {tb / 2**30:.2f} GiB \"\n"
  "        f\"(train {plan['train_bytes']:,}; do-not-train dev/validation {plan['do_not_train_bytes']:,}; cards {plan['card_bytes']:,})\")\n"
  "    plan[\"dry_run\"] = True\n"
  "    return plan\n"
  "\n"
  "\n"
  "def _in_colab() -> bool:\n"
  "    return \"google.colab\" in sys.modules or \"COLAB_RELEASE_TAG\" in os.environ or os.path.isdir(\"/content/sample_data\")\n"
  "\n"
  "\n"
  "def acquire(root, only=None, *, dry_run=False, now_utc=None, datasets=None, stage_dir=None, keep_stage=False,\n"
  "            verify_after_copy=True, allow_local=False, log=print, ext_dirname: str = EXTERNAL_DIRNAME, pin_existing: bool = True,\n"
  "            after_files=None) -> dict:\n"
  "    \"\"\"Fetch (or, with dry_run, only plan) the selected datasets into <root>/external_data_v49/.  Idempotent and resumable.\n"
  "\n"
  "    root        the existing Drive folder (never created here); only NEW sub-folders of it are written.\n"
  "    only        None = all six, or keys / repo ids / folder names (list or comma string).\n"
  "    now_utc     retrieval time to record, 'YYYY-MM-DDTHH:MM:SSZ'; read from the clock inside this call when omitted (never at import).\n"
  "    datasets    override the table (tests only).\n"
  "    stage_dir   local-disk staging dir (default /content/_hf_stage_v49); must be outside root.\n"
  "    allow_local real runs refuse to start off-Colab unless this is True (tests only: the owner's PC must not be loaded).\n"
  "    ext_dirname the folder under root that receives the files (default external_data_v49; the notebook's RAW_SUFFIX makes it a fresh one).\n"
  "    pin_existing a dataset that already has a provenance.json (or the commit_pin.json that is written before its first file) is fetched at THAT\n"
  "                commit, not at the current head of the repo: an upstream edit of a README between two runs must not turn a re-run into a refusal.\n"
  "    after_files  callable(repo_id) run after all files of a dataset are in place and BEFORE its provenance.json is written (the notebook passes the\n"
  "                Drive flush: flush_and_unmount + mount), so provenance never exists for bytes Drive may not have stored yet.  It also runs on a re-run\n"
  "                for a dataset that still has no provenance.  An exception stops the run (the files stay; the re-run verifies them again).\n"
  "    \"\"\"\n"
  "    table = list(DATASETS if datasets is None else datasets)\n"
  "    validate_table(table)\n"
  "    sel = select(table, only)\n"
  "    if not sel:\n"
  "        raise ValueError(\"no dataset selected\")\n"
  "    if dry_run:\n"
  "        return _dry_run(sel, root, log, ext_dirname)\n"
  "\n"
  "    if not allow_local and not _in_colab():\n"
  "        raise AcquireError(\"refusing to download off Colab: this pipeline must not load the owner's PC (pass allow_local=True only for tests)\")\n"
  "    root = Path(root)\n"
  "    if str(root).replace(\"\\\\\", \"/\").startswith(\"/content/drive\") and not os.path.isdir(\"/content/drive/MyDrive\"):\n"
  "        raise AcquireError(\"Google Drive is not mounted: run drive.mount('/content/drive') first\")\n"
  "    if not root.is_dir():\n"
  "        raise AcquireError(f\"root folder does not exist (it is never created here): {root}\")\n"
  "    if now_utc is None:\n"
  "        now_utc = datetime.datetime.now(datetime.timezone.utc).strftime(\"%Y-%m-%dT%H:%M:%SZ\")\n"
  "    if not re.fullmatch(r\"\\d{4}-\\d\\d-\\d\\dT\\d\\d:\\d\\d:\\d\\dZ\", now_utc):\n"
  "        raise ValueError(f\"now_utc must look like 2026-11-09T12:00:00Z, got {now_utc!r}\")\n"
  "    if stage_dir is None:\n"
  "        stage_dir = DEFAULT_STAGE if _in_colab() else tempfile.mkdtemp(prefix=\"hf_stage_v49_\")\n"
  "    stage = Path(stage_dir)\n"
  "    if _is_within(stage, root) or _is_within(root, stage):\n"
  "        raise UnsafeLayout(f\"the staging dir {stage} must be outside the Drive root {root}\")\n"
  "    stage.mkdir(parents=True, exist_ok=True)\n"
  "    biggest = max(f[\"size\"] for d in sel for f in d[\"files\"])\n"
  "    need = (sum(f[\"size\"] for d in sel for f in d[\"files\"]) if keep_stage else biggest) + (256 << 20)\n"
  "    free = shutil.disk_usage(stage).free\n"
  "    if free < need:\n"
  "        raise AcquireError(f\"staging disk has {free:,} B free, need about {need:,} B\")\n"
  "    os.environ.setdefault(\"HF_HUB_DISABLE_TELEMETRY\", \"1\")\n"
  "\n"
  "    api_cls, dl = _hub()\n"
  "    ctx = _Ctx()\n"
  "    ctx.ext = root / ext_dirname\n"
  "    ctx.stage, ctx.download, ctx.log, ctx.now = stage, dl, log, now_utc\n"
  "    ctx.keep_stage, ctx.verify_after_copy = keep_stage, verify_after_copy\n"
  "    ctx.ext.mkdir(exist_ok=True)\n"
  "    api = api_cls()\n"
  "\n"
  "    # ---- phase 1: listing, drift and LICENCE for every selected dataset before any data file is downloaded\n"
  "    states = []\n"
  "    for ds in sel:\n"
  "        pinned = _pinned_commit(ctx.ext, ds) if pin_existing else None\n"
  "        if pinned:\n"
  "            log(f\"  {ds['repo']}: an earlier run recorded commit {pinned[:12]}; fetching that commit, not the current head\")\n"
  "            info = api.dataset_info(ds[\"repo\"], files_metadata=True, token=False, revision=pinned)\n"
  "        else:\n"
  "            info = api.dataset_info(ds[\"repo\"], files_metadata=True, token=False)\n"
  "        sha = _g(info, \"sha\")\n"
  "        if not isinstance(sha, str) or not re.fullmatch(r\"[0-9a-f]{40}\", sha):\n"
  "            raise VerificationError(f\"{ds['repo']}: the Hub did not return a commit sha\")\n"
  "        if pinned and sha != pinned:\n"
  "            raise VerificationError(f\"{ds['repo']}: asked for the recorded commit {pinned} and the Hub answered {sha}\")\n"
  "        live = _live_listing(info)\n"
  "        _check_unlisted(ds, live, log)\n"
  "        st = {\"ds\": ds, \"sha\": sha, \"exp\": {}, \"drift\": {}}\n"
  "        for f in ds[\"files\"]:\n"
  "            st[\"exp\"][f[\"path\"]], st[\"drift\"][f[\"path\"]] = _expected_for(ds, f, live)\n"
  "        card = next(f for f in ds[\"files\"] if f[\"role\"] == \"card\")\n"
  "        card_final = ctx.ext / dest_rel(ds, card)\n"
  "        if os.path.lexists(card_final):\n"
  "            try:\n"
  "                _verify_existing(card_final, st[\"exp\"][card[\"path\"]], dest_rel(ds, card))\n"
  "                raw = card_final.read_bytes()\n"
  "            except ExistingFileMismatch:\n"
  "                # a card truncated by a run that died before Drive had all its bytes (the dataset has no provenance yet): the licence check reads the\n"
  "                # verified staged card; phase 2 moves the short file aside (see _ensure_file)\n"
  "                if not (os.path.isfile(card_final) and os.path.getsize(card_final) < st[\"exp\"][card[\"path\"]][\"size\"]\n"
  "                        and not _has_readable_provenance(ctx.ext, ds)):\n"
  "                    raise\n"
  "                staged, _ = _stage(ctx, ds, card, sha, st[\"exp\"][card[\"path\"]])\n"
  "                if not _is_prefix_of(card_final, staged):\n"
  "                    raise\n"
  "                raw = Path(staged).read_bytes()\n"
  "        else:\n"
  "            staged, _ = _stage(ctx, ds, card, sha, st[\"exp\"][card[\"path\"]])\n"
  "            raw = Path(staged).read_bytes()\n"
  "        st[\"card_text\"] = raw.decode(\"utf-8\", errors=\"replace\")\n"
  "        st[\"card_ids\"], st[\"card_fm\"] = parse_card_licenses(st[\"card_text\"])\n"
  "        if set(st[\"card_ids\"]) != set(ds[\"licenses\"]):\n"
  "            raise LicenseMismatch(f\"{ds['repo']}: the card says licence {st['card_ids'] or 'NONE'}, expected {list(ds['licenses'])}; nothing was downloaded\")\n"
  "        log(f\"licence ok: {ds['repo']} -> {', '.join(st['card_ids'])}  (commit {sha[:12]})\")\n"
  "        states.append(st)\n"
  "\n"
  "    # ---- phase 2: the files\n"
  "    result = {\"dry_run\": False, \"root\": str(root), \"external_dir\": str(ctx.ext), \"retrieved_utc\": now_utc, \"datasets\": {}}\n"
  "    for i, st in enumerate(states, 1):\n"
  "        ds = st[\"ds\"]\n"
  "        log(f\"[{i}/{len(states)}] {ds['repo']}\")\n"
  "        _write_pin(ctx, st)\n"
  "        records = [_ensure_file(ctx, st, f) for f in ds[\"files\"]]\n"
  "        if after_files is not None and (any(r[\"status\"] == \"downloaded\" for r in records)\n"
  "                                        or not _has_readable_provenance(ctx.ext, ds)):\n"
  "            after_files(ds[\"repo\"])\n"
  "        prov = build_provenance(ctx, st, records)\n"
  "        state = _write_provenance(ctx, st, prov)\n"
  "        for role in NO_TRAIN_ROLES:\n"
  "            if any(r[\"role\"] == role for r in records):\n"
  "                marker = ctx.ext / ROLE_DIR[role] / \"DO_NOT_TRAIN.txt\"\n"
  "                if not os.path.lexists(marker):\n"
  "                    _write_new_text(marker, f\"Files under {ROLE_DIR[role]}/ are {role} data kept ONLY for decontamination. \"\n"
  "                                            \"Never train on them and never pass them to a converter.\\n\")\n"
  "        result[\"datasets\"][ds[\"repo\"]] = {\"commit_sha\": st[\"sha\"], \"provenance\": state, \"adoption\": ds[\"adoption\"][\"text\"],\n"
  "                                          \"files\": [{\"dest_rel\": r[\"dest_rel\"], \"role\": r[\"role\"], \"status\": r[\"status\"], \"size\": r[\"size\"], \"sha256\": r[\"sha256\"]} for r in records]}\n"
  "    result[\"total_bytes\"] = sum(f[\"size\"] for d in result[\"datasets\"].values() for f in d[\"files\"])\n"
  "    result[\"downloaded_bytes\"] = sum(f[\"size\"] for d in result[\"datasets\"].values() for f in d[\"files\"] if f[\"status\"] == \"downloaded\")\n"
  "    log(f\"done: {len(states)} datasets, {result['total_bytes']:,} bytes verified ({result['downloaded_bytes']:,} downloaded this run)\")\n"
  "    return result\n"
  "\n"
  "\n"
  "# ------------------------------------------------------------------------------------------------ for the converters\n"
  "def _pinned_commit(ext, ds) -> Optional[str]:\n"
  "    \"\"\"The commit sha recorded by an earlier run of this dataset: provenance.json, else the commit_pin.json that is written before the first file\n"
  "    (None when neither is valid).\"\"\"\n"
  "    for name in (\"provenance.json\", PIN_NAME):\n"
  "        p = Path(ext) / folder_of(ds[\"repo\"]) / name\n"
  "        try:\n"
  "            sha = json.loads(p.read_text(encoding=\"utf-8\")).get(\"commit_sha\")\n"
  "        except (OSError, ValueError, AttributeError):\n"
  "            continue\n"
  "        if isinstance(sha, str) and re.fullmatch(r\"[0-9a-f]{40}\", sha):\n"
  "            return sha\n"
  "    return None\n"
  "\n"
  "\n"
  "def _provenance_of(root, ds, ext_dirname: str = EXTERNAL_DIRNAME) -> dict:\n"
  "    p = Path(root) / ext_dirname / folder_of(ds[\"repo\"]) / \"provenance.json\"\n"
  "    if not os.path.isfile(p):\n"
  "        raise AcquireError(f\"{ds['repo']} has not been acquired: {p} is missing\")\n"
  "    return _read_provenance_json(p)\n"
  "\n"
  "\n"
  "def train_files(root, which, datasets=None, ext_dirname: str = EXTERNAL_DIRNAME, include_held: bool = False) -> list:\n"
  "    \"\"\"Paths of the TRAIN-eligible data files of one dataset (key, repo id or folder name).  dev/validation can never come out of here.\n"
  "    A dataset whose adoption status in provenance.json is 'hold' (ToolACE, WizardLM: generator unverified / low priority) is REFUSED (HeldDataset)\n"
  "    unless include_held=True: a held dataset is only converted so that the checks can be read, never as ordinary training input.\"\"\"\n"
  "    table = list(DATASETS if datasets is None else datasets)\n"
  "    ds = select(table, [which])[0]\n"
  "    prov = _provenance_of(root, ds, ext_dirname)\n"
  "    status = (prov.get(\"adoption\") or {}).get(\"status\") if isinstance(prov.get(\"adoption\"), dict) else None\n"
  "    if status == \"hold\" and not include_held:\n"
  "        raise HeldDataset(f\"{ds['repo']} is on hold ({(prov.get('adoption') or {}).get('text', 'adoption: hold')}); its train files are not handed out \"\n"
  "                          f\"unless the caller says include_held=True (a held dataset is converted only so that the checks can be read)\")\n"
  "    out = []\n"
  "    for f in prov[\"files\"]:\n"
  "        if f[\"role\"] == \"train\" and f[\"may_train\"]:\n"
  "            if any(part.startswith(\"do_not_train\") for part in PurePosixPath(f[\"dest_rel\"]).parts):\n"
  "                raise UnsafeLayout(f\"{f['dest_rel']} is under a do_not_train folder but marked train\")\n"
  "            out.append(Path(root) / ext_dirname / f[\"dest_rel\"])\n"
  "    return out\n"
  "\n"
  "\n"
  "def do_not_train_files(root, datasets=None, ext_dirname: str = EXTERNAL_DIRNAME) -> dict:\n"
  "    \"\"\"{repo: [paths]} of the dev/validation files, for decontamination only.\"\"\"\n"
  "    table = list(DATASETS if datasets is None else datasets)\n"
  "    out = {}\n"
  "    for ds in table:\n"
  "        p = Path(root) / ext_dirname / folder_of(ds[\"repo\"]) / \"provenance.json\"\n"
  "        if not os.path.isfile(p):\n"
  "            continue\n"
  "        prov = _read_provenance_json(p)\n"
  "        files = [Path(root) / ext_dirname / f[\"dest_rel\"] for f in prov[\"files\"] if f[\"role\"] in NO_TRAIN_ROLES]\n"
  "        if files:\n"
  "            out[ds[\"repo\"]] = files\n"
  "    return out\n"
  "\n"
  "\n"
  "def load_manifest(root, datasets=None, ext_dirname: str = EXTERNAL_DIRNAME) -> dict:\n"
  "    \"\"\"{repo: provenance.json content} for every acquired dataset.\"\"\"\n"
  "    table = list(DATASETS if datasets is None else datasets)\n"
  "    out = {}\n"
  "    for ds in table:\n"
  "        p = Path(root) / ext_dirname / folder_of(ds[\"repo\"]) / \"provenance.json\"\n"
  "        if os.path.isfile(p):\n"
  "            out[ds[\"repo\"]] = _read_provenance_json(p)\n"
  "    return out\n"
  "\n"
  "\n"
  "def main(argv=None) -> int:\n"
  "    ap = argparse.ArgumentParser(description=\"Fetch the six approved Hugging Face datasets into Drive (run on Colab).\")\n"
  "    ap.add_argument(\"--root\", default=DEFAULT_ROOT)\n"
  "    ap.add_argument(\"--only\", default=None, help=\"comma separated keys / repo ids\")\n"
  "    ap.add_argument(\"--dry-run\", action=\"store_true\")\n"
  "    ap.add_argument(\"--stage-dir\", default=None)\n"
  "    ap.add_argument(\"--keep-stage\", action=\"store_true\")\n"
  "    ap.add_argument(\"--no-verify-after-copy\", action=\"store_true\")\n"
  "    ap.add_argument(\"--allow-local\", action=\"store_true\", help=\"tests only: the owner's PC must not be loaded\")\n"
  "    ap.add_argument(\"--now-utc\", default=None)\n"
  "    a = ap.parse_args(argv)\n"
  "    res = acquire(a.root, a.only, dry_run=a.dry_run, now_utc=a.now_utc, stage_dir=a.stage_dir, keep_stage=a.keep_stage,\n"
  "                  verify_after_copy=not a.no_verify_after_copy, allow_local=a.allow_local)\n"
  "    if not a.dry_run:\n"
  "        brief = {r: {\"commit\": d[\"commit_sha\"], \"provenance\": d[\"provenance\"], \"files\": {x[\"status\"]: sum(1 for y in d[\"files\"] if y[\"status\"] == x[\"status\"]) for x in d[\"files\"]}}\n"
  "                 for r, d in res[\"datasets\"].items()}\n"
  "        print(json.dumps({\"retrieved_utc\": res[\"retrieved_utc\"], \"total_bytes\": res[\"total_bytes\"], \"downloaded_bytes\": res[\"downloaded_bytes\"], \"datasets\": brief}, indent=1))\n"
  "    return 0\n"
  "\n"
  "\n"
  "if __name__ == \"__main__\":\n"
  "    sys.exit(main())\n"
 ),
 "checks.py": (
  "#!/usr/bin/env python3\n"
  "\"\"\"checks.py - streaming quality / contamination / duplicate checks for the GeoCPU v49 external-data pipeline.\n"
  "\n"
  "Runs on a Google Colab CPU runtime (12 GB RAM).  Standard library only (+ optional numpy, pyarrow when the Colab image has\n"
  "them).  Nothing here needs a secret, a network connection or a GPU.  Every function that reads rows takes any iterable of\n"
  "dict rows (use iter_rows(path)); nothing ever loads a whole dataset; results are deterministic (fixed seeds, no use of the\n"
  "per-process hash() for anything that is persisted or reported).\n"
  "\n"
  "NOTEBOOK COOKBOOK (names are exactly the ones defined below)\n"
  "    from checks import *\n"
  "    V   = load_vocab(DRIVE_ROOT, out_dir=OUT)                        # 3,898-entry model vocabulary (3,893 single characters)\n"
  "    rows = lambda: iter_rows(path)                                   # jsonl / parquet / SQuAD-like json, always streaming\n"
  "    oov  = oov_stats(rows(), V)                                      # per-row OOV share, share of rows > 1 %, top OOV characters\n"
  "    ln   = length_stats(rows(), group_key=lambda r: r.get(\"source\")) # len(prompt+target)+1, share > 2048 and > 1900\n"
  "    ex   = exact_dups(rows())                                        # exact duplicates inside one dataset\n"
  "    nd   = near_dups_within(rows(), threshold=0.75)                  # MinHash (k=128, char 5-grams, LSH 32x4) near duplicates; see MinHasher for the cut\n"
  "    cr   = cross_near_dups(rows(), iter_rows(OLD_CORPUS), sample_every=1)   # new rows vs the 423,348-row old corpus\n"
  "    mx   = cross_dataset_near_dups({\"jaquad\": rows_a(), \"hotpot\": rows_b()})  # dataset x dataset duplicate matrix\n"
  "    EF   = find_eval_files(DRIVE_ROOT)                               # eval manifests (bundle zip on Drive) -> EvalFiles\n"
  "    EI   = build_eval_index(EF.sources)                              # digit-stripped 13-gram shingles of all eval texts\n"
  "    dc   = decontaminate(rows(), EI)                                 # flagged_positions, boilerplate report, <=5 examples\n"
  "    sm   = semantic_neighbors(eval_questions, rows())                # TF-IDF look-alikes for human reading (NOT exhaustive)\n"
  "    ok, stdout, err = code_sandbox_check(code)                       # GeoCPU-executor-compatible re-run of python blocks\n"
  "    sample_for_reading(rows(), n=100, seed=7, out_path=\".../read_100_<dataset>.txt\", vocab=V)   # the by-eye 100 rows\n"
  "Rows that fail a check are not removed by these functions; they return positions / flags and the caller filters (a second\n"
  "streaming pass with drop_flagged / oov_filter).  Outputs written to Google Drive go through write_new_file(): it only ever\n"
  "creates files and refuses (FileExistsError) when a file exists with a different sha256.\n"
  "\n"
  "MEMORY / TIME (measured on the owner's Windows PC with synthetic 2,000-character rows; a Colab Linux CPU is similar or faster)\n"
  "    MinHash signature (k=128, 5-grams): ~2.8 ms/row with numpy (k=64 was 1.9), ~38 ms/row pure Python -> 300k rows: ~14 min / ~3 h\n"
  "        (numpy is on every Colab image; MinHasher(use_numpy=False) gives the identical signatures, only slower)\n"
  "    MinHashIndex: ~1.2 KB/row measured (512 B signature + 256 B band tables + ~290 B snippet + list slack) -> 300k rows ~ 360 MB;\n"
  "        build() holds one transient copy of the signatures (154 MB at 300k rows).  The old corpus is streamed, never held;\n"
  "        streaming all 423,348 old rows costs ~20-25 min (cross_near_dups(sample_every=N) divides that by N).\n"
  "    EvalIndex: one python int per distinct eval 13-gram: 146k shingles = 12 MB RSS for the small manifests, so the 9 MB of\n"
  "        all eval texts (incl. benchmark_8100_manifest.jsonl, 4.9 MB) is expected at a few hundred MB, not more.\n"
  "    ExactIndex: ~100 B per row.  Sandbox: ~0.11 s per snippet wall time with 2 workers on Windows (process start), less on Linux.\n"
  "    Everything else is O(1) memory per row.\n"
  "READ THE RESULTS WITH THESE REAL-DATA LESSONS (see real_smoke_output.json, 2,000 rows of the JGLUE family against the real vocabulary\n"
  "and the small eval manifests):  the 1 % OOV rule is per ROW, so one rare kanji drops an 80-character row (57 of 2,000 rows were over\n"
  "1 % while only 0.09 % of all characters were OOV); rows that share one passage (JSQuAD / JaQuAD / HotpotQA: several questions per\n"
  "context) are near duplicates of each other by character 5-grams (J 0.83-0.94) - pass text_fn=<question + answer only> to\n"
  "near_dups_within when the passage is not what should be deduplicated; and a shared 13-gram is a lead, not a verdict\n"
  "(a law title appears in both a Wikipedia row and a JamC-QA item): read decontaminate()['examples'].\n"
  "\"\"\"\n"
  "from __future__ import annotations\n"
  "\n"
  "import array\n"
  "import ast\n"
  "import bisect\n"
  "import collections\n"
  "import dataclasses\n"
  "import errno\n"
  "import glob as _glob\n"
  "import gzip\n"
  "import hashlib\n"
  "import heapq\n"
  "import io\n"
  "import itertools\n"
  "import json\n"
  "import math\n"
  "import operator\n"
  "import os\n"
  "import re\n"
  "import secrets\n"
  "import signal\n"
  "import subprocess\n"
  "import sys\n"
  "import tempfile\n"
  "import threading\n"
  "import time\n"
  "import unicodedata\n"
  "import zipfile\n"
  "import zlib\n"
  "from concurrent.futures import ThreadPoolExecutor\n"
  "from typing import Any, Callable, Dict, Iterable, Iterator, List, Mapping, Optional, Sequence, Tuple, Union\n"
  "\n"
  "if array.array(\"I\").itemsize != 4:                       # pragma: no cover\n"
  "    raise ImportError(\"checks.py needs a 4-byte array('I')\")\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Constants: where things live on Google Drive (found with the Drive connector on 2026-11-09)\n"
  "# =====================================================================================================================\n"
  "DRIVE_ROOT = \"/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106\"        # Drive folder id 1f4ny9Sr26rOnU8mOoEyrKxbQw60iH2Aj\n"
  "DRIVE_ROOT_FOLDER_ID = \"1f4ny9Sr26rOnU8mOoEyrKxbQw60iH2Aj\"\n"
  "\n"
  "# The existing corpus (inherited untouched).  Drive file id 1hBbEQ8eovdZZXNGj7PLHN3O4co7zGIBR, 614,482,850 bytes, 423,348 rows.\n"
  "OLD_CORPUS_REL = \"_v48_dataset_cache/train_40500_exact.jsonl\"\n"
  "OLD_CORPUS = DRIVE_ROOT + \"/\" + OLD_CORPUS_REL\n"
  "OLD_CORPUS_ROWS = 423_348\n"
  "\n"
  "# The model vocabulary is NOT a loose file in the run folders.  It is key \"vocab\" (token -> id, 3,898 entries = 3,893 single\n"
  "# characters + <pad> <unk> <bos> <eos> <sep>, ids 0..3897) of base_metadata.json inside the nested zip\n"
  "#   GeoCPU_v47_fullmodel_bundle_v1.zip (Drive id 1c87pqt4UoVV3I6ECv__Iip9Ob1KotkP0, 26,925,754 B; the _READY copy 1EYF4CmXLP-...\n"
  "#   has the same size)  ->  assets/r6_bundle.zip  ->  base_metadata.json\n"
  "# (the metadata also says parameters = 247,559,448 and model_protocol = character_next_token).\n"
  "# Two stand-alone vocab.json / omni_domain_vocab_v4.json files (68,705 B) exist elsewhere on Drive (folders of 2026-10-17/21,\n"
  "# not under GeoCPU_v47_fullmodel_20261106); they were NOT opened and are not used.\n"
  "VOCAB_BUNDLE_ZIPS = (\"GeoCPU_v47_fullmodel_bundle_v1.zip\", \"GeoCPU_v47_fullmodel_bundle_v1_READY.zip\")\n"
  "VOCAB_NESTED_ZIP = \"assets/r6_bundle.zip\"\n"
  "VOCAB_MEMBER = \"base_metadata.json\"\n"
  "VOCAB_DRIVE_PATH = DRIVE_ROOT + \"/\" + VOCAB_BUNDLE_ZIPS[0] + \"::\" + VOCAB_NESTED_ZIP + \"::\" + VOCAB_MEMBER + \"[vocab]\"\n"
  "EXPECTED_VOCAB_SIZE = 3898\n"
  "EXPECTED_SINGLE_CHARS = 3893\n"
  "VOCAB_SPECIALS = (\"<pad>\", \"<unk>\", \"<bos>\", \"<eos>\", \"<sep>\")\n"
  "\n"
  "# Eval manifests (Drive: inside GeoCPU_v47_fullmodel_bundle_v1.zip -> assets/r6_bundle.zip -> data/<name>; code/ in the outer zip).\n"
  "EVAL_BUNDLE_ZIPS = VOCAB_BUNDLE_ZIPS\n"
  "EVAL_NESTED_ZIP = VOCAB_NESTED_ZIP\n"
  "EVAL_REQUIRED_JSONL = {\n"
  "    \"benchmark_8100_manifest.jsonl\": \"JMMLU / JamC-QA 8,100-item benchmark manifest\",\n"
  "    \"jamcqa_manifest_test.jsonl\": \"JamC-QA test manifest\",\n"
  "    \"eval_swe_step_control_heldout.jsonl\": \"SWE step-control heldout\",\n"
  "    \"eval_swe_step_control_heldout_v43labels.jsonl\": \"SWE step-control heldout (v43 labels)\",\n"
  "    \"eval_v33_state_tracking_heldout.jsonl\": \"state tracking heldout (v33)\",\n"
  "    \"eval_v38_decomposed_state_bench200.jsonl\": \"decomposed state bench (v38)\",\n"
  "    \"eval_v41_logical_instruction_operators_heldout.jsonl\": \"T5 logical instruction operators (v41)\",\n"
  "    \"eval_v42_search_retrieval_aligned.jsonl\": \"T6 search retrieval, aligned prompts (v42)\",\n"
  "    \"eval_v42_search_retrieval_heldout.jsonl\": \"T6 search retrieval, heldout prompts (v42)\",\n"
  "}\n"
  "EVAL_REQUIRED_PY = {                                     # python sources whose string literals are eval prompts\n"
  "    \"tier8_readonly_v44fix.py\": \"T8 prompts (4 hard-coded CASES; outer bundle zip, code/)\",\n"
  "}\n"
  "# v48 unseen_eval / dev sets: 4 families x 2 files x 128 rows, selfauthored templates.  They are NOT stored as stand-alone\n"
  "# files on Drive (only the evaluation event folders step40500_v48sets/ and step61500/ are); local copy:\n"
  "# research/v48_balanced_augmentation_20261108/data/.  Optional: found when somebody copies them under the Drive root.\n"
  "EVAL_OPTIONAL_V48 = tuple(f\"{fam}_{kind}.jsonl\" for fam in (\"evidence\", \"japanese\", \"logic\", \"python\") for kind in (\"unseen_eval\", \"dev\"))\n"
  "EVAL_OPTIONAL_SUITE_ZIPS = (\"GeoCPU_v47_train_eval_suite_v2.zip\", \"GeoCPU_v47_continue30500_train_eval_v1.zip\",\n"
  "                            \"GeoCPU_v47_python_tir_eval_25500_28000_v1.zip\")      # small zips at the Drive root (30-218 KB)\n"
  "# The other frozen eval suites of the owner's registry (research/v47_practical_20261107/frozen/registry.json, 'leakage_sources', the\n"
  "# files that are not T1-T8 manifests).  Each is searched as a stand-alone file, in the bundle's data/ folder and (by file name) inside\n"
  "# the suite zips; one that is not found is reported as NOT covered (find_eval_files(...).missing_optional / the report), never skipped\n"
  "# silently.  python120 = selection_120.json; engineering64 = engineering_heldout.jsonl; visual10 = visual_heldout.jsonl;\n"
  "# new_public:* = the *_dev.jsonl files.\n"
  "EVAL_REGISTRY_EXTRA = {\n"
  "    \"dev_overlay.jsonl\": \"public_dev80: public search development overlay\",\n"
  "    \"dev_numina_tir_strict_numeric.jsonl\": \"python120 / math_python: NuminaMath-TIR strict numeric dev\",\n"
  "    \"dev_tldr_terminal.jsonl\": \"new_public:terminal dev\",\n"
  "    \"japanese_math_python_dev.jsonl\": \"new_public:japanese_math_python dev\",\n"
  "    \"plos_science_dev.jsonl\": \"new_public:plos_science dev\",\n"
  "    \"python_docs_ja_core_dev.jsonl\": \"new_public:python_ja dev\",\n"
  "    \"python_reference_dev.jsonl\": \"new_public:python_reference dev\",\n"
  "    \"science_code_complete_dev.jsonl\": \"new_public:science_code dev\",\n"
  "    \"science_engineering_verified_dev.jsonl\": \"new_public:science_engineering_verified dev\",\n"
  "    \"search_post_result_dev.jsonl\": \"new_public:search_next dev\",\n"
  "    \"engineering_heldout.jsonl\": \"engineering64 held-out\",\n"
  "    \"visual_heldout.jsonl\": \"visual10 held-out\",\n"
  "    \"dev_after_final_result_198.jsonl\": \"search_final198\",\n"
  "    \"selection_120.json\": \"python120: 80 Japanese + 40 NuminaMath-TIR selection\",\n"
  "}\n"
  "# which file stands for which tier of the owner's T1-T8 (registry.json 'suites'); T4 and T7 share one source file\n"
  "EVAL_TIER_FILES = {\n"
  "    \"T1 held-out SWE step control\": (\"eval_swe_step_control_heldout.jsonl\", \"eval_swe_step_control_heldout_v43labels.jsonl\"),\n"
  "    \"T2 decomposed state tracking\": (\"eval_v33_state_tracking_heldout.jsonl\", \"eval_v38_decomposed_state_bench200.jsonl\"),\n"
  "    \"T3 JamC-QA manifest\": (\"jamcqa_manifest_test.jsonl\",),\n"
  "    \"T4 broad 8100\": (\"benchmark_8100_manifest.jsonl\",),\n"
  "    \"T5 logical instruction operators\": (\"eval_v41_logical_instruction_operators_heldout.jsonl\",),\n"
  "    \"T6 search retrieval\": (\"eval_v42_search_retrieval_aligned.jsonl\", \"eval_v42_search_retrieval_heldout.jsonl\"),\n"
  "    \"T7 closed/open-book A/B (same source file as T4)\": (\"benchmark_8100_manifest.jsonl\",),\n"
  "    \"T8 code/explanation form probe\": (\"tier8_readonly_v44fix.py\",),\n"
  "}\n"
  "EVAL_TEXT_KEYS = (\"prompt\", \"question\", \"choices\", \"options\", \"target\", \"answer\", \"full_text\", \"input\", \"instruction\", \"text\",\n"
  "                  \"context\", \"expected_query\", \"final_answer\", \"python_code\")\n"
  "MAX_LEN = 2048                                           # context length in characters; a row needs len(prompt+target)+1 <= MAX_LEN\n"
  "\n"
  "ProgressCB = Callable[[str, int], None]\n"
  "_SKIP_DIRS = {\"checkpoints\", \"raw\", \"artifacts\", \"__pycache__\"}\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Small utilities: progress, safe writes, hashing\n"
  "# =====================================================================================================================\n"
  "class _Ticker:\n"
  "    \"\"\"Calls cb(label, n_done) about every `every` items.  cb may be None.\"\"\"\n"
  "    __slots__ = (\"cb\", \"label\", \"every\", \"n\", \"_next\")\n"
  "\n"
  "    def __init__(self, cb: Optional[ProgressCB], label: str, every: int = 20000):\n"
  "        self.cb, self.label, self.every, self.n = cb, label, max(1, int(every)), 0\n"
  "        self._next = self.every\n"
  "\n"
  "    def tick(self, k: int = 1) -> None:\n"
  "        self.n += k\n"
  "        if self.cb is not None and self.n >= self._next:\n"
  "            self.cb(self.label, self.n)\n"
  "            while self._next <= self.n:\n"
  "                self._next += self.every\n"
  "\n"
  "    def done(self) -> None:\n"
  "        if self.cb is not None:\n"
  "            self.cb(self.label + \" [done]\", self.n)\n"
  "\n"
  "\n"
  "def make_printer(min_interval_s: float = 10.0, stream=None) -> ProgressCB:\n"
  "    \"\"\"A ready-made progress callback: prints 'elapsed label: n (rate/s)' at most every min_interval_s seconds.\"\"\"\n"
  "    t0, last = time.time(), [0.0]\n"
  "\n"
  "    def cb(label: str, done: int) -> None:\n"
  "        now = time.time()\n"
  "        if label.endswith(\"[done]\") or now - last[0] >= min_interval_s:\n"
  "            last[0] = now\n"
  "            el = now - t0\n"
  "            print(f\"[{el:8.1f}s] {label}: {done:,}\" + (f\" ({done / el:,.0f}/s)\" if el > 0 else \"\"), file=stream or sys.stderr, flush=True)\n"
  "    return cb\n"
  "\n"
  "\n"
  "def sha256_file(path: str, chunk: int = 1 << 20) -> str:\n"
  "    h = hashlib.sha256()\n"
  "    with open(path, \"rb\") as f:\n"
  "        while True:\n"
  "            b = f.read(chunk)\n"
  "            if not b:\n"
  "                return h.hexdigest()\n"
  "            h.update(b)\n"
  "\n"
  "\n"
  "_FSYNC_UNSUPPORTED = {getattr(errno, n) for n in (\"EINVAL\", \"ENOTSUP\", \"EOPNOTSUPP\", \"ENOSYS\") if hasattr(errno, n)}\n"
  "\n"
  "\n"
  "def fsync_file(fd: int) -> None:\n"
  "    \"\"\"os.fsync that ignores ONLY 'this file system cannot fsync' (EINVAL / ENOTSUP / EOPNOTSUPP / ENOSYS).  Every other error (EIO, ENOSPC,\n"
  "    EDQUOT, ...) is raised: on a FUSE mount such as Google Drive the quota-full or I/O error that buffered writes defer shows up here, and\n"
  "    swallowing it would record a failed write as a successful one.\"\"\"\n"
  "    try:\n"
  "        os.fsync(fd)\n"
  "    except OSError as e:\n"
  "        if e.errno not in _FSYNC_UNSUPPORTED:\n"
  "            raise\n"
  "\n"
  "\n"
  "def quarantine_part(path: str) -> str:\n"
  "    \"\"\"Move a leftover `<x>.part` of a dead run aside as `<x>.part.stale-N` (never deleted: the owner's rule); returns the new name.\"\"\"\n"
  "    n = 1\n"
  "    while os.path.lexists(f\"{path}.stale-{n}\"):\n"
  "        n += 1\n"
  "    dst = f\"{path}.stale-{n}\"\n"
  "    os.rename(path, dst)\n"
  "    return dst\n"
  "\n"
  "\n"
  "def write_new_file(path: str, data: Union[str, bytes], *, encoding: str = \"utf-8\") -> str:\n"
  "    \"\"\"Create `path` with `data`.  Never overwrites: if the file exists with the same sha256 nothing happens ('identical'); with a\n"
  "    different sha256 FileExistsError is raised.  The bytes go to `<path>.part` first (flushed and fsynced) and are renamed into place\n"
  "    only when `path` still does not exist, so a runtime that dies in the middle of a write leaves a `.part`, never a truncated final\n"
  "    file (a leftover `.part` of an earlier run is moved aside, not deleted).  Text is written UTF-8 with lone surrogates as backslash\n"
  "    escapes (they occur in raw web text and would otherwise crash the write).  Returns 'created' or 'identical'.\"\"\"\n"
  "    if isinstance(data, str):\n"
  "        data = data.encode(encoding, \"backslashreplace\")\n"
  "    path = os.fspath(path)\n"
  "    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)\n"
  "\n"
  "    def compare() -> str:\n"
  "        old = sha256_file(path)\n"
  "        new = hashlib.sha256(data).hexdigest()\n"
  "        if old == new:\n"
  "            return \"identical\"\n"
  "        raise FileExistsError(f\"refusing to overwrite {path}: existing sha256 {old[:16]}... != new {new[:16]}...\") from None\n"
  "    if os.path.lexists(path):\n"
  "        return compare()\n"
  "    part = path + \".part\"\n"
  "    if os.path.lexists(part):\n"
  "        quarantine_part(part)\n"
  "    with open(part, \"xb\") as f:\n"
  "        f.write(data)\n"
  "        f.flush()\n"
  "        fsync_file(f.fileno())\n"
  "    if os.path.lexists(path):                                    # somebody created it while we wrote: never replace it\n"
  "        return compare()\n"
  "    os.rename(part, path)\n"
  "    return \"created\"\n"
  "\n"
  "\n"
  "def digest64(text: str) -> int:\n"
  "    \"\"\"Deterministic 64-bit digest of a string (blake2b); used for persisted / cross-process keys.\"\"\"\n"
  "    return int.from_bytes(hashlib.blake2b(text.encode(\"utf-8\", \"surrogatepass\"), digest_size=8).digest(), \"big\")\n"
  "\n"
  "\n"
  "def _string_leaves(obj: Any, depth: int = 0) -> Iterator[str]:\n"
  "    if isinstance(obj, str):\n"
  "        yield obj\n"
  "    elif depth < 5 and isinstance(obj, dict):\n"
  "        for v in obj.values():\n"
  "            yield from _string_leaves(v, depth + 1)\n"
  "    elif depth < 5 and isinstance(obj, (list, tuple)):\n"
  "        for v in obj:\n"
  "            yield from _string_leaves(v, depth + 1)\n"
  "\n"
  "\n"
  "def default_text(row: Any) -> str:\n"
  "    \"\"\"The text of a row: full_text, else prompt + target, else chat messages, else every string value joined by newlines.\"\"\"\n"
  "    if isinstance(row, str):\n"
  "        return row\n"
  "    ft = row.get(\"full_text\")\n"
  "    if isinstance(ft, str):\n"
  "        return ft\n"
  "    p, t = row.get(\"prompt\"), row.get(\"target\")\n"
  "    if isinstance(p, str) and isinstance(t, str):\n"
  "        return p + t\n"
  "    for key in (\"messages\", \"conversations\"):\n"
  "        msgs = row.get(key)\n"
  "        if isinstance(msgs, list) and msgs:\n"
  "            return \"\\n\".join(str(m.get(\"content\", m.get(\"value\", \"\"))) if isinstance(m, dict) else str(m) for m in msgs)\n"
  "    return \"\\n\".join(_string_leaves(row))\n"
  "\n"
  "\n"
  "def default_id(row: Any, pos: int = -1) -> str:\n"
  "    if isinstance(row, dict):\n"
  "        for k in (\"id\", \"_id\", \"uid\", \"q_id\", \"question_id\", \"sentence_pair_id\"):\n"
  "            if row.get(k) is not None:\n"
  "                return str(row[k])\n"
  "    return f\"#{pos}\"\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# iter_rows: jsonl / parquet / SQuAD-like json, always streaming\n"
  "# =====================================================================================================================\n"
  "_WS_COMMA = re.compile(r\"[\\s,]*\")\n"
  "_JSON_TOP_ARRAY = re.compile(r\"\\A\\s*\\[\")\n"
  "_JSON_DATA_KEY = re.compile(r'\\A\\s*\\{(?:\\s*\"[^\"\\\\]*\"\\s*:\\s*(?:\"(?:[^\"\\\\]|\\\\.)*\"|-?\\d[\\d.eE+-]*|true|false|null)\\s*,)*\\s*\"data\"\\s*:\\s*\\[')\n"
  "\n"
  "\n"
  "def _flatten_squad_article(art: dict) -> Iterator[dict]:\n"
  "    \"\"\"SQuAD / JaQuAD / JSQuAD article -> one flat dict per question.\"\"\"\n"
  "    title = art.get(\"title\", \"\")\n"
  "    for pi, para in enumerate(art.get(\"paragraphs\", [])):\n"
  "        ctx = para.get(\"context\", \"\")\n"
  "        for qa in para.get(\"qas\", []):\n"
  "            answers = qa.get(\"answers\") or []\n"
  "            row = {k: v for k, v in qa.items() if k not in (\"id\", \"question\", \"answers\", \"is_impossible\")}\n"
  "            row.update({\"id\": qa.get(\"id\"), \"title\": title, \"para_index\": pi, \"context\": ctx, \"question\": qa.get(\"question\", \"\"),\n"
  "                        \"answers\": answers, \"answer_texts\": [a.get(\"text\") for a in answers if isinstance(a, dict)],\n"
  "                        \"answer_starts\": [a.get(\"answer_start\") for a in answers if isinstance(a, dict)],\n"
  "                        \"is_impossible\": bool(qa.get(\"is_impossible\", False))})\n"
  "            yield row\n"
  "\n"
  "\n"
  "def _stream_array_items(f, buf: str, chunk: int = 1 << 20) -> Iterator[Any]:\n"
  "    \"\"\"Yield the items of a JSON array whose '[' is already consumed; `buf` is the text right after it.  Items must be\n"
  "    objects (a number cut at a chunk boundary could be mis-read).  Only the current item plus < 1 chunk is in memory.\"\"\"\n"
  "    dec = json.JSONDecoder()\n"
  "    pos = 0\n"
  "    while True:\n"
  "        while True:\n"
  "            pos = _WS_COMMA.match(buf, pos).end()\n"
  "            if pos < len(buf):\n"
  "                break\n"
  "            more = f.read(chunk)\n"
  "            if not more:\n"
  "                raise ValueError(\"unexpected end of JSON array\")\n"
  "            buf, pos = buf[pos:] + more, 0\n"
  "        if buf[pos] == \"]\":\n"
  "            return\n"
  "        grow = chunk\n"
  "        while True:\n"
  "            try:\n"
  "                obj, end = dec.raw_decode(buf, pos)\n"
  "                break\n"
  "            except json.JSONDecodeError:\n"
  "                more = f.read(grow)\n"
  "                if not more:\n"
  "                    raise\n"
  "                buf, pos = buf[pos:] + more, 0\n"
  "                grow = min(grow * 2, 1 << 26)\n"
  "        yield obj\n"
  "        pos = end\n"
  "        if pos > (1 << 20) and pos > len(buf) // 2:\n"
  "            buf, pos = buf[pos:], 0\n"
  "\n"
  "\n"
  "def _iter_jsonl(path: str, strict: bool, stats: Optional[dict] = None) -> Iterator[Any]:\n"
  "    opener = gzip.open if path.lower().endswith(\".gz\") else open\n"
  "    # strict: undecodable bytes and bad lines raise.  lenient (strict=False): undecodable bytes become U+FFFD (the converter reads the\n"
  "    # same way), bad lines are skipped and counted in stats[\"bad_lines\"] / stats[\"decode_replacements\"] when a dict is given.\n"
  "    with opener(path, \"rt\", encoding=\"utf-8\", errors=\"strict\" if strict else \"replace\", newline=\"\\n\") as f:\n"
  "        for ln, line in enumerate(f, 1):\n"
  "            line = line.strip()\n"
  "            if not line:\n"
  "                continue\n"
  "            if stats is not None and not strict and \"�\" in line:\n"
  "                stats[\"decode_replacements\"] = stats.get(\"decode_replacements\", 0) + 1\n"
  "            try:\n"
  "                yield json.loads(line)\n"
  "            except json.JSONDecodeError as e:\n"
  "                if strict:\n"
  "                    raise ValueError(f\"{path}:{ln}: bad JSON line: {e}\") from e\n"
  "                if stats is not None:\n"
  "                    stats[\"bad_lines\"] = stats.get(\"bad_lines\", 0) + 1\n"
  "\n"
  "\n"
  "def _iter_json(path: str, squad_flatten: bool) -> Iterator[Any]:\n"
  "    opener = gzip.open if path.lower().endswith(\".gz\") else open\n"
  "    with opener(path, \"rt\", encoding=\"utf-8-sig\") as f:\n"
  "        head = f.read(1 << 20)\n"
  "        m = _JSON_TOP_ARRAY.match(head)\n"
  "        if not m:\n"
  "            m = _JSON_DATA_KEY.match(head)\n"
  "        if m:\n"
  "            items: Iterable[Any] = _stream_array_items(f, head[m.end():])\n"
  "        else:                                            # unknown layout: whole-file parse (files of this kind are < 100 MB)\n"
  "            obj = json.loads(head + f.read())\n"
  "            if isinstance(obj, list):\n"
  "                items = obj\n"
  "            elif isinstance(obj, dict) and isinstance(obj.get(\"data\"), list):\n"
  "                items = obj[\"data\"]\n"
  "            else:\n"
  "                items = [obj]\n"
  "        for it in items:\n"
  "            if squad_flatten and isinstance(it, dict) and \"paragraphs\" in it:\n"
  "                yield from _flatten_squad_article(it)\n"
  "            else:\n"
  "                yield it\n"
  "\n"
  "\n"
  "def _iter_parquet(path: str, columns: Optional[Sequence[str]], batch_size: int) -> Iterator[dict]:\n"
  "    try:\n"
  "        import pyarrow.parquet as pq\n"
  "    except ImportError as e:                             # pragma: no cover\n"
  "        raise ImportError(\"reading .parquet needs pyarrow (pip install -q pyarrow)\") from e\n"
  "    pf = pq.ParquetFile(path)\n"
  "    for batch in pf.iter_batches(batch_size=batch_size, columns=list(columns) if columns else None):\n"
  "        yield from batch.to_pylist()\n"
  "\n"
  "\n"
  "def iter_rows(path: str, *, columns: Optional[Sequence[str]] = None, limit: Optional[int] = None, batch_size: int = 512,\n"
  "              squad_flatten: bool = True, strict: bool = True, progress: Optional[ProgressCB] = None,\n"
  "              progress_every: int = 50000, label: Optional[str] = None, stats: Optional[dict] = None) -> Iterator[Any]:\n"
  "    \"\"\"Stream the rows of a .jsonl(.gz) / .parquet / .json(.gz) file as dicts.\n"
  "      .jsonl   one JSON value per line (blank lines skipped; strict=True raises on a bad line or an undecodable byte; False replaces\n"
  "               undecodable bytes, skips bad lines and counts them in `stats` (bad_lines, decode_replacements) when a dict is passed)\n"
  "      .parquet pyarrow record batches of `batch_size` rows (only `columns` are read); RAM ~ one row group (~100-300 MB for 330 MB files)\n"
  "      .json    a top-level array of objects, or {\"data\": [...]} (SQuAD / JaQuAD); decoded item by item.  With squad_flatten\n"
  "               every article with \"paragraphs\" is flattened to one dict per question: id, title, context, question, answers,\n"
  "               answer_texts, answer_starts, is_impossible.  Any other layout is parsed whole (use only for small files).\n"
  "    \"\"\"\n"
  "    p = os.fspath(path)\n"
  "    low = p.lower()\n"
  "    if low.endswith((\".jsonl\", \".jsonl.gz\", \".ndjson\")):\n"
  "        gen = _iter_jsonl(p, strict, stats)\n"
  "    elif low.endswith(\".parquet\"):\n"
  "        gen = _iter_parquet(p, columns, batch_size)\n"
  "    elif low.endswith((\".json\", \".json.gz\")):\n"
  "        gen = _iter_json(p, squad_flatten)\n"
  "    else:\n"
  "        raise ValueError(f\"unsupported file type: {p}\")\n"
  "    tick = _Ticker(progress, label or os.path.basename(p), progress_every)\n"
  "    try:\n"
  "        for i, row in enumerate(gen):\n"
  "            if limit is not None and i >= limit:\n"
  "                break\n"
  "            tick.tick()\n"
  "            yield row\n"
  "    finally:\n"
  "        gen.close()\n"
  "    tick.done()\n"
  "\n"
  "\n"
  "def row_count_hint(path: str) -> Optional[int]:\n"
  "    \"\"\"Exact row count for parquet (footer only); None for other formats.\"\"\"\n"
  "    if os.fspath(path).lower().endswith(\".parquet\"):\n"
  "        import pyarrow.parquet as pq\n"
  "        return pq.ParquetFile(path).metadata.num_rows\n"
  "    return None\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Vocabulary + OOV\n"
  "# =====================================================================================================================\n"
  "@dataclasses.dataclass\n"
  "class VocabInfo:\n"
  "    chars: frozenset\n"
  "    source: str\n"
  "    n_entries: int = 0\n"
  "    specials: Tuple[str, ...] = ()\n"
  "    derived: bool = False\n"
  "    warnings: List[str] = dataclasses.field(default_factory=list)\n"
  "\n"
  "    def __contains__(self, c: str) -> bool:\n"
  "        return c in self.chars\n"
  "\n"
  "    def __len__(self) -> int:\n"
  "        return len(self.chars)\n"
  "\n"
  "    @property\n"
  "    def sha256_chars(self) -> str:\n"
  "        return hashlib.sha256(\"\".join(sorted(self.chars)).encode(\"utf-8\", \"surrogatepass\")).hexdigest()\n"
  "\n"
  "    def as_dict(self) -> dict:\n"
  "        return {\"source\": self.source, \"n_entries\": self.n_entries, \"n_single_chars\": len(self.chars), \"specials\": list(self.specials),\n"
  "                \"derived\": self.derived, \"sha256_of_sorted_chars\": self.sha256_chars, \"warnings\": list(self.warnings)}\n"
  "\n"
  "\n"
  "def _vocab_dict_from_obj(obj: Any) -> Optional[dict]:\n"
  "    if isinstance(obj, dict):\n"
  "        v = obj.get(\"vocab\")\n"
  "        if isinstance(v, dict) and v:\n"
  "            return v\n"
  "        m = obj.get(\"model\")\n"
  "        if isinstance(m, dict) and isinstance(m.get(\"vocab\"), dict):\n"
  "            return m[\"vocab\"]\n"
  "        if len(obj) >= 100 and all(isinstance(x, int) for x in obj.values()):\n"
  "            return obj\n"
  "    return None\n"
  "\n"
  "\n"
  "def vocab_from_json_bytes(data: bytes, source: str, *, expected_size: Optional[int] = EXPECTED_VOCAB_SIZE,\n"
  "                          expected_single: Optional[int] = EXPECTED_SINGLE_CHARS) -> VocabInfo:\n"
  "    \"\"\"VocabInfo from the bytes of a vocab.json (token -> id) or a base_metadata.json (key 'vocab').  Raises ValueError when the\n"
  "    file has no vocab or its size differs from the expected 3,898 entries / 3,893 single characters (None disables a check).\"\"\"\n"
  "    vd = _vocab_dict_from_obj(json.loads(data.decode(\"utf-8-sig\")))\n"
  "    if vd is None:\n"
  "        raise ValueError(\"no vocab dict in \" + source)\n"
  "    chars = frozenset(k for k in vd if isinstance(k, str) and len(k) == 1)\n"
  "    specials = tuple(sorted(k for k in vd if isinstance(k, str) and len(k) != 1))\n"
  "    if expected_size is not None and len(vd) != expected_size:\n"
  "        raise ValueError(f\"{source}: {len(vd)} vocab entries, expected {expected_size}\")\n"
  "    if expected_single is not None and len(chars) != expected_single:\n"
  "        raise ValueError(f\"{source}: {len(chars)} single-character entries, expected {expected_single}\")\n"
  "    return VocabInfo(chars=chars, source=source, n_entries=len(vd), specials=specials)\n"
  "\n"
  "\n"
  "def _walk_limited(root: str, max_depth: int) -> Iterator[Tuple[str, List[str]]]:\n"
  "    root = os.path.abspath(root)\n"
  "    base = root.rstrip(os.sep).count(os.sep)\n"
  "    for dirpath, dirnames, filenames in os.walk(root):\n"
  "        depth = dirpath.rstrip(os.sep).count(os.sep) - base\n"
  "        dirnames[:] = sorted(d for d in dirnames if d not in _SKIP_DIRS and not d.startswith(\".\")) if depth < max_depth else []\n"
  "        yield dirpath, sorted(filenames)\n"
  "\n"
  "\n"
  "def _bundle_zip_paths(drive_root: str) -> List[str]:\n"
  "    out = [os.path.join(drive_root, n) for n in VOCAB_BUNDLE_ZIPS if os.path.isfile(os.path.join(drive_root, n))]\n"
  "    for p in sorted(_glob.glob(os.path.join(drive_root, \"GeoCPU_v47_fullmodel_bundle*.zip\"))):\n"
  "        if p not in out:\n"
  "            out.append(p)\n"
  "    return out\n"
  "\n"
  "\n"
  "def load_vocab(drive_root: str = DRIVE_ROOT, out_dir: Optional[str] = None, *, corpus_path: Optional[str] = None,\n"
  "               allow_derive: bool = True, max_depth: int = 3, expected_size: Optional[int] = EXPECTED_VOCAB_SIZE,\n"
  "               expected_single: Optional[int] = EXPECTED_SINGLE_CHARS, cache_name: str = \"vocab_cache.json\",\n"
  "               progress: Optional[ProgressCB] = None) -> VocabInfo:\n"
  "    \"\"\"The model's character vocabulary.  Search order:\n"
  "      0. out_dir/vocab_cache.json (only a cache of a NON-derived result is trusted);\n"
  "      1. base_metadata.json['vocab'] inside GeoCPU_v47_fullmodel_bundle_v1.zip -> assets/r6_bundle.zip (VOCAB_DRIVE_PATH): the model's\n"
  "         own file, always preferred;\n"
  "      1b. only when no bundle yields it: a stand-alone vocab / tokenizer / base_metadata JSON in the Drive run folders (names vocab*.json,\n"
  "         *vocab.json, tokenizer*.json, base_metadata.json, depth <= max_depth); only a file with exactly 3,898 entries / 3,893 single\n"
  "         characters is accepted - anything else is ignored with a warning (and the source says it was a loose file);\n"
  "      2. else (allow_derive) the union of the characters of every full_text of OLD_CORPUS (one streaming pass).  That is the\n"
  "         census of what the model has SEEN (3,498 characters on 2026-11-09), not its vocabulary: it lacks ~400 vocabulary\n"
  "         characters and has 5 the vocabulary lacks.  A warning says so; the result is cached with derived=True.\n"
  "    The result (with its source and warnings) is cached as JSON in out_dir when given.\"\"\"\n"
  "    warnings: List[str] = []\n"
  "    cache_path = os.path.join(out_dir, cache_name) if out_dir else None\n"
  "    if cache_path and os.path.isfile(cache_path):\n"
  "        try:\n"
  "            with open(cache_path, encoding=\"utf-8\") as cf:\n"
  "                c = json.load(cf)\n"
  "            if not c.get(\"derived\") and isinstance(c.get(\"chars\"), str):\n"
  "                return VocabInfo(chars=frozenset(c[\"chars\"]), source=\"cache:\" + str(c.get(\"source\")), n_entries=int(c.get(\"n_entries\", 0)),\n"
  "                                 specials=tuple(c.get(\"specials\", ())), warnings=list(c.get(\"warnings\", [])))\n"
  "        except (ValueError, OSError):\n"
  "            warnings.append(\"ignored unreadable vocab cache \" + cache_path)\n"
  "    info: Optional[VocabInfo] = None\n"
  "    names = re.compile(r\"(?i)^(vocab.*\\.json|.*vocab\\.json|tokenizer.*\\.json|base_metadata\\.json)$\")\n"
  "    if os.path.isdir(drive_root):\n"
  "        # the model's own file first: base_metadata.json inside the bundle.  A stray vocab*.json elsewhere on Drive that merely has\n"
  "        # 3,898 entries must never override it (a stale vocabulary would silently drive the OOV filter for every row).\n"
  "        for zp in _bundle_zip_paths(drive_root):\n"
  "            try:\n"
  "                with zipfile.ZipFile(zp) as z:\n"
  "                    if VOCAB_NESTED_ZIP not in z.namelist():\n"
  "                        continue\n"
  "                    with zipfile.ZipFile(io.BytesIO(z.read(VOCAB_NESTED_ZIP))) as inner:\n"
  "                        if VOCAB_MEMBER not in inner.namelist():\n"
  "                            continue\n"
  "                        info = vocab_from_json_bytes(inner.read(VOCAB_MEMBER), f\"{zp}::{VOCAB_NESTED_ZIP}::{VOCAB_MEMBER}[vocab]\",\n"
  "                                                     expected_size=expected_size, expected_single=expected_single)\n"
  "                        break\n"
  "            except (ValueError, OSError, zipfile.BadZipFile) as e:\n"
  "                warnings.append(f\"rejected bundle {zp}: {e}\")\n"
  "        if info is None:\n"
  "            for dirpath, files in _walk_limited(drive_root, max_depth):\n"
  "                for fn in files:\n"
  "                    if not names.match(fn):\n"
  "                        continue\n"
  "                    p = os.path.join(dirpath, fn)\n"
  "                    try:\n"
  "                        if os.path.getsize(p) > 5 << 20:\n"
  "                            continue\n"
  "                        with open(p, \"rb\") as vf:\n"
  "                            info = vocab_from_json_bytes(vf.read(), p, expected_size=expected_size, expected_single=expected_single)\n"
  "                        warnings.append(\"vocabulary taken from a loose file, not from the model bundle: \" + p)\n"
  "                        break\n"
  "                    except (ValueError, OSError) as e:\n"
  "                        warnings.append(f\"rejected vocab candidate: {e}\")\n"
  "                if info:\n"
  "                    break\n"
  "    if info is None:\n"
  "        if not allow_derive:\n"
  "            raise FileNotFoundError(\"model vocabulary not found under \" + drive_root + \"; warnings: \" + \"; \".join(warnings))\n"
  "        cp = corpus_path or os.path.join(drive_root, OLD_CORPUS_REL)\n"
  "        chars: set = set()\n"
  "        tick = _Ticker(progress, \"derive vocab from \" + os.path.basename(cp), 50000)\n"
  "        for row in iter_rows(cp):\n"
  "            chars.update(default_text(row))\n"
  "            tick.tick()\n"
  "        tick.done()\n"
  "        warnings.append(\"DERIVED from the old corpus census (union of full_text characters), not the model's own vocabulary file\")\n"
  "        info = VocabInfo(chars=frozenset(chars), source=\"derived:\" + cp, n_entries=len(chars), derived=True)\n"
  "    info.warnings = warnings + info.warnings\n"
  "    if cache_path:\n"
  "        try:\n"
  "            write_new_file(cache_path, json.dumps({\"chars\": \"\".join(sorted(info.chars)), \"source\": info.source, \"n_entries\": info.n_entries,\n"
  "                                                   \"specials\": list(info.specials), \"derived\": info.derived, \"warnings\": info.warnings,\n"
  "                                                   \"sha256_of_sorted_chars\": info.sha256_chars}, ensure_ascii=False))\n"
  "        except FileExistsError:\n"
  "            info.warnings.append(\"vocab cache exists with different content; left untouched\")\n"
  "    return info\n"
  "\n"
  "\n"
  "# The training runtime maps every row through this table before it looks characters up (qwenputer_controller_v30.py,\n"
  "# _BASE_NORM_DICT + the full-width ASCII loop; convert.py keeps the same copy).  OOV must be measured AFTER it.\n"
  "_RUNTIME_NORM = {\n"
  "    \"ϵ\": \"ε\", \"ϕ\": \"φ\", \"ϑ\": \"θ\", \"ϰ\": \"κ\", \"ϱ\": \"ρ\", \"ς\": \"σ\", \"ο\": \"o\",\n"
  "    \"Α\": \"A\", \"Β\": \"B\", \"Ε\": \"E\", \"Ζ\": \"Z\", \"Η\": \"H\", \"Ι\": \"I\", \"Κ\": \"K\", \"Μ\": \"M\", \"Ν\": \"N\", \"Ο\": \"O\", \"Ρ\": \"P\",\n"
  "    \"Τ\": \"T\", \"Υ\": \"Y\", \"Χ\": \"X\", \"＞\": \">\", \"＜\": \"<\", \"⇔\": \"↔\", \"↦\": \"→\", \"⊨\": \"⊢\", \"⊤\": \"T\", \"∅\": \"φ\", \"≢\": \"≠\",\n"
  "    \"÷\": \"/\", \"⋅\": \"・\", \"∙\": \"・\", \"⟨\": \"(\", \"⟩\": \")\", \"⟪\": \"(\", \"⟫\": \")\", \"⟦\": \"[\", \"⟧\": \"]\", \"⌈\": \"[\", \"⌉\": \"]\",\n"
  "    \"⌊\": \"[\", \"⌋\": \"]\", \"−\": \"-\",\n"
  "}\n"
  "for _c in range(0xFF01, 0xFF5F):\n"
  "    if chr(_c) not in (\"（\", \"）\", \"！\", \"？\", \"：\", \"～\"):\n"
  "        _RUNTIME_NORM[chr(_c)] = chr(_c - 0xFEE0)\n"
  "RUNTIME_NORMALIZE = str.maketrans(_RUNTIME_NORM)\n"
  "\n"
  "\n"
  "def verify_runtime_normalizer(controller_source: str) -> Tuple[bool, str]:\n"
  "    \"\"\"Compare RUNTIME_NORMALIZE with `_BASE_NORM_DICT` (+ full-width loop) parsed from the text of qwenputer_controller_v30.py\n"
  "    (inside assets/r6_bundle.zip).  Returns (identical, explanation).\"\"\"\n"
  "    tree = ast.parse(controller_source)\n"
  "    for node in tree.body:\n"
  "        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == \"_BASE_NORM_DICT\" for t in node.targets):\n"
  "            d = dict(ast.literal_eval(node.value))\n"
  "            for c in range(0xFF01, 0xFF5F):\n"
  "                if chr(c) not in (\"（\", \"）\", \"！\", \"？\", \"：\", \"～\"):\n"
  "                    d[chr(c)] = chr(c - 0xFEE0)\n"
  "            if d == _RUNTIME_NORM:\n"
  "                return True, f\"identical ({len(d)} entries)\"\n"
  "            only_a = sorted(set(d) - set(_RUNTIME_NORM))\n"
  "            only_b = sorted(set(_RUNTIME_NORM) - set(d))\n"
  "            diff = sorted(k for k in set(d) & set(_RUNTIME_NORM) if d[k] != _RUNTIME_NORM[k])\n"
  "            return False, f\"controller-only keys {only_a[:10]}, checks-only keys {only_b[:10]}, differing values {diff[:10]}\"\n"
  "    return False, \"_BASE_NORM_DICT not found\"\n"
  "\n"
  "\n"
  "def _chars_of(vocab: Any) -> frozenset:\n"
  "    if isinstance(vocab, VocabInfo):\n"
  "        return vocab.chars\n"
  "    return vocab if isinstance(vocab, frozenset) else frozenset(vocab)\n"
  "\n"
  "\n"
  "def oov_fraction(text: str, chars: frozenset) -> Tuple[float, int]:\n"
  "    \"\"\"(share of characters not in `chars`, their count).\"\"\"\n"
  "    if not text:\n"
  "        return 0.0, 0\n"
  "    bad = set(text) - chars\n"
  "    if not bad:\n"
  "        return 0.0, 0\n"
  "    n = sum(text.count(c) for c in bad)\n"
  "    return n / len(text), n\n"
  "\n"
  "\n"
  "def oov_stats(rows: Iterable[Any], vocab: Any, *, text_fn: Optional[Callable[[Any], str]] = None, normalize: Union[bool, dict] = True,\n"
  "              threshold: float = 0.01, top_k: int = 30, keep_per_row: bool = False, group_key: Optional[Callable[[Any], str]] = None,\n"
  "              progress: Optional[ProgressCB] = None, progress_every: int = 50000) -> dict:\n"
  "    \"\"\"Out-of-vocabulary statistics of the row texts.  The texts are first mapped through the runtime's symbol normaliser\n"
  "    (normalize=True; False measures raw characters; a str.maketrans table is also accepted).\n"
  "    Returns n_rows, n_chars, oov_chars, oov_frac (overall), rows_with_any_oov, rows_over_threshold and share_rows_over_threshold\n"
  "    (strictly more than `threshold` = 1 % of a row's characters), rows_changed_by_normalizer, histogram of per-row fractions,\n"
  "    top_oov (char, codepoint, name, count, rows), per_group (same numbers per group_key value) and, with keep_per_row,\n"
  "    per_row_frac (array('f') of the per-row fractions in stream order).\"\"\"\n"
  "    chars = _chars_of(vocab)\n"
  "    table = RUNTIME_NORMALIZE if normalize is True else (normalize if isinstance(normalize, dict) else None)\n"
  "    text_fn = text_fn or default_text\n"
  "    tick = _Ticker(progress, \"oov_stats\", progress_every)\n"
  "\n"
  "    def new() -> dict:\n"
  "        return {\"n_rows\": 0, \"n_chars\": 0, \"oov_chars\": 0, \"rows_with_any_oov\": 0, \"rows_over_threshold\": 0, \"empty_rows\": 0}\n"
  "    total, groups = new(), collections.defaultdict(new)\n"
  "    char_count: collections.Counter = collections.Counter()\n"
  "    char_rows: collections.Counter = collections.Counter()\n"
  "    hist = collections.Counter()\n"
  "    per_row = array.array(\"f\") if keep_per_row else None\n"
  "    changed = 0\n"
  "    for row in rows:\n"
  "        text = text_fn(row)\n"
  "        if table is not None:\n"
  "            t2 = text.translate(table)\n"
  "            if t2 != text:\n"
  "                changed += 1\n"
  "            text = t2\n"
  "        frac, n_bad = oov_fraction(text, chars)\n"
  "        targets = [total] + ([groups[group_key(row)]] if group_key else [])\n"
  "        for s in targets:\n"
  "            s[\"n_rows\"] += 1\n"
  "            s[\"n_chars\"] += len(text)\n"
  "            s[\"oov_chars\"] += n_bad\n"
  "            s[\"empty_rows\"] += not text\n"
  "            if n_bad:\n"
  "                s[\"rows_with_any_oov\"] += 1\n"
  "            if frac > threshold:\n"
  "                s[\"rows_over_threshold\"] += 1\n"
  "        if n_bad:\n"
  "            for c in set(text) - chars:\n"
  "                char_count[c] += text.count(c)\n"
  "                char_rows[c] += 1\n"
  "        hist[\"0\" if frac == 0 else (\"(0,0.1%]\" if frac <= 0.001 else (\"(0.1%,1%]\" if frac <= 0.01 else (\"(1%,5%]\" if frac <= 0.05 else \">5%\")))] += 1\n"
  "        if per_row is not None:\n"
  "            per_row.append(frac)\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "\n"
  "    def fin(s: dict) -> dict:\n"
  "        s = dict(s)\n"
  "        s[\"oov_frac\"] = s[\"oov_chars\"] / s[\"n_chars\"] if s[\"n_chars\"] else 0.0\n"
  "        s[\"share_rows_over_threshold\"] = s[\"rows_over_threshold\"] / s[\"n_rows\"] if s[\"n_rows\"] else 0.0\n"
  "        s[\"share_rows_with_any_oov\"] = s[\"rows_with_any_oov\"] / s[\"n_rows\"] if s[\"n_rows\"] else 0.0\n"
  "        return s\n"
  "    out = fin(total)\n"
  "    out.update({\"threshold\": threshold, \"normalized\": table is not None, \"rows_changed_by_normalizer\": changed,\n"
  "                \"vocab_size\": len(chars), \"histogram\": {k: hist[k] for k in (\"0\", \"(0,0.1%]\", \"(0.1%,1%]\", \"(1%,5%]\", \">5%\")},\n"
  "                \"top_oov\": [{\"char\": c, \"codepoint\": f\"U+{ord(c):04X}\", \"name\": unicodedata.name(c, \"\"), \"count\": n, \"rows\": char_rows[c]}\n"
  "                            for c, n in sorted(char_count.items(), key=lambda kv: (-kv[1], kv[0]))[:top_k]],\n"
  "                \"n_distinct_oov_chars\": len(char_count)})\n"
  "    if group_key:\n"
  "        out[\"per_group\"] = {g: fin(s) for g, s in sorted(groups.items())}\n"
  "    if per_row is not None:\n"
  "        out[\"per_row_frac\"] = per_row\n"
  "    return out\n"
  "\n"
  "\n"
  "def oov_filter(rows: Iterable[Any], vocab: Any, *, text_fn: Optional[Callable[[Any], str]] = None, threshold: float = 0.01,\n"
  "               normalize: Union[bool, dict] = True) -> Iterator[Tuple[Any, float, bool]]:\n"
  "    \"\"\"Yield (row, oov_fraction, keep) with keep = fraction <= threshold (rows with MORE than 1 % are dropped).\"\"\"\n"
  "    chars = _chars_of(vocab)\n"
  "    table = RUNTIME_NORMALIZE if normalize is True else (normalize if isinstance(normalize, dict) else None)\n"
  "    text_fn = text_fn or default_text\n"
  "    for row in rows:\n"
  "        text = text_fn(row)\n"
  "        if table is not None:\n"
  "            text = text.translate(table)\n"
  "        frac, _ = oov_fraction(text, chars)\n"
  "        yield row, frac, frac <= threshold\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Length statistics\n"
  "# =====================================================================================================================\n"
  "def row_length(row: Any, text_fn: Optional[Callable[[Any], str]] = None) -> int:\n"
  "    \"\"\"len(prompt + target) + 1 (the +1 is the end token); a row needs <= MAX_LEN (2048).\"\"\"\n"
  "    if text_fn is None and isinstance(row, dict) and isinstance(row.get(\"prompt\"), str) and isinstance(row.get(\"target\"), str):\n"
  "        return len(row[\"prompt\"]) + len(row[\"target\"]) + 1\n"
  "    return len((text_fn or default_text)(row)) + 1\n"
  "\n"
  "\n"
  "def _percentile(items: List[Tuple[int, int]], total: int, q: float) -> int:\n"
  "    rank, cum = max(1, math.ceil(q * total)), 0\n"
  "    for v, c in items:\n"
  "        cum += c\n"
  "        if cum >= rank:\n"
  "            return v\n"
  "    return items[-1][0] if items else 0\n"
  "\n"
  "\n"
  "def length_stats(rows: Iterable[Any], *, text_fn: Optional[Callable[[Any], str]] = None, group_key: Optional[Callable[[Any], str]] = None,\n"
  "                 limit: int = MAX_LEN, soft_limits: Sequence[int] = (1900,), progress: Optional[ProgressCB] = None,\n"
  "                 progress_every: int = 50000) -> dict:\n"
  "    \"\"\"Distribution of L = len(prompt+target)+1.  Per group (and 'ALL'): n, mean, min, max, p50/p90/p95/p99 (exact), over_2048 =\n"
  "    rows with L > 2048 (cannot be trained), over_1900 (L > 1900), chars_over_2000 (len(text) > 2000, the plan's cap),\n"
  "    their shares, full_text_mismatch (rows whose full_text != prompt + target) and a histogram in steps of 256.\"\"\"\n"
  "    counters: Dict[str, collections.Counter] = collections.defaultdict(collections.Counter)\n"
  "    mism: collections.Counter = collections.Counter()\n"
  "    tick = _Ticker(progress, \"length_stats\", progress_every)\n"
  "    for row in rows:\n"
  "        L = row_length(row, text_fn)\n"
  "        names = [\"ALL\"] + ([str(group_key(row))] if group_key else [])\n"
  "        bad = isinstance(row, dict) and isinstance(row.get(\"full_text\"), str) and isinstance(row.get(\"prompt\"), str) \\\n"
  "            and isinstance(row.get(\"target\"), str) and row[\"full_text\"] != row[\"prompt\"] + row[\"target\"]\n"
  "        for n in names:\n"
  "            counters[n][L] += 1\n"
  "            mism[n] += bad\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "    out = {}\n"
  "    for name, cnt in sorted(counters.items()):\n"
  "        items = sorted(cnt.items())\n"
  "        n = sum(cnt.values())\n"
  "        d = {\"n\": n, \"mean\": sum(v * c for v, c in items) / n if n else 0.0, \"min\": items[0][0] if items else 0, \"max\": items[-1][0] if items else 0}\n"
  "        for q in (0.5, 0.9, 0.95, 0.99):\n"
  "            d[f\"p{int(q * 100)}\"] = _percentile(items, n, q)\n"
  "        over = sum(c for v, c in items if v > limit)\n"
  "        d[\"over_2048\"], d[\"share_over_2048\"] = over, (over / n if n else 0.0)\n"
  "        for s in soft_limits:\n"
  "            o = sum(c for v, c in items if v > s)\n"
  "            d[f\"over_{s}\"], d[f\"share_over_{s}\"] = o, (o / n if n else 0.0)\n"
  "        o = sum(c for v, c in items if v > 2001)\n"
  "        d[\"chars_over_2000\"], d[\"share_chars_over_2000\"] = o, (o / n if n else 0.0)\n"
  "        d[\"full_text_mismatch\"] = mism[name]\n"
  "        hist: collections.Counter = collections.Counter()\n"
  "        for v, c in items:\n"
  "            hist[min(v // 256, 10)] += c\n"
  "        d[\"histogram_256\"] = {(f\"{k * 256}-{k * 256 + 255}\" if k < 10 else \"2560+\"): hist[k] for k in sorted(hist)}\n"
  "        out[name] = d\n"
  "    return out\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Exact duplicates\n"
  "# =====================================================================================================================\n"
  "def _norm_key(text: str, mode: str) -> str:\n"
  "    if mode == \"none\":\n"
  "        return text\n"
  "    if mode == \"nfkc\":\n"
  "        text = unicodedata.normalize(\"NFKC\", text)\n"
  "    return \" \".join(text.split())\n"
  "\n"
  "\n"
  "class ExactIndex:\n"
  "    \"\"\"digest -> first tag.  ~100 B per row; add() returns the tag of the earlier identical text or None.\"\"\"\n"
  "\n"
  "    def __init__(self, normalize: str = \"ws\"):\n"
  "        self.normalize = normalize\n"
  "        self._seen: Dict[int, Any] = {}\n"
  "\n"
  "    def __len__(self) -> int:\n"
  "        return len(self._seen)\n"
  "\n"
  "    def key(self, text: str) -> int:\n"
  "        return digest64(_norm_key(text, self.normalize))\n"
  "\n"
  "    def add(self, text: str, tag: Any = None) -> Any:\n"
  "        k = self.key(text)\n"
  "        prev = self._seen.get(k)\n"
  "        if prev is None:\n"
  "            self._seen[k] = tag if tag is not None else True\n"
  "            return None\n"
  "        return prev\n"
  "\n"
  "    def contains(self, text: str) -> Any:\n"
  "        return self._seen.get(self.key(text))\n"
  "\n"
  "\n"
  "def build_exact_index(rows: Iterable[Any], *, text_fn: Optional[Callable[[Any], str]] = None, normalize: str = \"ws\",\n"
  "                      progress: Optional[ProgressCB] = None) -> ExactIndex:\n"
  "    \"\"\"Index of a (large) reference stream, e.g. the 423,348-row old corpus: ~45 MB, one streaming pass.\"\"\"\n"
  "    idx, text_fn = ExactIndex(normalize), text_fn or default_text\n"
  "    tick = _Ticker(progress, \"build_exact_index\", 50000)\n"
  "    for i, row in enumerate(rows):\n"
  "        idx.add(text_fn(row), i)\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "    return idx\n"
  "\n"
  "\n"
  "def exact_dups(rows: Iterable[Any], *, text_fn: Optional[Callable[[Any], str]] = None, normalize: str = \"ws\",\n"
  "               against: Optional[ExactIndex] = None, max_examples: int = 5, progress: Optional[ProgressCB] = None,\n"
  "               progress_every: int = 50000) -> dict:\n"
  "    \"\"\"Exact duplicates (after whitespace normalisation; text_fn picks full_text / prompt / question ...).\n"
  "    within_dup_positions: stream positions of rows equal to an EARLIER row of the same stream (keep-first);\n"
  "    against_dup_positions: positions of rows already present in `against` (cross-dataset / vs-old-corpus).\"\"\"\n"
  "    text_fn = text_fn or default_text\n"
  "    own, within, cross, examples = ExactIndex(normalize), [], [], []\n"
  "    n = 0\n"
  "    tick = _Ticker(progress, \"exact_dups\", progress_every)\n"
  "    for i, row in enumerate(rows):\n"
  "        n += 1\n"
  "        text = text_fn(row)\n"
  "        first = own.add(text, i)\n"
  "        if first is not None:\n"
  "            within.append(i)\n"
  "            if len(examples) < max_examples:\n"
  "                examples.append({\"position\": i, \"id\": default_id(row, i), \"first_position\": first if first is not True else None, \"text\": text[:160]})\n"
  "        if against is not None:\n"
  "            hit = against.contains(text)\n"
  "            if hit is not None:\n"
  "                cross.append(i)\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "    out = {\"n_rows\": n, \"n_unique\": len(own), \"within_dups\": len(within), \"within_dup_rate\": len(within) / n if n else 0.0,\n"
  "           \"within_dup_positions\": within, \"examples\": examples}\n"
  "    if against is not None:\n"
  "        out.update({\"against_dups\": len(cross), \"against_dup_rate\": len(cross) / n if n else 0.0, \"against_dup_positions\": cross,\n"
  "                    \"against_index_rows\": len(against)})\n"
  "    return out\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# MinHash (k = 128 permutations, character 5-grams) + LSH banding (32 bands x 4 rows)\n"
  "# =====================================================================================================================\n"
  "_M32 = 0xFFFFFFFF\n"
  "\n"
  "\n"
  "class MinHasher:\n"
  "    \"\"\"MinHash signatures of the character n-gram set of a text.\n"
  "      * text -> NFKC, casefold, whitespace collapsed; shingles = distinct character n-grams (default 5); each shingle is hashed with\n"
  "        zlib.crc32 over its UTF-32 bytes (32 bit);\n"
  "      * permutation j: h_j(x) = (a_j * x + b_j) mod 2^32 with odd a_j (a bijection of the 32-bit space), a_j / b_j derived from\n"
  "        sha256(\"checks.minhash:<seed>:<j>\"); signature[j] = min over shingles of h_j(shingle).\n"
  "    The expected share of equal signature slots of two texts is the Jaccard similarity of their shingle sets (standard error\n"
  "    0.035 at J = 0.8 with k = 128).  numpy is used when importable (identical output; pass use_numpy=False for pure Python).\n"
  "\n"
  "    HOW SHARP THE CUT IS (binomial, k = 128): a pair is reported when the share of equal slots is >= the cut.  With the cut at\n"
  "    0.75 (the pipeline's choice: target 0.8 minus a 0.05 margin) a pair of true Jaccard 0.90 is found with probability ~1.00,\n"
  "    0.85 ~0.999, 0.80 ~0.93, 0.75 ~0.55, 0.70 ~0.13, 0.65 ~0.01, 0.60 ~0.00.  With the cut at exactly the target the pair AT the target is\n"
  "    found only half of the time, which is a property of any MinHash estimate, not a defect.  near_dups_within drops a row only when it\n"
  "    is near a row that is kept (no chains: the survivor of every dropped row is itself near it).\"\"\"\n"
  "\n"
  "    def __init__(self, k: int = 128, ngram: int = 5, seed: int = 20261109, use_numpy: Optional[bool] = None):\n"
  "        self.k, self.ngram, self.seed = int(k), int(ngram), int(seed)\n"
  "        a, b = [], []\n"
  "        for j in range(self.k):\n"
  "            d = hashlib.sha256(f\"checks.minhash:{self.seed}:{j}\".encode()).digest()\n"
  "            a.append(int.from_bytes(d[0:4], \"little\") | 1)\n"
  "            b.append(int.from_bytes(d[4:8], \"little\"))\n"
  "        self._coeffs = list(zip(a, b))\n"
  "        self._np = None\n"
  "        if use_numpy is not False:\n"
  "            try:\n"
  "                import numpy as np\n"
  "                self._np, self._A, self._B = np, np.array(a, dtype=np.uint64), np.array(b, dtype=np.uint64)\n"
  "            except Exception:                            # noqa: BLE001\n"
  "                if use_numpy:\n"
  "                    raise\n"
  "\n"
  "    @property\n"
  "    def backend(self) -> str:\n"
  "        return \"numpy\" if self._np is not None else \"pure-python\"\n"
  "\n"
  "    @staticmethod\n"
  "    def normalize(text: str) -> str:\n"
  "        return \" \".join(unicodedata.normalize(\"NFKC\", text).casefold().split())\n"
  "\n"
  "    def shingles(self, text: str) -> List[int]:\n"
  "        s = self.normalize(text)\n"
  "        n = self.ngram\n"
  "        if not s:\n"
  "            return []\n"
  "        if len(s) < n:\n"
  "            return [zlib.crc32(s.encode(\"utf-32-le\", \"surrogatepass\"))]\n"
  "        b = s.encode(\"utf-32-le\", \"surrogatepass\")\n"
  "        crc = zlib.crc32\n"
  "        return list({crc(b[4 * i:4 * (i + n)]) for i in range(len(s) - n + 1)})\n"
  "\n"
  "    def signature(self, text: str) -> Optional[array.array]:\n"
  "        \"\"\"array('I') of k slots, or None when the text has no shingles (empty after normalisation).\"\"\"\n"
  "        xs = self.shingles(text)\n"
  "        if not xs:\n"
  "            return None\n"
  "        if self._np is not None:\n"
  "            np = self._np\n"
  "            arr = np.asarray(xs, dtype=np.uint64)\n"
  "            best = None\n"
  "            mask = np.uint64(_M32)\n"
  "            for s in range(0, len(arr), 4096):\n"
  "                m = ((arr[s:s + 4096, None] * self._A[None, :] + self._B[None, :]) & mask).min(axis=0)\n"
  "                best = m if best is None else np.minimum(best, m)\n"
  "            sig = array.array(\"I\")\n"
  "            sig.frombytes(best.astype(np.uint32).tobytes())\n"
  "            return sig\n"
  "        return array.array(\"I\", [min([(a * x + b) & _M32 for x in xs]) for a, b in self._coeffs])\n"
  "\n"
  "\n"
  "class MinHashIndex:\n"
  "    \"\"\"Signatures of added rows + LSH tables (`bands` bands of k/bands slots; default 32 x 4 for k = 128: a pair with Jaccard 0.8\n"
  "    collides in some band with probability ~1.0000, one with 0.7 with 0.9998, 0.6 with 0.988, 0.5 with 0.873; candidates are verified by\n"
  "    the share of equal slots).\n"
  "    Tables are sorted arrays (build() once after adding), queried with bisect; at most `max_per_band` earliest rows per bucket\n"
  "    are inspected (a bucket that large is boilerplate: its earliest members stand for it).\"\"\"\n"
  "\n"
  "    def __init__(self, hasher: Optional[MinHasher] = None, bands: int = 32, snippet_chars: int = 120):\n"
  "        self.hasher = hasher or MinHasher()\n"
  "        self.k = self.hasher.k\n"
  "        if self.k % bands:\n"
  "            raise ValueError(\"k must be a multiple of bands\")\n"
  "        self.bands, self.r = bands, self.k // bands\n"
  "        self.sigs = array.array(\"I\")\n"
  "        self.valid = bytearray()\n"
  "        self.labels: List[Any] = []\n"
  "        self.tags: List[Any] = []\n"
  "        self.snippet_chars = snippet_chars\n"
  "        self.snippets: List[str] = []\n"
  "        self._tables: List[Tuple[array.array, array.array]] = []\n"
  "        self._built = False\n"
  "\n"
  "    def __len__(self) -> int:\n"
  "        return len(self.valid)\n"
  "\n"
  "    def add_signature(self, sig: Optional[array.array], label: Any = None, tag: Any = None, snippet: str = \"\") -> int:\n"
  "        if self._built:\n"
  "            raise RuntimeError(\"index already built; create a new MinHashIndex to add rows\")\n"
  "        pos = len(self.valid)\n"
  "        if sig is None:\n"
  "            self.sigs.extend([0] * self.k)\n"
  "            self.valid.append(0)\n"
  "        else:\n"
  "            self.sigs.extend(sig)\n"
  "            self.valid.append(1)\n"
  "        self.labels.append(label)\n"
  "        self.tags.append(tag)\n"
  "        if self.snippet_chars:\n"
  "            self.snippets.append(snippet[:self.snippet_chars])\n"
  "        return pos\n"
  "\n"
  "    def add_text(self, text: str, label: Any = None, tag: Any = None) -> int:\n"
  "        return self.add_signature(self.hasher.signature(text), label, tag, text if self.snippet_chars else \"\")\n"
  "\n"
  "    def sig_at(self, pos: int) -> array.array:\n"
  "        return self.sigs[pos * self.k:(pos + 1) * self.k]\n"
  "\n"
  "    def build(self) -> \"MinHashIndex\":\n"
  "        n, k4, r4 = len(self.valid), self.k * 4, self.r * 4\n"
  "        ids = [i for i in range(n) if self.valid[i]]\n"
  "        self._tables = []\n"
  "        raw = memoryview(self.sigs).cast(\"B\")\n"
  "        try:\n"
  "            for b in range(self.bands):\n"
  "                off = b * r4\n"
  "                keys = [zlib.crc32(raw[i * k4 + off:i * k4 + off + r4]) for i in ids]\n"
  "                order = sorted(range(len(ids)), key=keys.__getitem__)                 # stable: ids ascend inside a bucket\n"
  "                self._tables.append((array.array(\"I\", (keys[o] for o in order)), array.array(\"I\", (ids[o] for o in order))))\n"
  "        finally:\n"
  "            raw.release()\n"
  "        self._built = True\n"
  "        return self\n"
  "\n"
  "    def agreement(self, sig: array.array, pos: int) -> float:\n"
  "        other = self.sigs[pos * self.k:(pos + 1) * self.k]\n"
  "        return sum(map(operator.eq, sig, other)) / self.k\n"
  "\n"
  "    def query_sig(self, sig: array.array, threshold: float = 0.8, *, below: Optional[int] = None, exclude: Optional[int] = None,\n"
  "                  max_per_band: int = 64) -> List[Tuple[int, float]]:\n"
  "        \"\"\"[(position, estimated Jaccard)] of indexed rows with agreement >= threshold; below=i only looks at rows < i.\"\"\"\n"
  "        if not self._built:\n"
  "            raise RuntimeError(\"call build() before querying\")\n"
  "        raw, r4, cand = sig.tobytes(), self.r * 4, set()\n"
  "        for b, (sk, sid) in enumerate(self._tables):\n"
  "            key = zlib.crc32(raw[b * r4:(b + 1) * r4])\n"
  "            j, m, cnt = bisect.bisect_left(sk, key), len(sk), 0\n"
  "            while j < m and sk[j] == key and cnt < max_per_band:\n"
  "                pid = sid[j]\n"
  "                if below is not None and pid >= below:\n"
  "                    break\n"
  "                cand.add(pid)\n"
  "                j += 1\n"
  "                cnt += 1\n"
  "        out = []\n"
  "        for pid in sorted(cand):\n"
  "            if pid == exclude:\n"
  "                continue\n"
  "            est = self.agreement(sig, pid)\n"
  "            if est >= threshold:\n"
  "                out.append((pid, est))\n"
  "        return out\n"
  "\n"
  "\n"
  "def near_dups_within(rows: Iterable[Any], *, text_fn: Optional[Callable[[Any], str]] = None, id_fn: Optional[Callable[[Any, int], str]] = None,\n"
  "                     threshold: float = 0.8, k: int = 128, bands: int = 32, max_per_band: int = 64, max_examples: int = 5,\n"
  "                     snippet_chars: int = 120, hasher: Optional[MinHasher] = None, progress: Optional[ProgressCB] = None,\n"
  "                     progress_every: int = 20000) -> dict:\n"
  "    \"\"\"Near-duplicate clusters inside one stream, greedy keep-first: rows are taken in order and a row is dropped (listed in\n"
  "    dup_positions) only when it is similar (estimated Jaccard >= threshold) to a row that is KEPT; it is then attached to the most\n"
  "    similar kept row (root_of[i], the survivor).  There are no chains: a~b~c with J(a, c) < threshold keeps a and c (an earlier\n"
  "    version used connected components, where a chain of 31 near pairs removed rows with Jaccard ~0 to the survivor).  Two survivors can\n"
  "    still be near each other when the LSH or the max_per_band cap misses the pair.  cluster_size_hist says how many clusters (a survivor\n"
  "    plus its dropped rows) of each size there are (sizes above 9 are pooled as '10+') and largest_cluster the biggest.  `threshold` is\n"
  "    applied to the ESTIMATE (see MinHasher for how sharp that is).\n"
  "    Memory: see the module docstring (~1.2 KB per row at k = 128).\"\"\"\n"
  "    text_fn, id_fn = text_fn or default_text, id_fn or default_id\n"
  "    idx = MinHashIndex(hasher or MinHasher(k=k), bands=bands, snippet_chars=snippet_chars)\n"
  "    tick = _Ticker(progress, \"minhash signatures\", progress_every)\n"
  "    for i, row in enumerate(rows):\n"
  "        idx.add_text(text_fn(row), id_fn(row, i))\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "    idx.build()\n"
  "    n = len(idx)\n"
  "    roots = list(range(n))                                            # roots[i] == i: kept; else the kept row that i duplicates\n"
  "    examples, tick = [], _Ticker(progress, \"minhash query\", progress_every)\n"
  "    for i in range(n):\n"
  "        tick.tick()\n"
  "        if not idx.valid[i]:\n"
  "            continue\n"
  "        best = None\n"
  "        for j, est in idx.query_sig(idx.sig_at(i), threshold, below=i, max_per_band=max_per_band):\n"
  "            if roots[j] == j and (best is None or est > best[1]):         # candidates ascend by position: ties keep the earliest\n"
  "                best = (j, est)\n"
  "        if best is not None:\n"
  "            j, est = best\n"
  "            roots[i] = j\n"
  "            if len(examples) < max_examples:\n"
  "                examples.append({\"position\": i, \"id\": idx.labels[i], \"similar_to_position\": j, \"similar_to_id\": idx.labels[j],\n"
  "                                 \"est_jaccard\": round(est, 3), \"text\": idx.snippets[i] if idx.snippets else \"\",\n"
  "                                 \"other_text\": idx.snippets[j] if idx.snippets else \"\"})\n"
  "    tick.done()\n"
  "    dup_positions = [i for i in range(n) if roots[i] != i]\n"
  "    sizes = collections.Counter(roots)\n"
  "    size_hist = collections.Counter((\"10+\" if c >= 10 else str(c)) for c in sizes.values() if c > 1)\n"
  "    return {\"n_rows\": n, \"n_rows_without_shingles\": n - sum(idx.valid), \"threshold\": threshold, \"k\": idx.k, \"bands\": bands,\n"
  "            \"cluster_size_hist\": {key: size_hist[key] for key in sorted(size_hist, key=lambda x: (len(x), x))},\n"
  "            \"hasher_backend\": idx.hasher.backend, \"n_dups\": len(dup_positions), \"dup_rate\": len(dup_positions) / n if n else 0.0,\n"
  "            \"dup_positions\": dup_positions, \"root_of\": roots, \"n_clusters_gt1\": sum(1 for r, c in sizes.items() if c > 1),\n"
  "            \"largest_cluster\": max(sizes.values()) if sizes else 0, \"examples\": examples}\n"
  "\n"
  "\n"
  "def cross_near_dups(index_rows: Iterable[Any], probe_rows: Iterable[Any], *, text_fn: Optional[Callable[[Any], str]] = None,\n"
  "                    probe_text_fn: Optional[Callable[[Any], str]] = None, id_fn: Optional[Callable[[Any, int], str]] = None,\n"
  "                    threshold: float = 0.8, k: int = 128, bands: int = 32, max_per_band: int = 64, sample_every: int = 1,\n"
  "                    max_examples: int = 5, snippet_chars: int = 120, progress: Optional[ProgressCB] = None,\n"
  "                    progress_every: int = 20000) -> dict:\n"
  "    \"\"\"New rows (index side, held in memory, <= ~300k) against a big reference stream (probe side, streamed once, never held -\n"
  "    e.g. the 423,348-row old corpus).  Returns how many index rows have a near duplicate in the probe stream\n"
  "    (index_rows_with_match / index_match_rate; matched_index_positions to filter them) and how many probe rows matched.\n"
  "    sample_every=N inspects only every N-th probe row (probe position % N == 0): the index match rate is then a LOWER bound\n"
  "    (about 1/N of the true matches are seen) while probe_match_rate stays an unbiased estimate.\"\"\"\n"
  "    text_fn = text_fn or default_text\n"
  "    probe_text_fn = probe_text_fn or text_fn\n"
  "    id_fn = id_fn or default_id\n"
  "    idx = MinHashIndex(MinHasher(k=k), bands=bands, snippet_chars=snippet_chars)\n"
  "    tick = _Ticker(progress, \"cross: index signatures\", progress_every)\n"
  "    for i, row in enumerate(index_rows):\n"
  "        idx.add_text(text_fn(row), id_fn(row, i))\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "    idx.build()\n"
  "    matched, scanned, probe_hits, examples = set(), 0, 0, []\n"
  "    tick = _Ticker(progress, \"cross: probe stream\", progress_every)\n"
  "    for pi, row in enumerate(probe_rows):\n"
  "        tick.tick()\n"
  "        if sample_every > 1 and pi % sample_every:\n"
  "            continue\n"
  "        text = probe_text_fn(row)\n"
  "        sig = idx.hasher.signature(text)\n"
  "        if sig is None:\n"
  "            continue\n"
  "        scanned += 1\n"
  "        hits = idx.query_sig(sig, threshold, max_per_band=max_per_band)\n"
  "        if hits:\n"
  "            probe_hits += 1\n"
  "            matched.update(j for j, _ in hits)\n"
  "            if len(examples) < max_examples:\n"
  "                j, est = max(hits, key=lambda h: h[1])\n"
  "                examples.append({\"probe_position\": pi, \"probe_id\": id_fn(row, pi), \"probe_text\": text[:snippet_chars or 120],\n"
  "                                 \"index_position\": j, \"index_id\": idx.labels[j], \"index_text\": idx.snippets[j] if idx.snippets else \"\",\n"
  "                                 \"est_jaccard\": round(est, 3)})\n"
  "    tick.done()\n"
  "    n_idx = sum(idx.valid)\n"
  "    return {\"index_rows\": len(idx), \"index_rows_with_match\": len(matched), \"index_match_rate\": len(matched) / n_idx if n_idx else 0.0,\n"
  "            \"probe_rows_scanned\": scanned, \"probe_rows_with_match\": probe_hits, \"probe_match_rate\": probe_hits / scanned if scanned else 0.0,\n"
  "            \"sample_every\": sample_every, \"index_rate_is_lower_bound\": sample_every > 1, \"threshold\": threshold,\n"
  "            \"matched_index_positions\": sorted(matched), \"examples\": examples}\n"
  "\n"
  "\n"
  "def cross_dataset_near_dups(named_rows: Mapping[str, Iterable[Any]], *, text_fn: Optional[Callable[[Any], str]] = None,\n"
  "                            threshold: float = 0.8, k: int = 128, bands: int = 32, max_per_band: int = 64, snippet_chars: int = 80,\n"
  "                            progress: Optional[ProgressCB] = None, progress_every: int = 20000, deadline: Optional[float] = None,\n"
  "                            info: Optional[dict] = None) -> dict:\n"
  "    \"\"\"Dataset x dataset duplicate matrix.  rows_with_match_in[A][B] = number of rows of A that have a near duplicate (estimated\n"
  "    Jaccard >= threshold) among the rows of B; for A == B it is the number of rows that belong to some duplicate pair inside A\n"
  "    (both members count).  All datasets share one index (~1.2 KB per row in total).\n"
  "    deadline = an absolute time.time() value: when it passes, the signature phase or the query phase stops and info[\"truncated\"]\n"
  "    (a dict passed by the caller) says which; the matrix then only covers the rows handled so far.\"\"\"\n"
  "    text_fn = text_fn or default_text\n"
  "    idx = MinHashIndex(MinHasher(k=k), bands=bands, snippet_chars=snippet_chars)\n"
  "    names = list(named_rows)\n"
  "    late = lambda: deadline is not None and time.time() > deadline            # noqa: E731\n"
  "    stopped = False\n"
  "    for ds, rows in named_rows.items():\n"
  "        tick = _Ticker(progress, f\"signatures[{ds}]\", progress_every)\n"
  "        for i, row in enumerate(rows):\n"
  "            if i % 512 == 0 and late():\n"
  "                stopped = True\n"
  "                break\n"
  "            idx.add_text(text_fn(row), default_id(row, i), tag=ds)\n"
  "            tick.tick()\n"
  "        tick.done()\n"
  "        if stopped:\n"
  "            break\n"
  "    if stopped and info is not None:\n"
  "        info[\"truncated\"] = \"signatures\"\n"
  "    idx.build()\n"
  "    n_rows = collections.Counter(t for t, v in zip(idx.tags, idx.valid) if v)\n"
  "    matrix = {a: collections.Counter() for a in names}\n"
  "    examples: List[dict] = []\n"
  "    tick = _Ticker(progress, \"cross-dataset query\", progress_every)\n"
  "    for i in range(len(idx)):\n"
  "        tick.tick()\n"
  "        if i % 512 == 0 and late():\n"
  "            if info is not None:\n"
  "                info.setdefault(\"truncated\", \"query\")\n"
  "                info[\"queried_rows\"] = i\n"
  "            break\n"
  "        if not idx.valid[i]:\n"
  "            continue\n"
  "        hits = idx.query_sig(idx.sig_at(i), threshold, exclude=i, max_per_band=max_per_band)\n"
  "        for other in {idx.tags[j] for j, _ in hits}:\n"
  "            matrix[idx.tags[i]][other] += 1\n"
  "        for j, est in hits:\n"
  "            if idx.tags[j] != idx.tags[i] and len(examples) < 5:\n"
  "                examples.append({\"a\": idx.tags[i], \"a_id\": idx.labels[i], \"a_text\": idx.snippets[i], \"b\": idx.tags[j], \"b_id\": idx.labels[j],\n"
  "                                 \"b_text\": idx.snippets[j], \"est_jaccard\": round(est, 3)})\n"
  "                break\n"
  "    tick.done()\n"
  "    return {\"threshold\": threshold, \"rows\": dict(n_rows), \"rows_with_match_in\": {a: dict(c) for a, c in matrix.items()}, \"examples\": examples}\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Eval files, eval index, decontamination (digit-stripped 13-gram character shingles)\n"
  "# =====================================================================================================================\n"
  "class EvalFilesMissing(FileNotFoundError):\n"
  "    \"\"\"A required eval file could not be found (or an eval file is unreadable).\"\"\"\n"
  "\n"
  "\n"
  "@dataclasses.dataclass(frozen=True)\n"
  "class EvalSource:\n"
  "    label: str\n"
  "    chain: Tuple[str, ...]            # (file,) or (zip, member[, nested member...]); nested zips are read through memory\n"
  "    kind: str                         # 'jsonl' | 'json' | 'py'\n"
  "    required: bool = True\n"
  "    description: str = \"\"\n"
  "\n"
  "    @property\n"
  "    def display(self) -> str:\n"
  "        return \"::\".join(self.chain)\n"
  "\n"
  "\n"
  "def _guess_kind(name: str) -> str:\n"
  "    low = name.lower()\n"
  "    return \"jsonl\" if low.endswith((\".jsonl\", \".ndjson\")) else (\"py\" if low.endswith(\".py\") else \"json\")\n"
  "\n"
  "\n"
  "def _read_chain(chain: Sequence[str]) -> bytes:\n"
  "    if len(chain) == 1:\n"
  "        with open(chain[0], \"rb\") as f:\n"
  "            return f.read()\n"
  "    z = zipfile.ZipFile(chain[0])\n"
  "    try:\n"
  "        for member in chain[1:-1]:\n"
  "            inner = zipfile.ZipFile(io.BytesIO(z.read(member)))\n"
  "            z.close()\n"
  "            z = inner\n"
  "        return z.read(chain[-1])\n"
  "    finally:\n"
  "        z.close()\n"
  "\n"
  "\n"
  "def eval_item_strings(obj: Any, text_keys: Sequence[str] = EVAL_TEXT_KEYS) -> List[str]:\n"
  "    \"\"\"The text fields of one eval record (each choice is its own string, so 13-grams never span fields).\"\"\"\n"
  "    out: List[str] = []\n"
  "    if isinstance(obj, dict):\n"
  "        for k in text_keys:\n"
  "            v = obj.get(k)\n"
  "            if isinstance(v, str):\n"
  "                out.append(v)\n"
  "            elif isinstance(v, (list, tuple)):\n"
  "                out.extend(x for x in v if isinstance(x, str))\n"
  "        if not out:\n"
  "            out = [s for s in _string_leaves(obj) if len(s) >= 13]\n"
  "    elif isinstance(obj, str):\n"
  "        out.append(obj)\n"
  "    return out\n"
  "\n"
  "\n"
  "def iter_eval_items(src: Union[EvalSource, str], text_keys: Sequence[str] = EVAL_TEXT_KEYS) -> Iterator[List[str]]:\n"
  "    \"\"\"Items (list of strings) of an eval source.  jsonl/json: the EVAL_TEXT_KEYS fields of every record;\n"
  "    py: every string literal of >= 20 characters (the Tier-8 prompts are literals in tier8_readonly_v44fix.py).\"\"\"\n"
  "    if isinstance(src, str):\n"
  "        src = EvalSource(os.path.basename(src), (src,), _guess_kind(src))\n"
  "    data = _read_chain(src.chain)\n"
  "    if src.kind == \"py\":\n"
  "        for node in ast.walk(ast.parse(data.decode(\"utf-8\"))):\n"
  "            if isinstance(node, ast.Constant) and isinstance(node.value, str) and len(node.value) >= 20:\n"
  "                yield [node.value]\n"
  "    elif src.kind == \"jsonl\":\n"
  "        for line in data.decode(\"utf-8-sig\").splitlines():\n"
  "            if line.strip():\n"
  "                items = eval_item_strings(json.loads(line), text_keys)\n"
  "                if items:\n"
  "                    yield items\n"
  "    else:\n"
  "        obj = json.loads(data.decode(\"utf-8-sig\"))\n"
  "        for rec in (obj if isinstance(obj, list) else (obj.get(\"data\", [obj]) if isinstance(obj, dict) else [obj])):\n"
  "            items = eval_item_strings(rec, text_keys)\n"
  "            if items:\n"
  "                yield items\n"
  "\n"
  "\n"
  "@dataclasses.dataclass\n"
  "class EvalFiles:\n"
  "    sources: List[EvalSource] = dataclasses.field(default_factory=list)\n"
  "    missing_required: List[str] = dataclasses.field(default_factory=list)\n"
  "    missing_optional: List[str] = dataclasses.field(default_factory=list)\n"
  "    notes: List[str] = dataclasses.field(default_factory=list)\n"
  "\n"
  "    def report(self) -> str:\n"
  "        lines = [f\"{len(self.sources)} eval sources\"]\n"
  "        lines += [f\"  {'R' if s.required else 'o'} {s.label}: {s.display}\" for s in self.sources]\n"
  "        lines += [f\"  MISSING (required): {m}\" for m in self.missing_required]\n"
  "        lines += [f\"  missing (optional): {m}\" for m in self.missing_optional]\n"
  "        lines += [f\"  note: {n}\" for n in self.notes]\n"
  "        return \"\\n\".join(lines)\n"
  "\n"
  "\n"
  "def _index_loose(roots: Sequence[str], max_depth: int) -> Dict[str, str]:\n"
  "    \"\"\"name -> path of every stand-alone file under the roots (depth-limited walk, first root / first path wins).  One pass.\"\"\"\n"
  "    out: Dict[str, str] = {}\n"
  "    for root in roots:\n"
  "        if os.path.isdir(root):\n"
  "            for dirpath, files in _walk_limited(root, max_depth):\n"
  "                for fn in files:\n"
  "                    out.setdefault(fn, os.path.join(dirpath, fn))\n"
  "    return out\n"
  "\n"
  "\n"
  "def _glob_fallback(name: str, roots: Sequence[str]) -> Optional[str]:\n"
  "    \"\"\"Recursive glob (slow on a Drive mount; also looks into the folders the walk skips).  Only used for names that the\n"
  "    index and the bundle zips did not provide.\"\"\"\n"
  "    for root in roots:\n"
  "        hits = sorted(_glob.glob(os.path.join(root, \"**\", name), recursive=True)) if os.path.isdir(root) else []\n"
  "        if hits:\n"
  "            return hits[0]\n"
  "    return None\n"
  "\n"
  "\n"
  "def locate_registry(name: str, loose: Mapping[str, str], bundles: Sequence[tuple], members: Mapping[str, Tuple[str, str]]) -> Optional[Tuple[str, ...]]:\n"
  "    \"\"\"Where an EVAL_REGISTRY_EXTRA file is: a stand-alone file, the bundle's nested data/ folder, or a suite zip member (by file name).\"\"\"\n"
  "    if name in loose:\n"
  "        return (loose[name],)\n"
  "    for zp, outer, nested in bundles:\n"
  "        if nested is not None and f\"data/{name}\" in nested:\n"
  "            return (zp, EVAL_NESTED_ZIP, f\"data/{name}\")\n"
  "    return members.get(name)\n"
  "\n"
  "\n"
  "def find_eval_files(drive_root: str = DRIVE_ROOT, *, extra_dirs: Sequence[str] = (), strict: bool = True, max_depth: int = 4) -> EvalFiles:\n"
  "    \"\"\"Locate the eval texts the training rows must not overlap.  For every name in EVAL_REQUIRED_JSONL / EVAL_REQUIRED_PY:\n"
  "    a stand-alone file under drive_root / extra_dirs wins; otherwise the file is taken from the Drive bundle\n"
  "    GeoCPU_v47_fullmodel_bundle_v1.zip (-> assets/r6_bundle.zip -> data/<name>; code/<name> for .py); otherwise a recursive glob\n"
  "    is tried.  Optional: the v48 unseen_eval/dev jsonl (EVAL_OPTIONAL_V48), the other frozen suites of the owner's registry\n"
  "    (EVAL_REGISTRY_EXTRA: python120, engineering64, visual10, the new_public dev sets ... as stand-alone files, in the bundle or in the\n"
  "    suite zips; the ones not found are listed in missing_optional) and the .jsonl members of the small suite zips whose names contain\n"
  "    eval / heldout / manifest / bench.  strict=True raises EvalFilesMissing, naming every missing required file.\"\"\"\n"
  "    ef = EvalFiles()\n"
  "    roots = [drive_root] + [d for d in extra_dirs]\n"
  "    bundles = []                                           # (zip path, outer names, nested names or None)\n"
  "    for zp in _bundle_zip_paths(drive_root) if os.path.isdir(drive_root) else []:\n"
  "        try:\n"
  "            with zipfile.ZipFile(zp) as z:\n"
  "                outer = set(z.namelist())\n"
  "                nested = None\n"
  "                if EVAL_NESTED_ZIP in outer:\n"
  "                    with zipfile.ZipFile(io.BytesIO(z.read(EVAL_NESTED_ZIP))) as inner:\n"
  "                        nested = set(inner.namelist())\n"
  "                bundles.append((zp, outer, nested))\n"
  "        except (OSError, zipfile.BadZipFile) as e:\n"
  "            ef.notes.append(f\"unreadable bundle {zp}: {e}\")\n"
  "\n"
  "    loose = _index_loose(roots, max_depth)\n"
  "\n"
  "    def locate(name: str, folder: str) -> Optional[Tuple[str, ...]]:\n"
  "        if name in loose:\n"
  "            return (loose[name],)\n"
  "        for zp, outer, nested in bundles:\n"
  "            if nested is not None and f\"data/{name}\" in nested:\n"
  "                return (zp, EVAL_NESTED_ZIP, f\"data/{name}\")\n"
  "            if f\"{folder}/{name}\" in outer:\n"
  "                return (zp, f\"{folder}/{name}\")\n"
  "        p = _glob_fallback(name, roots)\n"
  "        return (p,) if p else None\n"
  "    for table, folder, kind in ((EVAL_REQUIRED_JSONL, \"data\", \"jsonl\"), (EVAL_REQUIRED_PY, \"code\", \"py\")):\n"
  "        for name, desc in table.items():\n"
  "            chain = locate(name, folder)\n"
  "            if chain is None:\n"
  "                ef.missing_required.append(f\"{name} ({desc})\")\n"
  "            else:\n"
  "                ef.sources.append(EvalSource(label=name, chain=chain, kind=kind, required=True, description=desc))\n"
  "    for name in EVAL_OPTIONAL_V48:\n"
  "        p = loose.get(name)\n"
  "        if p:\n"
  "            ef.sources.append(EvalSource(label=\"v48:\" + name, chain=(p,), kind=\"jsonl\", required=False, description=\"v48 additive eval/dev set\"))\n"
  "        else:\n"
  "            ef.missing_optional.append(\"v48:\" + name)\n"
  "    if ef.missing_optional:\n"
  "        ef.notes.append(\"v48 unseen_eval/dev jsonl are not on Drive as stand-alone files (local copy: research/v48_balanced_augmentation_\"\n"
  "                        \"20261108/data/); copy them under the Drive root or pass extra_dirs to include them\")\n"
  "    pat = re.compile(r\"(?i)(eval|heldout|manifest|bench).*\\.jsonl$\")\n"
  "    suite_zips = [loose[zn] for zn in EVAL_OPTIONAL_SUITE_ZIPS if zn in loose]\n"
  "    suite_zips += [pth for name, pth in sorted(loose.items()) if re.fullmatch(r\"(?i)GeoCPU_v47_.*eval.*\\.zip\", name) and pth not in suite_zips]\n"
  "    members: Dict[str, Tuple[str, str]] = {}                 # basename -> (zip, member) for the registry names found inside suite zips\n"
  "    for zp in suite_zips:\n"
  "        try:\n"
  "            with zipfile.ZipFile(zp) as z:\n"
  "                for info in z.infolist():\n"
  "                    base = os.path.basename(info.filename)\n"
  "                    if base in EVAL_REGISTRY_EXTRA and info.file_size <= 50 << 20:\n"
  "                        members.setdefault(base, (zp, info.filename))\n"
  "        except (OSError, zipfile.BadZipFile) as e:\n"
  "            ef.notes.append(f\"unreadable suite zip {zp}: {e}\")\n"
  "    for name, desc in EVAL_REGISTRY_EXTRA.items():\n"
  "        chain = locate_registry(name, loose, bundles, members)\n"
  "        if chain is None:\n"
  "            ef.missing_optional.append(\"registry:\" + name + f\" ({desc})\")\n"
  "        else:\n"
  "            ef.sources.append(EvalSource(label=\"registry:\" + name, chain=chain, kind=_guess_kind(name), required=False, description=desc))\n"
  "    for zp in suite_zips:\n"
  "        try:\n"
  "            with zipfile.ZipFile(zp) as z:\n"
  "                for info in z.infolist():\n"
  "                    base = os.path.basename(info.filename)\n"
  "                    if pat.search(base) and base not in EVAL_REGISTRY_EXTRA and info.file_size <= 20 << 20:\n"
  "                        ef.sources.append(EvalSource(label=f\"suite:{os.path.basename(zp)}:{base}\", chain=(zp, info.filename),\n"
  "                                                     kind=\"jsonl\", required=False, description=\"eval member of a suite zip\"))\n"
  "        except (OSError, zipfile.BadZipFile):\n"
  "            pass                                              # already noted above\n"
  "    ef.sources.sort(key=lambda s: (not s.required, s.label))\n"
  "    if ef.missing_required and strict:\n"
  "        raise EvalFilesMissing(\"required eval files not found under \" + \", \".join(roots) + \":\\n  \" + \"\\n  \".join(ef.missing_required)\n"
  "                               + \"\\n(searched stand-alone files, \" + \", \".join(VOCAB_BUNDLE_ZIPS) + \" -> \" + EVAL_NESTED_ZIP\n"
  "                               + \" -> data/, and a recursive glob)\")\n"
  "    return ef\n"
  "\n"
  "\n"
  "_STRIP_DIGITS_WS = re.compile(r\"[\\d\\s]+\")\n"
  "\n"
  "\n"
  "WHOLE_ITEM_MIN_CHARS = 40          # an eval text of at least this many normalised characters is a whole item: a row that contains it is flagged\n"
  "\n"
  "\n"
  "def normalize_for_overlap(text: str) -> str:\n"
  "    \"\"\"The training loader's symbol map (RUNTIME_NORMALIZE: U+2212 -> '-', Greek capitals -> Latin, brackets, ...), NFKC, casefold, every\n"
  "    decimal digit (any script) and every whitespace character removed: '12冊で' and '15冊で' become equal, and text that the loader turns\n"
  "    into identical tokens is identical here too (a superset of what the loader does).\"\"\"\n"
  "    t = unicodedata.normalize(\"NFKC\", text.translate(RUNTIME_NORMALIZE)).translate(RUNTIME_NORMALIZE)\n"
  "    return _STRIP_DIGITS_WS.sub(\"\", t.casefold())\n"
  "\n"
  "\n"
  "class EvalIndex:\n"
  "    \"\"\"Set of the digit-stripped n-gram (default 13) shingles of all eval texts + the normalised eval texts (for locating examples).\n"
  "    Shingles are held as python hash() ints of the n-gram strings: the process-local seed is irrelevant because nothing hashed is\n"
  "    persisted or reported (shingle strings are); the chance that one false match occurs in a whole run is ~1e-4.\n"
  "    Per source: items, texts, indexable_texts (>= n normalised characters), shingles (distinct shingles the source CONTAINS, summed over its\n"
  "    texts - not only the ones new to the index: a source that repeats another one still holds its own), new_shingles (the ones it added).\n"
  "    `whole`: the eval texts of >= whole_min_chars normalised characters, keyed by the hash of their first n-gram (see whole_matches).\"\"\"\n"
  "\n"
  "    def __init__(self, ngram: int = 13, whole_min_chars: int = WHOLE_ITEM_MIN_CHARS):\n"
  "        self.ngram = ngram\n"
  "        self.whole_min_chars = max(int(whole_min_chars), ngram)\n"
  "        self.whole: Dict[int, List[Tuple[int, int]]] = {}       # first-shingle hash -> [(last-shingle hash, doc id)]\n"
  "        self._whole_seen: set = set()\n"
  "        self.hashes: set = set()\n"
  "        self.docs: List[Tuple[str, int, str]] = []        # (source label, item index, normalised text)\n"
  "        self.per_source: Dict[str, Dict[str, int]] = {}\n"
  "        # shingle -> number of distinct eval ITEMS (label, item) that contain it; only shingles of >= 2 items are stored (absent = 1).\n"
  "        # This, not the number of training rows, decides what is boilerplate: a shingle that many eval items share is a template\n"
  "        # phrase, a shingle that one eval item owns is content - however many training rows copy it.\n"
  "        self.eval_df: Dict[int, int] = {}\n"
  "        self._cur_key: Optional[Tuple[str, int]] = None\n"
  "        self._cur_seen: set = set()\n"
  "\n"
  "    def __len__(self) -> int:\n"
  "        return len(self.hashes)\n"
  "\n"
  "    def add_text(self, label: str, item: int, text: str) -> int:\n"
  "        norm = normalize_for_overlap(text)\n"
  "        n = self.ngram\n"
  "        st = self.per_source.setdefault(label, {\"items\": 0, \"texts\": 0, \"indexable_texts\": 0, \"shingles\": 0, \"new_shingles\": 0})\n"
  "        st[\"texts\"] += 1\n"
  "        if len(norm) < n:\n"
  "            return 0\n"
  "        st[\"indexable_texts\"] += 1\n"
  "        hset = {hash(norm[i:i + n]) for i in range(len(norm) - n + 1)}\n"
  "        if (label, item) != self._cur_key:                # texts of ONE item (question + choices) count once per shingle\n"
  "            self._cur_key, self._cur_seen = (label, item), set()\n"
  "        fresh = hset - self._cur_seen\n"
  "        self._cur_seen |= hset\n"
  "        before = len(self.hashes)\n"
  "        for h in fresh:\n"
  "            if h in self.hashes:\n"
  "                self.eval_df[h] = self.eval_df.get(h, 1) + 1\n"
  "        self.hashes.update(fresh)\n"
  "        doc_id = len(self.docs)\n"
  "        self.docs.append((label, item, norm))\n"
  "        if len(norm) >= self.whole_min_chars:             # a whole item: findable by (first n-gram, last n-gram) whatever the df of its shingles\n"
  "            key = hash(norm)\n"
  "            if key not in self._whole_seen:               # items that differ only in digits have the same normalised text: one entry\n"
  "                self._whole_seen.add(key)\n"
  "                self.whole.setdefault(hash(norm[:n]), []).append((hash(norm[-n:]), doc_id))\n"
  "        st[\"shingles\"] += len(hset)\n"
  "        st[\"new_shingles\"] += len(self.hashes) - before\n"
  "        return len(self.hashes) - before\n"
  "\n"
  "    def whole_matches(self, norm: str, hset: set, inter: set, limit: int = 1) -> List[Tuple[str, int, str]]:\n"
  "        \"\"\"[(source, item, eval text)] of whole eval texts (>= whole_min_chars normalised characters) that occur verbatim inside the\n"
  "        normalised row text `norm`.  hset = the row's n-gram hashes, inter = the ones that are also eval shingles.  A verbatim copy of an\n"
  "        eval item is contamination however many other items share its template phrases (their shingles may all be boilerplate).\"\"\"\n"
  "        out: List[Tuple[str, int, str]] = []\n"
  "        if not self.whole:\n"
  "            return out\n"
  "        for h in inter & self.whole.keys():\n"
  "            for last, doc in self.whole[h]:\n"
  "                if last in hset:\n"
  "                    label, item, text = self.docs[doc]\n"
  "                    if text in norm:\n"
  "                        out.append((label, item, text))\n"
  "                        if len(out) >= limit:\n"
  "                            return out\n"
  "        return out\n"
  "\n"
  "    def find_sources(self, shingle: str, limit: int = 3) -> List[dict]:\n"
  "        out = []\n"
  "        for label, item, norm in self.docs:\n"
  "            p = norm.find(shingle)\n"
  "            if p >= 0:\n"
  "                out.append({\"source\": label, \"item\": item, \"context\": norm[max(0, p - 25):p + len(shingle) + 25]})\n"
  "                if len(out) >= limit:\n"
  "                    break\n"
  "        return out\n"
  "\n"
  "    def stats(self) -> dict:\n"
  "        return {\"ngram\": self.ngram, \"distinct_shingles\": len(self.hashes), \"normalised_texts\": len(self.docs), \"per_source\": self.per_source,\n"
  "                \"shingles_in_several_eval_items\": len(self.eval_df), \"max_eval_items_per_shingle\": max(self.eval_df.values(), default=1),\n"
  "                \"whole_item_texts\": len(self._whole_seen), \"whole_item_min_chars\": self.whole_min_chars}\n"
  "\n"
  "\n"
  "def build_eval_index(sources: Iterable[Union[EvalSource, str, Tuple[str, Iterable[str]]]], *, ngram: int = 13,\n"
  "                     text_keys: Sequence[str] = EVAL_TEXT_KEYS, progress: Optional[ProgressCB] = None) -> EvalIndex:\n"
  "    \"\"\"EvalIndex over EvalSource objects (find_eval_files(...).sources), plain file paths, or (label, iterable of texts) pairs.\"\"\"\n"
  "    idx = EvalIndex(ngram)\n"
  "    for src in sources:\n"
  "        if isinstance(src, tuple) and len(src) == 2 and not isinstance(src[0], EvalSource) and not isinstance(src[1], str):\n"
  "            label, texts = src\n"
  "            for i, t in enumerate(texts):\n"
  "                idx.add_text(label, i, t)\n"
  "                idx.per_source[label][\"items\"] += 1\n"
  "            continue\n"
  "        s = src if isinstance(src, EvalSource) else EvalSource(os.path.basename(src), (src,), _guess_kind(src))\n"
  "        try:\n"
  "            for i, strings in enumerate(iter_eval_items(s, text_keys)):\n"
  "                for t in strings:\n"
  "                    idx.add_text(s.label, i, t)\n"
  "                idx.per_source[s.label][\"items\"] += 1\n"
  "        except (OSError, ValueError, zipfile.BadZipFile, KeyError, SyntaxError) as e:\n"
  "            raise EvalFilesMissing(f\"cannot read eval source {s.display}: {e}\") from e\n"
  "        if progress:\n"
  "            progress(f\"eval index: {s.label}\", len(idx))\n"
  "    return idx\n"
  "\n"
  "\n"
  "def decontaminate(rows: Iterable[Any], index: EvalIndex, *, text_fn: Optional[Callable[[Any], str]] = None,\n"
  "                  id_fn: Optional[Callable[[Any, int], str]] = None, boilerplate_eval_df: int = 20, min_shared: int = 1,\n"
  "                  max_examples: int = 5, row_prefix_chars: int = 160, progress: Optional[ProgressCB] = None,\n"
  "                  progress_every: int = 20000) -> dict:\n"
  "    \"\"\"Flag rows that share a digit-stripped n-gram (13) with any eval text.  A shingle is BOILERPLATE (instruction templates, 'which\n"
  "    of the following ...') when it occurs in at least `boilerplate_eval_df` (20) distinct EVAL ITEMS - the count is taken on the eval\n"
  "    side (EvalIndex.eval_df), never on the training side: an eval question owned by one item has eval df 1, so it flags every row that\n"
  "    copies it however many rows do (a train-side count would wave a widely copied leak through).  Boilerplate is reported and never\n"
  "    flags a row.  A row is flagged when it shares >= min_shared non-boilerplate shingles, OR when it contains a WHOLE eval text of\n"
  "    >= index.whole_min_chars (40) normalised characters verbatim (rows_flagged_whole_item): eval items that differ only in digits\n"
  "    (T1, T6, the v48 unseen_eval sets) share every shingle, so all their shingles are boilerplate by the df rule and a verbatim copy\n"
  "    of a whole item would otherwise pass; copying a template phrase alone (a part of the item) is still only boilerplate.  Two limits of\n"
  "    this rule that were measured and are NOT fixed (the thresholds 13 / 40 / 20 stay: false positives cannot be measured without the\n"
  "    real rows): an eval text of 13-39 normalised characters whose 13-grams are ALL boilerplate can be copied verbatim without being\n"
  "    flagged (the v33 prompt template, 50 items, and one v42 question), and a template phrase that is itself an eval text of 40+\n"
  "    characters flags EVERY row that contains it - read rows_flagged_whole_item per dataset before trusting the drop counts.\n"
  "    One streaming pass.  Returns n_rows, rows_with_any_shared_shingle, rows_boilerplate_only, rows_flagged, flagged_positions (stream\n"
  "    positions, to drop with drop_flagged), shared_count_hist, boilerplate {n_shingles, top[20]: shingle, eval_items, rows},\n"
  "    real_shingles_top (the non-boilerplate shingles that most rows share: a big number means one eval text is copied widely),\n"
  "    examples (<= max_examples: id, shared shingles, the row's first characters, where the shingle sits in the eval text) and the index\n"
  "    stats.  Limits: exact digit-stripped substring overlap only - paraphrases, translations and re-worded numbers are NOT caught (see\n"
  "    semantic_neighbors for a small look-alike search).\"\"\"\n"
  "    text_fn, id_fn = text_fn or default_text, id_fn or default_id\n"
  "    ng, ev, eval_df = index.ngram, index.hashes, index.eval_df\n"
  "    df: collections.Counter = collections.Counter()\n"
  "    shingle_text: Dict[int, str] = {}\n"
  "    hits: Dict[int, Tuple[str, Tuple[int, ...], str]] = {}\n"
  "    whole_pos: Dict[int, Tuple[str, int, str]] = {}          # position -> (source, item, whole eval text) of rows that contain a whole eval item\n"
  "    n_rows = n_short = rows_any = 0\n"
  "    tick = _Ticker(progress, \"decontaminate\", progress_every)\n"
  "    for pos, row in enumerate(rows):\n"
  "        tick.tick()\n"
  "        n_rows += 1\n"
  "        text = text_fn(row)\n"
  "        norm = normalize_for_overlap(text)\n"
  "        if len(norm) < ng:\n"
  "            n_short += 1\n"
  "            continue\n"
  "        hset = {hash(norm[i:i + ng]) for i in range(len(norm) - ng + 1)}\n"
  "        inter = hset & ev\n"
  "        if not inter:\n"
  "            continue\n"
  "        rows_any += 1\n"
  "        first: Dict[int, int] = {}\n"
  "        for i in range(len(norm) - ng + 1):\n"
  "            h = hash(norm[i:i + ng])\n"
  "            if h in inter and h not in first:\n"
  "                first[h] = i\n"
  "        real = []\n"
  "        for h in inter:\n"
  "            df[h] += 1\n"
  "            if h not in shingle_text:\n"
  "                shingle_text[h] = norm[first[h]:first[h] + ng]\n"
  "            if eval_df.get(h, 1) < boilerplate_eval_df:\n"
  "                real.append(h)\n"
  "        whole = index.whole_matches(norm, hset, inter)\n"
  "        if whole:\n"
  "            whole_pos[pos] = whole[0]\n"
  "        if real or whole:\n"
  "            hits[pos] = (id_fn(row, pos), tuple(sorted(real, key=lambda h: shingle_text[h])), text[:row_prefix_chars])\n"
  "    tick.done()\n"
  "    boiler = {h for h in df if eval_df.get(h, 1) >= boilerplate_eval_df}\n"
  "    flagged: Dict[int, List[int]] = {}\n"
  "    for pos, (rid, hs, prefix) in hits.items():\n"
  "        if len(hs) >= min_shared or pos in whole_pos:\n"
  "            flagged[pos] = list(hs)\n"
  "    any_real = len(hits)\n"
  "    hist = collections.Counter()\n"
  "    for pos, real in flagged.items():\n"
  "        c = len(real)\n"
  "        hist[\"whole_item_only\" if c == 0 else (\"1\" if c == 1 else (\"2-4\" if c <= 4 else (\"5-19\" if c <= 19 else \"20+\")))] += 1\n"
  "    examples = []\n"
  "    for pos in sorted(flagged, key=lambda p: (-len(flagged[p]), p))[:max_examples]:\n"
  "        rid, _, prefix = hits[pos]\n"
  "        shingles = sorted(shingle_text[h] for h in flagged[pos])\n"
  "        whole_text = whole_pos[pos][2] if pos in whole_pos else None\n"
  "        probe = shingles[0] if shingles else whole_text[:ng]\n"
  "        examples.append({\"position\": pos, \"id\": rid, \"n_shared_shingles\": len(shingles), \"shared_shingles\": shingles[:3],\n"
  "                         \"contains_whole_eval_item\": pos in whole_pos, \"row_prefix\": prefix, \"eval_matches\": index.find_sources(probe)})\n"
  "    top = sorted(((shingle_text[h], eval_df.get(h, 1), df[h]) for h in boiler), key=lambda t: (-t[2], t[0]))\n"
  "    real_top = sorted(((shingle_text[h], eval_df.get(h, 1), d) for h, d in df.items() if h not in boiler), key=lambda t: (-t[2], t[0]))\n"
  "    return {\"n_rows\": n_rows, \"n_rows_shorter_than_ngram\": n_short, \"ngram\": ng, \"boilerplate_eval_df\": boilerplate_eval_df, \"min_shared\": min_shared,\n"
  "            \"rows_with_any_shared_shingle\": rows_any, \"rows_boilerplate_only\": rows_any - any_real,\n"
  "            \"rows_with_real_overlap\": any_real, \"rows_flagged\": len(flagged), \"flagged_positions\": sorted(flagged),\n"
  "            \"rows_flagged_whole_item\": len(whole_pos), \"whole_item_positions\": sorted(whole_pos), \"whole_item_min_chars\": index.whole_min_chars,\n"
  "            \"shared_count_hist\": {k: hist[k] for k in (\"1\", \"2-4\", \"5-19\", \"20+\", \"whole_item_only\")},\n"
  "            \"boilerplate\": {\"n_shingles\": len(boiler), \"top\": [{\"shingle\": sh, \"eval_items\": e, \"rows\": d} for sh, e, d in top[:20]]},\n"
  "            \"real_shingles_top\": [{\"shingle\": sh, \"eval_items\": e, \"rows\": d} for sh, e, d in real_top[:10]],\n"
  "            \"examples\": examples, \"eval_index\": index.stats(),\n"
  "            \"limits\": \"exact digit-stripped 13-gram overlap only; paraphrases/translations not detected\"}\n"
  "\n"
  "\n"
  "def drop_flagged(rows: Iterable[Any], flagged_positions: Iterable[int]) -> Iterator[Any]:\n"
  "    \"\"\"Second streaming pass: the rows whose stream position is not in flagged_positions.\"\"\"\n"
  "    drop = set(flagged_positions)\n"
  "    return (r for i, r in enumerate(rows) if i not in drop)\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Semantic look-alike hook (character bigram TF-IDF cosine, blocked by rare 4-grams) - for human reading, NOT exhaustive\n"
  "# =====================================================================================================================\n"
  "def _grams(s: str, n: int) -> List[str]:\n"
  "    return [s[i:i + n] for i in range(len(s) - n + 1)] if len(s) >= n else ([s] if s else [])\n"
  "\n"
  "\n"
  "def _tfidf_vec(s: str, idf: Dict[str, float], default_idf: float) -> Dict[str, float]:\n"
  "    cnt = collections.Counter(_grams(s, 2))\n"
  "    vec = {g: (1.0 + math.log(c)) * idf.get(g, default_idf) for g, c in cnt.items()}\n"
  "    norm = math.sqrt(sum(v * v for v in vec.values()))\n"
  "    return {g: v / norm for g, v in vec.items()} if norm else {}\n"
  "\n"
  "\n"
  "def semantic_neighbors(queries: Sequence[str], rows: Iterable[Any], *, text_fn: Optional[Callable[[Any], str]] = None,\n"
  "                       id_fn: Optional[Callable[[Any, int], str]] = None, top_n: int = 5, min_cos: float = 0.30, block_n: int = 4,\n"
  "                       max_block_df: int = 30, min_shared_blocks: int = 2, idf_buffer_rows: int = 5000, snippet_chars: int = 200,\n"
  "                       progress: Optional[ProgressCB] = None, progress_every: int = 20000, deadline: Optional[float] = None,\n"
  "                       info: Optional[dict] = None) -> List[dict]:\n"
  "    \"\"\"For a SMALL list of eval questions find the train rows most similar by character-bigram TF-IDF cosine (digits and spaces\n"
  "    stripped, NFKC).  Candidate rows are those sharing >= min_shared_blocks rare 4-grams (a 4-gram is a blocking key only if it\n"
  "    occurs in <= max_block_df of the queries); only candidates get a cosine.  One streaming pass over `rows`.  Returns, per\n"
  "    query, {\"query\", \"matches\": [{cosine, position, id, shared_blocks, snippet}]} (best first) - the numbers are for a human to read,\n"
  "    not a verdict.  LIMITS: lexical only (no embeddings); paraphrases that change the wording, translations and re-ordered\n"
  "    sentences are missed; IDF comes from the queries + the first `idf_buffer_rows` rows; recall is unknown, nothing is exhaustive.\n"
  "    deadline = an absolute time.time() value: when it passes the pass over `rows` stops and info[\"truncated\"] / info[\"rows_scanned\"]\n"
  "    (a dict passed by the caller) say so; the matches found so far are returned.\"\"\"\n"
  "    text_fn, id_fn = text_fn or default_text, id_fn or default_id\n"
  "    norm = lambda t: _STRIP_DIGITS_WS.sub(\"\", unicodedata.normalize(\"NFKC\", t).casefold())   # noqa: E731\n"
  "    qs = [norm(q) for q in queries]\n"
  "    q_blocks = [set(_grams(q, block_n)) if len(q) >= block_n else set() for q in qs]\n"
  "    dfq: collections.Counter = collections.Counter()\n"
  "    for s in q_blocks:\n"
  "        dfq.update(s)\n"
  "    block_to_q: Dict[str, List[int]] = collections.defaultdict(list)\n"
  "    for qi, s in enumerate(q_blocks):\n"
  "        for g in sorted(s):\n"
  "            if dfq[g] <= max_block_df:\n"
  "                block_to_q[g].append(qi)\n"
  "    it = iter(rows)\n"
  "    df2: collections.Counter = collections.Counter()\n"
  "    n_docs = len(qs)\n"
  "    for q in qs:\n"
  "        df2.update(set(_grams(q, 2)))\n"
  "    buffered: List[Tuple[Any, str]] = []\n"
  "    for row in itertools.islice(it, idf_buffer_rows):\n"
  "        s = norm(text_fn(row))\n"
  "        buffered.append((row, s))\n"
  "        df2.update(set(_grams(s, 2)))\n"
  "        n_docs += 1\n"
  "    idf = {g: math.log((n_docs + 1) / (c + 1)) + 1.0 for g, c in df2.items()}\n"
  "    default_idf = math.log(n_docs + 1) + 1.0\n"
  "    qvecs = [_tfidf_vec(q, idf, default_idf) for q in qs]\n"
  "    best: List[list] = [[] for _ in qs]\n"
  "    tick = _Ticker(progress, \"semantic_neighbors\", progress_every)\n"
  "\n"
  "    def handle(pos: int, row: Any, s: str) -> None:\n"
  "        tick.tick()\n"
  "        if len(s) < block_n:\n"
  "            return\n"
  "        shared: collections.Counter = collections.Counter()\n"
  "        for g in {s[i:i + block_n] for i in range(len(s) - block_n + 1)}:\n"
  "            lst = block_to_q.get(g)\n"
  "            if lst:\n"
  "                for qi in lst:\n"
  "                    shared[qi] += 1\n"
  "        vec = None\n"
  "        for qi, cnt in shared.items():\n"
  "            if cnt < min_shared_blocks or not qvecs[qi]:\n"
  "                continue\n"
  "            if vec is None:\n"
  "                vec = _tfidf_vec(s, idf, default_idf)\n"
  "            a, b = (vec, qvecs[qi]) if len(vec) <= len(qvecs[qi]) else (qvecs[qi], vec)\n"
  "            cos = sum(v * b.get(g, 0.0) for g, v in a.items())\n"
  "            if cos >= min_cos:\n"
  "                item = (cos, -pos, id_fn(row, pos), text_fn(row)[:snippet_chars], cnt)\n"
  "                if len(best[qi]) < top_n:\n"
  "                    heapq.heappush(best[qi], item)\n"
  "                else:\n"
  "                    heapq.heappushpop(best[qi], item)\n"
  "    scanned, stopped = 0, False\n"
  "    for pos, (row, s) in enumerate(buffered):\n"
  "        if deadline is not None and pos % 256 == 0 and time.time() > deadline:\n"
  "            stopped = True\n"
  "            break\n"
  "        handle(pos, row, s)\n"
  "        scanned += 1\n"
  "    if not stopped:\n"
  "        for pos, row in enumerate(it, start=len(buffered)):\n"
  "            if deadline is not None and pos % 256 == 0 and time.time() > deadline:\n"
  "                stopped = True\n"
  "                break\n"
  "            handle(pos, row, norm(text_fn(row)))\n"
  "            scanned += 1\n"
  "    if stopped and info is not None:\n"
  "        info[\"truncated\"] = True\n"
  "    if info is not None:\n"
  "        info[\"rows_scanned\"] = scanned\n"
  "    tick.done()\n"
  "    return [{\"query\": q, \"matches\": [{\"cosine\": round(c, 4), \"position\": -negpos, \"id\": rid, \"shared_blocks\": cnt, \"snippet\": snip}\n"
  "                                      for c, negpos, rid, snip, cnt in sorted(best[i], reverse=True)]} for i, q in enumerate(queries)]\n"
  "\n"
  "\n"
  "def format_semantic_report(results: Sequence[dict]) -> str:\n"
  "    out = [\"semantic look-alikes (character-bigram TF-IDF cosine; lexical only, not exhaustive)\"]\n"
  "    for i, r in enumerate(results, 1):\n"
  "        out.append(f\"\\n[{i}] QUERY: {r['query']}\")\n"
  "        if not r[\"matches\"]:\n"
  "            out.append(\"    (no candidate above the cosine floor)\")\n"
  "        for m in r[\"matches\"]:\n"
  "            out.append(f\"    cos={m['cosine']:.3f} pos={m['position']} id={m['id']} shared_4grams={m['shared_blocks']}\\n      {m['snippet']!r}\")\n"
  "    return \"\\n\".join(out) + \"\\n\"\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# Code blocks + sandbox\n"
  "# =====================================================================================================================\n"
  "_FENCE_OPEN = re.compile(r\"^([ \\t]{0,3})(`{3,}|~{3,})([^\\n]*)$\")\n"
  "_FENCE_CLOSE = re.compile(r\"^[ \\t]{0,3}(`{3,}|~{3,})[ \\t]*$\")\n"
  "_LLM_CODE = re.compile(r\"<llm-code>(.*?)</llm-code>\", re.S)\n"
  "_PY_LANGS = {\"python\", \"py\", \"python3\", \"py3\", \"ipython\", \"python2\"}\n"
  "\n"
  "\n"
  "@dataclasses.dataclass\n"
  "class CodeBlock:\n"
  "    lang: str\n"
  "    code: str\n"
  "    start_line: int\n"
  "    closed: bool\n"
  "\n"
  "\n"
  "def extract_code_blocks(text: str, *, include_unclosed: bool = False) -> List[CodeBlock]:\n"
  "    \"\"\"Fenced code blocks (``` or ~~~, info string = language, up to 3 spaces of indent) of a markdown-ish text, in order.\"\"\"\n"
  "    blocks: List[CodeBlock] = []\n"
  "    lines = text.replace(\"\\r\\n\", \"\\n\").replace(\"\\r\", \"\\n\").split(\"\\n\")\n"
  "    i = 0\n"
  "    while i < len(lines):\n"
  "        m = _FENCE_OPEN.match(lines[i])\n"
  "        if not m or (m.group(2)[0] == \"`\" and \"`\" in m.group(3)):\n"
  "            i += 1\n"
  "            continue\n"
  "        indent, fence, info = len(m.group(1)), m.group(2), m.group(3).strip()\n"
  "        lang = (info.split()[0] if info else \"\").strip(\"{}.\").lower()\n"
  "        j, body, closed = i + 1, [], False\n"
  "        while j < len(lines):\n"
  "            c = _FENCE_CLOSE.match(lines[j])\n"
  "            if c and c.group(1)[0] == fence[0] and len(c.group(1)) >= len(fence):\n"
  "                closed = True\n"
  "                break\n"
  "            ln = lines[j]\n"
  "            body.append(ln[indent:] if ln[:indent].strip() == \"\" else ln)\n"
  "            j += 1\n"
  "        if not closed:                                    # a truncated answer: drop the blank line that the final newline leaves\n"
  "            while body and not body[-1].strip():\n"
  "                body.pop()\n"
  "        if closed or include_unclosed:\n"
  "            blocks.append(CodeBlock(lang, \"\\n\".join(body), i + 1, closed))\n"
  "        i = j + 1\n"
  "    return blocks\n"
  "\n"
  "\n"
  "def extract_python_blocks(text: str, *, include_unlabeled: bool = False, include_llm_code_tags: bool = True,\n"
  "                          include_unclosed: bool = False) -> List[str]:\n"
  "    \"\"\"The Python code of a text, in order: fenced blocks labelled python / py / python3 / ipython (and unlabeled fences when\n"
  "    include_unlabeled), plus <llm-code>...</llm-code> tags (AcroYAMALEX format).  A block that is never closed (a truncated\n"
  "    answer) is skipped unless include_unclosed.  ```output / ```text / other languages are ignored.\"\"\"\n"
  "    found: List[Tuple[int, str]] = []\n"
  "    for b in extract_code_blocks(text, include_unclosed=include_unclosed):\n"
  "        if b.lang in _PY_LANGS or (include_unlabeled and b.lang == \"\"):\n"
  "            found.append((b.start_line, b.code))\n"
  "    if include_llm_code_tags:\n"
  "        for m in _LLM_CODE.finditer(text):\n"
  "            found.append((text.count(\"\\n\", 0, m.start()) + 1, m.group(1).strip(\"\\n\")))\n"
  "    found.sort(key=lambda t: t[0])\n"
  "    return [c for _, c in found]\n"
  "\n"
  "\n"
  "# --- the sandbox.  WHAT IT IS AND IS NOT.\n"
  "# The static rules are those of the GeoCPU math worker (executor_worker_v44r1.py in the r6 bundle: imports only from math cmath\n"
  "# decimal fractions statistics itertools functools operator collections, no classes, no attribute assignment, no underscore\n"
  "# attributes, no open/eval/exec/compile/getattr/globals ...) PLUS extra hardening found in review (import-from names, frame and\n"
  "# code attributes, str.format only on a literal with plain fields, sanitised module namespaces).  A row that passes here is\n"
  "# therefore also accepted by the model's own executor, and stays inside what the model can really call.\n"
  "#\n"
  "# The Python-level layers (static rules, restricted builtins/imports, an audit hook) are a FILTER, not a security boundary:\n"
  "# CPython sandboxes in pure Python always have escapes.  The boundary is the operating system: on the Colab runtime the child\n"
  "# runs as the unprivileged user 'nobody' (no supplementary groups, cannot read /content/drive, cannot write anywhere but its own\n"
  "# output files), under RLIMIT_CPU / AS / FSIZE / NPROC / NOFILE, in its own session that is killed on timeout, and the notebook\n"
  "# unmounts Drive while any untrusted code runs.  sandbox_selftest() proves these properties on the machine that runs the job.\n"
  "# There is NO weaker fallback: if the sandbox cannot start, the caller must stop (convert.Checks raises).\n"
  "EXECUTOR_ROOTS = (\"math\", \"cmath\", \"decimal\", \"fractions\", \"statistics\", \"itertools\", \"functools\", \"operator\", \"collections\")\n"
  "THIRD_PARTY_ROOTS = (\"sympy\", \"numpy\", \"pandas\")                     # additional roots of the executor; off by default\n"
  "SANDBOX_PROFILES = {\n"
  "    \"executor\": {\"roots\": EXECUTOR_ROOTS, \"allow_classes\": False, \"allow_attr_store\": False, \"allow_private_attrs\": False, \"dunder_names\": ()},\n"
  "}\n"
  "SANDBOX_BACKEND = \"checks.code_sandbox_check(profile=executor)\"\n"
  "\n"
  "\n"
  "class SandboxError(RuntimeError):\n"
  "    \"\"\"The sandbox cannot be started the way it must be (no weaker fallback exists).\"\"\"\n"
  "\n"
  "\n"
  "_VALIDATE_SRC = r'''\n"
  "import ast, re, string\n"
  "class StaticReject(Exception):\n"
  "    pass\n"
  "DENIED_NAMES = {'eval','exec','compile','open','input','help','dir','vars','globals','locals','getattr','setattr','delattr','breakpoint','memoryview','exit','quit','type','object','super','classmethod','staticmethod','property'}\n"
  "DENIED_ATTRS = {'save','savez','savez_compressed','load','loads','dump','fromfile','tofile','memmap','ctypes','ctypeslib','system','popen','fork','spawn','get_handle','urlopen','request','write','writelines','open','codegen','autowrap','ufuncify','os','sys','builtins','subprocess','socket','environ','attrgetter','methodcaller','importlib','import_module','gi_frame','gi_code','gi_yieldfrom','cr_frame','cr_code','cr_await','ag_frame','ag_code','ag_await','f_back','f_globals','f_locals','f_builtins','f_code','f_trace','tb_frame','tb_next','func_globals','func_code','func_closure','cell_contents','co_code','co_consts','co_names','mro','with_traceback'}\n"
  "IMPORT_DENIED = DENIED_NAMES | DENIED_ATTRS | {'call'}\n"
  "ALLOWED_DOTTED = {'collections.abc'}\n"
  "_READ_PREFIXES = ('read_','to_csv','to_excel','to_json','to_pickle','to_parquet','to_hdf','to_sql','to_feather','to_orc')\n"
  "_STR_FORMATTERS = ('format','format_map')\n"
  "MATCH_CLASS_OK = ('int','float','str','bool','list','tuple','dict','set','frozenset','complex','bytes')\n"
  "_FMT_TRAVERSAL = re.compile(r\"\\{[^{}]*__\")\n"
  "_FIELD_OK = re.compile(r\"[0-9]+|[A-Za-z][A-Za-z0-9_]*\")\n"
  "def _check_format(s):\n"
  "    try:\n"
  "        parts = list(string.Formatter().parse(s))\n"
  "    except ValueError:\n"
  "        raise StaticReject(\"format_string|malformed format string\")\n"
  "    for _lit, field, spec, _conv in parts:\n"
  "        if field and not _FIELD_OK.fullmatch(field):\n"
  "            raise StaticReject(\"format_string|only plain {}, {0} and {name} fields are allowed (no attribute or index access)\")\n"
  "        if spec:\n"
  "            _check_format(spec)\n"
  "def _attr_bad(a, cfg):\n"
  "    priv = a.startswith(\"__\") if cfg[\"allow_private_attrs\"] else a.startswith(\"_\")\n"
  "    return priv or a in DENIED_ATTRS or a.startswith(_READ_PREFIXES)\n"
  "def _name_bad(n, cfg):\n"
  "    return (n.startswith(\"__\") and n not in cfg[\"dunder_names\"]) or n in DENIED_NAMES\n"
  "def _strname_bad(a, cfg):\n"
  "    # a name that is carried as a plain STRING in the tree but is looked up as an ATTRIBUTE at run time (from-import names): the attribute\n"
  "    # rules plus the two str formatters, which are only allowed on a string literal (ast.Attribute branch below)\n"
  "    return _attr_bad(a, cfg) or a in _STR_FORMATTERS\n"
  "_MATCH_CLASS_NODE = getattr(ast, 'MatchClass', ())\n"
  "def validate(code, cfg):\n"
  "    roots = set(cfg[\"roots\"])\n"
  "    if not isinstance(code, str):\n"
  "        raise StaticReject(\"type|code must be a str\")\n"
  "    if len(code) > cfg.get(\"max_code_chars\", 32768):\n"
  "        raise StaticReject(\"too_long|%d characters\" % len(code))\n"
  "    if \"\\x00\" in code:\n"
  "        raise StaticReject(\"syntax|null byte\")\n"
  "    try:\n"
  "        tree = ast.parse(code, filename=\"<row-code>\", mode=\"exec\")\n"
  "    except SyntaxError as e:\n"
  "        raise StaticReject(\"syntax|%s\" % e.msg)\n"
  "    except (ValueError, RecursionError, MemoryError) as e:\n"
  "        raise StaticReject(\"syntax|%s\" % type(e).__name__)\n"
  "    bound, class_pattern_names = set(), set()\n"
  "    for node in ast.walk(tree):\n"
  "        if isinstance(node, (ast.Global, ast.Nonlocal)):\n"
  "            raise StaticReject(\"global_nonlocal|global/nonlocal declarations are disabled\")\n"
  "        if isinstance(node, (ast.AsyncFunctionDef, ast.Await, ast.AsyncFor, ast.AsyncWith)):\n"
  "            raise StaticReject(\"async|async code is disabled\")\n"
  "        if isinstance(node, ast.ClassDef) and not cfg[\"allow_classes\"]:\n"
  "            raise StaticReject(\"class_def|class definitions are disabled\")\n"
  "        if isinstance(node, ast.Name) and _name_bad(node.id, cfg):\n"
  "            raise StaticReject(\"name|\" + node.id)\n"
  "        # Names that are carried as plain STRINGS instead of ast.Name / ast.Attribute nodes.  Two kinds:\n"
  "        # (1) names that are LOOKED UP as an attribute at run time: ast.MatchClass.kwd_attrs (`case str(format=f)` is getattr(subject, 'format')\n"
  "        #     - it handed out str.format on a string built at run time) and the names of `from m import n`.  Patching one forbidden name at a\n"
  "        #     time failed twice (__class__, then format), so keyword attributes are DENY BY DEFAULT: a class pattern may not have any, and a\n"
  "        #     class pattern may only name a builtin type.  With no keyword attribute a positional pattern on int / str / ... matches the\n"
  "        #     subject itself (no attribute is read at all).  Honest math code essentially never needs a class pattern with attributes.\n"
  "        # (2) names that are only BOUND (as-patterns, star / mapping captures, except-as, import-as, def names, parameters, type\n"
  "        #     parameters, call keywords such as dict(format=1)): no attribute is ever read from them, so they only get the name rules below.\n"
  "        if getattr(node, \"kwd_attrs\", None):                                 # ast.MatchClass (Python 3.10+)\n"
  "            raise StaticReject(\"match_class|keyword attributes in class patterns are disabled: \" + \", \".join(str(a) for a in node.kwd_attrs))\n"
  "        if isinstance(node, _MATCH_CLASS_NODE):\n"
  "            if not (isinstance(node.cls, ast.Name) and node.cls.id in MATCH_CLASS_OK):\n"
  "                raise StaticReject(\"match_class|class patterns are only allowed on these builtin types: \" + \" \".join(MATCH_CLASS_OK))\n"
  "            class_pattern_names.add(node.cls.id)\n"
  "        for _fld in (\"name\", \"rest\", \"asname\"):\n"
  "            _bound = getattr(node, _fld, None)\n"
  "            if isinstance(_bound, str) and not (_fld == \"name\" and isinstance(node, ast.alias)) and _name_bad(_bound, cfg):\n"
  "                raise StaticReject(\"name|\" + _bound)\n"
  "            if isinstance(_bound, str) and not isinstance(node, ast.alias):\n"
  "                bound.add(_bound)\n"
  "        if isinstance(node, ast.arg):\n"
  "            if _name_bad(node.arg, cfg):\n"
  "                raise StaticReject(\"name|\" + node.arg)\n"
  "            bound.add(node.arg)\n"
  "        if isinstance(node, ast.Name) and isinstance(node.ctx, (ast.Store, ast.Del)):\n"
  "            bound.add(node.id)\n"
  "        if isinstance(node, ast.alias):\n"
  "            bound.add(node.asname or node.name.split(\".\")[0])\n"
  "        if isinstance(node, ast.Attribute):\n"
  "            a = node.attr\n"
  "            if _attr_bad(a, cfg):\n"
  "                raise StaticReject(\"attr|\" + a)\n"
  "            if isinstance(node.ctx, (ast.Store, ast.Del)) and not cfg[\"allow_attr_store\"]:\n"
  "                raise StaticReject(\"attr_store|attribute assignment is disabled: .\" + a)\n"
  "            if a in (\"format\", \"format_map\"):\n"
  "                v = node.value\n"
  "                if not (isinstance(v, ast.Constant) and isinstance(v.value, str)):\n"
  "                    raise StaticReject(\"format_string|str.format / format_map are only allowed on a string literal\")\n"
  "                _check_format(v.value)\n"
  "        if isinstance(node, ast.Import):\n"
  "            for n in node.names:\n"
  "                if n.name.split(\".\")[0] not in roots or (\".\" in n.name and n.name not in ALLOWED_DOTTED):\n"
  "                    raise StaticReject(\"import|\" + n.name)\n"
  "        if isinstance(node, ast.ImportFrom):\n"
  "            mod = node.module or \"\"\n"
  "            if node.level or not mod or mod.split(\".\")[0] not in roots or (\".\" in mod and mod not in ALLOWED_DOTTED):\n"
  "                raise StaticReject(\"import|\" + str(node.module))\n"
  "            for a in node.names:\n"
  "                if a.name == \"*\" or a.name.startswith(\"_\"):\n"
  "                    raise StaticReject(\"wildcard_import|wildcard or private import\")\n"
  "                if a.name in IMPORT_DENIED or _strname_bad(a.name, cfg):\n"
  "                    raise StaticReject(\"import_name|\" + a.name)\n"
  "        if isinstance(node, ast.Constant) and isinstance(node.value, str) and _FMT_TRAVERSAL.search(node.value):\n"
  "            raise StaticReject(\"format_string|attribute traversal inside a format string\")\n"
  "    clash = sorted(class_pattern_names & bound)                  # `str = namedtuple(...)` would turn `case str(x)` into a pattern on another class\n"
  "    if clash:\n"
  "        raise StaticReject(\"match_class|a name that a class pattern uses is rebound in this code: \" + \", \".join(clash))\n"
  "    return tree\n"
  "'''\n"
  "\n"
  "_CHILD_SRC = _VALIDATE_SRC + r'''\n"
  "import builtins, contextlib, importlib, io, json, os, sys, types\n"
  "cfg = json.loads(sys.argv[1])\n"
  "with open(sys.argv[2], encoding=\"utf-8\") as _f:\n"
  "    code = _f.read()\n"
  "try:\n"
  "    import resource\n"
  "except Exception:\n"
  "    resource = None\n"
  "if resource is not None:\n"
  "    def _lim(name, val):\n"
  "        try:\n"
  "            which = getattr(resource, name)\n"
  "            hard = resource.getrlimit(which)[1]\n"
  "            if hard != resource.RLIM_INFINITY:\n"
  "                val = min(val, hard)\n"
  "            resource.setrlimit(which, (val, val))\n"
  "        except Exception:\n"
  "            pass\n"
  "    _lim(\"RLIMIT_CPU\", int(cfg[\"cpu\"]))\n"
  "    if cfg.get(\"mem_mb\"):\n"
  "        _lim(\"RLIMIT_AS\", int(cfg[\"mem_mb\"]) << 20)\n"
  "    _lim(\"RLIMIT_FSIZE\", 1 << 20)\n"
  "    _lim(\"RLIMIT_CORE\", 0)\n"
  "    _lim(\"RLIMIT_NPROC\", 64)\n"
  "    _lim(\"RLIMIT_NOFILE\", 64)\n"
  "ROOTS = set(cfg[\"roots\"])\n"
  "_HIDE = {\"attrgetter\", \"methodcaller\", \"call\"}\n"
  "def _public(mod):\n"
  "    # a namespace of the PUBLIC, NON-MODULE attributes of a module: nothing like random._os, fractions.sys or dataclasses.inspect\n"
  "    return types.SimpleNamespace(**{k: v for k, v in vars(mod).items() if not k.startswith(\"_\") and not isinstance(v, types.ModuleType) and k not in _HIDE})\n"
  "NS = {}\n"
  "for r in sorted(ROOTS):\n"
  "    try:\n"
  "        NS[r] = _public(importlib.import_module(r))\n"
  "    except Exception:\n"
  "        pass\n"
  "DOTTED = {}\n"
  "if \"collections\" in NS:\n"
  "    try:\n"
  "        DOTTED[\"collections.abc\"] = _public(importlib.import_module(\"collections.abc\"))\n"
  "        NS[\"collections\"].abc = DOTTED[\"collections.abc\"]\n"
  "    except Exception:\n"
  "        pass\n"
  "LIB = [os.path.realpath(os.path.dirname(os.__file__)), os.path.realpath(os.path.join(sys.base_prefix, \"DLLs\"))]\n"
  "if cfg.get(\"third_party\"):\n"
  "    for _n in NS:\n"
  "        _fn = getattr(sys.modules.get(_n), \"__file__\", None)\n"
  "        if _fn:\n"
  "            LIB.append(os.path.realpath(os.path.dirname(_fn)))\n"
  "    LIB += [os.path.realpath(p) for p in sys.path if p.endswith((\"site-packages\", \"dist-packages\"))]\n"
  "def _in_lib(p):\n"
  "    try:\n"
  "        p = os.path.realpath(p)\n"
  "    except Exception:\n"
  "        return False\n"
  "    return any(p == r or p.startswith(r + os.sep) for r in LIB)\n"
  "_BLOCK_PREFIX = (\"socket.\", \"subprocess.\", \"ctypes.\", \"urllib.\", \"http.client.\", \"ftplib.\", \"smtplib.\", \"imaplib.\", \"poplib.\", \"telnetlib.\", \"webbrowser.\", \"shutil.\", \"pty.\", \"_posixsubprocess.\")\n"
  "_BLOCK_EVENTS = {\"os.system\", \"os.exec\", \"os.posix_spawn\", \"os.spawn\", \"os.fork\", \"os.forkpty\", \"os.kill\", \"os.killpg\", \"os.remove\", \"os.rmdir\", \"os.rename\", \"os.mkdir\", \"os.chdir\", \"os.chmod\", \"os.chown\", \"os.link\", \"os.symlink\", \"os.truncate\", \"os.putenv\", \"os.unsetenv\", \"os.utime\", \"os.setxattr\", \"os.removexattr\", \"os.listxattr\", \"os.chroot\", \"os.mkfifo\", \"os.mknod\"}\n"
  "def audit(event, args):\n"
  "    if event.startswith(_BLOCK_PREFIX) or event in _BLOCK_EVENTS:\n"
  "        raise PermissionError(\"operation disabled in the sandbox: \" + event)\n"
  "    if event in (\"os.listdir\", \"os.scandir\"):\n"
  "        if not args or not isinstance(args[0], (str, bytes, os.PathLike)) or not _in_lib(os.fsdecode(args[0])):\n"
  "            raise PermissionError(\"filesystem enumeration disabled\")\n"
  "    if event == \"open\":\n"
  "        f, mode, flags = args\n"
  "        if not isinstance(f, (str, bytes, os.PathLike)):\n"
  "            raise PermissionError(\"file descriptor access disabled\")\n"
  "        if flags & (os.O_WRONLY | os.O_RDWR | os.O_CREAT | os.O_TRUNC | os.O_APPEND):\n"
  "            raise PermissionError(\"filesystem writes disabled\")\n"
  "        if not _in_lib(os.fsdecode(f)):\n"
  "            raise PermissionError(\"filesystem reads outside the python library disabled\")\n"
  "sys.addaudithook(audit)\n"
  "_NAMES = ('abs','all','any','bin','bool','bytes','bytearray','chr','complex','dict','divmod','enumerate','filter','float','format','frozenset','hash','hex','int','isinstance','issubclass','iter','len','list','map','max','min','next','oct','ord','pow','print','range','repr','reversed','round','set','slice','sorted','str','sum','tuple','zip','Exception','ValueError','TypeError','ZeroDivisionError','RuntimeError','AssertionError','NameError','IndexError','KeyError','StopIteration','OverflowError','ArithmeticError','LookupError','True','False','None')\n"
  "SAFE = {n: getattr(builtins, n) for n in _NAMES if hasattr(builtins, n)}\n"
  "def safe_import(name, globals=None, locals=None, fromlist=(), level=0):\n"
  "    if level:\n"
  "        raise PermissionError(\"import disabled: relative import\")\n"
  "    if name in DOTTED:\n"
  "        return DOTTED[name] if fromlist else NS[name.split(\".\")[0]]\n"
  "    if \".\" in name or name.split(\".\")[0] not in ROOTS:\n"
  "        raise PermissionError(\"import disabled: \" + name)\n"
  "    if name not in NS:\n"
  "        raise ImportError(\"module unavailable: \" + name)\n"
  "    return NS[name]\n"
  "SAFE[\"__import__\"] = safe_import\n"
  "class OutputLimit(RuntimeError):\n"
  "    pass\n"
  "class Capped(io.TextIOBase):\n"
  "    def __init__(self, cap):\n"
  "        self.cap, self.parts, self.size = cap, [], 0\n"
  "    def write(self, text):\n"
  "        text = str(text)\n"
  "        room = max(0, self.cap - self.size)\n"
  "        if room:\n"
  "            self.parts.append(text[:room])\n"
  "            self.size += min(len(text), room)\n"
  "        if len(text) > room:\n"
  "            raise OutputLimit(\"captured output exceeded %d characters\" % self.cap)\n"
  "        return len(text)\n"
  "    def flush(self):\n"
  "        pass\n"
  "    def getvalue(self):\n"
  "        return \"\".join(self.parts)\n"
  "out, err = Capped(int(cfg[\"cap\"])), Capped(int(cfg[\"cap\"]))\n"
  "res = {\"ok\": False, \"stdout\": \"\", \"error\": \"\"}\n"
  "ns = {\"__builtins__\": SAFE, \"__name__\": \"__main__\" if cfg[\"dunder_names\"] else \"tir_math\"}\n"
  "try:\n"
  "    tree = validate(code, cfg)\n"
  "    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):\n"
  "        exec(compile(tree, \"<row-code>\", \"exec\"), ns, ns)\n"
  "    res[\"ok\"] = True\n"
  "except StaticReject as e:\n"
  "    res[\"error\"] = \"static:\" + str(e).replace(\"|\", \":\", 1)\n"
  "except OutputLimit as e:\n"
  "    res[\"error\"] = \"output_limit:\" + str(e)\n"
  "except BaseException as e:\n"
  "    res[\"error\"] = \"exception:\" + type(e).__name__ + \": \" + str(e)[:300]\n"
  "res[\"stdout\"] = out.getvalue()\n"
  "sys.__stdout__.write(cfg[\"marker\"] + json.dumps(res) + \"\\n\")\n"
  "sys.__stdout__.flush()\n"
  "'''\n"
  "\n"
  "_VALIDATE_NS: Dict[str, Any] = {}\n"
  "exec(compile(_VALIDATE_SRC, \"<checks-validate>\", \"exec\"), _VALIDATE_NS)       # same text the child runs\n"
  "\n"
  "\n"
  "def _profile_cfg(profile: str, allowed_modules: Optional[Iterable[str]], allow_third_party: bool) -> dict:\n"
  "    if profile not in SANDBOX_PROFILES:\n"
  "        raise ValueError(f\"unknown sandbox profile {profile!r} (use {sorted(SANDBOX_PROFILES)})\")\n"
  "    cfg = dict(SANDBOX_PROFILES[profile])\n"
  "    roots = list(allowed_modules) if allowed_modules is not None else list(cfg[\"roots\"])\n"
  "    if allow_third_party:\n"
  "        roots += [r for r in THIRD_PARTY_ROOTS if r not in roots]\n"
  "    cfg[\"roots\"] = sorted(set(roots))\n"
  "    cfg[\"dunder_names\"] = list(cfg[\"dunder_names\"])\n"
  "    cfg[\"third_party\"] = bool(allow_third_party)\n"
  "    return cfg\n"
  "\n"
  "\n"
  "def static_check(code: str, *, profile: str = \"executor\", allowed_modules: Optional[Iterable[str]] = None,\n"
  "                 allow_third_party: bool = False) -> str:\n"
  "    \"\"\"'' when the code passes the static rules, else '<category>:<detail>' (syntax, too_long, global_nonlocal, async, class_def, name,\n"
  "    attr, attr_store, import, import_name, wildcard_import, format_string, match_class).  Profile 'executor' = the GeoCPU math worker's rules\n"
  "    (imports only from math cmath decimal fractions statistics itertools functools operator collections; no classes, no attribute\n"
  "    assignment, no underscore attributes, no open/eval/exec/compile/getattr/globals..., no 'os'/'sys'/'subprocess'/'socket'\n"
  "    attributes, no frame / code introspection attributes, str.format only on a literal with plain fields, no keyword attribute in a\n"
  "    match class pattern and class patterns only on the builtin types int float str bool list tuple dict set frozenset complex bytes).\n"
  "    This is the ONLY profile:\n"
  "    the earlier 'wide' profile (classes, random, re, json ...) was removed because the model's own executor cannot run such code.\"\"\"\n"
  "    cfg = _profile_cfg(profile, allowed_modules, allow_third_party)\n"
  "    try:\n"
  "        _VALIDATE_NS[\"validate\"](code, cfg)\n"
  "    except _VALIDATE_NS[\"StaticReject\"] as e:\n"
  "        return str(e).replace(\"|\", \":\", 1)\n"
  "    return \"\"\n"
  "\n"
  "\n"
  "def _resolve_run_as(run_as: Union[None, bool, str]) -> Optional[Tuple[int, int, str]]:\n"
  "    \"\"\"None (no user switch) or (uid, gid, name) of the unprivileged account the child runs as.  'auto' = 'nobody' when this process is\n"
  "    root on POSIX (Colab), nothing otherwise (a local test run).  An explicit name that cannot be used raises SandboxError.\"\"\"\n"
  "    if run_as in (None, False):\n"
  "        return None\n"
  "    if os.name != \"posix\":\n"
  "        if run_as == \"auto\":\n"
  "            return None\n"
  "        raise SandboxError(f\"run_as={run_as!r}: user switching needs POSIX\")\n"
  "    import pwd\n"
  "    if run_as == \"auto\":\n"
  "        if os.geteuid() != 0:\n"
  "            return None\n"
  "        run_as = \"nobody\"\n"
  "    elif os.geteuid() != 0:\n"
  "        raise SandboxError(f\"run_as={run_as!r} needs root to switch users\")\n"
  "    try:\n"
  "        pw = pwd.getpwnam(str(run_as))\n"
  "    except KeyError as e:\n"
  "        raise SandboxError(f\"user {run_as!r} does not exist on this machine\") from e\n"
  "    if pw.pw_uid == 0:\n"
  "        raise SandboxError(\"refusing to run the sandbox child as root\")\n"
  "    return pw.pw_uid, pw.pw_gid, str(run_as)\n"
  "\n"
  "\n"
  "_POSIX = os.name == \"posix\"\n"
  "_LIVE_CHILDREN: Dict[int, bool] = {}                 # pid (= process-group id: every child is a session leader) of the sandbox children running right now\n"
  "_LIVE_LOCK = threading.RLock()                     # re-entrant on purpose: convert.py's SIGTERM handler runs in the main thread, possibly while _track_child holds the lock\n"
  "\n"
  "\n"
  "def _track_child(pid: int, running: bool) -> None:\n"
  "    with _LIVE_LOCK:\n"
  "        if running:\n"
  "            _LIVE_CHILDREN[pid] = True\n"
  "        else:\n"
  "            _LIVE_CHILDREN.pop(pid, None)\n"
  "\n"
  "\n"
  "def kill_live_sandbox_children() -> int:\n"
  "    \"\"\"SIGKILL the process group of every sandbox child that is running right now (POSIX only; returns how many groups were signalled).\n"
  "    Each child is a session leader (start_new_session), so a signal sent to the CONVERTER's process group never reaches it: convert.py calls\n"
  "    this from its SIGTERM / SIGINT handler and at exit, so that stopping a converter also stops the untrusted code it started.\"\"\"\n"
  "    if not _POSIX:\n"
  "        return 0\n"
  "    with _LIVE_LOCK:\n"
  "        pids = list(_LIVE_CHILDREN)\n"
  "    n = 0\n"
  "    for pid in pids:\n"
  "        try:\n"
  "            os.killpg(pid, signal.SIGKILL)\n"
  "            n += 1\n"
  "        except OSError:\n"
  "            pass                                       # already gone\n"
  "    return n\n"
  "\n"
  "\n"
  "def _file_tail(path: str, nbytes: int) -> bytes:\n"
  "    try:\n"
  "        size = os.path.getsize(path)\n"
  "        with open(path, \"rb\") as f:\n"
  "            if size > nbytes:\n"
  "                f.seek(size - nbytes)\n"
  "            return f.read(nbytes)\n"
  "    except OSError:\n"
  "        return b\"\"\n"
  "\n"
  "\n"
  "def _spawn(script_text: str, args: Sequence[str], *, timeout: float, flags: Sequence[str], python: Optional[str],\n"
  "           acct: Optional[Tuple[int, int, str]], files: Optional[Mapping[str, str]] = None) -> dict:\n"
  "    \"\"\"Run `python <flags> _run.py <args>` once in a fresh temp dir: cwd = that dir, stdin closed, minimal environment, stdout/stderr\n"
  "    to files (never pipes, so a child cannot fill this process's memory; only the tail is read), own session killed on timeout, and\n"
  "    on POSIX as the unprivileged `acct` with no supplementary groups.  Returns {returncode, out, err, timed_out, spawn_error}.\"\"\"\n"
  "    exe = python or sys.executable\n"
  "    env = {\"PYTHONHASHSEED\": \"0\", \"PYTHONIOENCODING\": \"utf-8\", \"PYTHONUTF8\": \"1\", \"PYTHONDONTWRITEBYTECODE\": \"1\"}\n"
  "    if os.name == \"nt\":\n"
  "        for k in (\"SYSTEMROOT\", \"SystemRoot\", \"windir\", \"TEMP\", \"TMP\"):\n"
  "            if k in os.environ:\n"
  "                env[k] = os.environ[k]\n"
  "        env[\"PATH\"] = os.path.dirname(exe)\n"
  "    with tempfile.TemporaryDirectory(prefix=\"v49sbx_\", ignore_cleanup_errors=True) as d:\n"
  "        script, out_p, err_p = os.path.join(d, \"_run.py\"), os.path.join(d, \"out.bin\"), os.path.join(d, \"err.bin\")\n"
  "        payload = {\"_run.py\": script_text}\n"
  "        payload.update(files or {})\n"
  "        for name, text in payload.items():\n"
  "            with open(os.path.join(d, name), \"w\", encoding=\"utf-8\", newline=\"\\n\") as f:\n"
  "                f.write(text)\n"
  "        kw: Dict[str, Any] = {}\n"
  "        if acct is not None:                                      # the unprivileged child must be able to read its script\n"
  "            os.chmod(d, 0o755)\n"
  "            for name in payload:\n"
  "                os.chmod(os.path.join(d, name), 0o644)\n"
  "            kw.update(user=acct[0], group=acct[1], extra_groups=[])\n"
  "        timed_out, rc, spawn_error = False, None, \"\"\n"
  "        with open(out_p, \"wb\") as fo, open(err_p, \"wb\") as fe:\n"
  "            try:\n"
  "                proc = subprocess.Popen([exe] + list(flags) + [script] + list(args), stdin=subprocess.DEVNULL, stdout=fo, stderr=fe, cwd=d,\n"
  "                                        env=env, start_new_session=(os.name == \"posix\"), **kw)\n"
  "            except (OSError, ValueError) as e:\n"
  "                proc, spawn_error = None, f\"{type(e).__name__}: {e}\"\n"
  "            if proc is not None:\n"
  "                if _POSIX:\n"
  "                    _track_child(proc.pid, True)\n"
  "                try:\n"
  "                    try:\n"
  "                        proc.wait(timeout=timeout)\n"
  "                    except subprocess.TimeoutExpired:\n"
  "                        timed_out = True\n"
  "                        try:\n"
  "                            if os.name == \"posix\":\n"
  "                                os.killpg(proc.pid, signal.SIGKILL)\n"
  "                            else:\n"
  "                                proc.kill()\n"
  "                        except OSError:\n"
  "                            pass\n"
  "                        proc.wait()\n"
  "                finally:\n"
  "                    if _POSIX:\n"
  "                        _track_child(proc.pid, False)\n"
  "                rc = proc.returncode\n"
  "        raw_out, raw_err = _file_tail(out_p, 1 << 18), _file_tail(err_p, 4096)\n"
  "    return {\"returncode\": rc, \"out\": raw_out, \"err\": raw_err, \"timed_out\": timed_out, \"spawn_error\": spawn_error}\n"
  "\n"
  "\n"
  "def code_sandbox_run(code: str, timeout: float = 5.0, *, profile: str = \"executor\", allowed_modules: Optional[Iterable[str]] = None,\n"
  "                     allow_third_party: bool = False, max_output_chars: int = 8192, mem_mb: Optional[int] = None,\n"
  "                     python: Optional[str] = None, run_as: Union[None, bool, str] = \"auto\") -> dict:\n"
  "    \"\"\"Run `code` in a fresh python subprocess.  Returns {ok, stdout, error, category, seconds}.\n"
  "    category: '' | static | timeout | output_limit | exception | crash.  error is '<category>:<detail>'.\n"
  "    Defence in depth (see the comment above SANDBOX_PROFILES: the Python layers are a filter, the OS is the boundary): only code that\n"
  "    passes the static rules is ever executed and the child checks it again; python -S -s -B -X utf8 (no site / user site / .pyc;\n"
  "    -I is not used because it would ignore PYTHONHASHSEED=0, which makes set order of strings reproducible); cwd = a fresh temp\n"
  "    dir; stdin closed; minimal environment; restricted builtins; modules are handed out as namespaces of their public non-module\n"
  "    attributes; an audit hook refuses sockets / subprocesses / ctypes / file writes / reads outside the python library; output is\n"
  "    capped at max_output_chars; wall-clock `timeout` (the whole process group is killed).  On POSIX: RLIMIT_CPU / AS (mem_mb, default\n"
  "    1024; 4096 with third party) / FSIZE / CORE / NPROC / NOFILE and, when this process is root (Colab), the child runs as the\n"
  "    unprivileged user 'nobody' (run_as='auto'; None disables; a name forces it).  Windows has no `resource`: only the wall-clock\n"
  "    timeout applies there (local unit tests only).\"\"\"\n"
  "    t0 = time.time()\n"
  "    cfg = _profile_cfg(profile, allowed_modules, allow_third_party)\n"
  "    rej = static_check(code, profile=profile, allowed_modules=allowed_modules, allow_third_party=allow_third_party)\n"
  "    if rej:\n"
  "        return {\"ok\": False, \"stdout\": \"\", \"error\": \"static:\" + rej, \"category\": \"static\", \"seconds\": time.time() - t0}\n"
  "    acct = _resolve_run_as(run_as)\n"
  "    marker = \"@@CHECKS-RESULT-\" + secrets.token_hex(8) + \"@@\"\n"
  "    cfg.update({\"cpu\": int(math.ceil(timeout)) + 1, \"cap\": int(max_output_chars), \"marker\": marker,\n"
  "                \"mem_mb\": mem_mb if mem_mb is not None else (4096 if allow_third_party else 1024)})\n"
  "    flags = [\"-s\", \"-B\", \"-X\", \"utf8\"] + ([] if allow_third_party else [\"-S\"])\n"
  "    r = _spawn(_CHILD_SRC, [json.dumps(cfg), \"code.py\"], timeout=timeout, flags=flags, python=python, acct=acct, files={\"code.py\": code})\n"
  "    secs = time.time() - t0\n"
  "    if r[\"spawn_error\"]:\n"
  "        return {\"ok\": False, \"stdout\": \"\", \"error\": \"crash:spawn:\" + r[\"spawn_error\"][:200], \"category\": \"crash\", \"seconds\": secs}\n"
  "    if r[\"timed_out\"]:\n"
  "        return {\"ok\": False, \"stdout\": \"\", \"error\": f\"timeout:{timeout}s\", \"category\": \"timeout\", \"seconds\": secs}\n"
  "    result = None\n"
  "    for line in reversed(r[\"out\"].decode(\"utf-8\", \"replace\").splitlines()):\n"
  "        if line.startswith(marker):\n"
  "            try:\n"
  "                result = json.loads(line[len(marker):])\n"
  "            except ValueError:\n"
  "                result = None\n"
  "            break\n"
  "    if result is None:\n"
  "        tail = r[\"err\"].decode(\"utf-8\", \"replace\").strip().splitlines()[-1:] or [\"\"]\n"
  "        return {\"ok\": False, \"stdout\": \"\", \"error\": f\"crash:returncode={r['returncode']}:{tail[0][:200]}\", \"category\": \"crash\", \"seconds\": secs}\n"
  "    cat = \"\" if result[\"ok\"] else result[\"error\"].split(\":\", 1)[0]\n"
  "    return {\"ok\": bool(result[\"ok\"]), \"stdout\": result[\"stdout\"], \"error\": result[\"error\"], \"category\": cat, \"seconds\": secs}\n"
  "\n"
  "\n"
  "def code_sandbox_check(code: str, timeout: float = 5.0, **kw: Any) -> Tuple[bool, str, str]:\n"
  "    \"\"\"(ok, stdout, error) - see code_sandbox_run for the flags, profile and limits.  error is '' when ok, else e.g.\n"
  "    'static:import:os', 'static:name:open', 'static:class_def:...', 'timeout:5s', 'exception:ZeroDivisionError: ...',\n"
  "    'output_limit:...', 'crash:returncode=...'.  `print(1 + 1)` -> (True, '2\\\\n', '').\"\"\"\n"
  "    r = code_sandbox_run(code, timeout, **kw)\n"
  "    return r[\"ok\"], r[\"stdout\"], r[\"error\"]\n"
  "\n"
  "\n"
  "# payloads that the static rules must refuse (each is harmless if it ever ran: it only prints or raises)\n"
  "SANDBOX_MUST_REJECT = (\n"
  "    (\"import os\", \"import os\\nprint(os.getcwd())\\n\"),\n"
  "    (\"operator.attrgetter walk\", \"from operator import attrgetter, methodcaller\\nprint(methodcaller('__subclasses__')(attrgetter('__class__.__base__')(0)))\\n\"),\n"
  "    (\"format-string traversal (literal)\", \"print('{0.__class__}'.format(1))\\n\"),\n"
  "    (\"format-string traversal (built)\", \"s = '{0._' + '_class__}'\\nprint(s.format(1))\\n\"),\n"
  "    (\"generator frame walk\", \"g = (g.gi_frame.f_back for _ in [1])\\nprint(next(g))\\n\"),\n"
  "    (\"class definition\", \"class A:\\n    pass\\n\"),\n"
  "    (\"import re\", \"import re\\nprint(re.sub('a', 'b', 'a'))\\n\"),\n"
  "    (\"import random\", \"import random\\nprint(random.random())\\n\"),\n"
  "    (\"open()\", \"print(open('x'))\\n\"),\n"
  "    (\"getattr()\", \"print(getattr(1, 'real'))\\n\"),\n"
  "    (\"match class pattern reads __class__\", \"match 0:\\n    case int(__class__=c):\\n        print(c)\\n\"),\n"
  "    (\"match capture named like a dunder\", \"match 0:\\n    case int() as __builtins__:\\n        print(1)\\n\"),\n"
  "    # keyword attributes in class patterns are refused whatever their name (the format / format_map hole: the str.format rule only existed on ast.Attribute)\n"
  "    (\"match keyword attribute: str.format on a built string\", \"s = '{0._' + '_class__}'\\nmatch s:\\n    case str(format=f):\\n        print(f(1))\\n\"),\n"
  "    (\"match keyword attribute: str.format_map\", \"s = '{a._' + '_class__}'\\nmatch s:\\n    case str(format_map=f):\\n        print(f({'a': 1}))\\n\"),\n"
  "    (\"match keyword attribute: format walks to __globals__\", \"def g():\\n    pass\\ns = '{0._' + '_globals__}'\\nmatch s:\\n    case str(format=f):\\n        print(f(g)[:60])\\n\"),\n"
  "    (\"match keyword attribute: format_map walks to __globals__\", \"def g():\\n    pass\\ns = '{a._' + '_globals__}'\\nmatch s:\\n    case str(format_map=f):\\n        print(f({'a': g})[:60])\\n\"),\n"
  "    (\"match keyword attribute with a benign-looking name\", \"match 3:\\n    case int(real=r):\\n        print(r)\\n\"),\n"
  "    (\"match class pattern on a name that is not a builtin type\", \"from collections import Counter\\nmatch Counter('a'):\\n    case Counter():\\n        print(1)\\n\"),\n"
  "    (\"match class pattern on a rebound builtin name\", \"from collections import namedtuple\\nstr = namedtuple('P', 'x')\\nmatch str(1):\\n    case str(a):\\n        print(a)\\n\"),\n"
  ")\n"
  "_PROBE_SRC = r'''\n"
  "import json, os\n"
  "res = {\"uid\": os.getuid() if hasattr(os, \"getuid\") else None, \"gid\": os.getgid() if hasattr(os, \"getgid\") else None}\n"
  "def t(label, fn):\n"
  "    try:\n"
  "        res[label] = \"ok:\" + str(fn())\n"
  "    except BaseException as e:\n"
  "        res[label] = \"error:\" + type(e).__name__\n"
  "t(\"listdir_drive\", lambda: len(os.listdir(\"/content/drive\")))\n"
  "t(\"listdir_mydrive\", lambda: len(os.listdir(\"/content/drive/MyDrive\")))\n"
  "t(\"read_etc_shadow\", lambda: len(open(\"/etc/shadow\").read()))\n"
  "res[\"can_write_mydrive\"] = os.access(\"/content/drive/MyDrive\", os.W_OK)\n"
  "res[\"can_write_content\"] = os.access(\"/content\", os.W_OK)\n"
  "res[\"can_write_cwd\"] = os.access(\".\", os.W_OK)\n"
  "print(json.dumps(res))\n"
  "'''\n"
  "\n"
  "\n"
  "def sandbox_selftest(*, timeout: float = 20.0, python: Optional[str] = None, require_os_containment: bool = False, light: bool = False,\n"
  "                     run_as: Union[None, bool, str] = \"auto\", drive_unmounted: bool = False) -> dict:\n"
  "    \"\"\"Proves on THIS machine that the sandbox works and refuses what it must.  Returns {ok, backend, run_as, checks: [{name, ok,\n"
  "    detail[, info]}], failed: [names], n_passed, n_info}.  light=True runs only the cheap start-up checks (convert.Checks uses it); the full run (the notebook,\n"
  "    step '3d') adds the escape payloads, the runtime-import smoke test, timeout handling and the OS probe.  With\n"
  "    require_os_containment=True a machine where the child cannot be switched to an unprivileged user (Windows, a non-root Linux\n"
  "    process) FAILS: that is the setting for the Colab run.  On a root POSIX process the OS probe additionally requires that the\n"
  "    child is not root, cannot list or write a mounted /content/drive and cannot read /etc/shadow.  drive_unmounted=True says that the\n"
  "    notebook unmounts Drive while code runs (and verifies it): the two Drive checks are then replaced by ONE information line (info=True,\n"
  "    never a failure, not counted in n_passed) that only reports what the child could see of the mounted Drive, because the unmount is the\n"
  "    barrier that matters for Drive (a mounted Drive that happens to be world-readable must not block the run).  This test runs while\n"
  "    Drive is still mounted: it proves nothing about the unmount, which happens only during 7a / 7b.\"\"\"\n"
  "    out: List[dict] = []\n"
  "\n"
  "    def add(name: str, ok: bool, detail: str = \"\", info: bool = False) -> bool:\n"
  "        # info=True: a line that only REPORTS something (it can never fail and is not counted as a passed check)\n"
  "        out.append({\"name\": name, \"ok\": bool(ok), \"detail\": detail, **({\"info\": True} if info else {})})\n"
  "        return bool(ok)\n"
  "\n"
  "    def finish(acct: Optional[Tuple[int, int, str]]) -> dict:\n"
  "        failed = [c[\"name\"] for c in out if not c[\"ok\"]]\n"
  "        return {\"ok\": not failed, \"backend\": SANDBOX_BACKEND, \"run_as\": None if acct is None else acct[2], \"light\": light, \"checks\": out,\n"
  "                \"failed\": failed, \"n_passed\": sum(1 for c in out if c[\"ok\"] and not c.get(\"info\")), \"n_info\": sum(1 for c in out if c.get(\"info\")),\n"
  "                \"platform\": sys.platform, \"python\": sys.version.split()[0]}\n"
  "\n"
  "    try:\n"
  "        acct = _resolve_run_as(run_as)\n"
  "    except SandboxError as e:\n"
  "        add(\"run_as\", False, str(e))\n"
  "        return finish(None)                    # the other checks would call code_sandbox_run with the same unusable run_as and raise out of here\n"
  "    else:\n"
  "        add(\"run_as\", True, \"no user switch (not root / not POSIX)\" if acct is None else f\"child runs as {acct[2]} (uid {acct[0]})\")\n"
  "    if require_os_containment:\n"
  "        add(\"os_containment_available\", acct is not None, \"the child must run as an unprivileged user\" if acct is None else f\"uid {acct[0]}\")\n"
  "    r = code_sandbox_run(\"print(1 + 1)\\n\", timeout, python=python, run_as=run_as)\n"
  "    add(\"print(1 + 1)\", r[\"ok\"] and r[\"stdout\"].strip() == \"2\", r[\"error\"] or r[\"stdout\"].strip())\n"
  "    r = code_sandbox_run(\"import os\\n\", timeout, python=python, run_as=run_as)\n"
  "    add(\"import os is refused\", (not r[\"ok\"]) and r[\"category\"] == \"static\", r[\"error\"])\n"
  "    r = code_sandbox_run(\"import math, cmath, decimal, fractions, statistics, itertools, functools, operator, collections\\nprint('ok')\\n\", timeout,\n"
  "                         python=python, run_as=run_as)\n"
  "    add(\"every allowed module imports\", r[\"ok\"] and r[\"stdout\"].strip() == \"ok\", r[\"error\"] or r[\"stdout\"].strip())\n"
  "    if not light:\n"
  "        for name, code, expect in (\n"
  "                (\"Fraction\", \"from fractions import Fraction\\nprint(Fraction(1, 3) + Fraction(1, 6))\\n\", \"1/2\"),\n"
  "                (\"Decimal\", \"from decimal import Decimal\\nprint(Decimal('0.1') + Decimal('0.2'))\\n\", \"0.3\"),\n"
  "                (\"statistics\", \"import statistics\\nprint(statistics.mean([1, 2, 3]))\\n\", \"2\"),\n"
  "                (\"collections\", \"import collections\\nprint(collections.Counter('aab').most_common(1))\\n\", \"[('a', 2)]\"),\n"
  "                (\"collections.abc\", \"from collections.abc import Iterable\\nprint(isinstance([1], Iterable))\\n\", \"True\"),\n"
  "                (\"operator\", \"import operator\\nprint(operator.add(1, 2), operator.mul(3, 4))\\n\", \"3 12\"),\n"
  "                (\"functools\", \"from functools import reduce\\nprint(reduce(lambda a, b: a * b, [1, 2, 3, 4]))\\n\", \"24\"),\n"
  "                (\"itertools\", \"import itertools\\nprint(list(itertools.permutations([1, 2, 3]))[0])\\n\", \"(1, 2, 3)\"),\n"
  "                (\"math\", \"import math\\nprint(round(math.sqrt(2), 4), math.gcd(12, 18))\\n\", \"1.4142 6\"),\n"
  "                (\"str.format literal\", \"print('{:>5}|{name}'.format(3, name='x'))\\n\", \"3|x\"),\n"
  "                (\"function and loop\", \"def f(n):\\n    return sum(i * i for i in range(n))\\nprint(f(10))\\n\", \"285\"),\n"
  "                (\"match without a class pattern\", \"match (3, 4):\\n    case (a, b) if a < b:\\n        print(a + b)\\n    case _:\\n        print(0)\\n\", \"7\"),\n"
  "                (\"match with a positional builtin class pattern\", \"match 5:\\n    case int(n):\\n        print(n + 1)\\n\", \"6\")):\n"
  "            r = code_sandbox_run(code, timeout, python=python, run_as=run_as)\n"
  "            add(\"runs: \" + name, r[\"ok\"] and r[\"stdout\"].strip() == expect, r[\"error\"] or r[\"stdout\"].strip())\n"
  "        for name, code in SANDBOX_MUST_REJECT:\n"
  "            r = code_sandbox_run(code, timeout, python=python, run_as=run_as)\n"
  "            add(\"refused: \" + name, (not r[\"ok\"]) and r[\"category\"] == \"static\", r[\"error\"] or (\"RAN: \" + r[\"stdout\"].strip()[:80]))\n"
  "        r = code_sandbox_run(\"import statistics\\nprint(statistics.random)\\n\", timeout, python=python, run_as=run_as)\n"
  "        add(\"module namespaces hide submodules (statistics.random)\", (not r[\"ok\"]) and (\"AttributeError\" in r[\"error\"] or r[\"category\"] == \"static\"),\n"
  "            r[\"error\"] or (\"RAN: \" + r[\"stdout\"].strip()[:80]))\n"
  "        r = code_sandbox_run(\"while True:\\n    pass\\n\", 1.0, python=python, run_as=run_as)\n"
  "        add(\"endless loop is stopped\", (not r[\"ok\"]) and r[\"category\"] in (\"timeout\", \"exception\", \"crash\"), r[\"error\"])\n"
  "        r = code_sandbox_run(\"print('x' * 100000)\\n\", timeout, python=python, run_as=run_as, max_output_chars=1000)\n"
  "        add(\"output is capped\", (not r[\"ok\"]) and r[\"category\"] == \"output_limit\", r[\"error\"])\n"
  "        probe = _spawn(_PROBE_SRC, [], timeout=timeout, flags=[\"-s\", \"-B\", \"-X\", \"utf8\", \"-S\"], python=python, acct=acct)\n"
  "        info: dict = {}\n"
  "        try:\n"
  "            info = json.loads(probe[\"out\"].decode(\"utf-8\", \"replace\").strip().splitlines()[-1])\n"
  "        except (ValueError, IndexError):\n"
  "            pass\n"
  "        if not info:\n"
  "            add(\"os probe ran\", False, probe[\"spawn_error\"] or probe[\"err\"].decode(\"utf-8\", \"replace\")[-200:])\n"
  "        else:\n"
  "            add(\"os probe ran\", True, json.dumps(info))\n"
  "            if acct is not None:\n"
  "                add(\"child is not root\", info.get(\"uid\") not in (0, None), f\"uid {info.get('uid')}\")\n"
  "                sees = str(info.get(\"listdir_drive\", \"\")).startswith(\"ok:\") or str(info.get(\"listdir_mydrive\", \"\")).startswith(\"ok:\")\n"
  "                writes = bool(info.get(\"can_write_mydrive\"))\n"
  "                if drive_unmounted:\n"
  "                    # NOT a proof that Drive is unmounted: 3d runs with Drive mounted; the unmount happens only during 7a / 7b and is enforced\n"
  "                    # by the notebook's _detach (which verifies it).  This line only reports what a child could see of a mounted Drive.\n"
  "                    add(\"info: what the child could see of Drive at 3d time (Drive is unmounted only during 7a/7b, enforced by the notebook _detach)\", True,\n"
  "                        (\"it could list it\" if sees else \"it saw nothing\") + (\"; it could write it\" if writes else \"\")\n"
  "                        + f\" [{info.get('listdir_drive')} / {info.get('listdir_mydrive')}]\", info=True)\n"
  "                else:\n"
  "                    add(\"child cannot list /content/drive\", not sees, f\"{info.get('listdir_drive')} / {info.get('listdir_mydrive')}\")\n"
  "                    add(\"child cannot write Drive\", not writes, str(info.get(\"can_write_mydrive\")))\n"
  "                add(\"child cannot read /etc/shadow\", not str(info.get(\"read_etc_shadow\", \"\")).startswith(\"ok:\"), str(info.get(\"read_etc_shadow\")))\n"
  "    return finish(acct)\n"
  "\n"
  "\n"
  "def code_sandbox_check_many(codes: Sequence[str], timeout: float = 5.0, *, workers: int = 2, progress: Optional[ProgressCB] = None,\n"
  "                            progress_every: int = 200, **kw: Any) -> List[Tuple[bool, str, str]]:\n"
  "    \"\"\"code_sandbox_check over many snippets with `workers` parallel subprocesses (Colab has 2 vCPUs); order is preserved.\"\"\"\n"
  "    tick = _Ticker(progress, \"sandbox\", progress_every)\n"
  "\n"
  "    def one(c: str) -> Tuple[bool, str, str]:\n"
  "        r = code_sandbox_check(c, timeout, **kw)\n"
  "        tick.tick()\n"
  "        return r\n"
  "    with ThreadPoolExecutor(max_workers=max(1, workers)) as ex:\n"
  "        out = list(ex.map(one, codes))\n"
  "    tick.done()\n"
  "    return out\n"
  "\n"
  "\n"
  "_NUM_COMMA = re.compile(r\"^-?\\d{1,3}(,\\d{3})+(\\.\\d+)?$\")\n"
  "\n"
  "\n"
  "def normalize_answer(s: Any) -> str:\n"
  "    s = unicodedata.normalize(\"NFKC\", str(s)).strip().replace(\" \", \"\").replace(\"\\u3000\", \"\").lower()\n"
  "    s = s.rstrip(\"。.\")\n"
  "    return s.replace(\",\", \"\") if _NUM_COMMA.match(s) else s\n"
  "\n"
  "\n"
  "def _as_number(s: str) -> Optional[float]:\n"
  "    try:\n"
  "        if \"/\" in s and s.count(\"/\") == 1:\n"
  "            a, b = s.split(\"/\")\n"
  "            return float(a) / float(b)\n"
  "        return float(s)\n"
  "    except (ValueError, ZeroDivisionError):\n"
  "        return None\n"
  "\n"
  "\n"
  "def answers_equal(output: Any, reference: Any, rel_tol: float = 1e-9, abs_tol: float = 1e-12) -> bool:\n"
  "    \"\"\"Output of the sandbox == reference answer: equal after NFKC / whitespace / trailing '。' / thousands-comma normalisation, or\n"
  "    equal as numbers (int, decimal, a/b fraction) within rel_tol.\"\"\"\n"
  "    a, b = normalize_answer(output), normalize_answer(reference)\n"
  "    if a == b:\n"
  "        return True\n"
  "    x, y = _as_number(a), _as_number(b)\n"
  "    return x is not None and y is not None and math.isclose(x, y, rel_tol=rel_tol, abs_tol=abs_tol)\n"
  "\n"
  "\n"
  "# =====================================================================================================================\n"
  "# The by-eye sample\n"
  "# =====================================================================================================================\n"
  "def sample_for_reading(rows: Iterable[Any], n: int = 100, seed: int = 0, out_path: Optional[str] = None, *,\n"
  "                       text_fn: Optional[Callable[[Any], str]] = None, title: str = \"\", vocab: Any = None,\n"
  "                       max_chars: int = 4000, progress: Optional[ProgressCB] = None) -> List[Tuple[int, Any]]:\n"
  "    \"\"\"Seeded sample of n rows (priority sampling: the n rows with the smallest sha256(seed:position); deterministic across\n"
  "    Python versions and independent of the stream length), returned as [(position, row)] in stream order and, with out_path,\n"
  "    written as a UTF-8 text file for a human: position, id, length, OOV share and characters (when `vocab` is given), prompt and\n"
  "    target (or the JSON of the row).  Refuses to overwrite a different existing file (write_new_file).\"\"\"\n"
  "    text_fn = text_fn or default_text\n"
  "    heap: List[Tuple[int, int, Any]] = []                  # max-heap on priority via negation\n"
  "    total = 0\n"
  "    tick = _Ticker(progress, \"sample_for_reading\", 100000)\n"
  "    for pos, row in enumerate(rows):\n"
  "        total += 1\n"
  "        pri = int.from_bytes(hashlib.sha256(f\"{seed}:{pos}\".encode()).digest()[:8], \"big\")\n"
  "        if len(heap) < n:\n"
  "            heapq.heappush(heap, (-pri, pos, row))\n"
  "        elif pri < -heap[0][0]:\n"
  "            heapq.heapreplace(heap, (-pri, pos, row))\n"
  "        tick.tick()\n"
  "    tick.done()\n"
  "    picked = sorted(((pos, row) for _, pos, row in heap), key=lambda t: t[0])\n"
  "    if out_path:\n"
  "        chars = _chars_of(vocab) if vocab is not None else None\n"
  "        out = [f\"# {title or 'sample'}  seed={seed}  n={len(picked)} of {total} rows (priority sampling; read every row by eye)\\n\"]\n"
  "        for k, (pos, row) in enumerate(picked, 1):\n"
  "            text = text_fn(row)\n"
  "            norm = text.translate(RUNTIME_NORMALIZE)\n"
  "            head = f\"==== [{k}/{len(picked)}] position={pos} id={default_id(row, pos)}  len(prompt+target)+1={row_length(row, None)}\"\n"
  "            if chars is not None:\n"
  "                frac, nb = oov_fraction(norm, chars)\n"
  "                bad = sorted(set(norm) - chars)\n"
  "                head += f\"  oov={frac:.4f}\" + (f\" oov_chars={[f'U+{ord(c):04X}' for c in bad][:12]}\" if bad else \"\")\n"
  "            out.append(head + \" ====\\n\")\n"
  "            if isinstance(row, dict) and isinstance(row.get(\"prompt\"), str) and isinstance(row.get(\"target\"), str):\n"
  "                for key in (\"source\", \"family\", \"split\", \"answer_type\", \"reference_final_answer\", \"license\"):\n"
  "                    if row.get(key) not in (None, \"\"):\n"
  "                        out.append(f\"{key}: {row[key]}\\n\")\n"
  "                out.append(\"--- prompt ---\\n\" + row[\"prompt\"][:max_chars] + \"\\n--- target ---\\n\" + row[\"target\"][:max_chars] + \"\\n\")\n"
  "                steps = row.get(\"tool_steps\")\n"
  "                if steps:\n"
  "                    out.append(f\"--- tool_steps ({len(steps)}), loss_mask_spans={row.get('loss_mask_spans')} ---\\n\")\n"
  "            else:\n"
  "                out.append(json.dumps(row, ensure_ascii=False, indent=1)[:max_chars] + \"\\n\")\n"
  "            out.append(\"\\n\")\n"
  "        write_new_file(out_path, \"\".join(out))\n"
  "    return picked\n"
  "\n"
  "\n"
  "__all__ = [n for n in dir() if not n.startswith(\"_\") and n not in {\n"
  "    \"annotations\", \"array\", \"ast\", \"bisect\", \"collections\", \"dataclasses\", \"gzip\", \"hashlib\", \"heapq\", \"io\", \"itertools\", \"json\", \"math\",\n"
  "    \"operator\", \"os\", \"re\", \"secrets\", \"signal\", \"subprocess\", \"sys\", \"tempfile\", \"time\", \"unicodedata\", \"zipfile\", \"zlib\",\n"
  "    \"ThreadPoolExecutor\", \"Any\", \"Callable\", \"Dict\", \"Iterable\", \"Iterator\", \"List\", \"Mapping\", \"Optional\", \"Sequence\", \"Tuple\", \"Union\"}]\n"
 ),
 "convert.py": (
  "#!/usr/bin/env python3\n"
  "\"\"\"GeoCPU v49 acquisition converters: raw Hugging Face files -> GeoCPU training rows (one converter per dataset).\n"
  "\n"
  "Runs on a Google Colab CPU runtime (12 GB RAM) over files that were downloaded from huggingface.co.  Everything is\n"
  "streaming: a 1 GB JSONL, a 330 MB parquet and a 37 MB JSON array are read record by record.  Nothing here needs a secret.\n"
  "\n"
  "    python convert.py --dataset jaquad|coding|tengentoppa|hotpot|toolace|wizard|all --raw-dir DIR --out-dir DIR [--limit N]\n"
  "                      [--start S] [--vocab base_metadata.json] [--max-chars 1900] [--sample-dump 100] [--no-sandbox]\n"
  "                      [--allow-unverified-python] [--require-os-containment]\n"
  "    (--start S --limit E converts the records with index S <= i < E: a SHARD of a long dataset; merge_shards() joins shards)\n"
  "\n"
  "Row contract = the existing families (see families/jglue_human/build_jglue_rows.py, families/python_a/common.py):\n"
  "    id, source, license, split, training_category, family, structural_template, difficulty, _v46_group,\n"
  "    prompt, target, full_text (== prompt + target exactly), loss_mask_spans, tool_steps, reference_final_answer, answer_type\n"
  "  + attribution, adoption, and family specific extras.  Prompt style \"問題: ...\\\\n条件と根拠を順に考えてください。\",\n"
  "  target \"\\\\n根拠: 「verbatim quote」\\\\n答え: X。\".  A row needs len(full_text)+1 <= 2048; the cap used here is --max-chars (1900).\n"
  "\n"
  "Rules every converter obeys (all re-checked by validate_row() on every emitted row):\n"
  "  * every input record ends as exactly ONE of: an emitted row, or a counted reject with a reason  (n_in == n_out + sum(rejects))\n"
  "  * rows with injected tool observations (python stdout, tool results) list the observation text in loss_mask_spans,\n"
  "    and tool_steps[k][\"observation\"] == full_text[a:b]; such rows must be 100% in-vocabulary and unchanged by the runtime's\n"
  "    symbol normaliser (otherwise the training runtime would discard the mask)\n"
  "  * a row with more than 1 % characters outside the model vocabulary is dropped\n"
  "  * validation / dev / test splits of the sources are never read (file finder + id guards)\n"
  "  * splits are by GROUP (hash of the group key): JaQuAD by article title, HotpotQA by connected component of supporting-paragraph\n"
  "    titles; the four synthetic datasets have NO natural group, so they split by ROW (SPLIT_UNIT says so in every manifest: their\n"
  "    dev / heldout files are not group-independent from train).  heldout 3 %, dev 1 %, train 96 %\n"
  "  * python rows are executed by checks.code_sandbox_check(profile=\"executor\") only (no builtin fallback) and kept only when the answer's\n"
  "    own stored output equals the sandbox output\n"
  "\n"
  "Nothing is fetched here; --raw-dir holds files that were downloaded elsewhere.  Output files are only ever created, never\n"
  "overwritten: if a target exists with a different sha256 the new content is kept under a .new-<sha8> name and the run fails (exit 3).\n"
  "\"\"\"\n"
  "from __future__ import annotations\n"
  "\n"
  "import argparse\n"
  "import ast\n"
  "import atexit\n"
  "import contextlib\n"
  "import hashlib\n"
  "import heapq\n"
  "import itertools\n"
  "import json\n"
  "import os\n"
  "import re\n"
  "import signal\n"
  "import sys\n"
  "import textwrap\n"
  "import unicodedata\n"
  "import zlib\n"
  "from collections import Counter, defaultdict\n"
  "from dataclasses import dataclass, replace as dc_replace\n"
  "from fractions import Fraction\n"
  "from pathlib import Path\n"
  "from typing import Callable, Iterable, Iterator\n"
  "\n"
  "HERE = Path(__file__).resolve().parent\n"
  "\n"
  "MAX_LEN = 2048                    # model context (characters); a row needs len(full_text) + 1 <= MAX_LEN\n"
  "DEFAULT_CAP = 1900                # rows are kept at <= 1,900 characters\n"
  "OOV_MAX_FRAC = 0.01               # rows with more than 1 % characters outside the vocabulary are dropped\n"
  "INSTR = \"条件と根拠を順に考えてください。\"\n"
  "FENCE = chr(96) * 3\n"
  "SPLIT_PCT = {\"heldout\": 3, \"dev\": 1}      # share of GROUPS; the rest (96 %) is train\n"
  "\n"
  "EN_ROW_MAX = 700                  # an English row (instruction or answer prose in English) is kept only up to this many characters\n"
  "# every manifest says what a split unit is, so that nobody reads a row-level dev / heldout file as group-independent\n"
  "_ROW_UNIT = (\"row only: NOT group-independent - siblings generated from the same seed or evolution chain can sit in train and in \"\n"
  "             \"dev / heldout (near duplicates are removed at about Jaccard 0.5 across splits, paraphrase siblings are not)\")\n"
  "SPLIT_UNIT = {\"jaquad\": \"group = article title: every question of one article is in one split\",\n"
  "              \"hotpot\": \"group = connected component of supporting-paragraph titles over ALL train records: questions that share a \"\n"
  "                        \"supporting paragraph are in one split\",\n"
  "              \"coding\": _ROW_UNIT, \"tengentoppa\": _ROW_UNIT, \"toolace\": _ROW_UNIT, \"wizard\": _ROW_UNIT}\n"
  "PY_FENCE_RE = re.compile(r\"(?i)(?:`{3,}|~{3,})[ \\t]*(?:python|py|python3|py3|ipython)\\b\")       # anywhere: also inline, after text on the same line\n"
  "_FENCE_MARK = re.compile(r\"`{3,}|~{3,}\")\n"
  "_TAG_LINE = re.compile(r\"[ \\t]*[A-Za-z0-9_+#.-]{0,15}[ \\t]*\")\n"
  "\n"
  "\n"
  "def has_python_block(text: str) -> bool:\n"
  "    \"\"\"True when `text` holds a fenced block that is (or looks like) Python: a python-tagged fence anywhere (also inline, after text) or an\n"
  "    UNTAGGED fence whose content parses as Python and calls print().  A python block with a stated output cannot be re-run for free text\n"
  "    (tengentoppa, wizardlm): such rows are dropped, not kept as unverified claims about what code prints.\"\"\"\n"
  "    if PY_FENCE_RE.search(text):\n"
  "        return True\n"
  "    for body in _FENCE_MARK.split(text)[1::2]:                   # the pieces between an opening and a closing fence marker\n"
  "        first, _, rest = body.partition(\"\\n\")\n"
  "        if first.strip() and _TAG_LINE.fullmatch(first):          # tagged with some other language: not looked at\n"
  "            continue\n"
  "        try:\n"
  "            tree = ast.parse(textwrap.dedent(rest if not first.strip() else body).strip())\n"
  "        except (SyntaxError, ValueError, RecursionError, MemoryError):\n"
  "            continue\n"
  "        if any(isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id == \"print\" for n in ast.walk(tree)):\n"
  "            return True\n"
  "    return False\n"
  "REQUIRED_FIELDS = (\"id\", \"source\", \"license\", \"split\", \"training_category\", \"family\", \"structural_template\", \"difficulty\",\n"
  "                   \"_v46_group\", \"prompt\", \"target\", \"full_text\", \"loss_mask_spans\", \"tool_steps\",\n"
  "                   \"reference_final_answer\", \"answer_type\", \"attribution\", \"adoption\", \"adoption_note\")\n"
  "\n"
  "\n"
  "# ====================================================================================================== small helpers\n"
  "def h(*parts) -> int:\n"
  "    return int(hashlib.sha256(\"\\x1f\".join(str(p) for p in parts).encode(\"utf-8\")).hexdigest()[:12], 16)\n"
  "\n"
  "\n"
  "def split_of(group_key: str) -> str:\n"
  "    v = h(\"split-v1\", group_key) % 100\n"
  "    if v < SPLIT_PCT[\"heldout\"]:\n"
  "        return \"heldout\"\n"
  "    if v < SPLIT_PCT[\"heldout\"] + SPLIT_PCT[\"dev\"]:\n"
  "        return \"dev\"\n"
  "    return \"train\"\n"
  "\n"
  "\n"
  "def pick(options, *key):\n"
  "    return options[h(*key) % len(options)]\n"
  "\n"
  "\n"
  "def clean_text(s) -> str:\n"
  "    \"\"\"Universal newlines; nothing else is touched (code and quotes must stay verbatim).\"\"\"\n"
  "    return str(s).replace(\"\\r\\n\", \"\\n\").replace(\"\\r\", \"\\n\")\n"
  "\n"
  "\n"
  "def clean_ws(s) -> str:\n"
  "    return \" \".join(str(s).split())\n"
  "\n"
  "\n"
  "_KANA = re.compile(r\"[\\u3040-\\u30ff]\")\n"
  "_HAN = re.compile(r\"[\\u4e00-\\u9fff]\")\n"
  "_LATIN = re.compile(r\"[A-Za-z]\")\n"
  "\n"
  "\n"
  "def detect_lang(s: str) -> str:\n"
  "    kana, han, latin = len(_KANA.findall(s)), len(_HAN.findall(s)), len(_LATIN.findall(s))\n"
  "    letters = kana + han + latin\n"
  "    if letters == 0:\n"
  "        return \"other\"\n"
  "    if kana and kana / letters > 0.03:\n"
  "        return \"ja\"\n"
  "    if han / letters > 0.3:\n"
  "        return \"zh\" if not kana else \"ja\"\n"
  "    if latin / letters > 0.5:\n"
  "        return \"en\"\n"
  "    return \"other\"\n"
  "\n"
  "\n"
  "class Reject(Exception):\n"
  "    \"\"\"One input record cannot become a row; .reason is the (counted) why.\"\"\"\n"
  "\n"
  "    def __init__(self, reason: str):\n"
  "        super().__init__(reason)\n"
  "        self.reason = reason\n"
  "\n"
  "\n"
  "# ====================================================================================================== vocabulary\n"
  "# Same map as qwenputer_controller_v30._BASE_NORM_DICT (+ the full-width ASCII loop): the training runtime applies it to every\n"
  "# row before tokenising; test_convert.py compares this copy with the controller source when that file is reachable.\n"
  "_NORM = {\n"
  "    \"ϵ\": \"ε\", \"ϕ\": \"φ\", \"ϑ\": \"θ\", \"ϰ\": \"κ\", \"ϱ\": \"ρ\", \"ς\": \"σ\", \"ο\": \"o\",\n"
  "    \"Α\": \"A\", \"Β\": \"B\", \"Ε\": \"E\", \"Ζ\": \"Z\", \"Η\": \"H\", \"Ι\": \"I\", \"Κ\": \"K\", \"Μ\": \"M\", \"Ν\": \"N\", \"Ο\": \"O\", \"Ρ\": \"P\",\n"
  "    \"Τ\": \"T\", \"Υ\": \"Y\", \"Χ\": \"X\", \"＞\": \">\", \"＜\": \"<\", \"⇔\": \"↔\", \"↦\": \"→\", \"⊨\": \"⊢\", \"⊤\": \"T\", \"∅\": \"φ\", \"≢\": \"≠\",\n"
  "    \"÷\": \"/\", \"⋅\": \"・\", \"∙\": \"・\", \"⟨\": \"(\", \"⟩\": \")\", \"⟪\": \"(\", \"⟫\": \")\", \"⟦\": \"[\", \"⟧\": \"]\", \"⌈\": \"[\", \"⌉\": \"]\",\n"
  "    \"⌊\": \"[\", \"⌋\": \"]\", \"−\": \"-\",\n"
  "}\n"
  "for _c in range(0xFF01, 0xFF5F):\n"
  "    if chr(_c) not in (\"（\", \"）\", \"！\", \"？\", \"：\", \"～\"):\n"
  "        _NORM[chr(_c)] = chr(_c - 0xFEE0)\n"
  "NORMALIZE = str.maketrans(_NORM)\n"
  "\n"
  "\n"
  "def load_vocab(path=None, drive_root=None):\n"
  "    \"\"\"(frozenset of single characters, source).  Search order: explicit path, $GEOCPU_VOCAB, next to this file,\n"
  "    src/v44fix/base_metadata.json in any parent directory, then checks.load_vocab(drive_root) (finds base_metadata.json inside the\n"
  "    bundle zip on Drive; never derives a vocabulary from the corpus).  base_metadata.json[\"vocab\"] maps token -> id.\"\"\"\n"
  "    cands = []\n"
  "    if path:\n"
  "        cands.append(Path(path))\n"
  "    if os.environ.get(\"GEOCPU_VOCAB\"):\n"
  "        cands.append(Path(os.environ[\"GEOCPU_VOCAB\"]))\n"
  "    cands.append(HERE / \"base_metadata.json\")\n"
  "    cands.extend(up / \"src\" / \"v44fix\" / \"base_metadata.json\" for up in HERE.parents)\n"
  "    for p in cands:\n"
  "        if p.is_file():\n"
  "            data = json.loads(p.read_text(encoding=\"utf-8\"))\n"
  "            v = data[\"vocab\"] if isinstance(data, dict) and \"vocab\" in data else data\n"
  "            keys = v.keys() if isinstance(v, dict) else v\n"
  "            return frozenset(k for k in keys if isinstance(k, str) and len(k) == 1), str(p)\n"
  "    ext = _load_external_checks()\n"
  "    if ext is not None and callable(getattr(ext, \"load_vocab\", None)):\n"
  "        try:\n"
  "            root = drive_root or os.environ.get(\"GEOCPU_DRIVE_ROOT\") or getattr(ext, \"DRIVE_ROOT\", \"\")\n"
  "            info = ext.load_vocab(root, allow_derive=False)\n"
  "            if len(info.chars) >= 1000 and not getattr(info, \"derived\", False):\n"
  "                return frozenset(info.chars), \"checks.load_vocab:\" + str(info.source)\n"
  "        except Exception:                                        # noqa: BLE001\n"
  "            pass\n"
  "    raise FileNotFoundError(\"model vocabulary not found: pass --vocab base_metadata.json (src/v44fix/base_metadata.json) or --drive-root\")\n"
  "\n"
  "\n"
  "# ====================================================================================================== checks (checks.py is REQUIRED to run code)\n"
  "# There is no builtin sandbox any more.  An earlier version fell back to a plain-script runner (full builtins, no audit hook) whenever\n"
  "# checks.code_sandbox_check failed its start-up test; review showed that fallback could write files and that nothing noticed which\n"
  "# backend was used.  Now python rows are executed ONLY by checks.code_sandbox_check(profile=\"executor\") - the rules of the model's own\n"
  "# executor - and Checks() raises SandboxUnavailable when that sandbox is missing or fails its self-test.  --no-sandbox (run_code=False)\n"
  "# converts everything except the python rows, which are then rejected.\n"
  "def _load_external_checks():\n"
  "    try:\n"
  "        if str(HERE) not in sys.path:\n"
  "            sys.path.insert(0, str(HERE))\n"
  "        import checks as ext                                     # sibling module written by the pipeline's checks stage\n"
  "        return ext\n"
  "    except Exception:\n"
  "        return None\n"
  "\n"
  "\n"
  "class SandboxUnavailable(RuntimeError):\n"
  "    \"\"\"python rows cannot be verified here: the executor-profile sandbox of checks.py is missing or failed its self-test (at start-up), or it\n"
  "    BROKE in the middle of the run (it could not start a child, the child died without a result, the unprivileged user vanished).\n"
  "    There is no weaker fallback, and a broken sandbox is never turned into per-row rejects: the converter stops (exit code 4).\"\"\"\n"
  "\n"
  "\n"
  "# What code_sandbox_check reports as 'crash' / 'crash:spawn:...' is a failure of the SANDBOX (the child could not be started, or ended\n"
  "# without a result line), not a verdict about the code under test.  Everything else is the code's own outcome and stays a per-row reject:\n"
  "# static:* (refused by the rules), timeout:*, output_limit:*, exception:* (wrong code, MemoryError at the memory limit, ...).\n"
  "SANDBOX_INFRA_CATEGORIES = (\"crash\", \"spawn\")\n"
  "\n"
  "\n"
  "def sandbox_reason_is_infrastructure(reason: str) -> bool:\n"
  "    \"\"\"True for a code_sandbox_check error string that says the sandbox itself failed ('crash:spawn:OSError: [Errno 28] ...', 'crash:returncode=...').\"\"\"\n"
  "    return str(reason).split(\":\", 1)[0] in SANDBOX_INFRA_CATEGORIES\n"
  "\n"
  "\n"
  "class Checks:\n"
  "    \"\"\"Vocabulary / OOV / sandbox services.  Python is run through checks.code_sandbox_check(profile=\"executor\") after\n"
  "    checks.sandbox_selftest(light=True) passed (otherwise SandboxUnavailable is raised when run_code=True).  OOV is always measured\n"
  "    per row here with the runtime normaliser (checks.oov_stats is a corpus-level report over rows: run it over the output files).\n"
  "    The backend actually used is recorded in .backends (and in every summary): backends[\"sandbox\"] is\n"
  "    'checks.code_sandbox_check(profile=executor)' or 'disabled'; backends[\"sandbox_run_as\"] is the OS account of the child ('nobody' on\n"
  "    Colab).  python_require_stored_output (default True) keeps a python row only when the answer's OWN stored output block equals the\n"
  "    sandbox stdout - the owner's rule 'keep only rows whose output equals the reference answer'; rows that run but have nothing to\n"
  "    compare with are counted (python_sandbox_ok_no_stored_output) and dropped, never kept as 'verified'.\"\"\"\n"
  "\n"
  "    def __init__(self, vocab=None, vocab_path=None, max_chars=DEFAULT_CAP, run_code=True, sandbox_timeout=5.0, use_external=True,\n"
  "                 python_require_stored_output=True, tengentoppa_steps=True, drive_root=None, require_os_containment=False,\n"
  "                 en_row_max=EN_ROW_MAX):\n"
  "        if vocab is None:\n"
  "            vocab, vocab_path = load_vocab(vocab_path, drive_root)\n"
  "        self.vocab = frozenset(vocab)\n"
  "        self.vocab_path = vocab_path\n"
  "        self.max_chars = min(int(max_chars), MAX_LEN - 1)\n"
  "        self.run_code = run_code\n"
  "        self.sandbox_timeout = sandbox_timeout\n"
  "        self.python_require_stored_output = python_require_stored_output\n"
  "        self.tengentoppa_steps = tengentoppa_steps                        # include the 'reasoning' steps when the row fits\n"
  "        self.hotpot_roots = None                                          # title -> union-find root (set by run_dataset for hotpot)\n"
  "        self.en_row_max = int(en_row_max)\n"
  "        self.backends = {\"oov\": \"builtin\", \"sandbox\": \"disabled\", \"sandbox_run_as\": None, \"static_check\": \"unavailable\"}\n"
  "        self._ext = _load_external_checks() if use_external else None\n"
  "        ext = self._ext\n"
  "        if ext is not None and callable(getattr(ext, \"static_check\", None)):\n"
  "            self.backends[\"static_check\"] = \"checks.static_check(profile=executor)\"\n"
  "        if run_code:\n"
  "            need = (\"code_sandbox_check\", \"static_check\", \"sandbox_selftest\", \"SANDBOX_BACKEND\")\n"
  "            if ext is None or not all(hasattr(ext, n) for n in need):\n"
  "                raise SandboxUnavailable(\"running python rows needs the sibling checks.py sandbox (use_external=True and checks.py next to \"\n"
  "                                         \"convert.py); there is no builtin fallback.  Pass run_code=False / --no-sandbox to convert without \"\n"
  "                                         \"executing code\")\n"
  "            st = ext.sandbox_selftest(light=True, require_os_containment=require_os_containment, timeout=max(15.0, 3 * sandbox_timeout))\n"
  "            if not st[\"ok\"]:\n"
  "                raise SandboxUnavailable(\"the sandbox self-test failed, so no code is run: \" +\n"
  "                                         \"; \".join(f\"{c['name']}: {c['detail']}\" for c in st[\"checks\"] if not c[\"ok\"]))\n"
  "            self.backends[\"sandbox\"] = ext.SANDBOX_BACKEND\n"
  "            self.backends[\"sandbox_run_as\"] = st[\"run_as\"]\n"
  "        if ext is not None and getattr(ext, \"RUNTIME_NORMALIZE\", None) == NORMALIZE:\n"
  "            self.backends[\"oov\"] = \"builtin per-row (same runtime normaliser as checks.RUNTIME_NORMALIZE; checks.oov_stats is corpus level)\"\n"
  "\n"
  "    # --- vocabulary\n"
  "    def oov_list(self, text: str):\n"
  "        return [c for c in text.translate(NORMALIZE) if c not in self.vocab]\n"
  "\n"
  "    def _oov(self, text: str):\n"
  "        t = text.translate(NORMALIZE)                             # 1:1 character map, so len() is unchanged\n"
  "        bad = set(t) - self.vocab                                 # set arithmetic runs at C speed; counting only when something is OOV\n"
  "        return (sum(t.count(c) for c in bad) if bad else 0), bad\n"
  "\n"
  "    def oov_stats(self, text: str) -> dict:\n"
  "        n_oov, bad = self._oov(text)\n"
  "        return {\"n_chars\": len(text), \"n_oov\": n_oov, \"frac\": n_oov / max(1, len(text)), \"bad\": sorted(bad)[:20]}\n"
  "\n"
  "    def oov_count(self, text: str) -> int:\n"
  "        return self._oov(text)[0]\n"
  "\n"
  "    def normalizer_changes(self, text: str) -> bool:\n"
  "        return text.translate(NORMALIZE) != text\n"
  "\n"
  "    # --- code\n"
  "    def static_check(self, code: str):\n"
  "        \"\"\"(ok, reason): the executor profile's static rules (checks.static_check).\"\"\"\n"
  "        if self._ext is None:\n"
  "            return False, \"checks_unavailable\"\n"
  "        why = self._ext.static_check(code, profile=\"executor\")\n"
  "        return (not why), why\n"
  "\n"
  "    def run_python(self, code: str) -> dict:\n"
  "        if not self.run_code:\n"
  "            return {\"ok\": False, \"stdout\": \"\", \"reason\": \"sandbox_disabled\"}\n"
  "        try:\n"
  "            ok, out, err = self._ext.code_sandbox_check(code, self.sandbox_timeout, profile=\"executor\")\n"
  "        except Exception as e:                                   # noqa: BLE001  SandboxError (the unprivileged user vanished), OSError (no space in /tmp), ...\n"
  "            # The code under test cannot raise out of code_sandbox_check (its own exceptions are caught in the child), so this is the SANDBOX\n"
  "            # failing.  Turning it into one more per-row reject would let a long run finish 'balanced' with every verified python row gone.\n"
  "            raise SandboxUnavailable(f\"the sandbox failed in the middle of the run ({type(e).__name__}: {str(e)[:300]}); \"\n"
  "                                     f\"no more code is run and the converter stops\") from e\n"
  "        if not ok and sandbox_reason_is_infrastructure(err):\n"
  "            raise SandboxUnavailable(f\"the sandbox failed in the middle of the run ({str(err)[:300]}); no more code is run and the converter stops\")\n"
  "        return {\"ok\": bool(ok), \"stdout\": str(out), \"reason\": str(err)}\n"
  "\n"
  "\n"
  "# ====================================================================================================== rows, stats, validation\n"
  "@dataclass\n"
  "class Spec:\n"
  "    key: str\n"
  "    hf: str\n"
  "    license: str\n"
  "    source: str\n"
  "    attribution: str\n"
  "    adoption: str\n"
  "    adoption_note: str\n"
  "    find: Callable\n"
  "    reader: Callable\n"
  "    convert: Callable\n"
  "\n"
  "\n"
  "class Stats:\n"
  "    \"\"\"Per-dataset counters.  Invariant: n_in == n_out + sum(rejects).\"\"\"\n"
  "\n"
  "    MAX_EXAMPLES = 5\n"
  "\n"
  "    def __init__(self, key: str):\n"
  "        self.key = key\n"
  "        self.n_in = 0\n"
  "        self.n_out = 0\n"
  "        self.rejects: Counter = Counter()\n"
  "        self.families: Counter = Counter()\n"
  "        self.splits: Counter = Counter()\n"
  "        self.family_splits: dict = defaultdict(Counter)\n"
  "        self.extra: Counter = Counter()\n"
  "        self.reject_examples: dict = {}\n"
  "\n"
  "    def reject(self, reason: str, ident: str = \"\", rec=None):\n"
  "        self.rejects[reason] += 1\n"
  "        ex = self.reject_examples.setdefault(reason, []) if (reason in self.reject_examples or len(self.reject_examples) < 200) else None\n"
  "        if ex is not None and len(ex) < self.MAX_EXAMPLES:\n"
  "            ex.append({\"id\": ident, \"excerpt\": _excerpt(rec)})\n"
  "\n"
  "    ACCEPT_FIELDS = ((\"verification\", \"verification_\"), (\"code_lang\", \"code_text_lang_\"), (\"answer_lang\", \"code_text_answer_lang_\"),\n"
  "                     (\"trim_level\", \"trim_level_\"), (\"user_lang\", \"accepted_user_lang_\"), (\"answer_format\", \"answer_format_\"),\n"
  "                     (\"has_python_block\", \"has_python_block_\"))\n"
  "\n"
  "    def accept(self, row: dict):\n"
  "        self.n_out += 1\n"
  "        self.families[row[\"family\"]] += 1\n"
  "        self.splits[row[\"split\"]] += 1\n"
  "        self.family_splits[row[\"family\"]][row[\"split\"]] += 1\n"
  "        for field, prefix in self.ACCEPT_FIELDS:                  # counters over EMITTED rows only\n"
  "            if field in row:\n"
  "                self.extra[prefix + str(row[field])] += 1\n"
  "        fam = row[\"family\"]\n"
  "        sup = len(row[\"target\"]) - sum(b - a for a, b in row[\"loss_mask_spans\"])           # characters the loss is computed on\n"
  "        self.extra[\"supervised_chars_\" + fam] += sup\n"
  "        if row.get(\"user_lang\") == \"en\" or row.get(\"answer_lang\") == \"en\" or str(row.get(\"language\", \"\")).startswith(\"en\"):\n"
  "            self.extra[\"english_rows_\" + fam] += 1\n"
  "            self.extra[\"english_supervised_chars_\" + fam] += sup\n"
  "\n"
  "\n"
  "def build_row(*, id, family, template, group, category, prompt, target, ref, answer_type, difficulty, spec: Spec, split,\n"
  "              spans=None, steps=None, **extra) -> dict:\n"
  "    row = {\n"
  "        \"id\": id, \"source\": spec.source, \"license\": spec.license, \"split\": split, \"training_category\": category,\n"
  "        \"family\": family, \"structural_template\": template, \"difficulty\": difficulty, \"_v46_group\": group,\n"
  "        \"prompt\": prompt, \"target\": target, \"full_text\": prompt + target,\n"
  "        \"loss_mask_spans\": [list(s) for s in (spans or [])], \"tool_steps\": steps or [],\n"
  "        \"reference_final_answer\": ref, \"answer_type\": answer_type, \"attribution\": spec.attribution,\n"
  "        \"adoption\": spec.adoption, \"adoption_note\": spec.adoption_note,\n"
  "    }\n"
  "    row.update(extra)\n"
  "    return row\n"
  "\n"
  "\n"
  "def validate_row(row: dict, ck: Checks):\n"
  "    \"\"\"None if the row is a valid GeoCPU row, else a short reason.\"\"\"\n"
  "    for k in REQUIRED_FIELDS:\n"
  "        if k not in row:\n"
  "            return \"missing:\" + k\n"
  "    p, t, f = row[\"prompt\"], row[\"target\"], row[\"full_text\"]\n"
  "    if not all(isinstance(x, str) for x in (p, t, f)) or not p or not t:\n"
  "        return \"bad_text\"\n"
  "    if f != p + t:\n"
  "        return \"full_text_mismatch\"\n"
  "    if len(f) > ck.max_chars:\n"
  "        return \"too_long\"\n"
  "    if len(f) + 1 > MAX_LEN:\n"
  "        return \"too_long_ctx\"\n"
  "    if any(0xD800 <= ord(c) <= 0xDFFF for c in f):\n"
  "        return \"surrogate\"\n"
  "    if row[\"split\"] not in (\"train\", \"dev\", \"heldout\"):\n"
  "        return \"bad_split\"\n"
  "    spans, steps = row[\"loss_mask_spans\"], row[\"tool_steps\"]\n"
  "    prev = len(p)\n"
  "    for s in spans:\n"
  "        if not (isinstance(s, (list, tuple)) and len(s) == 2 and all(isinstance(x, int) for x in s)):\n"
  "            return \"bad_span\"\n"
  "        a, b = s\n"
  "        if not (prev <= a < b <= len(f)):\n"
  "            return \"span_out_of_order_or_range\"\n"
  "        prev = b\n"
  "    if spans:\n"
  "        if len(steps) != len(spans) or any(f[a:b] != st.get(\"observation\") for (a, b), st in zip(spans, steps)):\n"
  "            return \"mask_does_not_cover_observation\"\n"
  "        if ck.normalizer_changes(f) or ck.oov_count(f):\n"
  "            return \"masked_row_not_clean\"\n"
  "    if ck.oov_stats(f)[\"frac\"] > OOV_MAX_FRAC:\n"
  "        return \"oov_frac\"\n"
  "    if ck.oov_count(t):                                              # the loader DELETES every out-of-vocabulary character: in the prompt (context) up to\n"
  "        return \"oov_in_target\"                                       # 1 % is tolerated, in the target it would silently corrupt what the model learns to write\n"
  "    for q in row.get(\"quotes\", []) or []:\n"
  "        if q not in p:\n"
  "            return \"quote_not_in_prompt\"\n"
  "    return None\n"
  "\n"
  "\n"
  "def run_converter(records: Iterable, stats: Stats, fn: Callable, ck: Checks) -> Iterator[dict]:\n"
  "    \"\"\"fn(idx, rec) -> row | raises Reject.  Every record ends as a validated row or one counted reject.\"\"\"\n"
  "    for idx, rec in records:\n"
  "        stats.n_in += 1\n"
  "        ident = f\"{stats.key}:{idx}\"\n"
  "        try:\n"
  "            if not isinstance(rec, dict):\n"
  "                raise Reject(\"not_an_object\")\n"
  "            row = fn(idx, rec)\n"
  "            why = validate_row(row, ck)\n"
  "            if why:\n"
  "                raise Reject(\"invalid_row:\" + why)\n"
  "            row[\"n_oov\"] = ck.oov_count(row[\"full_text\"])\n"
  "        except Reject as r:\n"
  "            stats.reject(r.reason, ident, rec)\n"
  "            continue\n"
  "        except SandboxUnavailable:\n"
  "            raise                                                # the sandbox broke: that is not one odd record (see Checks.run_python)\n"
  "        except Exception as e:                                   # noqa: BLE001  one odd record must not kill a 1 GB run\n"
  "            stats.reject(\"exception:\" + type(e).__name__, ident, rec)\n"
  "            continue\n"
  "        stats.accept(row)\n"
  "        yield row\n"
  "\n"
  "\n"
  "def _excerpt(rec) -> str:\n"
  "    try:\n"
  "        return json.dumps(rec, ensure_ascii=False)[:160]\n"
  "    except Exception:                                            # noqa: BLE001\n"
  "        return repr(rec)[:160]\n"
  "\n"
  "\n"
  "# ====================================================================================================== streaming readers\n"
  "def iter_jsonl(path) -> Iterator[dict]:\n"
  "    with open(path, encoding=\"utf-8\", errors=\"replace\", newline=\"\\n\") as f:\n"
  "        for line in f:\n"
  "            line = line.strip()\n"
  "            if line:\n"
  "                try:\n"
  "                    obj = json.loads(line)\n"
  "                except ValueError:\n"
  "                    obj = {\"__bad_json__\": line[:80]}\n"
  "                yield obj\n"
  "\n"
  "\n"
  "def iter_json_array(path, chunk: int = 1 << 20) -> Iterator[dict]:\n"
  "    \"\"\"Objects of a top-level JSON array, one at a time (falls back to JSONL when the file starts with '{').\"\"\"\n"
  "    dec = json.JSONDecoder()\n"
  "    with open(path, encoding=\"utf-8\", errors=\"replace\") as f:\n"
  "        first = f.read(1)\n"
  "        while first and first.isspace():\n"
  "            first = f.read(1)\n"
  "        if not first:\n"
  "            return\n"
  "        if first == \"{\":\n"
  "            f.seek(0)\n"
  "            yield from iter_jsonl(path)\n"
  "            return\n"
  "        if first != \"[\":\n"
  "            raise ValueError(\"not a JSON array: \" + str(path))\n"
  "        buf, i, eof = \"\", 0, False\n"
  "        while True:\n"
  "            while True:\n"
  "                while i < len(buf) and (buf[i].isspace() or buf[i] == \",\"):\n"
  "                    i += 1\n"
  "                if i < len(buf):\n"
  "                    break\n"
  "                if eof:\n"
  "                    return\n"
  "                data = f.read(chunk)\n"
  "                buf, i = data, 0\n"
  "                eof = not data\n"
  "            if buf[i] == \"]\":\n"
  "                return\n"
  "            while True:\n"
  "                try:\n"
  "                    obj, j = dec.raw_decode(buf, i)\n"
  "                    break\n"
  "                except json.JSONDecodeError:\n"
  "                    if eof:\n"
  "                        raise\n"
  "                    data = f.read(chunk)\n"
  "                    if not data:\n"
  "                        eof = True\n"
  "                    buf, i = buf[i:] + data, 0\n"
  "            yield obj\n"
  "            i = j\n"
  "\n"
  "\n"
  "def iter_parquet(path, columns=None, batch_size: int = 256) -> Iterator[dict]:\n"
  "    import pyarrow.parquet as pq                                 # present on Colab; pip install -q pyarrow otherwise\n"
  "    pf = pq.ParquetFile(str(path))\n"
  "    for batch in pf.iter_batches(batch_size=batch_size, columns=columns):\n"
  "        yield from batch.to_pylist()\n"
  "\n"
  "\n"
  "def _find(raw_dir, patterns, root_only=()):\n"
  "    root = Path(raw_dir)\n"
  "    found = set()\n"
  "    for pat in patterns:\n"
  "        found.update(p for p in root.glob(pat) if p.is_file())\n"
  "    out = []\n"
  "    for p in sorted(found):\n"
  "        rel = \"/\".join(p.relative_to(root).parts).lower()\n"
  "        if any(part.startswith(\"do_not_train\") for part in p.relative_to(root).parts):\n"
  "            continue                                              # acquire.py keeps dev / validation files there\n"
  "        if re.search(r\"(?:^|[/_.-])(dev|validation|val|test)(?:[/_.-]|$)\", rel):\n"
  "            continue                                              # never train on validation / dev / test files\n"
  "        out.append(p)\n"
  "    return out\n"
  "\n"
  "\n"
  "def _limited(it: Iterable, limit, start: int = 0):\n"
  "    \"\"\"(index, record) for start <= index < limit (limit None = to the end); indices are global (they keep counting over files).\"\"\"\n"
  "    return itertools.islice(enumerate(it), start or 0, limit)\n"
  "\n"
  "\n"
  "def find_jaquad(d):\n"
  "    return _find(d, [\"**/jaquad_train_*.json\"])\n"
  "\n"
  "\n"
  "def find_coding(d):\n"
  "    return _find(d, [\"**/train-*.parquet\"])\n"
  "\n"
  "\n"
  "def find_tengentoppa(d):\n"
  "    return _find(d, [\"**/completed_sets.jsonl\"])\n"
  "\n"
  "\n"
  "def find_hotpot(d):\n"
  "    files = [p for p in _find(d, [\"**/distractor/train-*.parquet\"])]\n"
  "    if not files:                                                 # flat folder: accept only if it is clearly not fullwiki\n"
  "        files = [p for p in _find(d, [\"train-*.parquet\", \"**/train-*.parquet\"]) if \"fullwiki\" not in str(p).lower()]\n"
  "    return files\n"
  "\n"
  "\n"
  "def find_toolace(d):\n"
  "    return _find(d, [\"**/data.json\"])\n"
  "\n"
  "\n"
  "def find_wizard(d):\n"
  "    return _find(d, [\"**/train-*.parquet\"])\n"
  "\n"
  "\n"
  "def read_jaquad(files, limit=None, start=0):\n"
  "    n = 0\n"
  "    for fp in files:\n"
  "        d = json.loads(Path(fp).read_text(encoding=\"utf-8\"))\n"
  "        for art in d.get(\"data\", []):\n"
  "            title = str(art.get(\"title\", \"\")).strip()\n"
  "            for para in art.get(\"paragraphs\", []):\n"
  "                ctx = para.get(\"context\", \"\")\n"
  "                for qa in para.get(\"qas\", []):\n"
  "                    if limit is not None and n >= limit:\n"
  "                        return\n"
  "                    if n >= start:\n"
  "                        yield n, {\"id\": qa.get(\"id\"), \"title\": title, \"context\": ctx, \"question\": qa.get(\"question\", \"\"),\n"
  "                                  \"question_type\": qa.get(\"question_type\", \"\"), \"answers\": qa.get(\"answers\", [])}\n"
  "                    n += 1\n"
  "\n"
  "\n"
  "def _chain(files, reader):\n"
  "    for fp in files:\n"
  "        yield from reader(fp)\n"
  "\n"
  "\n"
  "def read_messages_parquet(files, limit=None, start=0):\n"
  "    return _limited(_chain(files, lambda p: iter_parquet(p, columns=[\"messages\"])), limit, start)\n"
  "\n"
  "\n"
  "def read_tengentoppa(files, limit=None, start=0):\n"
  "    return _limited(_chain(files, iter_jsonl), limit, start)\n"
  "\n"
  "\n"
  "def read_hotpot(files, limit=None, start=0):\n"
  "    cols = [\"id\", \"question\", \"answer\", \"type\", \"level\", \"supporting_facts\", \"context\"]\n"
  "    return _limited(_chain(files, lambda p: iter_parquet(p, columns=cols)), limit, start)\n"
  "\n"
  "\n"
  "def read_toolace(files, limit=None, start=0):\n"
  "    return _limited(_chain(files, iter_json_array), limit, start)\n"
  "\n"
  "\n"
  "def build_hotpot_roots(files) -> dict:\n"
  "    \"\"\"{supporting-paragraph title: root title} where two titles are joined whenever one question uses both: the connected components\n"
  "    over ALL train records (one cheap pass over the supporting_facts column).  Two questions that share a supporting paragraph then get\n"
  "    the same group key and so the same split; the root is the smallest title of its component, so the key does not depend on file order.\"\"\"\n"
  "    parent: dict = {}\n"
  "\n"
  "    def find(x):\n"
  "        while parent[x] != x:\n"
  "            parent[x] = parent[parent[x]]\n"
  "            x = parent[x]\n"
  "        return x\n"
  "    for fp in files:\n"
  "        for rec in iter_parquet(fp, columns=[\"supporting_facts\"]):\n"
  "            sf = rec.get(\"supporting_facts\") or {}\n"
  "            titles = sorted({str(t) for t in (sf.get(\"title\") or [])})\n"
  "            for t in titles:\n"
  "                parent.setdefault(t, t)\n"
  "            for t in titles[1:]:\n"
  "                a, b = find(titles[0]), find(t)\n"
  "                if a != b:\n"
  "                    parent[max(a, b)] = min(a, b)\n"
  "    return {t: find(t) for t in parent}\n"
  "\n"
  "\n"
  "# ====================================================================================================== (a) JaQuAD\n"
  "JAQUAD_MAX_PASSAGE = 700\n"
  "PASSAGE_HEAD = [\"次の文章を読んで質問に答えてください。\", \"文章を読んで、文章に書かれていることだけで質問に答えてください。\",\n"
  "                \"以下の文章を読んで質問に答えてください。\"]\n"
  "\n"
  "\n"
  "def sentence_spans(body: str):\n"
  "    \"\"\"[(start, end)]: a sentence ends after '。' (plus closing brackets) or at a line break; blank pieces are dropped.\"\"\"\n"
  "    spans, s = [], 0\n"
  "    for m in re.finditer(r\"。[」』）)]*|\\n\", body):\n"
  "        spans.append((s, m.end()))\n"
  "        s = m.end()\n"
  "    if s < len(body):\n"
  "        spans.append((s, len(body)))\n"
  "    return [(a, b) for a, b in spans if body[a:b].strip()]\n"
  "\n"
  "\n"
  "_INTERROGATIVE = re.compile(r\"(?:ですか|でしょうか|ますか|は何|を何|が何|とは|何年|何人|何番目|何回|いつ|どこ|だれ|誰|どの|どれ|どのように|なぜ|何)\")\n"
  "_CONTENT_CHAR = re.compile(r\"[\\u4e00-\\u9fff\\u30a0-\\u30ffA-Za-z0-9]\")\n"
  "\n"
  "\n"
  "def _cmp_norm(s: str) -> str:\n"
  "    return re.sub(r\"\\s+\", \"\", unicodedata.normalize(\"NFKC\", s)).casefold()\n"
  "\n"
  "\n"
  "def shares_content(question: str, quote: str, n: int = 3) -> bool:\n"
  "    \"\"\"True when the question and the quote share a character n-gram (default 3) that carries content (a kanji, katakana, latin letter\n"
  "    or digit), after the interrogative words (何 / どこ / ですか ...) are cut out of the question.  A quote that shares nothing with the\n"
  "    question does not support it - the JaQuAD card's own 'Multiple sentence reasoning' example (the question names ビワタナゴ, the\n"
  "    evidence sentence never does) is the case this catches.\"\"\"\n"
  "    q = _INTERROGATIVE.sub(\"\\n\", _cmp_norm(question).replace(\"?\", \"\\n\").replace(\"？\", \"\\n\"))\n"
  "    t = _cmp_norm(quote)\n"
  "    for frag in q.split(\"\\n\"):\n"
  "        for i in range(len(frag) - n + 1):\n"
  "            g = frag[i:i + n]\n"
  "            if g in t and _CONTENT_CHAR.search(g):\n"
  "                return True\n"
  "    return False\n"
  "\n"
  "\n"
  "def _answers_list(a):\n"
  "    \"\"\"JaQuAD raw: [{text, answer_start, answer_type}]; datasets-server view: {text: [..], answer_start: [..], answer_type: [..]}.\"\"\"\n"
  "    if isinstance(a, dict) and isinstance(a.get(\"text\"), list):\n"
  "        return [{\"text\": t, \"answer_start\": s, \"answer_type\": (a.get(\"answer_type\") or [\"\"] * len(a[\"text\"]))[i]}\n"
  "                for i, (t, s) in enumerate(zip(a[\"text\"], a[\"answer_start\"]))]\n"
  "    if isinstance(a, dict):\n"
  "        return [a]\n"
  "    return list(a or [])\n"
  "\n"
  "\n"
  "def _jaquad_one(idx, rec, st: Stats, ck: Checks, spec: Spec):\n"
  "    qid = str(rec.get(\"id\") or \"\")\n"
  "    if qid[:3] in (\"de-\", \"te-\"):\n"
  "        raise Reject(\"not_train_id\")                             # dev / test ids never enter training\n"
  "    title = str(rec.get(\"title\") or \"\").strip()\n"
  "    ctx = str(rec.get(\"context\") or \"\")\n"
  "    q = clean_ws(rec.get(\"question\") or \"\")\n"
  "    answers = _answers_list(rec.get(\"answers\"))\n"
  "    if not qid or not title or not ctx or not q:\n"
  "        raise Reject(\"missing_field\")\n"
  "    if not answers:\n"
  "        raise Reject(\"no_answer\")\n"
  "    if \"\\r\" in ctx:\n"
  "        raise Reject(\"context_has_cr\")\n"
  "    a = answers[0]\n"
  "    raw_text, start = str(a.get(\"text\", \"\")), int(a.get(\"answer_start\", -1))\n"
  "    if not raw_text.strip() or start < 0:\n"
  "        raise Reject(\"empty_answer\")\n"
  "    if ctx[start:start + len(raw_text)] != raw_text:\n"
  "        raise Reject(\"offset_mismatch\")\n"
  "    lead = len(ctx) - len(ctx.lstrip())\n"
  "    body = ctx.strip()\n"
  "    ans = raw_text.strip()\n"
  "    s0 = start - lead + (len(raw_text) - len(raw_text.lstrip()))\n"
  "    e0 = s0 + len(ans)\n"
  "    if s0 < 0 or e0 > len(body):\n"
  "        raise Reject(\"answer_outside_body\")\n"
  "    if \"\\n\" in ans:\n"
  "        raise Reject(\"answer_has_newline\")\n"
  "    if len(body) > JAQUAD_MAX_PASSAGE:\n"
  "        raise Reject(\"passage_too_long\")\n"
  "    hit = [sp for sp in sentence_spans(body) if sp[0] <= s0 and e0 <= sp[1]]\n"
  "    if len(hit) != 1:\n"
  "        raise Reject(\"answer_spans_sentences\")\n"
  "    quote = body[hit[0][0]:hit[0][1]].strip()\n"
  "    if ans not in quote or len(quote) < 4 or \"\\n\" in quote:\n"
  "        raise Reject(\"quote_check\")\n"
  "    qtype = str(rec.get(\"question_type\") or \"\")\n"
  "    st.extra[\"question_type_seen_\" + (qtype[:40] or \"(none)\")] += 1                   # distribution for the report\n"
  "    if re.search(r\"(?i)multi\", qtype):\n"
  "        raise Reject(\"question_type_multiple_sentence\")           # the evidence is not (only) the one sentence that holds the span\n"
  "    if re.search(r\"(?i)ambiguous\", qtype):\n"
  "        raise Reject(\"question_type_ambiguous\")                   # the annotators themselves marked the question as not clearly answerable\n"
  "    if not shares_content(q, quote):\n"
  "        raise Reject(\"quote_shares_no_content_with_question\")\n"
  "    if ck.oov_count(quote + ans):\n"
  "        raise Reject(\"oov_in_quote_or_answer\")\n"
  "    head = pick(PASSAGE_HEAD, \"jaq-head\", qid)\n"
  "    prompt = f\"問題: {head}\\n文章（{title}）: {body}\\n質問: {q}\\n{INSTR}\"\n"
  "    target = f\"\\n根拠: 「{quote}」\\n答え: {ans}。\"\n"
  "    gk = \"jaquad:\" + title\n"
  "    return build_row(id=f\"v49acq_jaquad_{qid}\", family=\"jaquad_human\", template=\"acq_jaquad_span\", group=\"v49_jaquad_human\",\n"
  "                     category=\"evidence\", prompt=prompt, target=target, ref=ans, answer_type=\"span\", difficulty=2, spec=spec,\n"
  "                     split=split_of(gk), quotes=[quote], jaquad_question_type=str(rec.get(\"question_type\") or \"\"),\n"
  "                     jaquad_answer_type=str(a.get(\"answer_type\") or \"\"), group_key=gk, quality_flag=\"human annotated (Skelter Labs)\")\n"
  "\n"
  "\n"
  "def convert_jaquad(records, stats: Stats, ck: Checks, spec: Spec):\n"
  "    return run_converter(records, stats, lambda i, r: _jaquad_one(i, r, stats, ck, spec), ck)\n"
  "\n"
  "\n"
  "# ====================================================================================================== fenced code parsing\n"
  "LANG_ALIAS = {\"py\": \"python\", \"python3\": \"python\", \"py3\": \"python\", \"c++\": \"cpp\", \"cxx\": \"cpp\", \"cc\": \"cpp\", \"js\": \"javascript\",\n"
  "              \"node\": \"javascript\", \"ts\": \"typescript\", \"sh\": \"bash\", \"shell\": \"bash\", \"zsh\": \"bash\", \"c#\": \"csharp\",\n"
  "              \"cs\": \"csharp\", \"rs\": \"rust\", \"golang\": \"go\", \"rb\": \"ruby\", \"kt\": \"kotlin\"}\n"
  "OUTPUT_LANGS = frozenset(\"output console stdout result results\".split())\n"
  "CODE_LANGS = frozenset(\"\"\"cpp c rust java javascript typescript go csharp ruby php swift kotlin scala sql bash r perl lua haskell dart\n"
  "objectivec matlab julia fortran cobol elixir clojure groovy powershell vb vbnet assembly asm verilog\"\"\".split())\n"
  "\n"
  "\n"
  "def _norm_lang(tag: str) -> str:\n"
  "    tag = tag.strip().strip(\"{}.\").lower()\n"
  "    tag = tag.split()[0] if tag else \"\"\n"
  "    tag = tag.strip(\"{}.\")\n"
  "    return LANG_ALIAS.get(tag, tag)\n"
  "\n"
  "\n"
  "def parse_fences(text: str):\n"
  "    \"\"\"[{lang, code, start, end}] for the fenced blocks of `text`, or None when the fences are unbalanced / malformed.\"\"\"\n"
  "    lines = text.split(\"\\n\")\n"
  "    offs, o = [], 0\n"
  "    for ln in lines:\n"
  "        offs.append(o)\n"
  "        o += len(ln) + 1\n"
  "    blocks, open_i, lang, indent = [], None, \"\", \"\"\n"
  "    for i, ln in enumerate(lines):\n"
  "        s = ln.strip()\n"
  "        if not s.startswith(FENCE):\n"
  "            continue\n"
  "        if open_i is None:\n"
  "            tag = s[3:].strip()\n"
  "            if \"`\" in tag:\n"
  "                return None\n"
  "            open_i, lang, indent = i, _norm_lang(tag), ln[:len(ln) - len(ln.lstrip())]\n"
  "        else:\n"
  "            if s != FENCE:\n"
  "                return None\n"
  "            code = textwrap.dedent(\"\\n\".join(lines[open_i + 1:i]))\n"
  "            blocks.append({\"lang\": lang, \"code\": code, \"start\": offs[open_i], \"end\": offs[i] + len(ln)})\n"
  "            open_i = None\n"
  "    return None if open_i is not None else blocks\n"
  "\n"
  "\n"
  "def _numbers(s: str):\n"
  "    out = []\n"
  "    for m in re.findall(r\"-?\\d[\\d,]*(?:\\.\\d+)?\", s):\n"
  "        try:\n"
  "            out.append(float(m.replace(\",\", \"\")))\n"
  "        except ValueError:\n"
  "            pass\n"
  "    return out\n"
  "\n"
  "\n"
  "def _stored_outputs(asst: str, blocks, py):\n"
  "    \"\"\"Stored outputs of the answer that belong to the python block `py`: the blocks after it and BEFORE the next python block (a later block's\n"
  "    output must never validate this block), tagged output/console/stdout/result, or untagged/text blocks directly after a label such as\n"
  "    出力: / Output:.\"\"\"\n"
  "    nxt = min((b[\"start\"] for b in blocks if b[\"lang\"] == \"python\" and b[\"start\"] >= py[\"end\"]), default=None)\n"
  "    res = []\n"
  "    for b in blocks:\n"
  "        if b[\"start\"] < py[\"end\"] or b is py or (nxt is not None and b[\"start\"] >= nxt):\n"
  "            continue\n"
  "        if b[\"lang\"] in OUTPUT_LANGS:\n"
  "            res.append(b[\"code\"])\n"
  "        elif b[\"lang\"] in (\"\", \"text\", \"plaintext\", \"txt\"):\n"
  "            before = asst[max(0, b[\"start\"] - 80):b[\"start\"]]\n"
  "            if re.search(r\"(出力|実行結果|結果|output|result|expected)[^\\n]{0,40}[:：]?\\s*$\", before, re.I):\n"
  "                res.append(b[\"code\"])\n"
  "    return res\n"
  "\n"
  "\n"
  "def _same_output(a: str, b: str) -> bool:\n"
  "    na = [ln.rstrip() for ln in a.strip().splitlines()]\n"
  "    nb = [ln.rstrip() for ln in b.strip().splitlines()]\n"
  "    return na == nb\n"
  "\n"
  "\n"
  "def _prose_after(asst: str, blocks, py) -> str:\n"
  "    pieces, cur = [], py[\"end\"]\n"
  "    for b in blocks:\n"
  "        if b[\"start\"] >= py[\"end\"]:\n"
  "            pieces.append(asst[cur:b[\"start\"]])\n"
  "            cur = b[\"end\"]\n"
  "    pieces.append(asst[cur:])\n"
  "    return \"\\n\".join(pieces)\n"
  "\n"
  "\n"
  "_OUT_WORDS = re.compile(r\"(出力|結果|表示|返し|戻り|prints?\\b|printed|output|results?\\b|returns?\\b|displays?)\", re.I)\n"
  "\n"
  "\n"
  "def prose_number_contradiction(prose: str, stdout: str, code: str, user: str) -> bool:\n"
  "    \"\"\"True if a sentence of the prose that talks about the output states a number that occurs in neither stdout, code nor\n"
  "    question (single digits are ignored: list numbering).\"\"\"\n"
  "    ok = _numbers(stdout) + _numbers(code) + _numbers(user)\n"
  "    for sent in re.split(r\"[。\\n]|(?<=[.!?])\\s\", prose):\n"
  "        if not _OUT_WORDS.search(sent):\n"
  "            continue\n"
  "        for n in _numbers(sent):\n"
  "            if abs(n) < 10 and float(n).is_integer():\n"
  "                continue\n"
  "            if not any(abs(n - m) <= 1e-9 * max(1.0, abs(n)) for m in ok):\n"
  "                return True\n"
  "    return False\n"
  "\n"
  "\n"
  "# ====================================================================================================== (b) Synthetic-JP-EN-Coding\n"
  "PLAN_FALLBACK = \"コードを書いて実行し、出力を確かめる\"\n"
  "\n"
  "\n"
  "def pick_plan(asst: str, first_fence_start: int, ck: Checks) -> str:\n"
  "    \"\"\"One short Japanese sentence taken from the prose before the first code block; a fixed sentence otherwise.\"\"\"\n"
  "    for sent in re.split(r\"[。\\n]\", asst[:first_fence_start]):\n"
  "        sent = sent.strip()\n"
  "        if 8 <= len(sent) <= 80 and _KANA.search(sent) and not re.search(r\"[`*#<>|:：]\", sent) and not ck.oov_count(sent):\n"
  "            return sent\n"
  "    return PLAN_FALLBACK\n"
  "\n"
  "\n"
  "def strip_fences(text: str, blocks) -> str:\n"
  "    \"\"\"The prose of an answer: `text` without its fenced code blocks (language detection must not be fooled by code).\"\"\"\n"
  "    out, cur = [], 0\n"
  "    for b in blocks or []:\n"
  "        out.append(text[cur:b[\"start\"]])\n"
  "        cur = b[\"end\"]\n"
  "    out.append(text[cur:])\n"
  "    return \"\\n\".join(out)\n"
  "\n"
  "\n"
  "def english_gate(ck: Checks, ulang: str, alang: str, n_chars: int):\n"
  "    \"\"\"Rejects an English row longer than ck.en_row_max characters: long English rows are not used as they are (owner's plan 5-3);\n"
  "    short English rows stay as a skeleton.\"\"\"\n"
  "    if (ulang == \"en\" or alang == \"en\") and n_chars > ck.en_row_max:\n"
  "        raise Reject(\"english_row_too_long\")\n"
  "\n"
  "\n"
  "def _coding_one(idx, rec, st: Stats, ck: Checks, spec: Spec):\n"
  "    msgs = rec.get(\"messages\")\n"
  "    if not isinstance(msgs, list) or not msgs:\n"
  "        raise Reject(\"no_messages\")\n"
  "    ui = next((i for i, m in enumerate(msgs) if isinstance(m, dict) and m.get(\"role\") == \"user\"), None)\n"
  "    if ui is None:\n"
  "        raise Reject(\"no_user_message\")\n"
  "    ai = next((i for i in range(ui + 1, len(msgs)) if isinstance(msgs[i], dict) and msgs[i].get(\"role\") == \"assistant\"), None)\n"
  "    if ai is None:\n"
  "        raise Reject(\"no_assistant_message\")\n"
  "    if len(msgs) > ai + 1:\n"
  "        st.extra[\"multiturn_first_pair_only\"] += 1\n"
  "    user, asst = clean_text(msgs[ui].get(\"content\") or \"\").strip(), clean_text(msgs[ai].get(\"content\") or \"\").strip()\n"
  "    if not user or not asst:\n"
  "        raise Reject(\"empty_message\")\n"
  "    if len(user) > ck.max_chars - 60:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    blocks = parse_fences(asst)\n"
  "    if blocks is None:\n"
  "        raise Reject(\"unbalanced_fence\")\n"
  "    if not blocks:\n"
  "        raise Reject(\"no_code_block\")\n"
  "    ulang = detect_lang(user)\n"
  "    alang = detect_lang(strip_fences(asst, blocks))               # language of the PROSE: code letters would make everything 'en'\n"
  "    py = [b for b in blocks if b[\"lang\"] == \"python\"]\n"
  "    if py:\n"
  "        return _coding_python_row(idx, user, asst, blocks, py, ulang, alang, st, ck, spec)\n"
  "    return _coding_text_row(idx, user, asst, blocks, ulang, alang, st, ck, spec)\n"
  "\n"
  "\n"
  "def _coding_python_row(idx, user, asst, blocks, py, ulang, alang, st, ck, spec):\n"
  "    if not ck.run_code:\n"
  "        raise Reject(\"sandbox_failed:sandbox_disabled\")\n"
  "    chosen, first_reason = None, None\n"
  "    for b in py:\n"
  "        ok, why = ck.static_check(b[\"code\"].rstrip())            # the model's own executor rules (checks.static_check, profile executor)\n"
  "        if not ok:\n"
  "            first_reason = first_reason or why\n"
  "            continue\n"
  "        if \"print(\" not in b[\"code\"]:\n"
  "            first_reason = first_reason or \"no_print\"\n"
  "            continue\n"
  "        chosen = b\n"
  "        break\n"
  "    if chosen is None:\n"
  "        raise Reject(\"static:\" + (first_reason or \"no_candidate\"))\n"
  "    code = chosen[\"code\"].strip(\"\\n\").rstrip()\n"
  "    if FENCE in code:\n"
  "        raise Reject(\"fence_in_code\")\n"
  "    if len(user) + len(code) + 150 > ck.max_chars:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    res = ck.run_python(code)\n"
  "    if not res[\"ok\"]:\n"
  "        raise Reject(\"sandbox_failed:\" + (res.get(\"reason\") or \"unknown\"))\n"
  "    stdout = clean_text(res[\"stdout\"])\n"
  "    if not stdout.strip():\n"
  "        raise Reject(\"empty_stdout\")\n"
  "    if not stdout.endswith(\"\\n\"):\n"
  "        stdout += \"\\n\"\n"
  "    if FENCE in stdout:\n"
  "        raise Reject(\"fence_in_stdout\")\n"
  "    if re.search(r\"0x[0-9a-fA-F]{6,}\", stdout):\n"
  "        raise Reject(\"address_in_stdout\")                        # object repr: not reproducible\n"
  "    lines = [ln for ln in stdout.splitlines() if ln.strip()]\n"
  "    ans = lines[-1].strip()\n"
  "    if len(ans) > 120:\n"
  "        raise Reject(\"answer_line_too_long\")\n"
  "    stored = _stored_outputs(asst, blocks, chosen)\n"
  "    if stored and not any(_same_output(s, stdout) for s in stored):\n"
  "        raise Reject(\"stored_output_mismatch\")\n"
  "    verification = \"stored_output_equals_sandbox\" if stored else \"sandbox_only_no_stored_output\"\n"
  "    if not stored:\n"
  "        st.extra[\"python_sandbox_ok_no_stored_output\"] += 1       # ran, but nothing independent to compare with: counted for the report\n"
  "        if ck.python_require_stored_output:\n"
  "            raise Reject(\"no_stored_output_to_corroborate\")\n"
  "    if prose_number_contradiction(_prose_after(asst, blocks, chosen), stdout, code, user):\n"
  "        raise Reject(\"prose_number_contradicts_output\")\n"
  "    plan = pick_plan(asst, blocks[0][\"start\"], ck)\n"
  "    prompt = f\"問題: {user}\\n道具: Python\\n思考:\"\n"
  "    head = f\"\\n計画: {plan}。\\n{FENCE}python\\n{code}\\n{FENCE}\"\n"
  "    obs = f\"\\n{FENCE}output\\n{stdout}{FENCE}\\n\"\n"
  "    tail = f\"確認: 出力の最後の行を答えとする。\\n答え: {ans}。\"\n"
  "    target = head + obs + tail\n"
  "    a = len(prompt) + len(head)\n"
  "    english_gate(ck, ulang, alang, len(prompt) + len(target))\n"
  "    gk = f\"coding:{idx}\"\n"
  "    return build_row(id=f\"v49acq_coding_{idx:07d}\", family=\"coding_pytool\", template=\"acq_coding_pytool\", group=\"v49_coding_pytool\",\n"
  "                     category=\"python\", prompt=prompt, target=target, ref=ans, answer_type=\"output_last_line\", difficulty=3,\n"
  "                     spec=spec, split=split_of(gk), spans=[(a, a + len(obs))],\n"
  "                     steps=[{\"index\": 0, \"code\": code, \"stdout\": stdout, \"observation\": obs, \"success\": True}],\n"
  "                     python_code=code, expected_stdout=stdout, plan_line=plan, verification=verification,\n"
  "                     sandbox_backend=ck.backends[\"sandbox\"], sandbox_run_as=ck.backends[\"sandbox_run_as\"], user_lang=ulang,\n"
  "                     answer_lang=alang, group_key=gk,\n"
  "                     quality_flag=\"llm-generated instruction+code; output replaced by sandbox stdout, equal to the answer's stored output\")\n"
  "\n"
  "\n"
  "def _coding_text_row(idx, user, asst, blocks, ulang, alang, st, ck, spec):\n"
  "    if any(not b[\"lang\"] for b in blocks):\n"
  "        raise Reject(\"untagged_code_fence\")\n"
  "    code_blocks = [b for b in blocks if b[\"lang\"] in CODE_LANGS]\n"
  "    if not code_blocks:\n"
  "        raise Reject(\"no_code_language_block\")\n"
  "    prompt = f\"問題: {user}\\n{INSTR}\"\n"
  "    target = \"\\n\" + asst\n"
  "    if len(prompt) + len(target) > ck.max_chars:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    english_gate(ck, ulang, alang, len(prompt) + len(target))\n"
  "    lang = code_blocks[0][\"lang\"]\n"
  "    gk = f\"coding:{idx}\"\n"
  "    return build_row(id=f\"v49acq_coding_{idx:07d}\", family=\"code_text\", template=\"acq_coding_codetext\", group=\"v49_coding_codetext\",\n"
  "                     category=\"code\", prompt=prompt, target=target, ref=\"\", answer_type=\"free_code_text\", difficulty=3, spec=spec,\n"
  "                     split=split_of(gk), code_lang=lang, n_code_blocks=len(blocks), user_lang=ulang, answer_lang=alang,\n"
  "                     verification=\"unverified_no_execution\", group_key=gk,\n"
  "                     quality_flag=\"llm-generated; code not executed (unverified)\")\n"
  "\n"
  "\n"
  "def convert_coding(records, stats: Stats, ck: Checks, spec: Spec):\n"
  "    return run_converter(records, stats, lambda i, r: _coding_one(i, r, stats, ck, spec), ck)\n"
  "\n"
  "\n"
  "# ====================================================================================================== (c) Tengentoppa\n"
  "REFUSAL_RE = re.compile(r\"^(?:申し訳|すみません|ごめんなさい|残念ながら|私はAI|私は人工知能|AIとして|言語モデルとして|I'm sorry|I am sorry|I cannot|I can't|As an AI)\")\n"
  "REFUSAL_ANY = re.compile(r\"(お答えできません|回答できません|対応できません|お手伝いできません|お答えすることができません)\")\n"
  "BOILER_RE = re.compile(r\"^(?:もちろん|承知(?:いたし)?ました|かしこまりました|了解(?:しました|です)|はい、(?:もちろん|承知)|ご質問ありがとう|ご依頼ありがとう|以下に(?:回答|説明|示)|以下の通り)\")\n"
  "TERMINATORS = tuple(\"。！？」』）)\")\n"
  "TENGEN_FLAG = \"qwen2.5-32b generated, unverified\"\n"
  "\n"
  "\n"
  "def latin_ratio(s: str) -> float:\n"
  "    core = [c for c in s if not c.isspace()]\n"
  "    return sum(1 for c in core if \"A\" <= c <= \"Z\" or \"a\" <= c <= \"z\") / max(1, len(core))\n"
  "\n"
  "\n"
  "def answer_format(a: str):\n"
  "    \"\"\"'prose' (ends with a sentence terminator), 'fenced' (balanced fences, ends with the closing fence), 'json' (the whole answer is\n"
  "    valid JSON), or None = cut off (max_tokens) / unfinished.\"\"\"\n"
  "    if a.endswith(TERMINATORS):\n"
  "        return \"prose\"\n"
  "    if a.endswith(FENCE) and parse_fences(a):\n"
  "        return \"fenced\"\n"
  "    if a[-1:] in (\"}\", \"]\"):\n"
  "        try:\n"
  "            json.loads(a)\n"
  "            return \"json\"\n"
  "        except ValueError:\n"
  "            pass\n"
  "    return None\n"
  "\n"
  "\n"
  "def looks_repetitive(s: str) -> bool:\n"
  "    b = s.encode(\"utf-8\")\n"
  "    return len(b) > 300 and len(zlib.compress(b, 6)) / len(b) < 0.22\n"
  "\n"
  "\n"
  "def reads_as_steps(s: str) -> bool:\n"
  "    return len(re.findall(r\"(?m)^\\s*(?:\\d+[.．)）]|[-・*]\\s|ステップ\\s*\\d)\", s)) >= 2\n"
  "\n"
  "\n"
  "def _tengentoppa_one(idx, rec, st: Stats, ck: Checks, spec: Spec):\n"
  "    if \"__bad_json__\" in rec:\n"
  "        raise Reject(\"bad_json_line\")\n"
  "    instr = clean_text(rec.get(\"instruction\") or \"\").strip()\n"
  "    refined = clean_text(rec.get(\"refined_answer\") or \"\").strip()\n"
  "    reasoning = clean_text(rec.get(\"reasoning\") or \"\").strip()\n"
  "    if not instr or not refined:\n"
  "        raise Reject(\"empty_field\")\n"
  "    if has_python_block(refined):\n"
  "        raise Reject(\"python_block_not_executed\")                 # a python block with a stated output cannot be re-run here: dropped\n"
  "    if REFUSAL_RE.match(refined[:40]) or REFUSAL_ANY.search(refined[:120]):\n"
  "        raise Reject(\"refusal_opening\")\n"
  "    if BOILER_RE.match(refined[:40]):\n"
  "        raise Reject(\"boilerplate_opening\")\n"
  "    if latin_ratio(refined) > 0.30:\n"
  "        raise Reject(\"latin_heavy\")\n"
  "    fmt = answer_format(refined)\n"
  "    if fmt is None:\n"
  "        raise Reject(\"answer_not_terminated\")\n"
  "    if looks_repetitive(refined):\n"
  "        raise Reject(\"repetitive\")\n"
  "    prompt = f\"問題: {instr}\\n{INSTR}\"\n"
  "    target = f\"\\n答え: {refined}\"\n"
  "    family, tmpl = \"tengentoppa_answer\", \"acq_tengentoppa_answer\"\n"
  "    if len(prompt) + len(target) > ck.max_chars:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    if (ck.tengentoppa_steps and reasoning and reads_as_steps(reasoning) and latin_ratio(reasoning) <= 0.30 and not looks_repetitive(reasoning)\n"
  "            and not has_python_block(reasoning)):\n"
  "        t2 = f\"\\n手順: {reasoning}\\n答え: {refined}\"\n"
  "        if len(prompt) + len(t2) <= ck.max_chars:\n"
  "            target, family, tmpl = t2, \"tengentoppa_steps\", \"acq_tengentoppa_steps\"\n"
  "    gk = f\"tengentoppa:{idx}\"\n"
  "    return build_row(id=f\"v49acq_tengentoppa_{idx:07d}\", family=family, template=tmpl, group=\"v49_tengentoppa\", category=\"free_text\",\n"
  "                     prompt=prompt, target=target, ref=refined, answer_type=\"free_text\", difficulty=2, spec=spec,\n"
  "                     split=split_of(gk), group_key=gk, answer_format=fmt, has_python_block=False,\n"
  "                     quality_flag=TENGEN_FLAG)\n"
  "\n"
  "\n"
  "def convert_tengentoppa(records, stats: Stats, ck: Checks, spec: Spec):\n"
  "    return run_converter(records, stats, lambda i, r: _tengentoppa_one(i, r, stats, ck, spec), ck)\n"
  "\n"
  "\n"
  "# ====================================================================================================== (d) HotpotQA (distractor, train only)\n"
  "HOTPOT_LEVEL_NOTE = {0: \"supporting paragraphs up to the last supporting sentence + 2 distractors (2 sentences)\",\n"
  "                     1: \"same, distractors cut to 1 sentence\", 2: \"supporting paragraphs only\",\n"
  "                     3: \"supporting sentences + paragraph opening\", 4: \"supporting sentences only\"}\n"
  "HOTPOT_HEAD = \"次の英語の段落を読んで、質問に英語で短く答えてください。\"\n"
  "\n"
  "\n"
  "def _answer_in_evidence(answer: str, evidence: str) -> bool:\n"
  "    \"\"\"The (normalised) answer occurs in the evidence as whole word(s): 'Spa' is not found in 'Spain'.\"\"\"\n"
  "    return bool(answer) and re.search(r\"(?<!\\w)\" + re.escape(answer) + r\"(?!\\w)\", evidence) is not None\n"
  "\n"
  "\n"
  "def _hotpot_one(idx, rec, st: Stats, ck: Checks, spec: Spec):\n"
  "    rid = str(rec.get(\"id\") or \"\")\n"
  "    q, ans = clean_ws(rec.get(\"question\") or \"\"), clean_ws(rec.get(\"answer\") or \"\")\n"
  "    sf, ctx = rec.get(\"supporting_facts\") or {}, rec.get(\"context\") or {}\n"
  "    if not rid or not q or not ans:\n"
  "        raise Reject(\"missing_field\")\n"
  "    by_title = {}\n"
  "    for t, ss in zip(ctx.get(\"title\", []), ctx.get(\"sentences\", [])):\n"
  "        by_title.setdefault(str(t), [clean_ws(s) for s in ss])\n"
  "    supp = {}\n"
  "    for t, i in zip(sf.get(\"title\", []), sf.get(\"sent_id\", [])):\n"
  "        supp.setdefault(str(t), set()).add(int(i))\n"
  "    if len(supp) != 2:\n"
  "        raise Reject(\"not_two_support_paragraphs\")\n"
  "    for t, ids in supp.items():\n"
  "        ss = by_title.get(t)\n"
  "        if ss is None or any(i < 0 or i >= len(ss) or not ss[i] for i in ids):\n"
  "            raise Reject(\"bad_supporting_fact\")\n"
  "    full_len = sum(len(\" \".join(ss)) for ss in by_title.values()) + len(q) + 120\n"
  "    if full_len <= ck.max_chars:\n"
  "        st.extra[\"fit_untrimmed_all_paragraphs\"] += 1\n"
  "    distr = sorted((t for t, ss in by_title.items() if t not in supp and any(ss)), key=lambda t: h(\"hp-dist\", rid, t))[:2]\n"
  "\n"
  "    def text_of(t, lv):\n"
  "        ss = by_title[t]\n"
  "        if t in supp:\n"
  "            ids = sorted(supp[t])\n"
  "            if lv <= 2:\n"
  "                keep = ss[:ids[-1] + 1]\n"
  "            elif lv == 3:\n"
  "                keep = ([ss[0]] if 0 not in ids else []) + [ss[i] for i in ids]\n"
  "            else:\n"
  "                keep = [ss[i] for i in ids]\n"
  "        else:\n"
  "            keep = ss[:2 if lv == 0 else 1]\n"
  "        return \" \".join(x for x in keep if x)\n"
  "\n"
  "    last_len = None\n"
  "    chosen = None\n"
  "    for lv in range(5):\n"
  "        titles = list(supp) + (distr if lv <= 1 else [])\n"
  "        titles.sort(key=lambda t: h(\"hp-order\", rid, t))\n"
  "        paras = [(t, text_of(t, lv)) for t in titles]\n"
  "        quotes_by_para = [(n + 1, t, [by_title[t][i] for i in sorted(supp[t])]) for n, (t, _) in enumerate(paras) if t in supp]\n"
  "        body = \"\\n\".join(f\"段落{n + 1}（{t}）: {txt}\" for n, (t, txt) in enumerate(paras))\n"
  "        prompt = f\"問題: {HOTPOT_HEAD}\\n{body}\\n質問: {q}\\n{INSTR}\"\n"
  "        find = \"探す: \" + \"と\".join(f\"段落{n}（{t}）\" for n, t, _ in quotes_by_para) + \"が必要。\"\n"
  "        qlines = [f\"根拠: 「{s}」\" for _, _, ss in quotes_by_para for s in ss]\n"
  "        yn = ans.lower() in (\"yes\", \"no\")\n"
  "        target = \"\\n\" + find + \"\\n\" + \"\\n\".join(qlines) + f\"\\n答え: {ans}。\"\n"
  "        last_len = len(prompt) + len(target)\n"
  "        if last_len <= ck.max_chars:\n"
  "            chosen = (lv, prompt, target, quotes_by_para, yn)\n"
  "            break\n"
  "    if chosen is None:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    lv, prompt, target, qbp, yn = chosen\n"
  "    quotes = [s for _, _, ss in qbp for s in ss]\n"
  "    norm = lambda s: re.sub(r\"\\s+\", \" \", s.lower()).strip()      # noqa: E731\n"
  "    if not yn and not _answer_in_evidence(norm(ans).strip(\" .,;:\\\"'\"), norm(\" \".join(quotes))):\n"
  "        raise Reject(\"answer_not_in_evidence\")\n"
  "    if ck.oov_count(\" \".join(quotes) + ans + \" \".join(t for _, t, _ in qbp)):\n"
  "        raise Reject(\"oov_in_quote_or_answer\")\n"
  "    first = min(supp)\n"
  "    gk = \"hotpot:\" + (ck.hotpot_roots or {}).get(first, first)    # union-find root over shared supporting paragraphs (run_dataset)\n"
  "    diff = {\"easy\": 2, \"medium\": 3, \"hard\": 4}.get(str(rec.get(\"level\")), 3)\n"
  "    return build_row(id=f\"v49acq_hotpot_{rid}\", family=\"hotpot_en_short\", template=\"acq_hotpot_en_short\", group=\"v49_hotpot_en_short\",\n"
  "                     category=\"evidence\", prompt=prompt, target=target, ref=ans, answer_type=\"yes_no_en\" if yn else \"span_en\",\n"
  "                     difficulty=diff, spec=spec, split=split_of(gk), quotes=quotes, hotpot_type=str(rec.get(\"type\") or \"\"),\n"
  "                     hotpot_level=str(rec.get(\"level\") or \"\"), trim_level=lv, trim_note=HOTPOT_LEVEL_NOTE[lv], group_key=gk,\n"
  "                     language=\"en-passages/ja-frame\", quality_flag=\"human annotated (HotpotQA); English passages\")\n"
  "\n"
  "\n"
  "def convert_hotpot(records, stats: Stats, ck: Checks, spec: Spec):\n"
  "    return run_converter(records, stats, lambda i, r: _hotpot_one(i, r, stats, ck, spec), ck)\n"
  "\n"
  "\n"
  "# ====================================================================================================== (e) ToolACE (adoption: hold)\n"
  "TOOL_MARKER = \"list of functions in JSON format that you can invoke:\"\n"
  "TOOLACE_HOLD = \"hold (generator unverified: the card and the paper text do not name the generator model)\"\n"
  "\n"
  "\n"
  "def parse_tool_functions(system: str):\n"
  "    i = system.find(TOOL_MARKER)\n"
  "    if i < 0:\n"
  "        return None\n"
  "    j = system.find(\"[\", i + len(TOOL_MARKER))\n"
  "    if j < 0:\n"
  "        return None\n"
  "    try:\n"
  "        obj, _ = json.JSONDecoder().raw_decode(system, j)\n"
  "    except ValueError:\n"
  "        return None\n"
  "    return obj if isinstance(obj, list) and all(isinstance(f, dict) and f.get(\"name\") for f in obj) else None\n"
  "\n"
  "\n"
  "def compact_function(f: dict, desc_chars: int = 90) -> str:\n"
  "    params = f.get(\"parameters\") if isinstance(f.get(\"parameters\"), dict) else {}\n"
  "    props = params.get(\"properties\") if isinstance(params.get(\"properties\"), dict) else {}\n"
  "    req = params.get(\"required\") if isinstance(params.get(\"required\"), list) else []\n"
  "    parts = [f\"{k}{'' if k in req else '?'}: {(v or {}).get('type', 'any') if isinstance(v, dict) else 'any'}\" for k, v in props.items()]\n"
  "    desc = clean_ws(f.get(\"description\") or \"\")\n"
  "    if len(desc) > desc_chars:\n"
  "        desc = desc[:desc_chars].rsplit(\" \", 1)[0].rstrip(\" ,;:\")\n"
  "    return f\"- {f['name']}({', '.join(parts)}): {desc}\"\n"
  "\n"
  "\n"
  "def is_tool_call(value: str) -> bool:\n"
  "    \"\"\"ToolACE assistant turns that are calls look like [Name(arg=value, ...), Name2(...)] (names may contain spaces / parentheses).\"\"\"\n"
  "    v = value.strip()\n"
  "    return v.startswith(\"[\") and v.endswith(\")]\") and \"(\" in v\n"
  "\n"
  "\n"
  "def _toolace_one(idx, rec, st: Stats, ck: Checks, spec: Spec):\n"
  "    if \"__bad_json__\" in rec:\n"
  "        raise Reject(\"bad_json_item\")\n"
  "    conv = rec.get(\"conversations\")\n"
  "    if not isinstance(conv, list) or len(conv) < 2 or conv[0].get(\"from\") != \"user\":\n"
  "        raise Reject(\"no_user_turn\")\n"
  "    user = clean_text(conv[0].get(\"value\") or \"\").strip()\n"
  "    lang = detect_lang(user)\n"
  "    st.extra[\"lang_\" + lang] += 1                                 # every item with a user turn, before any other filter\n"
  "    funcs = parse_tool_functions(str(rec.get(\"system\") or \"\"))\n"
  "    if not funcs:\n"
  "        # about a third of the file lists its tools as YAML / XML / markdown text and calls them in other syntaxes; not converted\n"
  "        raise Reject(\"tool_list_not_json\")\n"
  "    steps, final, i = [], None, 1\n"
  "    while i < len(conv):\n"
  "        m = conv[i]\n"
  "        who, val = m.get(\"from\"), clean_text(m.get(\"value\") or \"\").strip()\n"
  "        if who == \"assistant\" and val and is_tool_call(val):\n"
  "            obs = None\n"
  "            if i + 1 < len(conv) and conv[i + 1].get(\"from\") == \"tool\":\n"
  "                obs = clean_text(conv[i + 1].get(\"value\") or \"\").strip()\n"
  "                i += 1\n"
  "            steps.append((val, obs))\n"
  "            i += 1\n"
  "            continue\n"
  "        if who == \"assistant\":\n"
  "            final = val or None\n"
  "            i += 1\n"
  "        break\n"
  "    if i < len(conv):\n"
  "        st.extra[\"multiturn_first_exchange_only\"] += 1\n"
  "    if not user:\n"
  "        raise Reject(\"empty_user\")\n"
  "    if steps and final is None and any(o is not None for _, o in steps):\n"
  "        raise Reject(\"no_final_answer_after_tool\")\n"
  "    if steps and final is None and len(steps) != 1:\n"
  "        raise Reject(\"multiple_calls_without_result\")\n"
  "    if not steps and not final:\n"
  "        raise Reject(\"empty_assistant\")\n"
  "    if any(o is not None and not o for _, o in steps):\n"
  "        raise Reject(\"empty_tool_result\")\n"
  "    if steps and final and steps[-1][1] is None:\n"
  "        raise Reject(\"final_after_call_without_result\")\n"
  "    sigs = \"\\n\".join(compact_function(f) for f in funcs)\n"
  "    prompt = f\"問題: {user}\\n道具: 関数呼び出し\\n関数一覧:\\n{sigs}\\n思考:\"\n"
  "    if len(prompt) > ck.max_chars - 40:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    spans, tsteps, target = [], [], \"\"\n"
  "    base = len(prompt)\n"
  "    for k, (call, obs) in enumerate(steps):\n"
  "        target += (\"\\n\" if k == 0 else \"\") + f\"呼び出し: {call}\"\n"
  "        if obs is not None:\n"
  "            o = f\"\\n結果: {obs}\\n\"\n"
  "            a = base + len(target)\n"
  "            spans.append((a, a + len(o)))\n"
  "            tsteps.append({\"index\": k, \"call\": call, \"result\": obs, \"observation\": o, \"success\": True})\n"
  "            target += o\n"
  "        else:\n"
  "            target += \"\\n\"\n"
  "    if final:\n"
  "        target += f\"答え: {final}\"\n"
  "        family, ans = (\"toolace_call\", final) if steps else (\"toolace_nocall\", final)\n"
  "    else:\n"
  "        target = target.rstrip(\"\\n\")\n"
  "        family, ans = \"toolace_callonly\", steps[0][0]\n"
  "    if not target.startswith(\"\\n\"):\n"
  "        target = \"\\n\" + target\n"
  "        spans = [(a + 1, b + 1) for a, b in spans]\n"
  "    if len(prompt) + len(target) > ck.max_chars:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    gk = f\"toolace:{idx}\"\n"
  "    return build_row(id=f\"v49acq_toolace_{idx:06d}\", family=family, template=\"acq_\" + family, group=\"v49_toolace\", category=\"tool_use\",\n"
  "                     prompt=prompt, target=target, ref=ans, answer_type=\"tool_call\" if family == \"toolace_callonly\" else \"free_text\",\n"
  "                     difficulty=3, spec=spec, split=split_of(gk), spans=spans, steps=tsteps, n_functions=len(funcs),\n"
  "                     n_calls=len(steps), user_lang=lang, group_key=gk,\n"
  "                     quality_flag=\"synthetic (ToolACE); generator unverified\")\n"
  "\n"
  "\n"
  "def convert_toolace(records, stats: Stats, ck: Checks, spec: Spec):\n"
  "    return run_converter(records, stats, lambda i, r: _toolace_one(i, r, stats, ck, spec), ck)\n"
  "\n"
  "\n"
  "# ====================================================================================================== (f) wizardlm8x22b (adoption: hold / low priority)\n"
  "_ARITH = re.compile(r\"(?<![0-9.A-Za-z])(?P<expr>\\(?-?\\d+(?:\\.\\d+)?\\)?(?:\\s*[-+*/]\\s*\\(?-?\\d+(?:\\.\\d+)?\\)?)+)\\s*=\\s*\"\n"
  "                    r\"(?P<res>-?\\d+(?:\\.\\d+)?)(?![\\d.]|\\s*[-+*/^]\\s*[\\d(])\")\n"
  "WIZARD_HOLD = \"hold / low priority (the owner's materials plan lists it under 'avoid: obvious mathematical errors')\"\n"
  "\n"
  "\n"
  "def arith_normalize(s: str) -> str:\n"
  "    s = unicodedata.normalize(\"NFKC\", s).translate({0xD7: \"*\", 0xF7: \"/\", 0x2212: \"-\", 0x2013: \"-\"})\n"
  "    return re.sub(r\"(?<=\\d),(?=\\d{3}(?!\\d))\", \"\", s)\n"
  "\n"
  "\n"
  "def safe_eval_fraction(expr: str):\n"
  "    \"\"\"Exact value of an arithmetic expression (+ - * / parentheses, unary minus, decimals) or None.  No eval().\"\"\"\n"
  "    try:\n"
  "        tree = ast.parse(expr.strip(), mode=\"eval\")\n"
  "    except (SyntaxError, ValueError, RecursionError):\n"
  "        return None\n"
  "    n_nodes = [0]\n"
  "\n"
  "    def ev(n):\n"
  "        n_nodes[0] += 1\n"
  "        if n_nodes[0] > 40:\n"
  "            raise ValueError(\"too big\")\n"
  "        if isinstance(n, ast.Expression):\n"
  "            return ev(n.body)\n"
  "        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):\n"
  "            return Fraction(str(n.value))\n"
  "        if isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.USub, ast.UAdd)):\n"
  "            v = ev(n.operand)\n"
  "            return -v if isinstance(n.op, ast.USub) else v\n"
  "        if isinstance(n, ast.BinOp) and isinstance(n.op, (ast.Add, ast.Sub, ast.Mult, ast.Div)):\n"
  "            a, b = ev(n.left), ev(n.right)\n"
  "            if isinstance(n.op, ast.Add):\n"
  "                return a + b\n"
  "            if isinstance(n.op, ast.Sub):\n"
  "                return a - b\n"
  "            if isinstance(n.op, ast.Mult):\n"
  "                return a * b\n"
  "            if b == 0:\n"
  "                raise ZeroDivisionError\n"
  "            return a / b\n"
  "        raise ValueError(\"node\")\n"
  "\n"
  "    try:\n"
  "        return ev(tree)\n"
  "    except (ValueError, ZeroDivisionError, OverflowError):\n"
  "        return None\n"
  "\n"
  "\n"
  "def check_arithmetic(text: str):\n"
  "    \"\"\"[(statement, ok)] for the simple 'a op b ... = c' statements of `text`.  Rounded results are accepted (|diff| <= half a unit\n"
  "    of the last written digit); a statement preceded by an operator / letter / '=' is skipped (chain or equation in a variable).\"\"\"\n"
  "    s = arith_normalize(text)\n"
  "    out = []\n"
  "    for m in _ARITH.finditer(s):\n"
  "        prev = s[:m.start()].rstrip()[-1:]\n"
  "        if prev and (prev in \"+-*/^(=\" or (prev.isascii() and prev.isalnum())):\n"
  "            continue\n"
  "        expr, res = m.group(\"expr\"), m.group(\"res\")\n"
  "        if expr.count(\"(\") != expr.count(\")\") or not re.search(r\"\\d\\s*[-+*/]\\s*\\(?-?\\d\", expr):\n"
  "            continue\n"
  "        val = safe_eval_fraction(expr)\n"
  "        if val is None:\n"
  "            continue\n"
  "        dec = len(res.split(\".\")[1]) if \".\" in res else 0\n"
  "        ok = abs(val - Fraction(res)) <= Fraction(1, 2 * 10 ** dec)\n"
  "        out.append((f\"{expr.strip()} = {res}\", ok))\n"
  "        if len(out) >= 60:\n"
  "            break\n"
  "    return out\n"
  "\n"
  "\n"
  "def _wizard_one(idx, rec, st: Stats, ck: Checks, spec: Spec):\n"
  "    msgs = rec.get(\"messages\")\n"
  "    if not isinstance(msgs, list) or not msgs:\n"
  "        raise Reject(\"no_messages\")\n"
  "    ui = next((i for i, m in enumerate(msgs) if isinstance(m, dict) and m.get(\"role\") == \"user\"), None)\n"
  "    ai = None if ui is None else next((i for i in range(ui + 1, len(msgs)) if isinstance(msgs[i], dict) and msgs[i].get(\"role\") == \"assistant\"), None)\n"
  "    if ui is None or ai is None:\n"
  "        raise Reject(\"no_user_assistant_pair\")\n"
  "    if len(msgs) > ai + 1:\n"
  "        st.extra[\"multiturn_first_pair_only\"] += 1\n"
  "    user, asst = clean_text(msgs[ui].get(\"content\") or \"\").strip(), clean_text(msgs[ai].get(\"content\") or \"\").strip()\n"
  "    if not user or not asst:\n"
  "        raise Reject(\"empty_message\")\n"
  "    stmts = check_arithmetic(asst)                               # statistics over ALL parseable rows, before the length filter\n"
  "    has_py = has_python_block(asst)\n"
  "    prompt = f\"問題: {user}\\n{INSTR}\"\n"
  "    target = f\"\\n答え: {asst}\"\n"
  "    fits = len(prompt) + len(target) <= ck.max_chars\n"
  "    if stmts:\n"
  "        wrong = sum(1 for _, ok in stmts if not ok)\n"
  "        st.extra[\"arith_rows_checked\"] += 1\n"
  "        st.extra[\"arith_stmts_checked\"] += len(stmts)\n"
  "        st.extra[\"arith_stmts_wrong\"] += wrong\n"
  "        st.extra[\"arith_rows_wrong\"] += wrong > 0\n"
  "        if fits:\n"
  "            st.extra[\"fit_arith_rows_checked\"] += 1\n"
  "            st.extra[\"fit_arith_rows_wrong\"] += wrong > 0\n"
  "    if has_py:\n"
  "        raise Reject(\"python_block_not_executed\")                 # a python block with a stated output cannot be re-run here: dropped\n"
  "    if not fits:\n"
  "        raise Reject(\"row_too_long\")\n"
  "    if stmts and any(not ok for _, ok in stmts):\n"
  "        raise Reject(\"arith_statement_wrong\")\n"
  "    if latin_ratio(asst) > 0.30:\n"
  "        raise Reject(\"latin_heavy\")\n"
  "    if looks_repetitive(asst):\n"
  "        raise Reject(\"repetitive\")\n"
  "    nums = _numbers(unicodedata.normalize(\"NFKC\", asst))\n"
  "    family = \"wizard_math\" if stmts else \"wizard_text\"\n"
  "    gk = f\"wizard:{idx}\"\n"
  "    extra = {\"arith_statements_checked\": len(stmts), \"has_python_block\": False, \"group_key\": gk,\n"
  "             \"quality_flag\": \"wizardlm-2-8x22b generated, unverified\"}\n"
  "    if stmts and nums:\n"
  "        extra[\"final_number_guess\"] = nums[-1]\n"
  "    return build_row(id=f\"v49acq_wizard_{idx:07d}\", family=family, template=\"acq_\" + family, group=\"v49_wizard\", category=\"free_text\",\n"
  "                     prompt=prompt, target=target, ref=asst, answer_type=\"free_text\", difficulty=3, spec=spec, split=split_of(gk),\n"
  "                     **extra)\n"
  "\n"
  "\n"
  "def convert_wizard(records, stats: Stats, ck: Checks, spec: Spec):\n"
  "    return run_converter(records, stats, lambda i, r: _wizard_one(i, r, stats, ck, spec), ck)\n"
  "\n"
  "\n"
  "# ====================================================================================================== registry\n"
  "DATASETS: dict = {\n"
  "    \"jaquad\": Spec(\"jaquad\", \"SkelterLabsInc/JaQuAD\", \"CC-BY-SA-3.0\",\n"
  "                   \"JaQuAD (So, Byun, Kang, Cho; Skelter Labs) - CC BY-SA 3.0 - https://huggingface.co/datasets/SkelterLabsInc/JaQuAD\",\n"
  "                   \"So et al. (2022) JaQuAD: Japanese Question Answering Dataset for Machine Reading Comprehension, arXiv:2202.01764; \"\n"
  "                   \"contexts from Japanese Wikipedia (CC BY-SA); train files only\",\n"
  "                   \"candidate\", \"human annotated; group split by article title\", find_jaquad, read_jaquad, convert_jaquad),\n"
  "    \"coding\": Spec(\"coding\", \"llm-jp/Synthetic-JP-EN-Coding-Dataset\", \"Apache-2.0\",\n"
  "                   \"llm-jp/Synthetic-JP-EN-Coding-Dataset (subset of Aratako/Synthetic-JP-EN-Coding-Dataset-801k) - Apache-2.0\",\n"
  "                   \"LLM-jp (Kiyomaru, Kodama); instructions by Nemotron-4-340B-Instruct, Phi-3-medium, Mixtral-8x22B-Instruct, calm3-22b-chat; \"\n"
  "                   \"no post-filtering by the authors\",\n"
  "                   \"candidate\",\n"
  "                   \"python rows: executed by the executor-profile sandbox, stored output replaced by the sandbox stdout, and KEPT ONLY \"\n"
  "                   \"when the answer's own stored output block equals it (verification = stored_output_equals_sandbox; rows that ran but \"\n"
  "                   \"had no stored output are counted and dropped); code_text rows (other languages) are unverified, not executed; long \"\n"
  "                   \"English rows are dropped\", find_coding,\n"
  "                   read_messages_parquet, convert_coding),\n"
  "    \"tengentoppa\": Spec(\"tengentoppa\", \"DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k\", \"Apache-2.0\",\n"
  "                        \"DeL-TaiseiOzaki/Tengentoppa-sft-qwen2.5-32b-reasoning-100k - Apache-2.0\",\n"
  "                        \"Generated with Qwen2.5-32B-Instruct (multi-persona instructions, refined answers); no human check besides the card's note\",\n"
  "                        \"candidate\", \"qwen2.5-32b generated, unverified; refined_answer only (initial_answer dropped)\",\n"
  "                        find_tengentoppa, read_tengentoppa, convert_tengentoppa),\n"
  "    \"hotpot\": Spec(\"hotpot\", \"hotpotqa/hotpot_qa\", \"CC-BY-SA-4.0\",\n"
  "                   \"HotpotQA distractor setting, train split (Yang et al., 2018) - CC BY-SA 4.0 - https://hotpotqa.github.io/\",\n"
  "                   \"Yang, Qi, Zhang, Bengio, Cohen, Salakhutdinov, Manning (2018) HotpotQA: A Dataset for Diverse, Explainable Multi-hop \"\n"
  "                   \"Question Answering, EMNLP; English Wikipedia paragraphs; validation split never used\",\n"
  "                   \"candidate\", \"short English rows only: English passages inside a Japanese instruction frame\",\n"
  "                   find_hotpot, read_hotpot, convert_hotpot),\n"
  "    \"toolace\": Spec(\"toolace\", \"Team-ACE/ToolACE\", \"Apache-2.0\",\n"
  "                    \"Team-ACE/ToolACE - Apache-2.0 (dataset card)\",\n"
  "                    \"Liu et al. (2024) ToolACE: Winning the Points of LLM Function Calling, arXiv:2409.00920; generator model not stated\",\n"
  "                    \"hold\", TOOLACE_HOLD, find_toolace, read_toolace, convert_toolace),\n"
  "    \"wizard\": Spec(\"wizard\", \"llm-jp/wizardlm8x22b-logical-math-coding-sft-ja\", \"Apache-2.0\",\n"
  "                   \"llm-jp/wizardlm8x22b-logical-math-coding-sft-ja - Apache-2.0 (from kanhatakeyama/wizardlm8x22b-logical-math-coding-sft-ja \"\n"
  "                   \"and ..._additional-ja)\",\n"
  "                   \"LLM-jp (Kiyomaru, Kodama); texts generated with WizardLM-2-8x22B\",\n"
  "                   \"hold\", WIZARD_HOLD, find_wizard, read_messages_parquet, convert_wizard),\n"
  "}\n"
  "\n"
  "\n"
  "def summarize(stats: Stats, ck: Checks | None = None) -> dict:\n"
  "    \"\"\"JSON-able summary of one converter run (counts, rejects by reason, per-family splits, derived rates, backends).\"\"\"\n"
  "    spec = DATASETS.get(stats.key)\n"
  "    rej_total = sum(stats.rejects.values())\n"
  "    d = {\n"
  "        \"dataset\": stats.key, \"hf\": spec.hf if spec else \"\", \"license\": spec.license if spec else \"\",\n"
  "        \"adoption\": spec.adoption if spec else \"\", \"adoption_note\": spec.adoption_note if spec else \"\",\n"
  "        \"n_in\": stats.n_in, \"n_out\": stats.n_out, \"keep_rate\": round(stats.n_out / stats.n_in, 6) if stats.n_in else None,\n"
  "        \"reject_total\": rej_total, \"balanced\": stats.n_in == stats.n_out + rej_total,\n"
  "        \"rejects\": dict(stats.rejects.most_common()), \"families\": dict(stats.families), \"splits\": dict(stats.splits),\n"
  "        \"family_splits\": {k: dict(v) for k, v in stats.family_splits.items()}, \"extra\": dict(stats.extra),\n"
  "    }\n"
  "    e = stats.extra\n"
  "    if stats.key == \"wizard\":\n"
  "        d[\"arith_error_rate\"] = {\n"
  "            \"statements_checked\": e[\"arith_stmts_checked\"], \"statements_wrong\": e[\"arith_stmts_wrong\"],\n"
  "            \"statement_error_rate\": round(e[\"arith_stmts_wrong\"] / e[\"arith_stmts_checked\"], 6) if e[\"arith_stmts_checked\"] else None,\n"
  "            \"rows_with_checked_statements\": e[\"arith_rows_checked\"], \"rows_with_a_wrong_statement\": e[\"arith_rows_wrong\"],\n"
  "            \"row_error_rate\": round(e[\"arith_rows_wrong\"] / e[\"arith_rows_checked\"], 6) if e[\"arith_rows_checked\"] else None,\n"
  "            \"rows_fitting_cap_checked\": e[\"fit_arith_rows_checked\"], \"rows_fitting_cap_with_wrong\": e[\"fit_arith_rows_wrong\"],\n"
  "            \"note\": \"only simple 'a op b = c' statements are recomputed (exact fractions, rounded results accepted); lower bound of the true error rate\"}\n"
  "    if stats.key == \"toolace\":\n"
  "        d[\"language\"] = {\"parsed_en\": e[\"lang_en\"], \"parsed_zh\": e[\"lang_zh\"], \"parsed_ja\": e[\"lang_ja\"], \"parsed_other\": e[\"lang_other\"],\n"
  "                         \"kept_en\": e[\"accepted_user_lang_en\"], \"kept_zh\": e[\"accepted_user_lang_zh\"]}\n"
  "    if stats.key == \"hotpot\":\n"
  "        d[\"fit\"] = {\"rows_emitted\": stats.n_out, \"fit_untrimmed_all_paragraphs\": e[\"fit_untrimmed_all_paragraphs\"],\n"
  "                    \"by_trim_level\": {k: e[f\"trim_level_{k}\"] for k in range(5)}, \"level_note\": HOTPOT_LEVEL_NOTE}\n"
  "    if stats.key == \"coding\":\n"
  "        d[\"verification\"] = {k[len(\"verification_\"):]: v for k, v in e.items() if k.startswith(\"verification_\")}\n"
  "        d[\"code_text_languages\"] = {k[len(\"code_text_lang_\"):]: v for k, v in e.items() if k.startswith(\"code_text_lang_\")}\n"
  "        d[\"python\"] = {\"rows_kept_stored_output_equals_sandbox\": e[\"verification_stored_output_equals_sandbox\"],\n"
  "                       \"ran_in_sandbox_but_had_no_stored_output\": e[\"python_sandbox_ok_no_stored_output\"],\n"
  "                       \"note\": \"rows that ran but had no stored output block are NOT kept as verified python rows (owner's rule: keep only rows whose \"\n"
  "                               \"output equals the reference); the second number is how many there were\"}\n"
  "    if stats.key == \"jaquad\":\n"
  "        d[\"question_types\"] = {k[len(\"question_type_seen_\"):]: v for k, v in e.items() if k.startswith(\"question_type_seen_\")}\n"
  "    d[\"split_unit\"] = SPLIT_UNIT.get(stats.key, \"\")\n"
  "    d[\"group_independent_dev_heldout\"] = stats.key in (\"jaquad\", \"hotpot\")\n"
  "    fams = sorted(stats.families)\n"
  "    d[\"supervised_chars\"] = {f: e[\"supervised_chars_\" + f] for f in fams}\n"
  "    d[\"english\"] = {f: {\"rows\": e[\"english_rows_\" + f], \"supervised_chars\": e[\"english_supervised_chars_\" + f]} for f in fams if e[\"english_rows_\" + f]}\n"
  "    d[\"english_rows_total\"] = sum(v[\"rows\"] for v in d[\"english\"].values())\n"
  "    if ck is not None:\n"
  "        d[\"backends\"] = dict(ck.backends)\n"
  "        d[\"max_chars\"] = ck.max_chars\n"
  "        d[\"vocab_size\"] = len(ck.vocab)\n"
  "        d[\"vocab_path\"] = ck.vocab_path\n"
  "    return d\n"
  "\n"
  "\n"
  "# ====================================================================================================== output\n"
  "class SplitWriter:\n"
  "    \"\"\"train/dev/heldout JSONL files for one dataset.  Files are written under a .part name and only renamed into place when the\n"
  "    target does not exist; an existing target with a different sha256 is never touched (new content kept as .new-<sha8>).\"\"\"\n"
  "\n"
  "    SPLITS = (\"train\", \"dev\", \"heldout\")\n"
  "\n"
  "    def __init__(self, directory):\n"
  "        self.dir = Path(directory)\n"
  "        self.dir.mkdir(parents=True, exist_ok=True)\n"
  "        self.tmp = {sp: self.dir / f\"{sp}.jsonl.part-{os.getpid()}\" for sp in self.SPLITS}\n"
  "        self.fh = {sp: open(p, \"w\", encoding=\"utf-8\", newline=\"\\n\") for sp, p in self.tmp.items()}\n"
  "        self.counts: Counter = Counter()\n"
  "\n"
  "    def write(self, row: dict):\n"
  "        line = json.dumps(row, ensure_ascii=False)\n"
  "        line = line.replace(\"\\u2028\", \"\\\\u2028\").replace(\"\\u2029\", \"\\\\u2029\").replace(\"\\x85\", \"\\\\u0085\")\n"
  "        self.write_line(row[\"split\"], line)\n"
  "\n"
  "    def write_line(self, split: str, line: str):\n"
  "        \"\"\"One already serialised row (used when shards are merged).\"\"\"\n"
  "        self.fh[split].write(line.rstrip(\"\\n\") + \"\\n\")\n"
  "        self.counts[split] += 1\n"
  "\n"
  "    @staticmethod\n"
  "    def _free(path: Path) -> Path:\n"
  "        \"\"\"`path`, or path.1, path.2, ... : the first name that does not exist (existing files are never replaced).\"\"\"\n"
  "        cand, i = path, 0\n"
  "        while cand.exists():\n"
  "            i += 1\n"
  "            cand = path.with_name(path.name + f\".{i}\")\n"
  "        return cand\n"
  "\n"
  "    def close(self) -> dict:\n"
  "        info = {}\n"
  "        for sp in self.SPLITS:\n"
  "            self.fh[sp].close()\n"
  "        for sp in self.SPLITS:\n"
  "            tmp = self.tmp[sp]\n"
  "            data_sha = hashlib.sha256()\n"
  "            with open(tmp, \"rb\") as f:\n"
  "                for chunk in iter(lambda: f.read(1 << 20), b\"\"):\n"
  "                    data_sha.update(chunk)\n"
  "            sha = data_sha.hexdigest()\n"
  "            target = self.dir / f\"{sp}.jsonl\"\n"
  "            entry = {\"rows\": self.counts[sp], \"bytes\": tmp.stat().st_size, \"sha256\": sha}\n"
  "            if not target.exists():\n"
  "                tmp.rename(target)\n"
  "                entry[\"status\"] = \"written\"\n"
  "            else:\n"
  "                old = hashlib.sha256(target.read_bytes()).hexdigest()\n"
  "                if old == sha:\n"
  "                    entry[\"status\"] = \"identical_existing\"\n"
  "                    tmp.unlink()                                   # only this run's own temp file; the existing target is untouched\n"
  "                else:\n"
  "                    kept = self._free(self.dir / f\"{sp}.jsonl.new-{sha[:8]}\")\n"
  "                    tmp.rename(kept)\n"
  "                    entry[\"status\"] = \"REFUSED_existing_file_differs\"\n"
  "                    entry[\"kept_as\"] = kept.name\n"
  "            info[f\"{sp}.jsonl\"] = entry\n"
  "        return info\n"
  "\n"
  "\n"
  "def _write_new(path: Path, text: str) -> str:\n"
  "    \"\"\"Create `path` with `text`; never overwrite a file with different content.\"\"\"\n"
  "    data = text.encode(\"utf-8\")\n"
  "    if path.exists():\n"
  "        if path.read_bytes() == data:\n"
  "            return \"identical_existing\"\n"
  "        alt = SplitWriter._free(path.with_name(path.name + \".new-\" + hashlib.sha256(data).hexdigest()[:8]))\n"
  "        _atomic_write(alt, data)\n"
  "        return \"REFUSED_existing_file_differs:\" + alt.name\n"
  "    _atomic_write(path, data)\n"
  "    return \"written\"\n"
  "\n"
  "\n"
  "def _atomic_write(path: Path, data: bytes) -> None:\n"
  "    \"\"\"Write to <path>.part-<pid> and rename: the file never exists under its final name half written (a runtime killed in the middle leaves a\n"
  "    .part-<pid>, which nothing reads; the pipeline treats manifest.json as the 'finished' marker, so it is written last, see run_dataset).\"\"\"\n"
  "    part = path.with_name(path.name + f\".part-{os.getpid()}\")\n"
  "    with open(part, \"wb\") as f:\n"
  "        f.write(data)\n"
  "        f.flush()\n"
  "    os.replace(part, path)\n"
  "\n"
  "\n"
  "def format_dump(rows) -> str:\n"
  "    out = []\n"
  "    for r in rows:\n"
  "        out.append(\"=\" * 100)\n"
  "        out.append(f\"{r['id']}  family={r['family']}  split={r['split']}  len={len(r['full_text'])}  adoption={r['adoption']}\")\n"
  "        out.append(\"-\" * 40 + \" PROMPT\")\n"
  "        out.append(r[\"prompt\"])\n"
  "        out.append(\"-\" * 40 + \" TARGET   (loss_mask_spans=\" + json.dumps(r[\"loss_mask_spans\"]) + \")\")\n"
  "        out.append(r[\"target\"])\n"
  "        out.append(\"\")\n"
  "    return \"\\n\".join(out)\n"
  "\n"
  "\n"
  "def subdir_for(raw_dir, key: str) -> Path:\n"
  "    spec = DATASETS[key]\n"
  "    root = Path(raw_dir)\n"
  "    for cand in (root / key, root / spec.hf.replace(\"/\", \"__\"), root / spec.hf.split(\"/\")[1], root / spec.hf):\n"
  "        if cand.is_dir():\n"
  "            return cand\n"
  "    return root\n"
  "\n"
  "\n"
  "def load_provenance(raw_dir):\n"
  "    \"\"\"provenance.json written by acquire.py next to the dataset's files (None if absent / unreadable).\"\"\"\n"
  "    p = Path(raw_dir) / \"provenance.json\"\n"
  "    if p.is_file():\n"
  "        try:\n"
  "            d = json.loads(p.read_text(encoding=\"utf-8\"))\n"
  "            return d if isinstance(d, dict) and isinstance(d.get(\"files\"), list) else None\n"
  "        except ValueError:\n"
  "            return None\n"
  "    return None\n"
  "\n"
  "\n"
  "def apply_provenance(spec: Spec, prov: dict) -> Spec:\n"
  "    \"\"\"Row source / license from provenance['row_defaults'] and the attribution lines (acquire.py's rules for converters).\"\"\"\n"
  "    rd = prov.get(\"row_defaults\") or {}\n"
  "    lines = ((prov.get(\"attribution\") or {}).get(\"lines\")) or []\n"
  "    return dc_replace(spec, source=str(rd.get(\"source\") or spec.source), license=str(rd.get(\"license\") or spec.license),\n"
  "                      attribution=\" | \".join(str(x) for x in lines) or spec.attribution)\n"
  "\n"
  "\n"
  "def train_only(files, raw_dir, prov: dict):\n"
  "    \"\"\"(kept, dropped): only files that provenance lists with role == train and may_train (acquire.train_files' rule).\"\"\"\n"
  "    ok = {f[\"dest_rel\"] for f in prov[\"files\"] if f.get(\"role\") == \"train\" and f.get(\"may_train\")}\n"
  "    kept, dropped = [], []\n"
  "    for fp in files:\n"
  "        rel = \"/\".join(Path(fp).relative_to(raw_dir).parts)\n"
  "        (kept if any(d == rel or d.endswith(\"/\" + rel) for d in ok) else dropped).append(fp)\n"
  "    return kept, dropped\n"
  "\n"
  "\n"
  "def stats_state(stats: Stats) -> dict:\n"
  "    \"\"\"The raw counters of a Stats (JSON-able), stored in every manifest so that merge_shards can add shards up exactly.\"\"\"\n"
  "    return {\"n_in\": stats.n_in, \"n_out\": stats.n_out, \"rejects\": dict(stats.rejects), \"families\": dict(stats.families), \"splits\": dict(stats.splits),\n"
  "            \"family_splits\": {k: dict(v) for k, v in stats.family_splits.items()}, \"extra\": dict(stats.extra),\n"
  "            \"reject_examples\": stats.reject_examples}\n"
  "\n"
  "\n"
  "def run_dataset(key: str, raw_dir, out_dir, ck: Checks, limit=None, sample_dump: int = 100, seed: int = 20261109, start: int = 0) -> dict:\n"
  "    \"\"\"Convert one dataset (or, with start / limit, the records start <= index < limit: one shard).  The input is the TRAIN files only.\"\"\"\n"
  "    spec = DATASETS[key]\n"
  "    files = spec.find(raw_dir)\n"
  "    prov = load_provenance(raw_dir)\n"
  "    not_train = []\n"
  "    if prov:\n"
  "        spec = apply_provenance(spec, prov)\n"
  "        files, not_train = train_only(files, raw_dir, prov)\n"
  "    if not files:\n"
  "        raise FileNotFoundError(f\"{key}: no train-eligible raw files under {raw_dir}\")\n"
  "    stats = Stats(key)\n"
  "    if key == \"hotpot\":\n"
  "        ck.hotpot_roots = build_hotpot_roots(files)               # group key = component of shared supporting paragraphs (all train records)\n"
  "    rows = spec.convert(spec.reader(files, limit, start), stats, ck, spec)\n"
  "    writer = SplitWriter(Path(out_dir) / key)\n"
  "    heap, tie = [], 0\n"
  "    for row in rows:\n"
  "        writer.write(row)\n"
  "        if sample_dump:\n"
  "            tie += 1\n"
  "            heapq.heappush(heap, (-h(\"dump\", seed, row[\"id\"]), tie, row))\n"
  "            if len(heap) > sample_dump:\n"
  "                heapq.heappop(heap)\n"
  "    info = writer.close()\n"
  "    summary = summarize(stats, ck)\n"
  "    summary[\"raw_files\"] = [{\"name\": Path(p).relative_to(Path(raw_dir)).as_posix(), \"bytes\": p.stat().st_size} for p in files]\n"
  "    summary[\"provenance\"] = {\"used\": bool(prov), \"commit_sha\": (prov or {}).get(\"commit_sha\"),\n"
  "                             \"skipped_not_train_files\": [Path(p).relative_to(Path(raw_dir)).as_posix() for p in not_train]}\n"
  "    summary[\"row_source\"], summary[\"row_license\"] = spec.source, spec.license\n"
  "    summary[\"limit\"] = limit if not start else None\n"
  "    summary[\"range\"] = {\"start\": int(start), \"stop\": limit}\n"
  "    manifest = dict(summary, files={n: {k: v for k, v in e.items() if k in (\"rows\", \"bytes\", \"sha256\")} for n, e in info.items()},\n"
  "                    stats_state=stats_state(stats))\n"
  "    d = Path(out_dir) / key\n"
  "    summary[\"files\"] = info                                       # per-run statuses (written / identical_existing / REFUSED) stay out of the manifest\n"
  "    statuses = {}\n"
  "    if sample_dump:\n"
  "        dump_rows = [r for _, _, r in sorted(heap, key=lambda t: (-t[0], t[1]))]\n"
  "        statuses[\"sample_dump.txt\"] = _write_new(d / \"sample_dump.txt\", format_dump(dump_rows))\n"
  "    ex = [{\"reason\": k, **e} for k, v in stats.reject_examples.items() for e in v]\n"
  "    statuses[\"rejects_sample.jsonl\"] = _write_new(d / \"rejects_sample.jsonl\", \"\".join(json.dumps(e, ensure_ascii=False) + \"\\n\" for e in ex))\n"
  "    statuses[\"manifest.json\"] = _write_new(d / \"manifest.json\", json.dumps(manifest, ensure_ascii=False, indent=1))      # LAST: the marker of 'finished'\n"
  "    summary[\"write_status\"] = statuses\n"
  "    refused = any(v[\"status\"].startswith(\"REFUSED\") for v in info.values()) or any(s.startswith(\"REFUSED\") for s in statuses.values())\n"
  "    summary[\"refused_overwrite\"] = refused\n"
  "    return summary\n"
  "\n"
  "\n"
  "def merge_shards(key: str, shard_roots, out_dir, sample_dump: int = 100, seed: int = 20261109) -> dict:\n"
  "    \"\"\"Join the shards of one dataset into the folder a single run would have produced.  shard_roots: the --out-dir of every shard (each holds\n"
  "    <key>/ with train/dev/heldout.jsonl and manifest.json), in any order; the shards must tile the record range 0..end without a gap or an\n"
  "    overlap.  Rows are concatenated in record order, the counters of the manifests are added (so n_in == n_out + rejects still holds), the\n"
  "    100-row reading dump is drawn again over the merged rows, and the merged files are written next to nothing else under out_dir/<key>/\n"
  "    (never over an existing file).\"\"\"\n"
  "    mans = []\n"
  "    for r in shard_roots:\n"
  "        mp = Path(r) / key / \"manifest.json\"\n"
  "        m = json.loads(mp.read_text(encoding=\"utf-8\"))\n"
  "        rg = m.get(\"range\") or {\"start\": 0, \"stop\": m.get(\"limit\")}\n"
  "        mans.append((int(rg[\"start\"] or 0), rg[\"stop\"], Path(r) / key, m))\n"
  "    mans.sort(key=lambda t: t[0])\n"
  "    if not mans:\n"
  "        raise ValueError(\"merge_shards: no shards\")\n"
  "    if mans[0][0] != 0:\n"
  "        raise ValueError(f\"merge_shards: the first shard starts at record {mans[0][0]}, not 0\")\n"
  "    for (s0, e0, _, _), (s1, _, _, _) in zip(mans, mans[1:]):\n"
  "        if e0 is None or int(e0) != s1:\n"
  "            raise ValueError(f\"merge_shards: gap or overlap between a shard ending at {e0} and the next starting at {s1}\")\n"
  "    if mans[-1][1] is not None:\n"
  "        raise ValueError(\"merge_shards: the last shard must run to the end of the dataset (stop = None)\")\n"
  "    first = mans[0][3]\n"
  "    stats = Stats(key)\n"
  "    for _, _, _, m in mans:\n"
  "        st = m[\"stats_state\"]\n"
  "        if m.get(\"splits\") is None or not m.get(\"balanced\"):\n"
  "            raise ValueError(\"merge_shards: a shard manifest is not balanced\")\n"
  "        stats.n_in += st[\"n_in\"]\n"
  "        stats.n_out += st[\"n_out\"]\n"
  "        stats.rejects.update(st[\"rejects\"])\n"
  "        stats.families.update(st[\"families\"])\n"
  "        stats.splits.update(st[\"splits\"])\n"
  "        for fam, c in st[\"family_splits\"].items():\n"
  "            stats.family_splits[fam].update(c)\n"
  "        stats.extra.update(st[\"extra\"])\n"
  "        for reason, exs in st[\"reject_examples\"].items():\n"
  "            have = stats.reject_examples.setdefault(reason, [])\n"
  "            have.extend(exs[:max(0, Stats.MAX_EXAMPLES - len(have))])\n"
  "    writer = SplitWriter(Path(out_dir) / key)\n"
  "    heap, tie = [], 0\n"
  "    for _, _, d, _ in mans:\n"
  "        for sp in SplitWriter.SPLITS:\n"
  "            with open(d / f\"{sp}.jsonl\", encoding=\"utf-8\", newline=\"\\n\") as f:\n"
  "                for line in f:\n"
  "                    if not line.strip():\n"
  "                        continue\n"
  "                    writer.write_line(sp, line)\n"
  "                    if sample_dump:\n"
  "                        row = json.loads(line)\n"
  "                        tie += 1\n"
  "                        heapq.heappush(heap, (-h(\"dump\", seed, row[\"id\"]), tie, row))\n"
  "                        if len(heap) > sample_dump:\n"
  "                            heapq.heappop(heap)\n"
  "    info = writer.close()\n"
  "    summary = summarize(stats, None)\n"
  "    for k in (\"backends\", \"max_chars\", \"vocab_size\", \"vocab_path\", \"raw_files\", \"provenance\", \"row_source\", \"row_license\"):\n"
  "        if k in first:\n"
  "            summary[k] = first[k]\n"
  "    summary[\"limit\"] = None\n"
  "    summary[\"range\"] = {\"start\": 0, \"stop\": None, \"shards\": [[a, b] for a, b, _, _ in mans]}\n"
  "    manifest = dict(summary, files={n: {k: v for k, v in e.items() if k in (\"rows\", \"bytes\", \"sha256\")} for n, e in info.items()},\n"
  "                    stats_state=stats_state(stats))\n"
  "    d = Path(out_dir) / key\n"
  "    summary[\"files\"] = info\n"
  "    statuses = {}\n"
  "    if sample_dump:\n"
  "        statuses[\"sample_dump.txt\"] = _write_new(d / \"sample_dump.txt\", format_dump([r for _, _, r in sorted(heap, key=lambda t: (-t[0], t[1]))]))\n"
  "    ex = [{\"reason\": k, **e} for k, v in stats.reject_examples.items() for e in v]\n"
  "    statuses[\"rejects_sample.jsonl\"] = _write_new(d / \"rejects_sample.jsonl\", \"\".join(json.dumps(e, ensure_ascii=False) + \"\\n\" for e in ex))\n"
  "    statuses[\"manifest.json\"] = _write_new(d / \"manifest.json\", json.dumps(manifest, ensure_ascii=False, indent=1))      # LAST: the marker of 'finished'\n"
  "    summary[\"write_status\"] = statuses\n"
  "    summary[\"refused_overwrite\"] = any(v[\"status\"].startswith(\"REFUSED\") for v in info.values()) or any(x.startswith(\"REFUSED\") for x in statuses.values())\n"
  "    return summary\n"
  "\n"
  "\n"
  "@contextlib.contextmanager\n"
  "def forward_termination_to_sandbox(ext, posix: bool = os.name == \"posix\"):\n"
  "    \"\"\"While a converter runs, SIGTERM / SIGINT (the pipeline's _terminate, Colab's 'Interrupt execution') and a normal exit also kill the\n"
  "    sandbox child that is running: each child is a session leader, so the signal sent to the converter's process group does not reach it.\n"
  "    POSIX only (nothing happens elsewhere); the previous handlers are restored on exit.  Returns True when the handlers are installed.\"\"\"\n"
  "    if not posix or ext is None or not callable(getattr(ext, \"kill_live_sandbox_children\", None)):\n"
  "        yield False\n"
  "        return\n"
  "    old = {}\n"
  "\n"
  "    def handler(signum, _frame):\n"
  "        ext.kill_live_sandbox_children()\n"
  "        raise SystemExit(128 + int(signum))\n"
  "\n"
  "    for sig in (signal.SIGTERM, signal.SIGINT):\n"
  "        try:\n"
  "            old[sig] = signal.signal(sig, handler)\n"
  "        except (ValueError, OSError):                            # not the main thread / not allowed: the atexit net below still applies\n"
  "            pass\n"
  "    atexit.register(ext.kill_live_sandbox_children)\n"
  "    try:\n"
  "        yield True\n"
  "    finally:\n"
  "        atexit.unregister(ext.kill_live_sandbox_children)\n"
  "        ext.kill_live_sandbox_children()\n"
  "        for sig, h in old.items():\n"
  "            try:\n"
  "                signal.signal(sig, h)\n"
  "            except (ValueError, OSError):\n"
  "                pass\n"
  "\n"
  "\n"
  "def main(argv=None) -> int:\n"
  "    ap = argparse.ArgumentParser(description=\"GeoCPU v49 acquisition converters\")\n"
  "    ap.add_argument(\"--dataset\", required=True, help=\"|\".join(DATASETS) + \"|all\")\n"
  "    ap.add_argument(\"--raw-dir\", required=True)\n"
  "    ap.add_argument(\"--out-dir\", required=True)\n"
  "    ap.add_argument(\"--limit\", type=int, default=None, help=\"stop at record index N (exclusive) per dataset: the first N records\")\n"
  "    ap.add_argument(\"--start\", type=int, default=0, help=\"first record index (with --limit E: the shard S <= i < E)\")\n"
  "    ap.add_argument(\"--vocab\", default=None, help=\"base_metadata.json (default: search, then checks.load_vocab on --drive-root)\")\n"
  "    ap.add_argument(\"--drive-root\", default=None, help=\"Drive folder that holds the model bundle (for checks.load_vocab)\")\n"
  "    ap.add_argument(\"--max-chars\", type=int, default=DEFAULT_CAP)\n"
  "    ap.add_argument(\"--sample-dump\", type=int, default=100, help=\"seeded readable sample of N emitted rows per dataset (0 = off)\")\n"
  "    ap.add_argument(\"--no-sandbox\", action=\"store_true\", help=\"do not execute python (coding_pytool rows are then rejected)\")\n"
  "    ap.add_argument(\"--require-os-containment\", action=\"store_true\",\n"
  "                    help=\"the sandbox child must run as an unprivileged user (Colab: root -> nobody); otherwise the self-test fails\")\n"
  "    ap.add_argument(\"--sandbox-timeout\", type=float, default=5.0)\n"
  "    ap.add_argument(\"--no-external-checks\", action=\"store_true\", help=\"ignore a sibling checks.py (then --no-sandbox is required: no code can run)\")\n"
  "    ap.add_argument(\"--allow-unverified-python\", action=\"store_true\",\n"
  "                    help=\"coding: ALSO keep python rows that ran but have no stored output to compare with (default: dropped and counted)\")\n"
  "    ap.add_argument(\"--no-tengentoppa-steps\", action=\"store_true\", help=\"tengentoppa: never include the 'reasoning' field\")\n"
  "    a = ap.parse_args(argv)\n"
  "    keys = list(DATASETS) if a.dataset == \"all\" else [a.dataset]\n"
  "    for k in keys:\n"
  "        if k not in DATASETS:\n"
  "            print(\"unknown dataset:\", k, file=sys.stderr)\n"
  "            return 2\n"
  "    try:\n"
  "        ck = Checks(vocab_path=a.vocab, drive_root=a.drive_root, max_chars=a.max_chars, run_code=not a.no_sandbox, sandbox_timeout=a.sandbox_timeout,\n"
  "                    use_external=not a.no_external_checks, python_require_stored_output=not a.allow_unverified_python,\n"
  "                    tengentoppa_steps=not a.no_tengentoppa_steps, require_os_containment=a.require_os_containment)\n"
  "    except SandboxUnavailable as e:\n"
  "        print(f\"SANDBOX UNAVAILABLE: {e}\", file=sys.stderr)\n"
  "        return 4\n"
  "    code = 0\n"
  "    allsum = {}\n"
  "    for k in keys:\n"
  "        raw = subdir_for(a.raw_dir, k) if a.dataset == \"all\" else Path(a.raw_dir)\n"
  "        try:\n"
  "            with forward_termination_to_sandbox(ck._ext if ck.run_code else None):\n"
  "                s = run_dataset(k, raw, a.out_dir, ck, limit=a.limit, sample_dump=a.sample_dump, start=a.start)\n"
  "        except FileNotFoundError as e:\n"
  "            print(f\"[{k}] {e}\", file=sys.stderr)\n"
  "            code = max(code, 2)\n"
  "            continue\n"
  "        except SandboxUnavailable as e:                          # the sandbox broke in the middle of the run: no manifest is written for this dataset\n"
  "            print(f\"[{k}] SANDBOX FAILED DURING THE RUN: {e}\", file=sys.stderr)\n"
  "            return 4\n"
  "        allsum[k] = {x: s[x] for x in (\"n_in\", \"n_out\", \"keep_rate\", \"reject_total\", \"balanced\", \"families\", \"splits\", \"refused_overwrite\")}\n"
  "        print(json.dumps({k: allsum[k]}, ensure_ascii=False))\n"
  "        if s[\"refused_overwrite\"]:\n"
  "            code = 3\n"
  "    if len(keys) > 1:\n"
  "        _write_new(Path(a.out_dir) / \"convert_summary.json\", json.dumps(allsum, ensure_ascii=False, indent=1))\n"
  "    return code\n"
  "\n"
  "\n"
  "if __name__ == \"__main__\":\n"
  "    sys.exit(main())\n"
 ),
 "pipeline.py": (
  "#!/usr/bin/env python3\n"
  "\"\"\"pipeline.py - the steps of colab_acquire_check.ipynb as plain functions, so that they can be unit-tested locally on tiny fixtures.\n"
  "\n"
  "Runs on a Google Colab CPU runtime (12 GB RAM, no GPU).  Glue only: the work is done by acquire.py (download + verify),\n"
  "checks.py (OOV, length, duplicates, decontamination, sandbox) and convert.py (one converter per dataset).  Standard library only\n"
  "(+ pyarrow / numpy that the Colab image has).  Nothing here needs a secret, a GPU or a connection other than the one acquire.py opens\n"
  "to huggingface.co.\n"
  "\n"
  "Every step function prints one line   PASS [step] detail   or   FAIL [step] reason   and raises StepFailed (an AssertionError) on FAIL,\n"
  "so a notebook cell that calls it stops the run; the cells additionally `assert R[\"pass\"]`.\n"
  "\n"
  "Writing rules (the owner's): only NEW folders on Drive; a file that exists with a different sha256 is never overwritten (the step FAILs\n"
  "and says to set RUN_SUFFIX so that a fresh output folder is used); nothing is ever deleted on Drive.  Everything heavy happens on the\n"
  "Colab local disk (cfg.work, default /content/v49_work), which is scratch.  Every write to Drive goes to `<name>.part` first and is renamed\n"
  "into place only when the target still does not exist; cfg.drive_sync (the notebook: flush_and_unmount + mount) runs after every acquired\n"
  "dataset (before its provenance.json is written) and after steps 5, 7b, 8, 8b and 9, and steps 5 and 9 re-read what they list from the fresh mount.\n"
  "\n"
  "Flow (the numbers are the notebook's cells):\n"
  "  3c Drive remount probe -> 3d sandbox self-test (only when 'coding' is converted; Drive still mounted: proves the OS barrier) -> 4 dry run ->\n"
  "  5 acquire -> 6a local mirror of the train files -> 6b vocabulary + eval index -> 6c raw checks (counts vs the cards, OOV, length, 100-row\n"
  "  reading dumps) -> 7a converters on the first 2,000 records -> 7b converters on everything (in waves; Drive is unmounted while untrusted code\n"
  "  runs; coding and tengentoppa run in shards that are published one by one, so a disconnect loses at most one shard) -> 8 filters (exact +\n"
  "  MinHash near duplicates, 13-gram decontamination, old corpus, near siblings of train inside dev / heldout) -> 8b optional reviews with a\n"
  "  time budget -> 9 final rows, registry.json, REPORT_JA.md.\n"
  "Re-running a cell after a disconnect never conflicts with what the first run saved: states that hold volatile numbers (7a) are returned as they\n"
  "are, files that are rendered (registry.json, REPORT_JA.md) are rendered from the JSON round trip so that they are byte-identical in a new process.\n"
  "\"\"\"\n"
  "from __future__ import annotations\n"
  "\n"
  "import bisect\n"
  "import collections\n"
  "import contextlib\n"
  "import dataclasses\n"
  "import datetime\n"
  "import functools\n"
  "import hashlib\n"
  "import inspect\n"
  "import io\n"
  "import itertools\n"
  "import json\n"
  "import math\n"
  "import os\n"
  "import platform\n"
  "import shutil\n"
  "import signal\n"
  "import subprocess\n"
  "import sys\n"
  "import time\n"
  "import traceback\n"
  "import zipfile\n"
  "from pathlib import Path\n"
  "from typing import Any, Callable, Dict, List, Optional\n"
  "\n"
  "import acquire\n"
  "import checks\n"
  "import convert\n"
  "\n"
  "PIPELINE_VERSION = \"2\"\n"
  "OUT_NAME = \"v49_acquire_20261109\"\n"
  "ROWS_NAME = \"external_data_v49_rows\"\n"
  "STAGE_A_NAME = \"external_data_v49_stageA\"            # unfiltered converted rows; NOT tied to RUN_SUFFIX (a filter-only re-run must not reconvert)\n"
  "HOLD_DIRNAME = \"_HOLD_do_not_mix\"                    # rows of datasets whose adoption is on hold live here, never next to the candidates\n"
  "HOLD_KEYS = (\"toolace\", \"wizard\")\n"
  "CODE_EXEC_KEYS = (\"coding\",)                        # the only dataset whose converter RUNS code (python rows); the others never need the sandbox\n"
  "LEN_BUCKET = 64                                      # registry.json length histogram: 64-character buckets (the training plan's bucket width)\n"
  "GROUP_INDEPENDENT = (\"jaquad\", \"hotpot\")             # datasets whose dev / heldout are split by a natural group\n"
  "USER_EVAL_EXTRA = \"v49_eval_extra\"\n"
  "SPLITS = (\"train\", \"dev\", \"heldout\")\n"
  "KEYS = (\"coding\", \"tengentoppa\", \"jaquad\", \"hotpot\", \"toolace\", \"wizard\")                    # convert.py keys, report order\n"
  "CONVERT_ORDER = (\"coding\", \"tengentoppa\", \"hotpot\", \"wizard\", \"jaquad\", \"toolace\")           # slowest first\n"
  "ACQ_KEY = {\"coding\": \"coding\", \"tengentoppa\": \"tengentoppa\", \"jaquad\": \"jaquad\", \"hotpot\": \"hotpotqa\", \"toolace\": \"toolace\",\n"
  "           \"wizard\": \"wizardlm\"}\n"
  "EXPECTED_COUNTS = {\"coding\": 113796, \"tengentoppa\": 125000, \"jaquad\": 31748, \"hotpot\": 90447, \"toolace\": 11300, \"wizard\": 116422}\n"
  "COUNT_SOURCE = {\"coding\": \"dataset card: 113,796\", \"tengentoppa\": \"dataset card: 125,000\", \"jaquad\": \"owner's plan: train 31,748 (card: 39,696 in total)\",\n"
  "                \"hotpot\": \"dataset card: distractor train 90,447\", \"toolace\": \"task text: 11,300\", \"wizard\": \"dataset card: 116,422\"}\n"
  "KNOWN_VOCAB_SHA = \"8329446036ff708445cf08983267c2c599dae54b23efeb92be86129b153e8288\"        # sha256 of the sorted vocabulary characters\n"
  "V48_EVAL_NAMES = tuple(f\"{fam}_unseen_eval.jsonl\" for fam in (\"evidence\", \"japanese\", \"logic\", \"python\"))\n"
  "R_EXACT, R_NEAR, R_EVAL, R_OLD_EXACT, R_OLD_NEAR, R_SPLIT_LEAK = 1, 2, 3, 4, 5, 6\n"
  "REASON = {R_EXACT: \"exact_duplicate_within_dataset\", R_NEAR: \"near_duplicate_within_dataset\", R_EVAL: \"eval_overlap_13gram\",\n"
  "          R_OLD_EXACT: \"exact_duplicate_of_old_corpus\", R_OLD_NEAR: \"near_duplicate_of_old_corpus\",\n"
  "          R_SPLIT_LEAK: \"dev_heldout_row_with_a_near_sibling_in_train\"}\n"
  "STAGE_A_FILES = (\"train.jsonl\", \"dev.jsonl\", \"heldout.jsonl\", \"manifest.json\", \"sample_dump.txt\", \"rejects_sample.jsonl\")\n"
  "SHARD_FILES = (\"train.jsonl\", \"dev.jsonl\", \"heldout.jsonl\", \"manifest.json\", \"rejects_sample.jsonl\")\n"
  "FUNNEL = (\"converted\", \"after_exact_dedup\", \"after_near_dedup\", \"after_eval_decontam\", \"after_old_corpus_dedup\", \"after_split_hygiene\")\n"
  "BASE_NOT_COVERED = [\n"
  "    \"GSM8K / MATH test sets are not on Drive and were not downloaded (the owner approved exactly six datasets): \"\n"
  "    \"they are NOT part of the decontamination.  Put them as .jsonl/.json under <Drive root>/\" + USER_EVAL_EXTRA + \"/ and re-run \"\n"
  "    \"step 6b onwards (stage A is kept: a new RUN_SUFFIX only redoes the filters).\",\n"
  "    \"Semantic (paraphrase) overlap is only looked at, not removed: the character-bigram look-alike search is lexical and not exhaustive.\",\n"
  "    \"HotpotQA validation is indexed by its questions only (its passages are Wikipedia paragraphs shared with train by construction).\",\n"
  "    \"Upstream model terms (Nemotron-4, Phi-3, Mixtral, calm3, Qwen2.5, WizardLM-2) were NOT checked: only the dataset licences were.\",\n"
  "    \"Near-duplicate removal is a MinHash estimate: it reliably removes pairs of Jaccard >= 0.85, usually those near 0.8, and misses looser \"\n"
  "    \"paraphrases (see the cut and the detection table in the report).\",\n"
  "    \"Eval sentences shorter than 13 normalised characters (3 of the 589 T5 sentences) cannot form a 13-gram: a row that merely CONTAINS one of them \"\n"
  "    \"is not detected.  Eval texts of 40+ normalised characters are also matched as whole items (templated items that differ only in digits).\",\n"
  "    \"Eval texts of 13 to 39 normalised characters are matched only through their 13-grams: when ALL of such a text's 13-grams are template phrases \"\n"
  "    \"(shared by 20+ eval items) a verbatim copy of it is NOT flagged (an independent check measured this for the v33 prompt template, 50 items, and \"\n"
  "    \"one v42 question).  The thresholds (13 / 40 / 20) were left as they are because false positives cannot be measured without the real rows.  \"\n"
  "    \"The converse also holds: a template phrase that is itself an eval text of 40+ characters flags EVERY row that contains it, so read \"\n"
  "    \"rows_flagged_whole_item per dataset in the Colab report (section 2, and registry.json decontam.rows_flagged_whole_item_by_dataset) before trusting the drop counts.\",\n"
  "    \"Rows that import numpy / pandas / sympy are dropped by the sandbox (its allowed imports are the nine modules of the model's executor): a yield \"\n"
  "    \"loss, not a safety issue.\",\n"
  "]\n"
  "\n"
  "\n"
  "class StepFailed(AssertionError):\n"
  "    pass\n"
  "\n"
  "\n"
  "def require(cond, msg: str) -> None:\n"
  "    if not cond:\n"
  "        raise StepFailed(msg)\n"
  "\n"
  "\n"
  "def sha256_text(text: str) -> str:\n"
  "    return hashlib.sha256(text.encode(\"utf-8\")).hexdigest()\n"
  "\n"
  "\n"
  "def jdump(obj) -> str:\n"
  "    return json.dumps(obj, ensure_ascii=False, indent=1, sort_keys=True) + \"\\n\"\n"
  "\n"
  "\n"
  "def not_covered(missing_optional: Optional[List[str]] = None) -> List[str]:\n"
  "    out = list(BASE_NOT_COVERED)\n"
  "    if missing_optional:\n"
  "        out.append(\"These frozen eval suites of the owner's registry were NOT found on Drive and are NOT part of the decontamination: \"\n"
  "                   + \"; \".join(missing_optional))\n"
  "    return out\n"
  "\n"
  "\n"
  "# ====================================================================================================== configuration\n"
  "@dataclasses.dataclass\n"
  "class Cfg:\n"
  "    root: str = acquire.DEFAULT_ROOT\n"
  "    suffix: str = \"\"                                    # RUN_SUFFIX: a fresh OUTPUT folder after a refusal (never overwrite); stage A is kept\n"
  "    raw_suffix: str = \"\"                                # RAW_SUFFIX: a fresh folder for the downloaded files (only after a refused raw file)\n"
  "    work: str = \"/content/v49_work\"                     # local scratch (NOT Drive)\n"
  "    only: Optional[tuple] = None                        # convert.py keys; None = all six\n"
  "    smoke_limit: int = 2000\n"
  "    stop_on_count_mismatch: bool = False\n"
  "    expected_counts: Optional[dict] = dataclasses.field(default_factory=lambda: dict(EXPECTED_COUNTS))\n"
  "    expected_old_rows: Optional[int] = checks.OLD_CORPUS_ROWS\n"
  "    old_corpus: Optional[str] = None\n"
  "    max_chars: int = convert.DEFAULT_CAP\n"
  "    near_threshold: float = 0.8                         # the Jaccard we want to remove ...\n"
  "    near_margin: float = 0.05                           # ... the MinHash estimate is cut this much lower (see checks.MinHasher)\n"
  "    split_hygiene_threshold: float = 0.5                # a dev / heldout row whose estimated Jaccard with a train row is >= this is dropped\n"
  "    run_split_hygiene: bool = True\n"
  "    ngram: int = 13\n"
  "    boilerplate_eval_df: int = 20                       # a shingle shared by this many EVAL items is template boilerplate (never the train-side count)\n"
  "    min_shared: int = 1\n"
  "    wizard_max_error_rate: float = 0.01                 # shown in the report next to the measured rate; it no longer decides anything\n"
  "    wizard_min_statements: int = 300\n"
  "    run_sandbox: bool = True\n"
  "    require_os_containment: Optional[bool] = None       # None = yes on Colab (the child must run as an unprivileged user), no elsewhere\n"
  "    python_allow_unverified: bool = False               # True also keeps python rows without a stored output (default: dropped and counted)\n"
  "    parallel: int = 2\n"
  "    shard_keys: tuple = (\"coding\", \"tengentoppa\")       # converted in shards of shard_records records, each published when it is done\n"
  "    shard_records: int = 20000\n"
  "    max_unsharded_minutes: float = 120.0                # 7b refuses to start an unsharded dataset estimated above this\n"
  "    poll_s: float = 15.0\n"
  "    heartbeat_s: float = 120.0\n"
  "    vocab_override: Optional[str] = None                # tests only: a base_metadata.json on the local disk\n"
  "    allow_vocab_mismatch: bool = False                  # True: a vocabulary whose character set differs from the one on the owner's PC is accepted\n"
  "    eval_extra_dir: Optional[str] = None                # the notebook writes the embedded v48 unseen_eval texts here\n"
  "    require_v48: bool = True\n"
  "    run_semantic: bool = True\n"
  "    semantic_max_queries: int = 500\n"
  "    semantic_min_cos: float = 0.5\n"
  "    semantic_min_shared_blocks: int = 5\n"
  "    semantic_max_block_df: int = 5\n"
  "    semantic_budget_s: float = 900.0\n"
  "    run_cross: bool = True\n"
  "    cross_budget_s: float = 900.0\n"
  "    check_disk: bool = True\n"
  "    allow_local: bool = False                           # tests only (acquire refuses real downloads off Colab)\n"
  "    stage_dir: Optional[str] = None\n"
  "    drive_detach: Optional[Callable] = None             # the notebook: flush_and_unmount -> Drive is not visible while untrusted code runs\n"
  "    drive_attach: Optional[Callable] = None             # the notebook: mount again\n"
  "    drive_sync: Optional[Callable] = None               # the notebook: flush_and_unmount + mount (writes really reach Drive)\n"
  "    log: Callable = print\n"
  "    progress: Optional[Callable] = None\n"
  "    state: dict = dataclasses.field(default_factory=dict)\n"
  "    timings: list = dataclasses.field(default_factory=list)\n"
  "\n"
  "    def __post_init__(self):\n"
  "        self.root = str(self.root)\n"
  "        if self.only is not None:\n"
  "            bad = [k for k in self.only if k not in KEYS]\n"
  "            if bad:\n"
  "                raise ValueError(f\"unknown dataset key(s) {bad}; known: {list(KEYS)}\")\n"
  "            self.only = tuple(self.only)\n"
  "        if (self.drive_detach is None) != (self.drive_attach is None):\n"
  "            raise ValueError(\"drive_detach and drive_attach go together\")\n"
  "\n"
  "    # --- which datasets\n"
  "    @property\n"
  "    def keys(self) -> tuple:\n"
  "        return KEYS if self.only is None else tuple(k for k in KEYS if k in self.only)\n"
  "\n"
  "    def acq_only(self):\n"
  "        return None if self.only is None else [ACQ_KEY[k] for k in self.keys]\n"
  "\n"
  "    @property\n"
  "    def near_cut(self) -> float:\n"
  "        return round(self.near_threshold - self.near_margin, 6)\n"
  "\n"
  "    @property\n"
  "    def containment_required(self) -> bool:\n"
  "        return acquire._in_colab() if self.require_os_containment is None else bool(self.require_os_containment)\n"
  "\n"
  "    # --- places\n"
  "    @property\n"
  "    def ext_dirname(self) -> str:\n"
  "        return acquire.EXTERNAL_DIRNAME + self.raw_suffix\n"
  "\n"
  "    @property\n"
  "    def ext(self) -> Path:\n"
  "        return Path(self.root) / self.ext_dirname\n"
  "\n"
  "    @property\n"
  "    def only_tag(self) -> str:\n"
  "        \"\"\"'' for a full run, '_only_<keys>' for a trial run (ONLY): a trial run has its OWN output folder, so that it never collides with the full\n"
  "        run that follows it with the same RUN_SUFFIX.  The converted rows and the final rows are shared (they are per dataset).\"\"\"\n"
  "        return \"\" if len(self.keys) == len(KEYS) else \"_only_\" + \"-\".join(self.keys)\n"
  "\n"
  "    @property\n"
  "    def needs_sandbox(self) -> bool:\n"
  "        \"\"\"True when this run converts a dataset that executes code ('coding') and RUN_SANDBOX is on: only then the sandbox self-test is needed.\"\"\"\n"
  "        return self.run_sandbox and any(k in CODE_EXEC_KEYS for k in self.keys)\n"
  "\n"
  "    @property\n"
  "    def out(self) -> Path:\n"
  "        return Path(self.root) / (OUT_NAME + self.suffix + self.only_tag)\n"
  "\n"
  "    @property\n"
  "    def stage_a_root(self) -> Path:\n"
  "        \"\"\"Folder of the unfiltered converted rows.  RAW_SUFFIX gives a fresh one (other raw bytes: the old converted rows are never looked up).\"\"\"\n"
  "        return Path(self.root) / (STAGE_A_NAME + self.raw_suffix)\n"
  "\n"
  "    @property\n"
  "    def rows(self) -> Path:\n"
  "        return Path(self.root) / (ROWS_NAME + self.suffix)\n"
  "\n"
  "    @property\n"
  "    def old_corpus_path(self) -> Path:\n"
  "        return Path(self.old_corpus) if self.old_corpus else Path(self.root) / checks.OLD_CORPUS_REL\n"
  "\n"
  "    @property\n"
  "    def mirror(self) -> Path:\n"
  "        return Path(self.work) / \"mirror\"\n"
  "\n"
  "    @property\n"
  "    def vocab_file(self) -> Path:\n"
  "        return Path(self.work) / \"vocab_chars.json\"\n"
  "\n"
  "    def code_hashes(self) -> dict:\n"
  "        return {Path(m.__file__).name: hashlib.sha256(Path(m.__file__).read_bytes().replace(b\"\\r\\n\", b\"\\n\")).hexdigest()\n"
  "                for m in (acquire, checks, convert, sys.modules[__name__])}\n"
  "\n"
  "    def code_tag(self) -> str:\n"
  "        h = self.code_hashes()\n"
  "        return sha256_text(\"\".join(h[k] for k in sorted(h)))[:8]\n"
  "\n"
  "    def converter_hashes(self) -> dict:\n"
  "        \"\"\"Hashes of everything that decides what a converted row looks like: convert.py, the sandbox (source of the static rules and of the\n"
  "        child, how it is launched, the executor profile), the runtime symbol normaliser and the vocabulary.  pipeline.py, acquire.py and the\n"
  "        filter code of checks.py are NOT in it: changing them must not throw the (hours-long) converted rows away.\"\"\"\n"
  "        def norm(t: str) -> str:\n"
  "            return t.replace(\"\\r\\n\", \"\\n\")\n"
  "        sb = hashlib.sha256()\n"
  "        for part in (checks._VALIDATE_SRC, checks._CHILD_SRC, repr(sorted(checks.SANDBOX_PROFILES.items())), inspect.getsource(checks._spawn),\n"
  "                     inspect.getsource(checks.code_sandbox_run), inspect.getsource(checks._resolve_run_as), inspect.getsource(checks.static_check),\n"
  "                     checks.SANDBOX_BACKEND, repr(sorted(checks.RUNTIME_NORMALIZE.items()))):\n"
  "            sb.update(norm(part).encode(\"utf-8\"))\n"
  "            sb.update(b\"\\0\")\n"
  "        vocab = \"none\"\n"
  "        if self.vocab_file.is_file():\n"
  "            try:\n"
  "                vocab = hashlib.sha256(\"\".join(sorted(json.loads(self.vocab_file.read_text(encoding=\"utf-8\"))[\"vocab\"])).encode(\"utf-8\")).hexdigest()\n"
  "            except (ValueError, KeyError, OSError):\n"
  "                vocab = \"unreadable\"\n"
  "        params = {\"max_chars\": self.max_chars, \"python_allow_unverified\": self.python_allow_unverified, \"run_sandbox\": self.run_sandbox,\n"
  "                  \"python\": \"%d.%d\" % sys.version_info[:2]}              # what changes a converted row without being in a file: the cap, the python policy\n"
  "        return {\"convert.py\": hashlib.sha256(Path(convert.__file__).read_bytes().replace(b\"\\r\\n\", b\"\\n\")).hexdigest(),\n"
  "                \"checks.sandbox_and_normaliser\": sb.hexdigest(), \"vocabulary_chars\": vocab, \"parameters\": sha256_text(json.dumps(params, sort_keys=True))}\n"
  "\n"
  "    def raw_binding(self, key: str) -> dict:\n"
  "        \"\"\"What the unfiltered rows of `key` are made from: the pinned commit and the sha256 of every train file (from provenance.json).  Stage A and\n"
  "        the shards record it and refuse to be reused for other raw data (the converter tag alone does not know the raw bytes).\"\"\"\n"
  "        prov = acquire._provenance_of(self.root, _ds(key), self.ext_dirname)\n"
  "        return {\"commit_sha\": prov[\"commit_sha\"],\n"
  "                \"train_sha256\": sorted([f[\"dest_rel\"], f[\"sha256\"]] for f in prov[\"files\"] if f[\"role\"] == \"train\" and f.get(\"may_train\"))}\n"
  "\n"
  "    def converter_tag(self) -> str:\n"
  "        h = self.converter_hashes()\n"
  "        return sha256_text(\"\".join(h[k] for k in sorted(h)))[:10]\n"
  "\n"
  "    def conv_dir(self, key: str, smoke: bool = False) -> Path:\n"
  "        return Path(self.work) / ((\"smoke_\" if smoke else \"converted_\") + self.converter_tag()) / key\n"
  "\n"
  "    def shard_name(self, a: int, b: Optional[int]) -> str:\n"
  "        return f\"{a:09d}_{'end' if b is None else format(b, '09d')}\"\n"
  "\n"
  "    def shard_root(self, key: str, a: int, b: Optional[int]) -> Path:\n"
  "        \"\"\"--out-dir of one shard job (the rows are in <shard_root>/<key>/).\"\"\"\n"
  "        return Path(self.work) / (\"converted_\" + self.converter_tag()) / \"_shards\" / f\"{key}_{self.shard_name(a, b)}\"\n"
  "\n"
  "    def stage_a_drive(self, key: str) -> Path:\n"
  "        return self.stage_a_root / self.converter_tag() / key\n"
  "\n"
  "    def shard_drive(self, key: str, a: int, b: Optional[int]) -> Path:\n"
  "        return self.stage_a_root / self.converter_tag() / f\"{key}__shards\" / self.shard_name(a, b)\n"
  "\n"
  "    def mirror_dir(self, key: str) -> Path:\n"
  "        return self.mirror / acquire.folder_of(_ds(key)[\"repo\"])\n"
  "\n"
  "\n"
  "def make_cfg(**kw) -> Cfg:\n"
  "    \"\"\"Cfg with a progress printer that writes to stdout (visible in the notebook).\"\"\"\n"
  "    kw.setdefault(\"progress\", checks.make_printer(30.0, stream=sys.stdout))\n"
  "    return Cfg(**kw)\n"
  "\n"
  "\n"
  "def _ds(key: str) -> dict:\n"
  "    akey = ACQ_KEY[key]\n"
  "    return next(d for d in acquire.DATASETS if d[\"key\"] == akey)\n"
  "\n"
  "\n"
  "# ====================================================================================================== step plumbing\n"
  "def _flush_after_failure(cfg, name: str) -> None:\n"
  "    \"\"\"A step that writes to Drive and then fails still leaves what it wrote; make sure it really reaches Drive (never hides the failure).\"\"\"\n"
  "    if cfg.drive_sync is None:\n"
  "        return\n"
  "    try:\n"
  "        cfg.log(f\"  flushing Google Drive writes after the failure of [{name}] ...\")\n"
  "        cfg.drive_sync()\n"
  "    except Exception as e:                                                          # noqa: BLE001\n"
  "        cfg.log(f\"  WARNING: could not flush Drive after the failure ({type(e).__name__}: {e})\")\n"
  "\n"
  "\n"
  "def step(name: str, flush_on_failure: bool = False):\n"
  "    \"\"\"Decorator: prints the header and one PASS/FAIL line; fn(cfg, ...) returns (detail, result dict).  flush_on_failure: the steps that write\n"
  "    to Drive flush it (cfg.drive_sync) also when they fail.\"\"\"\n"
  "    def deco(fn):\n"
  "        @functools.wraps(fn)\n"
  "        def wrapper(cfg, *a, **kw):\n"
  "            t0 = time.time()\n"
  "            cfg.log(f\"=== {name} ===\")\n"
  "            try:\n"
  "                detail, res = fn(cfg, *a, **kw)\n"
  "            except StepFailed as e:\n"
  "                cfg.log(f\"FAIL [{name}] {e}\")\n"
  "                if flush_on_failure:\n"
  "                    _flush_after_failure(cfg, name)\n"
  "                raise\n"
  "            except Exception as e:                                                  # noqa: BLE001\n"
  "                tb = traceback.format_exc().strip().splitlines()[-6:]\n"
  "                cfg.log(f\"FAIL [{name}] {type(e).__name__}: {e}\\n    \" + \"\\n    \".join(tb))\n"
  "                if flush_on_failure:\n"
  "                    _flush_after_failure(cfg, name)\n"
  "                raise StepFailed(f\"{name}: {type(e).__name__}: {e}\") from e\n"
  "            out = dict(res or {})\n"
  "            out.update({\"pass\": True, \"step\": name, \"detail\": detail, \"seconds\": round(time.time() - t0, 1)})\n"
  "            cfg.timings.append({\"step\": name, \"seconds\": out[\"seconds\"]})\n"
  "            cfg.log(f\"PASS [{name}] {detail}  ({out['seconds']} s)\")\n"
  "            return out\n"
  "        wrapper.step_name = name\n"
  "        return wrapper\n"
  "    return deco\n"
  "\n"
  "\n"
  "def write_once(path, text: str) -> str:\n"
  "    \"\"\"Create a text file on Drive (via <path>.part); identical content is fine, different content is refused.\"\"\"\n"
  "    try:\n"
  "        return checks.write_new_file(str(path), text)\n"
  "    except FileExistsError as e:\n"
  "        raise StepFailed(f\"{e}.  Nothing was overwritten.  To produce new output, set RUN_SUFFIX (e.g. '_2') in the notebook and run again.\") from e\n"
  "\n"
  "\n"
  "def publish_file(src, dst, hint: Optional[str] = None) -> str:\n"
  "    \"\"\"Copy a local file to Drive: never over an existing file (re-checked right before the rename); verified by sha256; a partial copy of an\n"
  "    earlier run is continued when its bytes are exactly the start of the source (otherwise moved aside, never deleted).  `hint` replaces the\n"
  "    default advice of a refusal (set RUN_SUFFIX), which is wrong for folders that do not depend on RUN_SUFFIX (stage A).\"\"\"\n"
  "    src, dst = Path(src), Path(dst)\n"
  "    dst.parent.mkdir(parents=True, exist_ok=True)\n"
  "    want = checks.sha256_file(str(src))\n"
  "    if os.path.lexists(dst):\n"
  "        if checks.sha256_file(str(dst)) == want:\n"
  "            return \"identical\"\n"
  "        raise StepFailed(f\"refusing to overwrite {dst}: it exists with a different sha256.  \"\n"
  "                         + (hint or \"To produce new output, set RUN_SUFFIX (e.g. '_2') and run again.\"))\n"
  "    part = Path(str(dst) + \".part\")\n"
  "    n, sha, _resumed = acquire._copy_resume(str(src), str(part))\n"
  "    require(sha == want and n == src.stat().st_size, f\"copy of {src.name} to Drive does not match the source (kept as {part})\")\n"
  "    if os.path.lexists(dst):\n"
  "        raise StepFailed(f\"{dst} appeared while it was being copied; it was NOT overwritten (the finished copy stays at {part})\")\n"
  "    os.rename(part, dst)\n"
  "    return \"created\"\n"
  "\n"
  "\n"
  "def drive_sync(cfg: Cfg, why: str) -> None:\n"
  "    \"\"\"Make Drive writes really durable: the notebook's hook flushes and remounts (Drive/FUSE writes are asynchronous).\"\"\"\n"
  "    if cfg.drive_sync is not None:\n"
  "        cfg.log(f\"  flushing Google Drive writes ({why}) ...\")\n"
  "        cfg.drive_sync()\n"
  "\n"
  "\n"
  "@contextlib.contextmanager\n"
  "def untrusted_code_window(cfg: Cfg):\n"
  "    \"\"\"Drive is NOT mounted while code from the datasets runs: the owner's whole Drive (model weights, optimiser, the old corpus) is not\n"
  "    reachable from the VM during that time, whatever the sandbox does.  The converters only touch cfg.work.\"\"\"\n"
  "    if cfg.drive_detach is None:\n"
  "        yield False\n"
  "        return\n"
  "    cfg.log(\"  Google Drive is unmounted while code from the datasets runs ...\")\n"
  "    cfg.drive_detach()\n"
  "    try:\n"
  "        yield True\n"
  "    finally:\n"
  "        cfg.drive_attach()\n"
  "        cfg.log(\"  Google Drive is mounted again.\")\n"
  "\n"
  "\n"
  "def save_state(cfg: Cfg, name: str, obj) -> None:\n"
  "    write_once(cfg.out / \"state\" / f\"{name}.json\", jdump(obj))\n"
  "    cfg.state[name] = obj                                    # only after the write was accepted: a refused write must not leave the refused numbers in memory\n"
  "\n"
  "\n"
  "def load_state(cfg: Cfg, name: str):\n"
  "    if name in cfg.state:\n"
  "        return cfg.state[name]\n"
  "    p = cfg.out / \"state\" / f\"{name}.json\"\n"
  "    if p.is_file():\n"
  "        cfg.state[name] = json.loads(p.read_text(encoding=\"utf-8\"))\n"
  "        return cfg.state[name]\n"
  "    return None\n"
  "\n"
  "\n"
  "_CACHE: Dict[str, Any] = {}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 3: environment self-check\n"
  "@step(\"3 modules\")\n"
  "def step_modules(cfg: Cfg):\n"
  "    require(sys.version_info >= (3, 10), f\"Python {sys.version.split()[0]} is too old (need 3.10+)\")\n"
  "    acquire.validate_table(acquire.DATASETS)\n"
  "    require(set(KEYS) == set(convert.DATASETS), \"pipeline keys differ from convert.DATASETS\")\n"
  "    require(checks.RUNTIME_NORMALIZE == convert.NORMALIZE, \"checks.py and convert.py disagree on the runtime symbol normaliser\")\n"
  "    require(checks.SANDBOX_BACKEND == \"checks.code_sandbox_check(profile=executor)\", \"unexpected sandbox backend name\")\n"
  "    h = cfg.code_hashes()\n"
  "    return \"modules import; table valid; \" + \", \".join(f\"{k} {v[:8]}\" for k, v in sorted(h.items())), {\"code_sha256\": h}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 3c: Drive remount probe\n"
  "def expected_remounts(cfg: Cfg) -> int:\n"
  "    \"\"\"How many times a full run unmounts Drive and mounts it again, estimated from the card counts: one flush per acquired dataset (before its\n"
  "    provenance.json) and one after step 5, one window around 7a, one window per wave of converters in 7b, the flushes after 7b / 8 / 8b / 9, and\n"
  "    the final unmount of cell 10.  If Google asks for authorisation at every mount, this is how often the owner has to click.\"\"\"\n"
  "    jobs = 0\n"
  "    for k in cfg.keys:\n"
  "        total = EXPECTED_COUNTS[k]\n"
  "        jobs += len(shard_ranges(total, cfg.shard_records)) if (k in cfg.shard_keys and total > cfg.shard_records) else 1\n"
  "    waves = math.ceil(jobs / max(1, cfg.parallel))\n"
  "    syncs = (len(cfg.keys) + 5) if cfg.drive_sync is not None else 0\n"
  "    windows = (1 + waves) if cfg.drive_detach is not None else 0\n"
  "    return syncs + windows + 1\n"
  "\n"
  "\n"
  "@step(\"3c drive remount probe\")\n"
  "def step_drive_probe(cfg: Cfg):\n"
  "    \"\"\"Unmounts and mounts Drive a few times right at the start, with nothing running, so that the owner learns NOW whether Google asks for the\n"
  "    authorisation again at every mount (the run needs expected_remounts() of them, spread over hours).  Writes nothing.\"\"\"\n"
  "    if cfg.drive_sync is None and cfg.drive_detach is None:\n"
  "        return \"the Drive hooks are off (FLUSH_DRIVE_AFTER_STEPS and UNMOUNT_DRIVE_DURING_CODE_RUNS are False): nothing to probe\", {\"remounts_probed\": 0}\n"
  "    n = expected_remounts(cfg)\n"
  "    cfg.log(f\"  the full run unmounts Drive and mounts it again about {n} times (flush after every dataset, one window per wave of converters, flush after the \"\n"
  "            f\"writing steps).  Watch the screen during this probe: if a Google authorisation window appears at EVERY mount, it will appear about {n} times \"\n"
  "            f\"during the run, so stay near the screen during cells 5 and 7b (or set both switches in cell 3a to False and accept the trade-off written there).\")\n"
  "    done = 0\n"
  "    if cfg.drive_sync is not None:\n"
  "        cfg.drive_sync()\n"
  "        cfg.drive_sync()\n"
  "        done += 2\n"
  "    if cfg.drive_detach is not None:\n"
  "        cfg.drive_detach()\n"
  "        cfg.drive_attach()\n"
  "        done += 1\n"
  "    cfg.log(f\"  Drive was unmounted and mounted {done} times without a problem.  If you were asked to authorise more than once, expect that {n} times in this run.\")\n"
  "    return f\"Drive unmounted and mounted {done} times; the full run does it about {n} times\", {\"remounts_probed\": done, \"expected_remounts\": n}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 4: dry run\n"
  "def stale_files(cfg: Cfg) -> List[Path]:\n"
  "    \"\"\"Leftover *.stale-N copies of earlier dead runs (moved aside by the never-delete rule); the owner can remove them by hand.\"\"\"\n"
  "    out: List[Path] = []\n"
  "    for d in (cfg.ext, cfg.out, cfg.rows, cfg.stage_a_root):\n"
  "        if d.is_dir():\n"
  "            out += [p for p in d.rglob(\"*\") if p.is_file() and \".stale-\" in p.name]\n"
  "    return sorted(out)\n"
  "\n"
  "\n"
  "@step(\"4 dry run\")\n"
  "def step_dry_run(cfg: Cfg):\n"
  "    plan = acquire.acquire(cfg.root, cfg.acq_only(), dry_run=True, log=cfg.log, ext_dirname=cfg.ext_dirname)\n"
  "    sel = acquire.select(list(acquire.DATASETS), cfg.acq_only())\n"
  "    expect = acquire.total_bytes(sel)\n"
  "    require(plan[\"total_bytes\"] == expect and plan[\"n_files\"] == sum(len(d[\"files\"]) for d in sel),\n"
  "            f\"plan says {plan['total_bytes']:,} bytes / {plan['n_files']} files, the table says {expect:,}\")\n"
  "    biggest = max(f[\"size\"] for d in sel for f in d[\"files\"])\n"
  "    train = acquire.total_bytes(sel, (\"train\",))\n"
  "    notes = []\n"
  "    need_drive = expect + 2 * train + (1 << 30)               # raw files + stage A + final rows + dumps/state (7a measures the real size)\n"
  "    if cfg.check_disk:\n"
  "        Path(cfg.work).mkdir(parents=True, exist_ok=True)\n"
  "        need_local = biggest + train + (2 << 30)\n"
  "        free_local = shutil.disk_usage(cfg.work).free\n"
  "        require(free_local >= need_local, f\"local disk has {free_local:,} B free, need about {need_local:,} B (staging + mirror + rows)\")\n"
  "        notes.append(f\"local free {free_local / 2**30:.1f} GiB\")\n"
  "        try:\n"
  "            free_drive = shutil.disk_usage(cfg.root).free\n"
  "        except OSError:\n"
  "            free_drive = None\n"
  "        if free_drive:\n"
  "            require(free_drive >= need_drive, f\"Drive reports {free_drive:,} B free, need about {need_drive:,} B (raw files + converted rows + final rows)\")\n"
  "            notes.append(f\"Drive free {free_drive / 2**30:.1f} GiB (need about {need_drive / 2**30:.1f})\")\n"
  "        else:\n"
  "            notes.append(\"Drive free space unknown\")\n"
  "    stale = stale_files(cfg)\n"
  "    if stale:\n"
  "        total = sum(p.stat().st_size for p in stale)\n"
  "        cfg.log(f\"  note: {len(stale)} leftover '.stale-N' file(s) from earlier interrupted runs take {total / 2**20:,.0f} MiB on Drive; \"\n"
  "                f\"they are never deleted by this notebook - remove them by hand if you need the space: \" + \", \".join(str(p) for p in stale[:5]))\n"
  "        notes.append(f\"{len(stale)} stale files ({total / 2**20:,.0f} MiB)\")\n"
  "    return (f\"{plan['n_files']} files, expected total {expect:,} bytes = {expect / 1e9:.2f} GB \"\n"
  "            f\"(train {train:,}; dev/validation {plan['do_not_train_bytes']:,})\" + \"\".join(\"; \" + x for x in notes)), \\\n"
  "        {\"expected_total_bytes\": expect, \"n_files\": plan[\"n_files\"], \"need_drive_bytes\": need_drive, \"stale_files\": len(stale)}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 5: acquire\n"
  "def verify_acquired(cfg: Cfg) -> dict:\n"
  "    \"\"\"After acquire.acquire(): every selected dataset has provenance, the licence the plan expects, and all its files on Drive.\"\"\"\n"
  "    man = acquire.load_manifest(cfg.root, ext_dirname=cfg.ext_dirname)\n"
  "    out = {}\n"
  "    for key in cfg.keys:\n"
  "        ds = _ds(key)\n"
  "        prov = man.get(ds[\"repo\"])\n"
  "        require(prov is not None, f\"{ds['repo']}: provenance.json is missing - run step 5 first\")\n"
  "        require(set(prov[\"license\"][\"ids_from_card\"]) == set(ds[\"licenses\"]), f\"{ds['repo']}: licence on the card {prov['license']['ids_from_card']}\")\n"
  "        files = []\n"
  "        for f in prov[\"files\"]:\n"
  "            p = cfg.ext / f[\"dest_rel\"]\n"
  "            require(p.is_file() and p.stat().st_size == f[\"size\"], f\"{f['dest_rel']}: missing or wrong size on Drive\")\n"
  "            if f[\"role\"] != \"train\":                      # dev / validation (decontamination needs them intact) and the card: small, hashed after the remount; 6a hashes the train files\n"
  "                require(checks.sha256_file(str(p)) == f[\"sha256\"], f\"{f['dest_rel']}: sha256 differs from the verified file after the Drive remount (a lost or mangled write)\")\n"
  "            files.append({\"dest_rel\": f[\"dest_rel\"], \"role\": f[\"role\"], \"size\": f[\"size\"], \"sha256\": f[\"sha256\"]})\n"
  "        require(acquire.train_files(cfg.root, ds[\"key\"], ext_dirname=cfg.ext_dirname, include_held=True), f\"{ds['repo']}: no train files\")   # held datasets are converted for the checks\n"
  "        out[key] = {\"repo\": ds[\"repo\"], \"commit_sha\": prov[\"commit_sha\"], \"retrieved_utc\": prov[\"retrieved_utc\"],\n"
  "                    \"license_name\": prov[\"license\"][\"name\"], \"license_ids\": prov[\"license\"][\"ids_from_card\"],\n"
  "                    \"license_url\": prov[\"license\"][\"url\"], \"share_alike\": prov[\"license\"][\"share_alike\"],\n"
  "                    \"license_lines\": prov[\"license\"].get(\"card_licence_lines\") or [],\n"
  "                    \"attribution\": prov[\"attribution\"][\"lines\"], \"row_defaults\": prov[\"row_defaults\"],\n"
  "                    \"adoption_at_acquire\": prov[\"adoption\"], \"language\": prov[\"language\"], \"rows_per_card\": prov[\"rows_per_card\"],\n"
  "                    \"generator_notes\": prov[\"generator_notes\"], \"quality_notes\": prov[\"quality_notes\"],\n"
  "                    \"files\": files, \"train_bytes\": sum(f[\"size\"] for f in files if f[\"role\"] == \"train\"),\n"
  "                    \"do_not_train_bytes\": sum(f[\"size\"] for f in files if f[\"role\"] in acquire.NO_TRAIN_ROLES)}\n"
  "    return out\n"
  "\n"
  "\n"
  "@step(\"5 acquire\", flush_on_failure=True)\n"
  "def step_acquire(cfg: Cfg):\n"
  "    hook = (lambda repo: drive_sync(cfg, f\"step 5, the files of {repo} are in place; provenance.json comes next\")) if cfg.drive_sync is not None else None\n"
  "    res = acquire.acquire(cfg.root, cfg.acq_only(), log=cfg.log, allow_local=cfg.allow_local, stage_dir=cfg.stage_dir, ext_dirname=cfg.ext_dirname,\n"
  "                          after_files=hook)\n"
  "    drive_sync(cfg, \"step 5\")\n"
  "    info = verify_acquired(cfg)\n"
  "    save_state(cfg, \"step5_acquire\", info)\n"
  "    return (f\"{len(info)} datasets, {res['total_bytes']:,} bytes verified against the Hub (sha256 / git blob id), \"\n"
  "            f\"{res['downloaded_bytes']:,} downloaded in this run; licences match the plan\"), {\"datasets\": list(info)}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 6a: local mirror\n"
  "def mirror_train_files(cfg: Cfg, key: str) -> List[Path]:\n"
  "    \"\"\"Train files of one dataset inside the local mirror, named by the provenance record (dev/validation can never be listed).\"\"\"\n"
  "    d = cfg.mirror_dir(key)\n"
  "    prov = json.loads((d / \"provenance.json\").read_text(encoding=\"utf-8\"))\n"
  "    ext_folder = cfg.mirror\n"
  "    return [ext_folder / f[\"dest_rel\"] for f in prov[\"files\"] if f[\"role\"] == \"train\" and f.get(\"may_train\")]\n"
  "\n"
  "\n"
  "@step(\"6a local mirror\")\n"
  "def step_mirror(cfg: Cfg):\n"
  "    man = acquire.load_manifest(cfg.root, ext_dirname=cfg.ext_dirname)\n"
  "    total = copied = 0\n"
  "    for key in cfg.keys:\n"
  "        ds = _ds(key)\n"
  "        prov = man.get(ds[\"repo\"])\n"
  "        require(prov is not None, f\"{ds['repo']}: not acquired\")\n"
  "        exp = {f[\"dest_rel\"]: f for f in prov[\"files\"]}\n"
  "        folder = acquire.folder_of(ds[\"repo\"])\n"
  "        (cfg.mirror / folder).mkdir(parents=True, exist_ok=True)\n"
  "        for small in (\"provenance.json\", \"README.md\"):\n"
  "            if (cfg.ext / folder / small).is_file():\n"
  "                shutil.copyfile(cfg.ext / folder / small, cfg.mirror / folder / small)\n"
  "        for src in acquire.train_files(cfg.root, ds[\"key\"], ext_dirname=cfg.ext_dirname, include_held=True):     # held datasets are mirrored too (converted for the checks)\n"
  "            rel = src.relative_to(cfg.ext).as_posix()\n"
  "            e = exp[rel]\n"
  "            dst = cfg.mirror / rel\n"
  "            total += e[\"size\"]\n"
  "            if dst.is_file() and dst.stat().st_size == e[\"size\"] and checks.sha256_file(str(dst)) == e[\"sha256\"]:\n"
  "                continue\n"
  "            dst.parent.mkdir(parents=True, exist_ok=True)\n"
  "            part = Path(str(dst) + f\".part{os.getpid()}\")\n"
  "            if part.exists():                                                        # local scratch copy of our own: safe to replace\n"
  "                part.unlink()\n"
  "            n, sha = acquire._copy_hashing(str(src), str(part))\n"
  "            require(n == e[\"size\"] and sha == e[\"sha256\"], f\"{rel}: the local copy differs from the verified file on Drive\")\n"
  "            os.replace(part, dst)\n"
  "            copied += n\n"
  "    return f\"{total:,} bytes of train files are on the local disk and match the recorded sha256 ({copied:,} copied now)\", {\"bytes\": total}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 6b: vocabulary + eval index\n"
  "def _controller_source(cfg: Cfg) -> Optional[str]:\n"
  "    name = \"qwenputer_controller_v30.py\"\n"
  "    try:\n"
  "        for zp in checks._bundle_zip_paths(cfg.root):\n"
  "            with zipfile.ZipFile(zp) as z:\n"
  "                names = z.namelist()\n"
  "                for n in names:\n"
  "                    if n.endswith(name):\n"
  "                        return z.read(n).decode(\"utf-8\")\n"
  "                if checks.VOCAB_NESTED_ZIP in names:\n"
  "                    with zipfile.ZipFile(io.BytesIO(z.read(checks.VOCAB_NESTED_ZIP))) as inner:\n"
  "                        for n in inner.namelist():\n"
  "                            if n.endswith(name):\n"
  "                                return inner.read(n).decode(\"utf-8\")\n"
  "    except (OSError, zipfile.BadZipFile, UnicodeDecodeError):\n"
  "        return None\n"
  "    return None\n"
  "\n"
  "\n"
  "def _jaquad_dev_texts(paths):\n"
  "    seen = set()\n"
  "    for p in paths:\n"
  "        for r in checks.iter_rows(str(p)):\n"
  "            c = r.get(\"context\")\n"
  "            if isinstance(c, str) and c not in seen:\n"
  "                seen.add(c)\n"
  "                yield c\n"
  "            q = r.get(\"question\")\n"
  "            if isinstance(q, str):\n"
  "                yield q\n"
  "\n"
  "\n"
  "def _hotpot_validation_texts(paths):\n"
  "    for p in paths:\n"
  "        for r in checks.iter_rows(str(p), columns=[\"question\"]):\n"
  "            q = r.get(\"question\")\n"
  "            if isinstance(q, str):\n"
  "                yield q\n"
  "\n"
  "\n"
  "def decontam_sources(cfg: Cfg, ef) -> list:\n"
  "    \"\"\"Everything the training rows must not overlap: the eval manifests/code found by checks.find_eval_files, the v48 unseen_eval\n"
  "    texts, the registry's other frozen suites (python120, engineering64, visual10, the new_public dev sets) where they exist on Drive, the\n"
  "    dev/validation files that acquire.py keeps under do_not_train_*, and anything the owner drops into v49_eval_extra/.\"\"\"\n"
  "    sources = list(ef.sources)\n"
  "    dnt = acquire.do_not_train_files(cfg.root, ext_dirname=cfg.ext_dirname)\n"
  "    for repo, paths in sorted(dnt.items()):\n"
  "        if repo == \"SkelterLabsInc/JaQuAD\":\n"
  "            sources.append((\"jaquad_dev\", _jaquad_dev_texts(paths)))\n"
  "        elif repo == \"hotpotqa/hotpot_qa\":\n"
  "            sources.append((\"hotpot_validation\", _hotpot_validation_texts(paths)))\n"
  "    extra = Path(cfg.root) / USER_EVAL_EXTRA\n"
  "    if extra.is_dir():\n"
  "        for p in sorted(extra.iterdir()):\n"
  "            if p.is_file() and p.suffix.lower() in (\".jsonl\", \".ndjson\", \".json\"):\n"
  "                kind = \"jsonl\" if p.suffix.lower() in (\".jsonl\", \".ndjson\") else \"json\"\n"
  "                sources.append(checks.EvalSource(label=\"extra:\" + p.name, chain=(str(p),), kind=kind, required=False, description=\"owner's extra eval file\"))\n"
  "    return sources\n"
  "\n"
  "\n"
  "def eval_sources(cfg: Cfg):\n"
  "    \"\"\"(EvalFiles, all decontamination sources); cached for 8b after a kernel restart is rebuilt from the files.\"\"\"\n"
  "    try:\n"
  "        ef = checks.find_eval_files(cfg.root, extra_dirs=[cfg.eval_extra_dir] if cfg.eval_extra_dir else [], strict=True)\n"
  "    except checks.EvalFilesMissing as e:\n"
  "        raise StepFailed(f\"required eval files are missing, so decontamination would be incomplete: {e}\")\n"
  "    return ef, decontam_sources(cfg, ef)\n"
  "\n"
  "\n"
  "def tier_table(per_source: dict) -> List[dict]:\n"
  "    \"\"\"T1..T8 -> the files that stand for them and how many items / shingles each contributed to the eval index.\"\"\"\n"
  "    out = []\n"
  "    for tier, names in checks.EVAL_TIER_FILES.items():\n"
  "        out.append({\"tier\": tier, \"files\": {n: {\"items\": (per_source.get(n) or {}).get(\"items\", 0), \"shingles\": (per_source.get(n) or {}).get(\"shingles\", 0)} for n in names}})\n"
  "    return out\n"
  "\n"
  "\n"
  "def eval_source_problems(per_source: dict, required) -> Dict[str, List[str]]:\n"
  "    \"\"\"Required eval sources that cannot decontaminate anything: no item at all, or no text long enough to hold one n-gram (a manifest of ids /\n"
  "    metadata only).  A source that repeats the content of another one is fine: 'shingles' counts what a source CONTAINS, not what was new.\"\"\"\n"
  "    def get(label, field):\n"
  "        return (per_source.get(label) or {}).get(field, 0)\n"
  "    return {\"no_items\": sorted(l for l in required if get(l, \"items\") == 0),\n"
  "            \"no_shingles\": sorted(l for l in required if get(l, \"indexable_texts\") == 0 or get(l, \"shingles\") == 0)}\n"
  "\n"
  "\n"
  "@step(\"6b vocabulary and eval files\")\n"
  "def step_vocab_eval(cfg: Cfg):\n"
  "    notes: List[str] = []\n"
  "    # ---- the model vocabulary: never the census derived from a corpus\n"
  "    if cfg.vocab_override:\n"
  "        V = checks.vocab_from_json_bytes(Path(cfg.vocab_override).read_bytes(), str(cfg.vocab_override))\n"
  "    else:\n"
  "        V = checks.load_vocab(cfg.root, out_dir=str(cfg.out / \"state\"), allow_derive=False, progress=cfg.progress)\n"
  "    require(not V.derived, \"the vocabulary was derived from a corpus census, not read from the model's own file\")\n"
  "    require(len(V.chars) == checks.EXPECTED_SINGLE_CHARS and V.n_entries == checks.EXPECTED_VOCAB_SIZE,\n"
  "            f\"vocabulary has {len(V.chars)} single characters / {V.n_entries} entries, expected {checks.EXPECTED_SINGLE_CHARS} / {checks.EXPECTED_VOCAB_SIZE}\")\n"
  "    vocab_sha_match = V.sha256_chars == KNOWN_VOCAB_SHA\n"
  "    if not vocab_sha_match:\n"
  "        cfg.log(\"!\" * 20 + f\" MISMATCH: vocabulary character set sha256 {V.sha256_chars[:16]}... differs from the one seen on the owner's PC ({KNOWN_VOCAB_SHA[:16]}...)\")\n"
  "        require(cfg.allow_vocab_mismatch,\n"
  "                f\"the vocabulary ({V.source}) has a different character set than the model's vocabulary on the owner's PC: a stale file would \"\n"
  "                f\"silently decide which rows are dropped as out-of-vocabulary.  Fix the file, or set ALLOW_VOCAB_MISMATCH = True in the notebook \"\n"
  "                f\"if the model's vocabulary really changed.\")\n"
  "        notes.append(\"vocabulary sha256 MISMATCH vs the owner's PC copy (accepted: ALLOW_VOCAB_MISMATCH)\")\n"
  "    Path(cfg.work).mkdir(parents=True, exist_ok=True)\n"
  "    cfg.vocab_file.write_text(json.dumps({\"vocab\": {c: i for i, c in enumerate(sorted(V.chars))}, \"source\": V.source}, ensure_ascii=False), encoding=\"utf-8\")\n"
  "    _CACHE[\"vocab\"] = V\n"
  "    # ---- the runtime symbol normaliser: checks.py == convert.py == the controller source (when reachable)\n"
  "    require(checks.RUNTIME_NORMALIZE == convert.NORMALIZE, \"checks.py and convert.py disagree on the symbol normaliser\")\n"
  "    ctrl = _controller_source(cfg)\n"
  "    if ctrl is not None:\n"
  "        ok, why = checks.verify_runtime_normalizer(ctrl)\n"
  "        require(ok, f\"the runtime normaliser differs from qwenputer_controller_v30.py: {why}\")\n"
  "        notes.append(\"normaliser identical to the controller source\")\n"
  "    else:\n"
  "        notes.append(\"controller source not found in the bundle (normaliser not compared with it)\")\n"
  "    # ---- eval files\n"
  "    ef, sources = eval_sources(cfg)\n"
  "    cfg.log(ef.report())\n"
  "    labels = {s.label for s in ef.sources}\n"
  "    if cfg.require_v48:\n"
  "        missing = [n for n in V48_EVAL_NAMES if \"v48:\" + n not in labels]\n"
  "        require(not missing, f\"v48 unseen_eval texts not found: {missing}\")\n"
  "    index = checks.build_eval_index(sources, ngram=cfg.ngram, progress=lambda label, n: cfg.log(f\"  {label}: {n:,} shingles so far\"))\n"
  "    st = index.stats()\n"
  "    required = {s.label for s in ef.sources if s.required}\n"
  "    prob = eval_source_problems(st[\"per_source\"], required)\n"
  "    require(not prob[\"no_items\"], f\"eval sources without any item: {prob['no_items']}\")\n"
  "    require(not prob[\"no_shingles\"], f\"eval sources that contributed no {cfg.ngram}-gram at all (a manifest of ids / metadata only would index nothing and \"\n"
  "                                     f\"decontaminate nothing): {prob['no_shingles']}\")\n"
  "    require(st[\"distinct_shingles\"] > 0, \"the eval index is empty\")\n"
  "    for key in cfg.keys:\n"
  "        want = {\"jaquad\": \"jaquad_dev\", \"hotpot\": \"hotpot_validation\"}.get(key)\n"
  "        if want:\n"
  "            require(st[\"per_source\"].get(want, {}).get(\"items\", 0) > 0, f\"{want} was not indexed (dev/validation files for decontamination)\")\n"
  "    _CACHE[\"eval_index\"], _CACHE[\"eval_sources\"], _CACHE[\"eval_files\"] = index, sources, ef\n"
  "    vd = V.as_dict()\n"
  "    vd[\"source\"] = vd[\"source\"][len(\"cache:\"):] if vd[\"source\"].startswith(\"cache:\") else vd[\"source\"]          # same text on a re-run that reads the cache\n"
  "    tiers = tier_table(st[\"per_source\"])\n"
  "    cfg.log(\"  T1-T8 coverage: \" + \"; \".join(f\"{t['tier'].split(' ')[0]} \" + \"+\".join(f\"{n}({v['shingles']:,})\" for n, v in t[\"files\"].items()) for t in tiers))\n"
  "    info = {\"vocab\": dict(vd, sha_matches_owner_pc_copy=vocab_sha_match),\n"
  "            \"eval\": {\"sources\": [{\"label\": s.label, \"where\": s.display, \"required\": s.required} for s in ef.sources],\n"
  "                     \"extra_sources\": [s.label if isinstance(s, checks.EvalSource) else s[0] for s in sources[len(ef.sources):]],\n"
  "                     \"missing_optional\": ef.missing_optional, \"notes\": ef.notes, \"tiers\": tiers,\n"
  "                     \"index\": {\"ngram\": st[\"ngram\"], \"distinct_shingles\": st[\"distinct_shingles\"], \"per_source\": st[\"per_source\"],\n"
  "                               \"shingles_in_several_eval_items\": st[\"shingles_in_several_eval_items\"],\n"
  "                               \"max_eval_items_per_shingle\": st[\"max_eval_items_per_shingle\"],\n"
  "                               \"whole_item_texts\": st[\"whole_item_texts\"], \"whole_item_min_chars\": st[\"whole_item_min_chars\"]}},\n"
  "            \"notes\": notes}\n"
  "    save_state(cfg, \"step6b_vocab_eval\", info)\n"
  "    return (f\"vocabulary {len(V.chars)} chars from {V.source}; {len(ef.sources)} eval sources + {len(sources) - len(ef.sources)} dev/validation/extra sources, \"\n"
  "            f\"{st['distinct_shingles']:,} distinct {cfg.ngram}-gram shingles; {len(ef.missing_optional)} optional suites not found; \" + \"; \".join(notes)), info\n"
  "\n"
  "\n"
  "# ====================================================================================================== 6c: raw checks\n"
  "def _squad_text(r):\n"
  "    return \"\\n\".join([str(r.get(\"title\", \"\")), str(r.get(\"context\", \"\")), str(r.get(\"question\", \"\")), \" \".join(str(x) for x in r.get(\"answer_texts\") or [])])\n"
  "\n"
  "\n"
  "def _tengentoppa_text(r):\n"
  "    return str(r.get(\"instruction\", \"\")) + \"\\n\" + str(r.get(\"refined_answer\", \"\"))\n"
  "\n"
  "\n"
  "def _toolace_text(r):\n"
  "    return str(r.get(\"system\", \"\")) + \"\\n\" + \"\\n\".join(str(m.get(\"value\", \"\")) for m in (r.get(\"conversations\") or []) if isinstance(m, dict))\n"
  "\n"
  "\n"
  "RAW_TEXT = {\"jaquad\": _squad_text, \"tengentoppa\": _tengentoppa_text, \"toolace\": _toolace_text, \"coding\": checks.default_text,\n"
  "            \"wizard\": checks.default_text, \"hotpot\": checks.default_text}\n"
  "\n"
  "\n"
  "def raw_rows(cfg: Cfg, key: str, stats: Optional[dict] = None):\n"
  "    \"\"\"Rows of the mirrored train files.  Lenient like the converters (an undecodable byte or a damaged line does not abort a 1 GB file; it is\n"
  "    counted in `stats` and reported loudly): the strict reader is kept for the old corpus.\"\"\"\n"
  "    for p in mirror_train_files(cfg, key):\n"
  "        yield from checks.iter_rows(str(p), strict=False, stats=stats)\n"
  "\n"
  "\n"
  "def mismatch_banner(cfg: Cfg, key: str, n: int, exp: int) -> None:\n"
  "    cfg.log(\"!\" * 24 + f\" MISMATCH {key}: counted {n:,} records, {COUNT_SOURCE[key]} ({n - exp:+,}) \" + \"!\" * 24)\n"
  "\n"
  "\n"
  "@step(\"6c raw checks\")\n"
  "def step_raw_checks(cfg: Cfg):\n"
  "    V = _CACHE.get(\"vocab\")\n"
  "    require(V is not None, \"run step 6b first (vocabulary)\")\n"
  "    # a re-run after a disconnect does not read 1.9 GB again when the saved result was made from the same raw files, vocabulary and expectations\n"
  "    binding = {\"raw\": {k: cfg.raw_binding(k) for k in cfg.keys}, \"vocab_sha256_chars\": V.sha256_chars, \"expected_counts\": cfg.expected_counts}\n"
  "    done = load_state(cfg, \"step6c_raw_checks\")\n"
  "    if done is not None and done.get(\"binding\") == binding:\n"
  "        for m in done[\"mismatches\"]:\n"
  "            mismatch_banner(cfg, m[\"dataset\"], m[\"counted\"], m[\"expected\"])\n"
  "        if done[\"mismatches\"] and cfg.stop_on_count_mismatch:\n"
  "            raise StepFailed(\"record counts differ from the cards for: \" + \", \".join(m[\"dataset\"] for m in done[\"mismatches\"]) + \" (STOP_ON_COUNT_MISMATCH is on)\")\n"
  "        return (f\"already done (state found for the same raw files, vocabulary and expectations; {len(done['mismatches'])} count mismatch(es))\"), \\\n"
  "            {\"datasets\": done[\"datasets\"], \"mismatches\": done[\"mismatches\"]}\n"
  "    out, mism = {}, []\n"
  "    for key in cfg.keys:\n"
  "        tf = RAW_TEXT[key]\n"
  "        prefixes: collections.Counter = collections.Counter()\n"
  "        bad: dict = {}\n"
  "\n"
  "        def rows():\n"
  "            for r in raw_rows(cfg, key, bad):\n"
  "                if key == \"jaquad\":\n"
  "                    prefixes[str(r.get(\"id\"))[:3]] += 1\n"
  "                yield r\n"
  "        t0 = time.time()\n"
  "        oov = checks.oov_stats(rows(), V, text_fn=tf, progress=cfg.progress)\n"
  "        ln = checks.length_stats(raw_rows(cfg, key), text_fn=tf, progress=cfg.progress)[\"ALL\"]\n"
  "        n = oov[\"n_rows\"]\n"
  "        bad_lines, repl = bad.get(\"bad_lines\", 0), bad.get(\"decode_replacements\", 0)\n"
  "        if bad_lines or repl:\n"
  "            cfg.log(\"!\" * 24 + f\" BAD INPUT {key}: {bad_lines:,} damaged JSON line(s) skipped by this check (the converter counts them as rejects), \"\n"
  "                    f\"{repl:,} line(s) with undecodable bytes (replaced by U+FFFD; such rows fail the vocabulary filter) \" + \"!\" * 24)\n"
  "        require(n > 0 and ln[\"n\"] == n, f\"{key}: no records or the two passes disagree ({n} vs {ln['n']})\")\n"
  "        dump = cfg.out / \"reading\" / f\"raw_{key}_100.txt\"\n"
  "        checks.sample_for_reading(raw_rows(cfg, key), n=100, seed=7, out_path=str(dump), text_fn=tf,\n"
  "                                  title=f\"{key} RAW records (before conversion)\", vocab=V, progress=cfg.progress)\n"
  "        total = n + bad_lines\n"
  "        exp = (cfg.expected_counts or {}).get(key)\n"
  "        match = None if exp is None else (total == exp)\n"
  "        if match is False:\n"
  "            mismatch_banner(cfg, key, total, exp)\n"
  "            mism.append({\"dataset\": key, \"counted\": total, \"expected\": exp, \"source\": COUNT_SOURCE[key]})\n"
  "        if key == \"jaquad\":\n"
  "            badp = {p: c for p, c in prefixes.items() if p in (\"de-\", \"te-\")}\n"
  "            require(not badp, f\"jaquad train files contain dev/test ids: {badp}\")\n"
  "        out[key] = {\"n_raw\": n, \"bad_lines\": bad_lines, \"decode_replacements\": repl, \"n_total\": total, \"expected\": exp, \"count_match\": match,\n"
  "                    \"count_source\": COUNT_SOURCE[key],\n"
  "                    \"oov\": {k: oov[k] for k in (\"n_chars\", \"oov_frac\", \"rows_with_any_oov\", \"rows_over_threshold\", \"share_rows_over_threshold\", \"n_distinct_oov_chars\")},\n"
  "                    \"top_oov\": oov[\"top_oov\"][:8],\n"
  "                    \"length\": {k: ln[k] for k in (\"mean\", \"p50\", \"p90\", \"p99\", \"max\", \"over_2048\", \"share_over_2048\", \"over_1900\", \"share_over_1900\")},\n"
  "                    \"id_prefixes\": dict(prefixes) if key == \"jaquad\" else None, \"reading_file\": str(dump)}\n"
  "        cfg.log(f\"  {key}: {total:,} records, OOV chars {oov['oov_frac']:.4%}, rows with >1% OOV {oov['share_rows_over_threshold']:.2%}, \"\n"
  "                f\"rows over 2,048 chars {ln['share_over_2048']:.2%}  ({time.time() - t0:.0f} s)\")\n"
  "    if mism and cfg.stop_on_count_mismatch:\n"
  "        raise StepFailed(\"record counts differ from the cards for: \" + \", \".join(m[\"dataset\"] for m in mism) + \" (STOP_ON_COUNT_MISMATCH is on)\")\n"
  "    save_state(cfg, \"step6c_raw_checks\", {\"datasets\": out, \"mismatches\": mism, \"binding\": binding})\n"
  "    tail = \"all counts match the cards\" if not mism else f\"MISMATCH in {len(mism)}: \" + \", \".join(f\"{m['dataset']} {m['counted']:,} vs {m['expected']:,}\" for m in mism)\n"
  "    return f\"{len(out)} datasets read end to end (bad lines are counted, not fatal); {tail}\", {\"datasets\": out, \"mismatches\": mism}\n"
  "\n"
  "\n"
  "def n_records(raw: Optional[dict], key: str) -> Optional[int]:\n"
  "    \"\"\"Records the converter will read for `key`: parseable rows + damaged lines (it counts those as rejects).\"\"\"\n"
  "    d = ((raw or {}).get(\"datasets\", {}).get(key) or {})\n"
  "    if d.get(\"n_raw\") is None:\n"
  "        return None\n"
  "    return d[\"n_raw\"] + d.get(\"bad_lines\", 0)\n"
  "\n"
  "\n"
  "# ====================================================================================================== 3d: the sandbox must prove itself on THIS machine\n"
  "@step(\"3d sandbox self-test\")\n"
  "def step_sandbox_selftest(cfg: Cfg):\n"
  "    \"\"\"Runs before any dataset code does (cell 3d, right after the modules are loaded: a Linux failure shows up before 30-80 minutes of\n"
  "    downloads), with Drive still mounted: the child must run as an unprivileged user, must not be able to list or write the mounted Drive, must\n"
  "    refuse the escape payloads of the review, import every allowed module under the memory limit (RLIMIT_AS, -S), stop an endless loop and cap\n"
  "    its output.  No weaker fallback exists: if this fails, no code is run.  Only 'coding' executes code: a run without it skips the test.\"\"\"\n"
  "    if not cfg.needs_sandbox:\n"
  "        return (\"not needed: \" + (\"RUN_SANDBOX is off\" if not cfg.run_sandbox else \"none of the selected datasets executes code (only 'coding' does)\")), {\"selftest\": None}\n"
  "    r = checks.sandbox_selftest(require_os_containment=cfg.containment_required, drive_unmounted=cfg.drive_detach is not None)\n"
  "    failed = [c for c in r[\"checks\"] if not c[\"ok\"]]\n"
  "    n_info = sum(1 for c in r[\"checks\"] if c.get(\"info\"))\n"
  "    n_passed = sum(1 for c in r[\"checks\"] if c[\"ok\"] and not c.get(\"info\"))\n"
  "    for c in r[\"checks\"]:\n"
  "        cfg.log(f\"  {'info' if c.get('info') else ('ok  ' if c['ok'] else 'FAIL')} {c['name']}\" + (\"\" if c[\"ok\"] and not c.get(\"info\") else f\": {c['detail']}\"))\n"
  "    require(r[\"ok\"], f\"the sandbox self-test failed ({len(failed)} of {n_passed + len(failed)} checks): \" + \"; \".join(f\"{c['name']}: {c['detail']}\" for c in failed))\n"
  "    require(r[\"backend\"] == checks.SANDBOX_BACKEND, \"unexpected sandbox backend\")\n"
  "    if cfg.containment_required:\n"
  "        require(r[\"run_as\"] not in (None, \"root\"), \"the sandbox child does not run as an unprivileged user\")\n"
  "    rec = {\"backend\": r[\"backend\"], \"run_as\": r[\"run_as\"], \"n_checks\": n_passed, \"n_info\": n_info, \"containment_required\": cfg.containment_required,\n"
  "           \"drive_unmounted_during_code_runs\": cfg.drive_detach is not None,\n"
  "           \"checks\": [{\"name\": c[\"name\"], \"ok\": c[\"ok\"], **({\"info\": True} if c.get(\"info\") else {})} for c in r[\"checks\"]]}\n"
  "    _CACHE[\"sandbox\"] = rec                                    # valid for THIS process only: a new runtime proves itself again\n"
  "    text = jdump(rec)\n"
  "    write_once(cfg.out / \"state\" / f\"sandbox_selftest_{sha256_text(text)[:8]}.json\", text)          # a record, named by its content: never a conflict\n"
  "    return (f\"{n_passed} checks passed\" + (f\" (+ {n_info} information line(s) that prove nothing)\" if n_info else \"\") + f\"; backend {r['backend']}; child runs as {r['run_as'] or 'the notebook user (no OS barrier: local test only)'}\"), \\\n"
  "        {\"selftest\": r}\n"
  "\n"
  "\n"
  "def ensure_sandbox_ok(cfg: Cfg) -> dict:\n"
  "    \"\"\"The self-test result of THIS process (7a / 7b never start code without it); a fresh runtime runs it again, a result saved on Drive by an\n"
  "    earlier runtime is never trusted (the test is about the machine that is about to run the code).\"\"\"\n"
  "    st = _CACHE.get(\"sandbox\")\n"
  "    if st is None or (cfg.containment_required and st.get(\"run_as\") in (None, \"root\")):\n"
  "        step_sandbox_selftest(cfg)\n"
  "        st = _CACHE[\"sandbox\"]\n"
  "    require(st[\"backend\"] == checks.SANDBOX_BACKEND, \"the sandbox self-test belongs to another backend: run step 3d again\")\n"
  "    if cfg.containment_required:\n"
  "        require(st.get(\"run_as\") not in (None, \"root\"), \"the sandbox self-test shows no OS barrier: run step 3d again on the Colab runtime\")\n"
  "    return st\n"
  "\n"
  "\n"
  "def last_sandbox_record(cfg: Cfg) -> Optional[dict]:\n"
  "    \"\"\"What registry.json says about the sandbox: this process's self-test, else the newest record on Drive.\"\"\"\n"
  "    if _CACHE.get(\"sandbox\") is not None:\n"
  "        return _CACHE[\"sandbox\"]\n"
  "    files = sorted((cfg.out / \"state\").glob(\"sandbox_selftest_*.json\"), key=lambda q: q.stat().st_mtime) if (cfg.out / \"state\").is_dir() else []\n"
  "    return json.loads(files[-1].read_text(encoding=\"utf-8\")) if files else None\n"
  "\n"
  "\n"
  "# ====================================================================================================== 7: converters\n"
  "def new_job(key: str, start: int = 0, stop: Optional[int] = None, out_root: Optional[Path] = None, shard: bool = False) -> dict:\n"
  "    ident = key if not shard else f\"{key}@{start}-{'end' if stop is None else stop}\"\n"
  "    return {\"id\": ident, \"key\": key, \"start\": start, \"stop\": stop, \"out_root\": out_root, \"shard\": shard}\n"
  "\n"
  "\n"
  "def _pid_alive(pid: int) -> bool:\n"
  "    if pid <= 0:\n"
  "        return False\n"
  "    if os.name == \"nt\":\n"
  "        import ctypes\n"
  "        k32 = ctypes.windll.kernel32\n"
  "        h = k32.OpenProcess(0x1000, False, pid)                      # PROCESS_QUERY_LIMITED_INFORMATION\n"
  "        if not h:\n"
  "            return False\n"
  "        code = ctypes.c_ulong()\n"
  "        ok = k32.GetExitCodeProcess(h, ctypes.byref(code))\n"
  "        k32.CloseHandle(h)\n"
  "        return bool(ok) and code.value == 259                        # STILL_ACTIVE\n"
  "    try:\n"
  "        os.kill(pid, 0)\n"
  "    except ProcessLookupError:\n"
  "        return False\n"
  "    except PermissionError:\n"
  "        return True\n"
  "    return True\n"
  "\n"
  "\n"
  "def _terminate(job: dict, grace_s: float = 10.0) -> None:\n"
  "    \"\"\"Stop a converter and the sandbox children it started.  SIGTERM goes to the converter's process group; the sandbox children are session leaders\n"
  "    (checks._spawn, start_new_session) and are NOT in that group, so the converter forwards the signal itself (convert.forward_termination_to_sandbox ->\n"
  "    checks.kill_live_sandbox_children).  After the grace period the converter is SIGKILLed, which cannot forward anything: a child that is still\n"
  "    running then ends at its own RLIMIT_CPU (timeout + 1 s of CPU) - bounded, but not instant (POSIX code: not run on the Windows PC).\"\"\"\n"
  "    p = job[\"proc\"]\n"
  "    pid = getattr(p, \"pid\", None)\n"
  "    try:\n"
  "        if os.name == \"posix\" and pid:\n"
  "            os.killpg(pid, signal.SIGTERM)\n"
  "        elif hasattr(p, \"terminate\"):\n"
  "            p.terminate()\n"
  "    except (OSError, AttributeError):\n"
  "        pass\n"
  "    t0 = time.time()\n"
  "    while time.time() - t0 < grace_s and p.poll() is None:\n"
  "        time.sleep(0.2)\n"
  "    if p.poll() is None:\n"
  "        try:\n"
  "            if os.name == \"posix\" and pid:\n"
  "                os.killpg(pid, signal.SIGKILL)\n"
  "            elif hasattr(p, \"kill\"):\n"
  "                p.kill()\n"
  "        except (OSError, AttributeError):\n"
  "            pass\n"
  "\n"
  "\n"
  "def _converter_cmd(cfg: Cfg, job: dict, limit: Optional[int]) -> List[str]:\n"
  "    key = job[\"key\"]\n"
  "    cmd = [sys.executable, \"-X\", \"utf8\", str(Path(convert.__file__)), \"--dataset\", key, \"--raw-dir\", str(cfg.mirror_dir(key)),\n"
  "           \"--out-dir\", str(job[\"out_root\"]), \"--vocab\", str(cfg.vocab_file), \"--max-chars\", str(cfg.max_chars),\n"
  "           \"--sample-dump\", \"0\" if job[\"shard\"] else \"100\"]\n"
  "    if job[\"start\"]:\n"
  "        cmd += [\"--start\", str(job[\"start\"])]\n"
  "    stop = job[\"stop\"] if job[\"stop\"] is not None else limit\n"
  "    if stop:\n"
  "        cmd += [\"--limit\", str(stop)]\n"
  "    if not cfg.run_sandbox or key not in CODE_EXEC_KEYS:                 # only 'coding' runs code: the other converters never start the sandbox\n"
  "        cmd += [\"--no-sandbox\"]\n"
  "    elif cfg.containment_required:\n"
  "        cmd += [\"--require-os-containment\"]\n"
  "    if cfg.python_allow_unverified:\n"
  "        cmd += [\"--allow-unverified-python\"]\n"
  "    return cmd\n"
  "\n"
  "\n"
  "def _run_window(cfg: Cfg, jobs: List[dict], limit: Optional[int], smoke: bool) -> dict:\n"
  "    \"\"\"Run converter subprocesses (at most cfg.parallel at a time, in the given order) until all have finished.  An interrupt or any other\n"
  "    exception terminates every converter (and the sandbox children in their sessions) before it propagates: no orphan keeps running\n"
  "    behind a second 'Run all'.  A lock file per job refuses a second converter for the same job while the first one is alive.\"\"\"\n"
  "    logdir, lockdir = Path(cfg.work) / \"logs\", Path(cfg.work) / \"locks\"\n"
  "    logdir.mkdir(parents=True, exist_ok=True)\n"
  "    lockdir.mkdir(parents=True, exist_ok=True)\n"
  "    pending, running, rcs = list(jobs), [], {}\n"
  "    t0, last_beat = time.time(), time.time()\n"
  "    env = dict(os.environ, PYTHONUTF8=\"1\", PYTHONHASHSEED=\"0\")\n"
  "    try:\n"
  "        while pending or running:\n"
  "            while pending and len(running) < max(1, cfg.parallel):\n"
  "                job = pending.pop(0)\n"
  "                job[\"out_root\"].mkdir(parents=True, exist_ok=True)\n"
  "                lock = lockdir / f\"convert_{job['id'].replace('@', '_')}_{'smoke' if smoke else 'full'}.lock\"\n"
  "                if lock.is_file():\n"
  "                    try:\n"
  "                        other = int(lock.read_text().strip() or 0)\n"
  "                    except ValueError:\n"
  "                        other = 0\n"
  "                    require(not _pid_alive(other), f\"a converter for {job['id']} is still running (pid {other}); stop it (Runtime > Interrupt execution) or wait \"\n"
  "                                                   f\"for it - starting a second one would fight over the same files\")\n"
  "                cmd = _converter_cmd(cfg, job, limit)\n"
  "                lp = logdir / f\"convert_{'smoke' if smoke else 'full'}_{cfg.converter_tag()}_{job['id'].replace('@', '_')}.log\"\n"
  "                fh = open(lp, \"w\", encoding=\"utf-8\")\n"
  "                proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env, cwd=str(cfg.work), start_new_session=(os.name == \"posix\"))\n"
  "                lock.write_text(str(getattr(proc, \"pid\", 0) or 0), encoding=\"utf-8\")\n"
  "                running.append(dict(job, proc=proc, fh=fh, log=lp, t0=time.time(), lock=lock, out=job[\"out_root\"] / job[\"key\"]))\n"
  "                cfg.log(f\"  started convert.py --dataset {job['key']}\" + (f\" --start {job['start']}\" if job[\"start\"] else \"\")\n"
  "                        + (f\" --limit {job['stop'] if job['stop'] is not None else limit}\" if (job[\"stop\"] is not None or limit) else \"\") + f\"  (log {lp.name})\")\n"
  "            time.sleep(cfg.poll_s)\n"
  "            for job in list(running):\n"
  "                rc = job[\"proc\"].poll()\n"
  "                if rc is not None:\n"
  "                    job[\"fh\"].close()\n"
  "                    try:\n"
  "                        job[\"lock\"].unlink()\n"
  "                    except OSError:\n"
  "                        pass\n"
  "                    rcs[job[\"id\"]] = {\"rc\": rc, \"seconds\": round(time.time() - job[\"t0\"], 1), \"log\": str(job[\"log\"])}\n"
  "                    running.remove(job)\n"
  "                    cfg.log(f\"  finished {job['id']}: rc={rc} after {rcs[job['id']]['seconds']} s\")\n"
  "            if running and time.time() - last_beat >= cfg.heartbeat_s:\n"
  "                last_beat = time.time()\n"
  "                for job in running:\n"
  "                    done = sum(p.stat().st_size for p in job[\"out\"].glob(\"*.part-*\")) if job[\"out\"].is_dir() else 0\n"
  "                    cfg.log(f\"  [{(time.time() - t0) / 60:5.1f} min] {job['id']} still running, {done / 1e6:.1f} MB of rows written so far\")\n"
  "    except BaseException:\n"
  "        for job in running:\n"
  "            cfg.log(f\"  stopping converter {job['id']} ...\")\n"
  "            _terminate(job)\n"
  "            try:\n"
  "                job[\"fh\"].close()\n"
  "                job[\"lock\"].unlink()\n"
  "            except OSError:\n"
  "                pass\n"
  "        raise\n"
  "    return rcs\n"
  "\n"
  "\n"
  "def _run_converters(cfg: Cfg, jobs: List[dict], smoke: bool, limit: Optional[int] = None, wave_size: Optional[int] = None,\n"
  "                    on_wave: Optional[Callable] = None) -> dict:\n"
  "    \"\"\"convert.py as one subprocess per job.  wave_size=None: one window for all jobs (Drive stays unmounted for the whole window).\n"
  "    wave_size=n: the jobs run in waves of n; after each wave Drive is mounted again and on_wave(finished_ok_jobs, results) can publish what\n"
  "    is done, so a disconnect loses at most one wave.  Failed jobs raise StepFailed after the finished ones have been handed to on_wave.\"\"\"\n"
  "    chunks = [list(jobs)] if not wave_size else [list(jobs[i:i + wave_size]) for i in range(0, len(jobs), wave_size)]\n"
  "    results: dict = {}\n"
  "    for chunk in chunks:\n"
  "        if not chunk:\n"
  "            continue\n"
  "        with untrusted_code_window(cfg):\n"
  "            res = _run_window(cfg, chunk, limit, smoke)\n"
  "        results.update(res)\n"
  "        ok_jobs = [j for j in chunk if res.get(j[\"id\"], {}).get(\"rc\") == 0]\n"
  "        if on_wave and ok_jobs:\n"
  "            on_wave(ok_jobs, res)\n"
  "        bad = {k: v for k, v in res.items() if v[\"rc\"] != 0}\n"
  "        if bad:\n"
  "            tails = []\n"
  "            for k, v in bad.items():\n"
  "                tails.append(f\"--- {k} (rc {v['rc']}) log tail ---\\n\" + \"\\n\".join(Path(v[\"log\"]).read_text(encoding=\"utf-8\", errors=\"replace\").splitlines()[-15:]))\n"
  "            raise StepFailed(\"converter(s) failed: \" + \", \".join(f\"{k} rc={v['rc']}\" for k, v in bad.items()) + \"\\n\" + \"\\n\".join(tails))\n"
  "    return results\n"
  "\n"
  "\n"
  "def read_manifest(d: Path) -> dict:\n"
  "    p = Path(d) / \"manifest.json\"\n"
  "    require(p.is_file(), f\"{p} is missing\")\n"
  "    return json.loads(p.read_text(encoding=\"utf-8\"))\n"
  "\n"
  "\n"
  "def _check_manifest(cfg: Cfg, key: str, d: Path, limit: Optional[int], n_raw: Optional[int], start: int = 0, allow_empty: bool = False) -> dict:\n"
  "    \"\"\"limit = absolute stop index of the run (None = to the end), start = first index; n_raw = records of the whole dataset (or None).\"\"\"\n"
  "    man = read_manifest(d)\n"
  "    require(man.get(\"balanced\") is True, f\"{key}: converter bookkeeping is not balanced (n_in != n_out + rejects)\")\n"
  "    require(man[\"n_out\"] > 0 or allow_empty, f\"{key}: the converter emitted no row from {man['n_in']} records\")\n"
  "    if limit:\n"
  "        want = (limit if n_raw is None else min(limit, n_raw)) - start\n"
  "    else:\n"
  "        want = None if n_raw is None else n_raw - start\n"
  "    if want is not None:\n"
  "        require(man[\"n_in\"] == want, f\"{key}: converter read {man['n_in']:,} records, expected {want:,}\")\n"
  "    n_files = sum(e[\"rows\"] for e in man[\"files\"].values())\n"
  "    require(n_files == man[\"n_out\"], f\"{key}: split files hold {n_files} rows, manifest says {man['n_out']}\")\n"
  "    for sp in SPLITS:\n"
  "        require(man[\"files\"][f\"{sp}.jsonl\"][\"rows\"] == man[\"splits\"].get(sp, 0), f\"{key}: split counts disagree for {sp}\")\n"
  "    return man\n"
  "\n"
  "\n"
  "def iter_stage_a(cfg: Cfg, key: str):\n"
  "    \"\"\"(split, raw JSON line) of the unfiltered converted rows, train then dev then heldout; the position in this order is the row's index.\"\"\"\n"
  "    d = cfg.conv_dir(key)\n"
  "    for sp in SPLITS:\n"
  "        with open(d / f\"{sp}.jsonl\", encoding=\"utf-8\", newline=\"\\n\") as f:\n"
  "            for line in f:\n"
  "                if line.strip():\n"
  "                    yield sp, line\n"
  "\n"
  "\n"
  "def validate_rows(cfg: Cfg, key: str, ck: \"convert.Checks\") -> dict:\n"
  "    \"\"\"Independent pass over a dataset's converted files: every row re-read from JSON must satisfy convert.validate_row, ids are unique,\n"
  "    a group never spans two splits, no dev/test id of JaQuAD, spans cover the injected observations.\"\"\"\n"
  "    ids, group_split, n, bad = set(), {}, 0, collections.Counter()\n"
  "    for sp, line in iter_stage_a(cfg, key):\n"
  "        row = json.loads(line)\n"
  "        n += 1\n"
  "        why = convert.validate_row(row, ck)\n"
  "        if why:\n"
  "            bad[why] += 1\n"
  "        require(row[\"split\"] == sp, f\"{key}: row {row['id']} says split {row['split']} but sits in {sp}.jsonl\")\n"
  "        require(row[\"id\"] not in ids, f\"{key}: duplicate row id {row['id']}\")\n"
  "        ids.add(row[\"id\"])\n"
  "        g = row.get(\"group_key\")\n"
  "        if g is not None and key in GROUP_INDEPENDENT:                # rows of the other datasets are their own group: nothing to check\n"
  "            require(group_split.setdefault(g, sp) == sp, f\"{key}: group {g!r} appears in two splits ({group_split[g]} and {sp})\")\n"
  "        if key == \"jaquad\":\n"
  "            require(not row[\"id\"].startswith((\"v49acq_jaquad_de-\", \"v49acq_jaquad_te-\")), f\"{key}: dev/test id in the rows: {row['id']}\")\n"
  "    require(not bad, f\"{key}: rows that fail convert.validate_row after the JSON round trip: {dict(bad)}\")\n"
  "    return {\"rows\": n, \"groups\": len(group_split)}\n"
  "\n"
  "\n"
  "def _converted_checks(cfg: Cfg, key: str, V) -> dict:\n"
  "    paths = [str(cfg.conv_dir(key) / f\"{sp}.jsonl\") for sp in SPLITS]\n"
  "\n"
  "    def rows():\n"
  "        for p in paths:\n"
  "            yield from checks.iter_rows(p)\n"
  "    oov = checks.oov_stats(rows(), V, progress=cfg.progress)\n"
  "    ln = checks.length_stats(rows(), progress=cfg.progress)[\"ALL\"]\n"
  "    require(oov[\"rows_over_threshold\"] == 0, f\"{key}: {oov['rows_over_threshold']} converted rows have more than 1% characters outside the vocabulary\")\n"
  "    require(ln[\"over_2048\"] == 0 and ln[\"chars_over_2000\"] == 0, f\"{key}: converted rows longer than the cap (over 2,048: {ln['over_2048']}, over 2,000 chars: {ln['chars_over_2000']})\")\n"
  "    require(ln[\"full_text_mismatch\"] == 0, f\"{key}: {ln['full_text_mismatch']} rows with full_text != prompt + target\")\n"
  "    return {\"oov_frac\": oov[\"oov_frac\"], \"rows_with_any_oov\": oov[\"rows_with_any_oov\"], \"rows_over_threshold\": oov[\"rows_over_threshold\"],\n"
  "            \"normalizer_changed_rows\": oov[\"rows_changed_by_normalizer\"],\n"
  "            \"length\": {k: ln[k] for k in (\"n\", \"mean\", \"p50\", \"p90\", \"p99\", \"max\", \"over_1900\", \"over_2048\", \"chars_over_2000\", \"full_text_mismatch\")}}\n"
  "\n"
  "\n"
  "def _convert_ck(cfg: Cfg):\n"
  "    return convert.Checks(vocab_path=str(cfg.vocab_file), max_chars=cfg.max_chars, run_code=False, use_external=False)\n"
  "\n"
  "\n"
  "SANDBOX_INFRA_REJECT_PREFIXES = (\"sandbox_failed:crash\", \"sandbox_failed:spawn\", \"sandbox_failed:checks_error\")\n"
  "\n"
  "\n"
  "def sandbox_infrastructure_rejects(man: dict) -> Dict[str, int]:\n"
  "    \"\"\"Reject reasons of a manifest that say the SANDBOX failed (it could not start a child, the child ended without a result, the checks module\n"
  "    raised) as opposed to the code under test being wrong / slow / refused.  A balanced manifest hides them: every such row is counted as a reject.\"\"\"\n"
  "    return {k: v for k, v in (man.get(\"rejects\") or {}).items() if str(k).startswith(SANDBOX_INFRA_REJECT_PREFIXES)}\n"
  "\n"
  "\n"
  "def check_sandbox_backend(key: str, man: dict, cfg: Cfg) -> None:\n"
  "    \"\"\"A dataset that executes code must have been converted with the executor-profile sandbox, as an unprivileged user on Colab, and no row may have\n"
  "    been rejected because the sandbox itself failed (second net behind convert.SandboxUnavailable, which stops the converter at the first failure).\"\"\"\n"
  "    infra = sandbox_infrastructure_rejects(man)\n"
  "    require(not infra, f\"{key}: {sum(infra.values()):,} row(s) were rejected because the SANDBOX failed, not because of their code (\" +\n"
  "            \"; \".join(f\"{k} x {v:,}\" for k, v in sorted(infra.items(), key=lambda kv: -kv[1])[:5]) + f\"): the python rows that should have been verified are missing \"\n"
  "            f\"from the counts although the bookkeeping is balanced.  Fix the machine (free space in /tmp, the 'nobody' user, memory) and convert this dataset again \"\n"
  "            f\"(move {cfg.conv_dir(key)} and its Drive copy aside first, nothing is overwritten)\")\n"
  "    if key != \"coding\" or not cfg.run_sandbox:\n"
  "        return\n"
  "    b = man.get(\"backends\") or {}\n"
  "    require(b.get(\"sandbox\") == checks.SANDBOX_BACKEND, f\"{key}: converted with sandbox backend {b.get('sandbox')!r}, not {checks.SANDBOX_BACKEND!r}\")\n"
  "    if cfg.containment_required:\n"
  "        require(b.get(\"sandbox_run_as\") not in (None, \"root\"), f\"{key}: the sandbox child did not run as an unprivileged user ({b.get('sandbox_run_as')!r})\")\n"
  "\n"
  "\n"
  "def _dir_bytes(d: Path) -> int:\n"
  "    return sum(p.stat().st_size for p in Path(d).glob(\"*.jsonl\")) if Path(d).is_dir() else 0\n"
  "\n"
  "\n"
  "@step(\"7a convert smoke\")\n"
  "def step_convert_smoke(cfg: Cfg):\n"
  "    require(cfg.vocab_file.is_file(), \"run step 6b first (vocabulary file)\")\n"
  "    done = load_state(cfg, \"step7a_smoke\")\n"
  "    if done is not None:\n"
  "        # A re-run (new runtime, 'Run all' again) must not measure again: the saved numbers are wall-clock estimates, a second measurement differs\n"
  "        # slightly, and the write-once rule would refuse it.  The converters themselves are re-checked by 7b.\n"
  "        return (f\"already done (state found; the estimates of that run are used: about {(done.get('est_drive_bytes') or 0) / 2**30:.1f} GiB of Drive \"\n"
  "                f\"space for the rows)\"), {\"datasets\": {}, \"estimate_minutes\": done.get(\"estimate_minutes\"), \"est_converted_bytes\": done.get(\"est_converted_bytes\"),\n"
  "                                         \"est_drive_bytes\": done.get(\"est_drive_bytes\"), \"resumed\": True}\n"
  "    raw = load_state(cfg, \"step6c_raw_checks\")\n"
  "    require(raw is not None, \"run step 6c first (the record counts): without them 7a cannot check how many records each converter read, and the time and \"\n"
  "                             \"space estimates that 7b relies on would be silently wrong\")\n"
  "    if cfg.needs_sandbox:\n"
  "        ensure_sandbox_ok(cfg)\n"
  "    t0 = time.time()\n"
  "    jobs = [new_job(k, stop=cfg.smoke_limit, out_root=cfg.conv_dir(k, smoke=True).parent) for k in CONVERT_ORDER if k in cfg.keys]\n"
  "    rcs = _run_converters(cfg, jobs, smoke=True, limit=cfg.smoke_limit)\n"
  "    ck = _convert_ck(cfg)\n"
  "    res, est, est_bytes = {}, {}, 0\n"
  "    for key in cfg.keys:\n"
  "        d = cfg.conv_dir(key, smoke=True)\n"
  "        n_raw = n_records(raw, key)\n"
  "        man = _check_manifest(cfg, key, d, cfg.smoke_limit, n_raw)\n"
  "        check_sandbox_backend(key, man, cfg)\n"
  "        ids = set()\n"
  "        n = 0\n"
  "        for sp in SPLITS:\n"
  "            for r in checks.iter_rows(str(d / f\"{sp}.jsonl\")):\n"
  "                why = convert.validate_row(r, ck)\n"
  "                require(why is None, f\"{key}: smoke row {r.get('id')} invalid: {why}\")\n"
  "                require(r[\"id\"] not in ids and r[\"split\"] == sp, f\"{key}: duplicate id or wrong split file in the smoke rows\")\n"
  "                ids.add(r[\"id\"])\n"
  "                n += 1\n"
  "        require(n == man[\"n_out\"], f\"{key}: smoke files hold {n} rows, manifest says {man['n_out']}\")\n"
  "        res[key] = {\"n_in\": man[\"n_in\"], \"n_out\": man[\"n_out\"], \"keep_rate\": man[\"keep_rate\"], \"seconds\": rcs[key][\"seconds\"],\n"
  "                    \"top_rejects\": dict(list(man[\"rejects\"].items())[:5]), \"backends\": man.get(\"backends\")}\n"
  "        if n_raw and man[\"n_in\"]:\n"
  "            est[key] = round(rcs[key][\"seconds\"] * n_raw / man[\"n_in\"] / 60, 1)\n"
  "            est_bytes += int(_dir_bytes(d) / man[\"n_in\"] * n_raw)\n"
  "        cfg.log(f\"  {key}: {man['n_in']:,} in -> {man['n_out']:,} rows ({man['keep_rate']:.1%}), {rcs[key]['seconds']:.0f} s, sandbox: {(man.get('backends') or {}).get('sandbox')}\")\n"
  "    if est:\n"
  "        cfg.log(\"  rough full-run estimate (minutes, from this smoke run; Colab may differ; coding / tengentoppa run in shards): \"\n"
  "                + \", \".join(f\"{k} ~{v:g}\" for k, v in est.items()))\n"
  "        cfg.log(\"  keep the browser tab open during 7b (or use Colab's background execution): a closed tab can stop a running cell.  Finished \"\n"
  "                \"shards and datasets are already on Drive, so a re-run continues where it stopped.\")\n"
  "    est_drive = int(est_bytes * 2.5) + (256 << 20)            # stage A + final rows + dumps (the smoke rows are the first records: a rough measure)\n"
  "    out = {\"datasets\": res, \"estimate_minutes\": est, \"est_converted_bytes\": est_bytes, \"est_drive_bytes\": est_drive}\n"
  "    save_state(cfg, \"step7a_smoke\", {\"estimate_minutes\": est, \"est_converted_bytes\": est_bytes, \"est_drive_bytes\": est_drive})\n"
  "    return (f\"{len(res)} converters ran on the first {cfg.smoke_limit:,} records each; every row passed validate_row again after reading it back \"\n"
  "            f\"({time.time() - t0:.0f} s); about {est_drive / 2**30:.1f} GiB of Drive space is needed for the rows\"), out\n"
  "\n"
  "\n"
  "def restore_stage_a(cfg: Cfg, key: str) -> bool:\n"
  "    \"\"\"Copy the unfiltered rows of an earlier run from Drive to the local disk (resume).  False if there is none.  The Drive folder is keyed by\n"
  "    the converter tag (convert.py + sandbox + normaliser + vocabulary), so rows made by other converter code are simply not found.\"\"\"\n"
  "    src = cfg.stage_a_drive(key)\n"
  "    meta_p = src / \"stageA_meta.json\"\n"
  "    if not meta_p.is_file():\n"
  "        return False\n"
  "    meta = json.loads(meta_p.read_text(encoding=\"utf-8\"))\n"
  "    require(meta.get(\"converter\") == cfg.converter_hashes(),\n"
  "            f\"{src} was made by different converter code, parameters or another vocabulary than this notebook has.  Nothing was overwritten or deleted.\")\n"
  "    require(meta.get(\"raw\") == cfg.raw_binding(key),\n"
  "            f\"{src} was made from other raw data (another commit or other train files than the provenance of {cfg.ext_dirname}/ records).  Nothing was \"\n"
  "            f\"overwritten or deleted.  If the raw data really changed, set RAW_SUFFIX (a fresh raw folder also gets a fresh stage-A folder).\")\n"
  "    man = json.loads((src / \"manifest.json\").read_text(encoding=\"utf-8\"))\n"
  "    dst = cfg.conv_dir(key)\n"
  "    dst.mkdir(parents=True, exist_ok=True)\n"
  "    for name in STAGE_A_FILES:\n"
  "        s = src / name\n"
  "        require(s.is_file(), f\"{s} is missing from an earlier run\")\n"
  "        d = dst / name\n"
  "        want = man[\"files\"][name][\"sha256\"] if name in man[\"files\"] else None\n"
  "        if d.is_file() and (want is None or checks.sha256_file(str(d)) == want):\n"
  "            continue\n"
  "        shutil.copyfile(s, d)\n"
  "        if want is not None:\n"
  "            require(checks.sha256_file(str(d)) == want, f\"{name} restored from Drive does not match the manifest sha256\")\n"
  "    return True\n"
  "\n"
  "\n"
  "def _stage_a_hint(dst: Path) -> str:\n"
  "    return (f\"Stage A is shared by all output suffixes, so a new suffix does not help: rename the folder {dst} by hand (for example add '_old' to its name; \"\n"
  "            f\"nothing is deleted) and run cell 7b again.  If the raw data changed, set RAW_SUFFIX instead.\")\n"
  "\n"
  "\n"
  "def publish_stage_a(cfg: Cfg, key: str, V) -> None:\n"
  "    src, dst = cfg.conv_dir(key), cfg.stage_a_drive(key)\n"
  "    for name in STAGE_A_FILES:\n"
  "        publish_file(src / name, dst / name, hint=_stage_a_hint(dst))\n"
  "    meta = {\"schema\": \"v49_stage_a_meta/3\", \"converter\": cfg.converter_hashes(),                    # the filter code is not part of what stage A depends on\n"
  "            \"raw\": cfg.raw_binding(key), \"dataset_commit_sha\": (read_manifest(src).get(\"provenance\") or {}).get(\"commit_sha\"),\n"
  "            \"vocab_sha256_chars\": V.sha256_chars, \"limit\": None}\n"
  "    write_once(dst / \"stageA_meta.json\", jdump(meta))\n"
  "\n"
  "\n"
  "def ensure_stage_a(cfg: Cfg, key: str) -> None:\n"
  "    if (cfg.conv_dir(key) / \"manifest.json\").is_file():\n"
  "        return\n"
  "    require(restore_stage_a(cfg, key), f\"{key}: no converted rows locally or on Drive - run step 7b first\")\n"
  "\n"
  "\n"
  "def shard_ranges(n_records_total: int, size: int) -> List[tuple]:\n"
  "    \"\"\"Contiguous record ranges [(start, stop)], the last with stop None (to the end of the file whatever it holds).\"\"\"\n"
  "    k = max(1, math.ceil(n_records_total / max(1, size)))\n"
  "    return [(i * size, (i + 1) * size if i < k - 1 else None) for i in range(k)]\n"
  "\n"
  "\n"
  "def _shard_done_local(cfg: Cfg, key: str, a: int, b: Optional[int]) -> bool:\n"
  "    return (cfg.shard_root(key, a, b) / key / \"manifest.json\").is_file()\n"
  "\n"
  "\n"
  "def restore_shard(cfg: Cfg, key: str, a: int, b: Optional[int]) -> bool:\n"
  "    src = cfg.shard_drive(key, a, b)\n"
  "    if not (src / \"shard_meta.json\").is_file():\n"
  "        return False\n"
  "    meta = json.loads((src / \"shard_meta.json\").read_text(encoding=\"utf-8\"))\n"
  "    require(meta.get(\"converter\") == cfg.converter_hashes() and meta.get(\"range\") == [a, b],\n"
  "            f\"{src} was made by different converter code or for another range.  Nothing was overwritten or deleted.\")\n"
  "    require(meta.get(\"raw\") == cfg.raw_binding(key),\n"
  "            f\"{src} was made from other raw data (another commit or other train files than the provenance of {cfg.ext_dirname}/ records).  Nothing was \"\n"
  "            f\"overwritten or deleted.  If the raw data really changed, set RAW_SUFFIX.\")\n"
  "    man = json.loads((src / \"manifest.json\").read_text(encoding=\"utf-8\"))\n"
  "    dst = cfg.shard_root(key, a, b) / key\n"
  "    dst.mkdir(parents=True, exist_ok=True)\n"
  "    for name in SHARD_FILES:\n"
  "        s = src / name\n"
  "        require(s.is_file(), f\"{s} is missing from an earlier run\")\n"
  "        d = dst / name\n"
  "        want = man[\"files\"][name][\"sha256\"] if name in man[\"files\"] else None\n"
  "        if d.is_file() and (want is None or checks.sha256_file(str(d)) == want):\n"
  "            continue\n"
  "        shutil.copyfile(s, d)\n"
  "        if want is not None:\n"
  "            require(checks.sha256_file(str(d)) == want, f\"{name} restored from Drive does not match the shard manifest sha256\")\n"
  "    return True\n"
  "\n"
  "\n"
  "def publish_shard(cfg: Cfg, key: str, a: int, b: Optional[int]) -> None:\n"
  "    src, dst = cfg.shard_root(key, a, b) / key, cfg.shard_drive(key, a, b)\n"
  "    man = read_manifest(src)\n"
  "    require(man.get(\"balanced\") is True, f\"{key} shard {a}-{b}: converter bookkeeping is not balanced\")\n"
  "    require((man.get(\"range\") or {}).get(\"start\") == a and (man.get(\"range\") or {}).get(\"stop\") == b, f\"{key} shard {a}-{b}: manifest range differs\")\n"
  "    for name in SHARD_FILES:\n"
  "        publish_file(src / name, dst / name, hint=_stage_a_hint(dst))\n"
  "    write_once(dst / \"shard_meta.json\", jdump({\"schema\": \"v49_shard_meta/2\", \"key\": key, \"range\": [a, b], \"converter\": cfg.converter_hashes(),\n"
  "                                               \"raw\": cfg.raw_binding(key), \"n_in\": man[\"n_in\"], \"n_out\": man[\"n_out\"]}))\n"
  "\n"
  "\n"
  "def _order_jobs(jobs: List[dict]) -> List[dict]:\n"
  "    \"\"\"Slowest dataset first; the shards of different datasets alternate so that a wave of two pairs a coding shard with a tengentoppa shard.\"\"\"\n"
  "    by_key: Dict[str, List[dict]] = collections.defaultdict(list)\n"
  "    for j in jobs:\n"
  "        by_key[j[\"key\"]].append(j)\n"
  "    ordered, keys = [], [k for k in CONVERT_ORDER if k in by_key]\n"
  "    sharded = [k for k in keys if any(j[\"shard\"] for j in by_key[k])]\n"
  "    ordered += [j for group in itertools.zip_longest(*[by_key[k] for k in sharded]) for j in group if j is not None]\n"
  "    for k in keys:\n"
  "        if k not in sharded:\n"
  "            ordered += by_key[k]\n"
  "    return ordered\n"
  "\n"
  "\n"
  "def preflight_drive(cfg: Cfg, smoke: Optional[dict]) -> None:\n"
  "    need = (smoke or {}).get(\"est_drive_bytes\")\n"
  "    if not need or not cfg.check_disk:\n"
  "        return\n"
  "    try:\n"
  "        free = shutil.disk_usage(cfg.root).free\n"
  "    except OSError:\n"
  "        return\n"
  "    require(free >= need, f\"Drive reports {free:,} B free, the 7a smoke run extrapolates to about {need:,} B for the converted and final rows: not enough room\")\n"
  "\n"
  "\n"
  "@step(\"7b convert full\", flush_on_failure=True)\n"
  "def step_convert_full(cfg: Cfg):\n"
  "    V = _CACHE.get(\"vocab\")\n"
  "    require(V is not None and cfg.vocab_file.is_file(), \"run step 6b first (vocabulary)\")\n"
  "    raw = load_state(cfg, \"step6c_raw_checks\")\n"
  "    require(raw is not None, \"run step 6c first (the record counts): without them 7b cannot check that every converter read every record, cannot decide \"\n"
  "                             \"which datasets have to be converted in shards, and would not guard the time of an unsharded conversion\")\n"
  "    if cfg.needs_sandbox:\n"
  "        ensure_sandbox_ok(cfg)\n"
  "    smoke = load_state(cfg, \"step7a_smoke\")\n"
  "    preflight_drive(cfg, smoke)\n"
  "    jobs, resumed, plans = [], [], {}\n"
  "    for key in cfg.keys:\n"
  "        if (cfg.conv_dir(key) / \"manifest.json\").is_file() or restore_stage_a(cfg, key):          # local rows or the Drive copy of an earlier run\n"
  "            resumed.append(key)\n"
  "            continue\n"
  "        require(smoke is not None, f\"{key} has to be converted, but the state of cell 7a is missing: run cell 7a first (its numbers decide whether Drive has \"\n"
  "                                   f\"room for the rows and whether an unsharded dataset would take too long; without them both guards would be silently off)\")\n"
  "        total = n_records(raw, key)\n"
  "        if key in cfg.shard_keys and total is not None and total > cfg.shard_records:\n"
  "            plans[key] = shard_ranges(total, cfg.shard_records)\n"
  "            for a, b in plans[key]:\n"
  "                if _shard_done_local(cfg, key, a, b) or restore_shard(cfg, key, a, b):\n"
  "                    continue\n"
  "                jobs.append(new_job(key, a, b, cfg.shard_root(key, a, b), shard=True))\n"
  "        else:\n"
  "            est = ((smoke or {}).get(\"estimate_minutes\") or {}).get(key)\n"
  "            if est is not None and est > cfg.max_unsharded_minutes:             # (a require() with an f-string would format `est` even when it is None)\n"
  "                raise StepFailed(f\"{key}: the smoke run estimates ~{est:g} min for an unsharded conversion (limit {cfg.max_unsharded_minutes:g}); a disconnect \"\n"
  "                                 f\"would lose all of it.  Add '{key}' to SHARD_KEYS or raise MAX_UNSHARDED_MINUTES in cell 3a.\")\n"
  "            jobs.append(new_job(key, out_root=cfg.conv_dir(key).parent))\n"
  "    if resumed:\n"
  "        cfg.log(\"  resumed from earlier converted rows: \" + \", \".join(resumed))\n"
  "    jobs = _order_jobs(jobs)\n"
  "    if jobs:\n"
  "        cfg.log(f\"  {len(jobs)} conversion job(s) in waves of {max(1, cfg.parallel)}; each finished job is published to Drive at once\")\n"
  "    todo_keys = sorted({j[\"key\"] for j in jobs} | set(plans))\n"
  "\n"
  "    def finish(ok_jobs, _res):\n"
  "        for j in ok_jobs:\n"
  "            if j[\"shard\"]:\n"
  "                sman = _check_manifest(cfg, j[\"key\"], j[\"out_root\"] / j[\"key\"], j[\"stop\"], n_records(raw, j[\"key\"]), start=j[\"start\"], allow_empty=True)\n"
  "                check_sandbox_backend(j[\"key\"], sman, cfg)\n"
  "                publish_shard(cfg, j[\"key\"], j[\"start\"], j[\"stop\"])\n"
  "                cfg.log(f\"  shard {j['id']} published to Drive\")\n"
  "            else:\n"
  "                man = _check_manifest(cfg, j[\"key\"], cfg.conv_dir(j[\"key\"]), None, n_records(raw, j[\"key\"]))\n"
  "                check_sandbox_backend(j[\"key\"], man, cfg)\n"
  "                publish_stage_a(cfg, j[\"key\"], V)\n"
  "                cfg.log(f\"  {j['key']} published to Drive\")\n"
  "    if jobs:\n"
  "        _run_converters(cfg, jobs, smoke=False, wave_size=max(1, cfg.parallel), on_wave=finish)\n"
  "    for key, ranges in plans.items():\n"
  "        if (cfg.conv_dir(key) / \"manifest.json\").is_file():\n"
  "            continue\n"
  "        roots = [cfg.shard_root(key, a, b) for a, b in ranges]\n"
  "        for (a, b), r in zip(ranges, roots):\n"
  "            require((r / key / \"manifest.json\").is_file(), f\"{key}: shard {a}-{b} has no rows (run 7b again)\")\n"
  "        merged = convert.merge_shards(key, roots, cfg.conv_dir(key).parent)\n"
  "        require(not merged[\"refused_overwrite\"], f\"{key}: merged rows differ from an earlier merge in {cfg.conv_dir(key)}\")\n"
  "        cfg.log(f\"  {key}: {len(ranges)} shards merged into {merged['n_out']:,} rows\")\n"
  "    ck = _convert_ck(cfg)\n"
  "    res = {}\n"
  "    for key in cfg.keys:\n"
  "        d = cfg.conv_dir(key)\n"
  "        n_raw = n_records(raw, key)\n"
  "        man = _check_manifest(cfg, key, d, None, n_raw)\n"
  "        require(not man.get(\"limit\"), f\"{key}: these rows were made with --limit {man.get('limit')}, not a full run\")\n"
  "        check_sandbox_backend(key, man, cfg)\n"
  "        vr = validate_rows(cfg, key, ck)\n"
  "        require(vr[\"rows\"] == man[\"n_out\"], f\"{key}: {vr['rows']} rows read back, manifest says {man['n_out']}\")\n"
  "        cc = _converted_checks(cfg, key, V)\n"
  "        publish_stage_a(cfg, key, V)                                                        # identical files are skipped, nothing is overwritten\n"
  "        res[key] = {\"manifest\": man, \"validated\": vr, \"checks\": cc}\n"
  "        cfg.log(f\"  {key}: {man['n_in']:,} records -> {man['n_out']:,} rows; splits {man['splits']}; top rejects \"\n"
  "                + \", \".join(f\"{k}={v:,}\" for k, v in list(man[\"rejects\"].items())[:3]))\n"
  "    drive_sync(cfg, \"step 7b\")\n"
  "    return (f\"{len(res)} datasets converted ({len(todo_keys)} now, {len(resumed)} resumed); rows re-validated, no id/group/split problem, \"\n"
  "            f\"no row over the cap or over 1% OOV; unfiltered rows are kept on Drive under {STAGE_A_NAME}/{cfg.converter_tag()}/\"), {\"datasets\": res}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 8: filters\n"
  "def dedupe_text(row: dict) -> str:\n"
  "    \"\"\"Text for WITHIN-dataset duplicate detection.  One passage legitimately carries several questions (JaQuAD, HotpotQA), so for those the\n"
  "    question + evidence + answer part is compared, not the passage; everything else is compared whole.\"\"\"\n"
  "    if row.get(\"family\") in (\"jaquad_human\", \"hotpot_en_short\"):\n"
  "        p = row[\"prompt\"]\n"
  "        i = p.rfind(\"\\n質問: \")\n"
  "        return (p[i:] if i >= 0 else p) + row[\"target\"]\n"
  "    return row[\"full_text\"]\n"
  "\n"
  "\n"
  "def full_text(row: dict) -> str:\n"
  "    return row[\"full_text\"]\n"
  "\n"
  "\n"
  "def rows_at(cfg: Cfg, key: str, alive: bytearray):\n"
  "    for i, (_, line) in enumerate(iter_stage_a(cfg, key)):\n"
  "        if alive[i]:\n"
  "            yield json.loads(line)\n"
  "\n"
  "\n"
  "class Combined:\n"
  "    \"\"\"The surviving rows of several datasets as ONE stream (so that 'boilerplate' is judged over all rows); locate() maps back.\"\"\"\n"
  "\n"
  "    def __init__(self, cfg: Cfg, keys, alive: Dict[str, bytearray]):\n"
  "        self.cfg, self.keys, self.alive = cfg, list(keys), alive\n"
  "        self.maps, self.starts, tot = [], [], 0\n"
  "        for k in self.keys:\n"
  "            idx = [i for i, a in enumerate(alive[k]) if a]\n"
  "            self.maps.append(idx)\n"
  "            self.starts.append(tot)\n"
  "            tot += len(idx)\n"
  "        self.total = tot\n"
  "\n"
  "    def rows(self):\n"
  "        for k in self.keys:\n"
  "            yield from rows_at(self.cfg, k, self.alive[k])\n"
  "\n"
  "    def rows_of(self, key: str):\n"
  "        return rows_at(self.cfg, key, self.alive[key])\n"
  "\n"
  "    def locate(self, g: int):\n"
  "        j = bisect.bisect_right(self.starts, g) - 1\n"
  "        return self.keys[j], self.maps[j][g - self.starts[j]]\n"
  "\n"
  "\n"
  "def _drop(alive: Dict[str, bytearray], reason: Dict[str, bytearray], comb: Combined, positions, code: int) -> Dict[str, int]:\n"
  "    per: collections.Counter = collections.Counter()\n"
  "    for g in positions:\n"
  "        k, i = comb.locate(g)\n"
  "        if alive[k][i]:\n"
  "            alive[k][i] = 0\n"
  "            reason[k][i] = code\n"
  "            per[k] += 1\n"
  "    return dict(per)\n"
  "\n"
  "\n"
  "def _eval_queries(cfg: Cfg, sources) -> List[str]:\n"
  "    qs, seen = [], set()\n"
  "    for s in sources:\n"
  "        if not isinstance(s, checks.EvalSource):\n"
  "            continue\n"
  "        try:\n"
  "            for strings in checks.iter_eval_items(s):\n"
  "                q = strings[0].strip() if strings else \"\"\n"
  "                if len(q) >= 20 and q not in seen:\n"
  "                    seen.add(q)\n"
  "                    qs.append(q)\n"
  "        except Exception:                                                           # noqa: BLE001 - the semantic review is optional\n"
  "            continue\n"
  "    if len(qs) > cfg.semantic_max_queries:\n"
  "        qs = sorted(qs, key=sha256_text)[:cfg.semantic_max_queries]\n"
  "    return qs\n"
  "\n"
  "\n"
  "def _examples(items, k=5):\n"
  "    return items[:k]\n"
  "\n"
  "\n"
  "def load_alive(cfg: Cfg, keys) -> Dict[str, bytearray]:\n"
  "    \"\"\"Survivor masks rebuilt from the drop lists that step 8 wrote (so 8b and 9 do not depend on anything in memory).\"\"\"\n"
  "    alive = {}\n"
  "    for k in keys:\n"
  "        n = sum(1 for _ in iter_stage_a(cfg, k))\n"
  "        a = bytearray(b\"\\x01\") * n\n"
  "        p = cfg.out / \"filters\" / f\"{k}_dropped.tsv\"\n"
  "        require(p.is_file(), f\"{p} is missing: run step 8 first\")\n"
  "        for line in p.read_text(encoding=\"utf-8\").splitlines()[1:]:\n"
  "            a[int(line.split(\"\\t\", 1)[0])] = 0\n"
  "        alive[k] = a\n"
  "    return alive\n"
  "\n"
  "\n"
  "@step(\"8 filters\", flush_on_failure=True)\n"
  "def step_filters(cfg: Cfg):\n"
  "    \"\"\"Exact + MinHash duplicates inside a dataset, 13-gram decontamination, the old corpus, near siblings of train inside dev / heldout.  The\n"
  "    drop lists and the state are written as soon as these are done; the optional reviews are step 8b.\"\"\"\n"
  "    keys = cfg.keys\n"
  "    for k in keys:\n"
  "        ensure_stage_a(cfg, k)\n"
  "    index = _CACHE.get(\"eval_index\")\n"
  "    require(index is not None, \"run step 6b first (eval index)\")\n"
  "    raw = load_state(cfg, \"step6c_raw_checks\") or {}\n"
  "    stats: Dict[str, Any] = {k: {\"counts\": {}, \"examples\": {}} for k in keys}\n"
  "    alive: Dict[str, bytearray] = {}\n"
  "    reason: Dict[str, bytearray] = {}\n"
  "    for k in keys:\n"
  "        n = sum(1 for _ in iter_stage_a(cfg, k))\n"
  "        require(n == read_manifest(cfg.conv_dir(k))[\"n_out\"], f\"{k}: converted files and manifest disagree\")\n"
  "        alive[k], reason[k] = bytearray(b\"\\x01\") * n, bytearray(n)\n"
  "        stats[k][\"counts\"][\"converted\"] = n\n"
  "    cut = cfg.near_cut\n"
  "    # ---- 1. exact duplicates and 2. MinHash near duplicates, inside one dataset (keep first)\n"
  "    for k in keys:\n"
  "        comb = Combined(cfg, [k], alive)\n"
  "        ex = checks.exact_dups(comb.rows(), text_fn=dedupe_text, progress=cfg.progress)\n"
  "        _drop(alive, reason, comb, ex[\"within_dup_positions\"], R_EXACT)\n"
  "        stats[k][\"counts\"][\"after_exact_dedup\"] = sum(alive[k])\n"
  "        stats[k][\"examples\"][\"exact\"] = _examples(ex[\"examples\"])\n"
  "        comb = Combined(cfg, [k], alive)\n"
  "        nd = checks.near_dups_within(comb.rows(), text_fn=dedupe_text, threshold=cut, progress=cfg.progress)\n"
  "        _drop(alive, reason, comb, nd[\"dup_positions\"], R_NEAR)\n"
  "        stats[k][\"counts\"][\"after_near_dedup\"] = sum(alive[k])\n"
  "        stats[k][\"examples\"][\"near\"] = _examples(nd[\"examples\"])\n"
  "        stats[k][\"near\"] = {\"clusters\": nd[\"n_clusters_gt1\"], \"largest_cluster\": nd[\"largest_cluster\"], \"cluster_size_hist\": nd[\"cluster_size_hist\"],\n"
  "                            \"backend\": nd[\"hasher_backend\"], \"k\": nd[\"k\"], \"bands\": nd[\"bands\"], \"estimate_cut\": cut, \"target_jaccard\": cfg.near_threshold}\n"
  "        cfg.log(f\"  {k}: converted {stats[k]['counts']['converted']:,} -> exact {stats[k]['counts']['after_exact_dedup']:,} -> near {stats[k]['counts']['after_near_dedup']:,}\"\n"
  "                f\" (clusters {nd['n_clusters_gt1']:,}, largest {nd['largest_cluster']:,})\")\n"
  "    # ---- 3. decontamination against the eval texts: one pass over all datasets\n"
  "    comb = Combined(cfg, keys, alive)\n"
  "    dc = checks.decontaminate(comb.rows(), index, text_fn=full_text, boilerplate_eval_df=cfg.boilerplate_eval_df, min_shared=cfg.min_shared,\n"
  "                              max_examples=10, progress=cfg.progress)\n"
  "    whole_by = dict(collections.Counter(comb.locate(g)[0] for g in dc[\"whole_item_positions\"]))         # before _drop: positions refer to this stream\n"
  "    per = _drop(alive, reason, comb, dc[\"flagged_positions\"], R_EVAL)\n"
  "    for k in keys:\n"
  "        stats[k][\"counts\"][\"after_eval_decontam\"] = sum(alive[k])\n"
  "        stats[k][\"examples\"][\"eval_overlap\"] = []\n"
  "    glob: Dict[str, Any] = {\"decontam\": {\"n_rows\": dc[\"n_rows\"], \"rows_flagged\": dc[\"rows_flagged\"], \"rows_boilerplate_only\": dc[\"rows_boilerplate_only\"],\n"
  "                                         \"rows_flagged_whole_item\": dc[\"rows_flagged_whole_item\"], \"whole_item_min_chars\": dc[\"whole_item_min_chars\"],\n"
  "                                         \"rows_flagged_whole_item_by_dataset\": whole_by,\n"
  "                                         \"rows_with_any_shared_shingle\": dc[\"rows_with_any_shared_shingle\"], \"shared_count_hist\": dc[\"shared_count_hist\"],\n"
  "                                         \"boilerplate\": dc[\"boilerplate\"], \"real_shingles_top\": dc[\"real_shingles_top\"], \"examples\": dc[\"examples\"],\n"
  "                                         \"flagged_by_dataset\": per, \"ngram\": cfg.ngram, \"boilerplate_eval_df\": cfg.boilerplate_eval_df,\n"
  "                                         \"min_shared\": cfg.min_shared, \"limits\": dc[\"limits\"]}}\n"
  "    cfg.log(f\"  decontamination: {dc['rows_flagged']:,} of {dc['n_rows']:,} rows share a {cfg.ngram}-gram with an eval text (or contain a whole eval item of \"\n"
  "            f\"{dc['whole_item_min_chars']}+ characters: {dc['rows_flagged_whole_item']:,}, by dataset {whole_by}) and are dropped \"\n"
  "            f\"({dc['rows_boilerplate_only']:,} rows shared only phrases that {cfg.boilerplate_eval_df}+ eval items share and are kept); by dataset {per}\")\n"
  "    # ---- 4. the old corpus (exact, then MinHash): two streaming passes over the old rows, one index of all new rows\n"
  "    oc = cfg.old_corpus_path\n"
  "    require(oc.is_file(), f\"old corpus not found: {oc}\")\n"
  "    box = [0]\n"
  "\n"
  "    def counted(it):\n"
  "        for r in it:\n"
  "            box[0] += 1\n"
  "            yield r\n"
  "    t0 = time.time()\n"
  "    old_exact = checks.build_exact_index(counted(checks.iter_rows(str(oc))), text_fn=checks.default_text, progress=cfg.progress)\n"
  "    n_old = box[0]\n"
  "    cfg.log(f\"  old corpus: {n_old:,} rows found in {oc.name} ({oc.stat().st_size:,} bytes)  ({time.time() - t0:.0f} s)\")\n"
  "    if cfg.expected_old_rows is not None:\n"
  "        require(n_old == cfg.expected_old_rows, f\"the old corpus has {n_old:,} rows, the plan says {cfg.expected_old_rows:,}: wrong or incomplete file, \"\n"
  "                                                f\"so the dedupe against it would be incomplete\")\n"
  "    comb = Combined(cfg, keys, alive)\n"
  "    ex = checks.exact_dups(comb.rows(), text_fn=full_text, against=old_exact, progress=cfg.progress)\n"
  "    per_ex = _drop(alive, reason, comb, ex.get(\"against_dup_positions\", []), R_OLD_EXACT)\n"
  "    del old_exact\n"
  "    comb = Combined(cfg, keys, alive)\n"
  "    cr = checks.cross_near_dups(comb.rows(), checks.iter_rows(str(oc)), text_fn=full_text, probe_text_fn=checks.default_text,\n"
  "                                threshold=cut, progress=cfg.progress)\n"
  "    per_nr = _drop(alive, reason, comb, cr[\"matched_index_positions\"], R_OLD_NEAR)\n"
  "    glob[\"old_corpus\"] = {\"path\": str(oc), \"rows_found\": n_old, \"rows_expected\": cfg.expected_old_rows, \"bytes\": oc.stat().st_size,\n"
  "                          \"exact_dropped_by_dataset\": per_ex, \"near_dropped_by_dataset\": per_nr, \"probe_rows_scanned\": cr[\"probe_rows_scanned\"],\n"
  "                          \"probe_rows_with_match\": cr[\"probe_rows_with_match\"], \"examples\": cr[\"examples\"], \"threshold\": cfg.near_threshold,\n"
  "                          \"estimate_cut\": cut}\n"
  "    for k in keys:\n"
  "        stats[k][\"counts\"][\"after_old_corpus_dedup\"] = sum(alive[k])\n"
  "    cfg.log(f\"  old corpus dedupe: exact {per_ex}, near {per_nr}\")\n"
  "    # ---- 5. dev / heldout hygiene.  Only JaQuAD (by article) and HotpotQA (by shared paragraph) have a natural group; the four synthetic\n"
  "    # datasets split by ROW, so siblings of one seed can sit in train and in dev / heldout.  A dev / heldout row with a near sibling in train\n"
  "    # (estimated Jaccard >= split_hygiene_threshold) is dropped; paraphrase-level siblings remain, which is why these files are labelled\n"
  "    # 'not group-independent' everywhere.\n"
  "    hyg: Dict[str, Any] = {}\n"
  "    for k in keys:\n"
  "        stats[k][\"split_hygiene\"] = None\n"
  "        if not cfg.run_split_hygiene or k in GROUP_INDEPENDENT:\n"
  "            continue\n"
  "        comb = Combined(cfg, [k], alive)\n"
  "        positions, texts = [], []\n"
  "        for g, row in enumerate(comb.rows()):\n"
  "            if row[\"split\"] != \"train\":\n"
  "                positions.append(g)\n"
  "                texts.append({\"id\": row[\"id\"], \"full_text\": dedupe_text(row)})\n"
  "        info: Dict[str, Any] = {\"dev_heldout_rows\": len(positions), \"dropped\": 0, \"threshold_estimate\": cfg.split_hygiene_threshold}\n"
  "        if texts:\n"
  "            probe = ({\"id\": r[\"id\"], \"full_text\": dedupe_text(r)} for r in comb.rows() if r[\"split\"] == \"train\")\n"
  "            sib = checks.cross_near_dups(texts, probe, threshold=cfg.split_hygiene_threshold, progress=cfg.progress)\n"
  "            info.update({\"dropped\": len(sib[\"matched_index_positions\"]), \"train_rows_scanned\": sib[\"probe_rows_scanned\"], \"examples\": sib[\"examples\"][:3]})\n"
  "            _drop(alive, reason, comb, [positions[i] for i in sib[\"matched_index_positions\"]], R_SPLIT_LEAK)\n"
  "        hyg[k] = info\n"
  "        stats[k][\"split_hygiene\"] = info\n"
  "        if info[\"dropped\"]:\n"
  "            cfg.log(f\"  {k}: {info['dropped']:,} of {info['dev_heldout_rows']:,} dev / heldout rows had a near sibling in train and were dropped\")\n"
  "    glob[\"split_hygiene\"] = {\"datasets\": hyg, \"threshold_estimate\": cfg.split_hygiene_threshold, \"group_independent\": list(GROUP_INDEPENDENT)}\n"
  "    for k in keys:\n"
  "        stats[k][\"counts\"][\"after_split_hygiene\"] = sum(alive[k])\n"
  "        stats[k][\"counts\"][\"final_rows\"] = sum(alive[k])\n"
  "    # ---- the drop lists (one file per dataset) and the statistics: written NOW, before anything optional\n"
  "    for k in keys:\n"
  "        c = stats[k][\"counts\"]\n"
  "        seq = [c[x] for x in FUNNEL]\n"
  "        require(all(a >= b for a, b in zip(seq, seq[1:])), f\"{k}: the funnel is not monotone: {seq}\")\n"
  "        require(c[\"final_rows\"] == sum(alive[k]), f\"{k}: final count disagrees with the survivor mask\")\n"
  "        c[\"n_raw\"] = (raw.get(\"datasets\", {}).get(k) or {}).get(\"n_total\")\n"
  "        lines = [\"index\\tid\\treason\\n\"]\n"
  "        for i, (_, line) in enumerate(iter_stage_a(cfg, k)):\n"
  "            if not alive[k][i]:\n"
  "                lines.append(f\"{i}\\t{json.loads(line)['id']}\\t{REASON[reason[k][i]]}\\n\")\n"
  "        write_once(cfg.out / \"filters\" / f\"{k}_dropped.tsv\", \"\".join(lines))\n"
  "        stats[k][\"dropped_by_reason\"] = dict(collections.Counter(l.rstrip(\"\\n\").split(\"\\t\")[2] for l in lines[1:]))\n"
  "    out = {\"datasets\": stats, \"global\": glob}\n"
  "    save_state(cfg, \"step8_filters\", out)\n"
  "    drive_sync(cfg, \"step 8\")\n"
  "    fin = {k: stats[k][\"counts\"][\"final_rows\"] for k in keys}\n"
  "    return (f\"funnel done for {len(keys)} datasets; old corpus {n_old:,} rows found; decontamination dropped {dc['rows_flagged']:,}; \"\n"
  "            f\"dev/heldout near siblings dropped {sum(v['dropped'] for v in hyg.values()):,}; rows kept per dataset: {fin}\"), out\n"
  "\n"
  "\n"
  "@step(\"8b optional reviews\", flush_on_failure=True)\n"
  "def step_reviews(cfg: Cfg):\n"
  "    \"\"\"Semantic look-alikes of the eval questions (for human reading) and the dataset x dataset duplicate matrix.  Neither removes a row; each\n"
  "    has a wall-clock budget and says so when it was cut short.  Needs only the drop lists of step 8; result saved once (a re-run returns it).\"\"\"\n"
  "    keys = cfg.keys\n"
  "    for k in keys:\n"
  "        ensure_stage_a(cfg, k)\n"
  "    require(load_state(cfg, \"step8_filters\") is not None, \"run step 8 first\")\n"
  "    done = load_state(cfg, \"step8b_reviews\")\n"
  "    if done is not None:\n"
  "        return \"already done (state found; set RUN_SUFFIX to redo it)\", {\"reviews\": done}\n"
  "    alive = load_alive(cfg, keys)\n"
  "    final = Combined(cfg, keys, alive)\n"
  "    out: Dict[str, Any] = {\"semantic\": None, \"cross_dataset\": None, \"rows\": final.total}\n"
  "    if cfg.run_semantic:\n"
  "        info: dict = {}\n"
  "        try:\n"
  "            sources = _CACHE.get(\"eval_sources\") or eval_sources(cfg)[1]\n"
  "            qs = _eval_queries(cfg, sources)\n"
  "            res = checks.semantic_neighbors(qs, final.rows(), text_fn=full_text, id_fn=lambda r, p: r.get(\"id\"), top_n=3, min_cos=cfg.semantic_min_cos,\n"
  "                                            min_shared_blocks=cfg.semantic_min_shared_blocks, max_block_df=cfg.semantic_max_block_df,\n"
  "                                            progress=cfg.progress, deadline=time.time() + cfg.semantic_budget_s, info=info) if qs else []\n"
  "            hit = [r for r in res if r[\"matches\"]]\n"
  "            hit.sort(key=lambda r: -r[\"matches\"][0][\"cosine\"])\n"
  "            text = checks.format_semantic_report(hit[:300])\n"
  "            fname = f\"semantic_review_{hashlib.sha256(text.encode('utf-8')).hexdigest()[:8]}.txt\"\n"
  "            write_once(cfg.out / \"filters\" / fname, text)\n"
  "            best = [r[\"matches\"][0][\"cosine\"] for r in hit]\n"
  "            out[\"semantic\"] = {\"queries\": len(qs), \"queries_with_a_look_alike\": len(hit), \"best_cosine_ge_0.7\": sum(1 for c in best if c >= 0.7),\n"
  "                               \"best_cosine_ge_0.9\": sum(1 for c in best if c >= 0.9), \"min_cos\": cfg.semantic_min_cos,\n"
  "                               \"file\": str(cfg.out / \"filters\" / fname), \"note\": \"for reading only; no row was removed\",\n"
  "                               \"truncated\": bool(info.get(\"truncated\")), \"rows_scanned\": info.get(\"rows_scanned\"), \"rows_total\": final.total,\n"
  "                               \"budget_s\": cfg.semantic_budget_s, \"queries_sampled_from\": \"sha-sorted sample of the eval questions\"}\n"
  "            cfg.log(f\"  semantic look-alikes: {len(hit)} of {len(qs)} eval questions have a row with cosine >= {cfg.semantic_min_cos}\"\n"
  "                    + (f\"  [CUT SHORT by the {cfg.semantic_budget_s:g} s budget after {info.get('rows_scanned'):,} of {final.total:,} rows]\" if info.get(\"truncated\") else \"\"))\n"
  "        except StepFailed:\n"
  "            raise\n"
  "        except Exception as e:                                                      # noqa: BLE001\n"
  "            out[\"semantic\"] = {\"error\": f\"{type(e).__name__}: {e}\"}\n"
  "            cfg.log(f\"!!! WARNING: the optional semantic review failed ({type(e).__name__}: {e}); recorded in the report, the run continues\")\n"
  "    if cfg.run_cross:\n"
  "        info = {}\n"
  "        try:\n"
  "            cd = checks.cross_dataset_near_dups({k: final.rows_of(k) for k in keys}, text_fn=full_text, threshold=cfg.near_cut, progress=cfg.progress,\n"
  "                                                deadline=time.time() + cfg.cross_budget_s, info=info)\n"
  "            out[\"cross_dataset\"] = {\"rows\": cd[\"rows\"], \"rows_with_match_in\": cd[\"rows_with_match_in\"], \"examples\": cd[\"examples\"], \"threshold\": cfg.near_threshold,\n"
  "                                    \"estimate_cut\": cfg.near_cut, \"truncated\": info.get(\"truncated\"), \"budget_s\": cfg.cross_budget_s,\n"
  "                                    \"note\": \"report only; rows stay in their own dataset folder\"}\n"
  "            if info.get(\"truncated\"):\n"
  "                cfg.log(f\"  cross-dataset duplicate matrix CUT SHORT by the {cfg.cross_budget_s:g} s budget ({info['truncated']} phase)\")\n"
  "        except Exception as e:                                                      # noqa: BLE001\n"
  "            out[\"cross_dataset\"] = {\"error\": f\"{type(e).__name__}: {e}\"}\n"
  "            cfg.log(f\"!!! WARNING: the optional cross-dataset duplicate matrix failed ({type(e).__name__}: {e}); recorded, the run continues\")\n"
  "    save_state(cfg, \"step8b_reviews\", out)\n"
  "    drive_sync(cfg, \"step 8b\")\n"
  "    sem, cd = out[\"semantic\"] or {}, out[\"cross_dataset\"] or {}\n"
  "    return (f\"semantic review: {'error' if 'error' in sem else ('cut short' if sem.get('truncated') else ('done' if sem else 'off'))}; \"\n"
  "            f\"duplicate matrix: {'error' if 'error' in cd else ('cut short' if cd.get('truncated') else ('done' if cd else 'off'))}\"), {\"reviews\": out}\n"
  "\n"
  "\n"
  "# ====================================================================================================== 9: final rows, registry, report\n"
  "def adoption_verdict(cfg: Cfg, key: str, final_rows: int, conv: dict, final_verification: Optional[dict] = None) -> dict:\n"
  "    \"\"\"A recommendation from the checks, never a fixed number.  The owner decides; every verdict is provisional until the 100 rows\n"
  "    per dataset have been read by eye.  A dataset on hold (toolace, wizard) is NEVER promoted automatically.\"\"\"\n"
  "    why: List[str] = []\n"
  "    if final_rows == 0:\n"
  "        return {\"code\": \"reject\", \"ja\": \"不採用（採用できる行が0件）\", \"why\": [\"no row survived the checks\"]}\n"
  "    if key == \"toolace\":\n"
  "        return {\"code\": \"hold\", \"ja\": \"保留（生成モデルが未確認: カードにも論文にも明記なし）\", \"why\": [\"adoption is gated on the generator model being verified\"]}\n"
  "    if key == \"wizard\":\n"
  "        a = conv.get(\"arith_error_rate\") or {}\n"
  "        rate, n_st = a.get(\"statement_error_rate\"), a.get(\"statements_checked\") or 0\n"
  "        shown = \"測定不能\" if rate is None else f\"{rate:.2%}（{n_st:,}式中）\"\n"
  "        return {\"code\": \"hold\", \"ja\": f\"保留（計画の「数学の明らかな誤りがあるものは避ける」に該当。測定できた単純な算術式の誤り率は {shown}で、これは誤り率の下限です。\"\n"
  "                                      f\"自動では採用候補に上げません。100行を読んで、あなたが決めてください）\",\n"
  "                \"why\": [f\"measured statement error rate {rate} over {n_st} statements (a lower bound); the owner's plan lists this dataset under 'avoid'\"]}\n"
  "    if key == \"tengentoppa\":\n"
  "        return {\"code\": \"conditional\", \"ja\": \"条件付き候補（Qwen2.5-32B生成・人手確認なし。目視100行を読んでから判断）\", \"why\": [\"generated by Qwen2.5-32B-Instruct, not checked by humans\"]}\n"
  "    if key == \"coding\":\n"
  "        v = final_verification if final_verification is not None else {x[len(\"verification_\"):]: n for x, n in (conv.get(\"extra\") or {}).items() if x.startswith(\"verification_\")}\n"
  "        n_eq = v.get(\"stored_output_equals_sandbox\", 0)\n"
  "        n_un = v.get(\"unverified_no_execution\", 0)\n"
  "        return {\"code\": \"candidate\", \"ja\": f\"採用候補（実行して答え自身の保存出力と一致したpython行 {n_eq:,}行のみ。他言語のcode_text行 {n_un:,}行は未実行・未検証）\",\n"
  "                \"why\": [\"python rows: kept only when the answer's stored output equals the sandbox output\", \"code_text rows: not executed\"]}\n"
  "    if key == \"jaquad\":\n"
  "        return {\"code\": \"candidate\", \"ja\": \"採用候補（人手作成。CC BY-SA 3.0: 継承条件あり）\", \"why\": [\"human annotated\"]}\n"
  "    if key == \"hotpot\":\n"
  "        return {\"code\": \"candidate\", \"ja\": \"採用候補（人手作成・英語の短い行のみ。CC BY-SA 4.0: 継承条件あり）\", \"why\": [\"human annotated; English\"]}\n"
  "    return {\"code\": \"candidate\", \"ja\": \"採用候補\", \"why\": why}\n"
  "\n"
  "\n"
  "def _final_stats(rows_iter) -> dict:\n"
  "    \"\"\"One pass over the final rows (all splits): counts, characters and, for the TRAIN rows, the length information the training plan needs:\n"
  "    mean of len(prompt + target) + 1 (one EOS), mean masked characters and a 64-character-bucket histogram (key = bucket midpoint).\"\"\"\n"
  "    n, target_chars, masked, fam, ver = 0, 0, 0, collections.Counter(), collections.Counter()\n"
  "    tool_rows = 0\n"
  "    en_rows = en_chars = 0\n"
  "    by_split = {sp: {\"rows\": 0, \"target_chars\": 0, \"masked_chars\": 0} for sp in SPLITS}\n"
  "    tr_rows = tr_len = tr_masked = 0\n"
  "    hist: collections.Counter = collections.Counter()\n"
  "    for r in rows_iter:\n"
  "        n += 1\n"
  "        target_chars += len(r[\"target\"])\n"
  "        m = sum(b - a for a, b in r.get(\"loss_mask_spans\") or [])\n"
  "        masked += m\n"
  "        sp = r.get(\"split\")\n"
  "        if sp in by_split:\n"
  "            by_split[sp][\"rows\"] += 1\n"
  "            by_split[sp][\"target_chars\"] += len(r[\"target\"])\n"
  "            by_split[sp][\"masked_chars\"] += m\n"
  "            if sp == \"train\":\n"
  "                length = len(r[\"full_text\"]) + 1\n"
  "                tr_rows += 1\n"
  "                tr_len += length\n"
  "                tr_masked += m\n"
  "                hist[str(LEN_BUCKET * (length // LEN_BUCKET) + LEN_BUCKET // 2)] += 1\n"
  "        fam[r[\"family\"]] += 1\n"
  "        if r.get(\"verification\"):\n"
  "            ver[r[\"verification\"]] += 1\n"
  "        tool_rows += bool(r.get(\"tool_steps\"))\n"
  "        if r.get(\"user_lang\") == \"en\" or r.get(\"answer_lang\") == \"en\" or str(r.get(\"language\", \"\")).startswith(\"en\"):\n"
  "            en_rows += 1\n"
  "            en_chars += len(r[\"target\"]) - m\n"
  "    return {\"rows\": n, \"target_chars\": target_chars, \"masked_chars\": masked, \"supervised_chars\": target_chars - masked, \"families\": dict(fam),\n"
  "            \"verification\": dict(ver), \"rows_with_tool_steps\": tool_rows, \"english_rows\": en_rows, \"english_supervised_chars\": en_chars,\n"
  "            \"by_split\": {sp: dict(v, supervised_chars=v[\"target_chars\"] - v[\"masked_chars\"]) for sp, v in by_split.items()},\n"
  "            \"train_length\": {\"rows\": tr_rows, \"mean_len\": (tr_len / tr_rows) if tr_rows else None, \"mean_masked\": (tr_masked / tr_rows) if tr_rows else None,\n"
  "                             \"length_hist\": dict(sorted(hist.items(), key=lambda kv: int(kv[0])))}}\n"
  "\n"
  "\n"
  "LENGTH_NOTE = (\"mean_len = mean of len(prompt + target) + 1 (one EOS) over the final TRAIN rows; mean_masked = mean characters of the tool outputs that are not \"\n"
  "               \"supervised; length_hist = number of train rows per 64-character bucket of len(prompt + target) + 1, key = the bucket midpoint \"\n"
  "               \"(lower edge + 32); supervised_chars_by_split = target characters minus masked characters, per split\")\n"
  "\n"
  "\n"
  "_SA_URL = {\"CC-BY-SA-3.0\": \"https://creativecommons.org/licenses/by-sa/3.0/\", \"CC-BY-SA-4.0\": \"https://creativecommons.org/licenses/by-sa/4.0/\"}\n"
  "\n"
  "\n"
  "def find_apache_license_text() -> Optional[str]:\n"
  "    \"\"\"The standard Apache-2.0 text (from `Apache License` to `END OF TERMS AND CONDITIONS`), taken from the licence file of an installed\n"
  "    Apache-2.0 project (huggingface_hub is installed by cell 2), because Apache-2.0 section 4(a) asks that recipients get a copy of the\n"
  "    licence and no other source is reachable from the Colab runtime.  None when it cannot be found (the notice then says so).\"\"\"\n"
  "    from importlib import metadata\n"
  "    for name in (\"huggingface_hub\", \"requests\", \"pyarrow\"):\n"
  "        try:\n"
  "            dist = metadata.distribution(name)\n"
  "        except metadata.PackageNotFoundError:\n"
  "            continue\n"
  "        for f in dist.files or []:\n"
  "            if \"LICENSE\" not in f.name.upper():\n"
  "                continue\n"
  "            try:\n"
  "                text = f.read_text(encoding=\"utf-8\")\n"
  "            except (OSError, UnicodeDecodeError, ValueError):\n"
  "                continue\n"
  "            i, j = text.find(\"Apache License\"), text.find(\"END OF TERMS AND CONDITIONS\")\n"
  "            if 0 <= i < j and \"Version 2.0, January 2004\" in text[i:i + 300]:\n"
  "                return text[i:j + len(\"END OF TERMS AND CONDITIONS\")].strip() + \"\\n\"\n"
  "    return None\n"
  "\n"
  "\n"
  "def license_notice(info: dict, key: str, apache_text_available: bool) -> str:\n"
  "    L = [f\"LICENCE NOTICE for the rows of {info['repo']} (pinned commit {info['commit_sha']})\", \"\",\n"
  "         f\"Licence: {info['license_name']}  <{info['license_url']}>\", f\"SPDX-style ids on the dataset card: {', '.join(info['license_ids'])}\", \"\"]\n"
  "    lines = info.get(\"license_lines\") or []\n"
  "    if lines:\n"
  "        L += [\"Lines of the dataset card that mention the licence:\"] + [f\"  | {x}\" for x in lines] + [\"\"]\n"
  "    if any(i.lower().startswith(\"apache\") for i in info[\"license_ids\"]):\n"
  "        L += [\"Apache License 2.0, section 4: you must give recipients of the data or of derivative works a copy of the licence, keep the\",\n"
  "              \"attribution notices, and state that the files were changed (these rows are a converted, filtered adaptation).\",\n"
  "              \"The licence text is in LICENSE-Apache-2.0.txt next to this file.\" if apache_text_available else\n"
  "              \"The licence text could NOT be copied automatically (no installed Apache-2.0 package carried it); get it from https://www.apache.org/licenses/LICENSE-2.0.txt \"\n"
  "              \"and keep it with these rows.\"]\n"
  "    else:\n"
  "        L += [f\"Creative Commons {info['license_name']}: attribution + ShareAlike.  The full legal text is at {info['license_url']} (the licence deed link above).\",\n"
  "              \"These rows are an ADAPTATION of the source data (converted, shortened, filtered).  They may be shared only under CC BY-SA 3.0 / 4.0 \"\n"
  "              \"(or a licence CC lists as compatible), with the attribution in ATTRIBUTION.md.\",\n"
  "              \"DO NOT merge ShareAlike rows into one redistributed single-licence file: JaQuAD (CC BY-SA 3.0) and HotpotQA (CC BY-SA 4.0) both embed \"\n"
  "              \"Wikipedia text, and BY-SA 3.0 and 4.0 rows must stay separately labelled (each row also carries its own `license` and `source`).\"]\n"
  "    L += [\"\", \"Upstream model terms of the generating models (Nemotron-4, Phi-3, Mixtral, calm3, Qwen2.5, WizardLM-2) were NOT checked.\"]\n"
  "    return \"\\n\".join(L) + \"\\n\"\n"
  "\n"
  "\n"
  "def _attribution_md(info: dict, key: str, cfg: Cfg) -> str:\n"
  "    L = [\"# Attribution and licence (keep this file with the rows)\", \"\", f\"Dataset: {info['repo']} (pinned commit `{info['commit_sha']}`)\",\n"
  "         f\"Licence: {info['license_name']} <{info['license_url']}>\", \"Rows in this folder: converted to JSONL rows and filtered by v49_acquire (changes were made).\", \"\"]\n"
  "    L += [f\"- {x}\" for x in info[\"attribution\"]]\n"
  "    if info[\"share_alike\"]:\n"
  "        L += [\"\", \"ShareAlike: these rows are an adaptation of the source data and must be shared under CC BY-SA (or a compatible licence) with the same attribution.\",\n"
  "              \"Do not merge ShareAlike rows into one redistributed single-licence file: JaQuAD (BY-SA 3.0) and HotpotQA (BY-SA 4.0) both embed Wikipedia text and \"\n"
  "              \"must stay separately labelled.  See LICENSE_NOTICE.txt.\"]\n"
  "    L += [\"\", \"Each row's `source` and `license` fields carry the same information.  Upstream model terms were not checked.\", \"\", \"Origin and quality notes:\"]\n"
  "    L += [f\"- {x}\" for x in info[\"generator_notes\"] + info[\"quality_notes\"]]\n"
  "    return \"\\n\".join(L) + \"\\n\"\n"
  "\n"
  "\n"
  "_RUN_INFO_COUNTER = itertools.count(1)\n"
  "\n"
  "\n"
  "def verify_registry_on_drive(cfg: Cfg, reg: dict) -> int:\n"
  "    \"\"\"Re-hash every final file that registry.json lists (after drive_sync: from the fresh mount, not from a local cache).\"\"\"\n"
  "    n = 0\n"
  "    for k, d in reg[\"datasets\"].items():\n"
  "        for name, e in d[\"final\"][\"files\"].items():\n"
  "            p = Path(d[\"paths\"][\"rows\"]) / name\n"
  "            require(p.is_file() and checks.sha256_file(str(p)) == e[\"sha256\"], f\"{k}: {p} on Drive does not match the sha256 in registry.json\")\n"
  "            n += 1\n"
  "    return n\n"
  "\n"
  "\n"
  "@step(\"9 final rows and report\", flush_on_failure=True)\n"
  "def step_finalize(cfg: Cfg):\n"
  "    V = _CACHE.get(\"vocab\")\n"
  "    info5, info6b, raw, st8 = (load_state(cfg, n) for n in (\"step5_acquire\", \"step6b_vocab_eval\", \"step6c_raw_checks\", \"step8_filters\"))\n"
  "    require(all(x is not None for x in (info5, info6b, raw, st8)), \"earlier steps have not run (state missing): run cells 5, 6b, 6c and 8 first\")\n"
  "    reviews = load_state(cfg, \"step8b_reviews\") or {}\n"
  "    smoke7 = last_sandbox_record(cfg)\n"
  "    for k in cfg.keys:\n"
  "        ensure_stage_a(cfg, k)\n"
  "    ck = _convert_ck(cfg)\n"
  "    reg_ds: Dict[str, Any] = {}\n"
  "    final_dir = Path(cfg.work) / f\"final_{cfg.code_tag()}\"\n"
  "    apache = find_apache_license_text()\n"
  "    for k in cfg.keys:\n"
  "        man = read_manifest(cfg.conv_dir(k))\n"
  "        dropped = set()\n"
  "        for line in (cfg.out / \"filters\" / f\"{k}_dropped.tsv\").read_text(encoding=\"utf-8\").splitlines()[1:]:\n"
  "            dropped.add(int(line.split(\"\\t\", 1)[0]))\n"
  "        d = final_dir / k\n"
  "        d.mkdir(parents=True, exist_ok=True)\n"
  "        fh = {sp: open(d / f\"{sp}.jsonl\", \"w\", encoding=\"utf-8\", newline=\"\\n\") for sp in SPLITS}\n"
  "        n_by = collections.Counter()\n"
  "        try:\n"
  "            for i, (sp, line) in enumerate(iter_stage_a(cfg, k)):\n"
  "                if i in dropped:\n"
  "                    continue\n"
  "                fh[sp].write(line if line.endswith(\"\\n\") else line + \"\\n\")\n"
  "                n_by[sp] += 1\n"
  "        finally:\n"
  "            for f in fh.values():\n"
  "                f.close()\n"
  "        want = st8[\"datasets\"][k][\"counts\"][\"final_rows\"]\n"
  "        require(sum(n_by.values()) == want, f\"{k}: {sum(n_by.values())} rows written, the filter step kept {want}\")\n"
  "        files = {}\n"
  "        for sp in SPLITS:\n"
  "            p = d / f\"{sp}.jsonl\"\n"
  "            files[f\"{sp}.jsonl\"] = {\"rows\": n_by[sp], \"bytes\": p.stat().st_size, \"sha256\": checks.sha256_file(str(p))}\n"
  "\n"
  "        def final_rows(sps=SPLITS):\n"
  "            for sp in sps:\n"
  "                yield from checks.iter_rows(str(d / f\"{sp}.jsonl\"))\n"
  "        s_all = _final_stats(final_rows())\n"
  "        tl = s_all[\"train_length\"]\n"
  "        sup_by_split = {sp: s_all[\"by_split\"][sp][\"supervised_chars\"] for sp in SPLITS}\n"
  "        s_train = {\"supervised_chars\": sup_by_split[\"train\"]}\n"
  "        require(s_all[\"rows\"] == want, f\"{k}: re-read {s_all['rows']} rows, expected {want}\")\n"
  "        # every final row is still a valid row\n"
  "        bad = collections.Counter()\n"
  "        for r in final_rows():\n"
  "            why = convert.validate_row(r, ck)\n"
  "            if why:\n"
  "                bad[why] += 1\n"
  "        require(not bad, f\"{k}: final rows that fail validate_row: {dict(bad)}\")\n"
  "        verdict = adoption_verdict(cfg, k, want, man, s_all[\"verification\"])\n"
  "        hold = k in HOLD_KEYS\n"
  "        rd = cfg.rows / HOLD_DIRNAME / k if hold else cfg.rows / k\n"
  "        if hold:                                      # the marker comes FIRST: a run that dies after the rows must not leave held rows without it\n"
  "            write_once(rd / \"DO_NOT_TRAIN_UNTIL_APPROVED.txt\",\n"
  "                       f\"{k}: adoption is ON HOLD ({verdict['ja']}).  These rows were written so that the checks can be read; do not concatenate this folder into a \"\n"
  "                       f\"training set before the owner has approved it.  Every row also says adoption = hold.\\n\")\n"
  "        for name in files:\n"
  "            publish_file(d / name, rd / name)\n"
  "        reading = cfg.out / \"reading\" / f\"final_{k}_100.txt\"\n"
  "        checks.sample_for_reading(final_rows(), n=100, seed=7, out_path=str(reading), title=f\"{k} FINAL rows (after all filters)\", vocab=V, progress=cfg.progress)\n"
  "        a5 = info5[k]\n"
  "        write_once(rd / \"ATTRIBUTION.md\", _attribution_md(a5, k, cfg))\n"
  "        write_once(rd / \"LICENSE_NOTICE.txt\", license_notice(a5, k, apache is not None))\n"
  "        if apache is not None and any(i.lower().startswith(\"apache\") for i in a5[\"license_ids\"]):\n"
  "            write_once(rd / \"LICENSE-Apache-2.0.txt\", apache)\n"
  "        c = st8[\"datasets\"][k][\"counts\"]\n"
  "        manifest = {\"schema\": \"v49_rows_manifest/2\", \"dataset\": k, \"repo\": a5[\"repo\"], \"commit_sha\": a5[\"commit_sha\"], \"license\": a5[\"license_name\"],\n"
  "                    \"row_license\": a5[\"row_defaults\"][\"license\"], \"row_source\": a5[\"row_defaults\"][\"source\"], \"counts\": c, \"files\": files,\n"
  "                    \"supervised_chars\": {\"train\": s_train[\"supervised_chars\"], \"all\": s_all[\"supervised_chars\"]}, \"families\": s_all[\"families\"],\n"
  "                    \"supervised_chars_by_split\": sup_by_split, \"mean_len\": tl[\"mean_len\"], \"mean_masked\": tl[\"mean_masked\"], \"length_hist\": tl[\"length_hist\"],\n"
  "                    \"length_hist_bucket\": LEN_BUCKET, \"length_note\": LENGTH_NOTE,\n"
  "                    \"english\": {\"rows\": s_all[\"english_rows\"], \"supervised_chars\": s_all[\"english_supervised_chars\"]},\n"
  "                    \"verification\": s_all[\"verification\"], \"held\": hold,\n"
  "                    \"split_unit\": man.get(\"split_unit\"), \"group_independent_dev_heldout\": man.get(\"group_independent_dev_heldout\"),\n"
  "                    \"adoption_at_acquire\": a5[\"adoption_at_acquire\"], \"adoption_verdict\": verdict, \"code_sha256\": cfg.code_hashes(),\n"
  "                    \"converter_sha256\": cfg.converter_hashes(),\n"
  "                    \"note\": \"adoption is provisional until the 100 reading rows have been read by eye\"}\n"
  "        write_once(rd / \"manifest.json\", jdump(manifest))\n"
  "        # read back what is on Drive\n"
  "        for name, e in files.items():\n"
  "            require(checks.sha256_file(str(rd / name)) == e[\"sha256\"], f\"{k}: {name} on Drive differs from the file that was written\")\n"
  "        reg_ds[k] = {\"key\": k, \"repo\": a5[\"repo\"], \"commit_sha\": a5[\"commit_sha\"], \"retrieved_utc\": a5[\"retrieved_utc\"], \"held\": hold,\n"
  "                     \"license\": {\"name\": a5[\"license_name\"], \"ids\": a5[\"license_ids\"], \"url\": a5[\"license_url\"], \"share_alike\": a5[\"share_alike\"]},\n"
  "                     \"row_license\": a5[\"row_defaults\"][\"license\"], \"row_source\": a5[\"row_defaults\"][\"source\"], \"language\": a5[\"language\"],\n"
  "                     \"attribution\": a5[\"attribution\"], \"generator_notes\": a5[\"generator_notes\"], \"quality_notes\": a5[\"quality_notes\"],\n"
  "                     \"counts\": c, \"count_check\": {\"expected\": raw[\"datasets\"][k][\"expected\"], \"counted\": raw[\"datasets\"][k].get(\"n_total\", raw[\"datasets\"][k][\"n_raw\"]),\n"
  "                                                  \"match\": raw[\"datasets\"][k][\"count_match\"], \"source\": raw[\"datasets\"][k][\"count_source\"]},\n"
  "                     \"raw\": raw[\"datasets\"][k], \"convert\": {x: man.get(x) for x in (\"n_in\", \"n_out\", \"keep_rate\", \"rejects\", \"families\", \"splits\", \"extra\", \"backends\",\n"
  "                                                                                      \"arith_error_rate\", \"language\", \"fit\", \"verification\", \"python\", \"question_types\",\n"
  "                                                                                      \"supervised_chars\", \"english\", \"split_unit\", \"group_independent_dev_heldout\")},\n"
  "                     \"filters\": st8[\"datasets\"][k], \"supervised_chars\": {\"train\": s_train[\"supervised_chars\"], \"all\": s_all[\"supervised_chars\"]},\n"
  "                     \"supervised_chars_by_split\": sup_by_split, \"mean_len\": tl[\"mean_len\"], \"mean_masked\": tl[\"mean_masked\"], \"length_hist\": tl[\"length_hist\"],\n"
  "                     \"length_hist_bucket\": LEN_BUCKET, \"length_note\": LENGTH_NOTE,\n"
  "                     \"english\": {\"rows\": s_all[\"english_rows\"], \"supervised_chars\": s_all[\"english_supervised_chars\"]},\n"
  "                     \"final\": {\"by_split\": dict(n_by), \"families\": s_all[\"families\"], \"verification\": s_all[\"verification\"], \"rows_with_tool_steps\": s_all[\"rows_with_tool_steps\"],\n"
  "                               \"files\": files}, \"adoption\": {\"at_acquire\": a5[\"adoption_at_acquire\"], \"verdict\": verdict},\n"
  "                     \"paths\": {\"rows\": str(rd), \"reading_raw\": raw[\"datasets\"][k][\"reading_file\"], \"reading_final\": str(reading),\n"
  "                               \"converted_unfiltered\": str(cfg.stage_a_drive(k)), \"dropped_ids\": str(cfg.out / \"filters\" / f\"{k}_dropped.tsv\")}}\n"
  "        cfg.log(f\"  {k}: {c['converted']:,} converted -> {want:,} final rows ({dict(n_by)}); verdict {verdict['code']}\" + (\"  [HOLD folder]\" if hold else \"\"))\n"
  "    mism = raw[\"mismatches\"]\n"
  "    missing_optional = info6b[\"eval\"].get(\"missing_optional\") or []\n"
  "    reg = {\"schema\": \"v49_acquire_registry/2\", \"pipeline_version\": PIPELINE_VERSION, \"owner_decision\": acquire.OWNER_DECISION, \"code_sha256\": cfg.code_hashes(),\n"
  "           \"converter_sha256\": cfg.converter_hashes(),\n"
  "           \"paths\": {\"out\": str(cfg.out), \"rows\": str(cfg.rows), \"hold_rows\": str(cfg.rows / HOLD_DIRNAME), \"external_data\": str(cfg.ext), \"stage_a\": str(cfg.stage_a_root / cfg.converter_tag())},\n"
  "           \"only\": None if not cfg.only_tag else list(cfg.keys), \"complete_run\": not cfg.only_tag,\n"
  "           \"trainable\": [k for k in cfg.keys if k not in HOLD_KEYS], \"held\": [k for k in cfg.keys if k in HOLD_KEYS],\n"
  "           \"params\": {\"max_chars\": cfg.max_chars, \"oov_max_frac\": convert.OOV_MAX_FRAC, \"near_target_jaccard\": cfg.near_threshold, \"near_estimate_cut\": cfg.near_cut,\n"
  "                      \"minhash_k\": checks.MinHasher().k, \"minhash_bands\": checks.MinHashIndex().bands, \"ngram\": cfg.ngram,\n"
  "                      \"minhash_detection_probability\": {str(j): round(p, 3) for j, p in detection_probs(checks.MinHasher().k, cfg.near_cut).items()},\n"
  "                      \"boilerplate_eval_df\": cfg.boilerplate_eval_df, \"min_shared\": cfg.min_shared, \"split_pct_groups\": convert.SPLIT_PCT,\n"
  "                      \"split_hygiene_threshold_estimate\": cfg.split_hygiene_threshold, \"python_requires_stored_output\": not cfg.python_allow_unverified,\n"
  "                      \"en_row_max\": convert.EN_ROW_MAX, \"wizard_max_error_rate_shown\": cfg.wizard_max_error_rate, \"wizard_min_statements\": cfg.wizard_min_statements},\n"
  "           \"sandbox\": smoke7, \"vocab\": info6b[\"vocab\"], \"eval\": info6b[\"eval\"], \"old_corpus\": st8[\"global\"][\"old_corpus\"], \"decontam\": st8[\"global\"][\"decontam\"],\n"
  "           \"split_hygiene\": st8[\"global\"].get(\"split_hygiene\"), \"semantic_review\": reviews.get(\"semantic\"), \"cross_dataset_duplicates\": reviews.get(\"cross_dataset\"),\n"
  "           \"count_mismatches\": mism, \"not_covered\": not_covered(missing_optional), \"datasets\": reg_ds}\n"
  "    reg_text = jdump(reg)\n"
  "    reg = json.loads(reg_text)                        # the report is rendered from the JSON round trip: in a restarted process the saved states come back with\n"
  "    write_once(cfg.out / \"registry.json\", reg_text)   # sorted keys, in this process with insertion order - rendering from the original dict made the same\n"
  "    write_once(cfg.out / \"REPORT_JA.md\", render_report(reg, cfg))        # run produce a different REPORT_JA.md after a restart (and cell 9 was refused)\n"
  "    now = datetime.datetime.now(datetime.timezone.utc).strftime(\"%Y%m%dT%H%M%S_%fZ\")\n"
  "    try:\n"
  "        import numpy as _np\n"
  "        np_ver = _np.__version__\n"
  "    except Exception:                                                               # noqa: BLE001\n"
  "        np_ver = None\n"
  "    write_once(cfg.out / f\"run_info_{now}_{os.getpid()}_{next(_RUN_INFO_COUNTER)}.json\",\n"
  "               jdump({\"utc\": now, \"python\": sys.version.split()[0], \"platform\": platform.platform(), \"numpy\": np_ver,\n"
  "                      \"minhash_backend\": checks.MinHasher().backend, \"cpu_count\": os.cpu_count(), \"timings\": cfg.timings}))\n"
  "    drive_sync(cfg, \"step 9\")\n"
  "    n_checked = verify_registry_on_drive(cfg, reg)\n"
  "    tot = sum(v[\"counts\"][\"final_rows\"] for v in reg_ds.values())\n"
  "    return (f\"{tot:,} rows in {len(reg_ds)} datasets written to {cfg.rows.name}/ ({len(reg['held'])} on hold under {HOLD_DIRNAME}/); {n_checked} files re-hashed \"\n"
  "            f\"after a fresh Drive mount; registry.json and REPORT_JA.md written to {cfg.out.name}/; count mismatches: {len(mism)}\"), {\"registry\": reg}\n"
  "\n"
  "\n"
  "# ====================================================================================================== the Japanese report\n"
  "def detection_probs(k: int, cut: float, js=(0.9, 0.85, 0.8, 0.75, 0.7)) -> dict:\n"
  "    \"\"\"P(estimated Jaccard >= cut) for a pair of TRUE Jaccard j, with a k-permutation MinHash (binomial): how sharp the near-duplicate cut is.\"\"\"\n"
  "    kmin = math.ceil(cut * k - 1e-9)\n"
  "    return {j: sum(math.comb(k, i) * j ** i * (1 - j) ** (k - i) for i in range(kmin, k + 1)) for j in js}\n"
  "\n"
  "\n"
  "def _n(x) -> str:\n"
  "    return \"-\" if x is None else f\"{x:,}\"\n"
  "\n"
  "\n"
  "def _pct(x) -> str:\n"
  "    return \"-\" if x is None else f\"{x:.2%}\"\n"
  "\n"
  "\n"
  "def render_report(reg: dict, cfg: Cfg) -> str:\n"
  "    ds = reg[\"datasets\"]\n"
  "    pr = reg[\"params\"]\n"
  "    L = [\"# v49 外部データ 取得・検査レポート（自動生成）\", \"\",\n"
  "         \"このファイルは colab_acquire_check.ipynb が作りました。数字はすべて今回の実行の実測値で、事前に決めた件数はありません。\",\n"
  "         \"採用判定は検査結果からの暫定の提案です。各データセット100行の目視確認が済むまで確定しません。\", \"\"]\n"
  "    if reg.get(\"only\"):\n"
  "        L += [f\"**これは試運転（ONLY）の報告です。{', '.join(reg['only'])} だけを処理しており、6つすべての完全な実行ではありません。**\", \"\"]\n"
  "    L += [\n"
  "         \"## 1. 結果の一覧\", \"\",\n"
  "         \"| データセット | ライセンス | 元の件数 | 変換後 | 完全重複除去後 | 近似重複除去後 | 評価データ除外後 | 旧コーパス重複除去後 | dev/heldoutの近い兄弟除去後（採用候補の行数） | 採用判定 | 教師あり文字数（train） | 備考 |\",\n"
  "         \"|---|---|---:|---:|---:|---:|---:|---:|---:|---|---:|---|\"]\n"
  "    for k in KEYS:\n"
  "        d = ds.get(k)\n"
  "        if not d:\n"
  "            continue\n"
  "        c = d[\"counts\"]\n"
  "        note = []\n"
  "        if d[\"count_check\"][\"match\"] is False:\n"
  "            note.append(f\"件数がカードと不一致（数えた {_n(d['count_check']['counted'])} / 期待 {_n(d['count_check']['expected'])}）\")\n"
  "        if d[\"license\"][\"share_alike\"]:\n"
  "            note.append(\"継承条件あり\")\n"
  "        if d[\"held\"]:\n"
  "            note.append(f\"**保留**: {HOLD_DIRNAME}/ に置いてあります（学習に混ぜないでください）\")\n"
  "        if d[\"raw\"].get(\"bad_lines\"):\n"
  "            note.append(f\"壊れたJSON行 {_n(d['raw']['bad_lines'])}\")\n"
  "        L.append(f\"| {k} | {d['license']['name']} | {_n(c.get('n_raw'))} | {_n(c['converted'])} | {_n(c['after_exact_dedup'])} | {_n(c['after_near_dedup'])} | \"\n"
  "                 f\"{_n(c['after_eval_decontam'])} | {_n(c['after_old_corpus_dedup'])} | {_n(c['after_split_hygiene'])} | {d['adoption']['verdict']['ja']} | \"\n"
  "                 f\"{_n(d['supervised_chars']['train'])} | {'；'.join(note) or '-'} |\")\n"
  "    oc, dc = reg[\"old_corpus\"], reg[\"decontam\"]\n"
  "    L += [\"\", \"「元の件数」は取得したファイルを端から端まで数えた件数です。「変換後」は形式・長さ（1,900文字以内）・語彙外文字（1%超は除外）・コードの実行検証などを通った行です。\",\n"
  "          \"各段階は前の段階に残った行だけを対象にした順番の絞り込みです。\", \"\",\n"
  "          \"## 2. 全体の確認結果\", \"\",\n"
  "          f\"- 旧コーパス: {_n(oc['rows_found'])} 行を確認（計画の値 {_n(oc['rows_expected'])}）。ファイル `{oc['path']}`。\",\n"
  "          f\"- 語彙: {reg['vocab']['n_single_chars']} 文字（{reg['vocab']['source']}）。コーパスからの推定ではなくモデル自身の語彙ファイルです。\"\n"
  "          + (\"\" if reg[\"vocab\"].get(\"sha_matches_owner_pc_copy\") else \"（注意: PC上の語彙とハッシュが一致しません。ALLOW_VOCAB_MISMATCH で許可された実行です）\"),\n"
  "          f\"- 評価データ: {len(reg['eval']['sources'])} 件の評価ファイルに加え、{', '.join(reg['eval']['extra_sources']) or 'なし'} を照合に使用。\"\n"
  "          f\"13文字の断片（数字と空白を除いたもの）の種類数 {_n(reg['eval']['index']['distinct_shingles'])}。\",\n"
  "          f\"- 評価データとの一致で除外した行: {_n(dc['rows_flagged'])} 行（うち、{dc.get('whole_item_min_chars', 40)}文字以上の評価文が丸ごと入っていた行 {_n(dc.get('rows_flagged_whole_item'))} 行: \"\n"
  "          f\"数字だけが違う型どおりの問題は断片がすべて定型句になるので、丸ごとの一致を別に見ています。データセット別の丸ごと一致: {dc.get('rows_flagged_whole_item_by_dataset', '記録なし')}）。\"\n"
  "          f\"定型句そのものが{dc.get('whole_item_min_chars', 40)}文字以上の評価文でもあるときは、それを含む行がすべて除外されるので、この行数を読んでから除外数を信じてください。\"\n"
  "          f\"評価問題 {dc['boilerplate_eval_df']} 件以上が共有する定型句だけが一致した {_n(dc['rows_boilerplate_only'])} 行は残しました。\"\n"
  "          f\"データセット別: {dc['flagged_by_dataset']}。一致した断片の数の分布: {dc['shared_count_hist']}（1個だけの一致は、よくある言い回しの偶然の一致のことがあります。例は registry.json の decontam.examples）。\"\n"
  "          f\"定型句かどうかは「評価問題側で何件が共有しているか」で決めています（学習行側の数ではありません）。1件の評価問題にしかない文は、何行がそれを写していても除外されます。\",\n"
  "          f\"- 旧コーパスとの重複で除外した行: 完全一致 {oc['exact_dropped_by_dataset']}、近い重複（推定Jaccard {oc['estimate_cut']} 以上）{oc['near_dropped_by_dataset']}。\"]\n"
  "    sh = reg.get(\"split_hygiene\") or {}\n"
  "    if sh.get(\"datasets\"):\n"
  "        L.append(\"- dev / heldout の近い兄弟の除去（学習行と推定Jaccard {0} 以上で似ている dev / heldout 行を落とした数）: \".format(sh[\"threshold_estimate\"])\n"
  "                 + \"、\".join(f\"{k} {_n(v['dropped'])}/{_n(v['dev_heldout_rows'])}\" for k, v in sh[\"datasets\"].items()) + \"。\")\n"
  "    sem, cd = reg.get(\"semantic_review\"), reg.get(\"cross_dataset_duplicates\")\n"
  "    if sem and \"error\" not in sem:\n"
  "        cut = f\"（時間切れ: {_n(sem.get('rows_scanned'))} / {_n(sem.get('rows_total'))} 行まで）\" if sem.get(\"truncated\") else \"\"\n"
  "        L.append(f\"- 意味的な類似の目視用リスト: 評価問題 {_n(sem['queries'])} 問（sha順の標本）のうち {_n(sem['queries_with_a_look_alike'])} 問に似た行があります（`{sem['file']}`）。行は除外していません。{cut}\")\n"
  "    elif sem:\n"
  "        L.append(f\"- 意味的な類似の確認は失敗しました（{sem['error']}）。\")\n"
  "    else:\n"
  "        L.append(\"- 意味的な類似の確認（セル8b）はまだ実行されていません。\")\n"
  "    if cd and \"error\" not in cd:\n"
  "        L.append(f\"- データセット間の近似重複（推定Jaccard {cd['estimate_cut']} 以上）: {cd['rows_with_match_in']}\" + (\"（時間切れで途中まで）\" if cd.get(\"truncated\") else \"\"))\n"
  "    elif cd:\n"
  "        L.append(f\"- データセット間の近似重複の確認は失敗しました（{cd['error']}）。\")\n"
  "    L += [\"\", \"### T1〜T8 の評価データの照合状況\", \"\", \"| 区分 | ファイル（13文字の断片の数） |\", \"|---|---|\"]\n"
  "    for t in reg[\"eval\"].get(\"tiers\", []):\n"
  "        L.append(f\"| {t['tier']} | \" + \"、\".join(f\"{n}（{_n(v['shingles'])}）\" for n, v in t[\"files\"].items()) + \" |\")\n"
  "    L += [\"\", \"## 3. 件数の不一致（MISMATCH）\", \"\"]\n"
  "    if reg[\"count_mismatches\"]:\n"
  "        for m in reg[\"count_mismatches\"]:\n"
  "            L.append(f\"- **MISMATCH** {m['dataset']}: 数えた件数 {_n(m['counted'])} に対し、期待 {_n(m['expected'])}（{m['source']}）。カードや計画の数字が実際と違う可能性があります。\")\n"
  "    else:\n"
  "        L.append(\"- なし（6つともカード・計画の件数と一致）。\")\n"
  "    L += [\"\", \"## 4. データセットごとの詳細\", \"\"]\n"
  "    for k in KEYS:\n"
  "        d = ds.get(k)\n"
  "        if not d:\n"
  "            continue\n"
  "        c, cv, rw = d[\"counts\"], d[\"convert\"], d[\"raw\"]\n"
  "        L += [f\"### {k}（{d['repo']}）\", \"\",\n"
  "              f\"- 取得したコミット: `{d['commit_sha']}`（取得日時 {d['retrieved_utc']}）。言語: {d['language']}。\",\n"
  "              f\"- 採用判定: {d['adoption']['verdict']['ja']}\",\n"
  "              f\"- 変換での主な除外理由: \" + (\", \".join(f\"{r}={v:,}\" for r, v in list((cv.get('rejects') or {}).items())[:6]) or \"なし\"),\n"
  "              f\"- 元データの語彙外文字の割合: {_pct(rw['oov']['oov_frac'])}（1%超の行 {_pct(rw['oov']['share_rows_over_threshold'])}）。2,048文字超の行: {_pct(rw['length']['share_over_2048'])}。\",\n"
  "              f\"- 最終行数 {_n(c['final_rows'])}（分割 {d['final']['by_split']}）。ファミリー別: {d['final']['families']}。教師あり文字数 train {_n(d['supervised_chars']['train'])} / 全体 {_n(d['supervised_chars']['all'])}。\",\n"
  "              f\"- 英語の行: 最終 {_n(d['english']['rows'])} 行（教師あり {_n(d['english']['supervised_chars'])} 文字）。英語の行の長さの上限（{convert.EN_ROW_MAX} 文字）が掛かるのは coding の英語の行だけです。\"\n"
  "              f\"HotpotQA は英語の段落を読む行（英語であることが前提）、ToolACE は英語・中国語の関数呼び出しなので、この上限は掛けず、全体の上限（1,900文字）だけです。\"\n"
  "              f\"変換直後のファミリー別: 英語 {cv.get('english')}、教師あり文字数 {cv.get('supervised_chars')}。\",\n"
  "              f\"- 長さ（最終のtrain行）: len(prompt+target)+1 の平均 {'-' if d.get('mean_len') is None else format(d['mean_len'], '.1f')}、損失から外す文字の平均 \"\n"
  "              f\"{'-' if d.get('mean_masked') is None else format(d['mean_masked'], '.1f')}、教師あり文字数 train {_n(d['supervised_chars_by_split']['train'])} / dev {_n(d['supervised_chars_by_split']['dev'])} / \"\n"
  "              f\"heldout {_n(d['supervised_chars_by_split']['heldout'])}（registry.json の mean_len・length_hist・supervised_chars_by_split。ヒストグラムは64文字刻み）。\",\n"
  "              f\"- 分割の単位: {cv.get('split_unit')}。dev / heldout が学習行から独立しているか: \" + (\"はい（グループ単位）\" if cv.get(\"group_independent_dev_heldout\") else \"**いいえ**（行単位。dev / heldout をグループ独立の評価として使わないでください）\") + \"。\",\n"
  "              f\"- 絞り込みで落ちた行の内訳: {d['filters'].get('dropped_by_reason')}。\"]\n"
  "        near = d[\"filters\"].get(\"near\")\n"
  "        if near:\n"
  "            L.append(f\"- 近似重複: MinHash {near['k']}置換、推定Jaccard {near['estimate_cut']} 以上を同じ群として最初の1行だけ残す（目標は真のJaccard {near['target_jaccard']}）。\"\n"
  "                     f\"群が2行以上のもの {_n(near['clusters'])} 群、最大 {_n(near['largest_cluster'])} 行、群の大きさの分布 {near['cluster_size_hist']}。\"\n"
  "                     \"落とすのは、すでに残した行に似ている行だけです（鎖状につながっても、残す行と似ていない行は落としません）。\")\n"
  "        if d[\"final\"][\"verification\"]:\n"
  "            L.append(f\"- 検証の内訳（最終行）: {d['final']['verification']}。\")\n"
  "        py = cv.get(\"python\")\n"
  "        if py:\n"
  "            L.append(f\"- python行: 実行して保存出力と一致した（検証済みの）行は {_n(py['rows_kept_stored_output_equals_sandbox'])} 行だけです。実行はできたが比べる保存出力が無かった行 \"\n"
  "                     f\"{_n(py['ran_in_sandbox_but_had_no_stored_output'])} 行は「検証済み」として残さず、数だけ記録しました。\")\n"
  "        if cv.get(\"question_types\"):\n"
  "            L.append(f\"- JaQuAD の質問タイプ（変換に届いた行）: {cv['question_types']}。「Multiple sentence reasoning」と「Ambiguous」は証拠が1文に収まらない・答えが定まらないので除外しています。\")\n"
  "        if cv.get(\"arith_error_rate\"):\n"
  "            a = cv[\"arith_error_rate\"]\n"
  "            L.append(f\"- 算術式の検算: {_n(a['statements_checked'])} 式中 {_n(a['statements_wrong'])} 式が誤り（{_pct(a['statement_error_rate'])}）。行単位では {_pct(a['row_error_rate'])}。\"\n"
  "                     \"検算できたのは単純な「a 演算 b = c」だけなので、実際の誤り率の下限です。\")\n"
  "        L += [f\"- 目視用: 元データ100行 `{d['paths']['reading_raw']}`、最終行100行 `{d['paths']['reading_final']}`。\",\n"
  "              f\"- 出力: `{d['paths']['rows']}`（train / dev / heldout の jsonl、manifest.json、ATTRIBUTION.md、LICENSE_NOTICE.txt）。\", \"\"]\n"
  "    L += [\"## 5. 検査の範囲と限界\", \"\"] + [f\"- {x}\" for x in reg[\"not_covered\"]]\n"
  "    probs = detection_probs(pr[\"minhash_k\"], pr[\"near_estimate_cut\"])\n"
  "    L += [\"- 重複の判定: 同一データセット内は完全一致→MinHash（{k}置換・5文字断片。推定Jaccard {cut} 以上を重複とみなす。目標の真のJaccard {t} に対し、見つかる確率は計算上 \".format(\n"
  "        k=pr[\"minhash_k\"], cut=pr[\"near_estimate_cut\"], t=pr[\"near_target_jaccard\"]) + \"、\".join(f\"真のJaccard {j} の組で約{p:.0%}\" for j, p in probs.items())\n"
  "          + \"）。JaQuAD と HotpotQA は1つの文章に複数の質問が付くため、質問＋根拠＋答えの部分で比べています。旧コーパスとは行全体で比べています。\",\n"
  "          \"- 評価データとの照合は数字と空白を除いた13文字の完全一致です。言い換えや翻訳は検出できません。13文字に満たない評価文は断片が作れず、その文だけを含む行は見つかりません。\"\n"
  "          \"13〜39文字の評価文は13文字の断片でしか照合しないので、その断片がすべて定型句（評価問題20件以上が共有）のときは、そのまま写した行も見つかりません\"\n"
  "          \"（別の確認で測った例: v33 のプロンプトの型 50 件と v42 の質問 1 件）。しきい値（13・40・20）は、誤って除外する割合をここでは測れないので変えていません。\",\n"
  "          \"- 件数は実行ごとに決まります。同じコードと同じデータなら同じ結果になります（乱数は固定）。\", \"\"]\n"
  "    L += [\"## 6. ライセンスと表示の義務\", \"\"]\n"
  "    for k in KEYS:\n"
  "        d = ds.get(k)\n"
  "        if d:\n"
  "            L.append(f\"- {k}: {d['license']['name']}\" + (\"。継承条件（ShareAlike）あり: 変換した行も同じライセンス系統で、出典を付けて共有する必要があります。\"\n"
  "                                                          \"JaQuAD（BY-SA 3.0）と HotpotQA（BY-SA 4.0）はどちらも Wikipedia の文章を含むので、1つの単一ライセンスのファイルに混ぜて再配布しないでください。\"\n"
  "                                                          if d[\"license\"][\"share_alike\"] else \"。ライセンス表示と変更点の明記が必要です（ライセンス本文は LICENSE-Apache-2.0.txt）。\")\n"
  "                     + f\" 出典の文言は `{d['paths']['rows']}/ATTRIBUTION.md` と `LICENSE_NOTICE.txt` にあります。\")\n"
  "    L += [\"- 生成モデル（Nemotron-4、Phi-3、Mixtral、calm3、Qwen2.5、WizardLM-2）の利用条件は確認していません。\", \"\"]\n"
  "    L += [\"## 7. 出力の場所\", \"\", f\"- 行データ（採用候補）: `{reg['paths']['rows']}/<データセット>/`\", f\"- 保留のデータセット（{', '.join(reg['held']) or 'なし'}）: `{reg['paths']['hold_rows']}/<データセット>/`（学習に混ぜない）\",\n"
  "          f\"- 絞り込み前の変換結果（再実行ではここから戻す）: `{reg['paths']['stage_a']}/`\", f\"- 目視ファイル・状態・この報告: `{reg['paths']['out']}/`\",\n"
  "          f\"- 取得した元ファイルと出所の記録: `{reg['paths']['external_data']}/`\", \"\"]\n"
  "    return \"\\n\".join(L)\n"
 ),
}
EVAL_EXTRA = {
 "evidence_unseen_eval.jsonl": (
  "{\"id\": \"v48_evidence_unseen_eval_32_2000\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号95の参照先は作業1号21。\\n資料2: 作業1号21の参照先は作業2号14。\\n資料3: 作業別2号57の登録値は39。\\n資料4: 作業2号14の参照先は作業3号35。\\n資料5: 作業別1号56の登録値は47。\\n資料6: 作業別0号91の登録値は61。\\n資料7: 作業3号35の登録値は50。\\n対象作業0号95から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号95から作業1号21から作業2号14から作業3号35です。終点の値は50です。\\n答え: 50。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2001\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号67の参照先は作業4号89。\\n資料2: 作業別3号70の登録値は28。\\n資料3: 作業別2号72の登録値は75。\\n資料4: 作業別1号75の登録値は18。\\n資料5: 作業別0号35の登録値は31。\\n資料6: 作業4号89の登録値は64。\\n資料7: 作業0号35の参照先は作業1号46。\\n資料8: 作業1号46の参照先は作業2号99。\\n資料9: 作業2号99の参照先は作業3号67。\\n対象作業0号35から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号35から作業1号46から作業2号99から作業3号67から作業4号89です。終点の値は64です。\\n答え: 64。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2002\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号63の参照先は作業1号64。\\n資料2: 作業4号94の参照先は作業5号55。\\n資料3: 作業2号15の参照先は作業3号24。\\n資料4: 作業3号24の参照先は作業4号94。\\n資料5: 作業別3号89の登録値は34。\\n資料6: 作業別0号74の登録値は48。\\n資料7: 作業別2号37の登録値は17。\\n資料8: 作業別4号19の登録値は61。\\n資料9: 作業5号55の登録値は24。\\n資料10: 作業1号64の参照先は作業2号15。\\n資料11: 作業別1号51の登録値は54。\\n対象作業0号63から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号63から作業1号64から作業2号15から作業3号24から作業4号94から作業5号55です。終点の値は24です。\\n答え: 24。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2003\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号17の登録値は95。\\n資料2: 作業4号43の参照先は作業5号12。\\n資料3: 作業6号51の登録値は35。\\n資料4: 作業0号59の参照先は作業1号57。\\n資料5: 作業別3号77の登録値は79。\\n資料6: 作業別0号97の登録値は20。\\n資料7: 作業5号12の参照先は作業6号51。\\n資料8: 作業別4号24の登録値は87。\\n資料9: 作業別5号56の登録値は15。\\n資料10: 作業3号91の参照先は作業4号43。\\n資料11: 作業2号72の参照先は作業3号91。\\n資料12: 作業別1号90の登録値は18。\\n資料13: 作業1号57の参照先は作業2号72。\\n対象作業0号59から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号59から作業1号57から作業2号72から作業3号91から作業4号43から作業5号12から作業6号51です。終点の値は35です。\\n答え: 35。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2004\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号92の登録値は80。\\n資料2: 作業別3号92の登録値は95。\\n資料3: 作業4号15の参照先は作業5号58。\\n資料4: 作業別2号29の登録値は62。\\n資料5: 作業1号40の参照先は作業2号72。\\n資料6: 作業別4号29の登録値は80。\\n資料7: 作業2号72の参照先は作業3号45。\\n資料8: 作業3号45の参照先は作業4号15。\\n資料9: 作業0号59の参照先は作業1号40。\\n資料10: 作業別0号63の登録値は73。\\n資料11: 作業5号58の登録値は73。\\n対象作業0号59から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号59から作業1号40から作業2号72から作業3号45から作業4号15から作業5号58です。終点の値は73です。\\n答え: 73。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2005\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号49の参照先は作業3号73。\\n資料2: 作業別1号27の登録値は52。\\n資料3: 作業別3号75の登録値は87。\\n資料4: 作業1号31の参照先は作業2号49。\\n資料5: 作業別4号48の登録値は45。\\n資料6: 作業別2号93の登録値は84。\\n資料7: 作業別5号11の登録値は82。\\n資料8: 作業6号45の登録値は24。\\n資料9: 作業5号51の参照先は作業6号45。\\n資料10: 作業0号91の参照先は作業1号31。\\n資料11: 作業3号73の参照先は作業4号27。\\n資料12: 作業別0号51の登録値は63。\\n資料13: 作業4号27の参照先は作業5号51。\\n対象作業0号91から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号91から作業1号31から作業2号49から作業3号73から作業4号27から作業5号51から作業6号45です。終点の値は24です。\\n答え: 24。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2006\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号77の参照先は作業2号67。\\n資料2: 作業3号59の参照先は作業4号47。\\n資料3: 作業5号61の登録値は80。\\n資料4: 作業0号30の参照先は作業1号77。\\n資料5: 作業2号67の参照先は作業3号59。\\n資料6: 作業別0号83の登録値は94。\\n資料7: 作業別4号60の登録値は84。\\n資料8: 作業別1号40の登録値は80。\\n資料9: 作業別2号81の登録値は62。\\n資料10: 作業別3号54の登録値は40。\\n資料11: 作業4号47の参照先は作業5号61。\\n対象作業0号30から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号30から作業1号77から作業2号67から作業3号59から作業4号47から作業5号61です。終点の値は80です。\\n答え: 80。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2007\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号91の登録値は14。\\n資料2: 作業別4号88の登録値は18。\\n資料3: 作業3号95の参照先は作業4号31。\\n資料4: 作業2号50の参照先は作業3号95。\\n資料5: 作業別3号57の登録値は73。\\n資料6: 作業0号21の参照先は作業1号40。\\n資料7: 作業別5号56の登録値は69。\\n資料8: 作業4号31の参照先は作業5号37。\\n資料9: 作業5号37の参照先は作業6号66。\\n資料10: 作業別1号30の登録値は68。\\n資料11: 作業1号40の参照先は作業2号50。\\n資料12: 作業6号66の登録値は59。\\n資料13: 作業別0号79の登録値は62。\\n対象作業0号21から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号21から作業1号40から作業2号50から作業3号95から作業4号31から作業5号37から作業6号66です。終点の値は59です。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2008\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号89の登録値は17。\\n資料2: 作業0号93の参照先は作業1号14。\\n資料3: 作業別0号25の登録値は88。\\n資料4: 作業2号79の参照先は作業3号91。\\n資料5: 作業別2号86の登録値は43。\\n資料6: 作業3号91の登録値は34。\\n資料7: 作業1号14の参照先は作業2号79。\\n対象作業0号93から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号93から作業1号14から作業2号79から作業3号91です。終点の値は34です。\\n答え: 34。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2009\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号35の登録値は43。\\n資料2: 作業2号65の参照先は作業3号40。\\n資料3: 作業別1号87の登録値は45。\\n資料4: 作業1号59の参照先は作業2号65。\\n資料5: 作業別2号47の登録値は67。\\n資料6: 作業別3号45の登録値は32。\\n資料7: 作業0号52の参照先は作業1号59。\\n資料8: 作業3号40の参照先は作業4号35。\\n資料9: 作業別0号75の登録値は21。\\n対象作業0号52から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号52から作業1号59から作業2号65から作業3号40から作業4号35です。終点の値は43です。\\n答え: 43。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2010\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号80の登録値は43。\\n資料2: 作業別3号68の登録値は37。\\n資料3: 作業2号80の参照先は作業3号50。\\n資料4: 作業別4号45の登録値は58。\\n資料5: 作業5号60の登録値は90。\\n資料6: 作業1号25の参照先は作業2号80。\\n資料7: 作業別1号50の登録値は40。\\n資料8: 作業3号50の参照先は作業4号49。\\n資料9: 作業4号49の参照先は作業5号60。\\n資料10: 作業0号10の参照先は作業1号25。\\n資料11: 作業別0号17の登録値は63。\\n対象作業0号10から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号10から作業1号25から作業2号80から作業3号50から作業4号49から作業5号60です。終点の値は90です。\\n答え: 90。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2011\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号17の参照先は作業3号28。\\n資料2: 作業別1号85の登録値は87。\\n資料3: 作業別0号77の登録値は72。\\n資料4: 作業4号65の参照先は作業5号42。\\n資料5: 作業0号23の参照先は作業1号58。\\n資料6: 作業3号28の参照先は作業4号65。\\n資料7: 作業5号42の参照先は作業6号90。\\n資料8: 作業別4号98の登録値は40。\\n資料9: 作業別5号11の登録値は85。\\n資料10: 作業別3号18の登録値は59。\\n資料11: 作業1号58の参照先は作業2号17。\\n資料12: 作業別2号36の登録値は56。\\n資料13: 作業6号90の登録値は37。\\n対象作業0号23から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号23から作業1号58から作業2号17から作業3号28から作業4号65から作業5号42から作業6号90です。終点の値は37です。\\n答え: 37。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2012\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号93の登録値は57。\\n資料2: 作業2号52の参照先は作業3号49。\\n資料3: 作業1号30の参照先は作業2号52。\\n資料4: 作業別0号50の登録値は31。\\n資料5: 作業4号66の参照先は作業5号26。\\n資料6: 作業3号49の参照先は作業4号66。\\n資料7: 作業0号81の参照先は作業1号30。\\n資料8: 作業別2号10の登録値は73。\\n資料9: 作業5号26の登録値は52。\\n資料10: 作業別4号82の登録値は47。\\n資料11: 作業別1号33の登録値は33。\\n対象作業0号81から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号81から作業1号30から作業2号52から作業3号49から作業4号66から作業5号26です。終点の値は52です。\\n答え: 52。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2013\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号83の登録値は82。\\n資料2: 作業5号43の参照先は作業6号94。\\n資料3: 作業3号44の参照先は作業4号51。\\n資料4: 作業別4号73の登録値は41。\\n資料5: 作業別1号33の登録値は94。\\n資料6: 作業別0号97の登録値は62。\\n資料7: 作業別3号26の登録値は11。\\n資料8: 作業6号94の登録値は22。\\n資料9: 作業別5号94の登録値は51。\\n資料10: 作業2号25の参照先は作業3号44。\\n資料11: 作業1号58の参照先は作業2号25。\\n資料12: 作業4号51の参照先は作業5号43。\\n資料13: 作業0号58の参照先は作業1号58。\\n対象作業0号58から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号58から作業1号58から作業2号25から作業3号44から作業4号51から作業5号43から作業6号94です。終点の値は22です。\\n答え: 22。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2014\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号58の参照先は作業3号74。\\n資料2: 作業5号67の登録値は47。\\n資料3: 作業4号28の参照先は作業5号67。\\n資料4: 作業別0号23の登録値は47。\\n資料5: 作業別3号80の登録値は46。\\n資料6: 作業1号80の参照先は作業2号58。\\n資料7: 作業別4号31の登録値は11。\\n資料8: 作業3号74の参照先は作業4号28。\\n資料9: 作業別1号68の登録値は72。\\n資料10: 作業0号11の参照先は作業1号80。\\n資料11: 作業別2号66の登録値は71。\\n対象作業0号11から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号11から作業1号80から作業2号58から作業3号74から作業4号28から作業5号67です。終点の値は47です。\\n答え: 47。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2015\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号95の参照先は作業5号37。\\n資料2: 作業別3号78の登録値は66。\\n資料3: 作業3号51の参照先は作業4号95。\\n資料4: 作業別1号74の登録値は65。\\n資料5: 作業1号11の参照先は作業2号85。\\n資料6: 作業別4号13の登録値は25。\\n資料7: 作業5号37の参照先は作業6号65。\\n資料8: 作業0号82の参照先は作業1号11。\\n資料9: 作業6号65の登録値は49。\\n資料10: 作業別5号73の登録値は48。\\n資料11: 作業2号85の参照先は作業3号51。\\n資料12: 作業別2号54の登録値は62。\\n資料13: 作業別0号99の登録値は67。\\n対象作業0号82から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号82から作業1号11から作業2号85から作業3号51から作業4号95から作業5号37から作業6号65です。終点の値は49です。\\n答え: 49。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2016\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号42の登録値は70。\\n資料2: 作業0号93の参照先は作業1号41。\\n資料3: 作業3号83の登録値は73。\\n資料4: 作業2号59の参照先は作業3号83。\\n資料5: 作業別1号52の登録値は74。\\n資料6: 作業1号41の参照先は作業2号59。\\n資料7: 作業別0号16の登録値は16。\\n対象作業0号93から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号93から作業1号41から作業2号59から作業3号83です。終点の値は73です。\\n答え: 73。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2017\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号16の登録値は48。\\n資料2: 作業4号53の登録値は42。\\n資料3: 作業3号37の参照先は作業4号53。\\n資料4: 作業別3号16の登録値は29。\\n資料5: 作業0号48の参照先は作業1号35。\\n資料6: 作業別1号95の登録値は67。\\n資料7: 作業1号35の参照先は作業2号76。\\n資料8: 作業2号76の参照先は作業3号37。\\n資料9: 作業別2号72の登録値は22。\\n対象作業0号48から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号48から作業1号35から作業2号76から作業3号37から作業4号53です。終点の値は42です。\\n答え: 42。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2018\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号50の登録値は11。\\n資料2: 作業別2号67の登録値は66。\\n資料3: 作業5号92の登録値は63。\\n資料4: 作業別0号75の登録値は28。\\n資料5: 作業3号50の参照先は作業4号26。\\n資料6: 作業4号26の参照先は作業5号92。\\n資料7: 作業0号96の参照先は作業1号25。\\n資料8: 作業別4号29の登録値は61。\\n資料9: 作業1号25の参照先は作業2号14。\\n資料10: 作業2号14の参照先は作業3号50。\\n資料11: 作業別1号70の登録値は20。\\n対象作業0号96から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号96から作業1号25から作業2号14から作業3号50から作業4号26から作業5号92です。終点の値は63です。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2019\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号37の参照先は作業3号84。\\n資料2: 作業別1号28の登録値は32。\\n資料3: 作業1号86の参照先は作業2号37。\\n資料4: 作業0号80の参照先は作業1号86。\\n資料5: 作業3号84の参照先は作業4号34。\\n資料6: 作業別4号40の登録値は89。\\n資料7: 作業別3号88の登録値は36。\\n資料8: 作業5号18の参照先は作業6号41。\\n資料9: 作業6号41の登録値は82。\\n資料10: 作業別5号79の登録値は47。\\n資料11: 作業4号34の参照先は作業5号18。\\n資料12: 作業別0号78の登録値は42。\\n資料13: 作業別2号45の登録値は32。\\n対象作業0号80から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号80から作業1号86から作業2号37から作業3号84から作業4号34から作業5号18から作業6号41です。終点の値は82です。\\n答え: 82。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2020\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号15の参照先は作業5号80。\\n資料2: 作業2号72の参照先は作業3号10。\\n資料3: 作業別2号48の登録値は73。\\n資料4: 作業0号35の参照先は作業1号35。\\n資料5: 作業別4号21の登録値は99。\\n資料6: 作業5号80の登録値は70。\\n資料7: 作業別1号79の登録値は19。\\n資料8: 作業別3号86の登録値は37。\\n資料9: 作業別0号87の登録値は94。\\n資料10: 作業1号35の参照先は作業2号72。\\n資料11: 作業3号10の参照先は作業4号15。\\n対象作業0号35から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号35から作業1号35から作業2号72から作業3号10から作業4号15から作業5号80です。終点の値は70です。\\n答え: 70。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2021\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号79の参照先は作業4号90。\\n資料2: 作業0号70の参照先は作業1号31。\\n資料3: 作業1号31の参照先は作業2号62。\\n資料4: 作業別5号50の登録値は36。\\n資料5: 作業別4号60の登録値は24。\\n資料6: 作業別3号10の登録値は69。\\n資料7: 作業別0号49の登録値は17。\\n資料8: 作業5号27の参照先は作業6号14。\\n資料9: 作業6号14の登録値は63。\\n資料10: 作業別2号52の登録値は94。\\n資料11: 作業4号90の参照先は作業5号27。\\n資料12: 作業別1号18の登録値は67。\\n資料13: 作業2号62の参照先は作業3号79。\\n対象作業0号70から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号70から作業1号31から作業2号62から作業3号79から作業4号90から作業5号27から作業6号14です。終点の値は63です。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2022\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号30の登録値は80。\\n資料2: 作業5号77の登録値は81。\\n資料3: 作業0号95の参照先は作業1号13。\\n資料4: 作業4号24の参照先は作業5号77。\\n資料5: 作業別0号63の登録値は82。\\n資料6: 作業1号13の参照先は作業2号31。\\n資料7: 作業別3号17の登録値は39。\\n資料8: 作業別2号75の登録値は90。\\n資料9: 作業2号31の参照先は作業3号65。\\n資料10: 作業別4号72の登録値は41。\\n資料11: 作業3号65の参照先は作業4号24。\\n対象作業0号95から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号95から作業1号13から作業2号31から作業3号65から作業4号24から作業5号77です。終点の値は81です。\\n答え: 81。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2023\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業6号47の登録値は59。\\n資料2: 作業別5号46の登録値は49。\\n資料3: 作業別1号78の登録値は79。\\n資料4: 作業4号13の参照先は作業5号43。\\n資料5: 作業別2号44の登録値は68。\\n資料6: 作業別0号62の登録値は53。\\n資料7: 作業別4号66の登録値は74。\\n資料8: 作業3号16の参照先は作業4号13。\\n資料9: 作業1号45の参照先は作業2号25。\\n資料10: 作業5号43の参照先は作業6号47。\\n資料11: 作業0号12の参照先は作業1号45。\\n資料12: 作業別3号45の登録値は35。\\n資料13: 作業2号25の参照先は作業3号16。\\n対象作業0号12から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号12から作業1号45から作業2号25から作業3号16から作業4号13から作業5号43から作業6号47です。終点の値は59です。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2024\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号23の参照先は作業1号65。\\n資料2: 作業別0号34の登録値は63。\\n資料3: 作業別1号35の登録値は15。\\n資料4: 作業別2号52の登録値は66。\\n資料5: 作業1号65の参照先は作業2号99。\\n資料6: 作業3号25の登録値は50。\\n資料7: 作業2号99の参照先は作業3号25。\\n対象作業0号23から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号23から作業1号65から作業2号99から作業3号25です。終点の値は50です。\\n答え: 50。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2025\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号75の登録値は63。\\n資料2: 作業別3号40の登録値は28。\\n資料3: 作業1号20の参照先は作業2号80。\\n資料4: 作業別2号22の登録値は95。\\n資料5: 作業2号80の参照先は作業3号64。\\n資料6: 作業0号84の参照先は作業1号20。\\n資料7: 作業4号22の登録値は73。\\n資料8: 作業3号64の参照先は作業4号22。\\n資料9: 作業別0号94の登録値は48。\\n対象作業0号84から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号84から作業1号20から作業2号80から作業3号64から作業4号22です。終点の値は73です。\\n答え: 73。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2026\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号67の登録値は29。\\n資料2: 作業1号48の参照先は作業2号12。\\n資料3: 作業別3号98の登録値は16。\\n資料4: 作業別1号26の登録値は98。\\n資料5: 作業3号72の参照先は作業4号94。\\n資料6: 作業2号12の参照先は作業3号72。\\n資料7: 作業4号94の参照先は作業5号28。\\n資料8: 作業5号28の登録値は25。\\n資料9: 作業0号57の参照先は作業1号48。\\n資料10: 作業別2号24の登録値は56。\\n資料11: 作業別4号56の登録値は55。\\n対象作業0号57から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号57から作業1号48から作業2号12から作業3号72から作業4号94から作業5号28です。終点の値は25です。\\n答え: 25。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2027\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号12の参照先は作業5号43。\\n資料2: 作業別4号24の登録値は78。\\n資料3: 作業別0号45の登録値は63。\\n資料4: 作業2号15の参照先は作業3号71。\\n資料5: 作業別2号59の登録値は10。\\n資料6: 作業別3号35の登録値は96。\\n資料7: 作業6号76の登録値は86。\\n資料8: 作業別1号13の登録値は48。\\n資料9: 作業1号30の参照先は作業2号15。\\n資料10: 作業3号71の参照先は作業4号12。\\n資料11: 作業0号45の参照先は作業1号30。\\n資料12: 作業5号43の参照先は作業6号76。\\n資料13: 作業別5号50の登録値は57。\\n対象作業0号45から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号45から作業1号30から作業2号15から作業3号71から作業4号12から作業5号43から作業6号76です。終点の値は86です。\\n答え: 86。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2028\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号36の登録値は90。\\n資料2: 作業1号25の参照先は作業2号82。\\n資料3: 作業別2号21の登録値は79。\\n資料4: 作業別4号42の登録値は70。\\n資料5: 作業2号82の参照先は作業3号63。\\n資料6: 作業別0号92の登録値は88。\\n資料7: 作業4号32の参照先は作業5号75。\\n資料8: 作業5号75の登録値は26。\\n資料9: 作業別3号97の登録値は99。\\n資料10: 作業0号87の参照先は作業1号25。\\n資料11: 作業3号63の参照先は作業4号32。\\n対象作業0号87から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号87から作業1号25から作業2号82から作業3号63から作業4号32から作業5号75です。終点の値は26です。\\n答え: 26。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2029\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号16の登録値は65。\\n資料2: 作業1号76の参照先は作業2号84。\\n資料3: 作業別2号56の登録値は89。\\n資料4: 作業0号98の参照先は作業1号76。\\n資料5: 作業4号17の参照先は作業5号12。\\n資料6: 作業5号12の参照先は作業6号33。\\n資料7: 作業6号33の登録値は63。\\n資料8: 作業2号84の参照先は作業3号27。\\n資料9: 作業別0号31の登録値は79。\\n資料10: 作業別4号35の登録値は29。\\n資料11: 作業3号27の参照先は作業4号17。\\n資料12: 作業別5号58の登録値は61。\\n資料13: 作業別3号86の登録値は22。\\n対象作業0号98から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号98から作業1号76から作業2号84から作業3号27から作業4号17から作業5号12から作業6号33です。終点の値は63です。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2030\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号71の参照先は作業2号25。\\n資料2: 作業2号25の参照先は作業3号44。\\n資料3: 作業別2号67の登録値は15。\\n資料4: 作業別4号69の登録値は23。\\n資料5: 作業3号44の参照先は作業4号89。\\n資料6: 作業別0号32の登録値は72。\\n資料7: 作業0号28の参照先は作業1号71。\\n資料8: 作業4号89の参照先は作業5号35。\\n資料9: 作業5号35の登録値は63。\\n資料10: 作業別1号90の登録値は49。\\n資料11: 作業別3号20の登録値は53。\\n対象作業0号28から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号28から作業1号71から作業2号25から作業3号44から作業4号89から作業5号35です。終点の値は63です。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_32_2031\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業5号35の参照先は作業6号93。\\n資料2: 作業別3号24の登録値は38。\\n資料3: 作業2号31の参照先は作業3号62。\\n資料4: 作業1号31の参照先は作業2号31。\\n資料5: 作業別1号19の登録値は91。\\n資料6: 作業4号77の参照先は作業5号35。\\n資料7: 作業別5号56の登録値は34。\\n資料8: 作業0号59の参照先は作業1号31。\\n資料9: 作業3号62の参照先は作業4号77。\\n資料10: 作業6号93の登録値は84。\\n資料11: 作業別4号38の登録値は92。\\n資料12: 作業別2号44の登録値は20。\\n資料13: 作業別0号83の登録値は15。\\n対象作業0号59から参照先を順にたどり、終点の値をそのまま答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号59から作業1号31から作業2号31から作業3号62から作業4号77から作業5号35から作業6号93です。終点の値は84です。\\n答え: 84。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2000\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号81の参照先は作業2号88。\\n資料2: 作業別2号86の登録値は56。\\n資料3: 作業2号88の参照先は作業3号52。\\n資料4: 作業3号52の登録値は62。\\n資料5: 作業0号92の参照先は作業1号81。\\n資料6: 作業別0号93の登録値は21。\\n資料7: 作業別1号71の登録値は63。\\n対象作業0号92から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号92から作業1号81から作業2号88から作業3号52です。終点の値は62です。\\n答え: 72。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2001\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号85の参照先は作業2号49。\\n資料2: 作業2号49の参照先は作業3号25。\\n資料3: 作業0号22の参照先は作業1号85。\\n資料4: 作業別1号16の登録値は28。\\n資料5: 作業別0号52の登録値は98。\\n資料6: 作業3号25の参照先は作業4号61。\\n資料7: 作業別3号45の登録値は51。\\n資料8: 作業4号61の登録値は89。\\n資料9: 作業別2号12の登録値は66。\\n対象作業0号22から参照先を順にたどり、終点の値を3だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号22から作業1号85から作業2号49から作業3号25から作業4号61です。終点の値は89です。\\n答え: 92。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2002\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号30の参照先は作業5号77。\\n資料2: 作業1号30の参照先は作業2号41。\\n資料3: 作業別4号26の登録値は85。\\n資料4: 作業別1号47の登録値は94。\\n資料5: 作業別3号53の登録値は50。\\n資料6: 作業別2号45の登録値は62。\\n資料7: 作業別0号92の登録値は22。\\n資料8: 作業5号77の登録値は80。\\n資料9: 作業2号41の参照先は作業3号45。\\n資料10: 作業0号39の参照先は作業1号30。\\n資料11: 作業3号45の参照先は作業4号30。\\n対象作業0号39から参照先を順にたどり、終点の値を12だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号39から作業1号30から作業2号41から作業3号45から作業4号30から作業5号77です。終点の値は80です。\\n答え: 92。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2003\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号60の登録値は47。\\n資料2: 作業別4号27の登録値は53。\\n資料3: 作業0号40の参照先は作業1号89。\\n資料4: 作業1号89の参照先は作業2号96。\\n資料5: 作業別5号46の登録値は41。\\n資料6: 作業5号84の参照先は作業6号82。\\n資料7: 作業別1号55の登録値は95。\\n資料8: 作業2号96の参照先は作業3号72。\\n資料9: 作業別2号65の登録値は85。\\n資料10: 作業3号72の参照先は作業4号98。\\n資料11: 作業6号82の登録値は45。\\n資料12: 作業別3号73の登録値は33。\\n資料13: 作業4号98の参照先は作業5号84。\\n対象作業0号40から参照先を順にたどり、終点の値を9だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号40から作業1号89から作業2号96から作業3号72から作業4号98から作業5号84から作業6号82です。終点の値は45です。\\n答え: 54。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2004\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号32の参照先は作業5号39。\\n資料2: 作業1号12の参照先は作業2号27。\\n資料3: 作業別2号62の登録値は21。\\n資料4: 作業2号27の参照先は作業3号67。\\n資料5: 作業別3号75の登録値は72。\\n資料6: 作業別4号43の登録値は25。\\n資料7: 作業別1号70の登録値は84。\\n資料8: 作業別0号60の登録値は23。\\n資料9: 作業0号79の参照先は作業1号12。\\n資料10: 作業3号67の参照先は作業4号32。\\n資料11: 作業5号39の登録値は53。\\n対象作業0号79から参照先を順にたどり、終点の値を6だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号79から作業1号12から作業2号27から作業3号67から作業4号32から作業5号39です。終点の値は53です。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2005\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号81の参照先は作業5号35。\\n資料2: 作業1号79の参照先は作業2号47。\\n資料3: 作業別4号51の登録値は48。\\n資料4: 作業別1号22の登録値は89。\\n資料5: 作業別2号83の登録値は79。\\n資料6: 作業別0号65の登録値は80。\\n資料7: 作業0号87の参照先は作業1号79。\\n資料8: 作業5号35の参照先は作業6号19。\\n資料9: 作業別3号83の登録値は39。\\n資料10: 作業別5号35の登録値は69。\\n資料11: 作業2号47の参照先は作業3号46。\\n資料12: 作業3号46の参照先は作業4号81。\\n資料13: 作業6号19の登録値は87。\\n対象作業0号87から参照先を順にたどり、終点の値を7だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号87から作業1号79から作業2号47から作業3号46から作業4号81から作業5号35から作業6号19です。終点の値は87です。\\n答え: 94。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2006\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号45の参照先は作業4号29。\\n資料2: 作業別2号44の登録値は93。\\n資料3: 作業4号29の参照先は作業5号48。\\n資料4: 作業1号11の参照先は作業2号95。\\n資料5: 作業0号59の参照先は作業1号11。\\n資料6: 作業2号95の参照先は作業3号45。\\n資料7: 作業別3号72の登録値は71。\\n資料8: 作業別4号57の登録値は46。\\n資料9: 作業5号48の登録値は74。\\n資料10: 作業別1号44の登録値は83。\\n資料11: 作業別0号13の登録値は58。\\n対象作業0号59から参照先を順にたどり、終点の値を9だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号59から作業1号11から作業2号95から作業3号45から作業4号29から作業5号48です。終点の値は74です。\\n答え: 83。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2007\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別5号63の登録値は15。\\n資料2: 作業6号30の登録値は49。\\n資料3: 作業別3号19の登録値は28。\\n資料4: 作業4号17の参照先は作業5号34。\\n資料5: 作業5号34の参照先は作業6号30。\\n資料6: 作業別0号60の登録値は34。\\n資料7: 作業2号53の参照先は作業3号64。\\n資料8: 作業0号32の参照先は作業1号13。\\n資料9: 作業1号13の参照先は作業2号53。\\n資料10: 作業別1号74の登録値は75。\\n資料11: 作業別2号72の登録値は72。\\n資料12: 作業別4号45の登録値は82。\\n資料13: 作業3号64の参照先は作業4号17。\\n対象作業0号32から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号32から作業1号13から作業2号53から作業3号64から作業4号17から作業5号34から作業6号30です。終点の値は49です。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2008\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号67の参照先は作業3号14。\\n資料2: 作業1号71の参照先は作業2号67。\\n資料3: 作業3号14の登録値は69。\\n資料4: 作業別0号53の登録値は79。\\n資料5: 作業別1号59の登録値は96。\\n資料6: 作業0号21の参照先は作業1号71。\\n資料7: 作業別2号59の登録値は48。\\n対象作業0号21から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号21から作業1号71から作業2号67から作業3号14です。終点の値は69です。\\n答え: 79。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2009\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号32の登録値は30。\\n資料2: 作業4号17の登録値は60。\\n資料3: 作業3号67の参照先は作業4号17。\\n資料4: 作業1号63の参照先は作業2号99。\\n資料5: 作業別0号84の登録値は40。\\n資料6: 作業0号71の参照先は作業1号63。\\n資料7: 作業別3号29の登録値は81。\\n資料8: 作業別2号52の登録値は87。\\n資料9: 作業2号99の参照先は作業3号67。\\n対象作業0号71から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号71から作業1号63から作業2号99から作業3号67から作業4号17です。終点の値は60です。\\n答え: 70。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2010\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号86の登録値は37。\\n資料2: 作業5号51の登録値は39。\\n資料3: 作業別4号93の登録値は93。\\n資料4: 作業3号82の参照先は作業4号67。\\n資料5: 作業4号67の参照先は作業5号51。\\n資料6: 作業別1号11の登録値は69。\\n資料7: 作業別3号47の登録値は52。\\n資料8: 作業1号98の参照先は作業2号41。\\n資料9: 作業別2号83の登録値は17。\\n資料10: 作業2号41の参照先は作業3号82。\\n資料11: 作業0号18の参照先は作業1号98。\\n対象作業0号18から参照先を順にたどり、終点の値を7だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号18から作業1号98から作業2号41から作業3号82から作業4号67から作業5号51です。終点の値は39です。\\n答え: 46。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2011\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号12の参照先は作業1号43。\\n資料2: 作業別4号97の登録値は25。\\n資料3: 作業別0号32の登録値は58。\\n資料4: 作業1号43の参照先は作業2号40。\\n資料5: 作業6号17の登録値は82。\\n資料6: 作業別3号94の登録値は71。\\n資料7: 作業別1号10の登録値は47。\\n資料8: 作業2号40の参照先は作業3号50。\\n資料9: 作業別2号16の登録値は62。\\n資料10: 作業3号50の参照先は作業4号14。\\n資料11: 作業別5号92の登録値は15。\\n資料12: 作業5号30の参照先は作業6号17。\\n資料13: 作業4号14の参照先は作業5号30。\\n対象作業0号12から参照先を順にたどり、終点の値を4だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号12から作業1号43から作業2号40から作業3号50から作業4号14から作業5号30から作業6号17です。終点の値は82です。\\n答え: 86。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2012\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号11の参照先は作業4号36。\\n資料2: 作業1号26の参照先は作業2号72。\\n資料3: 作業別0号91の登録値は64。\\n資料4: 作業0号12の参照先は作業1号26。\\n資料5: 作業別1号23の登録値は80。\\n資料6: 作業別4号19の登録値は13。\\n資料7: 作業2号72の参照先は作業3号11。\\n資料8: 作業4号36の参照先は作業5号17。\\n資料9: 作業別3号10の登録値は88。\\n資料10: 作業別2号91の登録値は79。\\n資料11: 作業5号17の登録値は84。\\n対象作業0号12から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号12から作業1号26から作業2号72から作業3号11から作業4号36から作業5号17です。終点の値は84です。\\n答え: 94。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2013\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別5号74の登録値は87。\\n資料2: 作業5号40の参照先は作業6号67。\\n資料3: 作業6号67の登録値は56。\\n資料4: 作業4号34の参照先は作業5号40。\\n資料5: 作業0号75の参照先は作業1号37。\\n資料6: 作業別4号36の登録値は48。\\n資料7: 作業別2号50の登録値は43。\\n資料8: 作業2号24の参照先は作業3号41。\\n資料9: 作業別0号76の登録値は23。\\n資料10: 作業3号41の参照先は作業4号34。\\n資料11: 作業1号37の参照先は作業2号24。\\n資料12: 作業別1号23の登録値は84。\\n資料13: 作業別3号58の登録値は98。\\n対象作業0号75から参照先を順にたどり、終点の値を5だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号75から作業1号37から作業2号24から作業3号41から作業4号34から作業5号40から作業6号67です。終点の値は56です。\\n答え: 61。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2014\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号19の登録値は46。\\n資料2: 作業3号40の参照先は作業4号30。\\n資料3: 作業別0号57の登録値は85。\\n資料4: 作業5号25の登録値は48。\\n資料5: 作業1号31の参照先は作業2号49。\\n資料6: 作業4号30の参照先は作業5号25。\\n資料7: 作業別3号11の登録値は29。\\n資料8: 作業0号16の参照先は作業1号31。\\n資料9: 作業2号49の参照先は作業3号40。\\n資料10: 作業別1号75の登録値は46。\\n資料11: 作業別2号87の登録値は87。\\n対象作業0号16から参照先を順にたどり、終点の値を9だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号16から作業1号31から作業2号49から作業3号40から作業4号30から作業5号25です。終点の値は48です。\\n答え: 57。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2015\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号29の参照先は作業2号93。\\n資料2: 作業別5号52の登録値は49。\\n資料3: 作業4号10の参照先は作業5号68。\\n資料4: 作業0号68の参照先は作業1号29。\\n資料5: 作業5号68の参照先は作業6号49。\\n資料6: 作業別2号54の登録値は90。\\n資料7: 作業2号93の参照先は作業3号90。\\n資料8: 作業3号90の参照先は作業4号10。\\n資料9: 作業別0号15の登録値は39。\\n資料10: 作業別1号78の登録値は80。\\n資料11: 作業別3号63の登録値は10。\\n資料12: 作業別4号69の登録値は87。\\n資料13: 作業6号49の登録値は23。\\n対象作業0号68から参照先を順にたどり、終点の値を11だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号68から作業1号29から作業2号93から作業3号90から作業4号10から作業5号68から作業6号49です。終点の値は23です。\\n答え: 34。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2016\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号94の登録値は63。\\n資料2: 作業別2号54の登録値は12。\\n資料3: 作業0号95の参照先は作業1号20。\\n資料4: 作業2号99の参照先は作業3号49。\\n資料5: 作業別0号69の登録値は67。\\n資料6: 作業1号20の参照先は作業2号99。\\n資料7: 作業3号49の登録値は63。\\n対象作業0号95から参照先を順にたどり、終点の値を11だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号95から作業1号20から作業2号99から作業3号49です。終点の値は63です。\\n答え: 74。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2017\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号19の登録値は48。\\n資料2: 作業1号97の参照先は作業2号52。\\n資料3: 作業別0号76の登録値は98。\\n資料4: 作業別1号84の登録値は89。\\n資料5: 作業4号64の登録値は22。\\n資料6: 作業0号54の参照先は作業1号97。\\n資料7: 作業3号14の参照先は作業4号64。\\n資料8: 作業別2号36の登録値は37。\\n資料9: 作業2号52の参照先は作業3号14。\\n対象作業0号54から参照先を順にたどり、終点の値を12だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号54から作業1号97から作業2号52から作業3号14から作業4号64です。終点の値は22です。\\n答え: 34。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2018\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号63の登録値は49。\\n資料2: 作業1号85の参照先は作業2号61。\\n資料3: 作業4号24の参照先は作業5号30。\\n資料4: 作業2号61の参照先は作業3号77。\\n資料5: 作業3号77の参照先は作業4号24。\\n資料6: 作業別4号15の登録値は78。\\n資料7: 作業別2号81の登録値は31。\\n資料8: 作業別0号56の登録値は65。\\n資料9: 作業別1号34の登録値は67。\\n資料10: 作業0号30の参照先は作業1号85。\\n資料11: 作業5号30の登録値は56。\\n対象作業0号30から参照先を順にたどり、終点の値を6だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号30から作業1号85から作業2号61から作業3号77から作業4号24から作業5号30です。終点の値は56です。\\n答え: 62。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2019\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号67の登録値は94。\\n資料2: 作業別4号85の登録値は58。\\n資料3: 作業別0号98の登録値は18。\\n資料4: 作業4号75の参照先は作業5号88。\\n資料5: 作業別2号64の登録値は51。\\n資料6: 作業別1号71の登録値は70。\\n資料7: 作業2号36の参照先は作業3号57。\\n資料8: 作業1号22の参照先は作業2号36。\\n資料9: 作業5号88の参照先は作業6号98。\\n資料10: 作業3号57の参照先は作業4号75。\\n資料11: 作業別5号34の登録値は77。\\n資料12: 作業0号34の参照先は作業1号22。\\n資料13: 作業6号98の登録値は58。\\n対象作業0号34から参照先を順にたどり、終点の値を11だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号34から作業1号22から作業2号36から作業3号57から作業4号75から作業5号88から作業6号98です。終点の値は58です。\\n答え: 69。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2020\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号29の参照先は作業2号94。\\n資料2: 作業0号11の参照先は作業1号29。\\n資料3: 作業別1号20の登録値は67。\\n資料4: 作業2号94の参照先は作業3号92。\\n資料5: 作業3号92の参照先は作業4号27。\\n資料6: 作業4号27の参照先は作業5号57。\\n資料7: 作業別4号71の登録値は15。\\n資料8: 作業別0号46の登録値は65。\\n資料9: 作業5号57の登録値は60。\\n資料10: 作業別3号31の登録値は14。\\n資料11: 作業別2号90の登録値は48。\\n対象作業0号11から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号11から作業1号29から作業2号94から作業3号92から作業4号27から作業5号57です。終点の値は60です。\\n答え: 70。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2021\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号46の登録値は40。\\n資料2: 作業5号99の参照先は作業6号98。\\n資料3: 作業別1号90の登録値は72。\\n資料4: 作業1号88の参照先は作業2号98。\\n資料5: 作業別4号56の登録値は87。\\n資料6: 作業6号98の登録値は28。\\n資料7: 作業別0号20の登録値は38。\\n資料8: 作業別5号83の登録値は20。\\n資料9: 作業3号27の参照先は作業4号15。\\n資料10: 作業別3号62の登録値は44。\\n資料11: 作業2号98の参照先は作業3号27。\\n資料12: 作業0号95の参照先は作業1号88。\\n資料13: 作業4号15の参照先は作業5号99。\\n対象作業0号95から参照先を順にたどり、終点の値を4だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号95から作業1号88から作業2号98から作業3号27から作業4号15から作業5号99から作業6号98です。終点の値は28です。\\n答え: 32。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2022\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号49の参照先は作業1号23。\\n資料2: 作業別0号42の登録値は39。\\n資料3: 作業5号23の登録値は55。\\n資料4: 作業別4号36の登録値は18。\\n資料5: 作業3号69の参照先は作業4号16。\\n資料6: 作業別3号17の登録値は28。\\n資料7: 作業2号24の参照先は作業3号69。\\n資料8: 作業4号16の参照先は作業5号23。\\n資料9: 作業別1号46の登録値は43。\\n資料10: 作業別2号79の登録値は23。\\n資料11: 作業1号23の参照先は作業2号24。\\n対象作業0号49から参照先を順にたどり、終点の値を8だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号49から作業1号23から作業2号24から作業3号69から作業4号16から作業5号23です。終点の値は55です。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2023\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号44の登録値は20。\\n資料2: 作業別3号75の登録値は93。\\n資料3: 作業別0号51の登録値は55。\\n資料4: 作業別5号88の登録値は12。\\n資料5: 作業4号78の参照先は作業5号61。\\n資料6: 作業別4号93の登録値は73。\\n資料7: 作業6号10の登録値は88。\\n資料8: 作業3号19の参照先は作業4号78。\\n資料9: 作業2号66の参照先は作業3号19。\\n資料10: 作業別2号96の登録値は52。\\n資料11: 作業1号36の参照先は作業2号66。\\n資料12: 作業0号61の参照先は作業1号36。\\n資料13: 作業5号61の参照先は作業6号10。\\n対象作業0号61から参照先を順にたどり、終点の値を8だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号61から作業1号36から作業2号66から作業3号19から作業4号78から作業5号61から作業6号10です。終点の値は88です。\\n答え: 96。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2024\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号55の参照先は作業3号76。\\n資料2: 作業1号84の参照先は作業2号55。\\n資料3: 作業別2号55の登録値は55。\\n資料4: 作業別0号32の登録値は50。\\n資料5: 作業別1号76の登録値は31。\\n資料6: 作業3号76の登録値は80。\\n資料7: 作業0号84の参照先は作業1号84。\\n対象作業0号84から参照先を順にたどり、終点の値を9だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号84から作業1号84から作業2号55から作業3号76です。終点の値は80です。\\n答え: 89。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2025\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号69の参照先は作業2号34。\\n資料2: 作業2号34の参照先は作業3号37。\\n資料3: 作業別0号53の登録値は90。\\n資料4: 作業4号43の登録値は85。\\n資料5: 作業別1号57の登録値は50。\\n資料6: 作業別2号43の登録値は15。\\n資料7: 作業0号24の参照先は作業1号69。\\n資料8: 作業3号37の参照先は作業4号43。\\n資料9: 作業別3号48の登録値は58。\\n対象作業0号24から参照先を順にたどり、終点の値を10だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号24から作業1号69から作業2号34から作業3号37から作業4号43です。終点の値は85です。\\n答え: 95。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2026\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号43の登録値は52。\\n資料2: 作業2号65の参照先は作業3号56。\\n資料3: 作業別4号64の登録値は47。\\n資料4: 作業0号95の参照先は作業1号48。\\n資料5: 作業5号87の登録値は55。\\n資料6: 作業1号48の参照先は作業2号65。\\n資料7: 作業4号70の参照先は作業5号87。\\n資料8: 作業別2号98の登録値は98。\\n資料9: 作業別0号34の登録値は14。\\n資料10: 作業別3号46の登録値は59。\\n資料11: 作業3号56の参照先は作業4号70。\\n対象作業0号95から参照先を順にたどり、終点の値を12だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号95から作業1号48から作業2号65から作業3号56から作業4号70から作業5号87です。終点の値は55です。\\n答え: 67。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2027\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号23の参照先は作業3号74。\\n資料2: 作業4号49の参照先は作業5号22。\\n資料3: 作業別3号48の登録値は56。\\n資料4: 作業別4号94の登録値は35。\\n資料5: 作業別5号32の登録値は37。\\n資料6: 作業3号74の参照先は作業4号49。\\n資料7: 作業0号65の参照先は作業1号69。\\n資料8: 作業1号69の参照先は作業2号23。\\n資料9: 作業5号22の参照先は作業6号24。\\n資料10: 作業別2号21の登録値は68。\\n資料11: 作業別1号73の登録値は37。\\n資料12: 作業別0号44の登録値は19。\\n資料13: 作業6号24の登録値は78。\\n対象作業0号65から参照先を順にたどり、終点の値を12だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号65から作業1号69から作業2号23から作業3号74から作業4号49から作業5号22から作業6号24です。終点の値は78です。\\n答え: 90。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2028\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号43の登録値は75。\\n資料2: 作業5号40の登録値は66。\\n資料3: 作業4号77の参照先は作業5号40。\\n資料4: 作業0号15の参照先は作業1号35。\\n資料5: 作業3号26の参照先は作業4号77。\\n資料6: 作業別3号89の登録値は31。\\n資料7: 作業別0号44の登録値は95。\\n資料8: 作業別1号98の登録値は71。\\n資料9: 作業1号35の参照先は作業2号28。\\n資料10: 作業別2号10の登録値は88。\\n資料11: 作業2号28の参照先は作業3号26。\\n対象作業0号15から参照先を順にたどり、終点の値を5だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号15から作業1号35から作業2号28から作業3号26から作業4号77から作業5号40です。終点の値は66です。\\n答え: 71。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2029\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号57の登録値は82。\\n資料2: 作業3号65の参照先は作業4号89。\\n資料3: 作業別5号39の登録値は34。\\n資料4: 作業5号60の参照先は作業6号53。\\n資料5: 作業0号15の参照先は作業1号98。\\n資料6: 作業別0号33の登録値は31。\\n資料7: 作業別1号69の登録値は98。\\n資料8: 作業別2号96の登録値は93。\\n資料9: 作業6号53の登録値は27。\\n資料10: 作業4号89の参照先は作業5号60。\\n資料11: 作業2号47の参照先は作業3号65。\\n資料12: 作業1号98の参照先は作業2号47。\\n資料13: 作業別3号88の登録値は83。\\n対象作業0号15から参照先を順にたどり、終点の値を7だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号15から作業1号98から作業2号47から作業3号65から作業4号89から作業5号60から作業6号53です。終点の値は27です。\\n答え: 34。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2030\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号90の登録値は16。\\n資料2: 作業4号25の参照先は作業5号39。\\n資料3: 作業別0号20の登録値は63。\\n資料4: 作業1号95の参照先は作業2号28。\\n資料5: 作業2号28の参照先は作業3号56。\\n資料6: 作業5号39の登録値は49。\\n資料7: 作業別4号62の登録値は71。\\n資料8: 作業別2号70の登録値は35。\\n資料9: 作業0号97の参照先は作業1号95。\\n資料10: 作業別1号73の登録値は30。\\n資料11: 作業3号56の参照先は作業4号25。\\n対象作業0号97から参照先を順にたどり、終点の値を9だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号97から作業1号95から作業2号28から作業3号56から作業4号25から作業5号39です。終点の値は49です。\\n答え: 58。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_33_2031\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号31の登録値は58。\\n資料2: 作業別2号98の登録値は87。\\n資料3: 作業4号80の参照先は作業5号29。\\n資料4: 作業3号47の参照先は作業4号80。\\n資料5: 作業2号69の参照先は作業3号47。\\n資料6: 作業6号16の登録値は31。\\n資料7: 作業別4号22の登録値は12。\\n資料8: 作業別5号37の登録値は53。\\n資料9: 作業0号89の参照先は作業1号44。\\n資料10: 作業別1号98の登録値は63。\\n資料11: 作業1号44の参照先は作業2号69。\\n資料12: 作業別0号38の登録値は66。\\n資料13: 作業5号29の参照先は作業6号16。\\n対象作業0号89から参照先を順にたどり、終点の値を5だけ増やして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号89から作業1号44から作業2号69から作業3号47から作業4号80から作業5号29から作業6号16です。終点の値は31です。\\n答え: 36。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2000\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号33の参照先は作業1号29。\\n資料2: 作業2号40の参照先は作業3号92。\\n資料3: 作業別0号69の登録値は85。\\n資料4: 作業1号29の参照先は作業2号40。\\n資料5: 作業別1号70の登録値は76。\\n資料6: 作業別2号86の登録値は62。\\n資料7: 作業3号92の登録値は27。\\n対象作業0号33から参照先を順にたどり、終点の値を4だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号33から作業1号29から作業2号40から作業3号92です。終点の値は27です。\\n答え: 23。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2001\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号17の登録値は39。\\n資料2: 作業別2号16の登録値は86。\\n資料3: 作業0号10の参照先は作業1号19。\\n資料4: 作業1号19の参照先は作業2号41。\\n資料5: 作業4号75の登録値は89。\\n資料6: 作業3号50の参照先は作業4号75。\\n資料7: 作業2号41の参照先は作業3号50。\\n資料8: 作業別1号67の登録値は48。\\n資料9: 作業別0号15の登録値は27。\\n対象作業0号10から参照先を順にたどり、終点の値を4だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号10から作業1号19から作業2号41から作業3号50から作業4号75です。終点の値は89です。\\n答え: 85。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2002\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号89の参照先は作業4号27。\\n資料2: 作業4号27の参照先は作業5号71。\\n資料3: 作業1号17の参照先は作業2号33。\\n資料4: 作業別4号21の登録値は82。\\n資料5: 作業5号71の登録値は58。\\n資料6: 作業2号33の参照先は作業3号89。\\n資料7: 作業0号14の参照先は作業1号17。\\n資料8: 作業別0号98の登録値は43。\\n資料9: 作業別1号12の登録値は24。\\n資料10: 作業別3号40の登録値は57。\\n資料11: 作業別2号21の登録値は43。\\n対象作業0号14から参照先を順にたどり、終点の値を2だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号14から作業1号17から作業2号33から作業3号89から作業4号27から作業5号71です。終点の値は58です。\\n答え: 56。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2003\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業6号64の登録値は66。\\n資料2: 作業2号77の参照先は作業3号67。\\n資料3: 作業別5号59の登録値は14。\\n資料4: 作業別4号30の登録値は91。\\n資料5: 作業別2号28の登録値は98。\\n資料6: 作業別0号71の登録値は87。\\n資料7: 作業3号67の参照先は作業4号50。\\n資料8: 作業1号50の参照先は作業2号77。\\n資料9: 作業5号40の参照先は作業6号64。\\n資料10: 作業別1号63の登録値は83。\\n資料11: 作業0号51の参照先は作業1号50。\\n資料12: 作業4号50の参照先は作業5号40。\\n資料13: 作業別3号80の登録値は69。\\n対象作業0号51から参照先を順にたどり、終点の値を7だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号51から作業1号50から作業2号77から作業3号67から作業4号50から作業5号40から作業6号64です。終点の値は66です。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2004\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号62の登録値は72。\\n資料2: 作業5号16の登録値は48。\\n資料3: 作業3号52の参照先は作業4号98。\\n資料4: 作業1号20の参照先は作業2号19。\\n資料5: 作業別3号50の登録値は70。\\n資料6: 作業4号98の参照先は作業5号16。\\n資料7: 作業別4号63の登録値は98。\\n資料8: 作業別1号95の登録値は94。\\n資料9: 作業別2号74の登録値は48。\\n資料10: 作業2号19の参照先は作業3号52。\\n資料11: 作業0号72の参照先は作業1号20。\\n対象作業0号72から参照先を順にたどり、終点の値を11だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号72から作業1号20から作業2号19から作業3号52から作業4号98から作業5号16です。終点の値は48です。\\n答え: 37。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2005\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号89の登録値は60。\\n資料2: 作業2号67の参照先は作業3号49。\\n資料3: 作業別2号54の登録値は17。\\n資料4: 作業別0号99の登録値は36。\\n資料5: 作業3号49の参照先は作業4号10。\\n資料6: 作業1号40の参照先は作業2号67。\\n資料7: 作業5号92の参照先は作業6号74。\\n資料8: 作業0号61の参照先は作業1号40。\\n資料9: 作業4号10の参照先は作業5号92。\\n資料10: 作業別5号40の登録値は24。\\n資料11: 作業別4号47の登録値は39。\\n資料12: 作業6号74の登録値は80。\\n資料13: 作業別1号74の登録値は19。\\n対象作業0号61から参照先を順にたどり、終点の値を11だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号61から作業1号40から作業2号67から作業3号49から作業4号10から作業5号92から作業6号74です。終点の値は80です。\\n答え: 69。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2006\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号63の参照先は作業1号63。\\n資料2: 作業4号55の参照先は作業5号31。\\n資料3: 作業別2号16の登録値は36。\\n資料4: 作業2号27の参照先は作業3号78。\\n資料5: 作業3号78の参照先は作業4号55。\\n資料6: 作業別0号41の登録値は87。\\n資料7: 作業別1号61の登録値は73。\\n資料8: 作業別4号28の登録値は76。\\n資料9: 作業1号63の参照先は作業2号27。\\n資料10: 作業別3号67の登録値は40。\\n資料11: 作業5号31の登録値は46。\\n対象作業0号63から参照先を順にたどり、終点の値を11だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号63から作業1号63から作業2号27から作業3号78から作業4号55から作業5号31です。終点の値は46です。\\n答え: 35。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2007\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号81の登録値は21。\\n資料2: 作業5号67の参照先は作業6号25。\\n資料3: 作業3号61の参照先は作業4号26。\\n資料4: 作業別5号39の登録値は71。\\n資料5: 作業別1号30の登録値は82。\\n資料6: 作業別3号60の登録値は72。\\n資料7: 作業0号37の参照先は作業1号17。\\n資料8: 作業6号25の登録値は89。\\n資料9: 作業別4号65の登録値は46。\\n資料10: 作業2号46の参照先は作業3号61。\\n資料11: 作業別0号15の登録値は83。\\n資料12: 作業1号17の参照先は作業2号46。\\n資料13: 作業4号26の参照先は作業5号67。\\n対象作業0号37から参照先を順にたどり、終点の値を4だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号37から作業1号17から作業2号46から作業3号61から作業4号26から作業5号67から作業6号25です。終点の値は89です。\\n答え: 85。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2008\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号43の参照先は作業2号90。\\n資料2: 作業別0号19の登録値は64。\\n資料3: 作業0号94の参照先は作業1号43。\\n資料4: 作業別1号29の登録値は50。\\n資料5: 作業3号36の登録値は84。\\n資料6: 作業別2号27の登録値は18。\\n資料7: 作業2号90の参照先は作業3号36。\\n対象作業0号94から参照先を順にたどり、終点の値を6だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号94から作業1号43から作業2号90から作業3号36です。終点の値は84です。\\n答え: 78。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2009\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号64の参照先は作業2号31。\\n資料2: 作業4号57の登録値は83。\\n資料3: 作業別2号18の登録値は36。\\n資料4: 作業別1号12の登録値は19。\\n資料5: 作業別3号35の登録値は44。\\n資料6: 作業0号77の参照先は作業1号64。\\n資料7: 作業2号31の参照先は作業3号41。\\n資料8: 作業3号41の参照先は作業4号57。\\n資料9: 作業別0号74の登録値は35。\\n対象作業0号77から参照先を順にたどり、終点の値を5だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号77から作業1号64から作業2号31から作業3号41から作業4号57です。終点の値は83です。\\n答え: 78。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2010\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号26の参照先は作業2号38。\\n資料2: 作業5号67の登録値は29。\\n資料3: 作業別1号78の登録値は75。\\n資料4: 作業0号36の参照先は作業1号26。\\n資料5: 作業3号96の参照先は作業4号55。\\n資料6: 作業4号55の参照先は作業5号67。\\n資料7: 作業別4号78の登録値は64。\\n資料8: 作業2号38の参照先は作業3号96。\\n資料9: 作業別0号50の登録値は62。\\n資料10: 作業別3号98の登録値は13。\\n資料11: 作業別2号26の登録値は47。\\n対象作業0号36から参照先を順にたどり、終点の値を4だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号36から作業1号26から作業2号38から作業3号96から作業4号55から作業5号67です。終点の値は29です。\\n答え: 25。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2011\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号65の登録値は35。\\n資料2: 作業別1号35の登録値は45。\\n資料3: 作業2号46の参照先は作業3号37。\\n資料4: 作業0号90の参照先は作業1号98。\\n資料5: 作業別2号50の登録値は74。\\n資料6: 作業4号21の参照先は作業5号96。\\n資料7: 作業6号17の登録値は82。\\n資料8: 作業別4号18の登録値は52。\\n資料9: 作業1号98の参照先は作業2号46。\\n資料10: 作業別5号13の登録値は52。\\n資料11: 作業別0号38の登録値は29。\\n資料12: 作業5号96の参照先は作業6号17。\\n資料13: 作業3号37の参照先は作業4号21。\\n対象作業0号90から参照先を順にたどり、終点の値を9だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号90から作業1号98から作業2号46から作業3号37から作業4号21から作業5号96から作業6号17です。終点の値は82です。\\n答え: 73。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2012\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号97の参照先は作業5号11。\\n資料2: 作業別0号62の登録値は14。\\n資料3: 作業別3号72の登録値は39。\\n資料4: 作業別4号68の登録値は20。\\n資料5: 作業別1号72の登録値は96。\\n資料6: 作業3号47の参照先は作業4号97。\\n資料7: 作業5号11の登録値は22。\\n資料8: 作業1号79の参照先は作業2号12。\\n資料9: 作業0号17の参照先は作業1号79。\\n資料10: 作業2号12の参照先は作業3号47。\\n資料11: 作業別2号51の登録値は89。\\n対象作業0号17から参照先を順にたどり、終点の値を8だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号17から作業1号79から作業2号12から作業3号47から作業4号97から作業5号11です。終点の値は22です。\\n答え: 14。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2013\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業5号20の参照先は作業6号64。\\n資料2: 作業別3号27の登録値は21。\\n資料3: 作業2号51の参照先は作業3号89。\\n資料4: 作業別2号55の登録値は21。\\n資料5: 作業4号18の参照先は作業5号20。\\n資料6: 作業別0号88の登録値は37。\\n資料7: 作業6号64の登録値は29。\\n資料8: 作業0号63の参照先は作業1号96。\\n資料9: 作業別5号39の登録値は62。\\n資料10: 作業別1号23の登録値は48。\\n資料11: 作業3号89の参照先は作業4号18。\\n資料12: 作業1号96の参照先は作業2号51。\\n資料13: 作業別4号57の登録値は15。\\n対象作業0号63から参照先を順にたどり、終点の値を5だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号63から作業1号96から作業2号51から作業3号89から作業4号18から作業5号20から作業6号64です。終点の値は29です。\\n答え: 24。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2014\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号54の登録値は85。\\n資料2: 作業0号74の参照先は作業1号73。\\n資料3: 作業別0号95の登録値は14。\\n資料4: 作業別1号56の登録値は15。\\n資料5: 作業別4号58の登録値は70。\\n資料6: 作業4号12の参照先は作業5号75。\\n資料7: 作業5号75の登録値は42。\\n資料8: 作業1号73の参照先は作業2号79。\\n資料9: 作業3号96の参照先は作業4号12。\\n資料10: 作業別3号92の登録値は65。\\n資料11: 作業2号79の参照先は作業3号96。\\n対象作業0号74から参照先を順にたどり、終点の値を11だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号74から作業1号73から作業2号79から作業3号96から作業4号12から作業5号75です。終点の値は42です。\\n答え: 31。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2015\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業6号65の登録値は81。\\n資料2: 作業別0号13の登録値は85。\\n資料3: 作業1号49の参照先は作業2号59。\\n資料4: 作業2号59の参照先は作業3号53。\\n資料5: 作業別1号99の登録値は77。\\n資料6: 作業別4号45の登録値は50。\\n資料7: 作業4号44の参照先は作業5号65。\\n資料8: 作業5号65の参照先は作業6号65。\\n資料9: 作業別3号91の登録値は49。\\n資料10: 作業別2号57の登録値は63。\\n資料11: 作業別5号42の登録値は98。\\n資料12: 作業3号53の参照先は作業4号44。\\n資料13: 作業0号61の参照先は作業1号49。\\n対象作業0号61から参照先を順にたどり、終点の値を9だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号61から作業1号49から作業2号59から作業3号53から作業4号44から作業5号65から作業6号65です。終点の値は81です。\\n答え: 72。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2016\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号64の登録値は78。\\n資料2: 作業別2号41の登録値は16。\\n資料3: 作業2号68の参照先は作業3号17。\\n資料4: 作業0号21の参照先は作業1号59。\\n資料5: 作業別1号11の登録値は56。\\n資料6: 作業1号59の参照先は作業2号68。\\n資料7: 作業3号17の登録値は72。\\n対象作業0号21から参照先を順にたどり、終点の値を3だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号21から作業1号59から作業2号68から作業3号17です。終点の値は72です。\\n答え: 69。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2017\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号21の登録値は13。\\n資料2: 作業3号16の参照先は作業4号88。\\n資料3: 作業別3号92の登録値は57。\\n資料4: 作業別2号36の登録値は48。\\n資料5: 作業2号41の参照先は作業3号16。\\n資料6: 作業0号93の参照先は作業1号59。\\n資料7: 作業4号88の登録値は68。\\n資料8: 作業別0号70の登録値は62。\\n資料9: 作業1号59の参照先は作業2号41。\\n対象作業0号93から参照先を順にたどり、終点の値を4だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号93から作業1号59から作業2号41から作業3号16から作業4号88です。終点の値は68です。\\n答え: 64。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2018\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号40の参照先は作業3号76。\\n資料2: 作業1号36の参照先は作業2号40。\\n資料3: 作業別2号24の登録値は23。\\n資料4: 作業0号10の参照先は作業1号36。\\n資料5: 作業別3号41の登録値は40。\\n資料6: 作業別0号34の登録値は58。\\n資料7: 作業別1号30の登録値は72。\\n資料8: 作業別4号96の登録値は17。\\n資料9: 作業4号17の参照先は作業5号69。\\n資料10: 作業3号76の参照先は作業4号17。\\n資料11: 作業5号69の登録値は71。\\n対象作業0号10から参照先を順にたどり、終点の値を10だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号10から作業1号36から作業2号40から作業3号76から作業4号17から作業5号69です。終点の値は71です。\\n答え: 61。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2019\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号45の登録値は57。\\n資料2: 作業別1号64の登録値は71。\\n資料3: 作業別5号52の登録値は40。\\n資料4: 作業0号54の参照先は作業1号11。\\n資料5: 作業別4号57の登録値は61。\\n資料6: 作業5号71の参照先は作業6号43。\\n資料7: 作業6号43の登録値は56。\\n資料8: 作業別3号95の登録値は28。\\n資料9: 作業別2号59の登録値は49。\\n資料10: 作業3号63の参照先は作業4号13。\\n資料11: 作業4号13の参照先は作業5号71。\\n資料12: 作業2号56の参照先は作業3号63。\\n資料13: 作業1号11の参照先は作業2号56。\\n対象作業0号54から参照先を順にたどり、終点の値を7だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号54から作業1号11から作業2号56から作業3号63から作業4号13から作業5号71から作業6号43です。終点の値は56です。\\n答え: 49。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2020\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号25の登録値は19。\\n資料2: 作業別1号24の登録値は74。\\n資料3: 作業1号49の参照先は作業2号48。\\n資料4: 作業4号54の参照先は作業5号79。\\n資料5: 作業別2号15の登録値は46。\\n資料6: 作業2号48の参照先は作業3号93。\\n資料7: 作業3号93の参照先は作業4号54。\\n資料8: 作業5号79の登録値は41。\\n資料9: 作業別3号63の登録値は17。\\n資料10: 作業別4号24の登録値は35。\\n資料11: 作業0号72の参照先は作業1号49。\\n対象作業0号72から参照先を順にたどり、終点の値を6だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号72から作業1号49から作業2号48から作業3号93から作業4号54から作業5号79です。終点の値は41です。\\n答え: 35。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2021\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号50の参照先は作業2号86。\\n資料2: 作業2号86の参照先は作業3号79。\\n資料3: 作業別4号82の登録値は18。\\n資料4: 作業4号96の参照先は作業5号81。\\n資料5: 作業3号79の参照先は作業4号96。\\n資料6: 作業別3号81の登録値は42。\\n資料7: 作業別2号47の登録値は92。\\n資料8: 作業別0号57の登録値は63。\\n資料9: 作業別1号62の登録値は79。\\n資料10: 作業0号98の参照先は作業1号50。\\n資料11: 作業6号26の登録値は39。\\n資料12: 作業別5号70の登録値は38。\\n資料13: 作業5号81の参照先は作業6号26。\\n対象作業0号98から参照先を順にたどり、終点の値を2だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号98から作業1号50から作業2号86から作業3号79から作業4号96から作業5号81から作業6号26です。終点の値は39です。\\n答え: 37。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2022\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号78の参照先は作業5号74。\\n資料2: 作業5号74の登録値は20。\\n資料3: 作業別4号84の登録値は75。\\n資料4: 作業3号68の参照先は作業4号78。\\n資料5: 作業別0号11の登録値は38。\\n資料6: 作業1号76の参照先は作業2号70。\\n資料7: 作業2号70の参照先は作業3号68。\\n資料8: 作業別2号53の登録値は41。\\n資料9: 作業別1号38の登録値は44。\\n資料10: 作業別3号39の登録値は41。\\n資料11: 作業0号24の参照先は作業1号76。\\n対象作業0号24から参照先を順にたどり、終点の値を8だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号24から作業1号76から作業2号70から作業3号68から作業4号78から作業5号74です。終点の値は20です。\\n答え: 12。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2023\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号24の参照先は作業4号23。\\n資料2: 作業別5号82の登録値は32。\\n資料3: 作業別1号59の登録値は96。\\n資料4: 作業5号12の参照先は作業6号72。\\n資料5: 作業2号66の参照先は作業3号24。\\n資料6: 作業別0号25の登録値は46。\\n資料7: 作業別2号76の登録値は16。\\n資料8: 作業別3号87の登録値は78。\\n資料9: 作業1号52の参照先は作業2号66。\\n資料10: 作業別4号15の登録値は41。\\n資料11: 作業4号23の参照先は作業5号12。\\n資料12: 作業0号74の参照先は作業1号52。\\n資料13: 作業6号72の登録値は67。\\n対象作業0号74から参照先を順にたどり、終点の値を4だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号74から作業1号52から作業2号66から作業3号24から作業4号23から作業5号12から作業6号72です。終点の値は67です。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2024\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号80の登録値は59。\\n資料2: 作業別0号56の登録値は98。\\n資料3: 作業1号65の参照先は作業2号93。\\n資料4: 作業0号91の参照先は作業1号65。\\n資料5: 作業3号13の登録値は32。\\n資料6: 作業2号93の参照先は作業3号13。\\n資料7: 作業別2号44の登録値は55。\\n対象作業0号91から参照先を順にたどり、終点の値を3だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号91から作業1号65から作業2号93から作業3号13です。終点の値は32です。\\n答え: 29。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2025\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業4号13の登録値は26。\\n資料2: 作業別3号35の登録値は90。\\n資料3: 作業3号74の参照先は作業4号13。\\n資料4: 作業別2号30の登録値は63。\\n資料5: 作業別1号86の登録値は87。\\n資料6: 作業別0号51の登録値は30。\\n資料7: 作業0号49の参照先は作業1号10。\\n資料8: 作業2号34の参照先は作業3号74。\\n資料9: 作業1号10の参照先は作業2号34。\\n対象作業0号49から参照先を順にたどり、終点の値を9だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号49から作業1号10から作業2号34から作業3号74から作業4号13です。終点の値は26です。\\n答え: 17。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2026\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号89の登録値は42。\\n資料2: 作業0号39の参照先は作業1号79。\\n資料3: 作業別4号93の登録値は26。\\n資料4: 作業別2号55の登録値は67。\\n資料5: 作業1号79の参照先は作業2号63。\\n資料6: 作業5号21の登録値は32。\\n資料7: 作業4号96の参照先は作業5号21。\\n資料8: 作業2号63の参照先は作業3号92。\\n資料9: 作業別0号89の登録値は54。\\n資料10: 作業3号92の参照先は作業4号96。\\n資料11: 作業別1号15の登録値は80。\\n対象作業0号39から参照先を順にたどり、終点の値を6だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号39から作業1号79から作業2号63から作業3号92から作業4号96から作業5号21です。終点の値は32です。\\n答え: 26。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2027\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号62の参照先は作業2号98。\\n資料2: 作業別3号84の登録値は70。\\n資料3: 作業別0号85の登録値は56。\\n資料4: 作業4号88の参照先は作業5号97。\\n資料5: 作業2号98の参照先は作業3号22。\\n資料6: 作業別1号14の登録値は16。\\n資料7: 作業5号97の参照先は作業6号46。\\n資料8: 作業3号22の参照先は作業4号88。\\n資料9: 作業0号61の参照先は作業1号62。\\n資料10: 作業別4号80の登録値は21。\\n資料11: 作業別5号29の登録値は35。\\n資料12: 作業別2号46の登録値は44。\\n資料13: 作業6号46の登録値は55。\\n対象作業0号61から参照先を順にたどり、終点の値を11だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号61から作業1号62から作業2号98から作業3号22から作業4号88から作業5号97から作業6号46です。終点の値は55です。\\n答え: 44。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2028\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号60の登録値は33。\\n資料2: 作業2号71の参照先は作業3号92。\\n資料3: 作業1号32の参照先は作業2号71。\\n資料4: 作業3号92の参照先は作業4号18。\\n資料5: 作業5号80の登録値は30。\\n資料6: 作業0号45の参照先は作業1号32。\\n資料7: 作業4号18の参照先は作業5号80。\\n資料8: 作業別3号68の登録値は30。\\n資料9: 作業別1号75の登録値は94。\\n資料10: 作業別0号15の登録値は53。\\n資料11: 作業別2号81の登録値は38。\\n対象作業0号45から参照先を順にたどり、終点の値を2だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号45から作業1号32から作業2号71から作業3号92から作業4号18から作業5号80です。終点の値は30です。\\n答え: 28。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2029\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号86の登録値は92。\\n資料2: 作業6号28の登録値は75。\\n資料3: 作業3号49の参照先は作業4号41。\\n資料4: 作業5号64の参照先は作業6号28。\\n資料5: 作業1号41の参照先は作業2号24。\\n資料6: 作業0号25の参照先は作業1号41。\\n資料7: 作業別2号95の登録値は47。\\n資料8: 作業別1号66の登録値は24。\\n資料9: 作業別0号31の登録値は33。\\n資料10: 作業別5号49の登録値は72。\\n資料11: 作業2号24の参照先は作業3号49。\\n資料12: 作業別3号68の登録値は78。\\n資料13: 作業4号41の参照先は作業5号64。\\n対象作業0号25から参照先を順にたどり、終点の値を10だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号25から作業1号41から作業2号24から作業3号49から作業4号41から作業5号64から作業6号28です。終点の値は75です。\\n答え: 65。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2030\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号48の登録値は35。\\n資料2: 作業別0号69の登録値は56。\\n資料3: 作業別3号65の登録値は44。\\n資料4: 作業2号90の参照先は作業3号94。\\n資料5: 作業5号13の登録値は38。\\n資料6: 作業別2号79の登録値は94。\\n資料7: 作業3号94の参照先は作業4号66。\\n資料8: 作業別1号81の登録値は27。\\n資料9: 作業4号66の参照先は作業5号13。\\n資料10: 作業1号99の参照先は作業2号90。\\n資料11: 作業0号35の参照先は作業1号99。\\n対象作業0号35から参照先を順にたどり、終点の値を7だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号35から作業1号99から作業2号90から作業3号94から作業4号66から作業5号13です。終点の値は38です。\\n答え: 31。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_34_2031\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別5号36の登録値は36。\\n資料2: 作業別2号42の登録値は60。\\n資料3: 作業別3号76の登録値は77。\\n資料4: 作業別1号29の登録値は19。\\n資料5: 作業0号90の参照先は作業1号90。\\n資料6: 作業3号44の参照先は作業4号26。\\n資料7: 作業1号90の参照先は作業2号25。\\n資料8: 作業別0号42の登録値は49。\\n資料9: 作業4号26の参照先は作業5号81。\\n資料10: 作業2号25の参照先は作業3号44。\\n資料11: 作業6号56の登録値は79。\\n資料12: 作業5号81の参照先は作業6号56。\\n資料13: 作業別4号57の登録値は79。\\n対象作業0号90から参照先を順にたどり、終点の値を6だけ減らして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号90から作業1号90から作業2号25から作業3号44から作業4号26から作業5号81から作業6号56です。終点の値は79です。\\n答え: 73。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2000\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号78の登録値は64。\\n資料2: 作業3号46の登録値は88。\\n資料3: 作業2号98の参照先は作業3号46。\\n資料4: 作業0号99の参照先は作業1号79。\\n資料5: 作業別1号52の登録値は28。\\n資料6: 作業別0号34の登録値は77。\\n資料7: 作業1号79の参照先は作業2号98。\\n対象作業0号99から参照先を順にたどり、終点の値を6で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号99から作業1号79から作業2号98から作業3号46です。終点の値は88です。\\n答え: 4。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2001\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号69の参照先は作業2号50。\\n資料2: 作業0号12の参照先は作業1号69。\\n資料3: 作業別2号44の登録値は92。\\n資料4: 作業別0号42の登録値は14。\\n資料5: 作業3号76の参照先は作業4号82。\\n資料6: 作業4号82の登録値は20。\\n資料7: 作業別3号15の登録値は91。\\n資料8: 作業別1号15の登録値は31。\\n資料9: 作業2号50の参照先は作業3号76。\\n対象作業0号12から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号12から作業1号69から作業2号50から作業3号76から作業4号82です。終点の値は20です。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2002\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号32の参照先は作業2号21。\\n資料2: 作業別4号98の登録値は44。\\n資料3: 作業5号28の登録値は88。\\n資料4: 作業別3号59の登録値は51。\\n資料5: 作業3号71の参照先は作業4号47。\\n資料6: 作業別2号24の登録値は22。\\n資料7: 作業別0号36の登録値は88。\\n資料8: 作業2号21の参照先は作業3号71。\\n資料9: 作業0号60の参照先は作業1号32。\\n資料10: 作業4号47の参照先は作業5号28。\\n資料11: 作業別1号79の登録値は57。\\n対象作業0号60から参照先を順にたどり、終点の値を11で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号60から作業1号32から作業2号21から作業3号71から作業4号47から作業5号28です。終点の値は88です。\\n答え: 0。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2003\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業6号94の登録値は64。\\n資料2: 作業別3号24の登録値は60。\\n資料3: 作業別0号20の登録値は86。\\n資料4: 作業別4号87の登録値は36。\\n資料5: 作業3号33の参照先は作業4号97。\\n資料6: 作業別5号72の登録値は62。\\n資料7: 作業2号96の参照先は作業3号33。\\n資料8: 作業4号97の参照先は作業5号24。\\n資料9: 作業5号24の参照先は作業6号94。\\n資料10: 作業0号23の参照先は作業1号92。\\n資料11: 作業1号92の参照先は作業2号96。\\n資料12: 作業別2号64の登録値は46。\\n資料13: 作業別1号29の登録値は77。\\n対象作業0号23から参照先を順にたどり、終点の値を5で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号23から作業1号92から作業2号96から作業3号33から作業4号97から作業5号24から作業6号94です。終点の値は64です。\\n答え: 4。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2004\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号16の登録値は52。\\n資料2: 作業3号90の参照先は作業4号27。\\n資料3: 作業別1号11の登録値は36。\\n資料4: 作業別0号42の登録値は19。\\n資料5: 作業4号27の参照先は作業5号33。\\n資料6: 作業2号82の参照先は作業3号90。\\n資料7: 作業5号33の登録値は35。\\n資料8: 作業1号69の参照先は作業2号82。\\n資料9: 作業0号75の参照先は作業1号69。\\n資料10: 作業別3号79の登録値は38。\\n資料11: 作業別4号16の登録値は65。\\n対象作業0号75から参照先を順にたどり、終点の値を5で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号75から作業1号69から作業2号82から作業3号90から作業4号27から作業5号33です。終点の値は35です。\\n答え: 0。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2005\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号90の登録値は47。\\n資料2: 作業別1号80の登録値は88。\\n資料3: 作業4号26の参照先は作業5号29。\\n資料4: 作業別2号93の登録値は51。\\n資料5: 作業別4号90の登録値は70。\\n資料6: 作業3号47の参照先は作業4号26。\\n資料7: 作業1号75の参照先は作業2号33。\\n資料8: 作業2号33の参照先は作業3号47。\\n資料9: 作業別3号41の登録値は87。\\n資料10: 作業別5号52の登録値は16。\\n資料11: 作業6号79の登録値は88。\\n資料12: 作業5号29の参照先は作業6号79。\\n資料13: 作業0号84の参照先は作業1号75。\\n対象作業0号84から参照先を順にたどり、終点の値を8で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号84から作業1号75から作業2号33から作業3号47から作業4号26から作業5号29から作業6号79です。終点の値は88です。\\n答え: 0。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2006\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号86の参照先は作業2号54。\\n資料2: 作業2号54の参照先は作業3号84。\\n資料3: 作業別1号63の登録値は70。\\n資料4: 作業0号31の参照先は作業1号86。\\n資料5: 作業別3号98の登録値は10。\\n資料6: 作業4号81の参照先は作業5号59。\\n資料7: 作業5号59の登録値は39。\\n資料8: 作業3号84の参照先は作業4号81。\\n資料9: 作業別4号50の登録値は52。\\n資料10: 作業別2号95の登録値は94。\\n資料11: 作業別0号81の登録値は66。\\n対象作業0号31から参照先を順にたどり、終点の値を9で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号31から作業1号86から作業2号54から作業3号84から作業4号81から作業5号59です。終点の値は39です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2007\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号78の登録値は68。\\n資料2: 作業6号81の登録値は45。\\n資料3: 作業別0号63の登録値は44。\\n資料4: 作業1号65の参照先は作業2号20。\\n資料5: 作業2号20の参照先は作業3号40。\\n資料6: 作業別3号11の登録値は66。\\n資料7: 作業4号87の参照先は作業5号10。\\n資料8: 作業別1号30の登録値は35。\\n資料9: 作業5号10の参照先は作業6号81。\\n資料10: 作業別2号35の登録値は17。\\n資料11: 作業3号40の参照先は作業4号87。\\n資料12: 作業別5号19の登録値は35。\\n資料13: 作業0号68の参照先は作業1号65。\\n対象作業0号68から参照先を順にたどり、終点の値を6で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号68から作業1号65から作業2号20から作業3号40から作業4号87から作業5号10から作業6号81です。終点の値は45です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2008\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号88の参照先は作業1号61。\\n資料2: 作業2号74の参照先は作業3号56。\\n資料3: 作業別1号91の登録値は75。\\n資料4: 作業1号61の参照先は作業2号74。\\n資料5: 作業別0号11の登録値は74。\\n資料6: 作業3号56の登録値は67。\\n資料7: 作業別2号10の登録値は62。\\n対象作業0号88から参照先を順にたどり、終点の値を9で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号88から作業1号61から作業2号74から作業3号56です。終点の値は67です。\\n答え: 4。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2009\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号53の参照先は作業1号88。\\n資料2: 作業別0号85の登録値は81。\\n資料3: 作業別3号12の登録値は47。\\n資料4: 作業1号88の参照先は作業2号53。\\n資料5: 作業3号80の参照先は作業4号39。\\n資料6: 作業別1号41の登録値は20。\\n資料7: 作業別2号28の登録値は23。\\n資料8: 作業2号53の参照先は作業3号80。\\n資料9: 作業4号39の登録値は21。\\n対象作業0号53から参照先を順にたどり、終点の値を11で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号53から作業1号88から作業2号53から作業3号80から作業4号39です。終点の値は21です。\\n答え: 10。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2010\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別0号52の登録値は68。\\n資料2: 作業別3号23の登録値は42。\\n資料3: 作業3号22の参照先は作業4号55。\\n資料4: 作業0号86の参照先は作業1号77。\\n資料5: 作業別1号30の登録値は68。\\n資料6: 作業別4号63の登録値は42。\\n資料7: 作業別2号61の登録値は22。\\n資料8: 作業1号77の参照先は作業2号25。\\n資料9: 作業5号40の登録値は38。\\n資料10: 作業2号25の参照先は作業3号22。\\n資料11: 作業4号55の参照先は作業5号40。\\n対象作業0号86から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号86から作業1号77から作業2号25から作業3号22から作業4号55から作業5号40です。終点の値は38です。\\n答え: 2。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2011\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業5号54の参照先は作業6号36。\\n資料2: 作業2号39の参照先は作業3号17。\\n資料3: 作業6号36の登録値は69。\\n資料4: 作業0号45の参照先は作業1号25。\\n資料5: 作業3号17の参照先は作業4号26。\\n資料6: 作業別5号29の登録値は64。\\n資料7: 作業別3号77の登録値は38。\\n資料8: 作業1号25の参照先は作業2号39。\\n資料9: 作業別0号13の登録値は80。\\n資料10: 作業4号26の参照先は作業5号54。\\n資料11: 作業別1号58の登録値は54。\\n資料12: 作業別2号96の登録値は55。\\n資料13: 作業別4号39の登録値は15。\\n対象作業0号45から参照先を順にたどり、終点の値を7で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号45から作業1号25から作業2号39から作業3号17から作業4号26から作業5号54から作業6号36です。終点の値は69です。\\n答え: 6。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2012\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号76の登録値は37。\\n資料2: 作業1号79の参照先は作業2号77。\\n資料3: 作業別4号51の登録値は64。\\n資料4: 作業3号25の参照先は作業4号79。\\n資料5: 作業5号74の登録値は51。\\n資料6: 作業別3号77の登録値は13。\\n資料7: 作業0号58の参照先は作業1号79。\\n資料8: 作業4号79の参照先は作業5号74。\\n資料9: 作業2号77の参照先は作業3号25。\\n資料10: 作業別0号76の登録値は34。\\n資料11: 作業別1号10の登録値は48。\\n対象作業0号58から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号58から作業1号79から作業2号77から作業3号25から作業4号79から作業5号74です。終点の値は51です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2013\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号37の参照先は作業1号35。\\n資料2: 作業4号30の参照先は作業5号95。\\n資料3: 作業別1号20の登録値は89。\\n資料4: 作業5号95の参照先は作業6号81。\\n資料5: 作業1号35の参照先は作業2号14。\\n資料6: 作業別4号34の登録値は39。\\n資料7: 作業別2号61の登録値は26。\\n資料8: 作業別0号78の登録値は64。\\n資料9: 作業3号77の参照先は作業4号30。\\n資料10: 作業6号81の登録値は45。\\n資料11: 作業別5号81の登録値は27。\\n資料12: 作業別3号79の登録値は40。\\n資料13: 作業2号14の参照先は作業3号77。\\n対象作業0号37から参照先を順にたどり、終点の値を2で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号37から作業1号35から作業2号14から作業3号77から作業4号30から作業5号95から作業6号81です。終点の値は45です。\\n答え: 1。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2014\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業3号91の参照先は作業4号98。\\n資料2: 作業2号71の参照先は作業3号91。\\n資料3: 作業1号58の参照先は作業2号71。\\n資料4: 作業別1号74の登録値は22。\\n資料5: 作業別3号35の登録値は94。\\n資料6: 作業0号61の参照先は作業1号58。\\n資料7: 作業別0号46の登録値は92。\\n資料8: 作業別2号45の登録値は85。\\n資料9: 作業別4号66の登録値は80。\\n資料10: 作業4号98の参照先は作業5号28。\\n資料11: 作業5号28の登録値は39。\\n対象作業0号61から参照先を順にたどり、終点の値を6で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号61から作業1号58から作業2号71から作業3号91から作業4号98から作業5号28です。終点の値は39です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2015\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号71の参照先は作業2号58。\\n資料2: 作業別3号85の登録値は22。\\n資料3: 作業5号21の参照先は作業6号22。\\n資料4: 作業2号58の参照先は作業3号83。\\n資料5: 作業4号87の参照先は作業5号21。\\n資料6: 作業別4号60の登録値は86。\\n資料7: 作業0号98の参照先は作業1号71。\\n資料8: 作業別2号60の登録値は56。\\n資料9: 作業別0号30の登録値は19。\\n資料10: 作業別1号83の登録値は25。\\n資料11: 作業3号83の参照先は作業4号87。\\n資料12: 作業別5号88の登録値は23。\\n資料13: 作業6号22の登録値は57。\\n対象作業0号98から参照先を順にたどり、終点の値を10で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号98から作業1号71から作業2号58から作業3号83から作業4号87から作業5号21から作業6号22です。終点の値は57です。\\n答え: 7。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2016\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別1号46の登録値は85。\\n資料2: 作業2号17の参照先は作業3号19。\\n資料3: 作業1号24の参照先は作業2号17。\\n資料4: 作業3号19の登録値は43。\\n資料5: 作業0号53の参照先は作業1号24。\\n資料6: 作業別2号44の登録値は31。\\n資料7: 作業別0号86の登録値は17。\\n対象作業0号53から参照先を順にたどり、終点の値を10で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号53から作業1号24から作業2号17から作業3号19です。終点の値は43です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2017\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号86の登録値は22。\\n資料2: 作業別0号67の登録値は24。\\n資料3: 作業3号82の参照先は作業4号17。\\n資料4: 作業1号29の参照先は作業2号12。\\n資料5: 作業別2号42の登録値は10。\\n資料6: 作業2号12の参照先は作業3号82。\\n資料7: 作業0号82の参照先は作業1号29。\\n資料8: 作業4号17の登録値は70。\\n資料9: 作業別1号51の登録値は55。\\n対象作業0号82から参照先を順にたどり、終点の値を4で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号82から作業1号29から作業2号12から作業3号82から作業4号17です。終点の値は70です。\\n答え: 2。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2018\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業5号90の登録値は33。\\n資料2: 作業3号55の参照先は作業4号23。\\n資料3: 作業2号48の参照先は作業3号55。\\n資料4: 作業別0号43の登録値は85。\\n資料5: 作業4号23の参照先は作業5号90。\\n資料6: 作業別1号99の登録値は46。\\n資料7: 作業別3号73の登録値は87。\\n資料8: 作業別2号28の登録値は59。\\n資料9: 作業1号86の参照先は作業2号48。\\n資料10: 作業0号42の参照先は作業1号86。\\n資料11: 作業別4号32の登録値は31。\\n対象作業0号42から参照先を順にたどり、終点の値を5で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号42から作業1号86から作業2号48から作業3号55から作業4号23から作業5号90です。終点の値は33です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2019\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号35の参照先は作業3号75。\\n資料2: 作業1号81の参照先は作業2号35。\\n資料3: 作業別4号12の登録値は75。\\n資料4: 作業別0号59の登録値は12。\\n資料5: 作業別1号67の登録値は11。\\n資料6: 作業6号34の登録値は26。\\n資料7: 作業別2号92の登録値は50。\\n資料8: 作業0号15の参照先は作業1号81。\\n資料9: 作業3号75の参照先は作業4号67。\\n資料10: 作業別3号18の登録値は95。\\n資料11: 作業別5号96の登録値は10。\\n資料12: 作業5号34の参照先は作業6号34。\\n資料13: 作業4号67の参照先は作業5号34。\\n対象作業0号15から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号15から作業1号81から作業2号35から作業3号75から作業4号67から作業5号34から作業6号34です。終点の値は26です。\\n答え: 2。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2020\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号99の登録値は99。\\n資料2: 作業0号19の参照先は作業1号86。\\n資料3: 作業4号20の参照先は作業5号51。\\n資料4: 作業1号86の参照先は作業2号93。\\n資料5: 作業別4号92の登録値は40。\\n資料6: 作業別0号96の登録値は72。\\n資料7: 作業別1号46の登録値は38。\\n資料8: 作業3号56の参照先は作業4号20。\\n資料9: 作業5号51の登録値は87。\\n資料10: 作業別3号25の登録値は33。\\n資料11: 作業2号93の参照先は作業3号56。\\n対象作業0号19から参照先を順にたどり、終点の値を4で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号19から作業1号86から作業2号93から作業3号56から作業4号20から作業5号51です。終点の値は87です。\\n答え: 3。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2021\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別2号79の登録値は97。\\n資料2: 作業別4号22の登録値は46。\\n資料3: 作業別0号54の登録値は42。\\n資料4: 作業0号51の参照先は作業1号65。\\n資料5: 作業別3号48の登録値は87。\\n資料6: 作業1号65の参照先は作業2号31。\\n資料7: 作業3号59の参照先は作業4号61。\\n資料8: 作業6号55の登録値は44。\\n資料9: 作業2号31の参照先は作業3号59。\\n資料10: 作業別1号91の登録値は97。\\n資料11: 作業5号46の参照先は作業6号55。\\n資料12: 作業別5号29の登録値は41。\\n資料13: 作業4号61の参照先は作業5号46。\\n対象作業0号51から参照先を順にたどり、終点の値を6で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号51から作業1号65から作業2号31から作業3号59から作業4号61から作業5号46から作業6号55です。終点の値は44です。\\n答え: 2。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2022\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別3号30の登録値は80。\\n資料2: 作業別4号45の登録値は68。\\n資料3: 作業2号35の参照先は作業3号32。\\n資料4: 作業別0号51の登録値は89。\\n資料5: 作業別2号27の登録値は78。\\n資料6: 作業3号32の参照先は作業4号32。\\n資料7: 作業別1号83の登録値は57。\\n資料8: 作業4号32の参照先は作業5号82。\\n資料9: 作業0号40の参照先は作業1号77。\\n資料10: 作業5号82の登録値は41。\\n資料11: 作業1号77の参照先は作業2号35。\\n対象作業0号40から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号40から作業1号77から作業2号35から作業3号32から作業4号32から作業5号82です。終点の値は41です。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2023\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号73の参照先は作業1号51。\\n資料2: 作業5号51の参照先は作業6号56。\\n資料3: 作業1号51の参照先は作業2号94。\\n資料4: 作業別5号43の登録値は79。\\n資料5: 作業別0号18の登録値は71。\\n資料6: 作業別3号47の登録値は63。\\n資料7: 作業別1号30の登録値は35。\\n資料8: 作業3号61の参照先は作業4号12。\\n資料9: 作業2号94の参照先は作業3号61。\\n資料10: 作業6号56の登録値は71。\\n資料11: 作業別2号27の登録値は57。\\n資料12: 作業別4号60の登録値は77。\\n資料13: 作業4号12の参照先は作業5号51。\\n対象作業0号73から参照先を順にたどり、終点の値を10で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号73から作業1号51から作業2号94から作業3号61から作業4号12から作業5号51から作業6号56です。終点の値は71です。\\n答え: 1。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2024\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号52の参照先は作業2号55。\\n資料2: 作業2号55の参照先は作業3号49。\\n資料3: 作業3号49の登録値は50。\\n資料4: 作業0号12の参照先は作業1号52。\\n資料5: 作業別1号21の登録値は44。\\n資料6: 作業別0号27の登録値は35。\\n資料7: 作業別2号80の登録値は56。\\n対象作業0号12から参照先を順にたどり、終点の値を8で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号12から作業1号52から作業2号55から作業3号49です。終点の値は50です。\\n答え: 2。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2025\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号25の参照先は作業2号56。\\n資料2: 作業2号56の参照先は作業3号89。\\n資料3: 作業4号54の登録値は85。\\n資料4: 作業別3号17の登録値は85。\\n資料5: 作業3号89の参照先は作業4号54。\\n資料6: 作業0号45の参照先は作業1号25。\\n資料7: 作業別1号99の登録値は96。\\n資料8: 作業別2号11の登録値は47。\\n資料9: 作業別0号87の登録値は22。\\n対象作業0号45から参照先を順にたどり、終点の値を7で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号45から作業1号25から作業2号56から作業3号89から作業4号54です。終点の値は85です。\\n答え: 1。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2026\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号63の参照先は作業2号86。\\n資料2: 作業5号86の登録値は29。\\n資料3: 作業別4号22の登録値は38。\\n資料4: 作業3号28の参照先は作業4号21。\\n資料5: 作業2号86の参照先は作業3号28。\\n資料6: 作業別3号32の登録値は69。\\n資料7: 作業4号21の参照先は作業5号86。\\n資料8: 作業別1号80の登録値は94。\\n資料9: 作業別2号11の登録値は46。\\n資料10: 作業0号44の参照先は作業1号63。\\n資料11: 作業別0号68の登録値は65。\\n対象作業0号44から参照先を順にたどり、終点の値を4で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号44から作業1号63から作業2号86から作業3号28から作業4号21から作業5号86です。終点の値は29です。\\n答え: 1。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2027\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号91の参照先は作業1号79。\\n資料2: 作業5号43の参照先は作業6号13。\\n資料3: 作業1号79の参照先は作業2号85。\\n資料4: 作業別3号75の登録値は55。\\n資料5: 作業別4号15の登録値は61。\\n資料6: 作業2号85の参照先は作業3号37。\\n資料7: 作業4号82の参照先は作業5号43。\\n資料8: 作業6号13の登録値は32。\\n資料9: 作業別0号22の登録値は34。\\n資料10: 作業別5号93の登録値は85。\\n資料11: 作業3号37の参照先は作業4号82。\\n資料12: 作業別2号44の登録値は14。\\n資料13: 作業別1号42の登録値は92。\\n対象作業0号91から参照先を順にたどり、終点の値を4で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号91から作業1号79から作業2号85から作業3号37から作業4号82から作業5号43から作業6号13です。終点の値は32です。\\n答え: 0。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2028\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業0号48の参照先は作業1号74。\\n資料2: 作業別1号14の登録値は12。\\n資料3: 作業別3号47の登録値は17。\\n資料4: 作業1号74の参照先は作業2号98。\\n資料5: 作業2号98の参照先は作業3号95。\\n資料6: 作業3号95の参照先は作業4号43。\\n資料7: 作業別2号60の登録値は90。\\n資料8: 作業別0号11の登録値は60。\\n資料9: 作業5号70の登録値は65。\\n資料10: 作業4号43の参照先は作業5号70。\\n資料11: 作業別4号47の登録値は91。\\n対象作業0号48から参照先を順にたどり、終点の値を11で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号48から作業1号74から作業2号98から作業3号95から作業4号43から作業5号70です。終点の値は65です。\\n答え: 10。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2029\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業2号16の参照先は作業3号42。\\n資料2: 作業6号67の登録値は72。\\n資料3: 作業5号35の参照先は作業6号67。\\n資料4: 作業別4号58の登録値は13。\\n資料5: 作業1号43の参照先は作業2号16。\\n資料6: 作業3号42の参照先は作業4号74。\\n資料7: 作業別5号69の登録値は30。\\n資料8: 作業0号37の参照先は作業1号43。\\n資料9: 作業4号74の参照先は作業5号35。\\n資料10: 作業別1号67の登録値は44。\\n資料11: 作業別3号79の登録値は34。\\n資料12: 作業別0号15の登録値は83。\\n資料13: 作業別2号64の登録値は59。\\n対象作業0号37から参照先を順にたどり、終点の値を9で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号37から作業1号43から作業2号16から作業3号42から作業4号74から作業5号35から作業6号67です。終点の値は72です。\\n答え: 0。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2030\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業1号37の参照先は作業2号52。\\n資料2: 作業4号60の参照先は作業5号37。\\n資料3: 作業5号37の登録値は57。\\n資料4: 作業別1号36の登録値は86。\\n資料5: 作業別3号43の登録値は55。\\n資料6: 作業別0号53の登録値は44。\\n資料7: 作業別4号83の登録値は39。\\n資料8: 作業0号52の参照先は作業1号37。\\n資料9: 作業2号52の参照先は作業3号67。\\n資料10: 作業別2号24の登録値は84。\\n資料11: 作業3号67の参照先は作業4号60。\\n対象作業0号52から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号52から作業1号37から作業2号52から作業3号67から作業4号60から作業5号37です。終点の値は57です。\\n答え: 9。\"}\n"
  "{\"id\": \"v48_evidence_unseen_eval_35_2031\", \"prompt\": \"問題: 次は架空の資料です。\\n資料1: 作業別4号36の登録値は90。\\n資料2: 作業別1号56の登録値は29。\\n資料3: 作業別5号26の登録値は69。\\n資料4: 作業1号81の参照先は作業2号72。\\n資料5: 作業4号70の参照先は作業5号42。\\n資料6: 作業5号42の参照先は作業6号31。\\n資料7: 作業2号72の参照先は作業3号99。\\n資料8: 作業別0号31の登録値は67。\\n資料9: 作業0号36の参照先は作業1号81。\\n資料10: 作業別3号52の登録値は48。\\n資料11: 作業3号99の参照先は作業4号70。\\n資料12: 作業6号31の登録値は42。\\n資料13: 作業別2号65の登録値は12。\\n対象作業0号36から参照先を順にたどり、終点の値を12で割った余りとして答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n参照順は作業0号36から作業1号81から作業2号72から作業3号99から作業4号70から作業5号42から作業6号31です。終点の値は42です。\\n答え: 6。\"}\n"
 ),
 "japanese_unseen_eval.jsonl": (
  "{\"id\": \"v48_japanese_unseen_eval_32_2000\", \"prompt\": \"問題: 選択肢: A=37, B=81, C=22, D=27。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは81だけです。\\n答え: B。\", \"options\": [37, 81, 22, 27]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2001\", \"prompt\": \"問題: 選択肢: A=29, B=16, C=10, D=14。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは10だけです。\\n答え: C。\", \"options\": [29, 16, 10, 14]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2002\", \"prompt\": \"問題: 選択肢: A=13, B=50, C=59, D=23。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは50だけです。\\n答え: B。\", \"options\": [13, 50, 59, 23]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2003\", \"prompt\": \"問題: 選択肢: A=39, B=80, C=16, D=45。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは80だけです。\\n答え: B。\", \"options\": [39, 80, 16, 45]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2004\", \"prompt\": \"問題: 選択肢: A=46, B=36, C=69, D=86。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは86だけです。\\n答え: D。\", \"options\": [46, 36, 69, 86]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2005\", \"prompt\": \"問題: 選択肢: A=29, B=15, C=46, D=10。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは10だけです。\\n答え: D。\", \"options\": [29, 15, 46, 10]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2006\", \"prompt\": \"問題: 選択肢: A=53, B=35, C=14, D=58。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは53だけです。\\n答え: A。\", \"options\": [53, 35, 14, 58]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2007\", \"prompt\": \"問題: 選択肢: A=55, B=38, C=19, D=84。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは84だけです。\\n答え: D。\", \"options\": [55, 38, 19, 84]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2008\", \"prompt\": \"問題: 選択肢: A=78, B=77, C=39, D=52。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは52だけです。\\n答え: D。\", \"options\": [78, 77, 39, 52]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2009\", \"prompt\": \"問題: 選択肢: A=36, B=12, C=78, D=96。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは96だけです。\\n答え: D。\", \"options\": [36, 12, 78, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2010\", \"prompt\": \"問題: 選択肢: A=33, B=23, C=72, D=68。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは72だけです。\\n答え: C。\", \"options\": [33, 23, 72, 68]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2011\", \"prompt\": \"問題: 選択肢: A=96, B=36, C=47, D=35。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは96だけです。\\n答え: A。\", \"options\": [96, 36, 47, 35]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2012\", \"prompt\": \"問題: 選択肢: A=22, B=19, C=15, D=94。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは94だけです。\\n答え: D。\", \"options\": [22, 19, 15, 94]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2013\", \"prompt\": \"問題: 選択肢: A=81, B=19, C=22, D=77。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは81だけです。\\n答え: A。\", \"options\": [81, 19, 22, 77]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2014\", \"prompt\": \"問題: 選択肢: A=69, B=65, C=99, D=36。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは65だけです。\\n答え: B。\", \"options\": [69, 65, 99, 36]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2015\", \"prompt\": \"問題: 選択肢: A=59, B=37, C=61, D=16。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは61だけです。\\n答え: C。\", \"options\": [59, 37, 61, 16]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2016\", \"prompt\": \"問題: 選択肢: A=58, B=41, C=44, D=33。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは41だけです。\\n答え: B。\", \"options\": [58, 41, 44, 33]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2017\", \"prompt\": \"問題: 選択肢: A=87, B=79, C=77, D=47。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは87だけです。\\n答え: A。\", \"options\": [87, 79, 77, 47]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2018\", \"prompt\": \"問題: 選択肢: A=39, B=17, C=67, D=97。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは97だけです。\\n答え: D。\", \"options\": [39, 17, 67, 97]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2019\", \"prompt\": \"問題: 選択肢: A=44, B=34, C=95, D=68。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは95だけです。\\n答え: C。\", \"options\": [44, 34, 95, 68]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2020\", \"prompt\": \"問題: 選択肢: A=46, B=16, C=11, D=74。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは74だけです。\\n答え: D。\", \"options\": [46, 16, 11, 74]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2021\", \"prompt\": \"問題: 選択肢: A=52, B=59, C=35, D=15。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは52だけです。\\n答え: A。\", \"options\": [52, 59, 35, 15]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2022\", \"prompt\": \"問題: 選択肢: A=16, B=30, C=38, D=36。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは30だけです。\\n答え: B。\", \"options\": [16, 30, 38, 36]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2023\", \"prompt\": \"問題: 選択肢: A=94, B=19, C=78, D=39。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは94だけです。\\n答え: A。\", \"options\": [94, 19, 78, 39]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2024\", \"prompt\": \"問題: 選択肢: A=26, B=67, C=35, D=21。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは21だけです。\\n答え: D。\", \"options\": [26, 67, 35, 21]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2025\", \"prompt\": \"問題: 選択肢: A=48, B=24, C=86, D=12。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは86だけです。\\n答え: C。\", \"options\": [48, 24, 86, 12]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2026\", \"prompt\": \"問題: 選択肢: A=29, B=44, C=59, D=96。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは96だけです。\\n答え: D。\", \"options\": [29, 44, 59, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2027\", \"prompt\": \"問題: 選択肢: A=57, B=25, C=28, D=80。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは80だけです。\\n答え: D。\", \"options\": [57, 25, 28, 80]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2028\", \"prompt\": \"問題: 選択肢: A=78, B=30, C=46, D=77。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは30だけです。\\n答え: B。\", \"options\": [78, 30, 46, 77]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2029\", \"prompt\": \"問題: 選択肢: A=17, B=53, C=77, D=69。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは53だけです。\\n答え: B。\", \"options\": [17, 53, 77, 69]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2030\", \"prompt\": \"問題: 選択肢: A=68, B=36, C=43, D=77。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは43だけです。\\n答え: C。\", \"options\": [68, 36, 43, 77]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_32_2031\", \"prompt\": \"問題: 選択肢: A=53, B=49, C=37, D=48。\\n条件を丁寧に読み取ってください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは53だけです。\\n答え: A。\", \"options\": [53, 49, 37, 48]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2000\", \"prompt\": \"問題: 選択肢: A=75, B=91, C=70, D=28。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは28だけです。\\n答え: D。\", \"options\": [75, 91, 70, 28]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2001\", \"prompt\": \"問題: 選択肢: A=40, B=74, C=49, D=54。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは49だけです。\\n答え: C。\", \"options\": [40, 74, 49, 54]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2002\", \"prompt\": \"問題: 選択肢: A=62, B=37, C=10, D=60。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは37だけです。\\n答え: B。\", \"options\": [62, 37, 10, 60]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2003\", \"prompt\": \"問題: 選択肢: A=40, B=66, C=61, D=65。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは66だけです。\\n答え: B。\", \"options\": [40, 66, 61, 65]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2004\", \"prompt\": \"問題: 選択肢: A=91, B=63, C=47, D=75。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは47だけです。\\n答え: C。\", \"options\": [91, 63, 47, 75]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2005\", \"prompt\": \"問題: 選択肢: A=82, B=87, C=59, D=20。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは59だけです。\\n答え: C。\", \"options\": [82, 87, 59, 20]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2006\", \"prompt\": \"問題: 選択肢: A=51, B=37, C=91, D=75。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは37だけです。\\n答え: B。\", \"options\": [51, 37, 91, 75]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2007\", \"prompt\": \"問題: 選択肢: A=14, B=64, C=74, D=43。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは14だけです。\\n答え: A。\", \"options\": [14, 64, 74, 43]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2008\", \"prompt\": \"問題: 選択肢: A=88, B=52, C=62, D=42。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは88だけです。\\n答え: A。\", \"options\": [88, 52, 62, 42]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2009\", \"prompt\": \"問題: 選択肢: A=62, B=54, C=91, D=19。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは19だけです。\\n答え: D。\", \"options\": [62, 54, 91, 19]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2010\", \"prompt\": \"問題: 選択肢: A=86, B=76, C=21, D=69。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは69だけです。\\n答え: D。\", \"options\": [86, 76, 21, 69]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2011\", \"prompt\": \"問題: 選択肢: A=71, B=25, C=65, D=40。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは25だけです。\\n答え: B。\", \"options\": [71, 25, 65, 40]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2012\", \"prompt\": \"問題: 選択肢: A=61, B=91, C=20, D=37。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは37だけです。\\n答え: D。\", \"options\": [61, 91, 20, 37]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2013\", \"prompt\": \"問題: 選択肢: A=52, B=48, C=93, D=91。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは48だけです。\\n答え: B。\", \"options\": [52, 48, 93, 91]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2014\", \"prompt\": \"問題: 選択肢: A=73, B=36, C=70, D=97。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは36だけです。\\n答え: B。\", \"options\": [73, 36, 70, 97]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2015\", \"prompt\": \"問題: 選択肢: A=59, B=40, C=75, D=85。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは59だけです。\\n答え: A。\", \"options\": [59, 40, 75, 85]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2016\", \"prompt\": \"問題: 選択肢: A=91, B=40, C=45, D=86。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは45だけです。\\n答え: C。\", \"options\": [91, 40, 45, 86]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2017\", \"prompt\": \"問題: 選択肢: A=98, B=97, C=61, D=68。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは68だけです。\\n答え: D。\", \"options\": [98, 97, 61, 68]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2018\", \"prompt\": \"問題: 選択肢: A=63, B=94, C=70, D=35。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは35だけです。\\n答え: D。\", \"options\": [63, 94, 70, 35]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2019\", \"prompt\": \"問題: 選択肢: A=62, B=64, C=58, D=96。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは58だけです。\\n答え: C。\", \"options\": [62, 64, 58, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2020\", \"prompt\": \"問題: 選択肢: A=82, B=67, C=10, D=86。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは67だけです。\\n答え: B。\", \"options\": [82, 67, 10, 86]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2021\", \"prompt\": \"問題: 選択肢: A=85, B=57, C=72, D=71。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは57だけです。\\n答え: B。\", \"options\": [85, 57, 72, 71]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2022\", \"prompt\": \"問題: 選択肢: A=53, B=70, C=58, D=10。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは58だけです。\\n答え: C。\", \"options\": [53, 70, 58, 10]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2023\", \"prompt\": \"問題: 選択肢: A=64, B=53, C=73, D=44。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは44だけです。\\n答え: D。\", \"options\": [64, 53, 73, 44]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2024\", \"prompt\": \"問題: 選択肢: A=62, B=16, C=65, D=21。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは16だけです。\\n答え: B。\", \"options\": [62, 16, 65, 21]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2025\", \"prompt\": \"問題: 選択肢: A=30, B=96, C=29, D=73。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは29だけです。\\n答え: C。\", \"options\": [30, 96, 29, 73]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2026\", \"prompt\": \"問題: 選択肢: A=23, B=96, C=20, D=87。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは23だけです。\\n答え: A。\", \"options\": [23, 96, 20, 87]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2027\", \"prompt\": \"問題: 選択肢: A=79, B=50, C=91, D=64。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは79だけです。\\n答え: A。\", \"options\": [79, 50, 91, 64]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2028\", \"prompt\": \"問題: 選択肢: A=64, B=42, C=94, D=18。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは18だけです。\\n答え: D。\", \"options\": [64, 42, 94, 18]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2029\", \"prompt\": \"問題: 選択肢: A=26, B=43, C=86, D=10。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは26だけです。\\n答え: A。\", \"options\": [26, 43, 86, 10]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2030\", \"prompt\": \"問題: 選択肢: A=59, B=90, C=95, D=73。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは59だけです。\\n答え: A。\", \"options\": [59, 90, 95, 73]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_33_2031\", \"prompt\": \"問題: 選択肢: A=68, B=80, C=54, D=95。\\n否定の表現を見落とさないでください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは68だけです。\\n答え: A。\", \"options\": [68, 80, 54, 95]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2000\", \"prompt\": \"問題: 選択肢: A=17, B=65, C=29, D=45。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは65だけです。\\n答え: B。\", \"options\": [17, 65, 29, 45]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2001\", \"prompt\": \"問題: 選択肢: A=25, B=34, C=19, D=63。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは63だけです。\\n答え: D。\", \"options\": [25, 34, 19, 63]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2002\", \"prompt\": \"問題: 選択肢: A=39, B=47, C=23, D=96。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは96だけです。\\n答え: D。\", \"options\": [39, 47, 23, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2003\", \"prompt\": \"問題: 選択肢: A=44, B=85, C=35, D=29。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは85だけです。\\n答え: B。\", \"options\": [44, 85, 35, 29]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2004\", \"prompt\": \"問題: 選択肢: A=25, B=92, C=39, D=33。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは92だけです。\\n答え: B。\", \"options\": [25, 92, 39, 33]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2005\", \"prompt\": \"問題: 選択肢: A=45, B=14, C=93, D=16。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは93だけです。\\n答え: C。\", \"options\": [45, 14, 93, 16]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2006\", \"prompt\": \"問題: 選択肢: A=45, B=38, C=16, D=10。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは10だけです。\\n答え: D。\", \"options\": [45, 38, 16, 10]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2007\", \"prompt\": \"問題: 選択肢: A=54, B=35, C=18, D=79。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは54だけです。\\n答え: A。\", \"options\": [54, 35, 18, 79]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2008\", \"prompt\": \"問題: 選択肢: A=99, B=89, C=69, D=53。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは53だけです。\\n答え: D。\", \"options\": [99, 89, 69, 53]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2009\", \"prompt\": \"問題: 選択肢: A=31, B=57, C=69, D=29。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは31だけです。\\n答え: A。\", \"options\": [31, 57, 69, 29]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2010\", \"prompt\": \"問題: 選択肢: A=88, B=83, C=12, D=24。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは83だけです。\\n答え: B。\", \"options\": [88, 83, 12, 24]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2011\", \"prompt\": \"問題: 選択肢: A=48, B=98, C=34, D=47。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは98だけです。\\n答え: B。\", \"options\": [48, 98, 34, 47]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2012\", \"prompt\": \"問題: 選択肢: A=50, B=48, C=33, D=58。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは50だけです。\\n答え: A。\", \"options\": [50, 48, 33, 58]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2013\", \"prompt\": \"問題: 選択肢: A=76, B=16, C=13, D=44。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは76だけです。\\n答え: A。\", \"options\": [76, 16, 13, 44]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2014\", \"prompt\": \"問題: 選択肢: A=14, B=75, C=25, D=34。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは75だけです。\\n答え: B。\", \"options\": [14, 75, 25, 34]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2015\", \"prompt\": \"問題: 選択肢: A=53, B=18, C=57, D=69。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは53だけです。\\n答え: A。\", \"options\": [53, 18, 57, 69]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2016\", \"prompt\": \"問題: 選択肢: A=28, B=29, C=15, D=95。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは95だけです。\\n答え: D。\", \"options\": [28, 29, 15, 95]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2017\", \"prompt\": \"問題: 選択肢: A=99, B=20, C=56, D=78。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは20だけです。\\n答え: B。\", \"options\": [99, 20, 56, 78]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2018\", \"prompt\": \"問題: 選択肢: A=36, B=66, C=78, D=42。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは42だけです。\\n答え: D。\", \"options\": [36, 66, 78, 42]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2019\", \"prompt\": \"問題: 選択肢: A=56, B=29, C=83, D=28。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは83だけです。\\n答え: C。\", \"options\": [56, 29, 83, 28]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2020\", \"prompt\": \"問題: 選択肢: A=22, B=69, C=79, D=96。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは96だけです。\\n答え: D。\", \"options\": [22, 69, 79, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2021\", \"prompt\": \"問題: 選択肢: A=48, B=57, C=30, D=26。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは30だけです。\\n答え: C。\", \"options\": [48, 57, 30, 26]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2022\", \"prompt\": \"問題: 選択肢: A=73, B=13, C=35, D=46。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは73だけです。\\n答え: A。\", \"options\": [73, 13, 35, 46]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2023\", \"prompt\": \"問題: 選択肢: A=17, B=52, C=35, D=38。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは52だけです。\\n答え: B。\", \"options\": [17, 52, 35, 38]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2024\", \"prompt\": \"問題: 選択肢: A=23, B=90, C=11, D=28。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは90だけです。\\n答え: B。\", \"options\": [23, 90, 11, 28]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2025\", \"prompt\": \"問題: 選択肢: A=60, B=47, C=45, D=48。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは60だけです。\\n答え: A。\", \"options\": [60, 47, 45, 48]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2026\", \"prompt\": \"問題: 選択肢: A=44, B=22, C=81, D=89。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは81だけです。\\n答え: C。\", \"options\": [44, 22, 81, 89]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2027\", \"prompt\": \"問題: 選択肢: A=99, B=77, C=17, D=81。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは81だけです。\\n答え: D。\", \"options\": [99, 77, 17, 81]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2028\", \"prompt\": \"問題: 選択肢: A=97, B=66, C=59, D=36。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは97だけです。\\n答え: A。\", \"options\": [97, 66, 59, 36]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2029\", \"prompt\": \"問題: 選択肢: A=46, B=78, C=48, D=63。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは63だけです。\\n答え: D。\", \"options\": [46, 78, 48, 63]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2030\", \"prompt\": \"問題: 選択肢: A=58, B=85, C=12, D=44。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは85だけです。\\n答え: B。\", \"options\": [58, 85, 12, 44]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_34_2031\", \"prompt\": \"問題: 選択肢: A=25, B=98, C=56, D=55。\\n条件を同時に満たすか確認してください。 「十の位が一の位より大きい」に該当するものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは98だけです。\\n答え: B。\", \"options\": [25, 98, 56, 55]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2000\", \"prompt\": \"問題: 選択肢: A=72, B=82, C=15, D=71。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは15だけです。\\n答え: C。\", \"options\": [72, 82, 15, 71]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2001\", \"prompt\": \"問題: 選択肢: A=32, B=95, C=22, D=96。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは22だけです。\\n答え: C。\", \"options\": [32, 95, 22, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2002\", \"prompt\": \"問題: 選択肢: A=21, B=31, C=15, D=91。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは15だけです。\\n答え: C。\", \"options\": [21, 31, 15, 91]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2003\", \"prompt\": \"問題: 選択肢: A=30, B=91, C=63, D=58。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは58だけです。\\n答え: D。\", \"options\": [30, 91, 63, 58]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2004\", \"prompt\": \"問題: 選択肢: A=14, B=87, C=84, D=20。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは14だけです。\\n答え: A。\", \"options\": [14, 87, 84, 20]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2005\", \"prompt\": \"問題: 選択肢: A=37, B=65, C=70, D=60。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは37だけです。\\n答え: A。\", \"options\": [37, 65, 70, 60]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2006\", \"prompt\": \"問題: 選択肢: A=73, B=83, C=20, D=45。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは45だけです。\\n答え: D。\", \"options\": [73, 83, 20, 45]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2007\", \"prompt\": \"問題: 選択肢: A=42, B=29, C=85, D=82。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは29だけです。\\n答え: B。\", \"options\": [42, 29, 85, 82]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2008\", \"prompt\": \"問題: 選択肢: A=71, B=80, C=83, D=38。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは38だけです。\\n答え: D。\", \"options\": [71, 80, 83, 38]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2009\", \"prompt\": \"問題: 選択肢: A=80, B=10, C=49, D=82。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは49だけです。\\n答え: C。\", \"options\": [80, 10, 49, 82]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2010\", \"prompt\": \"問題: 選択肢: A=84, B=70, C=56, D=92。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは56だけです。\\n答え: C。\", \"options\": [84, 70, 56, 92]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2011\", \"prompt\": \"問題: 選択肢: A=34, B=53, C=76, D=80。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは34だけです。\\n答え: A。\", \"options\": [34, 53, 76, 80]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2012\", \"prompt\": \"問題: 選択肢: A=63, B=27, C=80, D=40。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは27だけです。\\n答え: B。\", \"options\": [63, 27, 80, 40]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2013\", \"prompt\": \"問題: 選択肢: A=97, B=65, C=34, D=83。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは34だけです。\\n答え: C。\", \"options\": [97, 65, 34, 83]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2014\", \"prompt\": \"問題: 選択肢: A=74, B=32, C=22, D=51。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは22だけです。\\n答え: C。\", \"options\": [74, 32, 22, 51]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2015\", \"prompt\": \"問題: 選択肢: A=65, B=52, C=19, D=21。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは19だけです。\\n答え: C。\", \"options\": [65, 52, 19, 21]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2016\", \"prompt\": \"問題: 選択肢: A=74, B=13, C=75, D=63。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは13だけです。\\n答え: B。\", \"options\": [74, 13, 75, 63]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2017\", \"prompt\": \"問題: 選択肢: A=35, B=86, C=51, D=62。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは35だけです。\\n答え: A。\", \"options\": [35, 86, 51, 62]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2018\", \"prompt\": \"問題: 選択肢: A=52, B=73, C=45, D=60。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは45だけです。\\n答え: C。\", \"options\": [52, 73, 45, 60]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2019\", \"prompt\": \"問題: 選択肢: A=41, B=87, C=23, D=72。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは23だけです。\\n答え: C。\", \"options\": [41, 87, 23, 72]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2020\", \"prompt\": \"問題: 選択肢: A=87, B=50, C=61, D=12。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは12だけです。\\n答え: D。\", \"options\": [87, 50, 61, 12]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2021\", \"prompt\": \"問題: 選択肢: A=45, B=30, C=92, D=90。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは45だけです。\\n答え: A。\", \"options\": [45, 30, 92, 90]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2022\", \"prompt\": \"問題: 選択肢: A=51, B=96, C=61, D=49。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは49だけです。\\n答え: D。\", \"options\": [51, 96, 61, 49]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2023\", \"prompt\": \"問題: 選択肢: A=64, B=90, C=96, D=16。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは16だけです。\\n答え: D。\", \"options\": [64, 90, 96, 16]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2024\", \"prompt\": \"問題: 選択肢: A=95, B=91, C=35, D=72。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは35だけです。\\n答え: C。\", \"options\": [95, 91, 35, 72]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2025\", \"prompt\": \"問題: 選択肢: A=96, B=81, C=49, D=51。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは49だけです。\\n答え: C。\", \"options\": [96, 81, 49, 51]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2026\", \"prompt\": \"問題: 選択肢: A=41, B=77, C=40, D=96。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは77だけです。\\n答え: B。\", \"options\": [41, 77, 40, 96]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2027\", \"prompt\": \"問題: 選択肢: A=16, B=75, C=98, D=83。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは16だけです。\\n答え: A。\", \"options\": [16, 75, 98, 83]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2028\", \"prompt\": \"問題: 選択肢: A=33, B=41, C=53, D=32。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは33だけです。\\n答え: A。\", \"options\": [33, 41, 53, 32]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2029\", \"prompt\": \"問題: 選択肢: A=83, B=67, C=60, D=54。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは67だけです。\\n答え: B。\", \"options\": [83, 67, 60, 54]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2030\", \"prompt\": \"問題: 選択肢: A=20, B=76, C=33, D=21。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは33だけです。\\n答え: C。\", \"options\": [20, 76, 33, 21]}\n"
  "{\"id\": \"v48_japanese_unseen_eval_35_2031\", \"prompt\": \"問題: 選択肢: A=23, B=71, C=64, D=74。\\n結論を急がず選択肢を比較してください。 「十の位が一の位より大きい」に該当しないものを一つ選び、最後に記号で答えてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n各値を判定すると指定条件に合うのは23だけです。\\n答え: A。\", \"options\": [23, 71, 64, 74]}\n"
 ),
 "logic_unseen_eval.jsonl": (
  "{\"id\": \"v48_logic_unseen_eval_32_2000\", \"prompt\": \"問題: 数列[27, 28, 63, 27, 4, 43, 51, 46, 58]で、25以上かつ3の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 175。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2001\", \"prompt\": \"問題: 数列[1, 30, 53, 5, 14, 27, 11, 17, 6, 2, 22, 34]で、30以上かつ6の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 87。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2002\", \"prompt\": \"問題: 数列[51, 23, 43, 36, 50, 40, 38, 15, 34, 10, 20, 22, 37, 2, 18]で、17以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 307。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2003\", \"prompt\": \"問題: 数列[49, 58, 46, 12, 15, 40, 33, 59, 3, 34, 17, 18, 10, 48, 1, 27, 48, 32]で、25以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 425。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2004\", \"prompt\": \"問題: 数列[21, 51, 40, 5, 44, 31, 45, 45, 43, 15, 56, 27, 31, 34, 13]で、22以上かつ6の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 447。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2005\", \"prompt\": \"問題: 数列[59, 65, 13, 4, 45, 55, 28, 13, 24, 13, 43, 38, 64, 58, 31, 1, 44, 1]で、21以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 485。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2006\", \"prompt\": \"問題: 数列[15, 33, 1, 36, 35, 18, 47, 49, 65, 65, 37, 58, 54, 48, 1]で、24以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 362。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2007\", \"prompt\": \"問題: 数列[52, 26, 46, 54, 27, 17, 34, 42, 10, 39, 25, 15, 10, 31, 3, 36, 55, 64]で、19以上かつ4の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 379。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2008\", \"prompt\": \"問題: 数列[30, 9, 16, 52, 4, 21, 4, 23, 62]で、18以上かつ3の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 137。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2009\", \"prompt\": \"問題: 数列[41, 46, 48, 34, 42, 55, 48, 50, 59, 13, 25, 33]で、30以上かつ4の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 360。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2010\", \"prompt\": \"問題: 数列[18, 42, 8, 32, 3, 45, 50, 56, 9, 6, 6, 21, 29, 53, 42]で、20以上かつ4の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 282。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2011\", \"prompt\": \"問題: 数列[10, 2, 8, 14, 35, 20, 59, 43, 44, 46, 23, 18, 55, 44, 35, 15, 6, 30]で、22以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 384。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2012\", \"prompt\": \"問題: 数列[28, 28, 15, 21, 44, 26, 16, 65, 11, 10, 41, 42, 19, 63, 36]で、12以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 267。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2013\", \"prompt\": \"問題: 数列[63, 39, 10, 47, 38, 46, 42, 39, 9, 32, 4, 50, 19, 16, 11, 17, 12, 12]で、23以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 346。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2014\", \"prompt\": \"問題: 数列[57, 15, 35, 29, 16, 12, 62, 17, 62, 62, 20, 34, 52, 35, 22]で、25以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 358。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2015\", \"prompt\": \"問題: 数列[15, 65, 18, 34, 22, 9, 20, 27, 5, 20, 22, 56, 24, 16, 48, 49, 13, 18]で、15以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 349。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2016\", \"prompt\": \"問題: 数列[58, 55, 49, 60, 26, 43, 40, 64, 29]で、10以上かつ2の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 176。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2017\", \"prompt\": \"問題: 数列[56, 35, 2, 2, 42, 30, 60, 15, 43, 57, 47, 53]で、18以上かつ7の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 290。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2018\", \"prompt\": \"問題: 数列[33, 27, 4, 15, 5, 20, 44, 7, 54, 55, 9, 54, 54, 56, 60]で、25以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 155。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2019\", \"prompt\": \"問題: 数列[53, 20, 42, 32, 33, 24, 1, 22, 33, 59, 56, 51, 56, 6, 39, 22, 12, 48]で、9以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 320。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2020\", \"prompt\": \"問題: 数列[25, 4, 60, 35, 21, 9, 41, 1, 22, 7, 43, 63, 38, 2, 7]で、29以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 185。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2021\", \"prompt\": \"問題: 数列[45, 55, 32, 4, 3, 45, 21, 27, 2, 42, 51, 9, 39, 48, 41, 7, 41, 27]で、30以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 294。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2022\", \"prompt\": \"問題: 数列[5, 52, 52, 5, 8, 33, 40, 40, 14, 22, 26, 57, 30, 16, 60]で、19以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 232。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2023\", \"prompt\": \"問題: 数列[26, 52, 30, 4, 24, 7, 65, 64, 9, 27, 25, 42, 30, 16, 45, 43, 29, 60]で、27以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 257。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2024\", \"prompt\": \"問題: 数列[28, 59, 54, 50, 4, 27, 14, 20, 6]で、14以上かつ2の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 86。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2025\", \"prompt\": \"問題: 数列[43, 45, 17, 38, 5, 16, 59, 47, 15, 13, 43, 38]で、17以上かつ4の倍数ではない値だけを選び、合計を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 330。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2026\", \"prompt\": \"問題: 数列[20, 10, 57, 23, 47, 5, 24, 23, 25, 23, 63, 58, 46, 39, 45]で、25以上かつ6の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 380。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2027\", \"prompt\": \"問題: 数列[51, 55, 49, 3, 21, 48, 48, 32, 31, 59, 12, 64, 28, 12, 43, 25, 10, 43]で、24以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 429。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2028\", \"prompt\": \"問題: 数列[44, 62, 5, 30, 42, 52, 49, 57, 52, 38, 14, 28, 7, 52, 14]で、21以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 476。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2029\", \"prompt\": \"問題: 数列[46, 7, 6, 8, 15, 40, 21, 2, 17, 57, 48, 41, 26, 37, 24, 63, 55, 23]で、18以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 386。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2030\", \"prompt\": \"問題: 数列[11, 57, 57, 21, 6, 14, 64, 24, 35, 64, 58, 10, 26, 45, 56]で、24以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 406。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_32_2031\", \"prompt\": \"問題: 数列[55, 46, 50, 38, 65, 6, 39, 23, 5, 44, 1, 31, 12, 31, 62, 35, 37, 61]で、12以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 594。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2000\", \"prompt\": \"問題: 数列[65, 33, 54, 58, 37, 40, 7, 64, 38]で、28以上かつ4の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 6。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2001\", \"prompt\": \"問題: 数列[27, 26, 18, 32, 17, 52, 14, 28, 11, 1, 55, 27]で、16以上かつ6の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2002\", \"prompt\": \"問題: 数列[32, 10, 28, 42, 31, 26, 17, 30, 61, 11, 46, 61, 26, 63, 60]で、13以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 9。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2003\", \"prompt\": \"問題: 数列[25, 19, 48, 53, 40, 37, 53, 16, 59, 17, 48, 26, 43, 7, 13, 65, 36, 2]で、12以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 11。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2004\", \"prompt\": \"問題: 数列[63, 8, 2, 29, 1, 24, 36, 38, 5, 12, 13, 58, 59, 62, 47]で、23以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 7。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2005\", \"prompt\": \"問題: 数列[55, 13, 21, 5, 24, 51, 32, 5, 15, 44, 6, 2, 18, 30, 27, 2, 19, 13]で、26以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2006\", \"prompt\": \"問題: 数列[42, 36, 3, 8, 7, 38, 26, 64, 7, 57, 44, 51, 2, 23, 37]で、30以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2007\", \"prompt\": \"問題: 数列[22, 48, 50, 1, 35, 27, 4, 11, 45, 34, 20, 3, 33, 36, 63, 52, 43, 53]で、31以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2008\", \"prompt\": \"問題: 数列[18, 51, 46, 25, 4, 47, 36, 62, 17]で、20以上かつ6の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2009\", \"prompt\": \"問題: 数列[28, 29, 46, 17, 10, 8, 44, 3, 37, 52, 17, 45]で、27以上かつ5の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 6。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2010\", \"prompt\": \"問題: 数列[55, 65, 63, 16, 36, 51, 14, 41, 8, 47, 52, 17, 44, 47, 51]で、31以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2011\", \"prompt\": \"問題: 数列[51, 32, 61, 14, 63, 1, 13, 32, 63, 38, 20, 6, 18, 17, 24, 8, 4, 61]で、12以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 12。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2012\", \"prompt\": \"問題: 数列[17, 6, 59, 10, 55, 38, 8, 24, 25, 50, 58, 18, 38, 52, 20]で、31以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2013\", \"prompt\": \"問題: 数列[34, 30, 1, 11, 49, 62, 65, 44, 16, 61, 45, 21, 3, 26, 64, 20, 59, 21]で、28以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 7。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2014\", \"prompt\": \"問題: 数列[37, 29, 2, 22, 33, 22, 44, 6, 56, 6, 41, 14, 20, 26, 41]で、20以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2015\", \"prompt\": \"問題: 数列[63, 4, 12, 35, 29, 43, 26, 38, 23, 18, 32, 37, 55, 33, 57, 25, 47, 35]で、12以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 12。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2016\", \"prompt\": \"問題: 数列[7, 2, 13, 51, 5, 32, 29, 42, 58]で、26以上かつ5の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 5。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2017\", \"prompt\": \"問題: 数列[35, 15, 15, 47, 30, 49, 7, 20, 32, 50, 47, 45]で、28以上かつ3の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 6。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2018\", \"prompt\": \"問題: 数列[8, 38, 52, 39, 51, 45, 27, 32, 51, 65, 25, 30, 53, 57, 21]で、19以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 10。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2019\", \"prompt\": \"問題: 数列[51, 37, 17, 65, 48, 12, 50, 12, 17, 36, 63, 50, 51, 36, 29, 57, 46, 27]で、12以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2020\", \"prompt\": \"問題: 数列[28, 19, 18, 26, 16, 54, 47, 21, 13, 8, 10, 22, 46, 54, 25]で、17以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2021\", \"prompt\": \"問題: 数列[8, 43, 55, 18, 20, 29, 30, 14, 57, 9, 51, 35, 29, 53, 22, 22, 12, 61]で、23以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 7。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2022\", \"prompt\": \"問題: 数列[1, 13, 48, 2, 14, 43, 22, 34, 45, 56, 59, 29, 38, 48, 20]で、11以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 11。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2023\", \"prompt\": \"問題: 数列[24, 41, 32, 46, 38, 11, 13, 28, 35, 33, 3, 16, 55, 45, 42, 36, 33, 54]で、13以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 13。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2024\", \"prompt\": \"問題: 数列[41, 26, 21, 53, 43, 5, 22, 18, 64]で、13以上かつ4の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 7。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2025\", \"prompt\": \"問題: 数列[52, 15, 44, 38, 33, 44, 45, 48, 32, 38, 1, 14]で、21以上かつ7の倍数ではない値だけを選び、個数を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 9。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2026\", \"prompt\": \"問題: 数列[3, 52, 49, 47, 62, 64, 4, 4, 25, 61, 26, 26, 29, 30, 64]で、22以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 11。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2027\", \"prompt\": \"問題: 数列[42, 18, 56, 4, 35, 36, 46, 60, 56, 34, 41, 38, 42, 18, 12, 1, 47, 23]で、18以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 4。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2028\", \"prompt\": \"問題: 数列[10, 45, 42, 17, 51, 40, 55, 11, 26, 8, 8, 50, 49, 2, 49]で、11以上かつ7の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2029\", \"prompt\": \"問題: 数列[45, 25, 23, 13, 22, 22, 19, 2, 41, 46, 2, 65, 19, 29, 35, 62, 8, 11]で、11以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 14。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2030\", \"prompt\": \"問題: 数列[40, 19, 1, 64, 38, 31, 17, 57, 62, 11, 14, 21, 12, 45, 40]で、15以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 6。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_33_2031\", \"prompt\": \"問題: 数列[1, 29, 36, 9, 14, 35, 7, 8, 12, 29, 65, 59, 36, 46, 35, 57, 38, 35]で、17以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2000\", \"prompt\": \"問題: 数列[61, 12, 36, 34, 39, 38, 51, 1, 29]で、14以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 61。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2001\", \"prompt\": \"問題: 数列[62, 56, 42, 7, 54, 36, 11, 54, 12, 47, 13, 39]で、31以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 62。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2002\", \"prompt\": \"問題: 数列[5, 31, 1, 34, 8, 38, 11, 52, 39, 33, 34, 33, 31, 30, 38]で、20以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 39。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2003\", \"prompt\": \"問題: 数列[58, 35, 43, 48, 7, 38, 26, 64, 40, 19, 43, 50, 44, 58, 49, 9, 58, 38]で、21以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 64。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2004\", \"prompt\": \"問題: 数列[16, 49, 31, 21, 23, 49, 34, 41, 13, 63, 45, 36, 13, 54, 30]で、29以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2005\", \"prompt\": \"問題: 数列[59, 56, 43, 43, 56, 59, 35, 43, 47, 11, 7, 35, 44, 42, 27, 59, 15, 32]で、18以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2006\", \"prompt\": \"問題: 数列[8, 60, 57, 1, 65, 34, 7, 62, 13, 18, 57, 57, 47, 19, 49]で、31以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 65。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2007\", \"prompt\": \"問題: 数列[46, 65, 41, 41, 16, 54, 62, 43, 28, 37, 41, 6, 40, 39, 4, 35, 27, 53]で、23以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 65。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2008\", \"prompt\": \"問題: 数列[35, 35, 45, 30, 51, 38, 7, 56, 53]で、14以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 56。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2009\", \"prompt\": \"問題: 数列[51, 55, 41, 3, 36, 59, 34, 28, 48, 11, 46, 21]で、25以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2010\", \"prompt\": \"問題: 数列[28, 28, 26, 24, 21, 62, 54, 38, 64, 57, 19, 24, 38, 65, 37]で、10以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 64。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2011\", \"prompt\": \"問題: 数列[3, 30, 46, 1, 46, 26, 16, 61, 7, 3, 58, 12, 4, 29, 12, 11, 35, 4]で、24以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 61。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2012\", \"prompt\": \"問題: 数列[7, 24, 39, 59, 62, 25, 45, 56, 47, 33, 44, 43, 21, 54, 3]で、24以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 62。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2013\", \"prompt\": \"問題: 数列[11, 16, 53, 5, 16, 60, 2, 19, 3, 48, 48, 38, 56, 15, 15, 48, 21, 55]で、11以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 60。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2014\", \"prompt\": \"問題: 数列[26, 52, 56, 54, 8, 40, 46, 63, 12, 26, 54, 8, 24, 39, 37]で、20以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 56。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2015\", \"prompt\": \"問題: 数列[9, 22, 8, 43, 14, 25, 13, 43, 10, 45, 59, 25, 12, 15, 48, 42, 46, 38]で、22以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2016\", \"prompt\": \"問題: 数列[43, 4, 9, 23, 5, 38, 27, 42, 58]で、10以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 58。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2017\", \"prompt\": \"問題: 数列[58, 8, 27, 22, 44, 22, 8, 49, 55, 21, 58, 24]で、9以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 58。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2018\", \"prompt\": \"問題: 数列[24, 11, 54, 18, 58, 41, 29, 32, 63, 10, 46, 44, 3, 63, 25]で、27以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2019\", \"prompt\": \"問題: 数列[30, 41, 15, 50, 9, 42, 59, 36, 62, 7, 40, 7, 47, 13, 6, 26, 56, 42]で、16以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 62。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2020\", \"prompt\": \"問題: 数列[23, 52, 23, 9, 3, 23, 27, 31, 53, 44, 30, 28, 45, 15, 28]で、17以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 53。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2021\", \"prompt\": \"問題: 数列[17, 24, 60, 62, 53, 30, 58, 6, 8, 65, 5, 31, 50, 51, 28, 47, 25, 62]で、22以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 62。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2022\", \"prompt\": \"問題: 数列[11, 54, 64, 23, 58, 5, 4, 62, 44, 38, 28, 47, 45, 1, 41]で、30以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 64。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2023\", \"prompt\": \"問題: 数列[7, 62, 10, 10, 35, 33, 23, 3, 44, 41, 18, 21, 19, 18, 1, 46, 57, 62]で、16以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 62。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2024\", \"prompt\": \"問題: 数列[6, 23, 26, 47, 7, 21, 48, 48, 59]で、12以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2025\", \"prompt\": \"問題: 数列[52, 17, 27, 50, 17, 19, 33, 7, 62, 8, 18, 35]で、23以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 35。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2026\", \"prompt\": \"問題: 数列[51, 1, 59, 15, 50, 27, 9, 13, 10, 2, 22, 5, 2, 7, 51]で、9以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2027\", \"prompt\": \"問題: 数列[54, 15, 55, 65, 57, 28, 51, 25, 45, 24, 57, 25, 21, 59, 53, 29, 1, 20]で、25以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 65。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2028\", \"prompt\": \"問題: 数列[37, 24, 50, 50, 38, 58, 16, 25, 48, 20, 23, 54, 63, 42, 16]で、27以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 63。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2029\", \"prompt\": \"問題: 数列[27, 19, 41, 23, 65, 19, 16, 65, 15, 24, 33, 45, 11, 21, 52, 21, 13, 20]で、9以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 65。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2030\", \"prompt\": \"問題: 数列[12, 59, 16, 49, 52, 5, 56, 56, 16, 16, 47, 36, 25, 51, 50]で、14以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 59。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_34_2031\", \"prompt\": \"問題: 数列[30, 23, 32, 48, 60, 13, 41, 32, 40, 63, 2, 22, 43, 4, 7, 52, 25, 16]で、31以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 60。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2000\", \"prompt\": \"問題: 数列[57, 18, 21, 1, 53, 51, 25, 27, 52]で、10以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 12857。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2001\", \"prompt\": \"問題: 数列[53, 58, 53, 22, 61, 50, 45, 29, 24, 19, 54, 38]で、27以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 19513。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2002\", \"prompt\": \"問題: 数列[40, 65, 60, 9, 56, 10, 12, 43, 9, 23, 6, 38, 14, 53, 2]で、11以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 15788。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2003\", \"prompt\": \"問題: 数列[15, 55, 60, 3, 52, 65, 36, 27, 51, 27, 22, 57, 1, 27, 36, 30, 65, 51]で、13以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 22338。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2004\", \"prompt\": \"問題: 数列[50, 13, 15, 39, 60, 31, 26, 30, 57, 52, 37, 31, 17, 44, 28]で、9以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 14619。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2005\", \"prompt\": \"問題: 数列[25, 40, 10, 14, 50, 9, 35, 47, 60, 41, 46, 44, 60, 59, 9, 25, 37, 52]で、19以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 22071。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2006\", \"prompt\": \"問題: 数列[14, 29, 6, 22, 29, 19, 41, 28, 32, 6, 4, 23, 20, 8, 13]で、12以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 4422。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2007\", \"prompt\": \"問題: 数列[23, 57, 37, 62, 32, 46, 34, 30, 12, 32, 20, 37, 55, 7, 3, 27, 47, 58]で、21以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 12479。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2008\", \"prompt\": \"問題: 数列[61, 26, 65, 20, 65, 6, 12, 9, 42]で、24以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 12847。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2009\", \"prompt\": \"問題: 数列[63, 18, 58, 60, 15, 43, 43, 12, 56, 44, 30, 54]で、18以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 16738。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2010\", \"prompt\": \"問題: 数列[2, 45, 21, 20, 12, 15, 3, 40, 6, 46, 53, 58, 26, 51, 54]で、27以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 9889。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2011\", \"prompt\": \"問題: 数列[40, 45, 38, 51, 44, 36, 47, 57, 24, 38, 33, 49, 65, 26, 9, 38, 22, 50]で、21以上かつ4の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 25791。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2012\", \"prompt\": \"問題: 数列[53, 25, 34, 43, 12, 22, 47, 7, 13, 37, 18, 21, 55, 31, 39]で、9以上かつ4の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 16942。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2013\", \"prompt\": \"問題: 数列[43, 25, 51, 2, 43, 37, 36, 5, 4, 35, 6, 28, 25, 46, 65, 60, 34, 44]で、23以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 14368。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2014\", \"prompt\": \"問題: 数列[48, 48, 53, 46, 26, 29, 48, 43, 44, 32, 45, 19, 34, 58, 17]で、10以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 16421。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2015\", \"prompt\": \"問題: 数列[19, 9, 32, 51, 26, 44, 62, 24, 62, 38, 50, 3, 44, 22, 53, 1, 47, 18]で、9以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8061。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2016\", \"prompt\": \"問題: 数列[65, 24, 39, 34, 27, 27, 61, 4, 35]で、26以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 10327。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2017\", \"prompt\": \"問題: 数列[15, 60, 27, 18, 53, 65, 59, 11, 5, 25, 26, 54]で、18以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 11869。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2018\", \"prompt\": \"問題: 数列[26, 58, 31, 15, 48, 29, 15, 1, 32, 26, 15, 42, 23, 51, 34]で、24以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 8698。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2019\", \"prompt\": \"問題: 数列[4, 14, 38, 60, 38, 44, 20, 63, 26, 8, 25, 27, 30, 42, 64, 38, 17, 43]で、21以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 18743。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2020\", \"prompt\": \"問題: 数列[41, 12, 62, 47, 39, 41, 16, 21, 20, 52, 52, 54, 25, 64, 17]で、17以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 22195。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2021\", \"prompt\": \"問題: 数列[29, 58, 37, 59, 18, 27, 19, 39, 32, 52, 19, 22, 55, 36, 54, 22, 24, 3]で、15以上かつ4の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 19260。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2022\", \"prompt\": \"問題: 数列[3, 51, 49, 56, 58, 47, 27, 48, 2, 56, 17, 19, 3, 62, 15]で、31以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 20691。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2023\", \"prompt\": \"問題: 数列[47, 56, 60, 2, 31, 29, 29, 43, 13, 50, 53, 12, 55, 30, 51, 65, 41, 6]で、15以上かつ4の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 24442。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2024\", \"prompt\": \"問題: 数列[40, 13, 42, 36, 7, 61, 25, 50, 56]で、26以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 9117。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2025\", \"prompt\": \"問題: 数列[61, 34, 1, 8, 51, 20, 65, 54, 56, 1, 54, 62]で、17以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 20290。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2026\", \"prompt\": \"問題: 数列[47, 11, 40, 28, 43, 63, 26, 1, 6, 47, 27, 62, 47, 59, 45]で、23以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 18861。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2027\", \"prompt\": \"問題: 数列[23, 2, 65, 24, 27, 51, 49, 26, 42, 7, 47, 6, 8, 16, 22, 39, 36, 50]で、30以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 11335。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2028\", \"prompt\": \"問題: 数列[10, 59, 60, 34, 27, 25, 22, 45, 14, 2, 30, 43, 57, 61, 26]で、23以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 14861。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2029\", \"prompt\": \"問題: 数列[40, 40, 38, 3, 40, 64, 58, 33, 47, 37, 37, 39, 36, 37, 23, 42, 48, 45]で、11以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 28784。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2030\", \"prompt\": \"問題: 数列[10, 24, 42, 45, 58, 28, 42, 43, 56, 39, 19, 3, 56, 57, 42]で、26以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 19064。\"}\n"
  "{\"id\": \"v48_logic_unseen_eval_35_2031\", \"prompt\": \"問題: 数列[3, 18, 37, 51, 16, 12, 29, 18, 31, 9, 22, 58, 57, 62, 32, 5, 50, 4]で、11以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n条件と根拠を順に考えてください。\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n答え: 9021。\"}\n"
 ),
 "python_unseen_eval.jsonl": (
  "{\"id\": \"v48_python_unseen_eval_32_2000\", \"prompt\": \"問題: 数列[36, 60, 3, 29, 14, 31, 61, 18, 65]で、31以上かつ3の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[36, 60, 3, 29, 14, 31, 61, 18, 65]\\na=[x for x in v if x>=31 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n157\\n```\\n答え: 157。実行結果を確認しました。\", \"python_code\": \"v=[36, 60, 3, 29, 14, 31, 61, 18, 65]\\na=[x for x in v if x>=31 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2001\", \"prompt\": \"問題: 数列[8, 61, 31, 25, 15, 13, 38, 19, 1, 30, 15, 12]で、14以上かつ6の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[8, 61, 31, 25, 15, 13, 38, 19, 1, 30, 15, 12]\\na=[x for x in v if x>=14 and x%6!=0]\\nprint(sum(a))\\n```\\n```output\\n204\\n```\\n答え: 204。実行結果を確認しました。\", \"python_code\": \"v=[8, 61, 31, 25, 15, 13, 38, 19, 1, 30, 15, 12]\\na=[x for x in v if x>=14 and x%6!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2002\", \"prompt\": \"問題: 数列[45, 5, 46, 16, 25, 32, 40, 19, 48, 53, 1, 51, 26, 61, 39]で、30以上かつ2の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[45, 5, 46, 16, 25, 32, 40, 19, 48, 53, 1, 51, 26, 61, 39]\\na=[x for x in v if x>=30 and x%2!=0]\\nprint(sum(a))\\n```\\n```output\\n249\\n```\\n答え: 249。実行結果を確認しました。\", \"python_code\": \"v=[45, 5, 46, 16, 25, 32, 40, 19, 48, 53, 1, 51, 26, 61, 39]\\na=[x for x in v if x>=30 and x%2!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2003\", \"prompt\": \"問題: 数列[56, 18, 31, 31, 22, 41, 52, 41, 35, 39, 11, 30, 43, 39, 4, 12, 62, 43]で、27以上かつ2の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[56, 18, 31, 31, 22, 41, 52, 41, 35, 39, 11, 30, 43, 39, 4, 12, 62, 43]\\na=[x for x in v if x>=27 and x%2!=0]\\nprint(sum(a))\\n```\\n```output\\n343\\n```\\n答え: 343。実行結果を確認しました。\", \"python_code\": \"v=[56, 18, 31, 31, 22, 41, 52, 41, 35, 39, 11, 30, 43, 39, 4, 12, 62, 43]\\na=[x for x in v if x>=27 and x%2!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2004\", \"prompt\": \"問題: 数列[60, 44, 59, 4, 55, 4, 25, 20, 12, 19, 26, 39, 30, 1, 24]で、17以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[60, 44, 59, 4, 55, 4, 25, 20, 12, 19, 26, 39, 30, 1, 24]\\na=[x for x in v if x>=17 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n248\\n```\\n答え: 248。実行結果を確認しました。\", \"python_code\": \"v=[60, 44, 59, 4, 55, 4, 25, 20, 12, 19, 26, 39, 30, 1, 24]\\na=[x for x in v if x>=17 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2005\", \"prompt\": \"問題: 数列[52, 19, 42, 5, 64, 25, 40, 54, 36, 65, 52, 42, 6, 3, 48, 58, 56, 21]で、25以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[52, 19, 42, 5, 64, 25, 40, 54, 36, 65, 52, 42, 6, 3, 48, 58, 56, 21]\\na=[x for x in v if x>=25 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n412\\n```\\n答え: 412。実行結果を確認しました。\", \"python_code\": \"v=[52, 19, 42, 5, 64, 25, 40, 54, 36, 65, 52, 42, 6, 3, 48, 58, 56, 21]\\na=[x for x in v if x>=25 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2006\", \"prompt\": \"問題: 数列[49, 49, 25, 27, 63, 36, 55, 50, 48, 20, 43, 18, 48, 12, 8]で、12以上かつ6の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[49, 49, 25, 27, 63, 36, 55, 50, 48, 20, 43, 18, 48, 12, 8]\\na=[x for x in v if x>=12 and x%6!=0]\\nprint(sum(a))\\n```\\n```output\\n381\\n```\\n答え: 381。実行結果を確認しました。\", \"python_code\": \"v=[49, 49, 25, 27, 63, 36, 55, 50, 48, 20, 43, 18, 48, 12, 8]\\na=[x for x in v if x>=12 and x%6!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2007\", \"prompt\": \"問題: 数列[33, 50, 26, 7, 30, 51, 32, 25, 64, 20, 47, 11, 14, 51, 55, 35, 61, 39]で、23以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[33, 50, 26, 7, 30, 51, 32, 25, 64, 20, 47, 11, 14, 51, 55, 35, 61, 39]\\na=[x for x in v if x>=23 and x%5!=0]\\nprint(sum(a))\\n```\\n```output\\n404\\n```\\n答え: 404。実行結果を確認しました。\", \"python_code\": \"v=[33, 50, 26, 7, 30, 51, 32, 25, 64, 20, 47, 11, 14, 51, 55, 35, 61, 39]\\na=[x for x in v if x>=23 and x%5!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2008\", \"prompt\": \"問題: 数列[31, 51, 41, 48, 37, 26, 54, 17, 7]で、16以上かつ7の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[31, 51, 41, 48, 37, 26, 54, 17, 7]\\na=[x for x in v if x>=16 and x%7!=0]\\nprint(sum(a))\\n```\\n```output\\n305\\n```\\n答え: 305。実行結果を確認しました。\", \"python_code\": \"v=[31, 51, 41, 48, 37, 26, 54, 17, 7]\\na=[x for x in v if x>=16 and x%7!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2009\", \"prompt\": \"問題: 数列[37, 47, 22, 31, 60, 46, 25, 6, 42, 33, 30, 62]で、12以上かつ3の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[37, 47, 22, 31, 60, 46, 25, 6, 42, 33, 30, 62]\\na=[x for x in v if x>=12 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n270\\n```\\n答え: 270。実行結果を確認しました。\", \"python_code\": \"v=[37, 47, 22, 31, 60, 46, 25, 6, 42, 33, 30, 62]\\na=[x for x in v if x>=12 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2010\", \"prompt\": \"問題: 数列[13, 22, 28, 64, 26, 31, 52, 11, 27, 65, 14, 38, 28, 51, 36]で、10以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[13, 22, 28, 64, 26, 31, 52, 11, 27, 65, 14, 38, 28, 51, 36]\\na=[x for x in v if x>=10 and x%7!=0]\\nprint(sum(a))\\n```\\n```output\\n436\\n```\\n答え: 436。実行結果を確認しました。\", \"python_code\": \"v=[13, 22, 28, 64, 26, 31, 52, 11, 27, 65, 14, 38, 28, 51, 36]\\na=[x for x in v if x>=10 and x%7!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2011\", \"prompt\": \"問題: 数列[29, 57, 48, 55, 39, 57, 44, 26, 23, 63, 13, 5, 43, 37, 28, 10, 51, 10]で、26以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[29, 57, 48, 55, 39, 57, 44, 26, 23, 63, 13, 5, 43, 37, 28, 10, 51, 10]\\na=[x for x in v if x>=26 and x%7!=0]\\nprint(sum(a))\\n```\\n```output\\n486\\n```\\n答え: 486。実行結果を確認しました。\", \"python_code\": \"v=[29, 57, 48, 55, 39, 57, 44, 26, 23, 63, 13, 5, 43, 37, 28, 10, 51, 10]\\na=[x for x in v if x>=26 and x%7!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2012\", \"prompt\": \"問題: 数列[37, 21, 50, 44, 57, 16, 65, 2, 62, 8, 31, 52, 59, 62, 16]で、12以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[37, 21, 50, 44, 57, 16, 65, 2, 62, 8, 31, 52, 59, 62, 16]\\na=[x for x in v if x>=12 and x%7!=0]\\nprint(sum(a))\\n```\\n```output\\n551\\n```\\n答え: 551。実行結果を確認しました。\", \"python_code\": \"v=[37, 21, 50, 44, 57, 16, 65, 2, 62, 8, 31, 52, 59, 62, 16]\\na=[x for x in v if x>=12 and x%7!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2013\", \"prompt\": \"問題: 数列[14, 44, 18, 31, 9, 13, 8, 22, 50, 46, 2, 9, 3, 33, 45, 35, 42, 52]で、14以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[14, 44, 18, 31, 9, 13, 8, 22, 50, 46, 2, 9, 3, 33, 45, 35, 42, 52]\\na=[x for x in v if x>=14 and x%5!=0]\\nprint(sum(a))\\n```\\n```output\\n302\\n```\\n答え: 302。実行結果を確認しました。\", \"python_code\": \"v=[14, 44, 18, 31, 9, 13, 8, 22, 50, 46, 2, 9, 3, 33, 45, 35, 42, 52]\\na=[x for x in v if x>=14 and x%5!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2014\", \"prompt\": \"問題: 数列[64, 32, 51, 42, 13, 9, 27, 17, 58, 40, 25, 53, 59, 11, 65]で、24以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[64, 32, 51, 42, 13, 9, 27, 17, 58, 40, 25, 53, 59, 11, 65]\\na=[x for x in v if x>=24 and x%5!=0]\\nprint(sum(a))\\n```\\n```output\\n386\\n```\\n答え: 386。実行結果を確認しました。\", \"python_code\": \"v=[64, 32, 51, 42, 13, 9, 27, 17, 58, 40, 25, 53, 59, 11, 65]\\na=[x for x in v if x>=24 and x%5!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2015\", \"prompt\": \"問題: 数列[55, 53, 38, 63, 60, 32, 44, 52, 7, 48, 43, 24, 25, 65, 59, 9, 18, 9]で、11以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[55, 53, 38, 63, 60, 32, 44, 52, 7, 48, 43, 24, 25, 65, 59, 9, 18, 9]\\na=[x for x in v if x>=11 and x%5!=0]\\nprint(sum(a))\\n```\\n```output\\n474\\n```\\n答え: 474。実行結果を確認しました。\", \"python_code\": \"v=[55, 53, 38, 63, 60, 32, 44, 52, 7, 48, 43, 24, 25, 65, 59, 9, 18, 9]\\na=[x for x in v if x>=11 and x%5!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2016\", \"prompt\": \"問題: 数列[28, 8, 55, 3, 10, 13, 30, 60, 61]で、21以上かつ4の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[28, 8, 55, 3, 10, 13, 30, 60, 61]\\na=[x for x in v if x>=21 and x%4!=0]\\nprint(sum(a))\\n```\\n```output\\n146\\n```\\n答え: 146。実行結果を確認しました。\", \"python_code\": \"v=[28, 8, 55, 3, 10, 13, 30, 60, 61]\\na=[x for x in v if x>=21 and x%4!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2017\", \"prompt\": \"問題: 数列[23, 29, 40, 48, 57, 32, 57, 3, 13, 26, 18, 42]で、10以上かつ6の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[23, 29, 40, 48, 57, 32, 57, 3, 13, 26, 18, 42]\\na=[x for x in v if x>=10 and x%6!=0]\\nprint(sum(a))\\n```\\n```output\\n277\\n```\\n答え: 277。実行結果を確認しました。\", \"python_code\": \"v=[23, 29, 40, 48, 57, 32, 57, 3, 13, 26, 18, 42]\\na=[x for x in v if x>=10 and x%6!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2018\", \"prompt\": \"問題: 数列[1, 33, 58, 44, 25, 45, 30, 33, 17, 40, 23, 46, 21, 24, 61]で、27以上かつ6の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[1, 33, 58, 44, 25, 45, 30, 33, 17, 40, 23, 46, 21, 24, 61]\\na=[x for x in v if x>=27 and x%6!=0]\\nprint(sum(a))\\n```\\n```output\\n360\\n```\\n答え: 360。実行結果を確認しました。\", \"python_code\": \"v=[1, 33, 58, 44, 25, 45, 30, 33, 17, 40, 23, 46, 21, 24, 61]\\na=[x for x in v if x>=27 and x%6!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2019\", \"prompt\": \"問題: 数列[4, 7, 12, 46, 36, 23, 30, 37, 25, 12, 59, 18, 8, 37, 6, 12, 41, 35]で、12以上かつ4の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[4, 7, 12, 46, 36, 23, 30, 37, 25, 12, 59, 18, 8, 37, 6, 12, 41, 35]\\na=[x for x in v if x>=12 and x%4!=0]\\nprint(sum(a))\\n```\\n```output\\n351\\n```\\n答え: 351。実行結果を確認しました。\", \"python_code\": \"v=[4, 7, 12, 46, 36, 23, 30, 37, 25, 12, 59, 18, 8, 37, 6, 12, 41, 35]\\na=[x for x in v if x>=12 and x%4!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2020\", \"prompt\": \"問題: 数列[20, 61, 60, 15, 57, 61, 59, 11, 43, 17, 10, 13, 12, 28, 20]で、15以上かつ7の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[20, 61, 60, 15, 57, 61, 59, 11, 43, 17, 10, 13, 12, 28, 20]\\na=[x for x in v if x>=15 and x%7!=0]\\nprint(sum(a))\\n```\\n```output\\n413\\n```\\n答え: 413。実行結果を確認しました。\", \"python_code\": \"v=[20, 61, 60, 15, 57, 61, 59, 11, 43, 17, 10, 13, 12, 28, 20]\\na=[x for x in v if x>=15 and x%7!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2021\", \"prompt\": \"問題: 数列[8, 38, 13, 45, 58, 18, 12, 65, 57, 20, 51, 30, 22, 48, 42, 48, 18, 18]で、31以上かつ4の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[8, 38, 13, 45, 58, 18, 12, 65, 57, 20, 51, 30, 22, 48, 42, 48, 18, 18]\\na=[x for x in v if x>=31 and x%4!=0]\\nprint(sum(a))\\n```\\n```output\\n356\\n```\\n答え: 356。実行結果を確認しました。\", \"python_code\": \"v=[8, 38, 13, 45, 58, 18, 12, 65, 57, 20, 51, 30, 22, 48, 42, 48, 18, 18]\\na=[x for x in v if x>=31 and x%4!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2022\", \"prompt\": \"問題: 数列[35, 7, 8, 24, 54, 33, 63, 46, 24, 16, 30, 33, 25, 6, 62]で、12以上かつ4の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[35, 7, 8, 24, 54, 33, 63, 46, 24, 16, 30, 33, 25, 6, 62]\\na=[x for x in v if x>=12 and x%4!=0]\\nprint(sum(a))\\n```\\n```output\\n381\\n```\\n答え: 381。実行結果を確認しました。\", \"python_code\": \"v=[35, 7, 8, 24, 54, 33, 63, 46, 24, 16, 30, 33, 25, 6, 62]\\na=[x for x in v if x>=12 and x%4!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2023\", \"prompt\": \"問題: 数列[30, 38, 41, 44, 51, 7, 8, 51, 52, 65, 25, 52, 52, 45, 49, 52, 27, 56]で、21以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[30, 38, 41, 44, 51, 7, 8, 51, 52, 65, 25, 52, 52, 45, 49, 52, 27, 56]\\na=[x for x in v if x>=21 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n526\\n```\\n答え: 526。実行結果を確認しました。\", \"python_code\": \"v=[30, 38, 41, 44, 51, 7, 8, 51, 52, 65, 25, 52, 52, 45, 49, 52, 27, 56]\\na=[x for x in v if x>=21 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2024\", \"prompt\": \"問題: 数列[21, 47, 12, 30, 20, 47, 22, 26, 53]で、11以上かつ3の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[21, 47, 12, 30, 20, 47, 22, 26, 53]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n215\\n```\\n答え: 215。実行結果を確認しました。\", \"python_code\": \"v=[21, 47, 12, 30, 20, 47, 22, 26, 53]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2025\", \"prompt\": \"問題: 数列[28, 20, 28, 8, 12, 42, 8, 46, 49, 45, 30, 19]で、18以上かつ2の倍数ではない値だけを選び、合計を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[28, 20, 28, 8, 12, 42, 8, 46, 49, 45, 30, 19]\\na=[x for x in v if x>=18 and x%2!=0]\\nprint(sum(a))\\n```\\n```output\\n113\\n```\\n答え: 113。実行結果を確認しました。\", \"python_code\": \"v=[28, 20, 28, 8, 12, 42, 8, 46, 49, 45, 30, 19]\\na=[x for x in v if x>=18 and x%2!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2026\", \"prompt\": \"問題: 数列[21, 43, 38, 28, 50, 24, 14, 52, 22, 31, 19, 40, 25, 48, 38]で、16以上かつ2の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[21, 43, 38, 28, 50, 24, 14, 52, 22, 31, 19, 40, 25, 48, 38]\\na=[x for x in v if x>=16 and x%2!=0]\\nprint(sum(a))\\n```\\n```output\\n139\\n```\\n答え: 139。実行結果を確認しました。\", \"python_code\": \"v=[21, 43, 38, 28, 50, 24, 14, 52, 22, 31, 19, 40, 25, 48, 38]\\na=[x for x in v if x>=16 and x%2!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2027\", \"prompt\": \"問題: 数列[22, 31, 41, 53, 19, 29, 45, 41, 39, 9, 16, 34, 32, 8, 33, 38, 7, 35]で、30以上かつ2の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[22, 31, 41, 53, 19, 29, 45, 41, 39, 9, 16, 34, 32, 8, 33, 38, 7, 35]\\na=[x for x in v if x>=30 and x%2!=0]\\nprint(sum(a))\\n```\\n```output\\n318\\n```\\n答え: 318。実行結果を確認しました。\", \"python_code\": \"v=[22, 31, 41, 53, 19, 29, 45, 41, 39, 9, 16, 34, 32, 8, 33, 38, 7, 35]\\na=[x for x in v if x>=30 and x%2!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2028\", \"prompt\": \"問題: 数列[31, 14, 30, 17, 44, 56, 7, 59, 44, 23, 43, 31, 21, 15, 8]で、10以上かつ2の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[31, 14, 30, 17, 44, 56, 7, 59, 44, 23, 43, 31, 21, 15, 8]\\na=[x for x in v if x>=10 and x%2!=0]\\nprint(sum(a))\\n```\\n```output\\n240\\n```\\n答え: 240。実行結果を確認しました。\", \"python_code\": \"v=[31, 14, 30, 17, 44, 56, 7, 59, 44, 23, 43, 31, 21, 15, 8]\\na=[x for x in v if x>=10 and x%2!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2029\", \"prompt\": \"問題: 数列[4, 46, 31, 5, 16, 30, 9, 65, 64, 52, 47, 49, 35, 7, 53, 58, 41, 3]で、9以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[4, 46, 31, 5, 16, 30, 9, 65, 64, 52, 47, 49, 35, 7, 53, 58, 41, 3]\\na=[x for x in v if x>=9 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n557\\n```\\n答え: 557。実行結果を確認しました。\", \"python_code\": \"v=[4, 46, 31, 5, 16, 30, 9, 65, 64, 52, 47, 49, 35, 7, 53, 58, 41, 3]\\na=[x for x in v if x>=9 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2030\", \"prompt\": \"問題: 数列[52, 9, 21, 61, 58, 63, 2, 18, 20, 19, 56, 47, 27, 54, 24]で、19以上かつ3の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[52, 9, 21, 61, 58, 63, 2, 18, 20, 19, 56, 47, 27, 54, 24]\\na=[x for x in v if x>=19 and x%3!=0]\\nprint(sum(a))\\n```\\n```output\\n313\\n```\\n答え: 313。実行結果を確認しました。\", \"python_code\": \"v=[52, 9, 21, 61, 58, 63, 2, 18, 20, 19, 56, 47, 27, 54, 24]\\na=[x for x in v if x>=19 and x%3!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_32_2031\", \"prompt\": \"問題: 数列[44, 10, 16, 39, 34, 38, 65, 11, 56, 11, 55, 11, 65, 32, 43, 16, 6, 32]で、14以上かつ5の倍数ではない値だけを選び、合計を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[44, 10, 16, 39, 34, 38, 65, 11, 56, 11, 55, 11, 65, 32, 43, 16, 6, 32]\\na=[x for x in v if x>=14 and x%5!=0]\\nprint(sum(a))\\n```\\n```output\\n350\\n```\\n答え: 350。実行結果を確認しました。\", \"python_code\": \"v=[44, 10, 16, 39, 34, 38, 65, 11, 56, 11, 55, 11, 65, 32, 43, 16, 6, 32]\\na=[x for x in v if x>=14 and x%5!=0]\\nprint(sum(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2000\", \"prompt\": \"問題: 数列[40, 18, 20, 64, 3, 30, 65, 11, 56]で、13以上かつ3の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[40, 18, 20, 64, 3, 30, 65, 11, 56]\\na=[x for x in v if x>=13 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n5\\n```\\n答え: 5。実行結果を確認しました。\", \"python_code\": \"v=[40, 18, 20, 64, 3, 30, 65, 11, 56]\\na=[x for x in v if x>=13 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2001\", \"prompt\": \"問題: 数列[22, 5, 46, 15, 45, 1, 21, 15, 61, 41, 41, 26]で、17以上かつ3の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[22, 5, 46, 15, 45, 1, 21, 15, 61, 41, 41, 26]\\na=[x for x in v if x>=17 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n6\\n```\\n答え: 6。実行結果を確認しました。\", \"python_code\": \"v=[22, 5, 46, 15, 45, 1, 21, 15, 61, 41, 41, 26]\\na=[x for x in v if x>=17 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2002\", \"prompt\": \"問題: 数列[65, 30, 30, 27, 37, 36, 21, 9, 36, 1, 27, 25, 13, 12, 3]で、26以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[65, 30, 30, 27, 37, 36, 21, 9, 36, 1, 27, 25, 13, 12, 3]\\na=[x for x in v if x>=26 and x%6!=0]\\nprint(len(a))\\n```\\n```output\\n4\\n```\\n答え: 4。実行結果を確認しました。\", \"python_code\": \"v=[65, 30, 30, 27, 37, 36, 21, 9, 36, 1, 27, 25, 13, 12, 3]\\na=[x for x in v if x>=26 and x%6!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2003\", \"prompt\": \"問題: 数列[6, 60, 45, 15, 41, 49, 18, 54, 5, 13, 15, 18, 65, 52, 25, 29, 8, 43]で、26以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[6, 60, 45, 15, 41, 49, 18, 54, 5, 13, 15, 18, 65, 52, 25, 29, 8, 43]\\na=[x for x in v if x>=26 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n6\\n```\\n答え: 6。実行結果を確認しました。\", \"python_code\": \"v=[6, 60, 45, 15, 41, 49, 18, 54, 5, 13, 15, 18, 65, 52, 25, 29, 8, 43]\\na=[x for x in v if x>=26 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2004\", \"prompt\": \"問題: 数列[29, 55, 56, 3, 51, 32, 51, 45, 1, 42, 12, 65, 36, 47, 32]で、29以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[29, 55, 56, 3, 51, 32, 51, 45, 1, 42, 12, 65, 36, 47, 32]\\na=[x for x in v if x>=29 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n9\\n```\\n答え: 9。実行結果を確認しました。\", \"python_code\": \"v=[29, 55, 56, 3, 51, 32, 51, 45, 1, 42, 12, 65, 36, 47, 32]\\na=[x for x in v if x>=29 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2005\", \"prompt\": \"問題: 数列[21, 45, 61, 63, 57, 43, 50, 3, 44, 42, 29, 21, 37, 34, 56, 7, 56, 29]で、15以上かつ7の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[21, 45, 61, 63, 57, 43, 50, 3, 44, 42, 29, 21, 37, 34, 56, 7, 56, 29]\\na=[x for x in v if x>=15 and x%7!=0]\\nprint(len(a))\\n```\\n```output\\n10\\n```\\n答え: 10。実行結果を確認しました。\", \"python_code\": \"v=[21, 45, 61, 63, 57, 43, 50, 3, 44, 42, 29, 21, 37, 34, 56, 7, 56, 29]\\na=[x for x in v if x>=15 and x%7!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2006\", \"prompt\": \"問題: 数列[19, 26, 27, 43, 45, 15, 47, 35, 59, 10, 47, 65, 51, 37, 7]で、11以上かつ7の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[19, 26, 27, 43, 45, 15, 47, 35, 59, 10, 47, 65, 51, 37, 7]\\na=[x for x in v if x>=11 and x%7!=0]\\nprint(len(a))\\n```\\n```output\\n12\\n```\\n答え: 12。実行結果を確認しました。\", \"python_code\": \"v=[19, 26, 27, 43, 45, 15, 47, 35, 59, 10, 47, 65, 51, 37, 7]\\na=[x for x in v if x>=11 and x%7!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2007\", \"prompt\": \"問題: 数列[56, 40, 23, 10, 41, 23, 43, 15, 43, 14, 47, 54, 56, 30, 35, 35, 26, 65]で、26以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[56, 40, 23, 10, 41, 23, 43, 15, 43, 14, 47, 54, 56, 30, 35, 35, 26, 65]\\na=[x for x in v if x>=26 and x%4!=0]\\nprint(len(a))\\n```\\n```output\\n10\\n```\\n答え: 10。実行結果を確認しました。\", \"python_code\": \"v=[56, 40, 23, 10, 41, 23, 43, 15, 43, 14, 47, 54, 56, 30, 35, 35, 26, 65]\\na=[x for x in v if x>=26 and x%4!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2008\", \"prompt\": \"問題: 数列[15, 47, 44, 61, 14, 16, 31, 51, 3]で、10以上かつ4の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[15, 47, 44, 61, 14, 16, 31, 51, 3]\\na=[x for x in v if x>=10 and x%4!=0]\\nprint(len(a))\\n```\\n```output\\n6\\n```\\n答え: 6。実行結果を確認しました。\", \"python_code\": \"v=[15, 47, 44, 61, 14, 16, 31, 51, 3]\\na=[x for x in v if x>=10 and x%4!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2009\", \"prompt\": \"問題: 数列[22, 42, 27, 5, 7, 13, 8, 29, 48, 61, 20, 43]で、20以上かつ3の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[22, 42, 27, 5, 7, 13, 8, 29, 48, 61, 20, 43]\\na=[x for x in v if x>=20 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n5\\n```\\n答え: 5。実行結果を確認しました。\", \"python_code\": \"v=[22, 42, 27, 5, 7, 13, 8, 29, 48, 61, 20, 43]\\na=[x for x in v if x>=20 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2010\", \"prompt\": \"問題: 数列[9, 58, 46, 49, 12, 27, 32, 29, 50, 4, 25, 55, 27, 12, 33]で、11以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[9, 58, 46, 49, 12, 27, 32, 29, 50, 4, 25, 55, 27, 12, 33]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n8\\n```\\n答え: 8。実行結果を確認しました。\", \"python_code\": \"v=[9, 58, 46, 49, 12, 27, 32, 29, 50, 4, 25, 55, 27, 12, 33]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2011\", \"prompt\": \"問題: 数列[44, 61, 42, 41, 32, 10, 42, 4, 60, 58, 31, 38, 52, 37, 56, 35, 24, 36]で、17以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[44, 61, 42, 41, 32, 10, 42, 4, 60, 58, 31, 38, 52, 37, 56, 35, 24, 36]\\na=[x for x in v if x>=17 and x%6!=0]\\nprint(len(a))\\n```\\n```output\\n11\\n```\\n答え: 11。実行結果を確認しました。\", \"python_code\": \"v=[44, 61, 42, 41, 32, 10, 42, 4, 60, 58, 31, 38, 52, 37, 56, 35, 24, 36]\\na=[x for x in v if x>=17 and x%6!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2012\", \"prompt\": \"問題: 数列[19, 55, 60, 59, 51, 35, 44, 42, 63, 8, 44, 44, 60, 4, 43]で、18以上かつ6の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[19, 55, 60, 59, 51, 35, 44, 42, 63, 8, 44, 44, 60, 4, 43]\\na=[x for x in v if x>=18 and x%6!=0]\\nprint(len(a))\\n```\\n```output\\n10\\n```\\n答え: 10。実行結果を確認しました。\", \"python_code\": \"v=[19, 55, 60, 59, 51, 35, 44, 42, 63, 8, 44, 44, 60, 4, 43]\\na=[x for x in v if x>=18 and x%6!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2013\", \"prompt\": \"問題: 数列[60, 20, 24, 6, 36, 12, 10, 11, 41, 60, 24, 24, 5, 52, 58, 60, 58, 50]で、29以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[60, 20, 24, 6, 36, 12, 10, 11, 41, 60, 24, 24, 5, 52, 58, 60, 58, 50]\\na=[x for x in v if x>=29 and x%4!=0]\\nprint(len(a))\\n```\\n```output\\n4\\n```\\n答え: 4。実行結果を確認しました。\", \"python_code\": \"v=[60, 20, 24, 6, 36, 12, 10, 11, 41, 60, 24, 24, 5, 52, 58, 60, 58, 50]\\na=[x for x in v if x>=29 and x%4!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2014\", \"prompt\": \"問題: 数列[14, 15, 48, 57, 24, 53, 56, 28, 41, 62, 31, 15, 44, 1, 16]で、24以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[14, 15, 48, 57, 24, 53, 56, 28, 41, 62, 31, 15, 44, 1, 16]\\na=[x for x in v if x>=24 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n10\\n```\\n答え: 10。実行結果を確認しました。\", \"python_code\": \"v=[14, 15, 48, 57, 24, 53, 56, 28, 41, 62, 31, 15, 44, 1, 16]\\na=[x for x in v if x>=24 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2015\", \"prompt\": \"問題: 数列[19, 52, 9, 5, 32, 33, 22, 61, 14, 26, 23, 58, 5, 50, 11, 7, 10, 51]で、25以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[19, 52, 9, 5, 32, 33, 22, 61, 14, 26, 23, 58, 5, 50, 11, 7, 10, 51]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(len(a))\\n```\\n```output\\n3\\n```\\n答え: 3。実行結果を確認しました。\", \"python_code\": \"v=[19, 52, 9, 5, 32, 33, 22, 61, 14, 26, 23, 58, 5, 50, 11, 7, 10, 51]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2016\", \"prompt\": \"問題: 数列[25, 48, 64, 25, 37, 50, 23, 18, 56]で、20以上かつ6の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[25, 48, 64, 25, 37, 50, 23, 18, 56]\\na=[x for x in v if x>=20 and x%6!=0]\\nprint(len(a))\\n```\\n```output\\n7\\n```\\n答え: 7。実行結果を確認しました。\", \"python_code\": \"v=[25, 48, 64, 25, 37, 50, 23, 18, 56]\\na=[x for x in v if x>=20 and x%6!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2017\", \"prompt\": \"問題: 数列[56, 10, 38, 17, 24, 31, 4, 11, 18, 61, 23, 18]で、10以上かつ2の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[56, 10, 38, 17, 24, 31, 4, 11, 18, 61, 23, 18]\\na=[x for x in v if x>=10 and x%2!=0]\\nprint(len(a))\\n```\\n```output\\n5\\n```\\n答え: 5。実行結果を確認しました。\", \"python_code\": \"v=[56, 10, 38, 17, 24, 31, 4, 11, 18, 61, 23, 18]\\na=[x for x in v if x>=10 and x%2!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2018\", \"prompt\": \"問題: 数列[40, 2, 64, 2, 36, 14, 47, 62, 17, 2, 56, 39, 55, 20, 41]で、18以上かつ3の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[40, 2, 64, 2, 36, 14, 47, 62, 17, 2, 56, 39, 55, 20, 41]\\na=[x for x in v if x>=18 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n8\\n```\\n答え: 8。実行結果を確認しました。\", \"python_code\": \"v=[40, 2, 64, 2, 36, 14, 47, 62, 17, 2, 56, 39, 55, 20, 41]\\na=[x for x in v if x>=18 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2019\", \"prompt\": \"問題: 数列[49, 37, 3, 23, 27, 34, 11, 38, 34, 15, 40, 8, 8, 65, 50, 37, 15, 7]で、27以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[49, 37, 3, 23, 27, 34, 11, 38, 34, 15, 40, 8, 8, 65, 50, 37, 15, 7]\\na=[x for x in v if x>=27 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n7\\n```\\n答え: 7。実行結果を確認しました。\", \"python_code\": \"v=[49, 37, 3, 23, 27, 34, 11, 38, 34, 15, 40, 8, 8, 65, 50, 37, 15, 7]\\na=[x for x in v if x>=27 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2020\", \"prompt\": \"問題: 数列[44, 2, 55, 7, 49, 42, 7, 3, 56, 19, 46, 45, 58, 28, 60]で、15以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[44, 2, 55, 7, 49, 42, 7, 3, 56, 19, 46, 45, 58, 28, 60]\\na=[x for x in v if x>=15 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n8\\n```\\n答え: 8。実行結果を確認しました。\", \"python_code\": \"v=[44, 2, 55, 7, 49, 42, 7, 3, 56, 19, 46, 45, 58, 28, 60]\\na=[x for x in v if x>=15 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2021\", \"prompt\": \"問題: 数列[12, 34, 18, 13, 15, 50, 29, 23, 44, 49, 36, 28, 62, 29, 18, 49, 26, 34]で、23以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[12, 34, 18, 13, 15, 50, 29, 23, 44, 49, 36, 28, 62, 29, 18, 49, 26, 34]\\na=[x for x in v if x>=23 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n12\\n```\\n答え: 12。実行結果を確認しました。\", \"python_code\": \"v=[12, 34, 18, 13, 15, 50, 29, 23, 44, 49, 36, 28, 62, 29, 18, 49, 26, 34]\\na=[x for x in v if x>=23 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2022\", \"prompt\": \"問題: 数列[27, 7, 21, 9, 32, 36, 42, 58, 47, 65, 2, 22, 35, 32, 25]で、25以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[27, 7, 21, 9, 32, 36, 42, 58, 47, 65, 2, 22, 35, 32, 25]\\na=[x for x in v if x>=25 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n7\\n```\\n答え: 7。実行結果を確認しました。\", \"python_code\": \"v=[27, 7, 21, 9, 32, 36, 42, 58, 47, 65, 2, 22, 35, 32, 25]\\na=[x for x in v if x>=25 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2023\", \"prompt\": \"問題: 数列[54, 3, 27, 42, 37, 55, 13, 29, 59, 54, 8, 25, 40, 38, 39, 50, 4, 28]で、17以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[54, 3, 27, 42, 37, 55, 13, 29, 59, 54, 8, 25, 40, 38, 39, 50, 4, 28]\\na=[x for x in v if x>=17 and x%2!=0]\\nprint(len(a))\\n```\\n```output\\n7\\n```\\n答え: 7。実行結果を確認しました。\", \"python_code\": \"v=[54, 3, 27, 42, 37, 55, 13, 29, 59, 54, 8, 25, 40, 38, 39, 50, 4, 28]\\na=[x for x in v if x>=17 and x%2!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2024\", \"prompt\": \"問題: 数列[30, 23, 27, 59, 64, 48, 63, 9, 15]で、17以上かつ4の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[30, 23, 27, 59, 64, 48, 63, 9, 15]\\na=[x for x in v if x>=17 and x%4!=0]\\nprint(len(a))\\n```\\n```output\\n5\\n```\\n答え: 5。実行結果を確認しました。\", \"python_code\": \"v=[30, 23, 27, 59, 64, 48, 63, 9, 15]\\na=[x for x in v if x>=17 and x%4!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2025\", \"prompt\": \"問題: 数列[2, 2, 20, 17, 8, 21, 47, 3, 16, 23, 7, 17]で、11以上かつ3の倍数ではない値だけを選び、個数を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[2, 2, 20, 17, 8, 21, 47, 3, 16, 23, 7, 17]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(len(a))\\n```\\n```output\\n6\\n```\\n答え: 6。実行結果を確認しました。\", \"python_code\": \"v=[2, 2, 20, 17, 8, 21, 47, 3, 16, 23, 7, 17]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2026\", \"prompt\": \"問題: 数列[3, 14, 14, 40, 51, 65, 63, 5, 52, 1, 53, 63, 21, 23, 33]で、28以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[3, 14, 14, 40, 51, 65, 63, 5, 52, 1, 53, 63, 21, 23, 33]\\na=[x for x in v if x>=28 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n6\\n```\\n答え: 6。実行結果を確認しました。\", \"python_code\": \"v=[3, 14, 14, 40, 51, 65, 63, 5, 52, 1, 53, 63, 21, 23, 33]\\na=[x for x in v if x>=28 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2027\", \"prompt\": \"問題: 数列[51, 52, 1, 9, 10, 20, 26, 27, 8, 62, 34, 56, 16, 45, 45, 32, 39, 17]で、18以上かつ4の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[51, 52, 1, 9, 10, 20, 26, 27, 8, 62, 34, 56, 16, 45, 45, 32, 39, 17]\\na=[x for x in v if x>=18 and x%4!=0]\\nprint(len(a))\\n```\\n```output\\n8\\n```\\n答え: 8。実行結果を確認しました。\", \"python_code\": \"v=[51, 52, 1, 9, 10, 20, 26, 27, 8, 62, 34, 56, 16, 45, 45, 32, 39, 17]\\na=[x for x in v if x>=18 and x%4!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2028\", \"prompt\": \"問題: 数列[22, 42, 11, 49, 50, 19, 40, 42, 15, 9, 45, 55, 3, 22, 37]で、23以上かつ7の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[22, 42, 11, 49, 50, 19, 40, 42, 15, 9, 45, 55, 3, 22, 37]\\na=[x for x in v if x>=23 and x%7!=0]\\nprint(len(a))\\n```\\n```output\\n5\\n```\\n答え: 5。実行結果を確認しました。\", \"python_code\": \"v=[22, 42, 11, 49, 50, 19, 40, 42, 15, 9, 45, 55, 3, 22, 37]\\na=[x for x in v if x>=23 and x%7!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2029\", \"prompt\": \"問題: 数列[62, 61, 60, 44, 16, 10, 34, 42, 48, 24, 7, 33, 43, 33, 35, 60, 19, 53]で、22以上かつ2の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[62, 61, 60, 44, 16, 10, 34, 42, 48, 24, 7, 33, 43, 33, 35, 60, 19, 53]\\na=[x for x in v if x>=22 and x%2!=0]\\nprint(len(a))\\n```\\n```output\\n6\\n```\\n答え: 6。実行結果を確認しました。\", \"python_code\": \"v=[62, 61, 60, 44, 16, 10, 34, 42, 48, 24, 7, 33, 43, 33, 35, 60, 19, 53]\\na=[x for x in v if x>=22 and x%2!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2030\", \"prompt\": \"問題: 数列[13, 38, 35, 6, 11, 21, 64, 10, 2, 15, 44, 28, 14, 18, 3]で、11以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[13, 38, 35, 6, 11, 21, 64, 10, 2, 15, 44, 28, 14, 18, 3]\\na=[x for x in v if x>=11 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n9\\n```\\n答え: 9。実行結果を確認しました。\", \"python_code\": \"v=[13, 38, 35, 6, 11, 21, 64, 10, 2, 15, 44, 28, 14, 18, 3]\\na=[x for x in v if x>=11 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_33_2031\", \"prompt\": \"問題: 数列[22, 34, 37, 20, 55, 48, 1, 63, 15, 41, 15, 44, 7, 5, 29, 34, 22, 55]で、19以上かつ5の倍数ではない値だけを選び、個数を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[22, 34, 37, 20, 55, 48, 1, 63, 15, 41, 15, 44, 7, 5, 29, 34, 22, 55]\\na=[x for x in v if x>=19 and x%5!=0]\\nprint(len(a))\\n```\\n```output\\n10\\n```\\n答え: 10。実行結果を確認しました。\", \"python_code\": \"v=[22, 34, 37, 20, 55, 48, 1, 63, 15, 41, 15, 44, 7, 5, 29, 34, 22, 55]\\na=[x for x in v if x>=19 and x%5!=0]\\nprint(len(a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2000\", \"prompt\": \"問題: 数列[65, 36, 53, 63, 23, 15, 28, 9, 64]で、23以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[65, 36, 53, 63, 23, 15, 28, 9, 64]\\na=[x for x in v if x>=23 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n65\\n```\\n答え: 65。実行結果を確認しました。\", \"python_code\": \"v=[65, 36, 53, 63, 23, 15, 28, 9, 64]\\na=[x for x in v if x>=23 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2001\", \"prompt\": \"問題: 数列[56, 2, 11, 14, 41, 49, 53, 54, 48, 23, 19, 52]で、11以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[56, 2, 11, 14, 41, 49, 53, 54, 48, 23, 19, 52]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n56\\n```\\n答え: 56。実行結果を確認しました。\", \"python_code\": \"v=[56, 2, 11, 14, 41, 49, 53, 54, 48, 23, 19, 52]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2002\", \"prompt\": \"問題: 数列[61, 34, 46, 13, 30, 24, 39, 31, 32, 24, 26, 55, 12, 57, 25]で、12以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[61, 34, 46, 13, 30, 24, 39, 31, 32, 24, 26, 55, 12, 57, 25]\\na=[x for x in v if x>=12 and x%5!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n61\\n```\\n答え: 61。実行結果を確認しました。\", \"python_code\": \"v=[61, 34, 46, 13, 30, 24, 39, 31, 32, 24, 26, 55, 12, 57, 25]\\na=[x for x in v if x>=12 and x%5!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2003\", \"prompt\": \"問題: 数列[34, 32, 61, 12, 12, 55, 15, 33, 30, 24, 59, 9, 32, 12, 58, 11, 59, 19]で、27以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[34, 32, 61, 12, 12, 55, 15, 33, 30, 24, 59, 9, 32, 12, 58, 11, 59, 19]\\na=[x for x in v if x>=27 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n61\\n```\\n答え: 61。実行結果を確認しました。\", \"python_code\": \"v=[34, 32, 61, 12, 12, 55, 15, 33, 30, 24, 59, 9, 32, 12, 58, 11, 59, 19]\\na=[x for x in v if x>=27 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2004\", \"prompt\": \"問題: 数列[9, 49, 43, 48, 24, 53, 20, 55, 30, 42, 44, 37, 32, 16, 13]で、16以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[9, 49, 43, 48, 24, 53, 20, 55, 30, 42, 44, 37, 32, 16, 13]\\na=[x for x in v if x>=16 and x%7!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n55\\n```\\n答え: 55。実行結果を確認しました。\", \"python_code\": \"v=[9, 49, 43, 48, 24, 53, 20, 55, 30, 42, 44, 37, 32, 16, 13]\\na=[x for x in v if x>=16 and x%7!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2005\", \"prompt\": \"問題: 数列[56, 60, 4, 14, 9, 51, 9, 34, 57, 32, 58, 54, 16, 10, 28, 35, 33, 59]で、16以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[56, 60, 4, 14, 9, 51, 9, 34, 57, 32, 58, 54, 16, 10, 28, 35, 33, 59]\\na=[x for x in v if x>=16 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n59\\n```\\n答え: 59。実行結果を確認しました。\", \"python_code\": \"v=[56, 60, 4, 14, 9, 51, 9, 34, 57, 32, 58, 54, 16, 10, 28, 35, 33, 59]\\na=[x for x in v if x>=16 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2006\", \"prompt\": \"問題: 数列[4, 19, 28, 50, 19, 29, 24, 58, 65, 50, 10, 5, 50, 23, 21]で、31以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[4, 19, 28, 50, 19, 29, 24, 58, 65, 50, 10, 5, 50, 23, 21]\\na=[x for x in v if x>=31 and x%5!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n58\\n```\\n答え: 58。実行結果を確認しました。\", \"python_code\": \"v=[4, 19, 28, 50, 19, 29, 24, 58, 65, 50, 10, 5, 50, 23, 21]\\na=[x for x in v if x>=31 and x%5!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2007\", \"prompt\": \"問題: 数列[32, 9, 31, 43, 57, 37, 4, 16, 8, 57, 5, 59, 4, 21, 4, 22, 45, 40]で、26以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[32, 9, 31, 43, 57, 37, 4, 16, 8, 57, 5, 59, 4, 21, 4, 22, 45, 40]\\na=[x for x in v if x>=26 and x%6!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n59\\n```\\n答え: 59。実行結果を確認しました。\", \"python_code\": \"v=[32, 9, 31, 43, 57, 37, 4, 16, 8, 57, 5, 59, 4, 21, 4, 22, 45, 40]\\na=[x for x in v if x>=26 and x%6!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2008\", \"prompt\": \"問題: 数列[33, 52, 55, 37, 61, 19, 15, 32, 2]で、9以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[33, 52, 55, 37, 61, 19, 15, 32, 2]\\na=[x for x in v if x>=9 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n61\\n```\\n答え: 61。実行結果を確認しました。\", \"python_code\": \"v=[33, 52, 55, 37, 61, 19, 15, 32, 2]\\na=[x for x in v if x>=9 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2009\", \"prompt\": \"問題: 数列[8, 47, 60, 51, 5, 31, 17, 64, 38, 4, 17, 35]で、26以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[8, 47, 60, 51, 5, 31, 17, 64, 38, 4, 17, 35]\\na=[x for x in v if x>=26 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n64\\n```\\n答え: 64。実行結果を確認しました。\", \"python_code\": \"v=[8, 47, 60, 51, 5, 31, 17, 64, 38, 4, 17, 35]\\na=[x for x in v if x>=26 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2010\", \"prompt\": \"問題: 数列[54, 17, 22, 53, 1, 39, 38, 48, 3, 20, 61, 28, 58, 31, 2]で、17以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[54, 17, 22, 53, 1, 39, 38, 48, 3, 20, 61, 28, 58, 31, 2]\\na=[x for x in v if x>=17 and x%5!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n61\\n```\\n答え: 61。実行結果を確認しました。\", \"python_code\": \"v=[54, 17, 22, 53, 1, 39, 38, 48, 3, 20, 61, 28, 58, 31, 2]\\na=[x for x in v if x>=17 and x%5!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2011\", \"prompt\": \"問題: 数列[62, 20, 64, 63, 14, 22, 16, 59, 7, 13, 13, 12, 18, 7, 6, 57, 53, 9]で、25以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[62, 20, 64, 63, 14, 22, 16, 59, 7, 13, 13, 12, 18, 7, 6, 57, 53, 9]\\na=[x for x in v if x>=25 and x%5!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n64\\n```\\n答え: 64。実行結果を確認しました。\", \"python_code\": \"v=[62, 20, 64, 63, 14, 22, 16, 59, 7, 13, 13, 12, 18, 7, 6, 57, 53, 9]\\na=[x for x in v if x>=25 and x%5!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2012\", \"prompt\": \"問題: 数列[47, 52, 39, 52, 17, 42, 64, 10, 32, 3, 54, 14, 4, 32, 64]で、13以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[47, 52, 39, 52, 17, 42, 64, 10, 32, 3, 54, 14, 4, 32, 64]\\na=[x for x in v if x>=13 and x%7!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n64\\n```\\n答え: 64。実行結果を確認しました。\", \"python_code\": \"v=[47, 52, 39, 52, 17, 42, 64, 10, 32, 3, 54, 14, 4, 32, 64]\\na=[x for x in v if x>=13 and x%7!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2013\", \"prompt\": \"問題: 数列[62, 20, 39, 52, 36, 41, 61, 19, 30, 58, 37, 10, 20, 42, 33, 38, 60, 27]で、24以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[62, 20, 39, 52, 36, 41, 61, 19, 30, 58, 37, 10, 20, 42, 33, 38, 60, 27]\\na=[x for x in v if x>=24 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n62\\n```\\n答え: 62。実行結果を確認しました。\", \"python_code\": \"v=[62, 20, 39, 52, 36, 41, 61, 19, 30, 58, 37, 10, 20, 42, 33, 38, 60, 27]\\na=[x for x in v if x>=24 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2014\", \"prompt\": \"問題: 数列[51, 4, 15, 21, 26, 9, 13, 63, 52, 16, 23, 63, 52, 35, 8]で、17以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[51, 4, 15, 21, 26, 9, 13, 63, 52, 16, 23, 63, 52, 35, 8]\\na=[x for x in v if x>=17 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n52\\n```\\n答え: 52。実行結果を確認しました。\", \"python_code\": \"v=[51, 4, 15, 21, 26, 9, 13, 63, 52, 16, 23, 63, 52, 35, 8]\\na=[x for x in v if x>=17 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2015\", \"prompt\": \"問題: 数列[30, 25, 35, 10, 43, 24, 43, 12, 56, 62, 32, 56, 7, 55, 24, 11, 15, 39]で、31以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[30, 25, 35, 10, 43, 24, 43, 12, 56, 62, 32, 56, 7, 55, 24, 11, 15, 39]\\na=[x for x in v if x>=31 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n62\\n```\\n答え: 62。実行結果を確認しました。\", \"python_code\": \"v=[30, 25, 35, 10, 43, 24, 43, 12, 56, 62, 32, 56, 7, 55, 24, 11, 15, 39]\\na=[x for x in v if x>=31 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2016\", \"prompt\": \"問題: 数列[53, 56, 35, 27, 18, 38, 1, 23, 45]で、31以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[53, 56, 35, 27, 18, 38, 1, 23, 45]\\na=[x for x in v if x>=31 and x%2!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n53\\n```\\n答え: 53。実行結果を確認しました。\", \"python_code\": \"v=[53, 56, 35, 27, 18, 38, 1, 23, 45]\\na=[x for x in v if x>=31 and x%2!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2017\", \"prompt\": \"問題: 数列[41, 15, 45, 4, 26, 44, 40, 18, 58, 31, 52, 37]で、13以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[41, 15, 45, 4, 26, 44, 40, 18, 58, 31, 52, 37]\\na=[x for x in v if x>=13 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n58\\n```\\n答え: 58。実行結果を確認しました。\", \"python_code\": \"v=[41, 15, 45, 4, 26, 44, 40, 18, 58, 31, 52, 37]\\na=[x for x in v if x>=13 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2018\", \"prompt\": \"問題: 数列[40, 42, 42, 55, 26, 11, 1, 7, 14, 18, 42, 36, 35, 38, 21]で、17以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[40, 42, 42, 55, 26, 11, 1, 7, 14, 18, 42, 36, 35, 38, 21]\\na=[x for x in v if x>=17 and x%7!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n55\\n```\\n答え: 55。実行結果を確認しました。\", \"python_code\": \"v=[40, 42, 42, 55, 26, 11, 1, 7, 14, 18, 42, 36, 35, 38, 21]\\na=[x for x in v if x>=17 and x%7!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2019\", \"prompt\": \"問題: 数列[36, 40, 7, 27, 58, 19, 2, 23, 10, 9, 8, 29, 59, 55, 19, 60, 63, 34]で、19以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[36, 40, 7, 27, 58, 19, 2, 23, 10, 9, 8, 29, 59, 55, 19, 60, 63, 34]\\na=[x for x in v if x>=19 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n59\\n```\\n答え: 59。実行結果を確認しました。\", \"python_code\": \"v=[36, 40, 7, 27, 58, 19, 2, 23, 10, 9, 8, 29, 59, 55, 19, 60, 63, 34]\\na=[x for x in v if x>=19 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2020\", \"prompt\": \"問題: 数列[40, 38, 29, 62, 19, 59, 41, 42, 62, 62, 13, 19, 8, 5, 46]で、18以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[40, 38, 29, 62, 19, 59, 41, 42, 62, 62, 13, 19, 8, 5, 46]\\na=[x for x in v if x>=18 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n62\\n```\\n答え: 62。実行結果を確認しました。\", \"python_code\": \"v=[40, 38, 29, 62, 19, 59, 41, 42, 62, 62, 13, 19, 8, 5, 46]\\na=[x for x in v if x>=18 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2021\", \"prompt\": \"問題: 数列[61, 2, 49, 54, 43, 33, 44, 38, 32, 54, 2, 5, 3, 60, 30, 10, 8, 64]で、28以上かつ3の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[61, 2, 49, 54, 43, 33, 44, 38, 32, 54, 2, 5, 3, 60, 30, 10, 8, 64]\\na=[x for x in v if x>=28 and x%3!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n64\\n```\\n答え: 64。実行結果を確認しました。\", \"python_code\": \"v=[61, 2, 49, 54, 43, 33, 44, 38, 32, 54, 2, 5, 3, 60, 30, 10, 8, 64]\\na=[x for x in v if x>=28 and x%3!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2022\", \"prompt\": \"問題: 数列[57, 23, 35, 9, 7, 57, 8, 17, 64, 11, 1, 29, 30, 46, 4]で、24以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[57, 23, 35, 9, 7, 57, 8, 17, 64, 11, 1, 29, 30, 46, 4]\\na=[x for x in v if x>=24 and x%6!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n64\\n```\\n答え: 64。実行結果を確認しました。\", \"python_code\": \"v=[57, 23, 35, 9, 7, 57, 8, 17, 64, 11, 1, 29, 30, 46, 4]\\na=[x for x in v if x>=24 and x%6!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2023\", \"prompt\": \"問題: 数列[24, 64, 11, 28, 54, 13, 23, 62, 13, 30, 28, 34, 42, 42, 34, 58, 16, 48]で、9以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[24, 64, 11, 28, 54, 13, 23, 62, 13, 30, 28, 34, 42, 42, 34, 58, 16, 48]\\na=[x for x in v if x>=9 and x%7!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n64\\n```\\n答え: 64。実行結果を確認しました。\", \"python_code\": \"v=[24, 64, 11, 28, 54, 13, 23, 62, 13, 30, 28, 34, 42, 42, 34, 58, 16, 48]\\na=[x for x in v if x>=9 and x%7!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2024\", \"prompt\": \"問題: 数列[61, 33, 52, 20, 53, 44, 33, 34, 2]で、25以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[61, 33, 52, 20, 53, 44, 33, 34, 2]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n61\\n```\\n答え: 61。実行結果を確認しました。\", \"python_code\": \"v=[61, 33, 52, 20, 53, 44, 33, 34, 2]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2025\", \"prompt\": \"問題: 数列[50, 43, 15, 1, 49, 51, 26, 26, 47, 59, 1, 7]で、27以上かつ7の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[50, 43, 15, 1, 49, 51, 26, 26, 47, 59, 1, 7]\\na=[x for x in v if x>=27 and x%7!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n59\\n```\\n答え: 59。実行結果を確認しました。\", \"python_code\": \"v=[50, 43, 15, 1, 49, 51, 26, 26, 47, 59, 1, 7]\\na=[x for x in v if x>=27 and x%7!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2026\", \"prompt\": \"問題: 数列[23, 62, 6, 51, 8, 28, 5, 10, 23, 54, 4, 56, 53, 46, 16]で、30以上かつ5の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[23, 62, 6, 51, 8, 28, 5, 10, 23, 54, 4, 56, 53, 46, 16]\\na=[x for x in v if x>=30 and x%5!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n62\\n```\\n答え: 62。実行結果を確認しました。\", \"python_code\": \"v=[23, 62, 6, 51, 8, 28, 5, 10, 23, 54, 4, 56, 53, 46, 16]\\na=[x for x in v if x>=30 and x%5!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2027\", \"prompt\": \"問題: 数列[49, 56, 21, 7, 47, 40, 60, 15, 28, 2, 22, 29, 65, 47, 52, 47, 4, 14]で、19以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[49, 56, 21, 7, 47, 40, 60, 15, 28, 2, 22, 29, 65, 47, 52, 47, 4, 14]\\na=[x for x in v if x>=19 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n65\\n```\\n答え: 65。実行結果を確認しました。\", \"python_code\": \"v=[49, 56, 21, 7, 47, 40, 60, 15, 28, 2, 22, 29, 65, 47, 52, 47, 4, 14]\\na=[x for x in v if x>=19 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2028\", \"prompt\": \"問題: 数列[16, 62, 18, 10, 29, 39, 42, 17, 39, 60, 25, 56, 62, 21, 47]で、20以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[16, 62, 18, 10, 29, 39, 42, 17, 39, 60, 25, 56, 62, 21, 47]\\na=[x for x in v if x>=20 and x%2!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n47\\n```\\n答え: 47。実行結果を確認しました。\", \"python_code\": \"v=[16, 62, 18, 10, 29, 39, 42, 17, 39, 60, 25, 56, 62, 21, 47]\\na=[x for x in v if x>=20 and x%2!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2029\", \"prompt\": \"問題: 数列[37, 5, 6, 61, 38, 45, 28, 30, 12, 42, 29, 5, 60, 48, 43, 34, 51, 23]で、10以上かつ2の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[37, 5, 6, 61, 38, 45, 28, 30, 12, 42, 29, 5, 60, 48, 43, 34, 51, 23]\\na=[x for x in v if x>=10 and x%2!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n61\\n```\\n答え: 61。実行結果を確認しました。\", \"python_code\": \"v=[37, 5, 6, 61, 38, 45, 28, 30, 12, 42, 29, 5, 60, 48, 43, 34, 51, 23]\\na=[x for x in v if x>=10 and x%2!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2030\", \"prompt\": \"問題: 数列[48, 40, 16, 1, 46, 11, 1, 21, 8, 27, 48, 42, 24, 35, 16]で、13以上かつ6の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[48, 40, 16, 1, 46, 11, 1, 21, 8, 27, 48, 42, 24, 35, 16]\\na=[x for x in v if x>=13 and x%6!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n46\\n```\\n答え: 46。実行結果を確認しました。\", \"python_code\": \"v=[48, 40, 16, 1, 46, 11, 1, 21, 8, 27, 48, 42, 24, 35, 16]\\na=[x for x in v if x>=13 and x%6!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_34_2031\", \"prompt\": \"問題: 数列[35, 55, 25, 38, 33, 10, 9, 13, 20, 52, 40, 2, 22, 18, 62, 18, 46, 29]で、11以上かつ4の倍数ではない値だけを選び、最大値（該当なしなら0）を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[35, 55, 25, 38, 33, 10, 9, 13, 20, 52, 40, 2, 22, 18, 62, 18, 46, 29]\\na=[x for x in v if x>=11 and x%4!=0]\\nprint(max(a, default=0))\\n```\\n```output\\n62\\n```\\n答え: 62。実行結果を確認しました。\", \"python_code\": \"v=[35, 55, 25, 38, 33, 10, 9, 13, 20, 52, 40, 2, 22, 18, 62, 18, 46, 29]\\na=[x for x in v if x>=11 and x%4!=0]\\nprint(max(a, default=0))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2000\", \"prompt\": \"問題: 数列[57, 26, 45, 26, 25, 44, 46, 65, 53]で、21以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[57, 26, 45, 26, 25, 44, 46, 65, 53]\\na=[x for x in v if x>=21 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n18337\\n```\\n答え: 18337。実行結果を確認しました。\", \"python_code\": \"v=[57, 26, 45, 26, 25, 44, 46, 65, 53]\\na=[x for x in v if x>=21 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2001\", \"prompt\": \"問題: 数列[62, 37, 12, 42, 45, 49, 41, 40, 43, 20, 10, 27]で、12以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[62, 37, 12, 42, 45, 49, 41, 40, 43, 20, 10, 27]\\na=[x for x in v if x>=12 and x%5!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n13781\\n```\\n答え: 13781。実行結果を確認しました。\", \"python_code\": \"v=[62, 37, 12, 42, 45, 49, 41, 40, 43, 20, 10, 27]\\na=[x for x in v if x>=12 and x%5!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2002\", \"prompt\": \"問題: 数列[58, 43, 41, 47, 32, 53, 50, 25, 16, 20, 4, 4, 38, 31, 47]で、15以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[58, 43, 41, 47, 32, 53, 50, 25, 16, 20, 4, 4, 38, 31, 47]\\na=[x for x in v if x>=15 and x%3!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n21331\\n```\\n答え: 21331。実行結果を確認しました。\", \"python_code\": \"v=[58, 43, 41, 47, 32, 53, 50, 25, 16, 20, 4, 4, 38, 31, 47]\\na=[x for x in v if x>=15 and x%3!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2003\", \"prompt\": \"問題: 数列[7, 32, 10, 4, 7, 39, 54, 64, 57, 2, 25, 52, 37, 56, 1, 26, 52, 9]で、27以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[7, 32, 10, 4, 7, 39, 54, 64, 57, 2, 25, 52, 37, 56, 1, 26, 52, 9]\\na=[x for x in v if x>=27 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n6139\\n```\\n答え: 6139。実行結果を確認しました。\", \"python_code\": \"v=[7, 32, 10, 4, 7, 39, 54, 64, 57, 2, 25, 52, 37, 56, 1, 26, 52, 9]\\na=[x for x in v if x>=27 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2004\", \"prompt\": \"問題: 数列[60, 47, 56, 58, 36, 12, 62, 42, 22, 62, 24, 26, 5, 19, 5]で、16以上かつ4の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[60, 47, 56, 58, 36, 12, 62, 42, 22, 62, 24, 26, 5, 19, 5]\\na=[x for x in v if x>=16 and x%4!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n16546\\n```\\n答え: 16546。実行結果を確認しました。\", \"python_code\": \"v=[60, 47, 56, 58, 36, 12, 62, 42, 22, 62, 24, 26, 5, 19, 5]\\na=[x for x in v if x>=16 and x%4!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2005\", \"prompt\": \"問題: 数列[61, 18, 65, 20, 31, 16, 24, 54, 19, 13, 56, 10, 1, 18, 52, 48, 45, 51]で、16以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[61, 18, 65, 20, 31, 16, 24, 54, 19, 13, 56, 10, 1, 18, 52, 48, 45, 51]\\na=[x for x in v if x>=16 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n13894\\n```\\n答え: 13894。実行結果を確認しました。\", \"python_code\": \"v=[61, 18, 65, 20, 31, 16, 24, 54, 19, 13, 56, 10, 1, 18, 52, 48, 45, 51]\\na=[x for x in v if x>=16 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2006\", \"prompt\": \"問題: 数列[10, 28, 37, 6, 63, 36, 60, 32, 55, 60, 5, 23, 2, 12, 10]で、15以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[10, 28, 37, 6, 63, 36, 60, 32, 55, 60, 5, 23, 2, 12, 10]\\na=[x for x in v if x>=15 and x%5!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n8971\\n```\\n答え: 8971。実行結果を確認しました。\", \"python_code\": \"v=[10, 28, 37, 6, 63, 36, 60, 32, 55, 60, 5, 23, 2, 12, 10]\\na=[x for x in v if x>=15 and x%5!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2007\", \"prompt\": \"問題: 数列[37, 25, 63, 19, 65, 56, 50, 13, 14, 39, 2, 53, 26, 53, 10, 58, 35, 43]で、25以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[37, 25, 63, 19, 65, 56, 50, 13, 14, 39, 2, 53, 26, 53, 10, 58, 35, 43]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n20401\\n```\\n答え: 20401。実行結果を確認しました。\", \"python_code\": \"v=[37, 25, 63, 19, 65, 56, 50, 13, 14, 39, 2, 53, 26, 53, 10, 58, 35, 43]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2008\", \"prompt\": \"問題: 数列[31, 5, 43, 18, 52, 48, 14, 59, 54]で、9以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[31, 5, 43, 18, 52, 48, 14, 59, 54]\\na=[x for x in v if x>=9 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n14539\\n```\\n答え: 14539。実行結果を確認しました。\", \"python_code\": \"v=[31, 5, 43, 18, 52, 48, 14, 59, 54]\\na=[x for x in v if x>=9 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2009\", \"prompt\": \"問題: 数列[64, 62, 17, 35, 6, 10, 15, 63, 16, 2, 10, 17]で、18以上かつ4の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[64, 62, 17, 35, 6, 10, 15, 63, 16, 2, 10, 17]\\na=[x for x in v if x>=18 and x%4!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n9038\\n```\\n答え: 9038。実行結果を確認しました。\", \"python_code\": \"v=[64, 62, 17, 35, 6, 10, 15, 63, 16, 2, 10, 17]\\na=[x for x in v if x>=18 and x%4!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2010\", \"prompt\": \"問題: 数列[45, 55, 35, 31, 14, 34, 43, 24, 25, 23, 28, 15, 8, 41, 35]で、15以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[45, 55, 35, 31, 14, 34, 43, 24, 25, 23, 28, 15, 8, 41, 35]\\na=[x for x in v if x>=15 and x%5!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n7536\\n```\\n答え: 7536。実行結果を確認しました。\", \"python_code\": \"v=[45, 55, 35, 31, 14, 34, 43, 24, 25, 23, 28, 15, 8, 41, 35]\\na=[x for x in v if x>=15 and x%5!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2011\", \"prompt\": \"問題: 数列[59, 30, 34, 35, 59, 26, 65, 61, 34, 31, 5, 63, 65, 47, 21, 25, 51, 61]で、20以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[59, 30, 34, 35, 59, 26, 65, 61, 34, 31, 5, 63, 65, 47, 21, 25, 51, 61]\\na=[x for x in v if x>=20 and x%6!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n37873\\n```\\n答え: 37873。実行結果を確認しました。\", \"python_code\": \"v=[59, 30, 34, 35, 59, 26, 65, 61, 34, 31, 5, 63, 65, 47, 21, 25, 51, 61]\\na=[x for x in v if x>=20 and x%6!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2012\", \"prompt\": \"問題: 数列[20, 52, 22, 10, 2, 20, 1, 57, 4, 26, 42, 41, 10, 11, 34]で、16以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[20, 52, 22, 10, 2, 20, 1, 57, 4, 26, 42, 41, 10, 11, 34]\\na=[x for x in v if x>=16 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n10750\\n```\\n答え: 10750。実行結果を確認しました。\", \"python_code\": \"v=[20, 52, 22, 10, 2, 20, 1, 57, 4, 26, 42, 41, 10, 11, 34]\\na=[x for x in v if x>=16 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2013\", \"prompt\": \"問題: 数列[64, 52, 40, 1, 44, 8, 43, 58, 13, 47, 58, 53, 29, 52, 52, 51, 13, 31]で、9以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[64, 52, 40, 1, 44, 8, 43, 58, 13, 47, 58, 53, 29, 52, 52, 51, 13, 31]\\na=[x for x in v if x>=9 and x%6!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n34080\\n```\\n答え: 34080。実行結果を確認しました。\", \"python_code\": \"v=[64, 52, 40, 1, 44, 8, 43, 58, 13, 47, 58, 53, 29, 52, 52, 51, 13, 31]\\na=[x for x in v if x>=9 and x%6!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2014\", \"prompt\": \"問題: 数列[46, 12, 2, 30, 52, 52, 54, 5, 55, 53, 37, 7, 29, 60, 4]で、15以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[46, 12, 2, 30, 52, 52, 54, 5, 55, 53, 37, 7, 29, 60, 4]\\na=[x for x in v if x>=15 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n8044\\n```\\n答え: 8044。実行結果を確認しました。\", \"python_code\": \"v=[46, 12, 2, 30, 52, 52, 54, 5, 55, 53, 37, 7, 29, 60, 4]\\na=[x for x in v if x>=15 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2015\", \"prompt\": \"問題: 数列[37, 7, 10, 54, 57, 64, 17, 48, 46, 55, 52, 3, 33, 17, 61, 30, 21, 3]で、20以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[37, 7, 10, 54, 57, 64, 17, 48, 46, 55, 52, 3, 33, 17, 61, 30, 21, 3]\\na=[x for x in v if x>=20 and x%3!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n17031\\n```\\n答え: 17031。実行結果を確認しました。\", \"python_code\": \"v=[37, 7, 10, 54, 57, 64, 17, 48, 46, 55, 52, 3, 33, 17, 61, 30, 21, 3]\\na=[x for x in v if x>=20 and x%3!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2016\", \"prompt\": \"問題: 数列[17, 15, 25, 18, 36, 10, 19, 32, 22]で、24以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[17, 15, 25, 18, 36, 10, 19, 32, 22]\\na=[x for x in v if x>=24 and x%6!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n1649\\n```\\n答え: 1649。実行結果を確認しました。\", \"python_code\": \"v=[17, 15, 25, 18, 36, 10, 19, 32, 22]\\na=[x for x in v if x>=24 and x%6!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2017\", \"prompt\": \"問題: 数列[57, 36, 55, 1, 19, 12, 49, 34, 61, 13, 57, 4]で、9以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[57, 36, 55, 1, 19, 12, 49, 34, 61, 13, 57, 4]\\na=[x for x in v if x>=9 and x%5!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n15746\\n```\\n答え: 15746。実行結果を確認しました。\", \"python_code\": \"v=[57, 36, 55, 1, 19, 12, 49, 34, 61, 13, 57, 4]\\na=[x for x in v if x>=9 and x%5!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2018\", \"prompt\": \"問題: 数列[60, 22, 25, 8, 1, 3, 16, 16, 58, 13, 12, 10, 18, 57, 59]で、9以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[60, 22, 25, 8, 1, 3, 16, 16, 58, 13, 12, 10, 18, 57, 59]\\na=[x for x in v if x>=9 and x%5!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n11727\\n```\\n答え: 11727。実行結果を確認しました。\", \"python_code\": \"v=[60, 22, 25, 8, 1, 3, 16, 16, 58, 13, 12, 10, 18, 57, 59]\\na=[x for x in v if x>=9 and x%5!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2019\", \"prompt\": \"問題: 数列[57, 46, 65, 21, 8, 9, 21, 63, 10, 7, 30, 62, 11, 27, 19, 56, 19, 7]で、22以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[57, 46, 65, 21, 8, 9, 21, 63, 10, 7, 30, 62, 11, 27, 19, 56, 19, 7]\\na=[x for x in v if x>=22 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n15063\\n```\\n答え: 15063。実行結果を確認しました。\", \"python_code\": \"v=[57, 46, 65, 21, 8, 9, 21, 63, 10, 7, 30, 62, 11, 27, 19, 56, 19, 7]\\na=[x for x in v if x>=22 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2020\", \"prompt\": \"問題: 数列[11, 21, 56, 49, 27, 14, 51, 12, 11, 51, 35, 63, 55, 19, 65]で、25以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[11, 21, 56, 49, 27, 14, 51, 12, 11, 51, 35, 63, 55, 19, 65]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n20776\\n```\\n答え: 20776。実行結果を確認しました。\", \"python_code\": \"v=[11, 21, 56, 49, 27, 14, 51, 12, 11, 51, 35, 63, 55, 19, 65]\\na=[x for x in v if x>=25 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2021\", \"prompt\": \"問題: 数列[16, 38, 55, 4, 40, 45, 59, 39, 8, 48, 53, 1, 47, 34, 33, 19, 20, 34]で、12以上かつ6の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[16, 38, 55, 4, 40, 45, 59, 39, 8, 48, 53, 1, 47, 34, 33, 19, 20, 34]\\na=[x for x in v if x>=12 and x%6!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n22532\\n```\\n答え: 22532。実行結果を確認しました。\", \"python_code\": \"v=[16, 38, 55, 4, 40, 45, 59, 39, 8, 48, 53, 1, 47, 34, 33, 19, 20, 34]\\na=[x for x in v if x>=12 and x%6!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2022\", \"prompt\": \"問題: 数列[33, 27, 23, 59, 1, 4, 53, 26, 65, 38, 6, 4, 55, 43, 1]で、21以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[33, 27, 23, 59, 1, 4, 53, 26, 65, 38, 6, 4, 55, 43, 1]\\na=[x for x in v if x>=21 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n19856\\n```\\n答え: 19856。実行結果を確認しました。\", \"python_code\": \"v=[33, 27, 23, 59, 1, 4, 53, 26, 65, 38, 6, 4, 55, 43, 1]\\na=[x for x in v if x>=21 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2023\", \"prompt\": \"問題: 数列[11, 14, 41, 47, 26, 55, 35, 15, 47, 44, 39, 39, 27, 62, 28, 3, 36, 15]で、23以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[11, 14, 41, 47, 26, 55, 35, 15, 47, 44, 39, 39, 27, 62, 28, 3, 36, 15]\\na=[x for x in v if x>=23 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n20647\\n```\\n答え: 20647。実行結果を確認しました。\", \"python_code\": \"v=[11, 14, 41, 47, 26, 55, 35, 15, 47, 44, 39, 39, 27, 62, 28, 3, 36, 15]\\na=[x for x in v if x>=23 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2024\", \"prompt\": \"問題: 数列[63, 38, 47, 64, 16, 62, 13, 9, 46]で、27以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[63, 38, 47, 64, 16, 62, 13, 9, 46]\\na=[x for x in v if x>=27 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n6178\\n```\\n答え: 6178。実行結果を確認しました。\", \"python_code\": \"v=[63, 38, 47, 64, 16, 62, 13, 9, 46]\\na=[x for x in v if x>=27 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2025\", \"prompt\": \"問題: 数列[24, 4, 37, 23, 5, 56, 6, 57, 14, 38, 27, 20]で、29以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[24, 4, 37, 23, 5, 56, 6, 57, 14, 38, 27, 20]\\na=[x for x in v if x>=29 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n4618\\n```\\n答え: 4618。実行結果を確認しました。\", \"python_code\": \"v=[24, 4, 37, 23, 5, 56, 6, 57, 14, 38, 27, 20]\\na=[x for x in v if x>=29 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2026\", \"prompt\": \"問題: 数列[36, 26, 11, 57, 8, 12, 22, 24, 23, 65, 22, 35, 56, 51, 4]で、22以上かつ5の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[36, 26, 11, 57, 8, 12, 22, 24, 23, 65, 22, 35, 56, 51, 4]\\na=[x for x in v if x>=22 and x%5!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n13031\\n```\\n答え: 13031。実行結果を確認しました。\", \"python_code\": \"v=[36, 26, 11, 57, 8, 12, 22, 24, 23, 65, 22, 35, 56, 51, 4]\\na=[x for x in v if x>=22 and x%5!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2027\", \"prompt\": \"問題: 数列[42, 25, 5, 15, 37, 5, 23, 37, 39, 59, 21, 21, 41, 65, 59, 12, 12, 16]で、11以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[42, 25, 5, 15, 37, 5, 23, 37, 39, 59, 21, 21, 41, 65, 59, 12, 12, 16]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n17016\\n```\\n答え: 17016。実行結果を確認しました。\", \"python_code\": \"v=[42, 25, 5, 15, 37, 5, 23, 37, 39, 59, 21, 21, 41, 65, 59, 12, 12, 16]\\na=[x for x in v if x>=11 and x%3!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2028\", \"prompt\": \"問題: 数列[45, 40, 3, 33, 38, 53, 24, 47, 1, 2, 41, 1, 17, 46, 54]で、29以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[45, 40, 3, 33, 38, 53, 24, 47, 1, 2, 41, 1, 17, 46, 54]\\na=[x for x in v if x>=29 and x%3!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n11859\\n```\\n答え: 11859。実行結果を確認しました。\", \"python_code\": \"v=[45, 40, 3, 33, 38, 53, 24, 47, 1, 2, 41, 1, 17, 46, 54]\\na=[x for x in v if x>=29 and x%3!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2029\", \"prompt\": \"問題: 数列[42, 1, 64, 52, 47, 6, 58, 21, 27, 23, 9, 55, 29, 15, 3, 27, 32, 63]で、19以上かつ2の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[42, 1, 64, 52, 47, 6, 58, 21, 27, 23, 9, 55, 29, 15, 3, 27, 32, 63]\\na=[x for x in v if x>=19 and x%2!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n12472\\n```\\n答え: 12472。実行結果を確認しました。\", \"python_code\": \"v=[42, 1, 64, 52, 47, 6, 58, 21, 27, 23, 9, 55, 29, 15, 3, 27, 32, 63]\\na=[x for x in v if x>=19 and x%2!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2030\", \"prompt\": \"問題: 数列[33, 11, 38, 33, 14, 4, 52, 17, 35, 58, 20, 47, 29, 7, 19]で、18以上かつ3の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[33, 11, 38, 33, 14, 4, 52, 17, 35, 58, 20, 47, 29, 7, 19]\\na=[x for x in v if x>=18 and x%3!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n12548\\n```\\n答え: 12548。実行結果を確認しました。\", \"python_code\": \"v=[33, 11, 38, 33, 14, 4, 52, 17, 35, 58, 20, 47, 29, 7, 19]\\na=[x for x in v if x>=18 and x%3!=0]\\nprint(sum(x*x for x in a))\"}\n"
  "{\"id\": \"v48_python_unseen_eval_35_2031\", \"prompt\": \"問題: 数列[13, 3, 55, 58, 7, 39, 48, 28, 65, 44, 33, 30, 24, 32, 15, 65, 19, 17]で、10以上かつ7の倍数ではない値だけを選び、二乗和を求めてください。 操作の順序と例外条件を確認し、途中の値を取り違えないでください。\\n道具: Python\\n思考:\", \"target\": \"\\n必要な条件を先に確認し、最終値を検算します。\\n```python\\nv=[13, 3, 55, 58, 7, 39, 48, 28, 65, 44, 33, 30, 24, 32, 15, 65, 19, 17]\\na=[x for x in v if x>=10 and x%7!=0]\\nprint(sum(x*x for x in a))\\n```\\n```output\\n25233\\n```\\n答え: 25233。実行結果を確認しました。\", \"python_code\": \"v=[13, 3, 55, 58, 7, 39, 48, 28, 65, 44, 33, 30, 24, 32, 15, 65, 19, 17]\\na=[x for x in v if x>=10 and x%7!=0]\\nprint(sum(x*x for x in a))\"}\n"
 ),
}

CODE = Path("/content/v49")
CODE.mkdir(parents=True, exist_ok=True)
for name, text in FILES.items():
    (CODE / name).write_text(text, encoding="utf-8")
EXTRA = CODE / "eval_extra"
EXTRA.mkdir(exist_ok=True)
for name, text in EVAL_EXTRA.items():
    (EXTRA / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(CODE))
for _m in ("acquire", "checks", "convert", "pipeline"):          # 同じカーネルで2回目に動かすとき、古い版ではなくこの版を読み込むため
    sys.modules.pop(_m, None)
import acquire, checks, convert  # noqa: E401,F401
import pipeline as P

for _m in (acquire, checks, convert, P):                          # 読み込まれたコードが、いま書き出した文面と同じことを確かめる
    _file = Path(_m.__file__)
    assert _file.parent == CODE, f"FAIL [3b modules] {_m.__name__} が {CODE} 以外から読み込まれました"
    assert hashlib.sha256(_file.read_bytes()).hexdigest() == hashlib.sha256(FILES[_file.name].encode("utf-8")).hexdigest(), f"FAIL [3b modules] {_m.__name__} が埋め込みの版と違います。「ランタイム → セッションを再起動」してください"


def _detach():                                                    # コードを実行するあいだ: Drive を外す
    drive.flush_and_unmount()
    assert not os.path.exists("/content/drive/MyDrive"), "FAIL [drive] Drive が外れていません"


def _attach():
    drive.mount("/content/drive")
    assert os.path.isdir(DRIVE_ROOT), f"FAIL [drive] {DRIVE_ROOT} が見つかりません"


def _sync():                                                      # 書き込みを確実にする: 外してつなぎ直す（Drive の書き込みは非同期）
    drive.flush_and_unmount()
    drive.mount("/content/drive")
    assert os.path.isdir(DRIVE_ROOT), f"FAIL [drive] {DRIVE_ROOT} が見つかりません"


CFG = P.make_cfg(root=DRIVE_ROOT, suffix=RUN_SUFFIX, raw_suffix=RAW_SUFFIX, only=ONLY, stop_on_count_mismatch=STOP_ON_COUNT_MISMATCH,
                 allow_vocab_mismatch=ALLOW_VOCAB_MISMATCH, eval_extra_dir=str(EXTRA), run_sandbox=RUN_SANDBOX, shard_records=SHARD_RECORDS,
                 shard_keys=tuple(SHARD_KEYS), max_unsharded_minutes=MAX_UNSHARDED_MINUTES,
                 drive_detach=_detach if UNMOUNT_DRIVE_DURING_CODE_RUNS else None, drive_attach=_attach if UNMOUNT_DRIVE_DURING_CODE_RUNS else None,
                 drive_sync=_sync if FLUSH_DRIVE_AFTER_STEPS else None)
R3 = P.step_modules(CFG)
assert R3["pass"], R3["detail"]


## 3c. Drive の予行演習 / 3d. サンドボックスの自己診断
3c は Drive を数回つなぎ直して、許可の画面がそのたびに出るかを先に確かめます（何も書きません）。3d は、データ中のコードを動かす前に、この Colab の上で安全な実行ができることを確かめます（coding を処理するときだけ）。確かめられなければ止まります。


In [ ]:
# 3c. Drive のつなぎ直しの予行演習（何も書きません）: 全体で何回つなぎ直すか、許可の画面がそのたびに出るかを、先に確かめます
R3c = P.step_drive_probe(CFG)
assert R3c["pass"], R3c["detail"]


In [ ]:
# 3d. サンドボックスの自己診断: データ中のコードを実行する前に、この Colab の上で確かめます（Drive はまだつながっています）。coding を処理しない実行では飛ばします
R3d = P.step_sandbox_selftest(CFG)
assert R3d["pass"], R3d["detail"]


## 4. 試運転
取ってくるファイルと合計バイト数を表示するだけです。通信も書き込みもありません。


In [ ]:
# 4. 試運転: 何をどれだけ取ってくるかだけを表示します（通信なし・書き込みなし）
R4 = P.step_dry_run(CFG)
assert R4["pass"], R4["detail"]
print("取得予定の合計バイト数:", f"{R4['expected_total_bytes']:,}")


## 5. 取得
ライセンスをすべて確認してからデータを取ります。取得済みで正しいファイルは飛ばします。別の内容のファイルが既にあると止まります。


In [ ]:
# 5. 6つを取得（再開できます）。ライセンス確認 → コミット固定 → 大きさと sha256 の照合 → Drive の新しいフォルダへ
R5 = P.step_acquire(CFG)
assert R5["pass"], R5["detail"]


## 6. 元データの検査
6a で train のファイルだけをこの Colab のディスクに写し、6b で語彙と評価データを確かめ、6c で件数・語彙外文字・長さを調べます。


In [ ]:
# 6a. 学習に使ってよいファイル（train）だけを、この Colab のディスクに写して sha256 を照合します（dev / validation は写しません）
R6a = P.step_mirror(CFG)
assert R6a["pass"], R6a["detail"]


In [ ]:
# 6b. モデルの語彙（3,898 件）と評価データの場所を確かめ、13文字の断片の索引を作ります
R6b = P.step_vocab_eval(CFG)
assert R6b["pass"], R6b["detail"]


In [ ]:
# 6c. 元データの検査: 件数をカードと照合（違えば MISMATCH）、語彙外の文字、長さ、読んで確かめる100行
R6c = P.step_raw_checks(CFG)
assert R6c["pass"], R6c["detail"]
for m in R6c["mismatches"]:
    print("MISMATCH:", m)
print("読んで確かめる100行（元データ）:", CFG.out / "reading")


## 7. 変換
7a は最初の2,000件だけの試運転、7b が本番です。7b は時間がかかります（コードの実行検証）。コードを実行するあいだ Drive は外れます。


In [ ]:
# 7a. 変換の試運転: 各データセットの最初の2,000件だけ。行の形・長さ・語彙・マスクを検査し、通らなければ止まります
R7a = P.step_convert_smoke(CFG)
assert R7a["pass"], R7a["detail"]


In [ ]:
# 7b. 変換の本番。長い2つは小分け（シャード）にして、できた分から Drive の external_data_v49_stageA/ に残します（再実行では Drive から戻します）。タブは開いたままに
R7b = P.step_convert_full(CFG)
assert R7b["pass"], R7b["detail"]


## 8. 絞り込み
重複と評価データとの重複を取り除きます。各段階の件数が表になります。結果はこのセルの終わりに Drive に書かれます。8b は任意の確認で、時間の上限があります。


In [ ]:
# 8. 絞り込み: 完全一致 → 近い重複（MinHash）→ 評価データとの13文字一致 → 旧コーパス423,348行との重複 → dev / heldout の近い兄弟
R8 = P.step_filters(CFG)
assert R8["pass"], R8["detail"]


In [ ]:
# 8b. 任意の確認（評価問題に似た行の目視用リスト・データセット間の重複）。時間の上限つきで、行は除きません
R8b = P.step_reviews(CFG)
assert R8b["pass"], R8b["detail"]


## 9. 書き出し
最終の行・registry.json・REPORT_JA.md を Drive の新しいフォルダに書き、つなぎ直したあとのDriveから全ファイルの sha256 を確かめます。


In [ ]:
# 9. 最終の行（train / dev / heldout）を external_data_v49_rows/<データセット>/ に（保留の2つは _HOLD_do_not_mix/ に）、registry.json と REPORT_JA.md を v49_acquire_20261109/ に書きます
R9 = P.step_finalize(CFG)
assert R9["pass"], R9["detail"]


## 10. 終わり


In [ ]:
# 10. 報告を表示し、Drive への書き込みを確実に終わらせます
rep = CFG.out / "REPORT_JA.md"
assert rep.is_file(), "FAIL [10 finish] REPORT_JA.md がありません。セル9を先に実行してください"
print(rep.read_text(encoding="utf-8"))
print("PASS [10 finish] 報告:", rep)
from google.colab import drive

drive.flush_and_unmount()
print("Drive の書き込みを終えてアンマウントしました。続けて作業するときは、セル3a から実行し直してください。")
